# Демографический советник · ML-05 V7.3.0 — Advanced Data Reconstruction

**Полный автономный Google Colab:** 52 закреплённых источника Stage I (7.2), семь исследовательских модулей, дополнительные строго наблюдаемые реконструкции, сохранение модели/прогнозов, независимая QA, полный evidence в приватном GitHub и автоскачивание компактной аналитики.

**Принцип:** если целевой показатель не подтверждён исходником, опыт фиксируется `SKIPPED_NOT_OBSERVABLE`. Никакой синтетической мобильности, дорасчёта отсутствующих OD-рёбер или автоматического присвоения кампусам адреса по ближайшему населённому пункту. Все результаты V7 сохраняются, а дополнительные эксперименты маркируются отдельно.

**Доступ:** секрет Colab `TRIAD-RU-Colab`. Для запуска: `Runtime → Run all` в свежем сеансе. Автоматическая замена действует только на `stage2/ml05_education/` после технической проверки; исходные Stage I и другие ML остаются неизменными.


**V7.3.0 fixes:** обязательное раннее исполнение DuckDB SQL (до GitHub), атомарная реконструкция E02, независимые индикаторы прогресса и перенос финальных отметок этапов после всех подмодулей. Остальные модели V7.1 сохранены.


## 1. Единственная конфигурационная ячейка
Первый запуск: `MODE="SMOKE"` и `AUTO_PUBLISH_AFTER_FULL_PASS=False`. Для последующей валидации используйте `PILOT10`, затем `FULL`. Точные имена показателей для прогноза приёма можно утвердить только после первичного анализа словаря. Меняется только эта ячейка.

In [ ]:
# Единственная ячейка, в которой пользователь меняет параметры эксперимента.
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class ML5Config:
    VERSION: str = 'stage2-ml5-v7.3.0-preserved-full-research-verified-release'
    MODE: str = 'FULL'             # SMOKE | PILOT10 | FULL; FULL по всем строкам без загрузки всего E04/E05 в RAM
    REPO: str = 'AntonDozhdikov/Demograf_sovetnik'
    SOURCE_BRANCH: str = 'main'
    INPUT_INDEX: str = 'handoff/07_v2/dataset_index.json'
    EXPECTED_INDEX_BLOB_SHA: str = 'd6625ccddc24f24a1ee451e33c48b736d3a57043'
    TARGET_BRANCH: str = 'ml05-education-20261102T160726Z'
    OUTPUT_PREFIX: str = 'stage2/ml05_education'
    WORKDIR: str = '/content/ml5_v7_3_0'
    FEATURE_PUBLICATION_LAG_YEARS: int = 1
    LABEL_PUBLICATION_LAG_YEARS: int = 1
    HORIZON: int = 1
    ALLOW_RETROSPECTIVE_ONLY: bool = True  # Все исторические данные доступны в текущем, а не историческом vintage.
    REQUIRE_ORIGINAL_E01_FOR_PREDICTION: bool = True
    APPROVE_UNAMBIGUOUS_INDICATORS: bool = False  # Показатель для прогноза утверждается явно, не эвристикой
    APPROVED_VO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E02, проверенные по методике
    APPROVED_SPO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E01/СПО
    STRICT_OD_COVERAGE_BREAK: bool = True   # При разрыве допускается только описательная диагностика / разведочный тест
    ALLOW_RETROSPECTIVE_TESTS: bool = True  # Не выдавать за operational historical backtest
    MIN_YEARS_FOR_TEMPORAL: int = 3
    MIN_TRAIN_ROWS: int = 100   # For regional ML-5.3 overridden by REGION_MIN_TRAIN_ROWS
    MIN_DEV_ROWS: int = 20
    MIN_TEST_ROWS: int = 20
    MIN_MUNICIPALITIES_E0E1: int = 35
    MAX_REMOTE_INPUT_GIB: float = 40.0
    MAX_READ_ROWS_SMOKE: int = 120_000
    BATCH_ROWS: int = 20_000
    DUCKDB_MEMORY_LIMIT: str = '512MB'
    DUCKDB_THREADS: int = 1
    REGION_MIN_TRAIN_ROWS: int = 30
    REGION_MIN_TEST_ROWS: int = 25
    CATALOG_FULL_DOWNLOAD: bool = True   # FULL: strict 52/52 sha256 verified, no hidden skip
    CATALOG_SCAN_ALL_YEARS: bool = True  # all recorded year labels, without pandas materialization
    STAGE1_DISK_RESERVE_GIB: float = 5.0
    EXTRA_CATALOG_PROFILE_MAX_GROUPS: int = 1500
    MAX_PANDAS_ROWS: int = 150_000
    PANEL_IN_MEMORY_MAX_ROWS: int = 15_000
    PANEL_PREVIEW_ROWS: int = 2_000
    EXPORT_PARQUET_CHUNK_ROWS: int = 10_000
    EXCEL_PREVIEW_ROWS: int = 3_000
    MIN_FREE_RAM_GIB: float = 2.0
    BOOTSTRAP_REPS: int = 399
    ML54_FULL_NO_STRATA_CAP: bool = True
    ML54_FINAL_SEEDS: int = 50
    MAX_COUNT_PREDICTION_MULTIPLIER: float = 10.0
    REQUIRE_UNIQUE_MODEL_ARTIFACTS: bool = True
    REQUIRE_SPATIAL_BASELINES: bool = True
    RANDOM_SEED: int = 20261104
    DEV_SEEDS: tuple = (11,23,37,41,53,67,79,83,97,101)
    FINAL_SEEDS: tuple = tuple(range(101,151))
    AUTO_PUBLISH_AFTER_FULL_PASS: bool = True  # Публикует только финальный проверенный этап V7.
    PUBLISH_RESULTS_TO_PRIVATE_GITHUB: bool = True   # Только отдельный закрытый Release + свой сегмент ветки.
    DOWNLOAD_COMPACT_REPORTS: bool = True              # Скачать отчёты ДО долгой загрузки в GitHub.
    GH_RELEASE_PART_MIB: int = 160                    # Ограничить размер одного Release asset.
    GH_REQUIRE_ASSET_DIGEST: bool = True
    REQUIRE_COMPLETED_COMPUTATIONAL_CONTRACT: bool = True
    PUBLISH_PARTIAL_RUN_WITH_EXPLICIT_FAIL_STATUS: bool = False
    STORE_DERIVED_RESTRICTED_DATA: bool = True  # Только внутри приватного репозитория, при допустимости лицензии.
    PRINT_MAX_ROWS: int = 12

CFG=ML5Config()
assert CFG.MODE in ('SMOKE','PILOT10','FULL')
assert CFG.HORIZON>=1 and CFG.FEATURE_PUBLICATION_LAG_YEARS>=0
assert CFG.LABEL_PUBLICATION_LAG_YEARS>=0
ROOT=Path(CFG.WORKDIR)
for sub in ('inputs','derived','figures','models','exports','logs'):(ROOT/sub).mkdir(parents=True,exist_ok=True)
print('ML-05',CFG.VERSION,'|',CFG.MODE,'| input: immutable Stage I | публикация:',CFG.AUTO_PUBLISH_AFTER_FULL_PASS)


In [ ]:
# ML-05 V7.3.0 · self-contained immutable notebook; no Colab get_ipynb calls.
EMBEDDED_SOURCE_B85 = "c-q9hYjYb{mLT|jenndD>I9@D2!Ny>py3{fge0s<f(3xG-4rLC1QH}lco_*uVo~gn?W(e8U0v>~t&Zu~nW~!Z-i_Iajg6pX$)?^B{R_yyFu!Ea<KE1hnKwaFcJI2oL?ChVKF&S&-19#B|G~J>bh};eLc!Sl4~DT1|Klh2cyH*!(+fRkbhkCQ-@jndZ=DuCbQfAqtL0o34}0#|X*pvDe%e3aN8VsEYPzCZ`2MbO{Kw;`$1jgxAAfi9==kvDSI0*uUmia_`Rw@l@zL=!13tVyehd#jKYj%do*V!6gR#0YlU*=wU&+j6vc~^>^hcxAy65zpZp&DP(Ttkg9Q3{MXwn>a2K_r?ipGWD)5VL&e}Qga9KSmG8pb$2Jbn!QzlITB!&La;&B0)&>l#aguCraZc+ptM8^=FD%jYo7r^j!OU!43N+C$ruUmGyz&#(iSIZg8T(5R1{9oMKBsVkX$+QMbPDo=jPO+SO7pB_J<l^w#1PocNp@{+MPgvOJTFR0No%b<pDc)rj;G^c4kHE>{<5rFUbwE@2$9)AnJ{tn(bq!FnN&iFa5k6T06&rf~>6Fs7dy*~Lhf*xLeO4ENTW`N@yAI%=W5&a`{e!w|f2DL?KJjV6FJ1=1!-xxnH!B1}#$h!ABEf;piIDTwUKSyH41Zm@DXMAh24KVSyfPioaaHNKEI?rie-VpTpUD`BU?F)hk!u%+cwAD23D_Zdz+)6?(c+r4IPYKG8xl5cn438_5V5X(=FHhhZV&bFY@8L7-K5yo)30A!CFin5}>=vNK=LRA(%oyN9Fb5ol=HD2f)IX@Kua}qYwQ9qzUayyHw@cSo%AaJ6<3BO>P)~&WIPcF1`-5XaQ^g(n7Jep7K)7KO#uYk8pPhVp@_Tr82-Bf)zo36%1@P&2C;yo+t-3q|Gy6Ss1gkQzH)ShmPs8I9j?p&41B4kN3@nKz4LbyVe}<p1lP8FF_?4h%AYQ@<-@`A@PQD`C{DKgcj)pi*I4VAdj^GPy_Q@9jl|Nt;#^N9FSio8N9OriOH4uRi2A_NbYa=KFC=O3KIe$-k^M-a5G59IM8v3M#Kcd6<h=9f0hH&IV!5~jUw?Fc3z(fhzXn90503(Q+7@st1mD2J|ZDWSM<Hr9Hdi{n83<D7WS};DTP5R?b&o%z%Z~kDwXT#}sKf$v^;C~9h1TgXlxOn&rUHwDC_8rOrXr&Atpf3_BXnyn}uJ5^^CdMZo60-bkubW-4-PWY(AW@zDgf^84nL!x(YXWfyN11T&kpXBfhRzG^fuxd%c2DS#d`BmKb;S@keHg$WY0eR>fai}Lyi7Rs>VAl8qTPAmdUW{V+?;%)9&tiD!mmTZ!KZlOj|epwb0k?@o;2?+UpMMMUoldMT7rJ15!ZZ>KO&s|CXwjEPvvZ0M?wiAc;caNA_GJaA0b8x63i!x&(x72Q1>aHf2KA>5*fwuCP9>aN#p|v8(=&Fl$P~a-9Om=&u|9?nG4@PfKh+uolwq>dO_etoQvHfCtu^v;3jOrKXm(_m5a#dvRB&qEABbCh<A+~%<~@!aIYAn8A%xHc^eQu-@)NR@`!iAYrg@IjW2Pn2=oAVfJZ?1K4u<4Iht@VSF6kA;++fi)#?Z3I~P8|?f9g$(Wu(%8`oDV^;>r7Mx$J_Z)~ir*z2Wwy?DdvdaiQPedr^ki6B44Q6BS^hSxXW&<Z93^pX5ML%cOUS+A^A8@cQ!mf_>_OXCw95J&kJW=14R`A0r>!|||B#H2zu{h_4wzemL8J^vnmGw?DI=lTiKRk$X@ltaNSM$*Sb8(zsf#J{^~7;4JA(4&)YVD|rwFz6fq#te1NSRC53%&Cah9JC}pxewgt1c{j48~}NRU;C48S4F@p+w<Gk=JS`65pXaW4<}<{kGB#T+az>UXKoQ^^qdarYl6ODLuj|d45A_KD1r;NI;7)9=bOlc{ze1v1UTIu4SEJHrrC8o&-IK>Z#Wo@gGZL(c&$!z9C>BvjCZ@8ZP|PszUs*BfAsrF!E3*C`^Cm+;--TJ^c#@(g{48i-PtJwzu-T&%e8u?x>hiNdpC-P$#>68ue&gFkI4AUaO8UK=$_k}X~U+?;5TQ~+?~1SjymlQJRZ4S*YR94+%X<kzgIjXrlJ1#u0aRccxZ@I1)s3hBK<4)m&eS*2ORfv=5)SAI?nXrJ0kNx;AJP<Bf6Kf^Rs1;SYWGbrPc7vYUTB+KeJMQJm@bEe!1J~+#TGTU3PneosrYFy}><q-0$3ttg2q!s4bQ4>$TF_(k;J}p3~_^8dlcUHyU<jZMpoB-)z_Ew+8L@Z1#$MFF%X>=DB0L({H&CGQaQ!(Jsm#t(TV?z_d~?_VtzOb-RA6<agJ)d^x|+Y_?j>{Cqn<@8n!}ej(@1%{Aw*ZeN+Z+?sP1u4L!uBHcDhwVUNeYze?@%=pa4Og@{xoXchNjojtzmHg$ujr397XyEeJYvmi2k0Q7e;aehaeps!2uw1G6@63YWIClGEcx%DFcg3EwvynG$luM0`TA9iGQmFyhYOj=T+8>uowR*wm^vBRoF8t0)>3VtPTyNZ})+&FC=Ro^wX=SDQp<OFCYSlV!O67LhuCA?oTrjo=gD$kkGlxhm7-wHPeH4<dpBf}2zLK}ZlYno<N5p1+PQ2?EbUVFdmikpvk0{XHg9!uJGf^D>`9=kR2CKSRSu3sB<!sKrQLW*YEmxKrbmWz}N0z(3UaP_qH`Yq4*DE(SsvC7gLWE@%At+v?3zk;>7o8NBaKkA#+4n3n%PZ#_cnqzWQA5)DnB^WJZ33azPIKA5UA0TgtCc!ZG8^VxhB=NW!>$XHOr>dtq1*WtO;t#{IL9L-Cx@^PbhpAJnO{`7>8HWA5p9w@9y5+l6J}Ol|96;LZua<}0px#<kR}jumO#2)wwGY&TInW|sdDLq2#(TrAzYdJ{sFHsT%wTNq$H$g1W2|+?nve2iQhvo9P%#v8+K^Sib3cye&Vn58Otxw?(}2UDAybH_&NI%GtplY2GNa;D2Mnp7`h{(JWkix?RewCsMCax+fMWD*!9MdTdK0gWPuJxqrAFat(8_J6_|@RZq!QfZ>{=aKuEG#g6fUI$QZdhJn-tu%tB_)fJ<aF>a<#J-`L)RLwOUiAlyxSh~@I_&;a?YW*PvMQyZ;rtW}mO>!p=Sqf)K|2`K0Ml`TY?mp%fxuT~pnn{Te0mFoqg-5ogCdOn-U>e`o<X%04B<vu*}Sv#AJw7*_zEZvGsINoY`W9ftC>-H*4r1r7BQdzAu{EK2CmtVbZYTGw%;Z%|+yB6v`9J0xcV<3*e45(pgrFxSkj^*lyYb({#a{RR6&$J3)@r;H1>_Xn~cAfmfWkYgMmeC&osuG%bcRNG<F!iO<nhhsJUSp9H;dHxze$BzC<+hAHV9|`Ov+Z_)yS(2S?*jJ>3}wINct+0|!?;ej^GheJtYsnsbR%akSLz?Y{-U0&97ctOC?=FY0;t+@l0f!uR94D1;>XQebz?o)m4&PZht^AL%cXkwJS}K21KPobjyTP@LHHMXDd`=!@NQSiABJ}_-WZszY7ID=Qtjs(z#T2!+F1J_a?EuvEhUUF7jKL>b)!};+rW^9VVaM^^m?`00CHYhhdl@&ICt$@q{o$odB(oAs*UX%xDo49E_Qx{7=n7a92}Vi&DJbcH`W>ozX0T9rBPX5sg!HM)N?V)AUO}9Jpf4?#efyNRBKdjl$IidT>oNy9XK|a;Pq0yyi!>!Pkjf_r@BgO3Fr!OuJaK}0W?*t*uQ;qIV(SB&0VqPbJoI~b@_^Q<(hSM&boHR%4Kuu2p6gXO?E+lQzNIp<EG#hE4PqKN8X#Jp@`pVe}Q|RmA21y>cObKDXQ_4Jjyd)?}VD7$f*!>fzu89S+{X|waV>M0}p!R*2Z-Wzwj?17C_a>28x>J$Vi|v7ar7DA~Y@;LTUZX5R#b?kPTs93Y!pUqXZoV^jlpAjKLEC2Qvz*uqX-a5SuUr8-VI}(#0cB{m0`!GjH<(B||ygu-qurijle8L@2G40S}QbG~iaa?o)%ySwr(5Oy$pPRTHXNp{Zde=q2m?*>=Hd83`%W42*Td^O*o-0iv{Axe0gD8IBObZl#Q(BYxUwkO&0myHs5RDzP-J3yB`MU_6DjjWzp2zzZ7)BnVTLhJB+{fr9{jZbT%3=uJ|EnJkw9w_qNCZN#3!$7LW>am){)7>EI6Jb6Sa*N6=Z!6poXC{P^vQqebpV?R<@S>i-rN}kjp+tqLAocw^QzUbMgRe<Eh?v}hx@3M5`X3<xwq?A*STR9pV@OuV{A9P}*OyZDC%M{8E_(UoZ{cA!NeZQD9;L4^)Q-$F7i&<?yQ>5U~D>b-oil|>o;afsNSEkkm&eof3^GZ5>RKvjnTEIozb6fBU05BQ3_^dZ*0m}g&+y|(%!cX17j%TI|DI8=LdP!$`cUzrNYUqqW?Dt4pWw{T4QMU(oqq?i%2(}<)lJ(iNaCSmpWtk88JAxR#7&I-|v0iU7cD8{!OFPFyHf;&2$~xYYZwmbOH0qo_gEIYGI?n7|t9AA2m1{pv#~HH3|Jx6S=XTpOs6=r(ec=280Q3iA7f#vTMdU8o3buplrr}<H+W?8E>nYe@UcFvkUPggcD7daJ0OsGh02prHzA~4!qYc(qN=xNi)fG74_O-TiC4Wab8!fl(4{xI~({_8?F7UT4n-`QyN7Rr)$+-=*@bbK6{Ic8G2CjG;Fb~+(a8sc$Dq@>V3pm~$YAC%Fznj^<x_}Q`Zc4MJjN7N5Ovh_>I%YbZ&WOh5WZa&)YDRAF{&rDLA~SNGmX{KvTaIS~`|iPh7w01{0q(oavD+$|KLu1HB9GG?Pn>RY*XAY7FPF?%Kd_?auIm*yn+6{5CLZqoc7|L~j85BVW_(017EJ_^NyMm`;T_C)V+Rg6$`&c}EwIu|Z$&VmZFJp!O3W+$4iG<>mxLw0hf@?kZ)Uf|JR*36|Baju+^E8iTON%Dqm+p=LwbG><eruxvC;7Q&d}S1uGx15s8|#OVgnXqPXIH9T?dd_PS)a0vT`wO5~xJ805L4p-%iJ%fGc(Sd#N9zibqh%ojUXWrhg(>igOW#<j4j1;w0YINSVvGa#val&9fURFJ1)B5SinbtZfQ-fs@~oVfqytjHT6=bV@$&!RDi*usQFO+w=A(v!C2vus_ilV11?MW3t9EZ+=AjFV++)tjHO>lIVTnSI}blXsTa^MNSmNcK(W2%LeQC-N!6EE9$<Hmj37B@Mhj_Z`$qoYt42$doBZV9v1Y^w3a6TS-PUww5s?x1>iD-bHeAy*C~A1`i8Ga7xsHJ^}P!8&pK;p$m;{9@p~AI!StFn@HpNV!U+G0X7oaEp}*5AMg`*;ar_}3(>VU`8aEm?e+gG7RqT@k{KOVkx81(BedVh2W4I4-OAkED@b=&zBnp;+rYj4Mw`J@)-Y(tT@*eLDETo{jma)^c;N~9g!Am2cm8gxi3?Q3hB#F@7F3i4bA)7fI4Vo@mMR_#5<?*+;LDeNcoxqyv@^tXuUk>_i<b~#-+jYr3(YL@ZA*kGuWdOh8OuA!@5q2`_;sk(M_57heoxmNY6A6~f>*clO^4e0RTrcjMLl5U|7UT!Z>`!{bJ$S%>Et55J*n&U6lRXEB?0tMBJ^`aWY2IyZ!z1y*Qs}Db-35w2>cg+V(cJBfXZWjyUui7<Yne@F4EL@#hDZKK;LbZcyW^eh9z69wT6pt}J;BQtLf=AT_ucXR!RYP-T#@`}nGd?&gZrZn02ZEq1YbVnFR*-Z|6zOi(K1KwrxVv3<4FF;&@kgqTR1=ZW0}7gY{QPgckuyXHr_?Vce+jwxDA3GKAG{KfMDW`huy)L-&DRF1OxRtp4aK`6gP)=cj(T9B~QApg<s+FR5v#R3+j2PG+v}(EX}k#ePD8d4@mLr(8e>?03EktQcXbn3dTCk*tFz{$cROWyEK)yOyH^_?p=p^V)<0+IH(;t{s9GB3;~t?Gqrw-pMz{<;QwcSG3Wq8#$chfiqD3Nc0Uto*U9Im)$UHb-KgXiw2p8Ua`FG_6IWc&w8bp4)s&4*PlYR^u>M5-#mF3mMyB8EsdV5Z^YrBRaVWv<n6MuFCm>E4-~gL<5#_q6O(2CCroYJHTILL*>98||e>;6RJY78cGc&D@hriFjFEe6WXQ&&~x+p-kmDa=Rzi6*S<$iii_Y<)oN&CUHiztubct(F{NrM3V|F8u_kf;khWx_*`9setmKscOZdi-guUwLxjHlSk(!keH7G}wbcn4=3p96!Kv{6u|rqofzld&ZfJ2a@jMgGd`e{W1Vkw+koTca{;~6PlyUI0y(ja2}IhzcF(99?sDnCA<dBCu9GuI{fXr6?&HR1~BXbWANCB7pr~u)?i#FNfa)h`WUxi)T#qEC3b|1aYi;#VcAcu)7pXiZhZjAGw9dc9TbBH&||slc;ilU%^fH72ABV+)AaAbp6m2&XWIiXj9uGBF5lvZ-lr311X4Rai(aVW7V)9zjIZN&e<G3g@_v!+S8iTkO=?S*R)#Jf*&Jx91NhUAH4Sbjej(F?tGt7zUVnY~0QL%>dAf@`rnU_f>Fj`7kfayIfbYqJA+Xt#81RT3e;)~Ab~MW>X3}{}w?Yh?K-aSu!dW8l?evBn5a)nzKRkX0kH1zFGicsiT`!XMOfUfUxHmKx`G@l;(@-meJphZ??*l6oQ^LUWA`g*~O^`)IX0B)!dTwtp+ABoGI%p}VpPOpmxlk`R3~p(6JH5`hXzrt;O=H<TFz;NL)|ZK0jdw}yQrt&EsB|PuzUhN$)8W~`RXwx;NU$83RS`zL1k!$r;g|M>chrzvu;5>u)+PF;7hs$E02hR=Som;&vnBC83gR7*yO|M9MS33tY~%8`z+mPe<3A&DDe{k*btW{aftT5F$HPu56}3f?{^*|5>Eb;kU%S^Cp|Zl}xu?<@zeW7zcZD8N_@$ChWXM-m3bH4^HJUr4!DLu45{6-~Ym~;G$bbnve&4g_3AtJxza;VWw+8yJ^HrCL&q)tALWLbxYMZlgL(jIoS)P)a);9-^CDq)SHF<`ap7X#=XXq?>sdT~+G>kJW#$QCVwa4xQ*yjw|{)egblrAPCJikRm8?`I(T;JW4{!qW;4Bsi5IF~}O<-iQ5{BIWgn95o>CNxgZh{2Njh5IxFpjk0TI^}k!Z*T7beLXwKB5O_T?tCjxfTZkw;=!4G``#I@@8vIEyqrA+tk7;6MBN8XcQ`gu)jBEFtlLg^!uqvzVd!|Ch7$DK0}v{Ss+a}(1G1Cn**hrH0AlxBDI^COL4>e9EDBo8!hGklRz7Vdt;`%bJ=<S_+N1yQ44nXji`srS)By|j8TyT&pF-k`*I7E9!Nz{a2Riab{BNzaYFfCSWY%2ljtLMTj>2~D;lV+{ZKpffcDjg=;U7~b-$uYNoBIUMgF?h56GVo69C!21g&2hHT-Z7=Zg#Gtf3aoM>h+*0Z;EVST28+YuxPqdCQh$vwqff%GIB*-NA!EZ%Gy=hf7aEuO>~rtsmE|(&{aX2=f9OU=`$>sw4g%pctU)Q0A7aNKz)@a28QYU0z<)3K|g*d7P}X6=)_|Kt%n&iW?-SiQ=z+&7ODYA_!I@7>J2pIH!7>;B5H~<y+MC49`rj+l|ZA_yIy9e&B}67=(RHJw*^q$r&sYGyAPOk@Ni$bbq`EwW&#x7bsw}kJHU9R(wl|c<*m>}8f7@qW4F26M;Ar2U}AV8jLqsBz|l^-(+oWxPC&X6j@9aObTmXkvtXjD!0PoC5$lM*DhF$&db7B>wMd>(_2T{kf3neKi@r86D$@RW2^&@F^$mCmpM)W)w6R=i(D(9cX{}LNq91?0QL5p$wK9AQMhg6fs1<%CpMv09quMB~6tjz~W%LZ7=j<$0D=$@R%gO_PdN(RJH)>)|>noLpKc!$?cw>WFw@vcF-0vKi1tOc{i6=ltC{SJ|-IL=HB!U0`o@0P~$5-@Iuy4X1q2|ggWDiIw)IpLWJbtXJD$xqFf-qnYE|Y>_8l`$N(;wVVNeyLYGH#|b9dE#raVj0RpfVR}^?+*OUtUJhtgPLr#_Tn)xfy3Tbb;#bo6t4g5;zGfa%)In@L$Umz2k#XN_+uqy`0T%9mMTZ=c2JZVz}SjLUt>li7B4NJ91GGAhdx7iC*sS?nRfZfCzrIO!Q5O*`4AhEQR3j!+Rki0tdzOZnryKH-$ks(Lr~k^dK@I+C<R{xleBcl5Lx5RB^O5R0>gw94k;on(8{e?Uqw$6I+s+znIJB=PzD_J9pcJ-QLYihGaF|)(a-PDzS;8(x_(>N%Eqr5iffg$3wdUnjPWR-7$$Dk~|T8mXgxQymKd${VO;YrZsYR+y^>Avp?x=!-_or1Poi5@xX@1?g(hDXOT+5L(`)>YP!)K@SL`5qZjKrdhl0J>5nhZn=ugdg_bP=5D$tfl(ay40LORScR+giYy1pDnfUSV9uR!cCQHWQl$pH*(*XqPLSyocQkr!K_W>|??tyLGH2V%}>tS}NruGAF9}9yeT$}KC(!UFIqi=3S*3@+R1G+kF#K_dZ2UP??(L$4HO?pF+4lNHZP5Z99C$$BAZ!&Uia){uNNlTbRiZI)>?=d0#Bhtx8W)qX=HFw>flWMi2@Re}~7(z^@*vD$jgC>Onwt0gwb_vnu@Eq;BU#Sr_9F=hx3bR4IixNQ=b^2;|+-}PYT_W;ilGVJ`Mv;Be>wB8>+bXu&o6W6E3o%d2*#Xu%!%hZ&_h8R$bbPa^d8(Bz7~|s5$<Q;MfJj1Dv<Kjc;na<yopccE&)f+)@J73!he@4yu4u&sLf42vDK8=H5FZf7-%MLEK}eo!=#r6!-%lsqsbp2MTW!0!3n#(@9J7eR(pP&lxbGFw(h^f^1wWIp>wiTD6J)OweZQmXco>XT3Un<j;=dVIy`4u7N2UZ?a4~vhA{Pl6&WgiNGwZB;XG<EvlV)IvZupNSU7^q^$^4}^Zkg7^1AukN?CFg5bTiTGPa>>N)<WMLn4kjdBmo~_Fosi+>2wFp%}$|nDR^ZogC^i4O$d%j7=YvQlBVepiqtmKcY5wri5<|($9jRf{3bq~rlpXJf%9Dye=|L2C~j%vUICE}N4O_#e27r=1y0*qiO$UQ?(si`INv*k8b@e%`t;6GwDiad1-usYpF(7?x`3D#Ij>uu3&)ZTdWcr{rl~)ekT@|)6oRJ@vCwTTo}S~*K6P*+R?*#jyV0qu^NfuI+jaW-p#`eQm5H`VYvV(nVcJF(L?oOI{E#8nNw(6_8Y1Uix`S~Du|OB(oESS221E-Ic9y6G?U18i=VuC~1Wq(vEM{XG6pRx>{fVGY#y*imJdy{dg=#VN^h+9&_KArv1Tc>jy_fqS(e4!lfSTQb=cbZw5PfVsUK*b|0;)JG3$6@0K6qL!=zehDc6-C|o|r))ChfG^@^BYgZA2@~cXeX~eWmoSd&+Uc`$>5rT_9n`R0K<$(<QTDkUBjge#+O(_&B%(+9dK7>~>QRm{B1TN}sWXW;_slXmbks6h)tN-i{ZPr-LFRo+Dd^ffdF+b{R9bw19}9KEh?cBf4OL_WO_F@}F!2jVaQEGTx+@N~irL5ZPG^`RYiphx?tnk|)x8Dn4D1?;<@NXut^*CL~ZnDiNbVB4)zKZ-nO`g*(f5gF|50KP=VOq610YN+6>XUYZ3$OcPItSwQ&t$HWwTr}|j(qUmHstgnytSB|hS3lT^JdH@sM-A(C%3q!x36)qm-)%C{5=BXo1`W-m_Vi;IHdydc+143?Yojz9#(<K#zDY-EA!(Z8`%lNv`1d(k^iU|~R7usEo2IDN9=y;-GQNw`>%d7~7m_=7E7~e)=(40;20hi7brj8~nD!5H-zEx}w6I#gzfCjs$AqW<r#vrB>xvV7eEfIQOK&D_FC=ydVVVy`+MS9!Mj-M}*S^Lk&|K<2!!~QmKLEY=P_or*XNwSD6PfO+1$6*>zMk7*+3a=}R6?+!CrF2H<iXYRK2}R*1zpy2BD^kOTdcqs!QoVwQHk>%Y6ulSuPh>f)HVqCr_iq-&#KM}$#1VYWq*#b4SAtLY(?Oz~)ap1p{Q<gw#H?^Vw^#Jl2|s}W8v7Hrrv!8IcrbRl!pM`;EAq?q2X;?m`VwX@c84<yQ~cwa!--iKcp3Q11P&jf9fPb<Fdr=u`f*mPoSw3oJ1_^|d}4C6o9y)dZU<FF{d*nQrx~w@VJP}u#Nq&1_74s;ichn=cH6WR`GIU?aS<Y|cPnFCz6%o>4f;JG2tc~`?-uS67jf6RM`}svQT2Ab`%W>K)`6ZI!*`U(qP5#a6(JM+Z~hWaw<f&N3x=Qn<CV-A_azaX7t)JcxbG>cLBl!Mj$&B^)f_97yW+L*5f}dPnb5@kAMT8k6h2ioP!Zq4#|C=F+oV_qyvLk%vy@9=R>ri>@*TsDK%Dn2H1zS26&f1775|>I++l`BWXg_2Zj{fCzZE$rzm`tv22J#th%)toz<KL}jGF2y43RA<oOR^*X(s01$9^~<ZCS?dxEFQCuCV(mo)<Wvd%jOLf3=KTjnx&!G*nrqz;2t84b+Pu{2`ct=*cVs3wikBTbVFylCB4gn&dc0{J2!B*6Sk33BIzaXt`Xk+(i1hu~unt?vSbM3ygBRyk?8b0NyKq)Ic}iW#thX>MC_HliQX0MrnmxqSgpL{JdnB)|S;cOL*wVF^f2Bl^et?2{+++-Cn6KeNgmao2jh~ns-y$mi0z;U1?dS3tZQ-xUc$7qe=)J-{9WZ+_DUFxQpRI1rsw2ppUPgECrsyNv+hXYpdlo`l;oP9dci%>^@HrQ3w}>v+iq(`2K<tVhYFEkY73;1Dk@xaPX+9RULXttI$$}GYqL+ER*gP`2`&wZTPi?PUqQ-c+ef}&^N>XO0FKTB<1Q;2y6TX4B&qYws@sn6eyiSj82*FZNDE4d=qx|M<<`ZH@p45nYP3mcBd6(eqy)ZDPxBPrHJm`XYKHK#;$!bNrll)bJu%CApfWvO5IDxG)MScD2hTigx{;YCJv(Poip@+1H&sPy1z4lF+O@lA)eYErj>4a`ggs}2$Ds;rxgS{rJpz|M*Hd)ku`><!GYRJFB*h|O!7Ty5ut6`RSlctVlJD#*q>EjkLaw09}|u!@P2ENn&TtJ{fNOrD7%X4wuyHw;0(qVYq`>_-3i)IXYm&F`29$74-oXmG=J3Xz#4?J6Rt)K#cN_2U(6^;%%{#MNlg(gvZR|A9#bfQ%o_BF`@;uty7uJ+3YlE{;AduZlw?Y;8GzE#J|f%GYw1UCM(|z>KoR~BxD1Hc15Gr65xwD~T7h??8K_yLB3*)-#gcw{Epit;mw6Gy15P^qm{TS`A`99#M!Pef1p!QG|7YvBN`B(Rh=q}O0D^Nyi-q>KGr}!;uhqF{c;mgUTfB4Oey26wEhrn3hYeZWb<xYakjo7pEN){iqS4F<Zmfw{xQb7l=G`6QunX^I+t;?Q&fU52{(G~~?fYhQ>!SFh8+|T&{t*|Qey2x=y?}`dhZF2PpPBaz_OVCywF8TCgQ-PzUUV@Qk<eybujvd~I)HO2uFS0jLs$FG1^mCzLfV7=c&6?2I^Dg3x90%~otbp35+-P}Jg4u?co^He2<L2u*L>}2)?aA>7HxO~48wi5*=jAc+7S@su<g)kA<t0A<>3o1az?@|Grs_<j6(iBn%n#T@vghqMg^p21c2J#&;D$Ge=I{K3iJtNGWVm@OfHxGS^D4rh(S`5`8jHFaPZzNj}aSM1&z2l3+=A^VDT3d5FgrmGa~-J;0>LoJG1SM@4Ie)amN|rcH@@CujFZ^3h<JC5BloAzb`JT+1bn4LN?zzD99&26Axh0*pQK8>dZk9*y;2O*I;+D#+;m&DoVtHf0<n1bcoLNiT<hg1JghM24MnthPkLGz@2xmxoyockqChpg05sh>FyUeBN@(JbMBRigbsc_a_I6`FVAf+_-E<tz{3L_g5JOQLfes&;HY5iYYrl_xo{2-e29iP3RG_;Plbm2b66<U4ij+Cawt?d6A?K0ska8{8-r$MJQx<PXm@H~q1b1c0Z|{X^GlaEPU(LEzBUKl!Km==<%KJ`D@YFagBzQ$F)BD4`!SXa0i7|9|4F@YIA;`IzW*!cxn}>2N0d-Mn%7ERZ;M9UGlrUxD9sn}%OJj8Tyv2dGNS{Cct(UdiMjT9>yg`rzq=`*YrfGc=8<-PieKs3d2wM)qLGAM;{J^P)@DfJ*?KRRSrD#2@t%@iDngGj)p%6sK;swAgA8E$;EU)hQj<y|QdBOP;tQXo1x(lr#lMiPdE#28&ysznFU&Tx#V@3X{&(8CLAPc52cf8dUm+LT4X4GROcjwL5_}Z=EU1(R>oJ|f|IE=hoYO%RfWhn$UsDE&FzJ*@3;M2y8CG`PF3c~M87N{Lj!Z2>re_iB^gf9|%7Rs4sW8<^gF1Le1(qP;r}$^YUno*ni|Q{CF)@z#A^78A6VnOpWWr4J`^&R9`x|GN9pm>Y2Qt#XWJu9@B*kCSn}z&bR*!(<*qm<+Up_6W0@kUo2^qE+$apvd`lQOANy4M=mZ94!v^u^GxPql%`$f%c^%Fs+y4W^1_d{;p<TcyIuc<p1es-(yv(>`S>US=r54Mc`pzAvq2yrpKVm9>HUk(0L2*1LM!yioTMoKzIkWA^W4%Cfpy<dcj1A*mJp43^jnd#G^8H|HCy{P$x)p1+JaHw|3|B6f%`1|zOpXrf+v_~);Iq<Q;meSI}pUyqkzoRG#f7eXzgXq8PC_^sZrej*<B8#abq?2rtw|7V(!7e|lZ?5p#KU=JTb-MGH)b_@Sq@^l;^%U$Z{Yw?GhT>SN>UqkwaF$_w&9CmL`kLVQX+6-{!Y-M2?j$;6MZ@yeoL+KNUIJSv&t9T=RxuN4V}$*iNaQ?=as%uOBUg)F<A2if#{NBTUgB}yk`MKXgMJGO(@w4_g<gYA1u>gl!L&Gj<J9Aw#NYbq_9AfFoPu>z_^)@0vcnjlDVpw(DwKW2DW={~<o`D$#6J(YalIfGY+v?uAglDjyI%?)3K4^%!$tkHrBi`!H|Z@n5Tm`~zB!tp07j0$b_dR1IGoLz795m(9Dn?QHaVY^4b6L<F^3!ZkwX|hmoi1D*-~|NwSu-abdp%FtgV%o;mXlOh-yeWt4r*ASLPDwIivm=h`8NNBIEE&oPhHNoSEuqW2M@`kr4<&iC#$3{)&BXB1b6My(U@&$X*vCUUx<}oTk%?@<bF=LXMW0z)^R{hb@vB4>m2mm_P|5vsuOD`@WenK9^Mb4Aw0d*yL^3i6^h193J1wTsWd)Q^Z0~XOr?4h#DhQu;epwh$-*D3sQ$4MM@6AgH1}#P3ifzBI42g-418Aj6&zW(;27oN>JQX_}I6~rCQ^9xzrHpEfjligFdA|NPuHf)WE}PIWEYp9C?6lR7EQ3<F_=Y4aCVjQ=!DWVNbD;ZCJ@%Ma~5Y8T;Nu%*K9#qSMh8HuJv%YJVqdD57rXE8zrk7!;LJ5@Cs|ouZ_zUu#a(iiM9<>WqSsU-5iIQO83{X7SRX2xUy{@-3DP4EaDH3d)q7>`aU)R4|X-n7xWgO>XtjQJZp%09C!~;vdb`9`O<8S9NZ}GFf-vpeKcyMAu+Ax#J=pNq;Ko8T?9v_@1M2e@J*OanNK}VNF#>zdjD7UfjoU1-F&L{?j_gTf9bSZftS{n9*OJuHr*XHfi`Zoz7rp)bzB;D&D}aM7GFBe)cet{n2+5PRLOjPK&PbQ&JF5_(0M_?miT(Lm35RS~9G^cL+oY!1g_$_X~YG`dBV{JjC0sdesH)roz$AW-obe>ARc8x**7gz1V?UprSPt_-Gd3w>ot0dZ|q_o6FLFaN@Ih@o~$dn6aWCCK|c#O-!;xk(;5u^4a{n1^+Dg?~8zs=<7Y#!OJuBG6<<wKI6a2X4vS9Ah~n~t?&YZ@;_#1@WJ*kfGbO@{%cXsUKL=LvH@zand;6co3xd1ts59Yth^h~_U^xxk@+|?{Yj5|FXrhm1Wh>V!{&5si0;saPKG*H%_$c@;{}3SPbp6XhMZ}3_`+rgOSUrO0gaq478B59dS?NWgt`i_Wcur3ufx+&H`~Nc)Jb5_i*`%fmeRBK$Ymz|Zl`}&CPLI^sn_QfG0|qlnRwrbTTs%8XKJ>KBS|tS%PobyVzYule);k1@+Zv;JKwz4%+2Swf7-0{375mOpycn7QU#3a`-)P(L@(nt$>BN?8U0nl=(I+*o6Xw*JWTCnn)UKZd8uJ6mFkUDr)8At#_dvV=~k(hHU^^|r!R^gfmCD}joQaHHKb}u(6qAFDBmn&(`}5a;+FVhT&Cdx`YpVm1|u6!V`#AOPgbNk+R+l4<4+00Ie?;}xBME5HvYteVCvD1(;e^8Newj*{}dm@het4*dzh98J6W!7Twf_m$ha;3B~l`%R$Vnh)a7r|#)r4cwX#8J0*1Coou1+l_@!!TrCeVsr%nN;WokX7O;H)->G8{lLDgLGkvv>Ke@F-@9EKlq_D_-92N26lo2nzAU0SZ$Wk9C-#_DPbV1Y<kGy<fG?CEKHW>5j8)Wvk6a2<;<HgHWN=RU{6@sG>sSGop$N02l0j{2LII8M-21bIV!-MN5|Jj`^LIA7yNWu;NB87bbo@Rl2uy0NygvXaK$L=JcvXUQvj+m7EevPQKQefmz(xUo{L)>7f;>CpH@+oq51gU83Z$mucv#^g@%+j;tl94o({@N-Q@FYS4uqG`tUkByv_2XHI{xZsOK8)F%U3}LMXe<`s}411&GFGk~j3iR&7HHxm^lw|`;4g3WA!QX$19qHTh7hn%H{avqCSJ3zonTkYt#7<y(drYq)Z4`QyhKmT=CTTHsl<<@eE%QI6g5t&xx#kglBu)22r`t{cm4d!D`*coOVdR?7t}yv3dCL_}>F_)&47y2L=4xdvN-yxmUPrJ&78bbEws0PBi!+{9CS;MTtFmUKW-s0@tyGq|;_Jm(jj~of=+>MypKz*4{UzBPjO#<o!PxDQ!jRZ7OkeN8i6h*JDje9c<@%)4ZAk`esk;6#Ye12^^^o_Pp_KNmMPt0{8uGRqqY99c8I0UECGVXXlczu?>Meu+a1WxqPm$86QkqqMFg4Apwj-hG3OZ63dG_9X<3N~I^gA#kJIaxF!4vaRXknLtXNapY{6Qk4oUxt?(rws;f+6a)(9CuRT@dmy<v1*ROe{R*!UCZ`!MW6=jZFn6hqTeC8v7x59vG<`)!J&QVTdqW%fJe8HMR{He_L-Xr%k=)qk##P%O8~hH9+2JvHPTf>jYwIY`g8j$ThGo2uSCsvNZCU{K6_0Qa14B$|Q?Ruy(2>zEZh)3$=^zN`WK-AMrTcbS}X+nuER+eJy-V2_G;aGr+!Pt@%X7BW+u|^8gC~#WgJWqxWgo_NQZ0u`!*NMd`-|vFR**v2>NxS!>ek?S&OKn-P_Lr#S{ur)YxMoii=G%8MZV<|Frc(LDYiLExOiBL~p?f`a6PcGQe807aARRxdytg#!Zg-wIQ6d=ilxin%LMwq2cUKA7y>)X6G5e6ZoC&5tYP;0+R=srp-N&ex{Rc`lgqe0<LT$|c!Bx_Cjps-OF=y{}}|p`Qe-c?EUVKow2=l8-}}4#ijcC}?UaxkIL#r*!AkAi70mNvXDt;tP6wL^q9T<<x_x8;9<qaAI<@z2PjITE8p#lb`83>4prgG<rJIXNZtbUh`+7rV}ZFsUkdD8J>tjOI(xk@c5+(xBDsU_g5t6=D~(xwx;+pH6p6DiML~HZCmL~tDVv(n<eY5X2A42#XNcgnIt2c(*Ho$bV^FJQ?I$E0|<?_6vNp8VDGTYmgn{yXx{X&ZU`}t=8xH*n1u+)+ba=hBeYRu8IpV^Ruc(bBPKFsXV^HP=Eb<^6)+ePPS8Yd<(#BAmWe)wF~kUa8iaw_{S*ko{mY_!oyf@&KXb;f>{ccw1>%cugDI-aIzw8z^I^;xu_a(Mq{?-E3bP>d5@{%{fXC??<Jbnmj-2f+Qe^v7UB{{^8HUy6QcctHaxrY%5?(s~u~Aq%teb#gJ08(-%Mdh&j1q+B<b034M!CO`v02YeqNbSi#8G>i8VDfGb$Fl3jL>*<!zl^XP4<E!&zwadq0BR<65s^bN34pET6wR`*`kJ`O<mgnf_{qbNJ<DR%s;KpNPME7EI_|**ZyT^7?r;;(Uw<KnzEY<JvL8GAVtkY<Sm>#{FNS^t574CX)>rfwk!|j-s`m7eluo39t{R#&T#0jGty!*q#Z79rl?r}84%%};ZP>0o<*;iSdC8%8y>mM!H81)B`t-xDWqB46jP-3+*m}e^hd{8V1qQ7#NH{A2rhtypF@R8vG8j7b_L6Ll{2ti$aas-^f_n~!*tTeG;3ijq|pk7d5LBd?@>TbI&q~C*0Y@ZNd?>hWVi7QEZ=o{v3x<%K4|p;@`ZbXC$SDl<S%XxX)A`7jr$-v)z@Q1_kkgSmS5qGgHF;w)Vm_bv3I`Gp>R96XxG=kFX>QVk~8&SSh#Zb8W>VCKs2?=@5uj=_qC0oP48!;Ge6g4<s6X^U|wCn2pU$WrL18%wwEB43)t$3;EhSQYh0b-)l>f+&w#6Y)P%{jm@kG)bPv@|T^`0_Xi(LoPSao`Jgf|*2HbQ_<_?%7zDg`L(l_})v3-*}ocdd);o{rC=q<~_IVrp(I}gyUXu($LWFX24pE>F%dOgxZCN2Cqy(m7+nrw16Wo7WR=2ezb<O_cN<XU;9T&kBjvpl)`nTtVtnN$(Wdzkw%2NbaiDpVfty0lhW`52Ra*r@eFVj;Q~Z-~yRkVtCVT>oGOvyCK-(bVS`=BcU~<*#Uc?yy<E*w&M;eeR`@oonf$0isZz<=)bv`L|6nq|8v88PTI!#~@)+xUwMC718Me_kW-R1R2$f372fJZ&qZ=*1otf3;v)7mbw>_v4qP2ak2ie>0<FJ(FY_YL%&LmN(hz4+71(l&{=xW$0qy;NNh@Qo0-fkN2QtvTT{Uj(76(=2grq>bmBTg>S#Dkbb*hn8HH=SN+KgZ@o;R^un{a~zII0blSwV{l6nV$tAf+B=)$*R(wtH!`1mC{#Y)FSjCrGayFx5K6ds-^SEb*!67`&OW&W~rHFvIR$8#B(0G(?+vp)nodxZ<nSV6R_NY*5hqCZ0h_}8d*z~$9%m1gn_mjhrP{|o6q(KDSCW}hqRM8lr2(6uP#Rd<_kRT-Z&YL(LROl@O^9)Ge(E*Mx7TMEbq`MoP8lphNP$f4jcG0Zo@V3Ql|m%IU?+Q~ElV$h!x!w(!jC-4%T3$-oG;!BDfIO3dFsIy~;(2l<1(%e5p6LbB<m6PWevd-n^)%=g~kcmp7GUHWT{`p+)Av@PHeuOf2a4B+`U_`(VLqLy&+6j{}CHhkDz-2$#29f<PkjLn4laLG?bA0U#20LAHF9WO*yu}2_2!cXG*~oZb^joml6Gk%glx%>=Ou(}<a%;zblF6tTQwXtuaxj{QM<zcjq%%KIqpyPUvC1jLlxtz1;6o$D$Hhx$fY%|gzywc|;s_R41H~8O=!vYYan|+yMdX*|Br(#%^=Y_b6%h&bWEYI-Ko@*!XhvG^f=2a&@>(&BJ;Be{feR~|yW{cDE6mPfr2%e<rB-M0I`U@COPH>-R$i|v8VZaAkhNnobq-lk`^5&7>(m{2GO&E#EKSC{gON}SVKy_@8Mz}6y?B<ES)xk}eo^J2d;OM})g|H-Pfbeu(F~uynbNQ`bDOgsLZ4_|naSm5@>k8|9Gn0GE)NXkVt3H8CZn!((Gn}+V04!8MA~lLX|x!n=Cao1Y&LCOym<G%GunwsD;PVe2}4CD#2F+1J`VWF`E=nt1rsrFl)ES-(5p5lgL}+`MiQsedPRgcy`lR8Ps+aN=~C`scK-AdXoQMI+HHXjD&Qp8#9s+H{^NsA{0$(NxXMw6v#_!s0*1ZL=dW1{*)04o5C5Bk|IJT}Sf@=0^|I(^C|{)5O|R>^!&Ls_MFEHOB8Cif`V;p&v&!bId6>=o^x24<wF&b{eviq_x;r9~R&s-FQ*7(4Mq`~~kf6^4p;5;`k3%a&Cw*c>89mtWl@K$qXl&G06wYEwDNZDlqs-v$zuU4S1)&nw{F>025~4F#NnX2@SLCzB&_zR8efWjP97^;`7AYq_i4^>wg>U9~c%v~&%#RV-rvfb_v9(WBYy2T8%V7NM7mITuzGAK1_^?|00NIp3Q~1rdL7jF11NbUA6s5NMn}9}0LrXv%(6*8k!L2<x5keQ9n!j@a<PjhJL}=mkX>t+xbsrD$fjO0e<2@0g;6Pep1+<D<T8}N`%$3|_(p97qDjsNXI6O7HP;JTpz72%F8LLn>Ec%l-Z<c{><Mn*WoQ2yDF5~trzKRzz`G$x-xNw9yYbI6c*)%z9!ra<4y<JE3u2+;!ZK|k4VqpE2OcwyxG6;f|+oeX?zFBG9+K5)LRFp`#Hr@2~dcnwxLc2cd+yh3zJS%JUVXXoa^Ci`SmZwGpdP2n!n8^fU_cc`~78blfsCyYx0*!EjUQs_;|5$q_F^KyI+Lfk6FXF9<x692#kDXSp)7L!jIUr^`_){`WqW3@?EKw#p1db!Fy#)YS;uLKjSMI=OFJ#D`4<=uxJWY7fnen9X<jc;HuG!VXkVz8h&Jr2x-^l1NQU{=?!tcVSVhT|d_9VQ}WVkjZxJ}|>5~aKU0|!DXDV!p1l}pQPhx)ZFxyr&c$Rn7i3jIbNPk{~otU@OgE?o@B0-izf+x1#$ZRwWsPNP(VLxnM*%XX;|>~G|C(Jc1*y9TNmNW`ZKuD^;bF|w}MQlaT9x<9_+tecv!4kmF<%LaO7Z+8dVTu2cPPcgJ$3l-P4l*je?xROYr$Rij~K^nrRf!5F&IXw?!>D@Nw+;3~@NDTRKp;Ez2zf&|(b8d>e(?@|nM^XX+(jw3>Q}+vySR#>E;hx1LwS-LkR@`<x_wsyZ`|^CtC1ve!(|>PES8735)?M78^5n`h`+m+?#ge~Dx%XEI8K(@M$D%=(3~n0q@*Tz8a>VTsSz=|$c!?AwuVQOW#wmO}<szdoVEcyRI?~$*sc!<N4DhF|;@QSFD<P2vg=EprVRt(2C2=?Kh2#wpcWX-gZVf33QV6X%0g`B-yK)nW&;KA}FDo{%C)6*qT?7H@lSftiB$tS~MRl2M<|SJJJB(3z@nZOpBGsV5?aJD6`J>3`jF3#hD3m{1FE2I9%lwLceWiNc#!R)8qu3uv?C?9qDX$cCY@!c~fAaNaN6I;*rA8FPK0zuN<O=kZls{69Dif-SLX%0+=$xJ5vuMI!nsJd5Mvz%BrNA`|8*0gTi)0ybR%%c1a#wy9)ffn?uVnJ(bU6Hpif`jwUkVgIlA7fZ7$gwX*`xlxQ7_Win<ney&8<a)p4-<y#!A`yj`#)hj%*bi)=Qu;%WofqR*4oG<!xb6=9cN`*;Et{r0T_Fc28+j`~g)jo$$E&IDDa&<SMCHOQh58pydNWj57)SZc-fDJH^ZMIu&VnuK#-omIU!9gdN}GUCw&D2-hL=UaB`H+^ld#gg_d<{8vkBl^a;-pr~?=N``i^VIuiT&*^vCC>fetGGlw;Z0uHupkPV{VvBwy@aT}z42@EE(^Hi_(MfOK-BJ%kn4*NL5&Gwz!hTdtQ4biBAD{*ySVXGh2rng@)kmq3r7QAGRV>-PLRQ=4I!<sKFXErGxohc_ZGON28ox%qep9{kXJCChW5MHRb(TE0>7rM8F}o<zdZJ3cn9U@lyAcJcM`O#pzdb!g85O1O4w`pKlbE5z0%BSqm&>Bm(-S(<`)5pnMs3rJaz&SDZ~5W$4lrS0G@ZW!jQ8_})3b}kYGR30car1;8JiI8g*OQMY+xZMp0Xu(^mK<&ohngo-t_FFr_44qwMe$9sWVMb$x_Kn!1J&-4OW5F5N3a*pbj4DSR|vXb>+p*rJJmV4c7Zv66*V^*0UYNNJV*f+}sVD-BXgjYDvMQD7a4XjyKD9-z*b6`EEK<EyS$r#8+eQ>+KUE&#wci<feo^s&(L6HXBvZh{H+d&0+#QAt?o~Fj;^+b|ac(6~QyF_NPm5yZwuz)L>0hxQbgAn)6h)F%Sp~S4#>!8FkIHh6m?@M9<g>8uL3p;kr=1$yhN<hfk!AP)0Wb=Ci4v%hq{-9)sFh0iWb;=wo#lAnUBnGDhL^9g)FEvie`@)ip!m?OvnZ_=vdFM}Yt?&SX6MO~TC>mrgM&#;IiCpG69X!zfXhx?^Xk=*kCJ9NbtEXLKIjb-dkfXFJ1;TPnmr1wKt`CHu`nKD(t9YSa||=2$KY{qoG&c-iS>li5E-<Xutl9xZ6VFW<Q!C&!6i<I=?QlfTC*_rm96x-m_hIocxMFEXLPw@vjK2NTamUsO*2DitHM?B-|ZY|QHteVT=*i5Xyrqt0L?T>U~Lq^E%|LNvG+WeKO6&v4VZF~19}y0a;H_pMXjM?P)fW#}T|*BSRwIpU%7E2}(3nNsw^3`*1*xtrJhvokfXs7NbX5S0KiSScSGE-95}0%oBH5Ui`+8TH(j6xocX)7(XJG4b41A&z0@*?5#~AOUxyDfVsvvjRPnOSR#&F~1eo03VGe=rU$p$>cH*jM1cDFb4hZ9(gE9smcI;&gsI{GQ@aMXT&q4GS|r%EWPv+5Ezp_Pea|@tiU%*hu8AQBt(YZ8H?pG>FH<-%_6SI0ZPJe>0$^UlQ}~b7va5Wm~k3zo#A!OK1&ks`$&e@LLHvA7+(_G3{l@L^|eJUv6lq*YyuXhZ9T16r|}-?nkX%7&c>p*XqYP3Zlg8o`b8?`d74b$X62x;dy5FaFS(o~hXQ^Z)_lvnaEDqw5<U2VNc>}Cf6D9*jGLY7i&6#qio;<z?vQiX9kH#0OG%<V<`tO>)SOFlF49sD^LR|ghNIB?ffo@K+6=gec$&c`ZX+IOpogX=aA45fWVQHKv1smKu2-Gb8RKwKeHISXIgB_43SYRe=HC!*hcY;boeNPV^wL0MSLQsT6WNei890qvCdf&o-3;q3X#I%8TCFr8x9WY3J;XHiF)#}LPY23rOAE(Xyv6!-@)FG~IMq5DL$-$S5xR4M&Jg3TuMeXTY3bi8aLU+M)_xF+e+=FBH1<2EkfUh?CBknDw_3l@Dzai&=>5eMRoq!A-dQ=uzeIF{uj3oXMUCQ|I94XkXQ@1tas1x^yuY8m#iFR-q?byCIox(S(lcGTZM<vgcJjs;tPy&LtQYZa{7W&ujY@TBD#uRc*UPKbM%iX~0tt)?7XOuO%||j@@AFu8Pr)uren{x-F~FBRJ>Np)7>qhQsE)9`;lRE>7~S0-4DQC*k_W|2Md7XJ=-mh7kz>nFfF0S~B2Wqa@@UX?O}J6bUS~&?1L-;AQRjjGiv6JLJuvkVS`R3?RxBd!7^sn5Ok9QEwVMT|4ZA-B8>XWL4>q$~%4KjGT|lxQY>n^@VB_|8EUQwIR1vjERSO@7%~u7^Z2wxEP&V)fH=}R^LfY~~nC^dPkk8L65!)@X<r%uCe*w4zv?p_JoDlkS27>4A^!$QDxSztsliPG!{8fZd;pdna5f8W-vzXA465V5b7`}`;VWcYj6mJLdF@evcDj+zkPzPb;!@tvB_m`sS!~YFC^1n>$jUf?Pk0PWiL7Ut(o!(0JhWd2U8M!SRdP<Q=!=f5xn&iuPUZ@3~nFxco&y~$WY5XO>mWxFXTOlPCQs|)bg9H*oziwuyti)+?g6WiFi3|&jIrn$nk-({Lr`H*4aYz&HPPd6_w~5^&Rw#W(ai2=n({O3{jR)JOKd0~tL}<dgnW&bs>lGcT9HYhg)+k|f!(^_J{JBIzG`Vl%cS&+e3UT98YHXPwcSd~i^lClviolWj(yj7piI7C2u+itomf5dGPibwT2M<UoS%TnV%lu-{>8H#)7tBk|Ab$jWwJ@#K1rv)1Tri0~G+9fl%jZpeSJ3Un2wBH;6~KszBwzco;ej^4Su~p#`X(S|6rh(&MN_0dGcVD@_+dk&;ihfVSg8Qt&Lw^6hjcVp5d`#NI19{KOlfi)pOmoYz&PcTxBH&$G{+OCD{W~$qp@CETQ1e@TJ^)E5^I!*naWxzNj3Z=vwm7j1YOF!giAFkkDR?TaYikZQ*RIkZ-FR^X9H<e8c4tt=X4nJ98(M=)mmxQzPVAVEhpir;<t-8@8QYs5lwMCEQcJVK3NQ!J_VVI$s1)0!Mpgz#>$Gtskq*V%k&4hMV}#&QFF}AvQ$SwTf)^&tDbkgNq1aS0qiF&k&x3TDoG%BT9s<F^a|2y9yuLe`|0wKbm679^(HG^%qcFsm;{WH!~vzJzh@=p{}2J{H|Le<&OoQ%<{)bJk=FGicJeH1E&TtymsFx9C(;9YkxXs-0ZUT|@i&&wyA@PE&Qdrj`oaGeD#yh<TR~;u(7t3*j&^VCA}@reT4|yXLCf%&^@0;(+*#ZXjU)uuf+E9OhP=Jaf`4lzUF#_6q&Eu-*(^w`5zbbnfuA~p?=iyw71g2q?^c2U?nBC>OFqt*MhB0$3Q3(nr^UMDWlZ%3|6VpVT%^)iq{s?}*QrA5^^I~p2$Sy(S`*wqlU05uRbU!V5S&8Eg;kF#Yd6dFMtRvhy+RtMtVAPZ0glWcqW)}8r6{!@O#498(`DNIjZ`VhFj`UwKBm{c5~X#I3{mKw9NdqOC{HG{-DW}QSOIGvy=nvQdI(_J4=M3r7JzzAcd&zOf$!^g=K^3F9_>YAKkV~!AcDZnbS!|JhEei3nU<kA2w5U@!CS=eS3}gFvO=@jEadWV02k4;L&~{?EJjN8HnF(34kN$3kh^;2YSuYl82Rx(U?90r^&WCC^()SWNs8%5<iRQlrj)cXSJOg%b|J4@6JQO|+3(qlpRRbqBa)$C%R)&^ijM?=JrX)9k>3y+$tnhs?EON9vO<<kuD2w7Dly`lwQiO{wo(e*A+LCm`0?Zmg|1tMUs&L!;b(P!<SXQ5(dY2^21Wfp2?{X0hJh$%ASf6eDBLKti8K=5u>&(L>If6GeNNU})Smwa`Jr{Jwd28_{3B)mLw&<<@ZCe9%b?wmwRTx^0n?_cQ@<A8(LO8+#>I>A%yquOUaGulkEwP9$JMi2^;1@V{UD0X$N+H)W1wJX-mbtnz+iZZYxJuo9bLQ#OZnd<q^O_|oX1hiGmifm(C*KqaG~n;FUj*;Mqc=#{fIWt4OYrg{)R_fh=<zA3PGYQF_}_iX(*hnqS${zHWmDp_LW~`%lV6w-}$<opE}yVnaws^PB!=B>S4%ZhbL(vPp1!qtD9p)^h@HPToN`(<bOnVA2ed2N!M$l)!ZV^q=y%yZ(wHg?v!-8OQl9>rFzp|$m{Y89RC5f@gI->&j72i6qcB-g|iEepoI!n_SioYSH+8ko9lLYtx>IgT-@AR6wf{`mugs-u~AnZHbAH@$DUSem`=X5QV*rfiDbkU7l++;dY$fG+zsSGF~n~z0*fV850cwLmMLyagmy(Un=_?C)m+_Jt1MO4ODpzrd9`}8R$9OHv6=K<UiV&ksk&BO{n%b9U55+DE-!D01SyTuM~OYnnR;P2sxl6;#t&LjYmC(bDDPr0oBDdy2drI%`?S2WQYSv6w7gttRBo5eGYuE6A|MmfMg}Uatk8znVMwv_w?HgCZ8)`yTKQ(Bjs;1o%k;Sl^G<}-d@%g2%FSD46~?#MtBLO|1n<GVE^lCXqS&PBMg!P_ay{|g%etMeufw45_xkc`)o#>EYp|4B!_?{m&5gAW)~X+_C1-sXOeN;Mw7sSSOtv+&NMjX24XiaMHVS>#Z1}E0ZLF>)*j4PLfN%9NHCCz_=#)~F`_)8-DO?@`9_@^r)<k(iCJd>G^3oQmB(+8a7)nA5NT#_4z^TB=|DgPFOb3MWLy>e~@KUlC>?{&#LSEr!Vhl8;y$wmQ?MO+XeV*V!xUi6{BS2*|Vr4P<$^82%6+nXeA32zZ5Tt!g5;`;{i?q{ZPr@itc5@BZJro^DUPNBH58nulc&#QP3d?SLRv&kpzy#gYmr7We)keB)YE!wS5c;j{Om#A@wpXc8ogas8iQg(!|52JVzMM;hVivx%u}tM+eL;%w3oC=vD%UrN5D@XXU8=wVUn!qQs0sn^l4+QikgrI&57HT%GJo5)Dh$EgMy}xvJKe6qwfP8k2vGx1v3W3M(*UYWF9Edn##%669g)CpI~K{bE>+;c<R~ohSh*;Ta5}B<l^8pyq;jk-m)|Lxw4<CXEs$~d;aV&dsLI=1p)j9s&tGZHHQV{z!nJGH^6*Pm{&w}+LUy4AkLTRW?Rlrgza41yHOesrJFFmYr=_Y(!k2hDtTKs-{rY`|?eYMLo4bO4Fbi>pLXtpVmqNxDQ@YPjQ?TgA9PaLUSe=e4&Dn#yW`Q%mnC-y;BP^&7<*|rL3ZCe@rPE0FLm$yED~>llu>IqJ$S;p87U8X|R2vCrKtfNa`7yh|V|MKw46ZMj)C-*}!k2@Du*kU6<us19ZEo&&3fX+?0A$rIV;^?^pg?BeOEDv^6gW+*W2w^qG_9W~P!iUEqE%Y>DS9;4_5iL9D%DG;fLQL<qS1#5j|>l3_5QfKXLxrzfUd1*A|Bk$Xcb6N21R2KBbyh$q_c7@20FdsbhAcu8k0W1A^MILn2cZ5^v*lQU_@d4;#11Hb=nRpMYT`OQRDhf(U*7jm6xrU#Y?IM?O^3o?SY$H@kxgIpxyBedGE;Dw?Tt+(h{Vu0=B#hvaPZNK?%&5--3vaHpYWVbGIOqt{RRZPd38E81Ie-lO4btws>*E*896?d?yVjr=ca4!I{b+=-CyTgFuNh*rvqOoSx%g(_AhsSPX%yr&z4Wd50$PjzBZOhWE3H0FNjfg}Ua5eJSBAhl3=^-_h47MfYNFWbKUvbt~%lWKkK3FhJgIlFHEC>We5GUY*)RwKn|%OatDy%Nh=PZ^ZaE>7_=S%rR|oPHSR|(BquXL;?t;w+U;O0$tR1tg{l@D5(HZT+f6j0z4ArB|5p7lnn?m#*1`s6ic6j_ggW3-)TGi|L|fqkvQ3L%SKx9GYOiIu_JPa^WWILO<NKlE^?6vbju3N8+fU(Y-5tACa}x2f{0^Wyz}^7PZ342tDdQJQ5{^l_I-~+T0MX}>~$R<Ho^<w^Z@~M9D#p)U{0NpU$J>_M9y4_fUqSHpN<K!K(SFNg3*6}CU;r~(>RP+=f0a7NS~cx{sJLF57QC;#MgI#a9<Gu0LT%mzytYOhyzV|1>U6C>F@k(9;2LgnwSH#OZ0hB6yF{Whyt4OQ0;dP>_OY!SD2E7L{kdiT~zt*6uufggyHyj8KmxGN%-fqHW7fNNv7|K`$8!1PreMigl1!$mdraVt8FRDLo<{drUs{ThH4nKJ558LQIODK_=17_)1ItzhRM_&fOxa(0OrH8o$ekcHOB*P@pXt{HiHS=nlq%X?euptVfhEnytOmp-0rIBH3quO4qVH_?Y&ftZbShUS55}RPa!py*|&u`iVhmg0!^y|x9Nn7gJ^H>*@A47fDJ7(AsUsz8%0y%)}+{)Aqa;u(Pe3+Vb{u7uML((3a*F}k0KID+eRUYFi$GzL+U>N2^~eV`r#U%jJP0x=P^i!;7{5oO=7|aS_5u-((MBLB(3iwj}M2$w5D%Rl4Up6G0~Rh0Hz20n!eZI9t2^a+a27uDN}z`9*A{42KL)MrEdi<aLxiFwnB%XDu|X*U#}X{T^K!-29vSZX}MHqaUt(Bp^QfvqsFx{Fa@{gb+$X*&KORMi@Au#Zg(#;t<)C~%3^|>a*UKgyJTTp(Z9<a<Iq0Cl`6zFhz}xnpuD&!%&WVH0k_Qw){g-MQsSUS9ld<4`Yr%T2O8ze!BU7_s;)HvSpqGNy;^SEsxDW7tNWOa!N=u>joI-Vl^Ye{;0_cehd#Nm{^)BUU}{0DYMdhaC0~6MQx>VSieVtC?iD|A9Rm8Zn~K}QE-si}K(>Nug>MUrbdX|iYtgjuP>(kw?9&zrm#o4<`Vl|3G2$VynxsUhIU0B#7q2i<03>75@4(M$nQD-7d2*WL%S3^TKM@HJndMbuyrM;TYm0nt+u6RFbLY-wk)OxR=bT?iSdWj`cugrUIbG1xz{DBJe}&00m1wNvSBhEwkK`R+&gLTKV&B+G_waMduJt8l_Ib`@Q$!VA?kGY$=ftET7kedO;-i)NM~1AAq|OVFaYnUv!#e+@3UP;=kB1Gx>J->c`-4dO&PftQq=q!yzhL8i#Q01vQdU(JTpY9}OF*LaTwfDXV;)9eygvRmW(23-j>xe2IpuKuf*c)RMw0#=AN|DE<@~k$<>uAP^FPMA9AY|6uDxaZ#n8+tc?s*~)zTW)G(iUNAE?S4h0$p6^!OR#-E29VpFREq+WlwQ{2%7_;Mr5o1Vr|IGB9)gG^&89I$wy4|B7!yB-;@)mI}Hlz5w*{6VA^?v5!Zh<|5&l+|ZNXDZUcQWeE!@hJpk_p&Jyf5w@eYyGZ)dSRgKER70u&Hl*G>Pk)NZ)bBL%Fx-D-ZMi6%$F)HwXdUJtlSF%LI@!R1cE7IE+ip3AQ^*p-i1njH%|{^dLbd`l+jd7}CTET@7n$a2sop5pgbh)NTM<h*Q_nH0p)mfQ3SVRw2I4+OpdzOwpBXA3R5Zv>=~k9E%FY)G(2_SQkGB<NSi_R>YG=1$p7~tvj2%*I3`oar;mb_Uz*McA_LcR5@y_SY+4)@1xiHnbn!eG}eF_Hqoi89LV1y8sQOF*HXFeDxHUw~BE6Zmd1pXmStnHV-<q;PAYVQ&FuWqbO={z_o1o2ha#b##+n9%~-33DFqhAia1ss1eUy<8{+rhT-Pj-0+n<uIt@@L?u@fHU^GjG%^pY(5{^Y@y%{0p_GeY6E9X1^X;>#}!L3l*jrCV}gS~zG{}tHMy+x!ecW1ntaw?3YR@1qK}jt@@Y_UF>nClq~_nsLOrsOYOJ;*+iNIyI(gKw6W2>5<d31v_gpPGlJ_+zb}W=BDaT?Rz14Jv+OQ`wCTuEB3__<by%VPSCdbX7X1J`g8S<GB(Lm}o`X}~~3YLnOQ@D@9DUriduIaYOqodgw!3-gX_2}Or3gBGk2oCC8t$t8OX5u@68b^T(C~mtb7(~u+Y6K^7hNy1(Qgu$`%&Y1ef2#@V_nh7ku7e=HC<r138iZjpHcjimG{UMjdTI()h{hR-WQ}N4Ry+Vp7d=KHR?-l&PaPdt262R{UkCuRA2!d3wt6GYJ}>n^aS)O1eK(Q5v0Sb%X;#xZZ+uNlMc0Tg9sDL0o$iBOXX1_FiiB2*0|7(#d|0k*G?_=Z_;Dd2@*Ty+7R7sTtP;rrPmiBX$9mys(b7>%kRP#zb1_bV;HkA4r3;oBuH+66u!MW)aATKnA#1u6?1QdbpE0mS=CFX3B=vUa3E*NcHe_8pWWchWX)(bAk4sCM9J3yD!-prNqKHp~m^uAq;Fq_aNesrQ%3Mwa^qg*>bHNo4J*mLPSpKSSY^F@;4%<2)i@CrsS^|O6R_LdD!-4H@DK~;SrKdR{h2BU0))K&sbd9Pg$@BVc5UgkK9cLH#_4(5yXr{LczJ5wBeVWQC__SWF+ok2zO1)mGBAbR)poXaa8=0xU>7-MCQ}|C0@7(ziKd>|VCS0Pw-#UKzP%Vf4P?ho@`hNbHq2?(5;PJ^<52MA@aezB_?&DYw|0=CANo(bsr_7KB|3><zKNKM{hdj=6zUfcCz%0)X!)rKx1w&qBj^HKnMbQ>qi-0c4Zz*{Ai-&%VYmp`$LGjYhtSi<<qV~ghDgzkrl0oqvzdCvJkS<(e8BRV6tr>TPPDy;Z#L-cXzT@W)mHSwL^#^jp$Ec03_)!d&R1jR(F&p}S!_g@Zxn#u2HxEUHdz$`30!J>JeMm(A71e1G+8bIeFbluJAa&9aaYr2das1{X>9d}Y3g@$jBE=wi58&sAga)78xs##lg~#6_P+uIsN+<Hm{~FI8fvi?2fs+U)<tzcAXbMr^Kv9YKIx%VG*J%VTgPw@kk)V)}8dNT8{fU}&tjC%mR{Hhvv&4et%`V)Wo%UX!8{ngiq?OYNU<SLL!Dvs*T2ooLqDjot5VM>09WJ)bv>$m0uDu<r9LI@M!*AlC%A536zNybj3X_ZUA*ZYK<3Eb=@*z5uVY45X<KvuW$l8Y}!IU<H;o;wPQLv<(@cXluN^8rN<x(Tc3{ie3c_7a@Dvu#N)~65RxuSKoajRU5{EYC&BuF^c1l9bM`T@?c7;a>ertZZ<{=;D;NFNkreHhll%47G{$>*juNyk&a{6m7u_>d!qUWm}5BQ7iAN3oGF%nyFdx{!#TaFPyY6e%CaYt&Hxqd_$rgJXOsOII9n1yfAW5ILtv%Om0iUW)*^@(Qr)Xl%Mst)W0+FWo9HePD8=e%F0R!j`s8fK-STKi??Tki}gq$HNHX@6tB#`vo!f#+n}hxv^GB2qf4yrS4aF&yf8*c*x!m)MEl$KsGMm<(=l(EO50-lV&0wO)OKMG5H9-ALyBAfr{bi9ps*bPgWZlMZrOpBWzQkCJXqal!#rq8RCKjqZ8qJxCcBLF-}HPeA^<oEt(JII7fSxHq|6ZPjNCjVy!}it4v$5=fuVO&m-KuE~#(;!@%)?fhYZstmv?Xq}eEb{i972Ip@<o-`S~$SYp|4lvPGg2V#-q4e0S%M~OhAkN_kYji6Xa0`A#HiwhF*4&^UtA<p?z!Y^2rGZf?OZ<Q^qk3}VrT^_)gP!h502K${p8+%<{wx#e~+z&<T2Of85Wxy~VvL)NV3UmNc>_oyM13D=A*?D)|zB|HPU~(HnJ|Sky1sT83`<u{Jb21uX%J6#w?f^CO<4J2T+>r=632e+TOWSMi!o`CD<l_OqWD^kzzZ&SvZ9nC$?<0Uk)SNz!AEZ!=wNiUSn*=&3pi+jANYg2!oXNTQ$t^+dqh@+s`fmWcOMZXRh<?<|)GF(AUDwJ#->B5ebu5XuTB?0a!FcxVa;<Vhlui$KjeT`IwC&l>J*U$}^s*gq(B@jSmu)e|?aCS^6tJsnD<3Do4sde>=qx?t&}U((x^|;dTa6?dkf=07KbV4F*mpc=?)C@3v(1W(Y~E~d0AK<%gY6?xRqbRr>hw^`^!+6O9>U8yk`izBl+<@YugbOKJNVUamCKFrjT3i}nQ9CTl{Vop@W;fSgpA~Lrlx9D&+LLT)Z8-<+L%8CT~xZ9`pR<q&>1iSfKpBmKIszZ(A~Q0x}>Fk(9Y0jqW@k}RwL&=4n!J8^o!}jNWoi0`Y4JOu#C{BJV^P4GtGzic8oqPn#Myz=ueSPzn9)DfRx8B)AzVa@%;>(`TnFg+yfB+<1L(_o0`d`&qUB=ib5e8++)_6hmvvyW_HKyBl->_7^K{{2m`oo2^7LbR)o~jAB&2;`7W$VHqx{%4o0Ean}I74mSk~JzlxZ2PFJN<dL$AS$xoUTL$|szvycg;A(4bh5SCi`cBTBGy<UcUYVGDJv2(uT6Bdn)xtx$zU!EnY{>K`-T+?AuEz)|Z4YfOMa6b-D0)<BMo$%PV&eXdux^Nr$uoT4{BA0apW2WBNWPigP8X|Flbn#JbH^#sH-(e@1lW+bFtJlVTc%CYG5uOL`H7MIy?7jP1<7RX6Te7#1vg~uV-bH<9Fo#0OBN-2sf?v!4MR6=vcwxZdC#y5ec|><VzS#wlwOW*hIAZS`)p(?o#Zt0qgx!3|xWt((LT*$iU&ACmEY;Rj8l@?Kc-p)L*6N+A0@DM^kzX&yq)OOe|MK;B$Io1?%T{`**G<(*ACjCyyaI*0#sn-!$wZXs+7e3<5Zy<8EcGyAD_lfk7Y%Y7YA9cP-r7)-aDcK1+<-pgVQXjO^?ia6PgO<TY9j3{U9zhvF+US+e(QCk3_8-ej3!bmxsO4<tPr(|&*F?C29U03<`C4NDhv^&{}U<h)#Bel96^GABKZ;o);=`aEy{?eP()ZzGzL}VkpW}sQMNJ1Tq~ahP70Z5eQ)3hQ3j#ymr-wpSj6H4-I<H~q^u~QjkC1AUaQ_NFWa}Pf#%R&E3KC6manV`x1_8X;id=ix;0*LJQ>2y%0Y?n3W*;$nHVafP$06J#stv}CQy$*74b{%r*ge9Rl-)LH5v?ool0jMk1!E9sp#0-PK$=+lQScy6612AUEF7)ZSM2n7!nuK)SdiZ5d5d2o-`M2eif+*g&Ok5p{*_W*;Zl1#Pqw;r_P_a7WL&!ZQi}oN-U|LuZ$Tr<C>VhW%`%}R9Uh86)1Z`6zxXMsQMzN&y%9)gwrs<E1ar>)Mq}%rQ<5DzBRHY*!C^ZBR((uPh&<6&sNNJi_ysTi{yBKPpD)g3>Sq@nM<`u=pgHZKJFusUKw`e55>U(rWM<*GdRW9#f*(6$P1ko&Q5s8#~=dd<DtP0n}VApkwCVOaf2~u_=gk&f}Ye!rr8f(ysRP4r)HcBj?r8DxVPGQ$GPTSJ(rLB0{Xa<YW@XRK~Xb_1%Bsm=tvS5Mn<fVsWPx_(g`P>{u^0OMNI;xc)<U&sI|iGWnnLOw7$|rUg1!GIR3LGQj}Tj?f#u`mGYC}$yC?yKVa-ELngGtNz!y*!t`DVJL4O{uzw!Zy{4RJPuT_<WIW5@sBdUQLAo@9vlATsT;U4vegPLHb=qT2-sqRiK7Pek4GzIW)|uH`#jW}^`ESQ<g+H<1`h2$OUTe;`e{#R|q0@J}Q=Qh=OKas7QqqL$sD;t3fTvWy@yAD&!9MI3G@73+XBVs~dkum^Scdz**ba|d5MwWnrk@FH5=`<bdAxHZJe$;ys7ACn4~pwM-f&V7Mwu=7tYkuJVIV0*C5b$OWYf>+#HwJSvc=txJ!Fc0dTJb19~|ya^GnRFhqb3Ftrs)rKbet>ugFC0z<FtAUI;(bk1Sfoe=*N2M8u}8=&pqpnBiN9XUsZ?t#C6k>#bscFzV4fQwc3JeQ4IB>qys3H>2pZw~C(A=5z|F<VJ}-`GA2rP0)`ymGrx&3$(sqF5)6`HRfUjCW`3+S|l4U<Z`EjF5H{6U$Fflu>+w$54{^w#fhNz_?vh8a1jgb675%Crp;A$Efd=PbW$(MGLpNB@4u!$d_fv3pr>Jm3j(AzQ-3GAQRn?I{7@MoodM3K3sM7dT!B)zMI~v`u^@d8!CeOoWJ;4)CWq&WhT8GB&M|>anVrvGsxD(RXJxHXD~U{mAgROtJDBYppO<U$c{-#;OG5ihcbv+CaMj;Sr32w8&^~c)-QPrfp`XUoTX^ANS{1m*zGpe<HC+?m`bmLB_EpZ~|GDIX&N@xfsmpnWK^HhS$UGw;6>+3V0o6^YXcD)MuML(cDyimZaG&kco%F$}8$+qiP>>NAK;pxuM3d0W#k{uw2PnaiFgoa{jr#zFlg`-wr8^q%LDuvKc$ZE8wf?vMV2p#syOyhUTRW~)$NS{Wn}8ex9F9&?@3&F(c<h7?+Z}Slj<mc7uyklJDm(>88$gd{OJ}T-qjyDeUs+L*nA>kr9ZNZQ64~vS(kyl)64$&${`*Ui*uO=+r_^Y21gw<ZIf1Yr1Nu17g%Wy%co)TT9>O4AX2RLs)a9u>oim8uJ#>oW6X4ROzJqOb889xRP|RZF{BKd!%~Ud_d+X^=6KZ+>wUT!JBj?olD|AiHfBI;GG{(o9^!Z+3UQjn&h^n4tI83-Y>7{zk5Hq4!n{u-(xoRtz&{kSxK&dWE!V02%$-ar6ah6$KF38;Ylqs8V`A7zslJ1eI<=0ZWQ*+)w)vM)6Uh;r)`1wV&dmOrEEmp9J>bn_dXD1a=SE02v6=yUt&w`n@C;*QiYFkZuW}1wtvS3<Dt5$&xLucKA$9#;%OPRJA<>i~@dQon{=G$Vk6}xLBCPan;3Jv5_P(OSS;RqD{<R$Z=d5PP^M^qaswjk`AQ@&DFqP=thHG?}DF=$%kJl(;2hr9N3$ix<|-eEF-2<Eh=;6MVf7qi#wvAu?LlEGv&$$h<I@bl*0*5VM>qdX!t@!m|%!s(}DIR<$$f%QRTYEpIqyAoM!aI~~#_TBsTJ<s<K3VxfxvC~{3*c<y1AZw-5m@Bh8@W$~zv6mSd^lkbi0xdkY^z;4ex1R4WoMzJ*wWf7DE5AMrk3tEYl;<=m@Ya2&f9kM&xFZLo(C!YLaf*=;HMin4M6j?}+>4(rO|Bll#<Mrpr4|fOR^je^tdU2u;EK>!=?k`2{?KFvwW}a@fHb<2S8qL5mGU4}Ygiy9K4^ed>Y(x{*sXRkL<VV?gseaEaat-;wvvb`^`a~&(vkAZX*V9Ho%1J~em_^xA8Pnf&`6+6_s6?7xljSMR2pQ$$9y*Vr;NATX^&I6^pwGd15AO5A<0xXJTg>lDw!SVFSC#~9vaizp-hyyd_`B(FL*67xK=coo-Nv!vy`Mlx63v3n!8>rmp%YiqE@a~S8gZN-BFxdM8+3V=7}hgaD5b+(xkdPL&mE&qEe}!V52~BJtyCYT$j|__o2j)WD=G2O_PFA4rLw>VTa|6ZluRdSc2~&BTJSDwRJ+a8sWFVP8+P}_e+1Us*+N1d?x~=UWOE3A<5&vL$9n$eMAa-PV7<6<_j`h>RWcZ5N`i|RQaQLB#ju(nxbwOO*<`G#CI^ba4(@?qDX&1Oix_1`Yz6qiq+Eexa^Lqua%&WX1IA~r1F{kg27=&N(_Cvz~~~OAr0ST#m2nlm9?b?N=#f(&kBL@mG6y029@tw;D1S`jTgLlu`3*Hfk?*H(NBIYld(Jz^@4&(3x+er?LQ`DAa*z?+=4+9q|AQFC88AgoH<rJQMXSx5Ug9brO!kfm>_B4vxv^zN-X4~-vnGJr{bON)p>6a3ZD*Y0{9c|w8)BSb^3h^R@k?g@w5ha$GrhO1&(w~1{Q1RkeU%*5TEEhyjf1+40J)^epLX~IR|4Y;z8ZI==TCF?A{=Qj&7-hxcPL9Gxww4H%+*iEz@5R^7Xh}()AOU4}dVYV35tGs-LFsN{8@$x)K^2#I9bRI!aQ$p&+ppa!l~2(~|m{zlLwoA?0s6!*vt}IHN#SeY<Kd(jr2+cu;7;r{P@t_@wFdh7&K`0%N28MZ|MHCvwE)qQ2qM)}mk(5PdfAiC`;ujKuA6$S_66`RSse_^R<?!KNWK4WEu9R?fWQN>jdhG!ve**k+P|<kU0%-b?i>!Zi$q2-NWY9`A7n05}lZ;;-n%H3f1z?jWSIrt-6?s2UtQMtlTuhFQhR+D*IqL1VQ_xdPeW9!txWR@XO>2FZdf`YHDuJYp0j(4v$pTg6-^t1SsR9Ki8c?Qp_HBe2DVO=B{OLY^KxO5&#`3=My}4hF(^Nq81n0r1pPwtyBARQR+F3}$l|1i4UMM4HPf2aFqK@N;@m%w9Y@?SRStX+wiEF5ZUiUz9CEi`2ALf7c8Uu?+d^ik&%s=y)F?<b!Si`~21tgH444)J#K%f#!O(vevLatTb*_HyU<O4v3J=Oq_=XqkLoz<|W9I1;}P)Gg-}?;zv{dLO8M|m!|HO@7!ggpL~Buw^s5!#C>wQonN{NZ1IQK`{-~lgLvt)tlcv*@7(b&0fHJPUW@JnzGA6OENnQ*15s3^WSc#)St90ITIQ+59C?I<lN$zZ7OInC$N|IWbOmQ290xI8oSC=~CrDlhQJ$2TVs>`<6%o+q$bSp3kuQiXexamoB#!?DPO6}cO3vM4R4lkAN^ko9A;jvk1AuB3iusI0{2V`>I3sj->brz%T}=J}G?HDYMtiAN+v3BeD=VcQ8ck8Cc={xi?oh0i^`2VoNMf0G5t^KS#d3f$>67sVe-`THKq!PIMjYU<LMK-s8f#cV_zd;H2BO%^#b}9AW!&fmy&`KNUfg0vY9WipJ-CN<`ohPu$l$$X+T;KVw@?LW3)(0#5eFuu#;m`dHWrk@g`@nPKa37kLJtjXrH&GIFQRBh^>Hfh$Vm!DuAzZ_Z1SJ?b6ad>OY59E@f{6_;Fq4`-9?AV)k$~UsbB}0_Gr+vDa;liwh6BS_36MpozgTD?tx^v6XT>U_Kk(lItV3hi<peqmRxqqzF-X8N@c0RdqeqW-(rLO0d*UsIS#;_o_03x4J>3YW5oH?7r7ZZhnf-Q{ehCg2Ku({4Mae-Y4KaJXlX3<;cSa@Jw1#lA5Mwv*YPuxl0N%|i}hsnSy)Woe{kTtg@@>QI48<CF=FidYfTe;Lq&~Ek*C<>sx6h}aL|y1v>h-<f<@8R&I?h|2+v~0Zg8Ld3oItg8%~oEC9fE8gmGrYO6EOOyv!qWA9KSJlPN@JRS<s<!f2W=`kbA%BjGO{^cLa}3!$$fydk;+C<>4NU-sU<yN%<_6aD{wijH&E4M<H8q(s>P-M#~fki<3LLXff@nzt7u8YChRK+qs1TIL+b&Lo-5Ogx@D_hip(lAWEq|L!@mEyePy&Y64x>MPt&a-Wx~>Z<DM1}Hh%nLU#^V-ele^<GasufHd2SWyrIaDL`-Q_tJ<9KQ=W^51n>NyZ)YC`8d(XLnXkEG%#WvFaNB2MB3zHzZ?&O$tQE<sqWdc0)RsEl-hcavh~7p0v%do^YR*R09DJJ9M&SZ);+^3b&B9hl{WMA_cZe-N{i(8RvO!p_T-Su5$i_av+WzjjM~#542mm&o499bxwMMWwR2!`ove!-4P#(Sb!oNR5FJI9X@;&6&*FVwwK<PLSk83Q?mOdVMC!DeQGNd1I}@7*Opx<x?9I>ggrB-aq6fY4M$IcB$J;8r}tJ#lh41Vj7)@4PAXt`=%Ai&T5zLg+HTTIYjHX{AB;hbA1Xz-u8}2Br}wFqKNv{O$gi>oniZE5<OA#2XPdAnO>j&OIdTE0+ek8`L?j%UPbE<5XVh(>cof-~$UA`IDW$?DH;_f5qFPZc7T8R3yccGI4|Ym<5gK7A!gpSF9B0>8dW=RG;(ssvDPjhMxp+7>l4=@SLA)i5uVO}yIKzx8^29naGjq}ySKy0-Cd()rYTO;^RbD*qr^VQCnD%-`zA8Hje){P}7^3Yp6%ppx$<2vOQW`f!_?KxX%4c*E-J4MQshf`L=U?M{ljYIAe<bvRIosEh@V?oD{-h|wX=a{Z&Y9=D^0~SK>1EG>g9<KB*qKPXnZ}De7DgumH_pLxqY<YKdzbaO_`;!a;KxF%faxkVoGjzO3)O~*1Yo7@w+b6yC^cLO84!eKRv~21ssp8afu@o{ik~m2;U7{PqOyTFiEsBLj>P3$YEc$Fmchz3XL@JGnh+~yS<jj0b0Fo$$-ac4AJ;IW^Glb<N@qmB1NbleQ@%{Uvf+{uN{J~%FH~3>m(xro++*ji(p0lzc%*d;J63d?x4h<+9;mj?%B~aI>mGH-VUGQz0Cm^`zSsHhG-xo)E{H&*wQVm#(3mVZFq9EcCAp7e>_+-Pf*=JKLT42YCwg1fT~36%oOO24N1TBm)ZV%52fAS?Em`Td%SH7}FgR%ykygB@Dl6K-?1ecwh4rLmDXSe05T;A&BceL&tf;xsxUpPc*?JojEFf6Wx((__ETG2ea9DJzWBh_`u&tw`ct(elBrFDwG5dt%;fE%FL&m4S7SY?O0re3}+hnhu(c`fynW2hyBaJgK=E^0AeV<4kBZhPK7IIEt1?o3Mt-lq_>1X5%@X!XyP!6152*3sY==`&acmDevq;g~6<}s-_1Vx6%qXe+To!AG<#h;uQW8cy0*mY~xB}}HKWzW`nH;#wliBNEQ?|COA%lP~!soZvazVna`$8ZBN8T5k_I|M$Z@vi#2&RhJ)be_$XBkdQq?N)t>n*tBI{Jvbvs_o=ErLVKqb#UIJHDH(s6EpHJYm+vmvd*BanIX7GpkLb<%B%YX=Y$)Ia6R(v`Am!NG!GPuX0|;8;}Xr(ua9MF8!cM({Nb#X`sSlq=@C;gv@nHNi2(`jG);<s;Oq|L5viB5P<h7XGWo(l>0VwH_Wht(`U^pgi_K+>);@H`%N*O5wjch4CN4*BCRt~TuOq86eF5!ZY3~ZjjM+eMFt~d>lvc$Z#}?72a)SQTg$&@ZWrW<Q=2m~<R(GBD52=pyNd<~YW%69*wV{j79L1x9nC&D19H{5&6Ks9D{X*zK{}dq)2U#37;ETHl$AjZUw7erbz<vH+IOis{;v}fgU36Bx%(YkfeDJnz+@8h?VoZ_M(GWXo`4Gr?-!0n4*1qM5=enPJ<}U;s<avjKd!gTpJ3zChS~KeyT5O>?9z_F->u|7yf<iJ+xUspu(bz2H1|{Vtf!)P5Jr<w|MnKrN6n8EqQjwtjXY`f2pDbS_DYLo*@H0%RluQj4)JP9U$O^wR4Cu_AVPV4r?!JWig8f^Dn}$v^6a4Jgl^hKMAq33Kr51VV<;h8n9&*OR&%`yHLQ4FCFsnZxKVD8?=n<FsQxLAb^Z!aZ6U?%=FgNd=|MvW2c0PD2Sb@*kjn}~+2-f<mXCFgz0!!d?;YZAoo;uh&LLFn8C)^XaHtVZ8;%Jz6ab`Z+sBhgSD3-JgrHEh)tbJQ+S<Q^gIQ@h|A?yU#*3C?%DIL)U{Ks%*6j@SiiWBX%k~rL4U*DSb+wllEnzp~lPb)`v+g(g-gMx8F*Kk1SL<n^5aP33{M<8k;eLU-9S2$TZJ;qx3uZz~}o>{FG%=fB9qIR6@!@Y~dsv^8Qt%DqC>JlOxLS;f=mqbXW*PTS9>=0|}(s=lnl~M)UE!}6W%6$}X(zPvL_RDrJlhHn==F^;1{mWDZ_ERl%+3_hHz<(sH#78Wv5i?8V0TDLXJvs$EWZiuRN8|k4MJ5k;st|;Op~Qa0((8Yg3OQZ>uN;Pzue9)%w9k4*q{T>D^1ptHF@vJVfBcO|NTHdQ|M(j^?~kb#hvSl#QS}kY)t^Y8OpXY6gv9GpQ~)aM6Qne0s!RM?u~&l5YMA;eqbcbIFw-SUrCV=$7{qt0v028y?b|Z2#NNQlXy&*}2YV>KF+O<UnRwv(Z&AR6_;%EHV|{yVOV5|3x{0ZYnHO<<w%S-)uCJBUXe|v4*0Qv|eSM`-rhy;D?QRR>1O>)3qu=NCM)`Y-!BXxEy@PGSL4!9kM_+fJ4~wJ9oAs4u13x0UJAr$@-6iQPMWXgI=WE;=jS*;e9FM||UwWXQ<ue%3+ET$nFE>3*z`TY;*n+upLGvgw)jBu;b_*e+%Tv1Z>zAnoBbWxbsL&GS&F{!1asH&(H$NxzAaduai*!H!cV22Xwmgi)$4v$NKm02)wj%OzhXV6oa;OnR)aVYQ>Q{G}JW@6RvjHc=*~vr?6duh}_WXZ<wDrt;^LAsifj_j3pe2BtbL4(2SLjM@^?IeU0|q&Y?+;n0gTpcUuX-xa;^eOQqBTr(S;nbxG`QDliy3^b7N29&<AyKp7;-JDUl$1}BO~8c-N=Vm|MnUh^p>K=TQ^p=mvVy1EC(ocWFDkFIWv^4;;v%j`TLaf=!4XzF4R96-&V>e#0p-9wEdbcrzZt{N(00)r`NznR}h=eKFApT;ClQ$A817UwbjW4P4ZU~<~Icfm3JDF2f@34;i+_6jAJt8{Y*JN2E8^NOTon*^agt+;=e8_Myp(-VwK~5uiMW+wL4slEQi)uFuWv>)6NVS3$5%<l=;t^5s_~zB!uP#Q&x-pIV~73qYcl2dxA}!<T8Bz2jzB<WeE8Cpv;uMe`9^)Z9GTJB%C=98I45GHaQo}2Umit!9s8?cx3`W`_?*CC;2`x8|5>vbaQ=kwZ7$T)Hi>)-Pj6X)T<kt4YGrKKWc6*m3>z(POd;MkPPy0Gv4_S#uvXW3T_Q?VY2PdzjEuk@4P9THTYP;_z>CTB%Jdv^jYz3HFaHL5Uu;sI6fLiAli_V(rJV`ve3U;z0iPgU*(h)23KCeY!MqyIcE_rx&b)JL08AsanJ_BU&y8SDVJs6wuSdi_#iqxbL`;=#z2_af)vemQehK6$&vetxU7z!)md=XT$Y-C!MU=4f_T`1xFL$!sp8{fx-F(uPfSf92kElCh^5BE*WRnFACK2Nt8|DaE&k-;)9`Ak^z@aU%HvxvE0G~h_1I41mdidm|9bK~sg2YP(%dk(r{ezcQ9Nql$w;N;BKp$R_L{!SOra}elA5+vc5Sq@Qqrm9Y&R{L&4Z|0wEHISiix|UE4N3NrT_eDo_mv|_baC2D4Ei0nFAZYq+mytL|bXtzGS@va}}SZBK@K_cTK7H7`#&tpCmKtml_UianER{?7_e|QOl^+KZr|n*UGiYYUyZC?GGPzNLSw?G~65O^_52RMx#U|$s9Zqr#tn{8@KD5<-m_3WQL;%Eo}b*#s_<f`*h~Kd9yvaz^0=85XlB=)%j_**~HDXCpXicYcmOtB>wuv`S>+q2SMLF4f}y}zU!>{(|i}uuQj(h{$W!i3_Wo9%uS=7Re2uWH(1O*QaiRvZ^$z1kgh<Qtny`l*Rd`a=a{}589P%`%C`y;@scAX{(&0v8Eo*a^=V8#OjVt8JuS<c*Z<c3__)=}in&tCZEmlYY@Jrw9qE5j>lMKQxCgasiNcurvzjx6LbJjY$_^4J3l{AEEPE@<tIJ#7e6?B?U<SFp&ZeoCqB~FY_d=!2$l05W3Wp77$Z)_&{5SSpM~@AUL*P{?rhrA12R-GuBsR$}*o6c4k!buZa9E1qJ@@j=wOyiN$IhnWo`&qF#1BO`msb!lja2MetAf89I;+d6?P3*aTOwQERvmrCe^^gRuEYt=YeA-efhn0f`%9YU1<l|1r8B{c@;Qhwu+Xnjax*RMRlV8F)ryGN6#mesi()Y1?08GtH{Mvf&L-I=+_Ouh`KjO7it4v+Z8m6^x#;^ejD5PRYQb67BRJif0Tgx37=5O=nngQQ?r%IE^(1Du9rtjPx|Ac8{T}+MSRWniS71>HOt@@$YNys!$1x#G$1e%M0?*YSL<oh6jPWf&@qf!eLTEwxgiFAOoCyl<U3O#qlF;A3<m^`nVI<i&&h3siWi6gXh)`c=PB+?+bwrE_j#a{Nse!L+){*q5@LmwVB>n^Jw3B$0puY@bJ?Q>bR-)A)lbTic<CPgYY){_;CoPF0X{BH_yeBlNM)z<2p0Mw-vsOuSvi{$DI@ou7f+3a7wfBpxw7EvsfYnTRMe2vlw6r@s<Q$IHMJ{qD7jQ~DndFpqQs@n1em=oV+N?=h)=pEO7>NWJtzZ9-z-UA(jXMo=5H(xT_naGI#voI9naZEoy5|p+AZf&l>7u5{VJ?X76Q{ce1CfyeUxwt_-l3@%vlXXjp@Tj2mX~JPj4Pu^bVh}IO21I4LR7#D`=eT^1wH8^tT|VILsMZUq>x!GvIx6YgvoJVIiOLXr&^%WsC6#~i30Pbx9)KOI&!Nh;yox1QFWYZ-h*cqXzqBA(&M%6m7QTL)(*TMV(TJajlZ1)j3Hawbt79KaOpuj>L!N>`O;1HdxIn{-LEjJxI(Wd*w1nRMJRDm&iirzqwIxq@yx5i2>-H~`Tn5KK9%>`RXc6H(2G!h=5ivG@7RZMz5rWcZGBDXM$?$LY1|6$i{8pXAH)d5K1J8>n*bz53ro-!7lSwvKpXdUoQ=eR=-dxWzJ=S|PxT`R@0{76B3!uuyqQ2z61-AF0oEZvJ1YG!oGZVqe(5d2i(N$BNUV_@3<hl&2fJ2st>{`w<C)6g5#}1}s3@+Q(OxsD4fMQkv@6bPIgxL<1Wwqwo_)+AeOk6bLM%g1AJppFJ)|L0ALZtx_QmE5GRZtb1-maJMma(FphIM8JjT4I*-UumB+9mqlejHn?&Y+mE~SoI;QtY`q=Mh6BnFo6>@npGa35F9eCIf+SY=*BMY<x<>h{Xk@&<@%X<+J|`pR~rIn7xWM-69sXIcz_Kpl_FL2x8l+0CLz$b393Rp)??;teA#xcL{49tpyDD?xr?!$nsBtmx@pgm10`s)2`{!ZfJ9Iba%aQSlfNMz*G_Q{NOCV6(iS1Eplb>WFEbZh8I;uoCHf>dVCmP*p@3qS<JxGgWoQf@bP7mwUSY85IgRhm&dRF?hM(pMN6a-i+j8D*qWXoNUlS1}v+V0tS$<Yo939RW1~Q@HoiF$rWHctUw@c7WqX5bNt8OsL(=<=gay}z}EJi@qj+=aCFgtYqZ44+(mR~wroat37y4uCh}>|7Y3V~xPbbSK?ik;EkG*jYs){XqvFh8H0&KG7Zu$?Q%c&IYNR=%3bLKeA7^H=1mq2lSr67#5A~Ue?jV+9?#P9L(ITaCk!^BXL`vzMOGKXfB>p{|IKnxB?Jo{i%|mqu`xd`XIK&TiA7y1&B;@2&QTOaa%tGkV!Cde}))72;0_qtfHDJ^n^lFHRGFbD8i6xMiuw3LiRH%V^p2;TW3i6h{QuB7*u#RG9xmXz^WZ_TYMgXI9z(}YB=nS$#aJ9(AX+4A`T-ruN5%MI8n%k?Z_06}fn1n@D>gIA|WhtkWS!o0VL?&gK1Mb7>qIkMsJheW$vA)s_cP40{gzD)4c4RzqYL&#9N|jSNiesNSNZ7IMWAva6{9z2N7d(PJ*BhKtuu@Av|0JzmYjhfQ;O>oDCkHui&@)&gr#Rke1|=i$vI;Uvd6z*2aD{iK!&2p3gynh1Y>mvsPH@GJ6yS-|-mmF6>7UEY;f~{C`5rnAI7VpC`Qu`pIU1*<xOz3z<5_3}0@*Jwiq{L-*R=nqb8>duV2kF&3T`s>Sn^>?&%PC|n@pJ|8wqlbx@i!`|C=TzL2Mrtl2E{58nFj_XahUqz!gu>7r1f-@dRF8V1c>EC&d-Wau;eM=ZNIAr3}ej{2~q6$1|q|etm=`#TU>!S2IpKFFTe_p1;?1lAYFR`=RniVvDgnZ23lF*L{I1HTRCK*c(H@zFp|BE&RC-n|WjYc@JCWlodN5;onwdQ9(=383Ocg=g8jsGoBxh+{LV=(ISGTj~#l>)44G5bS~tcPGgH#>^&e(r&NTwzY=M<s?#wL9gTx9kLW`(lss?{lv;3h`GI37WPyJ~h#(&d$NjG@<9+dIcN3`{U&UobB7_XFT*uu`*Ogebs3MK9w~|2zg|qqU{MDJc`I#$rNRfMXU!*fuMHlHzDjCXykh+qbaK=T9$lFi+z&-XN`7ft|^avSVEAHVuNBtCXm+ZHc>c~V;&77VGX;RplWH9wav4K=-vs;ZkS6mFA3BM$_7SAVB0DTKWwLYUY#}Nqpqj-EcXs1pw<SEp=QD2LgRHA!N>mJr~5YM3>>HNYqz2D-=?pZK-pfuCd3{@2T4hFmMT@6-rNey?mJZY<dr2h}XJsn+n9;1QS2&odeVr=w=#|i+8@)*YV0%68QCz(fNyd~HxHoknWWf9|)PoI!M_(RLR8MmvR;zX`spoDDUwJ>LSEr@S-e2e8U+zYrwCfi8*ah6EHDcR~C#W2@L!;(%hoZpN4c@%fz5#nFVF5q{InI?LJ{g$rWa)5wwMx;NHu<wxasE_-~p+fw0kCo2^so`7%!cCT&p@TLJj>r20nB+)oB|pGdlXfopj$ye)w@(<GsL?HUwXwR6Sn=}mB?X&;=}fOn)MlsLwB~4mv5qol+9LkBLFv7$gYr{KuJ<Sf=+$zFp<WKRY>}&-zNH+#*Ha&jyqsGvbXpew{eb){I3Uv<3@p>hY!$j10;$%!@LuUm-Ygw+w{%`~yL5JJ9iX1zDW^pFJKrR{6}UNb<vZ@B@2Is;7dv#b78hU(%{|bW-q<Gy(<c(9lKdXDxIR1h8XiePDe{DVE^v;dcZ3!Qh!NNt;X{I^5x$C<K_NQsqpJbqS<)!7FqY_km=;rJyBbk@`D;fM(dJLT1Qt!;iD9cBBN9eu9FJ1@+s~V@-?D=o86cmk2tHD)JR<aG-HiEB&Bn6U={r97{rNwh|MU58DJ`ix^(ciDia7hl3|y6tXa$XbLvUJ})rDdaKP5%<L&KZ+=(w*Ffz|D`U`N>?6p}BL>h5=kz@d`TU8wVigW++nCCrpZ-Gfo)Z_KcZhiI=m8XqQ<lD&n0=m!GomM!Wh7aofodW&45$R!jDTaKXGnSOzO*0AmLJo7KJ^C31qDOVj0!C{A03EP&p-i}c95-nAAK~t|^cL0<tl%18V(;s{vv3nO)m7*FJRbnS>f)iItD|&sd?95eF9OCwhnfy4-dz7ZFd!zBTBMd@=s(|UyGL?~Sh3uW`0R8bdd`fEG13cGOr$q`4PXnp=V$lIx;p~Hgra;|Kf4|%7(wupNj(6C_kPO1%)z6BzN3mu}CIn+}H%vRWqpeUVb6;5?Ibt8Au6f1Fhu({WySsVLd^q763!%CkwSgw`9`yrw8WgoM_2r@`7lo^n^fDgjKFPX#d_=()4{3FO!BkH;hiEiX#zhFH<!@wa>W7s4E=9E?Gx(PTEe*dvr2mrV@%iJ5o}9hk(Q>t3EGvt#Q6_qP$l1)~P29~n;*Rr7-Npax*l9=cp&**Oj`A7s(@E~9u12S#A3-Y-ZV;MVs_o1wv*Mh<Ja-Y2yJU1HbxAe%ZPZxW4)G5KiQ4ZlE@wf1oPe5xe(Z2Bm2K>3=4zE7htl8y;g*Fe*<r}6nfeh}<yG>!iA?nukV^aAcr@^2R-lTj4{81A4Mq(xgWQCJ1WQOFPr!4nIz>N&Hij95{&IhO?W*te7Sl;pq>r{gj_<=%$qK&%GNAd<=vlzhj1iWuF?A*_R89@aIo;nN=B?(fFZltg;n$bS=wdWR2MYcjp)~=c(C@5YMkwtxcNM;u36HXhIY0HDq>6-#WtniP(HDrH3wI%SoU)FAZV+fXs#ep_MdfUl^XTaR(ESF42@lBG*Mv+&Fh_q&eD%{5Ea|Zbm=K9mw5)Zc*FZO#$7v4wN5r?{?Mk^vp>b?`7yfgGy?F3{5G9CvJ&HtnsRq17Yjn2_+nIa$Qn!r{+gCgBtFN^7#X}hu2(KQT@gtUe;K7a<-1ohO^S`Tj_4Y{%1pc;%8^PPeYZeyin4CepfB8Ju2as9Z3_4WCo12joo@8#F^C#&<o4K&RVL9d5uSs3`SVo#C-y?LzY9LSY4M)#lq5_rn;7bhpBclrj0*Zt$nXFHygE^+y$ajR~1S;_ATM_TCZZw<%2Gf1^Q*Vi#y!5;U$~B7;6Fx-Um5OH+cjq@&hVL(=tNc@%cdq7R9UreUaf9H!B=!MYhMB;CkPUD9$Gx7uaegq@?o?l`{vdzjFd9#SJ7+g%D|x_cf>e6vmCBV$6*o=MrmA^l=zy2(S)2qO2#;KReTm#alOa$$;Oh|oCo#O=to;eZJwD--hBIi4YFE6q0Z1U7cm!;tm+T)72E7z~G5{KsaPdHLLm;!H+7M4_wy<V4H$pY7MA@NrPuDzO0Gx)JbvPJYxbP^ECjY5qv#H#tD)+$G=CM#|-SRDo^(6$1ZUB|M(P%bBc0S!N7Yn4`|K!A;=YD4vO*2>(esHU?9s${(SVm@MFDwYR$@Th5eeK5NavSyK%{P~ujlyy=P9>uJTx(cy4(J;Ov9;Qx2(RZ-#t&J9LX?s$zP`*bUXVz@fA%q=XFfI)7`x5+RtAEd|LdK1?!EI)GPCQ>LvVpy1?8e)vlO1>jvOYL<I{jQ+ILNZG2Q?JWX;hAa8xmF3D_=F`}u{v=?xA_WL^tIo<Nosf<77X!fG%&#=L>p0!pv&m)&jGu}FnyIMYCejz=Irb2CC=2tT<!B}|Kk9Y#gznniR(m?gP?<$$JTe;kcSM92RIJ}G=zJyJve3GRA%r*@@U-95`;Oh~w&^Je*vEAo$PaG&>jGvv;ph$^YsJfLZp*cjS7Rpjgvg(i+z=>!B9uhqDH5Sv7I3q)-sh^;<CRK++~NGm0WK7%=PyCi#)Z)rdQo5^W^d>D_r`%n((Yb?cfc!*>kc+pY!zSc~H>pN(Nl5cc7wS&Na(CtS`OV*ANfmbM$R1z%dFrwV{Rq{bwuoJC1bzSDxE+|>Bh2^Nkza;O<zq1KUS&H#q_RJ!3BpejEd!a$(HYN@#tiN582$P&*s&}m~8Rk_a@yl)5k@S&3UAVXwUg#RoWz|KuI*f1ty-i?tx<_#X*QMmcna7=7^HkmH0N_@{4NphL&xT>7!z#ZX&f8G-_)<iO%I-^1UF&9kqFP6F2(=Z9{UU-s^=3k=Q%=p5ePPPJrFto?cWOs+%a;Di)8~Op2+t~6DFqu=-Eq%U2Ix>|^(f7jn?2%_)fi+zh6y=3f{%z=g(dsUt0-LZPKpLols`W5T}~p>-F9*fT+&ZH)2vaKi_;ouv7K_>TxfCf-Hs2GoOgj(7&z_L?kM^?fdz{xF6nx+v57fKWj{e6zSUqN+-%&yaHVK%eJfg{$ff!UC*;cY%;Iek$=ipgE8061%PTuD$yS;(g^mfC(6k>|bp#X0ien28TiuZv)TSW*P{9>MKq?Oj=kHx$sL}=4sm;vo2HNkr+HTpMQk+1;sG9Qk7(oncE?=Ijp=zkxKX!PIM)#+?SK^<D+5$Do%R7!+G81!a6{!J#)E$hR;6~Bu^nKxBzHn#uk4LD{b$4fPfWU<^R69CFb){->qC(O*F0}eOj|g+X+oqN-R3N*fzYzm<M+$L0YE-;g8PMv!R-LoJDdjj%$&q-BgD&R=CJBA&pcC;jYTwL?W)%UtFNjrtSn0R=Q&cGzBU9;VTIuODm9Ews7;cbslQB7>@G$)vfnSv1)7D<X4OB#%GyIDNRfYv`^)>Hc?#dMH0{!QCv0BtB7p5$_E98KkrNuj0e!}eS0mpwgYK^1$>iiXB=Y;5gb0zQlUaNZ)F&RhO-s&n4YK4u1G`tzDudTcty}7(~dwqK=BKdvoR`jFB=6bZey0N{*v8eX>vE}FtfyD!H57;am`>-`4fHzZ>oSvJBevVSC^MON)nv5TZI%W82`g$S~<4R+z;TK=8{@S{ElQ}WoT;B`zwFrbmSw-g(k#2ac{m~$Se|kNyb<*ngaOfV9c2^_*+Q5UYd<60r>{6wA@{VBTN!tV$2!1>>=*%b`l}GEE@j07%nwoDcZ*0KOVH&oUR~u1tdjq!w>$?S6hM~5`Fb5z5c;kBmF@hpdbH3)q_YYgg2|8N$5K0>zV8*FxZ3gBd?!i=H_&Ex@UJJ*EA_T~lgV8~&-+h<vfy*im`7+6=WSBSP&1ScM0P_m9fS7pd?Zuvi;{YiZ8ozfK_n|w;%Xh~zQ#`EPQMW&UIRXKcxMdGBP?A28`R#rzmhKSo|1HrauFj(;kCkR>)|c<n2>&$N+<|Ya>KY#nFIHE#FXZHSLba1|(Z8?aeQ7a(LH7vKJmKl9-^3J~%^$9yhj6DytV?pdzaPi##5FC10jviuLapT~^q`z`Jq;QD8_fbjrVsYVdG?V=G+f|<X$E8FD?RhzYgvNoyX7|x*<KDAg&DF$(>W~}QxWKkQFgJ5By%tmXSd)^c#=VXr9c{^o!#PuCb=5g>=>~qW||fu3lVGtcYE339b+3xrJjC#rq~u3YBO)<33|*iHtF0^2#sa5$?S>69BgIpfh6v(2`QiD?;h=5aenIBK1B9Hr6A7xwqk-WxaJk9)rBg;52oA(JH8~B0^$fKKx&6*2&AV)Q3Hr^QX2WL<V^38QQ|9U8$-j@d#c>GhJN|N-I5d@c)tNq!NR_YEGUR(0)6_Nobps>pQ{@w`VX>?sJJd{ADbjRQEc!%m9OT*XYW7z>3cTfDwcRk_CG56DcTX=6GK3+MR@v%N`Ca7Fb@8l9HO8Q61_AQ?b(OVH8$KPQ8%$B7N0uS{PFo$@9{r;I+55uN1I7}gf0gA=>-Sv=5A@n-t=DfCKtE%41DvKxgTX#xTpnl%BG%ZU=|}dmmDS}K*ccjD4dciaL#sg{jP72U1}-1v#x!(R$m1wH1!&=sX=%=(?*Vx;~|H;7IiASctvy7?Q;~B(36v)1Eu_Gd77_y#nW%5KCs8*!BGo4Ip}n{=!vb*H>nOWgsREBAr0R9Ug#l6+Ne(^kkl(~a0pZ4@v0I3SaJE#5a+D1vDl*G;}9<mpP_S0UZ;x$A&tAAPd{!Vdp~cn_fX*)Yz%_O2+}Ll5<l3feg-#kz{3wxZk%JD@eB~}9e4>d+!_j+!LhSFNmJ$6!j6AK@v#J60i1w7pPpPAuQiH24x`+Q+iWMo!QqZXK21$%t}wE}ngI`^CcTt<4!k{RlDOMeDjM`;%jB~zGqw<q*b*{<HWxIFSf>}G7zjoB@jc<AV6(-t+WbR=N-%ShO3!^%3l35N)v@&sHvW@l71Q8xm35{v9$;dHlA+CZvPCR+TaaJIG}Wd{;^^kmot(NWE!W(_m20axiLCfC@4~vmxfQ8!pAV~9A6H&O39y9uj5q1ek?l_H;<J}H&Xg$prCD*XjsE6g?wPDR^5FTbyXqJGp&5SQhko3F(nwR}%V~ieW97^FQ&5rU3gj#s_j*L*=d?-@gow$P^p;?wPg)`kvvCzGX#1!ue4esHNnbI@>{PX&-da6v-RGyyN{<F@7$je3yZL4@V=b~?!q$Sg-9Ycg@zCisA}cmQ+59`}@A;S0c#pi*&#g=bQ-|4M=Y8Tpj|~0h`u66HMzm7D-dIUhmKt$E>6na8>o;#M-&ls@a%25AYO(T~(QX@^5m5bnaYid`E7|Wd)WLqFjA)$yfy8QkY(ONV?X~3}ZZ{(7#_BYmIef89&Ld?tjJ48iYS1+&h1huk5Kc$5tHG(@n2NcTR2eBfuIei*Z^N9_SFbPM+FsvoMqBmGTa7J0sAj!&Z5^KkYf1HiwD$tm^f9*{$eTUO!O+a3TG0_|vNmvhkSb+)8^Z;F$_!|?Ga4LuLQ_18WK1AM7b;O-slf<($WLEVBSFXL{lQ?=M%Risxddzoxst*hj(e;nR>&G7;B%OJ>3htnq0ddH*E-0$>-1w_aDlOdn;ZEcxQO8Q7+5UjTnUHdJ}Ppd!4_(}nT<9$WA>1gK9O1@+3avUA$Kt~NWapLy9bARgHhsjk8qscb*Ol26vwV3H4GCd6CX^*?qshM32?X1fpIXO_}@V;Y^7*r2TP)q0>@Q%Y|~$3gsfH`+bSqeNVx71iGQ^_7XIeuD|_+yUJU%y)oSqSg1kYnXy14}_k&>W2gNrCbV5wY-S|{=laxoYPHn>SD!^IH`yAh0V1Q{~U<07v<I8$Q+Vg@j9gMndL)YBgsBZ!NFMlT-WIM~1`My%g5|$OS;1qgcAX(`z@(`k^25rub3bf||mo!2_GUYRG3tLUeMYbmEU0yv33<~2m$H#3QWXhAzg0$42e~>LE9~O32qjwRh0gUcu-sV&_4QiHaDAyZXZ#Eih-qk8$V7<DKi}z(fWI4wx>zexk6@~)~`$^NL<906K$J>PmxIO9owhN9gd3)*0o>KwFLAt-sY`~lv^pjzP6U%;R@10eIq9D=|JG{<S$IIsI=yXN*T|=JQrM0B5DHO`>(F4^6q>)nM9#Amlu%@k7fR2uW{dQ3tlI;?GblHxRJ4}3#LlSkDFs`lq+I*2i&=~I6P+qp^EHP@KPg){B-=S-cwlQCtyruE#%w!n>mU`6P5B<XyLEd7x7~Jk1&KZ5xl|e9Scaw1tb?=ANkqR8j(W%i3oYtN6CJxd&*3p5x;e5GZTKV6pmRzmvk6~r)oV}!cL`i!L@)yWny7ysTy4I)<97&vvRWG`@ykzm~-rs>{s=MXgoKM?xq3{*K>Egndu3dR$u2QYO`YJLY?RmSzoL(X?dnTOIkSNb+q;A}z!r~B=_)Gp&j4S;feH2^;1l`EX?i(Ef2N>=R2ECHj78&DV0$vN}D^>Aheg+=A9#$(0)!e+A^1!A}SShDi!{Mr*4!)5J*iQ>U9(@G~NgQTPPG^w55bn?PO25|&Dpk75D2@DPl37!8It`CO5e8g7LXecvpxch_9ziYbxd74D2mN-GQ((TZgObc6d&J&XW4~svAUZ#>{Lnj%G<~rwiOXJ;9U=wa!QfiI2I5aHfE|=k@C{g!7(t_CK|3rojZdRA-Q(u^TbO(FZO-v)sTU};JX%@5!9Gz0cY=cvPDR^mn~moB%ALlN(_P9L>y+->eBV~_p^MIZ-&^jdOQFLZa8?(**%Kmb7*dfS*VpYQNZd66b`*PQyl=bWqS?p8Q0s~5e?LKWkgI>P=7ldms5x)<I1wZBE3p@MH#D#<ZZ9{t);CerkS=I`)hhnXZcG2|?Dp=(qx+rLys7HJpmW&wQtCF+lW1L<9%|B6DQ@kE0^P|aEv;G3#?)?mc!J@>8|zCAKX4sV?ci-TV40d*o7?17&Ms+o!%m<^+TBcP0}rXutfn8KBNlkC&EGu=JVE+E=t0j!a^$1S1t_4a;#=_m`OLNnA%d|^6#gkfP2;wyHDSw@rZQ2G0a9l|YytMpaOC53_oX-DQ;JLp?w}_#|G9NK<iA*uMI;Q0DylvAv`gLF@)~+hLIHFc6eSt(s)<TB2Eo*&u30q~9s~T-#jkpl7b}lpuNeD#{`lEnKl_*<aJ}=dotRDEQvCa~zoje`949MsreiRD%@c4+pOLr8V?xiSJdEsF@iUH(MF^UwBFDy0&!5bsU?ml&OOLV1Y*(I>SLSJixR7v|$6@A^R0K$mI2`A^Yh4BTh+J`WgvygzMU6-1=}zX!>k_dhd$DB7x+D`UbKFmeK7^Tp>a<TiRD)*Sd8U~H2_~DQ%;-RFx-$O`l-gdCUC_z;`WsuT>(Pzd^|f1#X)=}-(tBgzfB9XG+;v*Xd3PDXr*h71<E<Mj+n7_@J#;bX9NvdSMSXUWVMF4X`f?bQ4%r8EiB<aOTxVt?m$x;{rNIg6*uAdKq#O`AMQi##I@yiUcUP&CNkxqG6V0`!(5w}7k>_<WgwA1HE~^E|^4f3FrDd>NuE^dAKqh$}iX6eup8ekXQux3biWHnJrv3$~>q6-8KP3DT`T+I)FiSiSu~yh5PeFm3q6Rb88r@~MVb%Nc^CF0}7JhK8^UAfo3n7SHt6-|6_sNe<0uPHsWltC{k27*3sV65nOtAZe6SF+=QVa}2NqPU-2O1>E-!pWFRh7pV1)}WbVfHED#3$%VOwc3Ws)C5oC(soMk%O)&1+tuiY#{jZ6Orxq8<Fyi#z+aCJdy5}kO6T-j-*T4)g%#^p9na33F#p48o6&D(Ixko9}~{4n6OX3W5f@jw)+d!AH4b^h#w}7<6hjsBwU>snDLPb;Ao$*n~nO`_9n(;_~-w(4MS${6LV38TP6A=;*=GmHd@=pH*?be&-_-35n6cXSUss(c*8$qOf<Voj}(~4;{X9;$Om2m2~oDa4~0+kH+K4vpIeEQ(LN1Q>_(WTL$jQX^eB2Oj@$SxT!CEScE)ElsrVErpB$Cc4Ex-f&Btp!at7^ZALjSXQGwmTyz1;;rk&h%B$LYsc;)OO?C3Q{sQw0dRKC>JLvC!0LR=67r^7^sL)Ifn+p!hYJPl@|V!NMC9@WI3R4wJfPiC1rH81uYV3(MOQnl`A^-Gp^S1^RR`alg6#bOu{Q;gh4fX$-TrB+}X9#I`CHGsc-?h)7wVW#fP#!?uB`ugpftKsa9rmrzmF#|(!YqP#_8v)iqGBPA5iDia?6YmpPFwvk#6hDEmjpv^{`_Dw*kg|Phk<Z@|#Fa>O&ltGrh4qk=pL~TEElPkG|0B{4FvRg=PIvDntB>I3{Vx({*8~ZD$d}_!iE4hMXR>FEEqo#BmiS1z+on)$5{g3ZsY#Ep`{>Y&JK!54r4wQiPHSvO|KSqc$XQZ;{`{vaJ=H9xs4CTga!6RI7mheOSnD*0B^Py8-*rm$>TV{`FlHZcbfW6>8RK20yX*@m1{<fZ^aSGGm+BUNf`D0HO8$&mQb@9hchf~F2K5^l&UvrZyKCG8G_@=1>u+pp#2|mZQ`nZjyvU;1G(r(CxFz30sIJ8&kopLjDi1QE$RTIcx~HQ|Z^1rlqN81)NDRtjsaMSyp(fGZgrXH13<7^VYW2}KXH1M{p}rkO&m!P@&rI$}&wSRcuYnj?&h&l9!D7hE6<$}gF?@ITs%krjdvJ7MJor;IFPDL|jkp*J4jY@+R;lb*bg02BCTn@UT7KF2LT+KpWZ9;HYT&auD-ukYuPHHIGTjPGj+sbtRs!|%2<Qk%9%K>8`H10QY)I%(JL)?-77Ljjh=Uc?KdITgwnO2ZskD|mt+t_Q{9X6f#fMMV<-+y0`>n`WTcB|a5R4;#H>KblN5Ic0bPn-xwu57;uCPmab?!_BbLE;8*B*23M_7lP>5zXoy}u`8Z#h%y7);+kWz`FmW||Zk$S6JJ2*ZpTCIxc^_E1-L057<R{NMj$C^L}paw;<{R&vJt=`f8n=pQuNv=T4AUz<3@e|B}<sV(@1#%v_is(JopBHh4`RU+OlQQ*H>-&}KpH;a8`uDu_~g)CAw0^9tLIGfa4nm3UoP$c~LC<Ra_-9<hg?YEK(ekzqtG7?pQ-fCg=%;i6&g(ha1vxSjb2_sImM$;@J3ei|&pIN)OlZzz18|Fbt=Kw4JlP)qSHr}ytG}KQh>(fKkT`N&KLe)!E&@U8HJxVfZnluDkWnAJ^|7Hq0{LqG@SgJ0dGynQ#sOOF;DIbN<hZ4S(1*3&NM(WFKCKWSvH(3+mgI47LYW4-WhDq*sJ8<y*U}u;7aO?=KX}-(x7R&OBGK;n0ecN(~;1|kHjsHu2(hE~JTP;g@U?!I(+e6@)gVcXhG~*<lfbPkfeoB~8y5}bRX}z7MW(C^-15OEc@rN!y)#5MMiIu{xhkVrFQecMCf)NNjpHnrYQQ{mBul+=#N8~`hp<r});|lQy3ZcHGOdWXsp1@K2l|P}=DkW@vWd&B-UHHVvR}32?O&Ob?F80KgwyBk$tx0@Sn7!u2&Rvsw-N5z6dWFxC(EOBU=>w^}_%@{%wx0fLiz+?k+;d22pOOxQl@h<UBwNhBK{xPYs9H;-@rbMkA1brBmiA7NN~JHx0)ON1?Kufgc1|k#88ae1?dX4PIPaOH@jc0i?6;DbnRpEqAW&+OiC(NH<(etxN)5t^_03fLz~uBdbGjA+oln+Rmv6`rty}f2#!{}SQ|cy+jdt13;Tbgj!v0`1LTDs~KBUlfIR%l<9%GlH&H9^~BHx@JhfW&j(kxVrE2*V{fF=e<-Tf$~BD!;t?94@y%+PiFtyI6cIFiR1P0pe-IgCoz#VGj1dOrJ=7kX=TX+L9=+7qhlTjfe3tvw>$nP|0QO)@bsxr!9AJ>?A8O71w6Z=4%EcA5a<igwj~w@QK-Drafvf)XJ~N?GZMQnhs9G9UT5w_uJ0mF<|FRL8nzCuO}66PkX-(Fw5~!l_ZR%l>=*<=8^T(F<fU>l*ncq|*3_+RSFExAVs}=8yl1%*RheMhW6kSu`T+@DsfzN-g>gpNlWa)cgo&_NS?V*do_`G@Z73p}mHVC^&0o?4(t_ahHQ8Phrla?nYE^Dz&pqvs)pRA8@coC806NvaTuL<C7=7gD4xl(0^3!9mXS|ojZChyE3YKu3P~!HCGM>V<?vK{vXA7oZ>c&3q5s=di(gD`}z@QTYg>sjxG_9L6_H315*x2W2)gk|1W03Geo>H`gJB#e>|dHD7Rn+3QfPKW#%>bH-m+h-z9ZbDc2lzMZsx-fgaJll|p7{(c5QJIe5uIvE^ajVR;~esypq9D~&>=m)Dk-Z`8NeH>WX0#IF57Km&#Q0WMM;q+vi4$HcUw@rt{wXmqooI)PYb-L)|_T1~%P*>0kE7gy*Wc5K@5V5psNi!9_=tST6?#D_g8wUT{UE4YoMT9y{;O4$V|E+%?Mr?MEoL}m@>i7VPgyLaQA3lp}$adBkCfZcp8O{*6%zjG=$8|ul_yv={3gZLR4oRG&RmnBZ(^Ke$PHGQt&3yEd^8Vye(d7iFqLIbNuz+T7~A)oTKa3ey0qKcX6B6amA4AqDdIFuw9X&`hkaBzF=BoF`#Ap8fy0VnFU4nnl5z`Z`|z6)1AnON+A?H<``ghNiW0b=no`^rZvb%rWv;wIE8MZajQiuq0Lg<10<ReXDWbNNT>YgmQohWWX8nA!9y(+=<ZG|fmZX!;;j-{hS4>~52)lYuzo2$hN_{CAYvGE>}cae{O?$iy3E`!vuAkFlhecbt&}I!%(@{Rf#YsDKoGs{7|<=u>8B$K!C|oJV-0;5@{L;Zbg4iEI))Rf*7UYn5o#&Kx?v==PB(5%Cbm<3XombRn(J@vU~6+~3S}dXqnk=EaBB^O-rv{6R6>^pYOanuJlYuTrcz9PCWo3)+!)*mxYaifl++W>NyZe++}oMAy~H`Ts?|gf+<ab3Uh|av@kw!kSR{vAdnw9PYH01tL$jo%tym!tM0DP3>TF+8QZ@BYe(#OBr<YPqHN63ddSM9ae8IJ5PvM3TFk{fxe8l0mI=qFB#X_<GTUsauEZ;1)OyI==ltVfL(07wd3ytvyIW2h~we-Oq7rAwMGKr)8A}tZLT*rFv-Q827{$*p7wW^*S6}2qK=V*h)%GK;eW0U0UZ}=Nq>8--fS-4TFaCa{9FlV$36G2(ahzLI-vi(UdbiAm^Z@5a3cUmfjo)ReflY>X&&;O@u?nFNL+?rc4qm4j=A`x^`!_MI#xnA$CB@o`due>N;CLcC1k*zB@0bwVa+sXYK(9><qSBvbotUm;ZmO%UC8J&RIzN&@>LsYi-=7$7!lak#=4(Tfc%y$0iVh2rsq#)Qy^cdd_#-a#{yGX$)BGIpFYcRo^5yRIsHpC7vN9gZYy>UV)DR~n?d(3nY1Z(^BhM+u_KTR@-zF0LfUMStB7lxrD>4!l6jN63Vb^4K%DmbJU1AvZ1Irvk~};rZ0+pnMgu4ODCEV-OTALaK-kxBw|^AU41FiHDMF)8h43VSUUMSq3`1jYo}4RIyM(7~6GoQTq?3qTv|XW}kWK~VT3$zFi7A~}TwZ0n(-=K7|KrTx(-6>$jvXk|f8|8Enzf6J137^qD0j<v3yJ@(W6SF<4h$nt)`h&FpWm@YM%3In##`Du0$v7=R>iqYscOM>Hw(i7y{`<rCxfwo7n!II*$w1@vGpQTtr)9#!i)~3kOtuYIQidQ3JknU`rV4?2#)x!l$%SF`tMQ*VV&*nNaR<cT6h<k+sDsCUFn4J1edms|3EcxX7dfj0?_ma;nJlCcdeFC(-b}Ep7}J!ZgAJcn4!F4XJ<JU^t|)kAu^LQq^#g_v6FmJ*(2pF1ZyvO8HwwNs6KnZK-!Q>J-FrKcBrx)wL7G()qf8M7(9v?-joUnmA9mt3w$0Lsc@`9WP8vmQ8v(KJnF_tsf`>`XlA!G$6@7PGBq1&DYVC|>=Xl1pvj1PGHn}KRfKBCdNoyBsmc@M@Ajb(W@pk(y0ZGI)t<ixqKGu|%NnLO|9^o`{O$SQdBWz(IUj^?AO%C3kss2H3tyw71gAp(mP0?%U=98nIm};T4s2Gi3LEcH_h2ON5RooPqr2<dTQ^tM-%OdV$?X3{_{%B$lvM*p-rNWp85gCMRF5yler85)^Yx4Bxqd1sM^s@?$!|X5^yso-*S><n9o#swGT2evU@+Z$=I=UDPoexM2RD_#SUBE`aymPip-iYh>q}q^(7w@(mzH>_zsi>p3+r=l5*Gpg?zT!wg2bLN=#bun>4*_x3-jif;Ituul`{qoxQTfpf&;uPWVzI2CZ@^7ZXD%o$th53B}n#eU@WY9k?$RKl*jlSX-mp~<d!oQSK4>;6IYPLm*cDly;6%H^c(4-9J#+C#AsvbWPGxE7#z7GZJo@u`o=Xeqk|A_#?7tGw|CV11d~>0*rKeBv<JA5JmN}8?H*a7{Hik#E;cWnk#t6x$r~N8FzM{S!INA_UF<rSqXY9yss)+aCQ9m^pOHW4-$*fw9MswFi5`#2e}U|qmQ(h#4;N*up6M+11d9KP{c67AFar4WXS#V@`qOA}@lTL@K+~4)r@o270jm-GdvkfEF^P7F;_M3Wz)!Oc2(d<(7JwUL7DIht?Z@FT6mNE9kqWP9h^YI1`DOLgXrfq8548@586Ib9FRI~i#~zUm&k9wWZ(XhLKq)=<d`4XVoy*5bxe<RZ5kWHU4!EA5YP*HK1fSAZS~$b3G?CXl-M0|MdUoI!T1gB9Wgo=Pcn4{|vg*~R!lm|U>g}BChBbkVWOj<3T&2J*Yd9?S%c`K~RjDk=QR?_osK2{wI$C8sVni+M854+xcQanMz2i1WYY6Ls=VLf5&)~m<yX7o}mZfXO?o(IYk-+k#k{^`=kY9@pmJOX*86i_Q?X1@nk|KjtNI8Tk7b&?-@^$`rQN+XRAv(g9!|i$c8GCqtMQ(v2<lH$D<#0uCD6ONTc$VC;XVKI%yWd0KD8%>25J-?Y;6`dWovoN42-W9Zyg1|r(mgn*Np1}%_rJ5d3KQ0c#!-C2uI%vFANcS-QcmawQFfkei2(2v4jzYyQ5%>x@RvCGq+e6sGVn_GZTl&&D+Bo8J@J?-{x#XNzNY&GSDS1XtQ-8C@lk*xd~IN);C~KAL;bYvJl9E{GJ4gN{6My|%?~LDhLWF7Ut9v8W-^3P<T`=DJDuH!t7efOv5k~G&Z7g)zvtVTNYL`RTwmFGJ9W4tTj$35#`X$ZX*pTRqJ6DG8!lc$Y;ROBce<Dq+WKxnfZX4o|F`qs&&;8AfKHT*{(3~a;ScG>@>uB_1Z~It@^cDk{7TyrzgDqyS%32<9%&vv6;(fa_OWn}`NFxq5MPkJ%s)omMNKdx-x7F_aHWwEB9!e`YOb{%d1qr?QN5)`^Ty`#2F~#$9_3^>NEvpU@pv~hUgs!8iZ{GvDzzFY<RRWIW+B7~-gU9>d9(5Eiyha%46SoY{eIj&7C5LKj$cn7h|l~~L=xBs`V~DTNzjObxP5i9K<?E62eU^CTH2-_D%jmsJz8|SeHg67N$&28<Ypyb3qB|yF$F6DvLvS5N?>=jyO{PnQYfccyk1yotOm0Zs=7AnTeqdpUo8)u7Pqe?8=M(%$QcH6dUe5cMR~g{(^>7ME9e-?qj!a%P#h0{al0H~>tHm9nq;GI@fpYFK9OdrPrdkrHJvjX93Z&pZGb$|N=)ka1j(TuVKyAWhw(i(gknDMS_hHWgj*|K%=s8<vg7AZYTo)1yQ@o$GC~vvOjt=@EZ&+*uObw~;fS!~Zn7(kQsvhe6<9SB_|UTErGjYvfhc1%7D#1fkUjLKE+H_It1WsmEGl()Pq+ciwwLAVq&3w;?U49D*=*ftjRt65t22&APRu+V)ln_mbK^SBUFo!5ZOvc37_Q^_-yska3F2RI;JmKI7yi!zpy{b}g*Gg<X|DIrNKSux{?r+y?V73c|4F4j@W{E9xcyH_v=j%&2%XO$LD?bX3Y8LT;t+8D0yzU_@S&@S#P>_d#ZLeMM60s`>%%D7T0U~&V@9CNH;3ek#~+{n1ELuRIVj4+`xvJeeku&)ARj=0ff<ynovsFg2hgf{jq26eMs<O@&QC<|A5p)5C3rR1BmXECR}4h0wW5UNb@n*{3SeK2ASlV~<3uC<e56k^`7V2cBv#mE=>zg)erSR#(t<;~@5?y;NdbxY>e*k3g(U|812lo0m;RF$BMFSge-f+6;v&b2sl7rV8ipLm+sz@vK(mBjK(k3b{`^k?@p<*t*4``oue=Dv=l4Cl9~b65&M5(8E=g*bYJk!%dzaer2zY|_CH5Okyw;fV5U3=>DDoqFvM&Q)jU082*8ZV4=y*^+_6(TS#5=?cz&(tVKc&_XRo5FHo+eaB^LE|4jO2etUKL*JxQ!vUd+@uGM`i^!DFJ<n1iOBD(Y_sTY_2z(A}*M`DAm!UeICdI5ujBR9%R5rKhlm~{+%b{b>U9g1VL;oLYb&ZNMF@?C0oHy<WB0$oXAIrAWcHsjoXbIZ{U-g_0{E-w<G8obc~BIwmS{R$Zd3eX>~nnENx?jZ@p~^9E;{#EwgW~WI|tv=Iq**s#o(~v2X3$(JBa{H<mZD1*l%`1u<uo6xU`ofn8go*{L_1jV*JaW*kkF{G=z7=wj%(aAxIvzlb|<QZ7pRn^&zo15`b*0e4FjkQ7k5qX148T;B0fAlS2TZlPKQHvcH}=Wkul7@NND30w^N+B<4dG69Uq_aYF>H@3H;h_eYC4tj9r7DZ;q<0O_}QqIUBag3J!V`zeT;D5`D=taJ{zP>f<izDpM`j}38VICQL%mC)kmgFcd`xSv&;c!N;urZ}_bhq6_5Qh=rZxJLwe7}n(hr2H04&s5(Xw#TyHB>(^7Hw$DE90Xfc}XJ(Nt7ISI^Fvv>LPlEq+MNva+TwLuiL*{lKI5kol$5e?Y)Y`GeL^9he!v0jfqLkFMv4F5iL5dywnfi=b1Y#wULPszZ@ST{Y%0*7(?de5cR(xPJ---In&wg6BaS-;0f`ksviWWi2aG6hoOJjnC_+P(bnolc~V6;);HevO8#Z|^fJuM*1Bia1G8Vg?0cn~>zk|fEl-fUz{5loz|0~fy7!~z)>7HkXPo`e>bThF&Xj*Em|FyTfCm~$zBqT~GGmKiU!2e)xJBd!#I)Iq!%jH-NkuRmH@iK4Y<}dV;8OFQEuOm12RN*N`q!z*3%SxiKBD|%ITp8i#cP}(S&$;iEK*33i?T+90*>4a=`096XI617vuxH>a+O!wJ062D>(O5Eq)hK6$45!oxSAJc)|PIr%)UbZc(4Lqi*8S8#E89d>poi7kCH`p=blxK{C|J_PfF+EF=8R!^1_#AtkWlUlxvxeawf!5e1HgQp<kW%%NBqnUJn94()b1zA4mdzQmgfVGZ^{hV9utwt!~@IkAb7sW8~J>P8LRKcM`wQ@!|gVb{=%kX6NR1fysgfK=SPdb91?@0Q*R{!#-wQEbWuG!an;%nw)1=#YJ<m&;D<{q#QT^G*3q%`_7T;ocN2&r=Xjbt~?&NI%DSCWi?e~KGZc$Q}j(}wP8?*X@!5Dj&e|{lDHSMb}RG=(8;1R{`MN_uqj8zO5;vr#YeiMWfS$45<MP>R9;jOT=19S(PbdZMsc%!I#>~57KHD-gc&}apiESwu6ER`EC<sd>y)7%&{G^~XiO+srY*QIV<l~<CAwL|j+NK;otGY*j?T=%rrJqoM!W~<?4RLhN=2!XvIVkgSWCNZ!)ZBFx5Uw^)vhn!T3*{S>KD4woR3h)vb?5^3;Sm_PUv3!NG7|=;B#fK5ygMV46^B<mYd$%`j)o_1P66oYD1jawJHFtxnMrHl7YbD)m8dXUM7nTa*vSCsyRS7b@mk6S7{Eyn`BD^ymmqdD=XMozYQXZxKzuPcM+kZG+$Os?W+b)n{7uEXi~edBDl2{W@^M904pP<GDKRUN78|c-G4Q(BDL|@eJAm2qz%jga1ur+xyF8NSRp{U)EG@G)?i^mU_udM@!ERCFmVx0a>fk%;^0&;J-Ss~G%hAAx>;Xd+1}IvvSGe}t~rpW7?28A37+oM=B`wyaCFoy(y;ehT}1e3b11sFPMM4>dMFr>=*^k$foBB49`*vm0Xco0{!B|x00-0gbXqHJ%(IoyF*Qu!{?~6}=5VZ>L)ut@-PAaW9ER?C#jC^Q&CPj@>ijH-`5v{F)R>X-_D;Rw@gAW9;J)G<j)+onA7(}aWMO2)7?mj9{t#F+ksv81)Rq`09agJ%`kegM)r<36S8KHf@@ZLaE9I?nU@8gQa%a~8+mM5Ii8bbM3`|zj%TGZ;(o_NoMZfV|4R!fx8DTBN$V?h_3zu@@m6qE6U}rza6=ZcqyN8Nyp3J@TqW1tA=<@G@g$eh7rO9#<1T*~^?Pr=Z2U(cjhdV<+^c%y^zY{j<&F1qH{Qq^s)zjQ+#4U2%87`uodsD%PX67z-SDAo`PQ1hmUQw`zfeiLxQZyP!E9xctt$q##C(nppFG@Ig3g#8{#TEs(R%|LNexpKZ+@NmCvYSY>EJ=5XZuj7(JZu3UyXdvMt%LpmNGDz*z1A=xt<wO=Hb~z`m@N;_pGK=OTPyV<33Dj)6(0_64_&-8Q)Z+YFtx7p%$d4Y*6j;7pso5=)384#Xqp5e8I{hfKo%RD`m7*nPT#oXN#Xaw4YPBts?<(X5qCl^M=p!vPuUoHE2=NcyOq@ak<T}WoR#_9WJ1p*5OX@6F7@5<NOZu}xUre@VjAiCNRIdDZaN;N0S0;>b6orT<ZB5Z%P~}7Pn62##gh=GhHgx%9#(Iyo)bW5+6JZ|I_$Mh^(s~jlPtyw>@Idato8E`srrVO#;=`ll1ULwz^TCdVR%AZ1oWJ}oZ+Y1D+QI05ypd3=hl2e-dI{%CVY-Jm$z<<O_y`xX`yKmqmxAT&OGntT$QKweP3e6tple~8Aqkzs)lSE(`*4@t=x&=iNLl#K+-TmPNxOq^GdA6hqd-7=!R$%rp-i6+l-`G<>%zz7N+LkjBh}G*!|A<H#!92jxYSSY!9p5%R4`z!}Al?Gx7Y;MRVLfjc}595I>O|pLO^sP#MZsemv;*OFk-e?xH%!PeYff3wBy;T{ZFmD>(}?tecpkJVgZlZ?&<sTwg0G{*tyn`dhY-G0AF*G+y>>7cW_8x{8M}bOox0E!rEL0)y|UTE2@!;i)QaiO|%f)oYDTqYmug=*V6FJt*g@vXF~aTq)1;)Qo96=)3vRz*jRT1V_b~Dt>gnI)NcItfqew{><mtBL1Wwh@axr08NuymUpMcdPX|M%R}(w<;HUhr+CnTPv@)iS7+wtXRaWtmdeV5PIPZDx*Of?5AFd;Cqt@>`%5XPoMG4wN8R3OI>GHp3E}a7<SWHCk)7PCcEjOwa@QAKJ2kW45id;P3=6(dG4;q&D0~OJu*v`KAIAH4g%Uvbt01)2R?cM@SFc8Kbs<6(j$8v52W-shRQzwF;^~Q?RknCwSmF^nGS-CN=!YvFZuJ1|PFJtY-6j8x1WEITZLTJw9iIl=II9(~;7-Y3s$vL9^;)gAglYzKFn|-j7mupdS}hs22oylIo#QHN{jcPcufb;HmZ%zk%73)#^<>#(_JAorO~aut(iVC@T%a-bdDPq5war+e+$0#`QF7^J^w}&2Fu!HT-9F)7x4t+8L8c$>kOk1Y&zHAoniAiez~kt_tdB}OgI<k$lpiTIBH6Cq8IQ~T0o=wIEf4gc)sKPSgD-$#z#4jk{(<}(F2UqDQK8~e6T_Jp4_p0%SbvOXQa#Qzy9YOW8*fN15dkwB$-r7>QSRrkC};|!>2jgA;n~v@^tA#m8?e^^USwr-4~g${1Q*DYBuVt(iWaw>P|pgDt<ZP3*&)gqWDFZ14Cy@8bT=L9ivzr-r#?}KrDnMP<bIgMwXCyGPHCNe=d%o)Tzq=jkEDZhKwnPBs;e_fTYvHid7;Bk^Mc734EJ66Rg_7m19MSc^i)lm_h#pHs_r!EwJUVuOq(7)2~xdhyE0kxX|BoY@>)rJDtrhG(;7bu^kVuvnT7Q?dClxqG87`e5|{4PWi^{vX+K?1ikL8~%rdqQ48yaz#^tlJ*S7MbuB|mTz5lqryjE0=u@kVCJ^Og9ujMXvC<dw%U;VvR6<V(gc15V(?PVZ+^>SvdxtixJbMJDry0Cwh%e@L2{dZn+xJzliRM}dQFCkc>^dmZC>?wqV_ZaxUkw?_O=3SW!u3la6ma21JR-Mmv?PPDdBm(k-l0xVz;_qhC@l2l(1eH^nQRQ~zg_<h6M=Pndj4o`vRll(nt-rCgDl{x21NZ7ieN%v|qHuGgzPhp9v}8fkb(LGlix`sa5<48F8GHl&43TLBAL0O2D~cYPMhv-F-eVfhoAs49vXa3>OYT*dC07><x_yj=8yn<tMp+ECkQrJMEr&yrv2RQ<xu&P?v}&`P6|bS`!n%F9$zh1c`xNuDgkD-oF@zrE`dI$>8yw6Ze`C5s1xOFXvYx~qhqd+NM3*I?N}(W_=Kk0Fpzie8=(S6$j*SreoVMoj9n)u{Ne(q#M#67-mkv*dnCT>eUkaGxQb1PD7EH(Sc)-5)`>mtlaU%GJc%+RKRtp~VL%$>fR(?hxT|YGux!%L*8F>85o%6@==Cj@RIDLT#?fJud7+3M^uOvpt`Pc7pyorEN{3%D+U@p1bFqX=tY}BmoP4$xu^ODE9T)<+hZ(_@Oi(4hfdP}?L1&qDcr<CT){b?p&VMfsiTuAF)l<{Ig-e`&>hUDjT)J;{3W+nBvrmv)RKc^B)X*j7KQ`M>0RHQ4^%M~O)oUmx+z%$t^aXQK-xkE;XH=pj>g->b6<=ELx@2=o#urPt&(e&@%6Z-essMSqk?+(sfV>B9!O1_egB9&N(A|IMJ>T7go+Pq6AwcNYZq$x2%ed%8NXb`2EcST+OG&rG1U;2ITI&wcCQi5O!Gf9bplH=ak8}xgp6>k|+l^h(-?6*iRCnlc*6MndYF$w&Y&|3iko`H?dO8ErdjgoQYUp0$JG1Fw8b9tkyRG?0$bl%13@eb!`hv|nk)DNA{&gvcVRVR@UFRb!Tb=M6SMtysY60N{F#~{_EdXv1@a_o%m&>FnD3(7l#f{e-Vi=FfHv||^Y9)4>Qm)FB8G_TK_JAJ4lDtz)F#req1XE|9<{EE&9a{U{djXTSYH~Cx_nO9Tj8(QZx!RBbH)|6`^jVki4FiV^DRbUo|zLAd-4&w}-SFcf>ZOmnOzuY-cAR=1!uU(-E(#Y1*wn*q;Vu(1|{FKkMD3gv$mfN8}9rK=DFxd$)HnDKcH<mZhVHCzF8G09sJ|&00035!l=f^Ea3!=HXAQO==Tu>3?UNE80$GYD>1}1g}&oegQp&X0q9gGEKDI!1z7@>TG8)7s#5=L;?YN<OSsw8#ZJneC+LovmI7vBs5T7aZ4kNuSdtbpOKNl_cD?MoK<ea0Ke7k-+HZcqWscH#@y08yJE)xul82-8E1oG43dg3rlmC3h0ans(h5uC!S!7h@2Zv9Sgs&;O7KA_(#7A!9FnK(X>4P)f8kS-c8(765;b8C5FHE1v=aJmxGCPbu8`YsPmYS>%i71v&V^{41@!cz^zaAO{~;FrL={7?J{w!0t$q<Y~AI^Ro-{6JQK5bsS;Re?bx9pQh31su>v~F44urL?Fi2uZ%3BjJ*7{npnyf&5(Eu-+{HLh-A1;XE;2bVv@BSq>Q!2h>2<N{Zp!gLzurg|Moi$La)xXuI^uZ<p(bogbt*+y5QZp>ix^-hKn6qK4wG!oCh802_HBmC3_`WEHk!k3l3R~9yoCBTL~N!G(ja8qK2{SOe#E}w{fo{<Zvm36ItuU2bd^lgwFD$evt4Tm3-h8E42L1)tW~jYM*Nm1_Y&nvbBM>A$N(HjP?udAble;wR7HsCPIdI%lYBwcdkaJ_a?=jr%xN3o9mlqKs6fqqHYjben%X7o%Iz$r@_b!t&7Yc<fv6%WW7lbmZ@X~Tt>rkCD})*CTf9zbdS7q6YJ88#@(Y>#Htx)NW<J`phgK_gfu&WGTTMDacYs+BCeM5avE#E-=;$lBF_$n$350E1u49f#5<p^OK1a9viLhMJ)o%&(N9uOLLqPTLIjARC|j$E*A_<>o!B*sQHx52aL;v$S4Op5z^>KwvWia6ywnvk-`&a;%)^BrL?>sO)&#d9NWGloBJ)Xedlgj$4U{t0JpA;vs<(xI&v}g%cwY4yYfD;n1h;q+PhVeHSl_$xXfS<Uq;NnDL{SgXg#$GnXjSLIXKF^B5lxd&3U!Z;k0NtdX1}SwrG3-7Kg~DwJGatrj{EoG?!h5?wZZ1-qr=ez-?`@2(o*9NLJX$0w7ba=crc1*w=0+&IAQ91fP$Q9zGrh8x3kEFe{gbDHp}bg&vvBr^C0Ths`KqL1un1{JwNn8g!l!~Gyn$C)CdNXVw2(S+<`%y!DYDXrCGiL$=wi6r$GPth8VFYa79NH!VIJz8Biwo<@91HMcoi&cDfw?!$Jt>f&L6bt#o;+fW#?uK2duwM}CJlg?gX4g)7%qS@okAoviw?D}PS-1!pzS3&D`7Ou{fkKXIclL@xxxK+TOptNTLrRR`YTMx%t(Oe5aqpjr+J^<qL@6A4g;RMF*Ba-ob8jZg>nJyOF(wgYBaXjJQoSP<5#cKjQHG32Pa2UN+K_q`DQZ0)@Bl2*w(FYTVWh9Bwh5E0=+6k<O_S?~u62+FWrF2BVHV<LqsHvDSss2{s3={1!Tu3G7%a-RF3Hchp39s?iw0?tYb)!`4shUDv1J{8w|k#}nn?VSqjdr|ouR-PQTQr9RP6|ys|<Zs@tBTq!j5~co=LKV4<hT=Po&H62vD2%|$Y5WKTkX{t`x(D68F4@p){$}Hc+sm+>qpkYpEm#(rkf<4foGjs`C9O}Rfr$zudpS+8SLF0Zot{RUTY>rP+>H?9CKmdm$Z!>fQH)#N9?Tq_B2pvGzY1IXU+d1JP>+l3xy<&;4;Fcj-Bw>-iRcusZIOk=wJ&@_$rb@;aGb<#GkPjJx9$M6s~CT_FmLb{(g#=8oab<BR3j?DV7CbKy%J<(kkcpLUWJN2S8xb=4G`$N>={tqg?YT#Ni$|wpGmhuX=v*dRwm{crgy!MkZ>Mc%CVGBrOJ)IKn)@KFpB*kA#EtEPijl;reV+TmUFb0P4<PHc^&;7;(pRa-6h`Jt<nA=+NLTDJxB1GXohDt(~g@akLCSdt9v9s>WJ`2$$v1?1%8|^jKxc`f-_0Y%nsRtG90teIBtu0*@{z!)xqSk2X2Oe9S?@xeN04BokQKVQUJDr_hH9>T1%zWuFgZThC~obEe8VaD=j=XE5FG&!B(%JC1GRTTVD!1Gn<S80EBA#@fZURWOehiS7xudYP*V&rRy-O{VsZ`^bZ100WNb*66d)aCtTb7ETt02*0q4%H!|*qLnB&zIZ#DF=Jy9jvX%u`4cDLs>Cn#JG66%baaU4sWv{i5YHe}%$Wh{qS!f`Fzq529xEmfcs&ht^LZdow{6-b@{G3@P&s0QEc<Y<DjKcKr=5k|Y$#}fFy=MI0+N`g^f^Kf<PwipIt?j%z>}U&hi@;w2?O}`&6yVNC`Dx$M4#y&n3nV4kaQyp7)k&iAll?B@<#*u1kR^~n!Lq&GkfKf`kbrJTn(VU89JQ!NtiPjx+2OGaSQKmK>K3|u)2AXDNLYAE{u19Fvn@1qqi4E6vB6M2GoTVra<>mV5gn|MDBX=uX9*W`(f2M8=Z#9qA#|50)_CA?iXe(WOp@ax2G5qDM`B14D9AN5G<4#8wg<<UhnyH@%#Kd07*eD$-wY9^vdN1!Ch-poq!y?_8@)G1-TP$ZMKyBb?0~%54qJXrUEaJ&tPSRu&x)^EN#}8MShu6v$+-b8!e|82@~rfzhiZh~$Z_E#e*wfMhBc>qeTMFJWk|~0#=g3@r-5?5kfwd%%EFabug%9VrfI+PO65wWihA~ped9?jUp;MNl#@A!tUFIKd3a@xr-w4<xC!ISAj9aE_QcTHV^UtuPC^-3o}7P<7qX@M7U>ueSPPmG8G@vp<7@7r0$l!D<b`0(I_clPHlfwYZ1n+Wu?oCrzreo<28W=7e<_kw{mli`{;$*&E;Wx*qQQU2*=l&woRdr-@j~UFV|^;n`M)AJVtspQdCTBYqLWu-s*do%NIw^~5ox=TWH|(hGqVZ>4OCExz#CXzy90dT`sUkK70f?9(5M45H|M`~Az!m??h#=@>j2ju)o`NTr{@Bxo9vE^^^K}8Hn2V6WfZ{B!Y?8)8b_At{2ve|n#_~G3K+xsn`a-<)Kl&#c=pM&U!w!1fRN6>eDe{hnc&$e=y!G<=>?64@wp%KH_qfj%#*S8FCe~*K6uQa<=Eu{2yER<NIlnVifCE-loBC55vh=9hZ#*@rNtp9(<c%H8UFqZ0h50(REv}=g=d?3^dSP?5pMcJD`yMmQ=)D@KL5gM3i-S9zlWI-ib~f~cmiY`>V8Ux^Mhx<s<Exwn)LkaLXR1lTR^^J4OT~xO(aZw7wEGCZ~db{n!mLr<m5^cRG!|y|Lg;e#Xp-~hHx6&v_s9eC`kAZ&dQ5XH#82b{?e)!C9*IY%CJB~iY^Oz(W$ku>fw%|?n2dOi`#gs+maD9lyGd+WHr6khL!lkq<e~i&2ay=E<`3y8XONZ3Xv4|Lb!L=+-k|Ji~;P;{K7R+A^#2asr;PswZ#B++c4e6Cz*;ng8}+*Q2VJegc;f{&$(V?aMv$8x`!}1Jf(8)s7i1%IKGey9Eq|=`JQUpa!me&3?+$dCtdbRH=*iEuAUmYB(bTqXXUJFNfa_4yOu~Wq9}Nxq@e{`6GK^37nR~pwXLze#>q$<n~fz`?Mih(RS0O4r3RE=U0!P}McZpz%PX9q1cbLOhnQ*#K^Xeo6&NXz&TL5ZfaR^X2{C0IHWVpANgcYqw!Fo@pXAtPo!GMi#=%rJnJ?XJ`72WUm3CZ4V6Y0zswIyzFH~09J&`&q5Pz~#m0h}G9YF8p4Bc>@FYjfCz?LC6*(2qs-hC#@_;o3xVq;^+CQ4{W(t#WqOJq{uPWcG=kzeVovdLfwCYb|8ehHWJS1OGWrMyUOgXFaI#jMFp(IaT;@v~pf*sQGY5Tuaf)v{`^AR#@dbjK=kdkQkC-pqUZX_|W$Z_o@wwP>;V#3PEUoYTq5bR-XPnm(kKFuxqsiMRRJM(QThaSge;57UGn?u1Kv)*gfTf9}jZarH|4YUkRO3pusDQsF$km`p{cq>?Pn&jnMG<&njFeu8We4>Hi^^ZbeC-S*#Yo)Y*l=}|^j{A7$U47i?xfkl?<G4ku5!YnGUrLTx}7CueP(-C+0SBfF~g4Yj=JPN$cZu<agz0n$99}E)Ak+re9Vo7a}us56>NF@4E`Ff}q-%6I(++Uc^=cgd#eayLnnE%yehKje<Xl{w)gFN~tv{@c9+mA=+6L#7Ym45Ucd(!=2e}8}PYPIts^1Use@=deUXfEGEGsyWrIXA##o-pAZ@QIXc_(-xBs1T26d1mG6m~N!(!}l-b>4<59zoBeNLIil9lm?t$^dWJ_KO>p%OHR8OD0(I&b<wz3YUK*|NQeR=7ZE-(JWDlEz46vYV{>^GjX)bK^)*xMJYh!^zlK*1@V00q56JGwk8s`s|8OwszAH=JUf*2)5tP!a5f|P5;OGc$x|*Q*H`Z5I;hs}gG(@tC_IiUoWF+zLZoe}iZqH=(eBoyp^;!qqrpOBxZERm(Sr$&}D|OE6*)%>SpR4r4mHKs9_sO3r_u+_4AHb&V_}cOINx|Dgizhk$Ae?c#bu>gYjS(=~_#SrzB7gN-$VK&u+uv`E_h^Fzeh(do4)!qp;&6D>i@KfE^$0Q14$w6jPeg+G3*&=5V7xos`|wut7lMzhvgX94X&gZS7!6JcDMotSvvT>kPw+@l4Bbs4!exulg-g8GYim^JB4AEY9lVbY2gEw0h6~3RA{Q_1D1WJOha#uQw_%(xo(IK(p0v6>a<kt%Wm>~&lomaV4;|gZYoCUW7AWc93cpC5JnAL`hS*7De5on8CUM+OqIP`3I{@9UnvKR%GZ1e&7`r95)$&?>MZDFk7=y5bK<>cH40_#tG`nMP_&$1A932h%UUED{z4sz@<FpUG#=uo!6vyO&G`qg*jGVv%BC;|FKTD0bcKm(B|HFA5TWhHyBd6V<`9z-3yX+z0d(R`Vqy&CS<P>i(9M6Cdec2;UbU*eeWIb@o+(;3k+x6z{5Kh3LPu%}uD>*D#RHLltoCp5>jmF!}a7Tue1U~EKIlr&S&l{LY#c745gQ_Gb>O(gMj*tXkT%u|^oIbB4Ws1PO93lkdcs%HIRNho`vh$ofG?;A=FDz$TEf6U1g|Hh)$p*86x^1$OACXEW&B^{-*5y2s7q#T{Qq}B}MUNCHU#qO##KGCRvxkzoCGLc4q}%yZ?=6{fdN6A3_w-C*7)TMVZLh+*0n@5E^U|;R)VG(H>IkG0@)*FRt0JP39yxQ&ijTC?_Qza;o{C(OC<fd=jqV=h)Tt7%Mkj+P=~9FYS#To5xtN|P&*Erg?e=UZ863fy7`PrVfiRa4X)4SCsE6tgwbR)nlGP|A&*df4k~bTQ-6g{Lv85gmk&<vnOG+Y_O{E&4&nhDJp6$?-dWP%U+21V-1X7Tz`llru#uZ44qX8IxK?Abp!irUe#`iU+uiJhFw!mndpg&}(B&Qwo{zTTS^Rse!ih<T}?xgB#-vO|$Z+nTZj^EVSn}e7{<PLvFhe<2V39vcsCr1VSTriHTjm_qi73nG|I_P~)&+WUjC1D2wT$H>$b01B(xWjndf%?GSI`<iKY6lBk#sa&}fKIheH1zv0kD-{?`!uHlI;GPaw8nX#V(t7X{-=UC>97ud6s;~do6BprB3LpVgT*=W`*|adJDr#!PJrUXrPJVaXb<%?><^*-)@al^g#vACDC8b54vlw1<E2>((6hT(xSJ5Aho;Ie>pzF?Tt4`j*l}Vc5B$}7!>{!bU%emro2yOj*PHbX?H6!HRm<<-G#G}bJ6P*(Fyw!o^CeoLk(8i5WJU4c)?QLNof(!f`3^U&iX=Z7rA4XGrAzbWavqSM=&Sl%*32l)v+{}fO)(tSv)8L^e%FSJ`VrJ!otDdI1y`mPc&2QSnsUdJIdcC17oTtgkY-RR{eHy;P*%WvYwL6Hb|f|8EHH<&$m3P4Oo#CiaCL-o1>7!DA93O^P@BFC$v=;N!sqm>oCQK)`bVB<a)YpEU}JS->+R^~W_{HTA?fhZtt6OliuGtEgEw-gt0z46Frff($_978oW|{RDp~~<PUsG#l7p)=I7b4+oN6q;Hg9%(d9As9^XBr6W#k8i<0%PIHpne9&R9iA_3X^d?Jm+YCh(zpKch*Kw%>f%*(n))e@%T>cBvxM!b-5<&qA%~vYQstGIn;1=EAg_?vK*#fKyIl^72}=MOtUXB*3{;Wn>B@J{Ib+t4i#PQ6q>(W`$6~Q6D^YFD4M)*V2v%*G4M-n*7jlJ?ksq@iFCib9)0_r)qqB!Uv-6vw*t%K-GHY+lU${1CnOq4^sd5;j_PHqb+G&2~`KKP*bt>m{gHZOf|=2Bm-Y^Z9;?c>_ZMNOk;N0v4G5-D!a7>yOC}ZAOrf@Ipwo+if3m$G7eA9nZXLOspVuk0KJ{^>*2g(s@o@_QH%}@{438(`y~IMkn=VsUI@pdos!w%*V-rLm$hg1!AP6PITsXZGeApdx7EW2%=$y<S^kG&0;qE^@N@?v3x{u1&G1%kc9hc#06lQ3)n|)9B&hbx#6vgR$HN{ccPi}|BB)MIcO}cnoL_Te1&vU|`H5>txLth7)^TaDhpkFO@v{exE|foc<8d-}w4J-uuj?ex*($FL>t;yhkQ^eYDu&8aYeF~UY!pW<B^pP&l^prp-I2~Q1@epf=n%RG3tIt+AaY_v2+24G51)4ius$AtFK*q1!K0s{qiKw!d%*MM*Ty6I(67ON0#X>kGx#t6Exh^FpW>;Q%9{4W#fV`pxA6evCsv|G9Ea3c@?E26XUCh%Z#9-ElUw7CQ~0H{G6RCpqu5WWg+dkHnT<{e^J4iCDDV)@(V-MS!*-<^kGjCE>|l1v#ujYua;1y8O`L#!>BwL<chaA`Vj~J1d2zwe6pKeyZSd*9u^^IHM^iM(T6K6qz4+>|f;nM~dBUTQc=3hnuAz5|T4V8+Q+sUQxV?PGO2KC3%{8wRH@l2M^`sx2Nmaw%+I8>N$|{HPA9WMzCLUxI;60S5k0{^W+Vy!Y0F@p6bzE}9HA(KmK^XNbm=xOUB2PT_uEXCAu4OO^yv>-*D}&K$_kOpZe!t&qC9pms#Q1s_o$y<|o7mYU)oRK7Ixs7@8R~al!vDj5ms-8yVT*@e4m{YItzP%t7~T0B99fx=G2ZJ94rVY+iAX~`hE9!GnJgTkXN3Ni;Z7^AS*BoY3a4YLg86`f81BVd!e5Nv<birZE1y)no2cq;K^N}9>KtKA21X48GUS7>iEJ(+=^pfX04wu6tnsPLeBQ<?Iv5v#m<=`n84BX@fk3`Q%&|j~Ptg(CVtcj@W10au8yem#9;vzNAAfVyx?ke!fQ$>}nz!HU4u`1HrqB@qP6b;^q<9)<0<)8WHqC_di2E_=_9)JTbkqn{NWR>2@wHdIdx!8&Bv~BA`*HUKn?v`iifbF#GuQE=2xavsDlv)$ZIN)UQVrxphSkbKAT{=p$m-mUN8wzxQg!7i!ip+f4-3NdcX^SX%S!o;_$A&s+Tdc@7v=0gI^8jk;jQ4b?1WIqqq*N5o>m4!I4iL8<>93GclcfSkq9x7-@sCI9X=y1U3v1B(4D{|8pyS)<wdvZMJrK(qkY&K#-*7#H*eUg^x<TL;asg1wzSegZh0yJ7s~fS)+5MF%IPmzW^VWOYgZk>gGfG%cmzIleh%Cm)NQJT*h2+dDxJPo6?}y32F^#Xx8EBisH}otBRo~`RRHv?7tX~quLdLhtD2V<%iUpNx=T`$Bgl+FdW0dcoU@jAgiEZAiN*uOQz*3p_yNbF8Vm;s@M!cW)F=Kd2Q#l$f>$fKHIiZ%oj>;PmUiF)(49nTc!Jh0V!q=EoJ|i>wx;i6d>k$W+*?ep@5a6E;b73N!4zK}l`p~7Iv)0plTtix%|MaTs9c^oy`=vPfQlivMh;>vGvT^A%|IDrMAs|LRRU9WhRu}Ai}$NIDzt$nrF0+RbJ%kD3R}YpD%I+oT}8R<3SOu4eB3YXL1**UJPuobf$wKT3!|K<bl6r&s8o17H~dMf-G)<Ks#by+XC^L}XKA#Tf#O}#sTG!Q560A;ow?d>Apgi|@*sqX*!m067*UE6HUSNYb-qLm<$aD-Uk;5P$QE<<KQll)w>aQ(aF8uYmsJw(lxF7VD*^m}dG|8>xK;_S!Jn73#dDG#pqinilqFc5;67r3jt>Xz&|jIke)Cpy=DHvJxV6u~7NRx&Cm2wu5r!cma0sP=yRuLPsb!4JEFAGU_;DbAymFRLSfD1jDJZf|BOr>_;A`kyF{Ln_;+R$(9K*#<>`;1^;G%9*3vfb!nK+1LkSVru1U&#I)wO?V>GlV4CknCTeZJq|hKgiG?TBV6eZe`~QMM45o*YO{?ab~Y>SdT7DL7=X$rkkj=xz~a4qZ;A=rS>m#4WKXMuEUY<I!=S#PTSt@^ix5Y~9nt<3c8rxf(J2E_-i_3dG&*TDM7e>Wkl!lPG;k=u<jhK<44B78Cjw6Z*eM_lb+bBg@Gad*mLIAtIT3$*{Fgp~%Xs7j`Vh^Ldvv_VkjD=$^-|>n%ZrLMj$07U-#r(ah2*L$mAym%hdO0Y#t=UbAt3|CC%)ND9Lg(?7i7XbCqBZb5mUSUb6VDTfbqVE7_u9k<oG!ITFxS8#pQeM2o3oK9<BqY9H24W9#pfEQXxH1oX>FQWXTC@blVTH+`O#?{r*EDb&o`yj7UdA?vUN5|F~r_LxnX<4@zIn<CNo+iBN#BiqL+VDfcxCxH%CUycg!v^9(YQ*!nF9b)qOI_TQAOcuxp51cT9RlI)$3%rBU7!;X&?X^$d4SF<;oO`6EI?<lkW$qJ8dt}0Z4TD}GW>>y8-dH70Mmi+bIKB2ei?u6q;4#R-;M9ooMmC&cAatwDbrRO^*8FbF!t~K?+xGN)Jf$l41_|Y$S;jZNb+3%xd<BiQ~*&5PibkpqlfGZt-L6q1y4b9KCxHqt??*K4d*=jso{L_t#Pcs#$`$0W=zd&9j8hvJcH#5Q`q6D(@tI>LpXd5$O{|BhGt803Zj;fXtA=%p#!B&6^6p=(6K5|ML$99Ydk`Z3FAQ$KiCi(v;<GkX72~hE=Kh(9|6gOAyS6D_~1138GYI>moJc&JIw$`+#ysx#=Li<b`yV&O6BrynWov5Ss0x{(5XQ0xKuxu(2)|lzuu2;55|o?x)YB|hfq0eRpHPW`XJ8@Mwk$f7SY83?)^B_X8@Pq4{p#cx0N<q(z_7AeKktr2=TL$!$GedRu-n1l#+B+BS&G@AcRQXlR*#or_k>uKN<NrA<k|hM^%C?8~_({It=lNBqy+QT?_!aXTsJ5{`79Q-}Y-vb$m&Y{2J5|@Gc-XnSiIXhX1wnzmy0~ZZOTA_1|~dpL3VJl0PYGb2WHz)?%TfBE45{E+S93uySP~z*`ulmGCNNK7bb6gCh#{io+6V(FN`p-bv-xgztuWP!5!jZRqcx!tJ~NZ(+|WSL;9Jj{V<alU<u)ldaskzN!@VrG6AkMST*^&sT90qF&rOiIZ@S_ggz2jt|3yfQ$&}GLLf?R;mk=4>?G&QLlBh*KUO%o1o!%|FGL@M@b7;gIWOUxd+0`_%sArs2}|pSd?(?;ye4_z&1wC(68mPx+(ieYJ?%_BvCo(P=8YJkWweAz`p6!i2qLH|8=kuAho~|?5Af@a21u5%FkD;{{}WTN4GrB)^;h(6k|QLYF%#mJz<_i!>L2ruAdHWtKg7)A1$jj!^$UY42xdtWYBFVS=*ho*DY_ZG}$zU#zLopqppF*5h_zcu<oj_!xnmmrS2AN2i`v(p<lW%VOE}h8}Afs(myppn1TdkPgb>(Te~_>ldRN|r3!RI;xs*W*a2ncB7%#>%s5NSHz*u!b$exNd1Ga{vH9H{T=wwaU3{SfTsTrcbs<2YNpe1^uWa0|M|06eW39fj{3CQnxjyd~R}<fdO)(twM1TdSrxCLgtv2dwQ62tWZ*E0%)wd#?!1e3R#^xQUoZT&HJ|{GyX?PEgVgGPUXu^)kIGc#y06fMx9ws|tbL?icUgZlmB2{V9n>x24hxV;pL_JkZp@ZN-_2Gkk(IRXa^v&nfCcL%9Gb3z*S)*#j3=BAPXnKsm0rzxp1EFX~#05YXFtPx(F8ngizH}2O8DX1VdW81OwwP;G&jd?ouCBk)@OeD**Q$jpmd~a3=8>W8V7w1<p=geZI<Q7829bk(Va<{kMKzydI0fGt3soP(adQwSu&)U32OT(48Qc!`e4!t_H|nCwk|077{(2IbK&~Z{xjI5tc>U>b{6Nt41o6z<nH`+OF3lH4zI`hEfrXk+*B$zosRy=3&K=>AYf+J|?ZYxV5RfaFla&}w)E85KF8go_DArZ;cJZh?zDpCB@)6{#csf)r1T`LuNJfBTsk(i`;Ei0JJni!+hVHhN-dPrBrSl}+@>{k#RDQn0%e|{Tpf%sk4m2%6yJvR-u_AI3oFG9-zm$7uw*m`tl)wzVMI=Kb7$<f{fH3`B?bO|LtLKDWrL&HcZZ8zkkYV0bqh}l``qSm-S;K)(oa>g-D5I0C7G^eCtBm!dXsCsz*Y?7e<GR3-oV}$ua^1x@Qy6x(lk-#ck}UPOsqaB0amlc4W4+~|)Vj=n&CY9MvbxDD(7If?vT&)?nh~G2%DHsqdcE0LSzc>6Nedyq$W;Tff^AOR7khd8VlUSKl={9!6wo6Bh{#<wU~&paneK*}xj_2A!nLx#vVw7QTg$7B%$lI*u>nYw`s0e4wE*lIM-nkqR2r6Tdg?AUZXynozk3M@A_t<DwS~=aJ2KW)uU&?}npMBCw*KbY1u9Ox<VD@;fsw?$poagSxp!}F<GRxZze-P1lLn+F1X7}8gMp_Y5|X%~2!<EO9tPV50!<JR2*3a+iIzF#OESr9W+rbknVPrWeJ9!3Th;!tRf=sTvMtM1n=c@Jh501=yPSKUZh(|L@1!ag(U)^hpX=}XJZ4(2D8sm9zUfD@u$*C{05kg|uUBMldPNSUOj4I;C#9-av>f1;kd?zO+L3WPuTB1ld|9CXZeDqQ=b+tB^Otr`7zA&gekFvFyG=<WJG<+fjO``dVz3%uGhwW1v56Ey1S|7Ca#cJ61>UZ0uKzU`<(XoBT!1;&n(IQHdtFFU0W;@0%ll};5>&FYU(d4Ac?<I;zPoAeDiyw%k>#zj!71Dy&T*AUE|xaoY7Dxe*%Mi@@!4_@V%nQWu{W_^rI~(4d18M;>@bO^Llp9a2g+EfKNE<R8taMjzn>AV!&fq)bcJz6nSlONNk1sTqV%huB2%qned+dx2d#cUU>duP-P%Ta$9Lf%4HXj;R43sIa(bMTl8JyO;k_$|IGWR#lJ}?;Tea=oonocDY>+=Wgw>#mBys03ZXcZt?oGtmh9x?c=<=l56=Tlm!QzVk<jYl=8Z6d>8(h`r$>WvgjW++~V!2u^d!}HwjWJB<!73D2EH2Q)&^)RqEnJ{(SDg8g&qS<t45GYX?J8KuyBq0&ZVE>rOox8VDG2b1TgBcav5bGqd>ay0<f$wZs7U27A0EhLMx?aj&F?N6a6*5O0dEiI!9wzF1{B1TDK&G)8NF20wySi0Wd_4F7OIq#>N!Gj@IU5==sz#uurP(is`*8p4HC&Ws%cS9N?QDHWaLaK86%ra+mHiAYAcPsx@^f*awfp$OZ1d%BQnZRlZfWYelw@yT&c%nOqzfwHmxGoP``IjVnE8mPbR}bsRPHrq(HqBQ9e8(F^QdRFd^Ax((d(yEd<0x^x;6;OGnAWPOqEbhwN7tuX{X01kQUHKH{!cgjYZQ&ccts6JdTy(YRKkYcX6riyw`X(&XfzF#hp(_W$SF!e6%*-Y+k_88<J>XCfXgVkm0P^Sm5Oc;o64zCk?)|6+&XKiEHKrJoEBpnpQobk~T%0<!E%8WvbuAm9cLrNmp^sPCZpgV1Lgg0pa*c;mIbErL_43jkNxE!S_=R(CPD7_0Kkb^Z(*T5XuX`XkKAK+|6Js^>K{Xut{;Z{x3m=eVJEj;fCRgWpYCKV~e-!&FMgcofmzG8V}kH&j~34J(=}-mY&nYHMMHeQ-I%G`luhR0i!+pU21;jD38gSyhK**2CNN)yDRkhtmBVTZg$r&>Gt}t~CAH3+<30HVvdpAK0-(XGTm{3GR$bn}Ln#n<Ax`fS4CnQsyOfyA3Kp#Wgl%)hHw7th`YShm11vKY5ZUIJ}HvGhGI<T`clvJ%+@9^Omgr1T)H34@TuuN|F~s5rw=OMoyXO%=isx8Ts#KWTs%-CQvbo&Qy@N2*|*u6NQogC1RKoEbJF)BqLV>Pa8tyL9YcRT2wT()L!VKVkFb5jRi!987o049f&^OF5qr`cYA&H91F&t{l4X6tO!rev1v%a)6YD7Ni)XqXT<5bfEpYGmo7n%xI-`X8iu4ogq*J(>lY$t6zJE-HY>uP2-5Z`;RSyjFucy^8b!9a!uYSB3K$+{4M7k(22d6tvV<D@Whs}YRe*_zgZHcwh~q8_BxEtoy^pE#=K+c0V(^NU=*$9VjVv9FzDB;Gw^aGob44lfMZVWYI?JnO)zNe~ZS@)TM^TRWd@wVD!%ehg3ySu_XvyU_uHIN)UOqS%TJjsE;KgqQKJs&uGvq0O4L%WU4u4D0)fPgb24{L25d80-g4@06os)wF<f%Y>W5y+96kGk_i;GBr^C^?d5SE2OfrE;-vCRaGME#6{p`R0%hCovLNZ`dtVD4uoAI=kpc?b1wv6C@m)XPt73}^(|{gkxykK}}N$wG`-LtGnQ5|pEaWc(Vs@xNl@<#uWGq{Ut@><0WH-w1@8KOr=nGz~CRqh`jD@YGN6F?^gX2_L`w<!i=RK6vwb`^Mq&;_Jj%Ca`_PEZV_?#)0&5dKK32Cp7lhQ(rQD5*;-Rqo;s;e$LyMi$e~3-v26azYNgyPd!sxPB662JQFzB?nHbpMj31VOwic;$y^8g#8=d)r;O4`<0`qAOe>Kd$Y)q)3jG`2d1g^Q?bJ7Jakp7)MQCO?!OV8(;*<WgKyp`mIOve&(G}NqA^NOi<OzB&<U%mG0*OmdTp$_>U7o}dsjY8>XZ~yk=M5oS;Hb06o54{DJB|aZACT2N`1cQgYb0<8V;uDG=Qvv&fNMm4BE<aXyb(};e@qbTh@OsQTc5blPy$+&nR9S0X2L=t&@#%Nwv%Q5k>#5R36ASgLiFekR4gC(r*qf~LX~19qBN4rOP30#3rpq2GA1-DiG(}$Y-WjQ!zJZz+#0~~a&_52vEoZm^is4`Ub=1t7{$b6S=DKFaG^Kms|DF1`!<MC>XRe=In8EdKHM~JrE1-mU)ikkXRgTWLOx4f7j|ZGJr?%-bh(BqQw>QkU0+?VZ7`HtV{_wO^BtOok>EG1deHy4RHOOkf(+3@7jM-z;WV$mvrgL{@yYT_Z!MO~S#NGO;#+$JXfA-w;Z<5);^hn{&G)O~b~dgO@n~dKD}a68hBaQhiHwol116kM=C@L`6``r`#N^?NYddj+_Z-!DDsEpuBsH_Z$JAp6=uUH1+i3t1bs9Ncez*sXC&ovkD5&sU=vy}BiYy5+otkEkI2*0iC<JGkvrnkpSM3pNTmA12&EFSN4if$iCWiP5k@pZe&p2LjLj=i!4lI`C=uwCF!{P+Vl$VG@Ot?Ir!`YVFS{X9XnHnn$c4!aZN_-kqK3l~9#IO$ic3cUz&xJ=W0-`lW+v{t$>%ue>*MGQI+Xy4mx}z#}%tyKnqI?(31OGAyCvG3arxE@)yWN@(z-&63p)Ig_wB~1g&}wb>W&=YabH0h1;(idf%>9}h+OCzk_*i9W?Cm05*R6Hff$LPE3ha=bC~;B<tJ%Dd4U!Yu=(1#hT?YTn?lO~4A>C;f(1X?H`bKSMcYSrUz6(9Nxx0?r^k)5bZFl|cI`wh9Q@iDU2DZoehK$44>T3D~x2yS!L|0p1JUCFo27iq}QCz`0yeP<K;9@HhIo^BZj4{fH<Xgo`{GQ70ZJHrv2uDM-ZSP6vu@zz#kA^q|@ZanqphT3&SFQ4J8SWFu>1YiK=Aq~i-u2Dh`tABQ5`Ym~m$ILlLxSx)=aA4RxkJKVs5{IZl~cNv-3cOT;k9%F%)^^-oO0as+WO8G6R-F?U~q8m4jA~B|KK3I1FY7`K&luq>sU^)w-=uukNd&$6Bc>?TLErahuOa|HKaJmdYwQ23sZD5spTJpk?5C{DJ_X^Uw)KFqV+7LwWD$nnbf(=VuT8UCeQtb$b6-xz&yy_J86MDJ;m=HkdX-X-R&CfEap@?Xmw)dEE%8X=@a2QUL6AD^lMRw>J;;UegdcSVt6*EC-R~-EM2gMcju_#zmXKeB)@+Q&nz9J-A6DDqi2!wp^sAvP5Po23s#>-p_b*dW^nE9&Gp-RjlG?iSqit`jrCxy2{E1OCMP(aR{Yb5Jrh3lYBjX#L65jDYu}m`m1gi=$8avKtlzMeHN$K`3bHn3JY&jUy&|4vSSQ$);9F9fmoPa#NIIRQ6H^c2f}A*H;o5e9Y~d`68DESfg=5OeDvbA~Ap!e^O$mzg*eE8&j>nW(vMJ<Jo(QgSvvw*nD7bWc(!OKxrU(Pf9EP)^J9>KeCS@wJ)j(UuFoi%VKFPD4Wfys@@G)xav8Vo%Ld&=;E}HWvvL`xE$yw%P!|HZnV{&KldinaBZzSh3Ccjz2Z5u5we;+9JW;W*tMzTMU$&6Uc0uIRg_YePx{<vW<?ER}XgzBZI?0b-mjkb-=ha=(`+a5CmJGLXmp7Zi&f;Nzr)yITNE(}U&zE<=C3ULIP)jk%+%s;~4zZRw}v{6z>qUr%6ZGI-G_ek-2E*8IiJbmGrtg1qKA7jcamPVoEJtuqFW7OKnVPh$MFCM>UL-CDH`$lJJ`Npdlimk}2@F0OLL-4G{(tm3+YT9<?o%-ALO$;^I+o<md@;>Q%l59y&9sau&=~wy%b)jJo{zz;L=vDY-Hj(^yXH4!4N<SDYm|FkJcrN4^v-}Cp0iZ>hPZSd=G7>(s=Dt?QJyVJ7sZh~l`Y=;q;SK}wvUYp3fvOPNmseIBo43}t@79o&9)I6?cdM~`r@m9CyW*Xly_>tWogdf+X(s~$j>mh63VnNUgaRis%~iux2pFX`k%-`X!FAfE8lLUHHN519uzw@8XISBoy0knK4ix2HSSh1{5_kNt0&O|V4E38n7{(7;r|5l-L3f{G8taHeVByB5XxK^Il7Ae)ca0|1t8c!^Y5`ha7Z=Od(($Of4~7d+bTI05li}cKh=h2u2C|MTf5w+`8~Td2N{TI&D_?ke6}#w@6mC4bv2Y$44qv%B8k@9Blu*@i%<?0P2Gx%U^>?X$G9!5p(&ZhzN_iZ;786vRh-7OOSH#r=<LZGwSVF#E{Qet~8$-Sfmo&8x4Dz%UTLeGtUJQz!^&7Lh^@WiL_tsDJPmskBx{$+BROQC41MAj-8#Q;|%zERwXYn4LnD=RdW-}jks`nR8R_MzFG+T$U6^q4{@t}((aT;R@)<6p+z8J1@ET1!|Fk!82Li!|P?SR1b;EOM7x2goOos0`mdZ~Rh?6nisGm+g*A}){jX<yRwK~7-@k=3y?3wuCaNgcW0ZtXv8UTPogpEj9WCnDd<`@K=26&)0tIlU|<+XJRXLgx3W+{!e?(*0zNf?OwhkRWc&r23$k!Oes37r(z)E_>%D{rNv=YC>08lMfr}Cg~mb1_+tBbP4|;ZYl(*9FM<WEf+5@3S?RMjXxse{#%~=#bO3QR(5}=4hM-F<ZV!mpsc7=Uh-pb=zhz1@y#3fqV7!r2<|WI)q2=vshTLWRPTNw57cq%K|0Qylh_rHq6l3{GgiUNHeZ!<yh}<v6RD0R4-MQ$KJsrwNRE`7h5x|jv@{&=C_p>v-mg4yI@;(EY0L{Dr4!Rs6cBQYi5e;spxt0KFEupCm1cAji~QI+mgzWx!R3el%#4$J926)93A&PEkM!AH%$714q8)>haRh`Vm~&*bE0q6m5&=kO&GuZnyk+QIV5e#9;i~ecL{ZR_CZ=*ru(+u*3-G{FN|z%!*Zk8>sN#Z~=5l~+(YR|8mI;QVUk+N6h2dcaJ|b)INX$Ylw_Q(>>5z*o2b9Z`Ip|csH5TY*s$-nBTEY4+V*|+gF740h_fq%=mHArEP0cNDYki}!yI2ktaTCYZqHDWc_JgY-$s4&f7_zG{sM5nWKo`xW+U8p0E_q^73$%CK8lT!(&cuo}X%7+6;L#cXQrL8~t<B;Is`zkqke`evQ9uIp7t#t~v-WzTQS6X1qOo33Mu3kLC83XjOo~V<zXk^^BmRCmKCNV1W?yfTj`_clH(LkSR61EFyQGX%9jl@!o}H|N^?IG87s=2efkiuNoOO4OTmgfdOiayvYIM^HT1IXv_Ps@;;y#ghX5io_58KIT8r10@!jzc6wVWP=$(>LfLMURZFNYE|<8p}k7^Fre4~wNZMuU7DBOmGZW(M(v{*sJ3$(MuRdykGQJ(|S5=on{TLg-YmxC=O$i)Wdy)6zx}P}ddROHQi{`xhKn3e28ic4{+v1H-$yYWP_!W|G7iy8@oI{^!0MD9*15$3A>R_Mnf+lEdpA6l2zviQ)Itzz&b#$U=j%kYdfmsYG6>PX#eIKmN3owY+i11icvM9~={feKp^k5rfdhc=$c^2@V;Hi>JhH{q9y{8v_I6yr?i4qi}zjQM<yxNW0qSY+Op26;~WHmb8f}KJ%Bqj8rrU8m~Aw_c43aI3mV7^KiMx2d8WggsnZTNzU3P=Mm_HHIQwFBKACAEaP;`1o<Ww7X2WD^B^*19@3(0cSFnC82ULUvo-DMid<)A+qgo?^Rp@~&R65Dy^W1jA-C2yYa61NeZo|@yRm#NHZC{Q^5HchA9_o{#kpMOo~O}=ojSm}6+17;{gZo>tVY9<FnZP~+26M~`z9&-g`9ak0&${~$KK<pifFNzxl7KTkvai_a?gWDT-n(b(jusWB1}=y1bt0U+5+E052kACUoKaxW&N{S)h}jkn)wpOsCu4UXMbAEJo&Lv_KBQx^3w20lgg<0nhw?Vot?dUUeXW<u_u$TBI2SjW*WW9Gei+(Zq7TM;rK?wFO#&ujs42v8?&XHP+HQVP3r?9dF0IrG2}@VT!DTM4qFFf4v!2bCkLRPOi!jscvj4nFWoJG_7_9wlyNaqPw{F%O6{2?cJkrH{rbb2PpBcj0hnbX>5Td4ChZ<t2o8JgfFjr<4rDLRmYl0qu~#zWS7t&e&9n;4$!Txk6DzD^Op1Y$u!dfaS`T}ZYPmSyQKGqW@m)~4vhs7!nR=&ch?Bu2IX)l~Pfy#Ug~hB5WB#PN^S~5v{(*!KQki(Z_L}^t7F<H!IuVI`3yTiij~RN)@xgheT)LE*kLc}2U!ta^r_to<4ea^jLCRxS=X18HQ5f#0EUdB#O}e;0Oj*~Y8W*N$Epl=x9UDIs5hudLdoU@SF4&)HeNfY{&F8#N`L5!zD7R!U4~7N!;{yR{@t0Z%H!Ix(zM96l@k4J~_N{Z*VrGCEuC2M`d0Dapz9SuHbWfQ_d2P8kJ@z8A-|$p;o<>Wh(I<ZXg#%RQaoY5%+5T0WGLbQTqHZhd9%Qdha=X6yW)3DOUcT9A?C$Jt*S6yA`qqxNDrDQEGR6hk{(5|a8JXg))4*_^wu+_BcsL>mS4B-fvz&ueIS0OS91W~}G@K*@$D-k^(+jU)UM;PDe)0Jmj2Y4JEuTpSd9>%);2#4MZouaYBFAJ56TCMO{NA;l#;q{DlDPKGMr?i|{J6VS+g{&kZ068?hg=Hup*Tk6;DpMm2mKsYWd94JRAjt|w|VU4O;0sE{nO7bxCLLj)b6F&u^%w<t{@@5fSX1-Ut}gb=KuZzgR>$^X{O&4CK*60s*hBoe*7Ky{p0U6E2XR5vq;!0L^?U4&{iLb&#|7&HyDTjrG0K)vve}*kg7<^LY1Oc>uWK~&_21Ehg&>G`n_*th0CXaQBa%_|D3~+I2sN*K}46yu6g+phkj%PvB~ICp8)ZcZ*jkVB*RTT@pLOem@PBm6kl+X$lU}_A=Er&8OO|*g|aC`?vs~4m#Lc^_X+>`%#80g0_C2EzLhu7uo#hAwpGtMmTN$22Cq9jDFb-KicXTA#3&pH26;b+{an&3`nrsqGbF7?_qegwM420gr!+dLrBs>TLh!aM&Dj<*5{s>-ROZ&cF)nbs9!pDSys>_F-M40x;w=SfnvfjK_Z;CQ{N_*@#R?NO;3P{jW3HfQ_e0uljL5vlB8u|~C9gl`rD&Uo=_MJeB~dh<_}i<r`C-44jDyy=eS{!@f!H~TIJQ3=AQiO9YnjsJC2=Xv5`X3e%gq(JsGn-OJ&_y%H~tpZYrVY4UU6EywER*#3NKg%YCtA&*kzuraIEWRlz&ZViLjr1!f}pIwLcx}h;6vcl9i*0OQ6ANL))$)VV+efm4u9SNe=Hh)}?BK(j-kpP`JBOzl-SstNS6j82oEeaLvRRjXO~V3t0n-?(|ySA*`!7c?c3*$F6+4=9evo9844+z?8oy7!K6M&S7aZoTA_lrP32phY(0=6s45XSlxIN)r-&+I+SOJolq*Qd|axW-K0=X<qL9xk@ILIbJ(2g9)9(7&aCpT&)W-(%bxDttbIw(Dz!81ZEn|h8XK77_HMnl4Kf?-RJGgn_~v%K_5;6}!q}z$O{@t6km+&ISX)4k(#sFg8MRbix(erCNvy=r1r`Q7ob?LS_)f}cWn`Lg#LhE1aG%OcybG*YF#I{$6&$9jqnE!v&qn9$b(7>61FYJUD{`=cqt^I%IOx64Oueo4csQBh$1nZ+hyR3qU~PD$r^!eXJt{JE$=u+cXXXlEcm0vKD}nx@@^aud>`)7^cl{b2T`xZji#dDhQiVhHKR4qDe)HmYS73O5|KfKtfyHpDNePBLjwJA4GybT7L*S%(LOm_R$UmXBe?_B4ub2tiPenOWNE4s&$;*Evkxm7u!zIr)yM3FcuDcM>D<ZJ6$hv^45<bhzk}e&bDr9iQ@v9ywN+d}?U@$&Qg=E4JQu+}n-fFF3lH$&8ZF?6@@OlxkszKsKf=3Bjy)lX=td!3#;s^2Xf=eNH!|W|E!^7x&m((F->CIFg@GChs(!D2r7pmhxz2d<`7gfqVffu!BqFipC<@c#Vko4a^M8UD2*X>S1Le}n!;Vll7+oenVwOQ_s`2_|~;V2V~yDhmhK~K^&FwzGV%I*x`O-j1p>F<%0Q*&cq4kISxnajY_pGE?P{-_uKM;g%qgPXpgjSGJu0+I!>gxlxmB8iHzBz{RLwa;0}{9J)wOZ_mzHjUU{@;Lh(gFdaDaE31}Y;}m$9XkHc|6-2ApZ^7Qpx{AodK8?%j6LXePlhKG0`0J(1U)7_e-HiMP5r?mlRutO%07cLVd=`-Ta7C_TMZF4p`q+#-#ixqvl1N)|K`k43q$NF!-sqFJas;Fj_Cb7r{BbexS+ZYgDTM(h%R=JzRYfv9f%V+xR;#z&p9<E)#`M5ROg_Yg~z;v;>pPYIr5!kRHw+I@C#FLvTqaRC6G+2`(wQxbW*BQTsqnjrM-tGcyT<P;Aa<v1tw$|rH5FEtiyLG<*mvK*?i`zLhRO1#f#sWGKNTT6V@OKz(F^27zJVJC&S*LaQNfzJQ{2GrL*cIqub7Y{GH66%M*33R-9m{93hHjp2uXU`;gKue@Sv`U_WkVI~nP3e;}-{&v;4wK?s^sU$+A-rR1@<duyZdBO!fM{KxoJP-e{M=GAD-cji%RJkM+0#&dHn1;-=yUX>g;-u*ZQNr!vdV^+^`1k99E>0itoQSkr>#+jRs;Fbs`?IY+l3`t<`SQHq}H&|pRol|uOQK4I7+&wtHKUdVgR<ZROGGB=JHJT&OaLNhjhnGLa<4f8~c^?*ubkJ2lgMa^MrqW^bn#l7s4nwIYrRUq$cH{PT?QRVJA`Yp$WsF8u=LxTbHyUApI?_@OT8xu68Y$^|--pAdc5{-{5IxGVq$nm@kj_tzLr;a{q)#&*5uC>fW~TR%CYVs#RvVjZ>&T5#+lU*Ras8dz>TbLa*H+OhLFg?C)<u4xK&uYZHE{(>?rCbaDELg{yWc8DaKZkNEA}H1JJCF!1Lc8y*=Zg1W1LOH0SIT@k#_Z$E-g78ui>B#&oay4xjQ~;O+?71v?p}C1)m*%MDYAE`8it0lPZkMC~5bQX_SmYC|UCv98$s>KTdrB&pNa+5f7?o3EsK04v%{~L%dYXI<`gAVyi>K$oe_!d`_3vL3D%O!3h^O8qF%fKK`zSDbLDUeP?xheT#sd-mYz|gE}L;Boc*G?E)Tyg@iKxkV97~7l{&R4IqP*)XQ+SPx*1;jadmAYk&S1@r}nx3ReK_oi9HQBlbd!M#bVq?U&cW^QAqfGz_RbvTfo5m-bg~)UR!l&|$r)-BoA}&<8RG*y;_(Oe3~R5SrCqp~kSW>9_2jZ~Qa;X5==rr4c;Jz7ZCE4Qlt|H)y3<SQLmrUy7Xj&x9WHE3#cHGy!Q`GF@jl%RUry>{o<5q_FlVSx%>r!8D;G$~<|g4Q%I0o2tZ%b{}WDyt4ZFKWYZ1CSX>B%vt$+nMosJos@F=OgJVU^G|tbe=czuNG9af%<Iy?EPoQnuqco{N%Hi}W45gMG~=|jF{LzO*Sz?xoh`&7`B!2keJG>njcSz{my#>l3HljYeMKgLC5rj75T$2^6h_Wm`c<A1jUqqFR_eqF$PEz%_meT6`7*gOf`xiN>60K!=Utu{wk+}{zY-!?y?pITy}YbN^*^x9*aGnq#s+~vK;IXvowV<*-IN&1+S~S2ZACEq?)Q=h)Kd~T_=|v2HgMFxu;sE;6njbwgsHd|LwkAupuEPn$}GMQr*wA@=}vBMH}<wbCAxdJw*9V1n4P7sEnjfMvNo;X3ltojOQKsVd7BS9$H~X`4IzenC~{bzYQ{6a7NGkcKCVAJJ;pFH^cSPi{o2Cf(!$lnm0-KRi-?L8fwC*)J_eNaf5xEd>EB{j$IY_?EuZrYboZe+DPIdKo>oQnL(A%+W%Hqo@|`||#_C=(ZPPFQO-dW2=|0!l)F^fQlx#Y<;d~^0G$JWyh2gTvnihx^^_=&yT&vL8d;x8#%?x|_#eY$XAtDq(Z;2C;7u_=xqvSE}WFnKwL64PL<ANZ-zhU`=1iAKhT(O`#uBv-7iVdUMlWBE7WOW@5#ts_(Q5*7USx`iGyx$#{?8jM!Z@SOMAS_c-4ZNT=0`Zce5DX7~lC-BWZJhA4i+Rg<J?WgZ`9Z$bjJVagdcq(UF;g7pjamZe1otr<ZhDFWeD8f!512JMR$7Am6nx(43rW-MQx-%qY5`1Ief;1;C_pO2g&0|C;$w2C%q+}U6r*H}iUSl>rDRq7A?{mcAqm$6QM^%&N)MVGi2V?{Y>Z}_Bqp8?P`o;v20%)sCo5QMBeFkYIQH7(-iQ@noc@@#FhReLtY?;jf{}x&IIY7CPo|)1B@+rKB$)6W9|?AqZ|&z6(M-+o^t^Ml$7a!2+2Q`7JKN!e!vN6Wd%@v63xSNTJSQ2%r+Rpv<>0P@bi3zS7Q}cjCaGQ-s-)MxF!XeOz_WMug2<i3sZPAWT+W!!x;xd(Mm^iA?R+Zt#UvOhyHNb~6|!wt{0L@`3Cq=%WjM^}8#eTQCPFix3S+Q<chzu^&sH$f3A<}J@X{Yiy}LWUxYjd6=I}W3(i3TZ5#e_v6g-w^|6>kuWiAUHr13R}Hhc!(D7uNvn@=t2U=FnfzfH-NvQ!uAB^!P&J$O%ZHHGVyKm?aPCbd@zt~?V+H~9M~AfxNY9PU6&K#G72#(YIQh|n|AHE9LA2!LR6ATa$9&y44)W}$Hy3KY6CnJdIlAU5l}KWc3M0O9e(pK*Qdc3q(9-Mxzdoa_9P!NzxYtTbj~<AA*|g1wu<B}DjGYD;&SqKi77B4u|>*_EAWY0**)mwxe2<T!kU@41=KUem%CvTL8_2RwJHgNLQH-fd*&E_j%w$&CbGSL#iZ@$_WWPbLM^GU^Wj8Fj>oG<FQX>`rm7zE#eM9_W@@9T4F=UCLU)(#*`9V#F8I=|4q*!zEzRQmycWus$p?M|V6t#5h)XaX6Y3x{@JBjRmgiKhTDUs@Wf9xFlOMKGR+mf*Z*5xI7EgVt*3)V`{rbo3Im*;nSbJID&I*j6+F(v<CMIUD*X8{~M-&_d2~Xhe&rZb4>p(q>y+{q1NN$36_t4peO#c)E}PQZ}l+tk*{DfiY~;heXblGClp51N-5@6)v!vE4&{}cz{%Yssbii-q^Sw9cPB@^QT5RP<Q9d7p{jCFF|a-CQ^v;$&Okot1kcD_WFOsv#Ou<m!VXa&xH-;=k7y_9mc+<<7N4vH<^fuEyb+DPK^xsDT^Pk4(%NI+6!7uX$0O5tcgT;zpSjPb>w$$cG2!9%1>)eoe5c8u6-RN@;YMR6{BcwVLeBCQ)Qqx#_9jPFtmq2<heKI$E$ClwmD$~HHYB?=Y{1b`PwWDP$<d|Bk&KX)k!o;76**`P{Dq*W1awbP&;phOkN1aNeEr-dioHb3D4J@LW=bTOl313mQ0Ij(xD!PE=^}rt9CVP(p>_JY9AST$Cfq$hSeE$CopnRb5xo@Y`{Hw*XT(*1*>c|rEA$s7lTpKkYcC4bN+HyFz4Y{>3yXfK9i*}z$W!hFr@RVi7~XN_b1~8fG0i}HCbgHId6pckN;!S`DLrc|mgJB#dV>6?(uzXkXbLO7w8v1S5gP00ER`Fl<dd9AjUA9hSKh4rvG6><#Di*gp$^UNuGzurIvGsR%;|7#%+ND`Q{GP1XulKB4eE*~qT3oW10G-Lql0AnU^u=f^j0e<NsL}@Fj8KT+%euVc}85gWOL~3@{OU-EnVz9^3W;1Rb8s2LaYvYeFS~-;k=5o2;oo#;k+tpzp}8{j8az@D^1^N^e{TDs-v&fjMU!;J`03wlRd5D!}0K#RuBD}4sqp#*!}TOTJ3ggEOfJn-#aZ{p+7!2rOg-qH8<Y5V8!H&7nM?RdL2;Zc<r<}Uumzn9u6+D4s08OoK&iFohhZR&pDxJ&E$G>ub<DG|8KU&m$D&yIRHqbq4Xv8zoG(5SHBUZfmFTXD1cb{R}8wqF=iYruiRGBo*+MRrh<Jv^kS%Tc)_Q_Sn&&K-I$$-KsP7}fkLnHl<gapz*Txh7#`;akVXC*fmr`Bec>-<P;R|^rM|c#{gr6i|2t)Y`9hH}Fb#(lIwiSnpRqqc<cz5}15NkLkp*8<R>`klJTvre)K-#*(<<&_uuJak%Rfqb8&u1Ha;ehM@-P<*qMXw3-t>5=<Fel(jRK)evE0@5EtqS#S;qBpebG;}q)KwMabKaBJh2nsYHX|g#QZW<XkmPR7*9ZBo*1R3fTkW+#e4hvA$^yhob$NX9Shx-Eu5p_=%g>gc*1m*VY68Rc@WpS+tT*1e{wvSR3C}@Di*(@Q&B}Sky$lX(0q13Hm-Q6%>UJJWWD?q&3%;;W3bbIBHY0;Gx=j-<oQ$tlbW!^WZ$9aX^e*Lx4IGI^f{1x$K?z%)Q7WKP$ltNiUU>LaK01TAO}$PcYgn-IhI1Bt=x$DkuZHDU-4Z$PnqW$Xz1(XatT!9Zc>2Ga5^~kY2LP_Drx%iEaBY9FT5Fn7~R7os2U!_2?^(RJRHDcIT}8whW+F)rP3ak9iCZ~z}meLl<2I36?{}v-KvcS5gm5x@3_I<`XtsRCgo&NaHSBKQy3kWXPNK68_2jfBk4c}chS=LoMMw6^Zw36%+IC4`DwuR=095LX}C*EDjdbQMydaTkcMruk7ZWx*Drs{@Q<HDC8hI$S09%%V57Z|!w$al3Jj!8D=LA&%ZQ1rx#&DINa|l=W_TQEqNv5HD)jQccO&%!3X<b?IEduiC^_#J!g@JWm&(NmE-dN`i^XP!iIYLYwgh-?0Sf9Jk4~oOZQki2`tmyeL<1;H$L?uWd#C1H+J$feD}_p&!ubUP^bY+-jjw>VhA`Sjz_&uHKRRkvm&&DLv02P%E{LSl!gRbZzpfcg$1>VI3rtbrnN_(6i4Z(ZbTP$Ofk%s2NV^e6a)6Ad&JBa63e#NP>6Nbl&SZ@ys2S>Rjm>yxZwn-cT`izwZ^-9BH%mj9nE+<;qqEZH4oB7PMq~F1H-8ddIfT!Rm*X4pBk?6?rJoEB`n`j2h5s#ePL4<1J@~y092qKf{)+tTEIJ%zjv`W7Rz-REcOmmxL04z;Rr2fmq6JO={IR}1Gi4Cr9@kNSIYd42IB8bVkKk>Lt?Aqdzb!-<)7i@VxehC`9bM1zsLY;e9%q8m310l}#eaM8zrXnJFMcProoUMZ;a7HUgbKm`csNc1wLb>xT!Z6yA(hxMPgdRya%pAew)jJ!zSc|=S|#~9!}V7Q9%ZNuXGWi_r2Wj;fWN2)<-9CB6<=xgM9^-*Mm-56^B?Em8B`V|p`bpVMW6Z`^w<fO4LqqWF0?dr6!DK4oZ)VUqe+rRANo{HClQoMNGF=WtYo`8?<Gj?h1s_{lWYgwfX8V^1hS9NqZdqP#kw%$R#8JrIfpU$zc!<)b+YO=nzfCouoQi|3P$}(#-DP$HFnBRDf;`Fo2s9}MF_pty9xSc+4IDqMklV#ZeJfF_h6sCe?~&Xz39HD8=FYhDLF#C^@*AgNMtY+QBA&FOC$Cn*RUhGJh>f}FnV;aa^Fe1^taW|u4dB$whlp3iOoK;_6C#k(k2AFFze;P;u{Bti%XsJv0lFTFQot@yj7_4Q;XN}35TSABnWTKaDGnT^iW_LWSIXZf+;!0QKfbo?3X5xmNvy`upOB-fT<@Qu&*b9SQej)c+8J!QRKztkhl6v+6@@c-3?xUsxxE|loU6f#|F`*&KrqRQ7kg8gtz1`1y+T~t`WP;ht$QCcVYOz1+gw=>OkXF0yT`z|CW__`qX3e(rL4PMn|gvXNN#T8L|6{)<4#azE(c$*Sgz`fs;WsoYcyHa?n3Pp`|0VVV@1=vVj^^rHE5M5#+>Q5ctQZau}fG-_hJ7xF=^ydy!v6BBelc(zS>zmp{?$;G_+c6Ir9qC@ul!5crytW|WcV#k1E;W!ibOb@lLir}HXQressaASnXSM!1^$2_>uc6AC^X9ui<ZZDRs9cEZPb@(@Jvb`Rl5FtV?m1Q_f$DP{AXf|^j@T&u5{)Tggw<Ke_5IgL~Vmqh1UkfW6y{R1tNjmn@1)gU;S?$;Z1PkNIhB5)v6i<xPn5D6lYa4}eDbwJv|o}CCNOj2Vec2-BYTH9P(U&FkN)!J@tqj8%eZM*&9L90Iz+1Dtxrk-;}5aLTpJw<+&^i_#c6px4fq?Ei@2y1J1*LQX>9YQ@F7X8RA0{dV*d@!kEtp;KaO|;Mx{AwqdV&JNia%$#@&Y{&~<#A62mP2uPLFIzlQL8d8fjC`QAF(6|KTo0Yl|DzGaeF@m?Rh*TUpw3A*l$NFEJ}hF7SW8NdPx2VTHI8a1FPXqV{d!4u7_oZXyYks<?Z_R`mJ?XTx#@ovaNCb<)v6n(^$`93{6rEm?KQw?HrY1ln1%wg@DNOn$b6c?0mbv8}Dt_wzjq#ZzIOffPCgg{Uv)Uh*%Ww95P*S9wNV|Ub!NdQVUPFFZg6rV!2Odkp#yYB}Jo!YN>}5-kHEDwq!29&DhQIAb{k)O~{iB@Mq2}oK&lgz0KX7Fe?(pm^c-8D4bO;$HI&_wji%=)oVNJm~(YD0~z4~@(M5DK?m#FNca=#C2qwy16wMm4vw(tQ4)a0KA8qfQMp_WRBSqJ9wx^$vc23{-yt^XtepWSK4yHPO+%%9Zq;`0(Cpu8O^*uR+r?7y5Zaj57)RBdmz*_R4MP~Mjr!_t;QCOa)!NQ(!OlJiYCFN(we8hAunYtHR%%e`b$i&X*X>)W*LU{r7Tg?$Vx@9({r39iuCqy}>&A&zxTK#q7!v!ZI^BT`zpd2kck7_@Z5G_*2oYv0vw#j|?h%!8avbxG6!p%n?Z#c)AUbHmhhu13_^Z?^`_Y~HcHQaGV0|ZmsT^z~yALd+U>D)7u~*%p!hVIXEShh)HBfG5aP!?@F<OeQ!m!&9kPOy2X=BLDU;qmb1%wHjrhF7yRR+!+lvt`Hos%Y!?0Ej@LU{ST%V9B$_R$JB0-d~cm`vM8!vQP+L>2ebiF}{E#I<)|%5JXJb||SZ!^Ur|FtbJP8I^VpeF-Kx<a;3K3Nvm8#4rrgyMm|OwfGQZ=9ssJ(~=m@3~XnsdsZ7;?^1V^1hq0c2do*v*$w(icsZWD*N^)c9Ny}~cVI^rFJ~IKi@|Oqa9)DXfY*Y;tp=i_2V1r6AMVw6BZNrbuG66&{PoW6S}}_VhElJDL6^T@&rs0`AVxqf=Nub3TP}ImMnFKt!4mAOV&>p?c!$xFL42)?%r?v+fedB}7c&4BXvY<D6=nY_ycgtO8B_=P;=^y?<o-1$xJxG@vzR==Gh$&8Z>2MrA1mXoa^$~wc13`1(4U@kQo<cJP1?e0U;qecLjuFHS@7a2?fiMB_~GKWQEZMo+2#;05H)~So*rP5gY^-@d0Sz{-@W1C5KfTRWEzvDm_!7hge#;%s10KV6-uT&eu(BvZ1OP(1+mn_GKGvfD7C{ew(!)#5}-T-v~0_UF>ok@4orK;sM(?H5foVVNGpTt&0@){a7#OBg4Q?*I9$JfiU%e$=uS|85$K=dyHG6z`R3%H4@xMKU&%Dj7HH~pLI&M&uj9%!=y=H!YEWo5mmiDsp?!3dPHH~I52XS++{c`C`hL~CDVd_*ufCs=MU<Br>H&*#D)W@m#GYW5AQ-E5;r&z;RFsUz!!c|Z?IEW2y2}9DilLw^WlReDGn?fJdugUpklr_vPiKB7#i9z`I`zr@R(FwParoA5N;Nv_9Zn028E=Um>ZT*Z{O=((8i^R`UMwBA9vTS&?+dNF>_kzj`T737uaM-u0@hJ1eh==dCd$i5?e#rzodkN30_?Y0m@^%#^y;VbCMfg=lbB{gY>~E9bpcM_F6bMoMMdI?D&|&n3}?#8I5EN3LFGrSV7rcPb7TcWqMzNG#@|NOILu3<rDIkcuI2|AUKDZWVk2f9xaZjZRV1TXDf{qo$o$(rx%JT?D#G#g7)409ps)j=jk5f(!igxl)4c=6ymcAIfKb!z#a)i~Ls7suXBykk^?PgeO;A)f8b8WcQkN@aGrgt&W4PI%m>#$K$(6$d|B)N@JxueOBnS>TO;B|WFl-1~0t?>&EeZNV4iPmrbwcu3Ti-$Y9Lm_t+S?CX$34)ZC_2O@2c;A*+^WZ|K{qKZM$7JMz>TJ=v#~`{%;(s1QR+aAtr{{Yr?Wr-=@p2NYJC}hWva*L)^->f;N0@-)v^{~sX>{F$RX?slZgzM6O)pz6edX*hpQTvm&4+IZ(*@%KI<vU92&Wv4()T<ddB%)-y9Is%hqdY^5yKa!!2{)%<>*5hsjuMfo87vtRe*T1Hu#ZXk%lYJ#O5JxAtystnb`G$?o>;?fPxBPW!PIt$l4-!bu<k-w1ObDQU#5de)vc$9{&{<YJVmQ=uLZEnEH5iOh|+`r4Zv#I6`&z~GnMmLmTKh>}JZ+u8J<QWZ*o*s?TsQT(Iz-8+rF-I$VcHX5rn1UO0suAU1Sp$4kbEl`!>TeZ9E8|!tSWl%@%2(T;((8PJ`piq4H^4IwDV{%x^2xke5D}ncH95nQ!nK)>VF9JWelHp+W2END;q0|`7fB^sry*C;U+n_5TqwDbyq(hM94^Af2K{A0OU}tsp%8yo8Y3~#IgGf_RlpeO+ejN{eQ2tZ<_NmIi&&k8m4jE3f5A19f@l*zh!MBqTKQ4(xcbWfAA*z2-wUXB>8R_nxJoi+BduCp{lSVERi8Ip3yEAggSNd>_U<xrxxT9`#FFCD_x_IWLNQ-z*wE9)Ff-h3vop4}@N_oZxc1uUC34inb=*0bk2W6U(SQ|9#LDIU{Kb6N}($O%en5j>Vfv4RvCRQ2HmBT^z{G}X$W*{`SqY_Q>ggUbzdc<w`;=Bz%GOEN5IZBU~SAF0FwSxq8?5ZbSbA=zEkeupK`hH3ok&DioWICp#xB>GVc_x&RI%+n|ON^y<i&!S=)2j?w99HzyxndTDePjL3P4xqvZrnx58oS<r%dI>pN9rEW1@rEtj26yga%N74qgXBlkui&B7>%~4Lc1dF_gL(PQ0*e8FizYX#zo&(5NR?!BCxhXr0lp<FneObD5Ll)edah{D<MpXz17`4Bxd8eLFU5x-L3WQ^;NidcOA|O0uhAcVta2Zb=+FQ%k@|%Ct{3*8_A}p;5@mB=HsuDpGd|~c{Ff&Zf0O6F4BwNdH`^qiYQJz?IV!&yU8}ppeWd2z>G-X6H^Yy*A%iL+?<hll2U+2Mx2i(4N@pMMqw7FJ3NVrkB?U}n5Sj@7AH!2aGD-pz!Zv!l5E7#I;|esP~__H6jO-fbSkrQe>Ni(^5)vzMxYW17kUGbL19Xrv@r}EVRRmza#l<fOwgm(_`&Yo28Ak<PmVOs!`1=D<-+C@XfQsdaXS9d8#0XFQ8!oU9(>l`#;%1I1alM{iDvb8-sLGqd9W&;h3?kZ_QVs~2dzV1Fk`@-*-16?#k3JIJEu_F)P#}E@gin{Ehe(mW)(E6V|u%gaELV=I9`+Eq;<$f5PZ<N?G|56_LCCkT|Ho%R`On9vG_eSWLV7;n09%VF0UYV-0M;%>ctZ-n+)O?cTO-645I)FIb~(^qG-yD-hwA)^%v)f4F~;Gp65E+9Ii;ev=yZu!3-bXJ7~4<RpFQ-NUNYV6%@`Te~)29Y@=_^2WNaKC25II^nKXEV*14q8d&k|R~fC0OxPfm5SvcU)NS_U#`as?IIn7cA}4#&>|M^G#P5wL)lq(~(QBN^*aP_yn&VG)Ec(;c+9+}t(*5N90;kc%3if{+Ry)ai1^(r!NEyR@gX@^X;SnLxB3Y+zMz6{ko4_E_27P6m;Jh9LTif*oG<%xq6?EFHfL?hY`BVk>H&J&^JoB1T2x+%THMT!xvwM@;i*B#pgw!Z(Ew?*i#_>wo4{9-;RH=Nehr0W1IzVs`<1$4~2r>3E!U4tcS}pk2gnoCxVTy&+2Cu+1GfL-(&-T>&sw#grJ|^`B{db?FuSto&G&gg<_`UoPNL3Q`L0-Oo@q6l7`9*In`p{5iU>`o5#an26jI$x~HyKh7{v|>(ov}x55s>3aiQZ?_N%Oxb>t^LaDo$%O2XHfJ9+sOWyONCeAJT-kA1%QIM}j~ovWUm6tfSbRvF*A7Lq5yl!|Fp%lKhN4*QtWHRE--!a$cB~<M>_c+zTg$>sS?G(5Jx~KbfHKdKyd@7U@jon7U~xGj2png?bZz_?ma{R8{7Wl*l*NCO<bGfO{H5-T`RN(dzxwDyYhdHqXn#4sAOLS-w_gbP$Qz+$#iQD?WI&c6V!UCnE;R(ot^^8WRR;o{Vtbk)u#GWgcIlNNBgdz#6pd6DM~G!DKl{nbLWjUnWAUP*Ui)_!gP;mdg=gG}dopPvV}syIq6-64<{V{6vWWoLo8+k_9^FTre)h9A?q)2I~Z{i~<0s&E88+CsYfkC<pB$bn*qLRw`-1*Z|tY`>k;gVY0I;L96RA!A(CpTRCV=C{ce>g?G}cOGH;dk7r>L%KyoN;z=6~c&T)5yWB|Cn^U9&C-5N??9V0dHH+s!3z6Ugh;uMs@mU*4vyxxye*i{;4V93Alz<=!DXfHYHdsPFXo5U?d8QzQLjGKuQ3(c_=U^+?wSlrOXZ1c#ndZz1h-J=bK?+u&AYlb)pyTi|R&_a)*a9<(qnLnJ8>x;*V(4>_2MBusDsc{)iack|@+&i84rImA;RumnM`)|8UM;SSM=pB7zHV7_o?g#ygK*bas0SnNc!5H!Kq>BGH&FZ?2oi$u6$IQi2urO&`*c><o(}{e{ig6#pA$AiTf}VSjG0T+0LH)|1sHU0GurBLDDWn^_ZdG1L~9nlE8=X6ILOAhmdxH4<Q&5_XihVE875g+*>A#yUY~pyOUu{8%28`_6yp`_%Ci}ri4oi|Th3|U&?062;cnIf|GzLum5D_%&u*FrB&OP4u~2&E=qMB+k@-?!#+^)uhlkRlgkNVAH%D1=5L@?QU7_%I@uEcccuO+6?lOuv8?~FTdPry$!s<qi6UKJtuZi)h*2;@oS~46l+nBjHM`1ohD`SDqIBO28X$T%D*2dtWt&4%bVd+X}oNBmT-`#HPY}HrE|6=3h%-Xp|hx=hSLAt!r(WwzwqvghT!xax;(BZ6o?Lt667v9kV6DcUSKcXyDpv;IuA7yxazek)sVP$=Dt^Uq_2qy*lK4RF!2u;YA2J+UU<?;wsKAZg!tb?KW9!8<j-f$2z#FyHd^S%I|M))taXA47%xH?T-MIvQwQOAFNJ#c`-ncs@<*6Lx!mdy7t2oq5NlQQ4dU4?>mQ{K-lKUqw_yQtp{Vd6}w_PD+XQuN?t0)3bD_=3L$IReg16eD}^^!(;wL{qB)A2egq`C*I@AK@uNqAKtmWcFY@QbJ4+kP(^=h8$=M9&CgsFP>Et*`m71zz71&)S)ABdBz=l3<`{T4h=*E1BS!noYbF_cUL6le@di>2+5-WBQV+i?*y%+?Vv_E&R8gwRIr6_37ILO6w0Oz7`N+~p0jo5-Si3Uu)wAfmFN?_kQu6TBw_`@xTX;+Fj|d39#VX`FaHt8mDxs8-em!5lFI7&djnh0#U??f8!tvd&dz-4lF^~&)+Ic~i$OOTV6K^zl-oy1``)Ay91N#NoXJ9ViDW$PQ5K9KVkbW_ZZ!+IC3J!#kmSc0w0xYvv)VUd*ba8K;1tI5Scu8kB0GvrQrfRcHhU+;3`?)dyI%Q@9?`HtRqh8j8kOAx0=Jt&c$lAoC@rW6Q#z5B#9~QqrGB?6mFj^PCj-@Km2U4ZEH+nIhN@0a7f|lG)7W1B>&B)B*^?eq?c{b@Brm#UrlmTgmH0m+O3o=a%@cEh!Vg6RV|+C1cR=JRP+sw*)t_b=s>TV3nrLgD7En7t{gJhWLIGd99Dcx&<z%@pmXZN^)#0zB<YA|Wfl7tqe&vm3w0OPvm&?^^+0%`~>Z7p^PZYgMkeha6$0(sK$1ohX*Ei!kjg7U&9>`ptv!m0oiZpvQyWBN6DRN7dQqQEiUkG>W+uQ4y3dK<$k#F?Pap_HZgTin)>j3(jdCQqmP$67}LvaUD7v?STdzlk0^>nB6+P>p^e{-&Ciy644a%`gBgaj=57lkznx0M4|$+UR$zLeft`qRSnRCbM5TPlINEJ9tsCo7$449}`Ad5@|SACX@S!Ut8|6J;?paXeM8ux;TL4GYLv!i$r|S8HqyF&2qV^n#;jJ)^5%$gLAay;zcx@V}9PCR&-EqLLvFvl!009!euy+x4~j>PBtb5I@#e<2$vTJK<TTJ3(~J_oYE!yANPjl0E``7x#;t#CbVJ2CSehB}px5#e48dnE{I^uJ}WqwJTwZGp7hRL+St+Nif2wya|RL;2jFIq2K~`HB5q})jkR)qZX1j2vctc2riadJ?u>>N0*h<f-&ceRJ&+8j;4vyozG#z9$<`dw}qBmPI*y!xBMd$pH*aFR5PJvvHdwKX+Kn|6TApxrHT|P%8EOd0v6R)KkDynZ8WxPcrb)nW3|&8bo+>J@(@#jp_f)rR->4Z_R+A{PF6%jRMPJ(zyZaBMN}t!F-Vh)QNoAl0gjsyM*lED{k7K~BvW)_U^IO9;5f{4P9bkv{=l7%GG_zZxs6%iP^X>OPm`J!?8zLBj{6*jGWR@DBR<WXD5tZm$TSb=6EBCP88@rPQ)`mQo2(4Wd1XGg;Lffb4;~ZC72d{pjXI`DM%yO?6#VH2rPU=5XAIW*&dy%88iEU0DRTB^bgQ<$vA12H$9Af8IXMQG!yvpY`l14V(rKPNES6%*hK^%2fxmas8&iA)94jLj5VaXeA9j{GXZEI<5y`z6iz;Fz7dXM(-lFFT>-N7`Id=@Qd5Vj}IipP}g&KEv>`zQwd8oIm68tfqPB1OAAa<^hb*hKBRm@v@I>=tUc)Pw?->z+}|FtmW+E&5Z+x6P+PDW^S3NLnfDNY{3Q8FR7t2Hu4-_6Nb<-xp-4?z^^e+$UB0-v^yjqfBK3BOngqx}Muq(`HeTet<R{C|odebeUg@?!y(j2y);|ACAa-!NUV#P)nbP|RNo4r9g#(W!j|Pb$HxZLH>tV%c`ye53PbYw68%V%h#W6D`$-l>LVC1nC3+)<j_?6U?s!2JEMlFz}o{^GmYqh-`(AsS`irGy?<d_V<1;a*0y=T(B-Z5dd<M5c)Up;_pDR;i2GX#7oa95y5L|l6^h{G493F*Ni&4+`is<<1o4LYN(@cU#kQH@Sp?IaJMy(1}VuAPj)DQM4ALF<>NlCP+1hT$|U0aaG6CP=yd)-ATOQ|l@uFC6aV7VM~vx828y4cC{cLGYB7Gw7_!3L`Zyc1^zCaP@ZSXau)e*1x4yZX1zd_&n)}ycGUHJ$zJ#)uXJs`EBMOWyyC+Mk3VpszP7acXAf)$>DGiGq(0=1Bg`f(U9$y9=AMz4kU!UOeFHitE?~ZUZ+~a#N#Poh>=^El<VCr}nVQW}Kkfr{pw!N8!HNV5qI*7Be6fee$!4XM7Aif2NnGZ5t5EA{&Q;G8&iJcD@U|0%vk9re~nM5GBmVmy$axiWUKov4v$i@_mVL<MWhB1h|h^{7}1E>Xyi}o06ELKDVOktcsL2_jnSW9|kcp%TP@=5hj;|MC0Is=)me!IS1Od|{i0?qCxAX&wISU`OXA<(?xjJ5e<3R%#&E~>aFiXpwMgR0U^bnZpY3+PqPcph*0YSTEXxSd)r+qbwbonV!I-nr-2=3*pDlq)2jhV4AMyS`aagQ=Z=6q$|ntTpyvK^BVvC=$Kn6ZdQG*4~-(_0~gt-F!O@tMSMJwo^0SS-=dVxC{GBVFvtUv6{CE?Jq&>MOYTFLxC~|3oMxwK(tH7h1}^|j6##1@B_jT_`}Qp1ulJ=23o~BX23tvddfg}w9y@-YteFaJ-QLSVc|StuT76f)zm&v!t*YkoE(DEUkFRYG}Y>>#a>(yZ<bC5peo)g&@>!$<KaE`7*W6~or98uRv|{>A!zI&X@ha+z&RM;ih@@8@ppFWyMdrSAQ}sV1-bUd-J9W$zXJ_19u4uy=@CwoNp;cf(Daiaii@mhD7)%FC9d6!ckgZ$JvCsA8fJUJx<KX}Tz;@jl5zDAb^F|ECF&#Xk(a#?jiP5S1k;G`GNxPAsy1}dI3o?qxxTSm-wq1AP@ENCU8H#?mUGp!F6B%WkGxea)6d$?or1Wu026U@P5vqtzrUC)ydfK~w!X8wzUjf5EzX0P3g5+OI<n6!IMLKJS4f+w>0VqBEK{-QY5^}K#F6y#KF#g^HQx!zcuy(aO+<<y^q@GFO&A}0;}l4T3YtoQA*=TY9ZV7`!X`tq1p~NM^7b^R7SA~^!DO&HJ+~02RU{V26S56XkziRHy<$|myGTyre}B;peGWzXvh&;$)+4@~qmMB3g`(x(iBVezO@8-X_*>^AY}DVbZ@?8?Iq;7jw%vEP>TqFi6BPXIoppE(ueg1nID{GOX8U$+Q<?}2ur8S463X-DySgFaS+i&hR}yIX7W}84hm4r+pts7E3_RZRY^3vSri;s0ub+D}mAj3J7o7oq^$I&naN!Lmr=%2CvU{xBr|w_C7X~SGWcmFJeC+g_ANZKMOM@S$H<kIokW-{*sK(Mo1gK+xg@(fMt8G=X%=$I>Tm>U4z~Ts9H0%!t=m`%Nml3e6xDqT~y>U<HIQIsixs@_KI~dD92{2~L44CdATNl*Q!vL{hI@DFmedmgShl&V&QC1C<DRz_uH%iMeE&6@pyAb%Ln8+S%QeNtGe1b4#2!waiISq7bL`Kt7>(v}p8oZo=^W(U=#VTk5^25>8L0hR?bJ2@U4P#F;?mS@F|JlDO6K@-K`$-a*jstYp$O$fH_jdh<d+XcvwF>&9I3gd%w>72|{ZT&&$m9x&Y&yV3?zn^EU*xpe(>hPf4!LOxG;^b|vs1wl>-Io0s6L{4&LU$?B%SIbZ;fYRCZP5G<ya$*^Lo`Fm9Bu{fpU`z&JZEE1ESDum~q?RCq9_C!*Q$47WTzbISLQp80#RKK8-?rLNIpZ`Ld5l;f6@n;i?0NRre8pu-0%K#FT=Ma1To5_W}AkYPxj=6>+4fwv=mzfHt+~(3eRSAMVHPW|jXb9m4K3XcfXZrnR+sJB~vu{K@^!Uebrt_CKceZrv6UH+=y&r%5^U;Rl#X)Eh)_QiD8DOux+>!SrS*4&%_t$#GeiYXJvU1e|val2;(86N2LtIDl}cKS%=Q#lf862}cINBSTol(@CJ=y-KNR6CZjYL~>Iwn$f)^8F^tVsl3_L=8`M<oMYH@2ZU^<euDM+n~^41=j;B6!%PMuK^Ua-7~V$EV3foWtx5^NnJ-$|6v`4RaEArr`Y!%af2X#(%lrG@-R$x_SqZ8vp&{h8Dnn_rSi|R{DB|#_$yB93+)oM$qL@oa1p|<bA1jL2tYTqEQal4Q{G*jj7GAd)VHJmNtCJxbhJ=iXQ(R(fkV`v6uFyoif*Z^g$0#XQ?Fy~f#+ssO>J26^^~m3|z|%ZapqSGWAU?t~`%Gf0970*;L}1^s=sR)(=p`W4POo-;mL<w4H^`ig3^#{1jF5`d3n>2W3WYx5smAUC2Cimjif*fqVffrF*&Pmp!{mVp@?z{MlK{{2F~d5DZ0>%mYc}Pu=<*U@q&L)YV9&jt>5UV*il{tonkbsSseVcXUg@?$c<A>D<9P-X-;kM}PNu^JLAw^T0*2=y;zGG%P{Y6jC^#Gsj|tIwf$hSbpk7|G_!w&1i>MSTX{7m-QVI-?Qja&7nFl`+JYYl+8KQF43Sf;;4;_%)3n%7)HXY%ijLb$HoWRnY2CKDARD2KQ`OC+v<t){n9t<g@gIEm?eDSztAG<gbNqNa}JV75g);Op`aMo2{_2a)~Q}beZ>Du{C&G3C$7FhB^bZp|QPIL|*S$gwHIc^5A%~Rr9TiLG!jknwr@ETt8R4}*W2M<fS8qMwOIqG$f_;R}59D9=O$f^DnS#F?T&*LLW^I%yPWtzyv$eyn$z1Wf^%(WA93~UzxN=*htd?-(@;K>93*~kBs+_4lmeRNNCw<M%+FT=BRMR#)Qkqmf0DG|cMDRm6nW+@9G4t@|`)+mKkMX!@u6=K8VubU~ssx}(fayp?}->nFl_AIA9@z?9r21pdo63}IZ=aRnZU9b2}cWn7PT#;8T;TrZ@EmliP3G{A&_KAPmyc1z|U9jSC<`S%$Pzl8y<AYQiJOa6CJ@?f^6tz+O;OMl9p19$dx)8KLN+RZhBTytcMWLDwa_5^MB2`l%?tXcit280zG9+fGvdRzdb0z|Q=`_*TFeSE`9nu^dG0j$_c{m2i10AKv77`;#W~X&LLi}UsY@C4?kAU_DH$x5FmI!Kq|7XNEr*xFO=vkdx4kDa6wX7h@tlaXswo6+P<C!og8q|3;iKV!P4$#e-$Oo5SV}=-;nu5E4&Kn!9^bAU|=A|1}>@y}Sgm@;Yt@$Z}&sMbB@LpMDsJYG8*~Lpz#0~T)iNO5?w=_&|UlFNrAQ=h~*^rNW{XPiyTED?nVhNgm{+EEaeNKJG?wM1~${s;F$YFme8hQ%vv2*B5ITK@jK#>s?1SmNc%#XQ3LWj{)RH>;g;i)=9U&^UpD@y5Fg(HN{nH41J#;p#mWhCw~EqG2#QjN&)fpRO_2~i0N8af>61=7QCG<~B>q1y<J5Uj8GW|Y{OwbJ38pX}f^6hf*qvO7b>$MrfKsEIfc0)cnX(Y_}oMP5OEzlo-=P~6hhYT1XMYHef8`p3R=bB}T3^iz9NZcO-`aikPB`dowrf9cQ5Coljj@>2wmUV(pp#~Joiz{_6<9P)Hn=!K%JuifZeyLzp)bj~R2e<}$bJox|qgsAGDz4(fe3sk)G7Z}L!AyI2T(WKY^0N?$Qh$?@|w=lk$f}p*j%M^y44Ol>hek5Sy;j+(M@Z$d#Knb6E18gY}*NpY0wqznPWazyw{<*wV3SRubsPhJwE@8m@rAwM+^LYld=87L-KzuDdVtk)q@if?NC=wi_!hb~_^|gd%K)P~u<4;+E;m`FqJtbxRv)3Fcexuwjzj;tze#J=fWt?RMZ}AO<(0^?*uT%R)A3dkgz07$g%|`wa^oMD~#yA5o!<c`MElUz2P?(&nI)41+Cy{R~i5}@A#^89alP|tRAPB0M@g<(|GhYkT4J=-99}q1PO#_*VfnWD4;xiB{R&UEJ%$zeaepz>;yRj{nuLnf%^ij#7Z^G>@HPHngMo#}S?~~ELFP`PZ!(%4@?YOpGuYq1w+uhsAgcA{Lzqe6S3AHI1#3;**Y?{QDF^dSH%yJE9^wi|Ne)Ze4C#^PWFcPcvg4aeAg>k%DLoa6FF%{YGW`|fCrQ`=&XOuS)Fy*s~6wl?Bs_jT90b~?uKQt?BTBe<YI3VqBP9XF-;;4l=?c$NtKHFN3UG03K(ywl`)p1#K81KAM%~ovIJT~Aosbg$%InIiab9f)~%)q>0$E^od>n**IrLj>h|8(4qxT&H^bfG20BhRX_l%9psa;TqC^Gn$eGmF!YqD|4bE@xOq!5QYo^lmD3S>z*O4-YFSs?cJ@N>Be(!IXO*l>Ov}sVJKtbW7YiKInB%h9?tQfQ)uAYFqFX>Iq(PAE~%O4!cL7?t<ikfeHr#R3d^crQJV4V0u=}h^<d<ToEcl6y;#>ZnM>`<Q*VcdCDq_{rdri9@>wkm8$)a44~K+U$8=IPqnc2i5tDyv;a^rX2eHbaeUBeVVGXUf#><j1QCKxjz<&B!zxXO{T^cRM9F|Si(8X+uO~SiGEjS#t~E2FQ~XOv{(hZCVJ;b7aVwc^?pABYYwp%}GtJ#xVXj$PHE%Avbb68QhcaPOdL=|D_zjW^+*lP*(;Q{WK8pJ=Yh(nPRSsKi@+0GxY;z(`W!XsSF?PmemQB%K;n~)C<GvFkV^~M=i^CQB*_7qpc-}U=V0X<n-CKow4EV<Vk{bS5<;*bci=oVm0TRa@+xx{1%ce!2;XixCr5WESZw_Pr99oA~IamQ?rH^%nPX*53&o5|q@JK{2Kgm`zY~UqdWYLnE$7~p%8DQkws4}CBmM$7)bZ!H++6IhCe<2&YG*dTZJVl1R`ry<y5#kP^gU(ktzgC*8+ruLeF(#24#snwvbU1DG^RRQXg&SW#*w_;n>H*;fbEFu{Sye1X25^@Mn{gkL#A4@*)t>#imUet+eP_49l$Hv2R}7&c4D*$gk$m=HqC?x1nY@3h16<4*uz)jQ!8Dswnkz<=3o8bNVzH0LYx@-k51lpav{F_pHzYgV*5(ZJFM)GNtBMZjsf3a5yXBgV!VgS*0@oZPd@-1nW`)J0D@RMe?n!A#y|`gdgKpc6qq>lN_>J}(H{M7NuAS39{F{<sAmJ_6kGG0D^9<W=HZ&@lxSq1H^BZIag>C$6GS)tI;5g)I0cXThdh9v9^hL1q!wu2AU%&V*V(_jwcntj7XH@Gq%v8>&$;S-f!De}pxJSqwPZcG(!pCKP^XGo%@z;&a@kY{l^Lp~?$Q;7De*<$i?kyleB5m3*peuZ587(dex@}<^!|GMYVoDaFv+?|(8+z90f%DEDJ%s*Q03}_^#m>;U&eB+TwgQ)t?tQPeTh~|Zz_^3X@sRC+E)oN(kgy{0JWC!<;Q~FXqsSElmj{v|D{aiGvDBL|nL;Yw=eYOkOrE}c4Jg>haSE3O%V}16M<KViVg(D1HIt(FnXbKxRh@&%)u}bF#Ccgub971U9TSDVNrk7_r-arGr-iqB4dw?fz$h{sFPPqy#0_Tcx;T0nPW9cr?fN`43o+`XSjg+(7*&d~CmG2@t6_b~x4|$fhn|V)=u>TSI=e43IO$B<x;ba-d`weLS^buh7o8d5(1*fZK_6QFPoIr^F?a`mK^WsaL6WQR7$c~9zQ@ee=VRWmqGPi!;g_*o;FmcE;HN!ofxvUjfc*6EpGAC853~TAaabp9*N}B9qq0;nDL@_jR=Qr-my+*6Q96dpJ#C=lNP|{ZsIYQt90&hXV|_E_-9TD8{xC6CX7j4z15!6K)h<d&w+6JDrnmLK&3+b>dRrvJCuBC4-wx%WZC!PRc*5?Ww|oHQx`nB!VE9t}DeWR{!G{B7QhXY62cu00^h=NI;a#5!2jx8l$`QKY9+7>;1Cydemm;Rs7cb{4$$qpI@xd2@TM?{?vWq8Z%2&j(Xl{@US%J*cjRIO3uw#O{HfW!6G!ik}vZaAgbrZ}g_Ce;x%!=#JEO3^#v*OU_Xs_BLPIMSOO3`%|bjH5m2aPedyzf{43eyM}y;}#p=a#22U`XbhS-3F|dx-!rpRWR=tl8BNB*}S7&)TbIS0&k!=cp~O^F`mI9Y@W#*V6I+eorzfk(}b)X*Y0Y+ho=kgD}AuH~oIj#_#U<ZeEA))qNofEi5%JUAruQH7_k)X2-1D9Wv1n<ciEDs&=O;(N$Z6bPnF?EXHap9FdrHb%}~xmm4`xAhqEt;}1hmb0L=O-tO{#tGZk+eox=`efEVo$aRs%YJ97@WL9P`p10;G%QKN@v`)Y4%-m`<kKjDhDeUQI<&4auP%s?5psbI^!^4DZd#%1ckWo=nh55I4%TWveK?!~eoi{0DpP=KA<FQ`;l&n)9+4#2~pp67B|KbAyL;DqZP5$@+8MA)IRuhax`+$utBHHvLj<kFEaTv7`uAe74A?vX#W6;X=UikXYUW3`)6ur{=%<iZQGtQ$1XncoYn_r17CVG6xC7@rS6W(_luV?2;XcN`?<{B#RjqP}+ez&%{yS`ctx3(L%SonblG{BYsN(*t09}I_mQDs&)3+*u!c-txFa3Z70=noQ&{j$!b`BDS1d#ZnxCl;cVtOCLuktqy8F_8EE3V-VKKzT4kYIq`gDf6Hp?*)I7jRR&3upMH~M_>*tooYl1RX1&X4ApL}_X2^`&U+i9z+iE<ps)KxyZ8wlX7pZZi+lbS8VyxrO42M`8k`)9Z2B}Tuf?t?j$Ja6kk2LEMjh4@%I*@RgL(k;ok?=sLZ)(fdDw=9S16Xcs6xz`^svzPHfwL!);DUH8{={+%>|NCrT;p+kTe2W?5?RzrI=r2wpGzG5@s6|Rc0GuXmFtxcMy!C6o-n<l%+`#%g)C>o+S>$cV?b(lw+6CIt)AG*8PuZn;d;OGg00l%}r)1<%$oP0Bm<O)`B9DM-8$yHHGhBAy|a}JJ1V^w9!5pV~odw$gMdX!@)j2X=70tAZHj?Je~&l8J6|p`*^|BkB}C-P^7QKPqZJd#+pybczSk~=iul~Rf5(LjQPr`DlG=Hb)r2yK8C^8lJLCPE}Rfsp4ZO05|0+E@ciZdoGF%d+9i{dKByA=d3mM%)&b^ns_BAIR|O;{jiA<IR6y!d_!ETEoG1<SR9CkImwEx-N0_QG+UcRFxPCl3LBl77OeNjZYRD$1usD;&yAV!bE<?ZUOc4Wc+}XjOV;HkYQTuG;KZRC*bkwRYl}qJfv02P-Y3Z;x#ksQ|Yohn*5Q)NJoieNxCbtSY9QRgU;isc2RUo<KVZP;5l2_89u$M8{QF!H0&A&%HWM?HNv(KS=7Hszz0?GU;$Zw48Q@#T@3u8@<?vn4SbMOr2M}%Q|=nt)4&1*OFxmQKi_zQtt@?=|D%7YOu!ghZ+!4reZ*cUBs&fA+&oJ(9J*nCjjfl`QozR42q;mo(T>#L3JwfX3!iB#GVN;SQ-SZ|sYI+yxw)S4dQ5S5s%AcLwNJ|ME`?&EB}``r4Zb<v>2i+wzKoN7Eyc%;G#?sa*j)A7g_Q6CwhqLo+?L7PgX8r>P}c&0b9;_mN8A&z`FwOI$}tu2ehKCkB70yCKGFI=9V&OYaRXC0dP)y<mqIvll<PkEnTjS$A@Gu&Odm_PB5E+kEMACENGy*vU>N0}t|1|RyY$}YrjPk-m0S}t3g?ul939(>69>TBG(X1GK&^=2NYyxRW!YTJxDL=5(QP#V#gQUSfI{?2~b2IU=fHQ4Ez(WOh!X}MEFh)23-?zhGumQ3{~ZSvR0g20|FI>y%MvHh8QnkM<oZBbgz6wPyj{^aH2InXOy-QA6)te26uUWAGAFkyJ5VlQ8<-QC*Dj~JDZDymyV{HW@=yR|xueN{Gpzbtw0`Oc!gvncP(1zB+;9T99#JN@zeW%!x%VN~F12snQr2rti2^e=}p+>D4T8$Z~++YoV}tRh6gTBFXsQo;&26|_4Dg4S-GOfVmZR`@)46>|)#6(GA?zq{4gHp63`;j#wH2W8=@F_YC*Aa33v?pax*6Lsr1DfyFMHybhSOa#=70-Q-;>r>FO;$_oy<_FBC5d4-a$n~`_JX~PNx<CZo5DC~}0EP62r-;)KFdW~2g8C*J#zdpik~PAtKyos{UfVlJ@cll_k@0XcSr8dgtv-8nTd27uZIG6*QUtLba0|x4XgHa|sna>(Ho=-gxQcdi0WvWTUd23xPgJt{7lH=vIeu4^&Znll@h5@FNh88ZU;F`z`MFH`&W>}8yD0NoDa#n(h{4I<HU-Ru;)LREE-YUbeH2e0zi0d$gjEhbQwB4WzlLr?0G#G4d?`9NBb8fzcB;6Zn||a$^(JSWh{w$g<iNg@UBV~<83CT;MVWNWGUT;*)Sk{m3!@k(zFmSQb{@?21vl%vKdRR^1Lm$+ESGaj%IwXzs%7ngprz+Y7w#haa&<p6Cas134L^A1cyYv?Avdj6aaS9ktFV$v>Sw3GwVI~q5)!&v?dtMNjuqSUrfY$XyVN7}y75(7ShE7JQxVpJsRr#O_j~Yh{aEemrjp$4!tm?FVEdUEn>|7)m|`Z4o|t7710htZ7`;To?(+vBRgXG@V#Tlw2kKIJ>Dt2L(!y1gzI&4x2|0RGOczGIih=agiOBE;Z&hJ(acDQ!@mBQ>@v~bp*OBhLw_F1?Vc1UY^<nr2B~EHlg!O?)|G14wg*!(lhll-S2lI!VeReXcrXDE_IMoRDWs+dxN$>rnT3U+6(7@p_xuFsln$F$8TTy&a9V8E-6DOk*m4_c5QI_Mxog&55{*Uetc!r#-aZ84!ufUDOxN+c*R~P0lJm#<@tEN(K(&<eQA)jt0)=d{}00V>VRqk7QD{;GB-S;h0UcWA2{5$k<Xen(EF2lR}>)xTSLxY@s#}i+F4{gfuh|j<E4wikdS(Se%euKTuA8a;$v}to2P_^>;WcKs!P?0fUA9|sCN{zSYuVYViUq%x1>R?pDtO?GE>!M;q5A)zzr6C++p;+{`L8_buxuqWTVJ{IuF!*8^aeYN^f5NjixxGSjj6xXKgCPik-lR(Wz6+V@v+eL1ltR?VYE?HUD_jM$Ngv&-h+f250E5aEW6(H}MDzR^oMjYIquHZo=1rO$xj`@Hv>fEQHEKNBYUWhKCgX3cif`Za-p=~qVW+y<n%=}-*-l6jAC3zYJ4yzs>f#&asFRF92)-84j!7p5Ze*qM^$3pKgQ1FR$qB9zJD?AnlZ9E%$%37e6`%*>9-U2SQJ&<7$S4;|ElSUO*O3A<Kv|=!unS}r+I3wgH?y8>ji*(|oxu|Zy&1>owZ~($%YIP2)^1x458klO&Xtgg2D(vol~P1@p>HQ>28IOyYX}!Bd~pQj;zDVMQDczY8BXg2DKRb_!AKxu&EVc(_+U^Cd))zOZ(%XQ&S+*QQOhO7i@Xw}hHm=0u9i#DWY~uZQ4RZ(_r_V4Ne&aiw>NIzysM|&0I&6qDI}u0c%vL)Y`C0w7-#{L?>>y#;!?gmGKhbI;1r8uYJxmiU06KN37qLf#6CygBm6`DzN8Dk$Xh>iX1;^iw6o?W9@6ftgRSEn^KPD5g>}HO(eC#~C}KzD2n4F4_c`-upy7va7<m&)jz<~(!DT_N9Akpx@MIc4qJna~g{GGfzcGTs^PPP$bcE&LJ088_DZm!7^KRVYHQt2v-)biwPxsZ%(9GDAd36}|ryudSDAjmU6|S3-ODl?zrZMyY;G6Eq<B{o)JkEZ8h9h#>wC0oDq+Y%juWmPXc3|bL5^i?JYE~1Q=AAImyC;WqNQ(AmdnDl#vg+~q7`t94W7p$lbRC`Tf#c%DMVJ!0j#=s(k@JV;)_ir=5uBx3#rp^+4HD=cNX-umsL8P%D@O>X)qFalxl1d(;@twMh5NYmb>i?cy#|x}bmV*ZIBy!*!)bYCh|K6tePb<d?CsjKBSt+jbBb90kgjlua5pN)hoh)i{tw56)jcKO)mb(QkJ!@43M!1ChT9C84@8g6@`lVP^Eg2o$&|09dpy(4k~R;d-4TML=1yZ{4LiJkcWa~Z?p+B^LcW&m`Y!yY<!O3OsE;k*3aymX@Sv`MrzZ9*Oz&gEy@G{d1ZFZl8g>LiGtEEuT(SKs%W1#LT774Adwq+%D#%xBO`O_BmS;t}Q)>M_EQOOXnvSp3%gaG0nY72f5zdDdyr?;P8jT&(mQI{SongYbJgBUkYOD7Z3HYOo;>v;2wS5`V@8FDArW?apC^&4jTf6mblTEA^-$u`Mwku*e*A?M3<D8y|>lKq&X@Hq{ZDP+MW+Mc9qj3)i%Nj$~MDI@EO8<04rH=&%Fh2K)?q?7r577pN*<>L3wWbaW&mfuFQ5>7~-5)i!f8Y@D418f{3ipbNDJrM{_)(&OglyvxK1R73)d?ji7OeN=YrZ`fX8C=P&4&+g(|sS0YgyEOWnr-y>7R?0rk^CEDk|HzD#gH2EWFs%$B2Nn8$q9f?0e6PxTbDaU_<T=5kpN}@VSNKL3Zu%QLj)r2M)}r9Su)NbTX0|=mShYw`<fn=R??hM9R5WVmumxAb-D#kgKinaccl#4p&gJdm@sToi^(HalT)Ct9q3a0Z<qlv<4CXQ_NtM9;CeeR^SlX*l0+dhy=QUy&Z)j<WsC8aSDpoWSy9@`LV&-ftz$FEFLIS9!xU&Vj-(9q;bjZ6YdbvAG2CS|B=HX+k*o|#X=#y$sq`82rfz&=O}wHJSKF|5eKHy$+p_qytR%=u#N3^RV}0Y=}Pe09CgTzE2fPLCiC9rc73O@@pgR;g_o;wV>@o_-hr(e!<Q6;t~JNqjrw|w8)&Q#5L*l|rjLJ<z7SAR(z%wr**Ywr6R7C73hD+S#=l1J9s<;UN(ofYUOY#fAw-3HjzB;!9>4slf!a?u@yTyq{*utCF>&fiKBm!Ud3Z>lnRs;Q(?b97V=t)vZ}ZU5B|wph!Nkd70!tenWY@F6$YbQAub{j?DzM;Zo_5M~Fht@m1uuS=1Kj<Ez&l4AdxSnBXrz}PzWlYqt4Ew90lgHw_%t9CrH=z_fejY^7}5L$7!yJ&5#$ei)60J#uwrPWfw=!GL_=r5b5RX_>2rc!S9A;bao-5Qc<lH;$uG8=jD;&uf2s>V?n~^h&*7=Z{4ygysj`V<;4x+3J*SQnNb*sD9)0rX|HI3lVwN8IECYYK9X|RLyPIL}zob4^{j0(T5IA1U%fI~=;G`B0uO*%K8;7q0zMofF#k0m}WH05-PkiE1r05M?bVr`$onEUu7$O$06$f%V86b{;cCsVf<ik^TqG24&3ZcohK_@=$wHHJb5oRSZ3^3eOVILx-2opJwWwCQH<op|zVAjXVh5I#68D}Zrw9MaRo~3t@C%M4+I*>=GA>I(8b>u2rzqO9ZS1|&M%1kWBCUQ0#*h&RQia4@WP=VIhKz`o|QM~N-hX<4vrm}c4pHG2d-3LXXn)$dN=L+BPB3<DPfpxVapGrYQRVS@O4h@zsIFI<MtMa4d&ZNII+=zh#ZbObpXm1;N#FtPlp0dG6Gzp>iZUz##EbjPQ!WhG%v-<wn?EB+6`u=!U-)o!aOpG$B60M;UrA%J%b`IBmajY`CuVz&78o<k-ER=Iap+YIad11q1dCD{}u+l0EqGg2{by2pru%rvLW4!E1S2%8s3IdA!At?<W@zud6FND@UCd@-0gqI&?;*r16j25pK|8lunEoZmb$pO`^w(sOP$~|Q<AiJ+Dypvy+0JM;=;s)|Jx**lT%@T!N9Gn(XACZQJk9gk703Sa#yu_8j5bbI~%@AR02$gA#b7DQq3*$*bpK>Q(WWz+kw_0>9tPD|fIX-|D*&CVpbkQ;`=k}%_Aw++2sN&VRov(-r;)2xP0PPVz2=zS_P((s?uTLg))xKE<RDAFN4W9ao7#c+VG(YLwm>)(t&M>@{!r3su*|9}g!0ZG8AaRB9K?$soMp1wVG>}5}(;1)&_{KeZ-DIt0am-S+-^*I0;vuT7nVm7MYQ6WGc5YcJo3XiMwF%s_b^sZ~fltdxaZi1@@7e6MYFVQh!Y0afb2BcIZH;BAz=lVdF<aRWt(<UAAI(GUi5a&ui;7ZvNb@YaozP$TS->LoC4w2$ryOAO>q@<8ay%*I!L-wv{GTHS`0DH&U~Vi&3dmWqPoM5d1eOmt;ELx-=7l0C^h{Q)J;gj7GwPmEq?;nF_hrksg<LH>x|c)v;Rqr5ov${P%c0<Wq0J*pJXtI^FNZSHAgA?$apAC(JnT)Ts5H?WTo#G0IR6aycm;ZYk_6kJ^7oFDIwIK>ST|k1AsAzk>fsO>JP|&;Hz9HOym`?)Hg-%0jEW~MTJ%Z&#5G;6wu==NQ<;zxp3wxGH@g}h^(9zi3HjZPjx#IAT1?4E7wHRR$e*5h8v@6mWsjTm>bn^(VnC036N<7A=S}EVlc%W;+ixMQJB>Lc!HZ_(aqGFTvjs0|vxPAyZZ)7iHT0qCIVrZ~jBt#3a>rH2WCv?dU!?*oXzssq_P9EP)8QRrcAYPg9Ovy}UNX6T!oPY(xhl^E$<EF#pXZeeW|_Zo(V6+?uUv9g=J_iZ{Cbw)yqV}+9PQ4V^~T;#T-(`ctghGcVyE0Jb@|Ep62sXXB7=bUO6RZBIxuqBl>jF<_H&WQd&7ivGW;BS*1X?xvGjPegjs@Jl1i2L4~Oci*O@$j(b<{W@XvW-%veC)SiZEMJ9DWrgga)ia}Up&x}b0P((2XM*=h8~8DT}3sRLr>Tn=TltDTIWIu5L4{QOVJwsvs3d`5acfBgK}`SW-(?14A-Ba_-mNt8LgWx&}nf`Srl4XHLH16xH3`+3E&RY}dAd+CXzI=q-Y=s6Ymr$fkK0^COEZ6n%^Z!PLu3QZud8o|d@(O3>?U@nlLWA?R)Xh9*Y%t?ZVI0Xb4C`J{rA1y94GxY2njA$PG1G!(A69qvrX_}xxW3IVkhVI>!-J)r8*x%Pp`6>mO47njyWL|5sFgy&}{Z{Xo%rJ0|Vx)T@|G(Y1va{6?U<j~eGa(PMR*K2bwLX}f93J+Ft4g=kM4#mJVKRD%^-Pg+p%3z4o*FlYi~%5c!E{|3Gc;(CjuRw<;Ys&MBe*+zRW=)#yY=J|g85rKKQGt~!mH>f7<k$0Gcj0JFPQvbYOHx;jFV$T{|#CoT8vr*hJgHF;f14RuD;p6y7<Q8)pJG5yv;!}pJzs%JYmQ76Xu<VTgF9=Yw)xZG}fYkf*QW2K#I@Nef{#6BKqRxui@>-Q8w?>6YiDQ8|(1q_4d^_Z?vz!VywfB($!LVzHTylgq-|@=%8rNBpLTpia5du62y-n@4uuPdm1p71R+WMoPtQiy>kX8+^7T?twJ$16mf*X8M3CAe^0|jF*H9H?T7X}Aq449#pk#&DleX9jUUIyd_&Y|pVRmM5ns)YPB4QxmP?CQmP=Q!P{6Jy21j&JM!&$9*g-rLUh%sZzkc!my!fv#{;vRf8Y$Y$V4kOL_{FC$e*w3DMxjNGwFUex3cvZ86?F3AcNBf}3H|qDc<Gau|M>FnC^!rwpg3|P5L9ItgY>zK4H66JQ!4cDINk$;qoCtI;dr2@;z6RRQx8bGIf@J6;E=zATTFXL-|&nodlDE?C%?2pl5fPrPPMTXBf7-){=UyhE_8X0*@WL6%y}D#gI!)63YJgJl`Nd%9^e3~)a%xVt(X-&IN90-elR5vMNGs|4ckKmcL-BOS49mO@5_LQ=6PC{RhF0Li)^#>yaC2=(Y|dKORd3a%5&f~4Bmk0DWa`#sfi#rm{f`;_e-4AFMlqA>i+yM6ax4qfz&*PU*a#fPo3Cte^+?I$ltyZ!`wo~mfALk3|S5oCXPF0Pst4(Mec_7jK;GD6rFPI(}k2XGH%cLYU)K~=dQx*GDKoD-rv9ILJ(2pX_-vNl{+tCXnS(MhkzH7R9wVo%rRZ^BQ-q1<!H`C&x+!y#gE6t&H)spF1G2?ux~y$cHJxPcxwgL(2$|Kjc!Uh-DDEW?**j;sz}`yh{|vm+T_74=P%%g5p$J(M!FXjXMntS1*;(+3jUiLG98M_eT7h71Qde*<`;rGl^MTILJ7sfi$-%)H+O!?o(~Ir(Vi1U3VOG~*hRBuv}AH}Ec-L76~0C=a!fI!K-IR~674U@E)k|lGKPnzlID+2C#k$QZpDq8JN0dhR<Ez!uJ35gY`3us685c)#*aeZKw4q%pqYb+gvsBXup&4UU@nl08^@C|w&5?DD{n29mdV2=FMPjr{YJ5v5fP7jjRS8TPm;j|w3Pc4+F|J&?y8wPi^*a~*D{u3ycL%mfH9O%#LqwKK<k%3vjjZ&Ng~AOBtSeC!PJODhBV#<!Xu;L@uwtgJqb9B+e54V;&IUylYWozP534iOkvV2a3QPvV+J>i0(Ec+xHFP`e@@@_l^Y<-yt-cm|Nh}`XM|2W>IZ2Y6bb*YFa9|ojH%~t|8U_?O7^LSJkOE{Jl?4zRF%UZp`V|QPgCKlH{ztcWI?n+zZNst9Ak;kmdD@-Mdre&&CeT{VbE|&xfq$CqK~RnZm!>kbsY?(A&t=A9g-@pvtz-5!5b!Bc--qu@+(uN`7qUridAe@iybB|Rha&6+SW{jF@x9-s$sE~zw@=%_5Cu`kaUTQe2yw~4@)W)DjzA+&CZU(IL6k;aY!9Sl<FKP=PUj~xNq4LGczSjWNZ)%_Xpa5ApXP+?E_l>a=&<jTu4IEC0=h(55qJU8AG`7+Vqo)(JC`TD?z{+5Ag_L6`||UbC!CO-k>lAEg&hN^Psl1wcWsj>+OcIe{9z7)_02js*YL{d%6?h3snl|dpl7Tr&mR(J1vwkl#OrGB*%E~rHqdPIVr&)kRIayN`iV28mFy3)Iw2dNRUCJMB|7SI6NlFyfT652j^?RWt;|R*+LR9%&nEoFB5OYXw{0fIdHrF!@c!wIMxYEd*=?CE$X{>;E2SG>AlVMAMVw01FmhZ|211?MV8tf8zWT;q`s}1YBtiEAGVXxG^o=*XoAIUHhJi5vrxodJ=2l3gpMrfZh(4qov_v@h3?(>R&9M_Z+iw(cOXrp6;dss9mP^ivnh@*hm|0_Ow$xD?^hPDmzy&Sd+($-Mj;3OL@|s`aBW%b4bVn39K(pUTWJbqPcBD=mCr26$DD<F8g+UTlD|Os$rzORr5-X}2{={Co>QhMvWSpG6LGfK&q=!ZM3_`xeiR4@)jx3V7blf{1rMUx<#Q_JnIe~$hCDR3h{UX~(ZuOkP|ye+3(tYdVewEKvCTJ(9-TSrwJ=SmUd}Qbdx{Wll>upIn03DcXGTd}?IuIfKGb(Zkxq)h!n=EG?@DZH0S>7$Vr405s%aQN<Et2)*p`0w9R_fZ(^vb6I?1tgXKNxWYyno1Is`)Em*sG2`|<2-p;Q^tbz(5)ioRmcIk`P`uez^GOFbME$zm*5y*xdP`QZ4gOi1u_*k&kQ6y8$B&GVHgL6hZ|P|29rhTA+YS*iLMANv-^`KEki+F4;a-PnKXsXL{`*)ZZ58m#lnRuTN75#L$AjcLb*>!0Q({IgYC#mEM?60y4-lXW0;;MQY;5RY%Z%O?Z@ZYV9wn8J;{UAS(-P@FZh6gxd_WjBiXL%x~Fw=|-CIsKbyj}hA8h2_?7Bo7*EVT98=ttN(Si4IW0dHL~5TD=kanttE2?WTpSa78r3)Pq32qC7>^6OAnt^+a<6g?$(!Y4mKmu(#@3a`kZW+R~fn%<cZ?Qjn7-ZF^xbW0+D=F+CSkCgH6oBtbr7-Sy=!(Ydo|WMKauZ?pC(b>F9%y(Nw(tyibcE2X3UD%CKV%1r3rY`jfp!P@!EE5eyZ%DCx@pjL7-UjBU`3<P8d#=}z9^ao7R6?e&*VkM~JNLxDji!Y$7{|&18neZ(93JMfOeF|j|hWTSMMnAC+RGISnB%r)jc4|Duu717ly>|KF+Tp>q)~mVqVE-#G8%{n1;prrYXwHF60i@ms<o=#S2&g0c($k{^F=IgNM*x|(Z-4|0Qr5!%&)d5%w{e~MqOa0Ps?uny36Yd2*^uEBL_!iKyagcTI0#PfCIFHcfdCDF5^06I>|~OeeKIrN$;_OZnv<QG*|pDK`<yCSwqn_ll~<r%VV>lCmv!&e04OWtRBW)j*L|&Teb?W|WPts1$CtI&6|a@_ZphupVlU=xXCmfH|MBrZ$a^@?0#{i>pq_0YsED%Eur&Aj7tBMDc1GgX1fKM`z+;Tb4~8!cg}$^XfteO#=@t>j(v7AwroYJV0I6z1z7FurwR|`|%ktx*2O;zZu6e#1*eyrjsnoA5;~eT(Hs&h^KG$3Ef*ome|Dj&>+gG<cHu&Co5-ZjpZ+-p^T>i@9V(WYwlF`!DYgccSkg4zDT%hL$43Xo=j2QWGZ~}}R?oCRt!HWj_le-N!#S$SK6*HJaIOTBnY*%Xa_iK%6Ci)t;04Y_=&X*K+<UVusu<3+AN02q8J=pS8K71f-P`>&LxeW~W6M}<zuUvdPF9UV{KU3=!Ds-yZA4jFx1{y&pIqV%ACRjy3J!UZ;Nkls8jTwi{9Ox|PH|NkXlZLqD&|hSt5)?SsE88eB+^&&RXl$X6As?|`Mb6E$RgE>jU0JVeu2$C~;vuX)`qVQBl77ua#?l*2II<vm)?gTrYzWkHYc2SwJ3`Eb<9^y^W-f-^*bGTEiVnv<DL34L%}U#uSWOZCfcZSwi<ltJII>IQqK(^!!0<w=3**U2=PY0|#XPfn4QF?E2dxWgR&a(?KA1N@zp$FXiw{mPrXwA3`V-Q}FpL{-AKYW4D@H#p0aT`PYR)r@!p?(fS@vPmF-iuV8$DZM0y%jKkkzAffK)~Y*kNhq@Hf|d-azq%;m`DZ-mO=*@BLY)bVqtwk;oa0hI=RD$p9{2g#19KMBFZup*gu~7x8Af)TF`+4!V;^!_fn%^my1G9-Q$c3V;!EpHma(q>Bw*=)E^U1R6iAZEusqYqC+<Sw;04C8-P5oxC|D7er3E5YW8gp&A-PLOs&hy1Sh@$+@j6olZ{#DdF`=vx^*dU|7dW!G7<w+gT8|67T2*ocf@Ef+@?i>Oe#RdO#@TLOUT)HEkaT(hnTL159r^74cSEBZ(L9KD;5vdZ>8c6RAQecL5<HJ0iLY4o1()2_#vey|j;d*-nDpN;D5586(35Ar?V5`@L=-SO#{XjoBMMR>2l3h#*E(5ay(o78m(1<ffJv-3wVntKs%Hn)7+Y`k6P?qPbh(_-&xnxV2kft+M5H&K;nAVPR7boa=qRtU(ybObq`Tbo+%_^OV(7VlX7$m)~xf_iBxutvV?r)+=}4$OxN8bXBVie_P{eSrJx<!VvxpBS3qDI6du+%S%zVJ`pj|D)Wz9_)JIVw94vtCIbktgt1Vzmz5gp7>@#niR>G@O#RY(P5$uAKGlsVn47~v_gk#N4)FE*1|Sl?k3i6(*YRtv0;b`hE;gCT$4nlWfI~h*4`!G3WI?Um3=P)}AfY3SkR2RQnu!GU{+;yb7*9NKga||-b4qmkDYD%_|IGvwP9xt7wzoIduWsJvEaQDNp8;EfMz)mhoYTV;Zm8Zv%w-025BVLqoPqqE1?2Fh=-Vx~V*FH+Ba_0yQgp_qv=9dv#M1>C1pP+_KyFE-UKk&D+bEp?wd=MYjGgS|$zB(!E&;`gC?6m04%18Y^5W_AY>nCS?|%4L4)YwucdqI@Uwl=O`MpU3v|VgGx#_q#Ca8x0gP4k;TMJ7%T1#!pnIgjgE%R)3r7j4*{rLE(@5OPLbqF}9*5GthdO?*2(lf~LvAG5dVqwm$RTb#PberkSLB#7Z>hNW_8gX^Q<{O{vX(mwS)?=46ibQ@$?7RpWK#r2gDW`Q*)}heLF4)4S;_7_Dr@XMs&c|=mqt%;yXxfdh7&5Dm@tl5qp80qB1T=Gd%aldn39O>=#PXAZw>Ukf?1Hp5M2P)(IfI5|4Nb@fWosk`*wD;bMwlE1<e82)q+66@l}#72!ylg<h0*0#<eFF!cXXMR@h(Dc4>RjgM-P|Gn$XikNRpW~5dtPJOty83Kx9yDgj(6DSK#0Et@j(T`vzgr@x^ozf+;j62oXFo(CMZjNq9F5qJ}7uIKTT*Yc>~Pwr#OtmGyPD+L#jdBK*%sl2K#<T0}AessmACfq|eu!kN@=;jSd&*Fzi?0SYsp)WY`tl29vXvNfhFo9>dX5FdbM4Jmrcie92z$YYQ{9>Rs(?P`>8C?P#a%mz^!!OPpUrKf!+n#|gfEQ6FBZ|Jl?IlD@(JJR1AzV~bKwDkuk*Go%fhd4~SoGsNf`B^Wnj)>y+L1@bWZ+)0?Iwy#}bMDXmz$9mMAo}B%7Cb$VEw(%MG#L4@tE~3C2v*h?1~(qw-_j|c)ue^9YcP5DaJ5?B;VPgy?e8Ltn6lQ!9Mu9}-=%bRK&FN8VSQ0l6EwRnW8Ot?(KShW_K3MBPa))ka_STEhx$2960zDhq!I};Wzaq)o~U;o>SrS7s1BgKf;>x(%%VEpJ)O<%%74gGAVcpTb_>cs$Qe?GcjV-Q?crd*H^QWDQ-YqUszLXW6j5yc&+4(i0<Z1}Wf`xFS|hPCf`=YPHJ?>hhsqYe4E3BPgGI>fJL5YiGaCmDlTrpr_wo#ORoxB^<~b#KFbVkHnMGuY^2(b6jVLE59xcGmZ$Cip6{?W*H9ySu&okkE@pz6|*Q5-EX!q#3^XN-lKJylT?ZMdk7BsXp^2U^NgTa{a_Se)2uT+VjBE*h<dJs+-=e(NL*BGBnZ4cbc2SCb|r;cZg$lHY5Cwu+g_)y#A&|V#MdL7pH3)e~!C!N2I7kCTbzsZW+!2yb3nhi#x$F^$3&76IZ!c{mH6h|aEnKsJZ6X0dA0pPMwQPCSr(1c?%NYXB$V%0lcN>-37^a*b3`qh=<N<i3VXnlZ@kd0mGN6?T7YpwlIN9iLuXG&?yu8nlywc45PuA&o$hU<;UWrA`Z^ADO7lOw(kON%a$2`+W5vcB_yOk+djFpTd5|G&Is^OWv{7IcCsWODCp?bdZ7n>&?_+IkITHJP&As;lK=*v2RAHo~5fCl|&Fj@U^|*u$f8S&om`{gK9o6X0El9XCtUQIK_%%d-CprhB`xSzT{%qcU}FVx%TYlrON0^$|r;G+jfIT^e+*)%T9R&UqHtSy3Ip-A#>GzEi2+t$Jyuw0u;J{bsPW!djsZ4ec(K3j8mJdkYdfo?+ryPiSb4^>&)J>2H>T;jw7YwB@t&>RV8L*zSoQy`?N3*k5S)pgTk-hfaV{5a=G+>)N&uvX*MnsB9egeQyBuOimQ-V95p!S*I9mcHs<n;qHS1q{Gp426?AZFz&(ZPyvLigc>Pu1ueQo+n)qc>yo?GEe}46aUn?T4f5h7{N3^KFp(3Sh@ml{3D$}gO2QhQd!-)h6YmkiA(Y)uXA3%QNf-^kKmlsr$`@v+H81+PdRxq`d0K^2fzox|vykfbj2MOH206#T2?z5wuL1VlvYIheO8GFk^Qn@Qz$cFeZ6sPz*g)liwYVBQdYTspRzMvm_YKtm4?{5c#=zp~qnLr}Oq6d9{Cx}M)&bp_whfPRx1dDe&nYx>n;p%y#h@ywC{SqoT-qK)PFWK<-zl$O`R7!8b_v{m+0p1tA@Perm;hp}a?a#Q17TPjOm18c^OnWHTvSyV^)H}sd=FUdpKNXVYmmT(PH%L*RK=Yt5bt`G!*hh2_IicY6Ae$rz<rAoe$<?2j@|qbt_mqvFK$#;vd0o@wG;FvnU;iszc3!0Pu)TSA3%=NPNQJYXHH-~=1$0q%vWR)D=)j$C97PUH4k%-MK$_o7*Ou<^Dd3E8ipFrOVzVUG{oLEE$})L0TG!Ef;S}xgQ!NU`cTzxvno?hI5&)_naU`W2cMZ!Q68eH@1RcYiUTj|qZL>1paa(oGQzya+@K$M4Vc^K%IlC4r23TJPuS#S#o3*cvwF;tKLk+~fgD=p+U8pI$IY+}0sun#r5P6h`}!9^Vlw>5j_?=P|Iu9cIx*QRHx;HzW(72UjsY>FF;*;P^^pkCYPRwf+M>bez4CINMjK@u{H%;nl~$o#?J#b*vQ;?$meK8u)VfiYip<#GwQ9_D+;vLYyYAhx(Zy-08n`L5N1nw%m>8>1vr{MVNMXL;UQ~jzsAGDr1cspC;>GFIZc55mYka<RvibQ7PGtgvW&u+&#Pd<+Ruo^-!McE18saxb_gm9)EA00sR$_%7CsX*=xfL#(Sz&p0W(DXfr&ZuB(B$>-WK|Gbgjt)*BtGCo3fPW<sppNE*VeSk#wq$;*l85@ZE7i2EO{*V*ttw_Fg?EkRXuk)1CZVs7hm~xQps<T&H!e}%4KlMfbK@r#u%tTm+*zRNOQoErzYI>dCbF@FFS!J&487qfU1f-X$EGjU(6`YuaRs(%__TbG0gnsa|`ID6v+j6ByTdiK$an!L6GD+(`MBvKD@xuys#4V6;^ob4yWd=>^5eDzzX>2g{4@g6wvdLpHqOR$W1E1n{Y;ZZq5O3UIB4KWwQ#1Tk29t1*T^d;2O2Ld0yc9Y%T#&lksp$76I|Gd6QqF;rtW=qTZYjF4IPCE&<Vm|NUiJ$jd2U&&})CKkfWwG6u|+eP*_&G6aYLxYytGoB&7$^YQ_h$pBoMzzG4eaQ-vs!aQAhJve_^dRhw}9Di%l;x~!i?_2&H0y~3vy$1O;2&|6+Fw>5J^|OHY4Q}}o==+M2T43vvj81RXD`6Tq5bX4j0m(NyIyX}O3>%C<!Z$I`jqLLZRUw>#Hb3{gT+XsHtyKkPZ_J7p(cXNoVC~N@;jn>Q9=nB%$!24xzPpN3hI^lI#lt)gZaEXy1WY*xVFOlqL1-2P_&eDkZ;`zY^>fmvn#_7^@jF82e$q`$rK8wg$Fh7SycJ#%r!Dgwh|!#`HbJ<p{GI~CUWxHzDvFLe$gbk;^kj_K_ncxIq)_GjB+%^9YhgHcLgD7{wwcr1Slevu-nj#mwz|2)o;6!_Ma5=1*<5C~pJ!Avo-R&LZy-7yM)Nx4%-pP=E*OyX-eDU-g1jrW=Z%ayC3S-EQNft!1zJ>2rfylOmGJIw%CMh*%N_d~&OWmbx+joQ?VK9sG|!I8m^^o4WG01vZ}4C&<9vj}F;4_J9qgxjBL<WXoUHY}S$w@2^tvc`t|`g3fE=8gPQkOorYBOc+kp=;g(e|w7=HZjZRkKk%#*2sqE2zu2OMpSM9OHO#B@7TariU#L?mGt2wA(^LFGUxJNXEvb*fJjy3IH@1D%soT<c6T0kk8XO~}vbJ_+E!dyQW||3murwa(4G_WmUib^fm+fuB%*1DWaJC8OD2e8%|Uh${ODLDM`Z{1^ec_=NzWRWPgwu8%k}2%qyOs^BT3Ga{OdZ`Tptj+HaPz>zTlXZ|Fh{SDN9Mnrj$L6W_AQAE5~KScU7PKxkSfcYr=97f_n92<?CW5C^bk|R7b!FurF8wR(z{!Vvk@16GIn+3Ol(Lbj#W7nTpX%PsXidTa2Q<(6*dHdpd$wi#8v4H<geM>k*!Nvc)`2EHIeewTZ{L97f8LjW)4+`ss;5>g$kU|D{ANJ=db<=p5fmgy8w*-8jLeY8k%d0qELMeV`r+|>V)PWd&9n7Aa0_BKuIbe1Mo@Sb00hatV0TB}<|Eph9mXtq2*FT}f_$w2EncjX*d-rq77V%e%mOL$!M}Cmx{hQ0?{RRxbWg)<b_J{CEdeZ5s@O4ciCAtrZ(~H8}4KaLxv7`TXAM)Mi!SKnpJRgjY(fS-<KFVaEVS7kqMhzbY&<f~bhTGxasP&@e@sS)s6lc)v@HGxJnhwlULgdIV1sPCK*0e!!HEGj*8mK+^F`p!v*F;koJR)C6ATy+O1h<{7+JwB-IZ}%DniGs12v(K5)m+f(Igx*k%UO^dRKgU|uqlv&3({&Uo#q7kyzmxtYix!X`>yf!lOs}zz>8*>LH5DwvZy@hHUgyz{`h%lfox3~u=-Yca9n(J*d3uVsK~hXJ<3^(A2FP+XP=30b)EqH`l_dfm_hkH{ZV6RAEpEN51KE?&A{vsS7oP0nARzU@^6*6np4vpof{wa_9w|cy3@DH*rV)175kGy@$Kkm{tkIZNkwk1L_f>_0tUHMyml?}bWVyNgdvsm1`>6Rc1+RSYG}m8kEkKNA=Tq--2$ITB%xrFVJAVyR*8q4Ekiqj(7iB{iy<@WRg@z+fKq2H2(u*7JeVfx<TAoXG4L~nys260baA-DG5gcU$?+^rkS6Ck#1o@aWvj{Ry~-y1r%^?aNzT_AZdV!&p~g6l<_Hs)Al<(wrsx}D51oue&wWXm1kZ$bqrw<`9?&iRF=6vR7pT9_1ylG$BNKW#%4w3j83%exg3L$(78dcAiqX_)R=D%WGcb8xYlh9Z*R{W>tfDqTfbxiI^>Tr#G>v~FQx^mTIY~f(pG&4!r4!5Kcz80EnI?7x^LoADRAeHZmCW-c>WwWXPi48HdrL6g2+B-c%TuaOa9FSYKA?N=6OqE=bJ-IkH-vMU893Y3)f)bo=T6h544B}IXLvd9(;TN-5d&FH(q?!9#5DmL!?+7Zd}I;5<OgX%`YyA34+d{!67z`QN~({ssbsPQio=nfM+iSZf32zVgHo{Ui44y{swtmli}6qh!^#4_mt)J8U~)v#a$?2ba(Iy0-1nzc19Xl}_;evMdPSj`q{SKa$Ex<$oX#oDP~aOOSTF~BQD7QUz)(hW>5V`Na*0kaLc_v&+XDVNJ?T#n@%}HWt2;?$<96-tF6PVL-K^~-?0&aZPoM|8)zGtygJFW#4e>&#ia9!hH9gs5gg9>?Zq9*m?wZko0o<+=8mLu>lk~C|%`%ZevvHWLtvlj&N#3vR+)JuIUR~ei&}N}0kLC$ZyZ9$@xnR=4CyFViJHwky=HgkLljH@-lN4p2$$LcPo^eUt6K2al1{y_#iS8-Pdiat#?*&0@WgebapHll@apFVII8dW1PC$UjLKNSx)<FcW#RzIJ;M}WXYg5SMaX&p_JX(&>yW}mEzdWv%v$lDcccytaEzf&5UE{k6?{b<ao!|2-1}#Re;A_4w;0ky_*VuE$^cSMgt4}y_<|lfEI31!u=qKo5s7_oiFDa+ppXA;2>TltS@pB8VkmNu;&M7J7!qn^yU2vK|2YiYVlA;8gnFuPeTgt$r!8jQ&K2Bc13=9+rwu<sZSLw<|RWrfN0oOK@D&@ow^p?O4keY69ZSSs&jLW+_$~(ZvExRk^k};Y<Ck>avYJq-U%3*Q7MY<eSMEn%SY+)378V+ei;GR*SJqG{1H=Kt74=?^5_WTRvwQVf|LYDsxB7l4Hn<C?l-#kN&n`RWfhvn6$VH|{OL$cvf#3RF!8bFo=ECpT75Te(c12aDGUZt)fjV{k0UoQoicNZ^>S3eh221(EqwVPs!@p{nN%M2L=3DDOxXUG~;A~FG|jw;7r<RR&k_K_$dDeNwuIz&jv8Q-qg)@G|~Hr7|Ji-`IUG870!<+AyOA010U%1}U97>9OBquh`(tiBogGr8is*@;|d#5r*Tqx2DX40uPX*6y#W1Nr5_0hv%o`2XH<fqx5>;X)Xoi8ZFgRQ?z(Zg4C-dwU-dfuxqjh-6=^dPiaN|5#jjr?T*qbm60x(Z&?d4BtOgDNrr+Cf1!M%hzTl^7ob8Y}qL-p@<wkOYSJB`u&rBpBy={*WBsW<LmL#@_85!(y6l|Op9`FsD_*bSnhH-mC+R_O1kktazAG0KT=m`b~=3v7KfJ#93^1C#2q1x2d~4-(2dJ5w$j<b$7;L6K%hfmp4SIDzCTKUo~m2dD5<NbMHMRy-(=T2Mwu&nDbswx{^$dZ)HzG`$U{9*HcG8+r0<JnYhIRhtG-cLul)qc@sBGY!T`NmUq=>ErQ}q3i#FFPbp)u>C>TKH*Q&`KtYat4>9Ckgy4HoaHJm;H!XPBIyLaLKK;J?1stAwzK;Se@VEy1(D6Xo2unO2K`7*aZqY7v&0Sw)!ZU7AhM!9;wR(+pUdQp~3$&hfHOR=e!5mG!Fj>nJE{sZ7tXf7_xTU;kNI@$K>IxrD9>vg?GmEE=4j?G_COOX%8!pmm-j&4qEhRrDTFx4Jha!;fjCAB}Fb23i4FK=<4kS}FQkgwScCX_~KQT#t&eTuw?Y44R45ucBW!(UNectZ<?oW57iYDoNN!*d1a-q7<Q??Oskg1RIOAOqB7CQCE_)LZb%a2UkJtk@bKp2!r|W`z@@3B&Ct(zfWV%TvrN&x%yG*8h^HlUS1K;`OW5#T99x1W8JmO-)_h6P?-q6O!k?l<UQrW$+_T#U@wG;cINXlMiOGHH3w-R%>im>J%rHVQmh|?k9d8<w04`BvA|FuK`peawm6BCxt?rr0h1n&@>+j?0$OG>z{=bgV*MAQH%g$9#G^5zF<3$?$M=fTy7d|K;uvVCL%rV74gm-7kgvLK_wl|1$fYGw;~J>n@YO~gmrLMAm5U9k#lWE+TEleuU1CV3dzNQkv7;K8bzK!F%+G=5o8oS29pvKdK-^LUC_ycnD2GIa$~d$irY6YVy!pn9-*@yPq~<F&VvSjn90N8`W|&DeIx~LrNZS7yD`;Y6r|oGH0)5_z3v#VxV>Hn&N)Xq#32Iq6UUWk=WN+*Sfc))Pl=@yNSbP;o2465!_(-Zb(v-yMiZ-QWjSRT9Xh2>oJJ~sB}_P)4K;;o?!=6>$|6HIb)m=~ngOQtJTjAt=PVm+4{5}wwmp6G9ew*cRPaH(?1=$`V<HOWxeNxjlQeWGikr?(YR%&=?UM8rQjlFwfHVTJ-mI(v{l=UWltR`P(mjLB-HhQaMtU!#CW)s#l;Q>wJ?9M3<L&`^K9s}7l__zFd@FxJ*C9!@Psmk-?PsptOp<SW9DBSBzTnhksP+5<=`z1^TyEqQjP{|wy?CB$aYOB<f<{7bV0FMmW)XD$RftNX?gV{9voU+#>vrban92j@8F^WN4Xm!O&AiSpQ@>jH#_q;OrT)QGkh2?@`p+<u`n`C>1hc(P`27$+#p>)QGJW84c7F1yc3#jm|0yc5zogrl9re)G_8Z}`^&;aXB6G7~@{skJng{tcsbw<v^Pp>&x4(8i=1fmAGJeo4+VSuz2Hq(~0+)L~F}hov?BQwpsCQi8+MA_?rB*C{FO^!py9fq`9z)4q^c~8onxxT{r4j?``2kK**;>X9Qxv)EIK_@f3R~KS`-Z=!;)@HM{oIQ8zD>mhorZN66`oC}kW(GV+{jWORGjG0B?{$SeZF{8*m;529VK>@dMb2qNNb+?b@$I4rZ!`{mLdnMHJ^lDAb<e~0kH47xYRU29+L?Q68+40ZH(0V#<zk--FwsP-n(qw<3q&vPq+m!itmdSJ)ifx`;#({pqV>P&RcF(9escLuzf{*T;vGzV3c{7&D_cGcVO>LbO7&LjDN?8keD~rDS>t4-N^CyWFONKp<)K3<{rWa<nbkSG6a1CGC&UUjiON6&(ZLSD+&7I_{Nn4{UOfW7?xmjt&!$lyzIO*v}+K~b6gKj1hnD*P<qtyr!-IbZDCqL$E{2h<Vnvj$&)e03@QhcLt(TDOM*A$(nTU@H2We|b#8TX8t1AIKb}=#CYdqWv^GIzZ;-2icI!eeV0Y8Xypa5)THivQ(|QJh$#c=#46#TameADVlCVjyb2b2OHX9bm{;ii(6`7gX)VR3kWR}av<z=>4$qLlEgTU$WY0}@$<4qP`{Il)ms(gv?Z|wprGUEDZ|5vO5;kbd9WPKLi;8yNs+eht*V!LK?h;vS$IVZ1ezQTObs9qsYxvQit!`o5xoszs#LRjS1Uy!w&I7-J_P8O<Xum0Ps-)QG7f5K;+Q`H`e0x!@ChHAZAmY-K9AuThspR-j>OCC}`7YH5IOLFP6T<^ImOkqoWstCP+&Y<q;H)QSSe8f+(Apr2@?=Sx4;(vl1^!toCjpMT@q;;*jxjL8sGlec~Z)KHE-C@~y@7FlNM@^mU9?=gNI|{=#y)HUjLd8MKMA|aUE64Mc3dy)(9afg?rGt*`HRZXNpm0%?1(Ev%D9auv|FfOS;+&#w92FWzuwhP<HL**!fJvpC?;Ew94>&a;*#=~y12`I(-JR*D0h8^c2g++|ce7q?Y^|fml{@LIPN7MecW^mI&30KRMx;K=fVGJ7L0UC1lUI)do37Cv^}6E%>Ge>&$TSRV2;B9Ac}C#++wU#4ie$XX*<$@kPr#C54^#zcn8P{H*c;3z$@iYxSj@m5xwXbqHd|QRMEiZMv4b=p{a!gAjJMnA8Dco0tE5PqA~!tyJNdVJ&RKRXD??*hBx0NFESgoi!&x^aN=S0Z9wFvOyOlb61a4M~-O~wazSN$}ow`=>u$y)YON)zW@E-RL2013kK!(&%0Hqx7#_sw~4MPRVS$y-(dTn(Fwj!{dgaj0FB<nb10(9>z3~*gryq;um8?0=h+B;8=RoaTWLmx@FM4+kHresF^J!37DREfNVYvmgek@wQNBo;KuzcuA9!#l&gz+PhH-ru3JMTVH6GUf|-_p$U^rmw!_d*{{9{f>L+$P`QvLogV?dV|~$zz`>WPJNb+HTpg09OfN9ldG6-g-qpBRfZyK&SKv9UKNp*tle=A5(eQ?{`$m@mrZ1Nl*5E(gS%!!!!>2tmlf~T^2Y66(-CyYL|DOfC5sK=4W!kwa)2Q1@FvLytFkX(?2qR~8-x+lYlr6(D{NR<eA5sLek_O8dO<olvZwumB4*i?M;xhNlZb)7Xuk&`HXDKX+L6L<r)fJ)-|1W`Qg~@m#MOSux=V6ydch3Y3v%E5m}5U;OdpYs&SI~z(g^<jomt9=58*&i+rTJ90$%*jWn3@EEO68a$An_=@4vE^=p~i>g%0u+(I&)Gf1~WwYt{R#C&y74L3~UdL_jL4sXjG&VlS{Vn4*>y*i#Y3@+%!1ErX9Ui;wI)kf2w;WVNrhfxqE!;>EY$X?OQ;w7*R_u^SSL@&3RC)K*dUnR9E<B7Gx5e!ddV{^Ul1e@;;-FKPC^G*8C&^0pw!7Ejf#c$TTOq{WfH7OvjUuYzp&3%U>f8&vx2s&Btur5oZ4vDEH@lF;&$JsJ#QWVNzcsn@CvOJ8kA-V!L={t2Of9f?4t!YF+dKj@x~W1+dgq|ih+JsF_kTh51!D%-51pLB@9#3}HBs?c~|%1f;qWTSKyv<fozmF;cB^S!-}xo{Zt0^Z=tVALv7S#+S39vm=7MyR|rGCiTQ2H%B_p><S-t~GHMuK4^H)j(RUEWo=bSAmc1$Jx_!H_8ICEB5}~<<hsBlHbu>yhPVrm>bVfGkEiP18?Pyq4Wk8%?*E$yc9zrz)5wxe!P^Esc+<LV5i$hr(|sL%sHYGN71d(GE9nZ(MHQ8x4?fgMY|ssj!q+EYsh>7mnjoTfpIaqvb6ZtTW>o9Ud|iv^7IjZFEDvt)SReD2XSwG&X$kteFYCSfg+6v8u#$?r?(tyE-7*-#{3gp&c73v+)J_zTDq6#Qt$PQF%kk)ma@VKx4<7Ro)x_?)5lBXnG(Q{9N5D5;7tx@V!okhA8IIsL@<%vz4&MCu*6zBMJj+9v}iH9cboGgiT-mA2V$ZhUJ(Q|vnc<F$MOaFZgXJI^T4_e$e{F`6pNoTo&+@TV~K_F2VxU`8L&BF91wr`M|_Mn`Oh%)pEO_`d?dgpzOgO|{1@x3Ek_|i08E^_*n&kDe=Vy&jeW_aLU}G7Gukx&UUsY{@M-^SseHWDgmp9FxsOg;P3~CBiu3pMtc9-2BzU0aqJYHACudVr%jM(c=JE2MxhVm$%Y%bcev<ab1#Ukoxtr~GdiimyWtU%RpE3GKNs)g8oAK9|*>1Bbv<N{QB*4B6Puhp*hJ>e%#{rTB@d(|HupAwop8FUJ$#+OAl$XiaLR_?+YJJ{B)!C>b5&3MYAm(eC!#_NEm~|UuiQ(t4pGUxzqBoF?C5f^sT$wsGD$Umi@p-J0J;Zh%S(T<lM9z7d1S$P)y|RJQI(izrWDWUb&R6Pu=t(X!D;gu7%?ZpkY}KcDcAL3Ne%&`2(j_M272@{Gg%@23js3q?ZLHR7+vIpduGSd=-QZb7M5wqYnXMESo+qsB$pneA<cIRr&t(+*EumWwcZ-Y+WiTZE9CZc4I`Jj>CHz9X<v^mpwd{Q|MrU`kHs@Z%PgM@7RwlWVf~%c&(2Q9{WyID7#xHNd`Tp^aIlm;Utzx(_fvXA!Wn+$w>HTN&xn_^zFO}NFaWl?gF)~-SyM4>fZFMW)S)sI!M7r5{>zmrGHBQ9n?zHcb%$>&!WqhCs`s!m6GATsWP%Ih5+#vRjdx?G7JD$@&CpN`tlpV6Pr=HUKIPAc9)J~4l@whB$^tX#4wNEkI!0iSnN5^G3tIh0v;BpqKN$g~(C9s((;ku{CWjVOH@)~(swn<Wn6p3xT5~(w;<sFez$}l|P0)m+($^RpFNNN@-?GZXn%$fB_Y))*DIb;{&AKw&>iKAQrMXUQUF)BcQ2h7vV!6^TmH_rN5c6UEFG~K*U|42eWj(wJ`uh!-bRIK}pFq)gl<G!lY@76Hh-(hs@Ib(+cS4PduKq%GhiGdCevo}HpoSkV@ofd4DR(CrDI3YzPA(UDgZaJu_o)(r;>bFE|{x%TUdSAPmO7_1s!394JRu(xx^b6^oi=i;WxAz-EP6QGCzoWSF&lmrG@h=zuWRQ&QdM0lDR=v7fSzpBvYk1de-NJ7Yj5<MZ8e~Gs7|LLgR{CVSQr~HK7@0hU-@ZrVC#b)rczY5&ruuibb}H+o_}oy}(ls4CW{ygBA%ZXd3M&1I$nMj8bUXZ4c5s}iqyswzJ^8PO-Oh=EpH_(Q#p87J;YoKAy4MIMA5O%}LLVqa6d0I?Poz>>FAXaxU(!|h0)ZmePTCLFZpY%t{k330z6_2rW^r6>4+qd4hw4*4Rp{pfv>^`AiXWj>3I6a!mtgXUaXD;;SKwH&cll4h8~*gWSI~NKtV2oZVGRF<|3p{lFA_43DNIh!_m!~a$<va2Sl<8XcN^99>grDLR&b}jwPDZBWA4a#d@`U=Lv|4;PiT?8>+AP1Wd_MCSx%3$HNbonS<&4sK;Krkwm%4<8y}uW!Om6?LWhpe!=S(chXD-kN4wRXI6y$wdX-Qwf}b>Y)*^4^7?x2sTL{J2SGRUIcM5Mso+<c26vj>em{6P4owN_(s1>5-V$1KG3vGU<Y@7}z@%1W*3v|H#)ukc0!?wBg;4v0IN22nA86DtIjH7n$by4gM*muGjX*kzd{U*Nm?^Wy7K%A{^C#W@o&E56&V5=_Cjg2Qj;)C9ppd-2kRVK>3;*vQn!pAx4r@Vw7;kR5SF=?6QLHcJXyP3Dl(ESpul&HbKCYo!SYgux<3B<QTvB08jtOVQG(4@qKrxasKg?J22qt!T8ZB!aTP6Uu|Qv?(w5#0VDsK656tK6?`-c|j5Zz=lLbFU{yOus^H$Ma#jk2CYbK5}H>H+dp??>^62Al;02jlPG-wJ((e+D4^%wZ74)UVXo^T@{we-@@(uPZz&)tdRn*219X-ORbn~!Me8n+rVC0V-d}@RSTMfn}7dqp>n$+PS}DzgS7B4uAFj>AJumXSa^Hu{c1f^JYJ5t2Aq|=Dt0$=n!bwkp?0TWe-TCTBGrkLoKvUVX{FNb8ZeqWuf2Ph;^p{Sd_7)?Z%p5^o&AE(87JE&@n*gY<t=|#*3o$`SYBLQoYlO;0_lMFrk~phTjfk$P83aVkoIjx(%g?xHs)r8Oav7x0>K_l;P`;hX9}Ml)`i)2XsLNbl^jvpH_kzWhy6%-iL}b)QIUU>M0F%GC$PdU!ngc;n%@Z(RJqhdjorR{PAO$tTot0Q^HV!vGrTk&m|amxY`8Px)cJ`R%7b$3_9ddsZ<=U4cTDCoNH4r1XXHvK|LFSeNHw#nN&h72Ty(uL%c(~)p~|R;lpvi9`{kwX!p(Rje??CB;j^r8S(@EfEs=@zH{^OG09Y|R-`c-%{?43RK>Jsr+_RBJrRM+Muzyr?!szq8Jf3;{MPehE@}&gxMt<O#bcVJ#g^RC(W7A)NbMV+lP&}*<*pDT!;kt>p!UXL20uqlWzc?1^A1H;Yn(xz!3r@@V#ue`QZ8JPh4q=Aa(Kr(!3>{)0+B_4VYy?fQ^ku9oBsWSz^c_#e1@D(4&MlK5^OEUEjei*EvK&$$qUKU7XV^|uhykD?I||!+l49yDL^)oJ;&2`B#{?4+ZEYq@@d*iqfP!acoXPuVO1lgY9ir3q91VxnAMK)(DID`CZ?O#xnaHG^NIh7`@2{(h5~xPf?>;1o8&-e3y$*K{Zs4R2bbbp_Gf2fR;&hi6mv4mA(~$^Id9+SYMrOD&dt;&sqSd)cz&~cv1}IfAQUyM!A0wW><$hGk#!YA4xB|&p3TR2&@@r&MuvETCs(b8=GdSoLmg2YF<HJ<6yw8?^$Gvk$RA;qXS$G6C`EzGG3-=hgNOBADx8w)GCd-!8%84(uGrKo}!ccv33%;X2D)YNy507*XlRqkg6wo6F^KozdopQKVy-#L}Fz4z*ElU3c!CJ~J1o-JRV@^^6ijO&(h#kbx>VqJ7-;~h#Z-MVyzH7f>Pf^+m1S`qFCr{W^#7J^K4=|J3Ew9s{GAJ=d<;Mcrm@5_HLrzv)z6oKeVHPNlF`m-$<55xl4adDyTsb(u6<CLYXhGRgpI$1PmNB1&kUSke6H(~Gu7)#!ev-Dq>>FtU$BF$?m`a{7=rp>z{E7gd*>v==;Avhmp3lYC<X`uc?MKR11CzrWLeaPP%8Rp!B`c88*?Zn*0>QaF=lml^kI63RSByUs`B6yONsO6=Q0|MeV(7uVJdu}Ka*po-_DFo9{??!HMtE$@Ug-+H2|XYS#*#A)X-WfN>z3%2a_IY-xS5Z|bbh08yIu9Kc;PtdipOg5pE&)w_rdno&b?~GG$I_H9S<jmT_6WxX+ONjq%Y2OpO8bClG{vhnh4oVWCkjs2UfmkJu9IxL%!Msa)+>7?e#G9D=ZZV3w{jbY}iKt8emx{B!I%9Mlc-qrzh+ILh4xpsq;o2$_sC63FeJPd+g)MK-obwjt13OF0c5@cv0y>PR^HLpC$@3PB;rzDOk1ExK~{Z{iYO-7?A6fF`?*1MFaz5Ia~hPNPwc~2&bY3khC4PSI?PPjN!ulc{PQ!d8wMvFbD$qnOl{a!Fql{<llwgH-g~*y2PjAmlE&8G@(-tEY6)2e5cHstzQUCmON4;gcF&n%i?1UzC)-aIvDy<hX14FCyx$XzqkQl{p%`;*^$W4jyy(DcxajIBMOB`i8zB<x%q%fJ&3Jxh#55IfVm?pZ`z$oZ5@LFb$#gPm5$nn3F23}9fT4hlPRJEPU|4a>sxDy#C2!L$^IET$-_bFO%fG|X||7cu4*%!blZmm<b#oyrAR6Ge|6(F_7;a>PNOpX+H)b!BMvdl)UOE-Hb9t%bvQAV`s%$znMtj=pjKAz+%4kfD|4BU+i=vvTwV3Y)zw|XTBaB;(lHU-GS{7{39FxfRNJ+6*eP3eG9=aPbjV_e1i9D83G|HFcS1xM(2kt?;A)_Y2zHn9@nzR2_B4?VFk00aNVPB_(g8}w7t;V@Vp5ULRB0mMGi_kWJK+CI8s1G9`JXE$^NEN^LtTl^BgY7Q&t#&MmmIXDwa3ySk)xWdkf_%U0=jYSouwPsyEiTw1a$G=F~Wvi3}iyREF1hhYXugp0<uqk!vRq`ONn3@1W?h7SHCubg#L|K=r7Fpc8a=CnXsN~VNS*tDAI}C`*WLgew7i;h)?|(rt_IhszyAZ<Bm-@+rrD(<6TBdFjVn>#)>-)%l(D}d2TH2cRDLeH@;n7!28!R$d2w?k?c|=Rzj#Q1C*_QhG1Tps}z%=?G1+y!f~L^T!2(+O3pkhK<NJUl9RqHcl(&AS7sc-lUKit*}p{vSY;O1TKy@x@vakU1FJ|7RI&j?(F)g*juEEXoao_pIyv;2v3p}OiDy(`%Dms%D@p@Da-T;ayp=*S^(UP@pQ#@TNb3RG;&%crl=LwZbU9pFxprgYHX6%snV*LE<>U|z4CC^W*W}3IEoG~evNNsWl5J)TPRg>x`^9U-09L44H|!~BOPNd8bs39?h@U^&8;;9jB+dUSez2yQL*Sd?ad*Uc1sKB`j)r|=$}#yyf*CP7>CrJ<YKS3&xSzlP!v-H?SZY3|76Fcpx(w;bow$SrgtLf^5T3}Mdc2;4Jv3hx#47krw`>}VwY^<RinFHKa&n8ae%>*xA|mRLzag}G2^Rp<5kTmxeh^eP*8<)+@(;0n#GkZIWov~*8JL`frK#H8hp#bD=uacj9R730HoZ#G`@P~@@x!9iv%TOiwrZO&%-u$9^KQXA!a4ejrqsQnFt<JwKcaWhjx+UK=A6YCx4qCB@^3*H`1f?XGwUk9MYZy)Za<eM7)C9IiG{_FrU3eh9AN0Fxi{v|#U#0=h6R2p&zk!>r!UFFOVq8lz?_w%Z-kyW%s$OA8FWKEo+q&59H=BKXXR8)0h>KoDsw-rie<!Hm?XQKKTHyx;sF8Mrd78rs>}B4wd-v6%9BUK<WYKtny&;%`2;a=j|+$o`7nkd!2g%yQTKRUM#1Oy*49pAr(W5{)HMzAq@d?@IKo>SAT(wS-*aeE<z$H^cZjMS8^2WN+8QCO><m&KI<+#kHCD-TnN#}=d_nEvvNb?%Z_uUC<iMacw*r{h1(?`Cr-WZ1gSeVxjw%+KBxrrg?C?d+y@w0@U`a>Y=9j|lZWVm5{B~*jJSC&S0Up2+oQUC3QQ!*U)53hqtk+X_RUJ-XxsV*t2_L7UQF>NL9xlLE<5N4MMzWBNv9K7rXaj47DfPl|XF#a6;V8SEqta})D?!_9c@^c1?x=thBkV8*(Z5*02jcH27G9dnNY9byBze?DG(j3An)H!NDio;E=0nCne#l_rJoK2cl4$;7l77lMXTf-$7IBk^Mmg5uy~PsEr_?JmIv_3?X96hD8pg7tvT!u3F|>z{t)<6SW)%Vg^*ChJJlbDNsyE1xa#tdRi0mbpz)HXV@C5U~fnY(+#>K_u6&t^M9KE-^Xd-ye!*gRhhcMhJ4ZU;ojWi@J_$5r)yi{C_m*2u)*di<92Ezd+-Q_0UEo1z3bhV%xy|Of~u|KEI_q=t!kxA^I0QWi?Cuvx+2~@1#?S&=uxj)aZkZzP&8wAOVxj{rX^+L|yZ*x^tHQ#<HUZFZ08DesL(x<CrvVPN`{RKlP3rPM@xOtT{pNPS<A~_sJ6U*EIqbV?@G%s2rF;^i;c@<~@6fB(i!9(@ITW`TqPHRIUrIQK>BliqP2{D7VzXZYQ8U8@i#A?%lLl}ajA4fVd9IGTQM1A#M?Y?8@Nnk1QMzKATyz83{)02fWh>Az)_(55|Z5pmUKG!7AcP^Hl@ddi_B$(29I3lZfzONGXLij4l8W<4|U&Q{pY&Rv3KEhj3p6AjO|04sM>$=$oLY&|Xj@ko8wmP5j2|gC7Y?WES)_8p{SUv2vA7Bd7(HY3cAD&>7LO(bbX?6N%F)@2bu!DhLoE)Z;AQiGP1`Ksh+E51BNVdQktY88|LUC*ak`QxGk1wgBp?p>tu-ylyw$~nXF|A1-FYSZ9BSVFgbO+<^5lo{ATaxgMlGVKY_}q5GNsc&kic0FOk{8Oz=S=o!66eZ~KVpYaraz1i_L1Y*sy{*c7nRG<8p>%OWqnD*`WB}UB1pya?C#}eecD1;uySUKa}`<N3%0j6)~{~f4iKsotDYQodn1)$bb(0+V&r!Zx;h>0I5<YAxeh6X>8=ehLrc*Y@q<J|g_rW#GbB~IYh^}A&nMT!Cw9D=p#j8u9t83O%!^ybRM-V>GU#z2F&3nGCZa^IrX|-Am@)jJRF}XC6%fd=-!DuptG#Hw)x;=>p#91FHd4N}Z?mthY%vGaUro{VJP#>%cBF$?a@5k6R14mYZK`2Ti*|K*%YIlqImSB>k<O2g5HDfSJ=p6FGJ&|vFzcg-eJEoE*1duBICD%S1Vip&sjSnrxB~|Uu}G~uGt3;t%n>$wCEIGfVWM<C?=WxX57Xua7$dT}SY3LpA>O1RRu|D3bScb`K_=u-W7DC9%O0W_Ya|B+lr0-^dos#(`_EEGpBrPsHt2HmRAVsK8jam5RdSkB3C*c^DfT_E{W&orK>o?8LwF>_p~xd6;5+nl;l64<->p|EEu27cRrTO=H&21ShRpS$PA1CeGRdYuVRF$w%fEDz(YecWVOG1rfy%ccTU$`Ic`$KBp^}($&B7~?h&%9u@{v;)#$@Cgo=n80KgJjU6`m@qyM)WOju#g!3pphb+v0Gls>_xlbcGKSusk|r&rW4$w_)!DI$I()@x86}HMotE>IU%QA8a5!L7GS{!vTc9F-^5Ukqm_}V<3*GA6E6Kzp}zv9#N=4{ifd|3B=HMujf(m-080_UJt191yOt2SEit=btFhv*O3C<y}faDHJ#i>rU>?jF!FZin86#Nx;h2)y#_b%5BCP=FMz$+Z389Dj?){>oHxQvO4-;&RtA0R0)r;Af>La%%uteju{8xtFru6``T}IyV(o)C3~j7e1C1t=^V7pcdqi9*-tvEC3f9upZg{=)_<SYUFSUb6ltl`0nk56LOa`K8iVq~^1r!-SK%O0^581b;+fQH)diy{b(5dp|sE|jSEw3xn4bIc3mN=<lBbcB>D@3Ubr?o>vvci76?MQ)zM}o9N5+IlML3x-9-akoxv1xQG<Zz{w&mL`KOav%qm3OtbocJ<P8$I}0AiIh1!#ok<EpE)HSz20NY#EEowKq%GTHY3f=Ds&hv?mmS<7@rAn0e#2FF05F4Sx!{>A@kKIMM4aYx>wqM-v#l&Dm_#6B7rDeAY1YZzIPC73mI%(&I!s&h#;X%E-m6_MBRDOqKD5wr2=xH+I=kju7bKMt}=|=?T(5_(X63)p~ua-cW>*wsJFXZy0On?T_7w#%c}koI8Z)WcDDpgEJF{s>n!?EmX)zYkt?tXzmsauS?$0(>yO@_LrEuatrnCy~XJD94q}(JjI#Pm~?@ezqYlyi;MzH*ji<$!v2-2_WVW4I9@u#7EYtssBo>c8;5gZF_?|l_q?<YlW|JZcDs8#Nweth&Butt!!GhTjQiC(LTkjKnafhpL3Eeh$y^GYxDFai2jhg534Gbm$r(&Iv6FTW#%Z`;S>LU~BtUEEIlzl(o+g_k&$emx4)d@l2*$)ZV{98w?<wZ1=$Wz8@DUtj^WY-G4gz5|BbRW97ap=e-r0SWCqkZ|IQQC6<9{v6;r0c6g%hJL<{yARH(Jcyp_F1?WCWOUgZR%+W>)l;l01-SKgQwV$q`7`z(<XS4-p8f+wF8IHGN_v<}O($zg%8)TZ36yUR=JG$W-6S9$cgkfZo7}hy91@;#2QC`^DoS1yMkw^s${bhV{gm`OV&fyr<Ann}CNznqd{70bP)xBESvG#h}bz!X1N3oVOxRN`&iyIE@@;qMe`ZDg0Rg{b`@{dBxf7P++ideqksmZZH4B=Xf;NS^u%Xt{(xj*VGeKw22hRoq!zz&gj3zU_EU=K-W<@BO=*JAg3$fl+5EGwSw6=ExV&LMg@$esAf$?oa~dHLMv?*K44r>stQZN7`3uE>R8K|b%ju^@*djHa~-ntm*NVYG3R~8%$bQ8Ff*$ujf4xTWTiea33!`wa<+g3_uf{$_LHqm&qp9C^#<i+zv*<MKPe<<(f9RJ8#kwj7GG$<n@y2mOtV#DZqhSv0aOOF;hwq^y;o*;3lg)FjoN0iQ?J0k>s#+P{Q6$H9IfwI^8D#f^SA?U4tnZq4Q2Q_8k3*pXfJUO`oz50LX0RSXJ~c=-hH2kFXRE9irnmj1M0kyfLsqK4*T5$qP<eAO%8fUnU7EQ`XXj4>8A&hqsQr=lEBe12Vy@(LQKaPm|sPfpu1c=!ne|FV`mI>zdO}Rqjr0}nlv^laHrp^p(9`o#5!K;d?sINV1@IUL4n--h~i^^IM(ZhN$lTv=bcWceTgs$jFR|GF}U~_#Aali35=W!?yr>Gz%DyrICdQ5DSCv`gNXD<W~hTovd3_wnaFkyJ^73u`zZUx|3{S~h&sU%eVs}1hsY$pj3vI%&w#69ul;8-_Sy)o#;{^Jc*E2GB|&f_@-~bBChd1>LI|k-CI0I(N)7dba)soDWxU8@JIOrQ3IkK5bNLJ(iQxYioD>FU>#t?EWTI_G{q^Ug;aoUgblE?-t3lI&0Q>(1PXQ(~L!Jr4wEqhX=!ud3@CDTHH%55GSDyKQ!|^2VbeB5q?zL|jPXaV`aRoLwMwK9oxj?osX*-q?lrh@ay2LjHonCq{7>>!JNMUCp*;D|Gz^HdRcT|aSrZ}19*UqVPkC=e<m8gw>WL!N@#F9FTPYj&SDG?<rLMZ!D1#vg|PvH8K%Ia!$d#8dK>`W#`soFpWnjDVH&7)?BzXY?NRj#$Njuv(m(Ra#HG3dT=b+6vk+%T1$-afge$fzKltZN(`rVqP;0!j;bTLe^p=z+mz0hNTYZ{II_AJK*ptS!GGEK1RPvoqSXZ%w6Zo7z7Xz@b71cr{Ur;e0uK0Nl{Aj`g9gBtlJ!k1)OR^2!a=VIHRAL;ehk*k6vi{b33cPDZ<|H<6Fq&<ZQ<@T4KGVr?}nB`!`?9Io!deY@e;G7}gU)y1$_F^|5a=K%!#!!(hh=aStRKJZp|aFUKtelt;=j4mnhn!e+6Jr3i={JPLyE}WKft5KiCi^a%64>#sjE9?4w%Jyas%M`SAk$HD1Dbl4ytbK^SxgE+dM5`@=XhT1jvmzAmw38Dn1W$L*QED1JoHzcwjw$}y(1z>ZgXpbHiP;at=z$y;TOZi{CnX1<0I!;0Z<G$&hehA1d(%ebqoZ<jq!H*0EAl97L2%D5#yenmx>12PAdhl<bVf66#~G`!-v~U#YV0>NXRO8!*bfZQv109R{%~{a{mo3iTs$fuv7Q`}xGGExcupeIW^{Dqmc#8`etfD3jP<c$b)VaTNf}%I6Fk3S*Du8#9La5Ct%nx&LY_^pZ;a1qI?HX0$$q9WpOEjG>g8lD&_ehGQzX{o@?%<6NxGxFRT5T0SRxhaXnIG?y@+OCMGr)e@rK8~d}Meepn!d+hH54ZwP-{OTqN@150I?r20yTKw45AZt(?Z)?-RjfDzM|`lpU-f5QW#Nm<)96joE=TLi=%c2674_0IF3oVE16NUO0B|F8aJ%P%$fZA=9dW2q}-&{LLe$C(dBqBdo>sav|Kh-Kf^@BY;7rez(g04X&%Pl-)Bsmeyg4ea24VQWtJYq*u`z0hn~mR%qJSX)*ZF(K{+<J)PCCD^;;^rg9(1DE$cn!`*69^YdCZhwfMLNJWR}P#ju@cqE*;mV3I-S@=>)ehqS(9O(_|7}7%txxofb(G+hBe{*7Mr~)24w)%*mTX`25A0DO0XoQaKZRR%w4{%@L&evWEx<PkS&NiT<9Vjf}p;9NoLS`Dbhv+G9>}Ow{67p2+UcyZA+P$>{We$CwW&CzDX|4_Y-0y+ClEB4&w^}cI_&9qb<}rz;mB}*Fsk;4EZIcIVuqs|k+@)Y;S%Jj{DEy<1EJ%U9LkJkS7jO@Ef<2J-aDh$+J;Yf@r(rR%*gK4{7P1(lvyY3`z@@?bzDVZon&({uUPhqI)*<9dVP%xsdt~bc)8<$cqnXykQI0h+Yc+7Mu99CSkY*2QaFVXcE6Z0`mUATm7{bc3++sQlcXkG*asBYMZRWvY&hTMSo2ZyQof(g0_;{<1u-kOE->+_(_CVd3w;wFQ_vx)@an{*&n$|}}c=na^tn@~&Oar~QTdb^RQVe&PoM*+WPj@x&bt`MmTfqZoxz~?Klby54<M1lK^IOw_Ed7kmskLH(yai(KIMG;Wn|6)h-Qeccn^bQ4>b|x{edR{`TKD?R^ise>_iv(V(W7`K6**%$W2*P(LWxI^LvF%nsp+#cf+aqT!4(5ahLZ+-E<iA!zxoyD`4X^d7W~qi)y8)AZZ0j|SXud&=sq`#*NTfYVU9*$PS~&D=bvb1aH1M`hXu9ZKYsOFo;0>Ve8IWjq)EY_-wn%H8O(p9VM7hH(4w2en`ZmsOPNqdqI23%c+c6OV0A|!2f><UR`>}$HQLomemlLJn|*D%lKEyScP9N?&(}Ax`))cLC|{#&TETD1>wuCDkFEG6Oec<CAp1rRE(mgo*^6TSSj3x+<55>Q2ORc}%fM*YHY)WG1i{G!IFS-3CGpm(l%OVoI~)_^;WStv9GsX^Ckg!Eru9WD@_SHYOKgRog4lr&D@oc6xkgOa>2xMZW7G+XmeHRwNxcSLXF7{ep8yCbU^Hp76=_L;KLG|Dx?kWwBI(#GAB!1|(B(tVF>nROUsimjSgz{gQcP(Zl3_>YqmvHQ_(+ACqIw68{ytv!%1#^`2_FXC3AvMS=7PAHr;@l~v4y%H%d#J%0|W6gg!k|y9rQlpW|TKh(&>$ftD&#1+h6e#9w6^!j6n5j+1rQ4_7+?@cYXbsqvS@V!8vWm$3vr+qx8`wx^N#VT3*5>c?h)-O-PTDPIufM^^RG$On!1IfvS-4Mx3g3p2<=@xUO}gzOG>0f|w5IMCSy<ja5e;vPXH3%xB!E{CES^;!1u)cWTx3HM1!_M04~rsxqDI3ghkp+KtNL;!23Fw%x_+Nq2FD_&SjBj(Wh|G2p9NYs_4n(1<EM!Iq`el<jToZlbgU0!h1f44h-9d&Kx8xF^+PghA2ggk*wKqtTn*q3qNvn+?Rwj}aL8K&&BA8Qd4AXXY7$UW=5gr1hv(IbLPG20OBXGCg^tAg&rRhK+FksavM0>ntuOI!CXRB|NA6CU&J!-LBLz;$`Pvb%V0L%6zc%w|kx1#Y4cpfNaBY*K1kpIn8$Q6g5zrggB$~w+SQ)JuAxwQ5zz>HJXF@90OZ57JF$)JrR_A>@0|2qgU%&jmGM|>go?^Nh)h=H8-kbMz?NIi{s;zd<46vXuPCz_9*Q?z|5B7%^K|Q;}J+}-ABgL+iM$J<msZ`<*k}98pw@Aq6;kF>OIuMq8EIaBhCv%Q;y?(OGb4CC*yACErnkCuq#B-aZG_i7@{pOegeAFIJu1}z$<fW(Vi;_5L<pc(y9i-CS&Tc&h4V3E=Ejy9yZ}3H1Bl`QFJe`^Wgh|*}kV1LBG8<U7x1{xCgz3m**)gNw&9!m3e9ag61uJb4Fn}CsONDrpQ6Z7B`w@0Vg;ye_ZO4FXqi@RK@}$$F<MOfAY;N$Vz3#zQM^+NSc~v%Sk03+T=BPaKt)byCD^mCW&b_;8jeeBH%gODjS!3YTA4X5<(5_rHO-}#?LI{ncdAu^%HY)Clo8^<*$1kEUiB(nnVb}yOaZ|f#Ri{Nplv4#!t*|uM_!)Y|@VnkrfMQ1Nv5w8mbg1{RB#z)LDjwU|TRbG)mhsT7-jxlP;Sn5bH^nCZn>c2fQhaqU*KIYUOS<>(x7Rry|Q$cJ?ZUNx629QCr#3V3NgT$5?tS7YygggeAw9Y#KsVK65}wISus+M-KkB@EQj0o1t&z*c9y#B{cFU!xW0iOE#bgTn=a*)lrjybQX|n(C-Z%pm<LNn)1~tn}0|-W94L$pGQO@4BH(k$2Qz2SVb2Z4mZ@&UCzADD{LOi5NWxqj9epRR?Q`RRHQ*3|B7P6%3{<)Ba!cfLymfDdFf=Ekw*_F`<66lz_Lf^*(-Ssw#LxYOa+qGiTvh9HFPLX4YDznKDMUS4b=8vNU5tYQ#}-$T0Q4)61JAG(jchL>YDUqz(fq2YfPztH?P<3lTwk<i7LAdR24GYRoU68*KSjEinVJp1i&@}IB)&>@tZ>GlDjf?i_Hm#b5nLUml%=ugVnAs70PHLZY9D{GA?J;3l!PUdWEd5x~zLbX)Xvu$bF_R1oXT|vhL5QB-KNL^gTW{t|-|*odx=|DU}<oOcb?D<{)c2%B$%})Fc%m*{V7~OxhDgjwR)llfqC3nL?bxJCo2G#cyGIqdX7cRh+}3v^a{c5kbfhD|@{)wNk5CA}BGz`@>&pQ~lBM5F|-h(x&{~LQj@QV}Y%}N$K&m`Otgtgr!P$Usi3n5Q-__n@ap)_#v|8BJ}i<XG?`Vv89#mpg40V;PF%@%G5_U9KKYkX1GK8#r|+O>5fD|xA-W^ZCoX<Ne5~FjI!^TmN;bQB)Op2uB3KH+wVPNqG~2&h?UwoO{=AfE>lXS(vU%V#~{UFNQGHiyz^ZQw0wyfEw7Y-x;$@oPjP-B3UXYbSOr-`+*c=KTqcFLgjt2Q>sx4nVZyt54}&Fm8&FBH_DbB}sw*jyJTE+*84L$9xeap;z{k`YByJKwvrv8mLBC5;I8Y^8OV=yg+bE8Tg9J}a-wP-SR|Y~n*H+=)2<Hgqm28zY5Pyd;ViR@qxEXm@;e>na?ODR^UNZLUyjw=JBCeTaWtnaoyKHP;0k%NnO|C)+l-VA*hHQKIWUkFnFW<QqI8`|n*u0uzW)R|PNdivI<LP=c5OtKS=m*YM8=K}e6@6i4SQMrdSybgNQh78!p&*VwqFqJWeC#A@_F|)55+;U_0z7eoug|F8$miIR@|Csmnyzg1mrkvSt>@m=qx4bvab~AWp_@{P3Dx1&9XM7sIBt#Ads|!NnRmaDtnSw92vdXV@5avV+6MxwroonD58V_a2Pp!WH$JdyBt@XH{PqW^yr^u}ep1ndm~%nx;t4m9w(_dJ230F3Gwi>NzHIDn;2`Q%l*sWN%u~bcE<4QDfZ7nxm{b3C3N{mfJ?OCprFK4`LF{n4)7?!*?7=8u7?A#EYb~$R6`X4g`qS8~Y&Y(0Ilw<i>(;kcd1hcF2#AwcA>7;H+6SAJ4Wz|#HW0&zadcR<r@T*Y*Q=G>s&1NVRbAU%RWZAjjqTkA%po!4Ft6|tG_hLQuB_HzxVV@<+}YR)&k1WL6UHE&Ey(OIk#dHQSX`LtMkNp%qFD~g&~U~@S?+-_D@<A^IZv1red=_{{ft8b;3WBYs;q;V9%%Nh=hjgZVR)iRcooB`%$ZIP8<Y*2WXT&k;ySr`x*^=XwEKYTkmxHxmpiAYCpI*50XS5OA+tTLdmvvhO20eE!+QZn3vm?CfzVDan8=WVHswmx{v~{~tyS;THf!><847WRy~XGC>W_92fHqW|gFW?AIf!I@HkcfCfvHN+sX~O@M^i474C7$F&E*>$<OB+X9W(P_miIgANiU(B(l15uALCoW1iBjtPDy~W3Ux#*7T94CvMgurB122JOe{A;d&EMvU|ekZT=^#1&a1EO8nk%xTKoEq_O;9AB>OFgV|rYJ*zM!hZ{W|bMIKraAxOLkhMiK@*RPDY*nbxRw@<9Vzzn-0v)^i`65v&0%)0oJeN{NW73BaHVVu8?#iC{%S7J^dMhR=}oU^J2e|q%`img?tv;Gb~|3u`FdoHsRzxws9oQPnVZ_7wPiLt-XF`ZcMnM!%Al4^bK&3KKInRVEq6kMN+@XDt;O!?K{zv-~mrMH)t_S<`JBy3f=OKo>Y3p(U6(Ehaw-;iYx)N`kUGl@Z`k~9j7+WdY+#cL%`syvxU{H4gl{25IsG1pTZi*&L;Gt~My1r`4?XR}NXX|1f)=}d2ec)m;d<fTNFxlCM`Vd1o!TSJaD<TPdrKyEoowuM>;zx6{9L^6U2-L}%_MvN9;09l=(|AD$Q@VNDr@Ex+PXja-CC+Us=ZWYZD4UJghNIRp1fL7_pQka_(OiE(HjON+_W8oU(Jfr*3GH=b<<K=KxN77EG;MhW=DbxL!Re@|-5f8EDu~MKSLt0z&L=xq%bUJC$jxaRxt+G<INNu8M%?Fh{N7Y)R-(t`bS)E;*zTa^rk*ylQa-VSiZCe?g+vvOznw4O`ciQb}nj-PMOF{P(J$``RfCz43{RnUB0G16c`wD3&7@zDJA>5`R9|qDrwHX}jdiBRBNuVHP@;CT(@?rX)CR}$PmbrtBr_VoKsaj?4px5Hu+L)tGwR>r&40Us*@Lv_`H?qS0WFY4t6`A~WQ{9L2wDWLfZq+~r7?@X>B{I%UvLa##AFfC}f=_tK=oSV>9oTKELl0V{NMtH55un!SQnLOuqq_9-mPE$N4>y}Wf|=g_F7z>Nz&x1{xNe`cr=}CXSW4)0B*u9VW;~c3U$hSvk*=0TDG6Z7#z@SZf*%gLk6aK?*lp3&Af>EOnT=PP5%83akP1=dn{4lj?CpsY^T!H<_3GWqI(#M-zU;Nu2&cuBA$vt2XZz{ih#(*o%$fCxF~7<9K@S$JBkUmgC$90s+V-|}<b<G-J$Q@f4uitM2_k@sB2CLSxyI$m4V$vzgLs$a0E1#<c@Z`FK`#C-7TrCA6v2b~Po;I-#Z4WPp_T631Ap(Pb>R~G(BCh^K3R4#BDG1)vXaKaP^O@G>Af<Z`F1#e-^R=&_<3y;hL|+&RY<X9Rqj1)2!ks$<?~cgF3;+ZaL~~fD0Es3OIhjLHsdPR!!YDJz0a}#<fxAg2JbShFx?h<lHJXX3Lm`uk+N(1us=MY(H=0YS;~4n`YSQCds12HhZ)JWfxU^@^Hi>;E_&m0$7UyDdkr^%mRBi+WJS#8%CfpY1|SwPzi6JMJd2XaB38*OM}=$(A1w^n>>VAQ2nR0C6x0pmNmEfi`{<cJB0Uq4ht7~RX`?R2qQUd-L@Iw!&ce7(AXJj6B!06vr=iOoTlAeWg_=@(?#p84q|R@Wg1Q1x7pm4unXpSiW*wZODKzs^!mH3l=F1yiTboA)%NrYWX<}#^WG3jR?FVC@#}~;WesNq`d>BPl!Qpsb7GeLuCLT`Gz6*<*pAW}VjSeQ5T9>crKp^QNdqMu+cq%Vrt)#6Xps20Qu^=IO0~MwbU<)!tHys2ggY;;xcW^R183)w!oGHCq-AW|k(-Z2xDe`>^vmm%27zv>rU{I>y#n^Ur2k!csGV&<9Q|R9~xuX^ebRIN$)xtS|Des>JNMc4ZFSbH>Xs5tKkNt*L91Y`mumQ)iTuz6C23|3R>m$Kr4!y2}4Xs>64pRjNjojiK<RPoeHjs<`K+Jwt2yXqGBt%TQ?ZW|Ga}rk`s3Cc}!UgXQq_rl2WsIJXK`fDh@E>FkOd&gVi}C%HEsUde7T!W`cO^OPk)LVO?X*vJ^3z7EC{J`g8#%TlnR!@QtNN`q3O>PrHjt(I$ghJO);Z`3gO;34`0jRP9i73Q#1P^AExBbFSQ^S>SQ*u5!b+R9Ipoo<5cO~(vIF>ITMslwEDWj|k6O>|r9~s`T6F-^L3fB+a8B+-Gq$n5a+At(G`U^Q=FCPU5QoAmm??=A{%Vax$XN9JC)N5EV)Lz|2f!vjyHmN1V63=xbR*|^?99Hq<OZH&1~J~=whu~$zZ?$-#m>pm@wo6X?hX(@I!U3j9!r0*h{Y_6gxF??=fk*w0WDUD=|c~y@`L!H-1{cklK$T3t*MY{$_j@HdWV@{p3SVZ=^<hu>Y$<LXMsN2CYqwlac8TXMNJs-+Y0(eLnHTY`pIR`0KIq|@juT2eQnELdTnoixxLiBtYz;Xie#Rd68!I2_-4efKbQ8Ar_xI2g~t6?*Cg|lOfoM7E-#t6z7WQ(UtIi28oJOzlMRMvSXc{mK-+5&<JFADe=dM@*--P8Y-`dA=LvVmUU;o@#o*$%FibQZeN5)P0HK%!*6bI;nn)4-2xCi-wr|!#mv%eXZ>;RCyjcrf-X_onOZfAuvt0BD#!EoQG;d_{eW{TOzJNcUFmN!hBcTyr&WvZZ=|PZ)dE<!{aZzcVrTvdz{nj=v!-MQH=bP?&N67XB{GWbruPEmqIlkoK&E-Ts@O#xQBOR&(kF;95U0VnKbRK^1_zW&2czE1TCkPQ#<OhlPx*${>TnhOWo1KsjjYmg4;CEyI=hg<W#~EI7uRq*Fa*LO2r+W(W{{D~@H@4O$l;01$iAgN*&qScLEA>$1a64%1A>{|&4Fod}cjwjy0o!3+2$sbcqd*04vG1(<D1?dtyNKa4SVR{IER(fMCVRB%bJA>aFVPt{+1c6w9yfN--`NGJwz*dQu|P`#VGkJ@j1kFH{o#p7k})N@v`T)FaSIBS;}7vgkB=W5lnT}^EXq9suh_-idV36lE@j!kD8W&0k4Vp0>9I}*4+e1KgwYh2V~Tosa6sm&=-kWVkV#84#bzx^+l)|>Ak?L(;p&{UA9VKc$ytha=#Ws#9`q&)ePI0t{3jJrAdAzo9-0uA$jgrf05PIg5}v~;{@laElp^BgX@7kBXoNQKkmIy4A9colHW{6jJV&@`FGH&-$J#3U*FwH`rnP&D`g3Q6w)xY|;p7hN<POQ9m+S-s`E3GiMaDGD(|sWTbr0n8@xpLFkO5;Hp_sN01APuCPv9*9SBoJhQu0?xNhH)p(I$PB#?~F5UfT{DU#7F*v56YUw60+jI8BLPZ03!KZUJw7pk6r#HO@PvnaU}g2)fJ($1xzL08e91|56Uy!!FdDiIv25L@9JXKs5q8PeAKYd@^a%<ua=loM6f*!S)F4T*2L9y@-B_jY_j1{4C-;yjvzaktkqhJIQoe%zoA&NF8_57Dg%r(Rqtuh&ee6Wn`jRLPsYGaxV;9M@C3uys#8aOUuW3Ld<d6t|;=MN_{;=HE&UJUae5>ZAqQ?y12;`4JsqjedzV_T1})XIAsspLb$zE+r-mJkg!rSb03~n?p8S%MO8Qpuv1gNY~m2@)NZ#B*?yYV88y8xmlGrc0{es$oXGo$r6VU_h}cI{g+VVB8zjd=s2J0T;epWzkQ#TT%wCyhUyJiCk7NC`>yU=fnH_}X8x2wSN`Ns458?E!@Lyl6chM!gXOi#jlrYCtwujz0Nc*sJI%mqG1?e)zkYj@RF{X1jC^{=p7bw@%gYMZlW@puV!~PKn+JAX6o}h$mfsL&mrbowEcHHeVj!Cew3z86AHo;MP+B@of)Wz(wbjM<s-~$qs9vyZE!CLix&`<ZsEfxXKPX;P^l5=S=*Vd@6GxCX@>w&IH`!bPCTsEr%ee3}?SVlB{L=bMdG@jlS;nhr1&y|?8D5o;xlesDtECl=3sQw757Jc7@O4hHVR>pBqvTEcX6BG2za1<w!^v2YqC`Q$wh~vk)4o1V1<Gr&&Q$qr9$uKrkAXwv$8CyX(9C)!5C@iv!ZK)Q-wP6>gp{551*nT%fE>P%bq9VF+O$g8bG{>P(r5xktqG<qVqjd0~EIurv3!M0^8<x2s9zc_2<S8+}(dWCjN145_!hHL3f>1H4JziS=P0f%6!^Clqa_e9eZ2B#m6J2g;Cl7op$Oj{6dm+#JoI2fVnC8*kVn$WKR#H%6b#=Xh;?V619MalmHT30KI~?{vm=4YFw^6)p?2zL=a;OYxy$$zZ?I#wS>LdggfeX1#pTh1^7T?kI%0jN6SQLDJ?%e4?plabWK(+AC*R6=>1!uL#b}OKZMDu|a%J)m4MHR!$n+Bc=iO^OiPDT__&SG-SSyY}P<RKfP>O%5BIcKabYv8C%>yo|Kbfve{7LhZ6dyA>QcQYjIlhO1Oq?+-b9zFI{og%0*{}92;8E43ITjj!zFcB?DI&4CAD3a`-#gKB^8SE|*ZkyTVN1D|Q?Uh!EGC4io@dPhx!I7$o$!lI+_q=PN1h1UY2vrND+vb#vx^dF;>Nm)bw+^I0afnowDF%Ln`AiK%3GrlMNzz96R4r#-tGYg(A#})-y%lnRsE6-?+=@ibL6KfwTS=wOO5R4f70eHYo>v2AqUoQo9x<%`czb=TUfJ2Ie_$sh0Gi7KM!LP0tW{;CX6T9pE(hkw1*b_mK)8Aag;BeMG3}f_I)PuAy64WfQ!~=&3=o7R?Jof1D*ZRuQv(5%9*Kml1GEychJ&Z|>IClSF><F@8K?~beUM@0I=oyc;{{|>j^R9LpM6R<FP)8|(y5XADB~1`Dks8r9T$j~5sc5d556fA9FkKwi>adFMvTt5aU7H-A~d5pD9MQ%YT(PXN_7z12=#d5N6O|LHMu}*8VS`??V+JE@FD|Kr2q6Djx2JK`5-m{OX#QRML{KuiI9Vs>ed*LKAN`~LiN&41FEnuXv(;lf!y#DhEQerQ($N^*0tyWBSoYmB^aaWpDV1RoF>KCCDtwwlW7P=cdC`0-MS4?<geGZ@3ZVMjif2Hj18LNI-K%)MCJ0-Me-9j<6zs=O}A+P59!#H2d+`xS7j8>%NjP#{T!SAy>pVq2HMd~3pB;Z3Ta+C-IM-SWQSBVkv+<_^P_Mz3C2ClM0Q61&C<jvc^H{)0sR$p&x%2xlX4xUllCFdR93_wn%~-;rum1-&tuRx|CtH@fbD<MpNz{n9^1(B<-D4&e&k6oh2hjNco(6n3+u@W09JPLSZ4&!WE}WKv8;B;Ag42P@#vvtw`z!k!wVH1{{22pa{QjcI7&K4N{u%@*&9!QH;|4zvj$FCL}m?pO3PwrCm5$rXL!D5W$46fwro$6j$kpC-BZ$y-pOXz!Ihc|J;6hCe24=EPpKQoMJDi83IYZ&yxE8uFA@Vywyzo^c%&3&-E2$`n!DsoMMA0Ac*-}_m|mfzu&csT6#ABIsB<$JC}?Q@hT3FAj_2-h2%vp=*dCL{G?<Yx^Q#Sf7mbtMXjYOUELGjS3;ZM8;R64-VY?uhNpi;Q3SgM_%jjt?9l5ndUp~$b20d%%4h(VK{<BV~H!a_|Zo5aWhDm4GJ~<)~EAFj-cFN_O{!aSoq@eu+2@8wftd99!M6oH#Nbzez`13g(yjNbnc5PZERt7Yw%7#)a*KMZ>*aB8!Qy7+tmG!h+E_-n%{Ig+9pv}j;=dZ=*Z88}qinUGz6kbmfncN1Zu8u7f_<2I%zi#!&U^h`NJxs@V7s$v3yGN3*$mt0Kei`h?tjn5%Ij!zM+v&+<2-gVgV04?+*$mJ;lbj5cQ!N=tL=UAl+ZSr!%cDqz(9&DkrdeA{?r#|dH!B;}Mz&`MT?YDR#K+UKWDh3q2=>=-9}XP)sR&<oyHhwP7Oo7ZSjtWC=>NW`Jmy6!2{2}Kh33qoR$nRqZZ);!Nvt(EneMuZ2X9|}8a+q<B?5kPR56#(cgKFsXDjmUX*^Lp;0fVpAE=9(F5ht?8GB6*4ND?Z1CO`UxgUCGIq437<LJU!!)R1NbtV3e?UZa!#N&{x!=(r4>cc?@v=R@abJLsHdo4+q;uhX&RLwj{fHGB`ze-`%$*8C?g1z}*?k-95(!<s~uztm5E9>)*Dl%-U6ogm8I1tDS7}q?@I*x0>)7~n3dpOH@-PLJs5{EES_=f~+u!!M2mD-%^*hu}Y%=OtDHlN=WTz8J=I4~PBl#ZdeVhN<=lfGG~UNCUG@q72gdd^84;yNyfqs4#`!@kp)%Yg|7EKtlz_}b3h?RT!<SnOQCR7SV&1xrg8pB1HR?|*#!j{*5PGinl{+Wb)h3%&X|x?*d$+Kh`gqeqq~Guf+;<!3Kn{msSmi_h4pnn8BR!}sD#fz<RhV@wi`5tsT3@1qIqwe*<YzkKyELq?&)@T*VR(EzLaLf}ZfAfTNu(ZiPkEM=oFFTS?Gm7bD!@5kDg8^L|hV_8C90>kBa1;1H;?4_l*JImdhoj2=`y;*%fsBUc6Fmz-+Frth~!Le}lK=KlFdix-ups{3(4i<cX;PT6hEB#)ubJ(Rh>5>h*H$GHRkHEk5_S5#nOjj@XQ3`@Z&yMzCIyiviY5ATgKaw$fn5--YoJg5+opN+X2bD;sn+f~C1mgZG86_x@=b}Y3b4T<3;jqt@1mlxqG`5T>Z2~HufJr{a<ma~nG>1&Oh|@2hLAQpe^M+H}Pw`q~C&#U|03)HywlC+cPP_VvfeU6G9Xb&h+wg1m;D}%bW0%MJ$l%Gq6O!+i&dDLrLzY<5v2aj_nHU`j&-7~v{^1u2uYrtRBbR32No=fFd`X#1lg3WHvQx>t6eZbZ%n3=#^JlkRs%6Bw>sam7B|%3L!*5%7Qfc#=WNN0AX3JcrppCTW7AHkOm$egOH!Y0dm<*-mG#K{|1`GTB-Z5-Ia%$l%ucBc!YYnFpM9Rr6(!LHyn0yMqGtHcC^P1^A9s*x?2ED*Zq|FO;8#Yt)%N6IjeK;I<2iOVnyE1&SGWF|&fhNoDgrcLy+9Ut=*47TL|2E$KnOvzM=8R%y+&HHRV@s=pl%$m2NlC}#{OVM6bkd*nK#G=ZEsPU4{=)4cDPo3$Ucj{wEYpW4AXHL>wxEd|o8KE_-@RO|#Xr(s!%Pea7mEprWvWCOO23_gFqio%(&+GWp*m9#%QqMe6e#pKzN@y=(T6A9iRTi%<<N~BL_!YRc}k^T5Qd@y=uGFnJ?!_><FOC)AKvQ22y|^Q?j8ZtpX{BPOIbYTj)qRfzziRk?IwyS04L)U5EoAiVR3SFtO`e-(#4Yj@?!<5d>cM+DH~ZXFggIeR|aqhFs5A)w9a1EneSRM{;*&E>38veEJOuDvn^aL$h?X92o%p5TRwRlN)Ass2ai{<+o)|87`8qtmDaX)Z^PcHGy;AUZ&ZGq|28>FPyLlv?%yql#prgdl6r~!B?+Fn@D`N0jgf^rScbxCMkj;uv?7=wptcE9gkM|5J47wp6HZm1GYc@z)vd~UwXs?)tX3L31(Dl{6Z>H`_bc_)dzE?=(`&@d06IX_5wiznK2VcyzKSCJOAuXn=l11JeQN{55ITw2X@wtTT$%3Hw|2LK+aG+Zy}+BwuKIhy8tl!r+h|SirblC#*zFI3$q*;-0ny^+;?0$f1>{GzDcB`A0{Ly!1Ioo0N71u|`)D6aaT|v}{VpXF@=zfl)cv&GEzn0X{2TrgU7^1`Rr0uu8r2!RY)zfLqR5t32x+v1@QOTms;bag5Kg9~pNJx2pjW~ujNo6v&Q>7aaDT6aLE+98q7(<)mHLl%t2=Rk8vS~e)L6k!8ar!|$8na(S~=6*Fcc)gJu$I-5Ww8-RO^NJ?^Wwnab$y9BTy%WcAGxRx`yl#@z<#i=DY9NhfJS8-;Rd9$4z_PT2XPoJ87c<rjSjmC1;_z)WZ8r{khzl4<KdVyt+J@12hP&4`G)gd7pB!0(khG;cq!<0p)1^jPn;@s3{4XATG%+o_iORsK$88nyMwb%o)#4g>Ui$UOJ)zj)N?)i`6<(mZf+(z7}7PSNtp0*)Mo6KzU&7Yc$H|v6~^=BZU6X<hX>cMJ=_3&y=ml;2;xa<mG7=H3_$QIr&kJ*Fn8|GG<szl{_~lnT*q6rhx(oCq_+*elUp98H!#Qk)U8XDeBEm!c9{n=)c8u7hnb^X)NBwFqh-<wRhf$@jR86ZZ2MPIkbmqe{6>fJPtcyX>eRj$LVO4o{1Ves0v`W&?pWMtS@A1{0%z1x;|0V3jV8U`cC76VkgHnfA5S6Xm|L7Fh-v(W`~L$;RQcFIWnB!?JC3pk?Xd17DcyEMp93(fhbETxZJyMBbC7K-(6lQfi!Y58U&BSV>X1+1Pl<AvE57h!$;QNhrNSC{a_%M3Rrxu8lQ3KEdD#zvXsKx4{Mu^T|(!nQZUq7b-TLBxiQvvG5>TUbRcd=gM%`($h)B^a4iyeTENi~JL8NLrfyu`JL5TQiO39C2wvn{G^hN5m|pi2yE;_72)|*hne~-dxt5IbQW#`dn#X>VBM)^I_iSYCH^;xOPwoqsECfs0LoN0QwKckkH%WZwv<&lB#L%<@Shd1pES7ED`>0zMC*Nye;_)<lEn^~@P+!ZM4daQY8XO<B&tAj@JZN;xnhQ(Cc&XTmX7oc$knRO_<1Ks~&0dxMFh1<z<bo^-mmdgqT2a|tEG{mu#EZpuZms|YpA0GZ+^OKWTv)n-6AX1wJ0SKBKI)E!G?Kk>LG&+ruZ(};wuu&&x(iFoz}Q~ZOLJw(sW^KCkM-%}k@km?aRB+*VJ~{kGiZ7Wi({8GYIir2JL|P=f09#i(+u~z<$N05TrMugu$YUjx0Y7o=6-naxOX19d-U33)M5hu&O63TFrZeT8*0jdw94gciz3j~e5ppRKUjR%{6x-Y>)rB_Z%~(#IQJ!M)r;Yhr0joPa@9;HUGwV9^LUGwyb)3ugI8uI0&10RXR&%9qE3tlg3B=;DSjuSG#2JFiVyI(-%e9-Dt^$i1$;v=`hR@&_eQcQX5@P3S0B6l!#S$9AJgc7IXMrm%0HG{=V2`7(JiEvo;+pV<h-!Dqyn14AExZ=8>dAr*QG@*SHd4cfX{KsNsC6rB#NXmcqQI*-bii=s?>6LTD19nUn^ebGYx8+o4{oL#a3-I^K#GMS&)rViB7*Kegu`xwICIH2cGf|`BZEmd@g?GOap(fb{7;-zwLLp%k*1P`ty4=|L|7*_O-@r_g>M*cym?s=H2(KSN?vh?{P|gV9&)5+)4OBW|TkhjO_<u==a~!?7!ydW8bs4X!~cx12gVQ?`>uPri3vnV#FJrkHWqP=Tx=05to|H%!{bQ!hAHu@!A(BWjc#wm2aPnQ2C=OP>1Idb%>}*u0Wo#OOzizC_C}8a$K1}TGe6C8;XI<4bCGsL~c#MO3qLV|1}{l{ZZR!Edpz)hHd3;W_-kx3bw--+=4Zpg}e`@3@b5icfEp&a!TM)`9_^-JH5JEsA;W``DzL`p#~b8or<$YiYbnjBp&*}hOV$(-!_Lu)4Hpp2WQQD5hKtF(X9PFyFyy0>9x5-@VI=7JJ<+Cq$f))D;7~)&Qs<xR3uV<ehsQHVift|^_GnlqNo&duw=DKsTUR@pVsU^kTY8WQnIub#B5KCzM!Z;9fXRpgwSEJt}(w=ni2g{tkblLF)EQ%2}bQlK()uJJ{6xRdn%W7TD7{ez3v1L$56x@cDm#CsCTRoDWp5M7|Xk6zoY5%Dg*fh-y+=Q#nD!ei%03{jLy2@i_%pSqctU5%Cta6{O7H=G*vjCPB&l&rKsBhw$D5?aT6&X$U7j&t;8$v|7HBw(jxxJZ>H^U2B|1FL(Dg{w18<=ZY(V0f0l17Kqc_wJ^W8=TIXPUPh7PVb@=2+Y$@)39LfaO<^bl}59AKQda7wuXj)}zr_ZR12vFW)5DzkrX~+iWF`8(;LykO_o@i6K4@u8VCxkyEJ$o}GO)V}*`OSy`3kL?_!nuklhR>o(f>_W@I;%bh;qU}j+f~|YKWB|h;EcNb{af%g5zW~Kdb(O(1sXZp4atyY(D%V$c;|9?abhbR4q%EqOs1gu*7jG;+{(<)&-NPlhR{nX7qoD46D*?4BKQg6VLl~q0G7dDeS+2&EdMcKaz4BGDr0~@b`WYZ{S%s!DL6G$54rX;TX>uYOarf??~701>Uz|Lt-cs9MfppDikd@odTNzjzy&E5QN!#TUUCE9oS_K#98o6LbxV&Eiis3GHNMPffn!F8E{9Y;0s=E_2oByC4du~cchoJ+=r8xL*q;=Ni!0IhGw)~f7g%;N<EdjR`6_GoD}(};tO5^QTdVA3a$BqJ{y92$;OU?~`2L<6PV{i|Zf=o=5~IT4y)tj9ui+qCmIY-^h^8CHn6os!iL^azns|_{3PyU+LysD9mMj|`?_XCf?WqLMpI{N3o><goY9+$&E!{_uOI=Ae!)$&H><gN$Li*fIE39Bym6vI9G7#RRP?w6rapRUev*GvTYpJj_*`Mi!vrdFgcmW5hTy7kfaR^hHY(?f+0aa{gsWm-D6~y!(+0iQ?6k~P<`;-mw86;!;%Jhi#fo6v}8JFb+DpM!WE+jgMNtf+YU<IhVdRn_uV>h21VKfOGv*DOij>qnvzC^u-S1I;e0+86@Dei){a$^k475Y2!M}JO$=+^3Z=DQ@L0~|6Dhgz|Nd1J-GlFNKvG=g)!<B+o}V?(#tXa0DB$x4ZHRcFWwg4v4=_Ax7z^wK}KFx5FlaVanFbcThpU<Z_)ARrL&uL-0b-9Dab9Q1$&i=NS6GYj*QgAFjn^zSczd-1<7{vXDq&rhjq_yu6V1d4?R=(eQW|LJ#MA4>m5dgGm?ca|>)rN2ZevIR)G2%{h$PEIlW2g#8cAo%~tpy>4e1*Q>}`L3V7`jn=R&i2=VN{!8o--}njU<Xn=0rlTg+{81<7xJ9FL?4@221>m8D}+8L`1Pj}W&d+{gJBlWXhS{6I$%H0=Auv0%SWQ-LpOdw|Nc32@fWZDp0Z_sg}aA){5+sOL65(|vmjhjzk0*K_t%#9u6Guf-~Kj%7OsKoeyy=vSx5JM*nrRJC_f=A`!5;G9(S!LeK?%o>f#dRBzQ*4@D<OVKGGR?ev`<4FSz($@ceU_OE{Fyx%ft=Qx{PL;+e9tJwY&huHM)m{J?U!f)V`kmX4L7qzF&x{@}Hz4TxLA8W*AXfBfp_X4pf2QXI9nQ=p(KmGa|oe>j*F_tT>uDx1Q!?q8<&Pl84|K#=YI-oeSJTZI3h?LI6OmKNg`1kpXkR9&N<M|iV6ez<}Jb8;0xWX+)59jB8+;mypSj>D_;;gzs>JUDQ*<T!!}B7!IV^bB@MA-aW+uzDz7z}n)@aj(2|V=>;_8zSsI;f{n<DCDo({oy#{Xm_V}ceh?u3RV>Nv9`!>%*R)L3VQJ`zM!YiNPhc`pr5!c+2h-|_bD}ka3@D}o(7!yCxTa?@Bb($hS*H>B|Ag}1x(ct*tN;Fo(2NYrx6)jVaQx@nVVvGX7Nm2;Wo5)lyP@Lcy%(>J`N4Ro>eQT4DNJw;zP;~ChYa+8SYHh1d=AAia=*XUy2e6DVtDkjz(30q-a-HzqOWEGyWqd2fos{sWV!$(Ng1doS5DL<n?9DLAl^|E+%&-ce^3<MG1pUEeiLql<ppP)TcycoLneRNNoTF1$WdFByhpi!0c<5J(knWiVV9C;$72PLl+cRR$zQH0~J?JvMkBdSKtLIH!ruPT%|2=ld9tRmoPHB&wN+$1oJAj9~96sCf07z(-{1}hzTMQ#a8Sr7(;CneQ)a3yM&>If7UDd&)v-*Zf?E5nXCej2K&cqfw617sJH^Z23qv(T_A&TdRiu`c(QlgA5O*vLB7jOy`x*$S$tDq4BS1O2^yu);9>>osYfA~o)+=+OnQ@kw;aO8Dlz9td?`>Gx@|uu9_G2=aG#4rLBi2kayKsX^C{4<Z`fw~QY3o$RA--gL6_{u@+yzQ_>|Q7<?!P7(8^yUWfkG6*ucdXVSFY(_#f_-3-6x=oHa&9m%;-WOz0nRzk7f;LNmPh9en)<IX=pcV+05?>+uaSSUl|9&;AEp+kc`@h=Z4DkYr%*7q5Pe8v-t7TDQ0T9h>yp4^Sfuw|OB2akw0whVf{~Dyqyb6`Mod#^K^pG8ELuqWf9EZJF3B>Fm2(uKloVycJi(t@gua=*7-%KP-9yv#Es6B~Cf)JZ0@KW)K+c+QST45$E1*E3&|%`78gqYd?IQV|N{n9V{80Iynj10fobb(CZlq*9V+UD8OkJh7*x&W^dF@AJ9$R=pq0hrwonx>VePc2VC7wLQ%HmqpBwIJD(6tC6q^Pt7yGtdcNh^cwmU=<TXLgGQ9X_$&aX0#wQ<{W#Yp>4dcRyCu8KyMHHW&O}eL(GHREXqZog;hX@f+4u|{u2vbS<1>hz)OkoQkW)LjwtTU4#U&?7?Ch?LAms}cvukp`62_^m?#jw9J$kjj0qE3eA2uc)YL(r2jcT~wKl7n$U;Dd@iCYKs>i4-OLQ8be>EYlh!DxqJNBy3i9-ruVK(A@`vu45jR-`yfVD{`;xaxN2Lcecx)mf>|%ep3Gbt9$q6Hm)mQ^#A!3t)%8OU^T&qNGSn>+=56*!VMoXNhx+1-Y$@6im*Tc#zUek^Hyvpo=hbtndD^7q%t|qB=`JzriziRSdt~V>Ygt^eTDfXXTA2Tdp8Km_C0k|6<KO@@8{ZUujg-7@Xwu6r;UER!LTs9fB~;)a!NXPh6iEl)H*+T5AV_W&+cq=na?g<lLv*P66eIlC^*<bG0{=U>i#U^+h^*dqrm0yK=mXY^Rct%X{#R`nJ9ROt8#1+)t{k_=OpW;EDd<s;>Bj@4YiZcT(UX|&!mS-7?v<>by#%5vxbUVmMU@(-QQIZ?%P)R0qsRRY;ZLKgD4&D_;V}X-FCl9-w^MX?4Prj41Di)llk7kwH|-Hyjyu!T`tga<=t(!cXo>WC$~wgZaq`Xom?N+7e?iNH0?G_ec4XAcT4nLIt*C0<lWa55*tH`K;xipZmGC<zf!t1@1<=x9fihl>i+Z`#$q6pp!z7f-)e%Ae?L+*_<Q5d1mUg{%tkXge2p8uh>glDGmTLVM=^ZrT7@r-3FpKzl)n09PCt#K16<z1Yy`4<w_geG^cxSt2%m9(vr(hngL|fQ<xB>QIVkoMjyr)}5)?W&kd8AE^4WztxU1C~C~^3w;|+ciMsk~>Ak3M3u&PqfQ!R+Wc{yRT6!DaQBr{Rg*7XdpBj7hDuYNXOA}%hvTw)%{GbS878>HT06k?gbo^&X(zA21mAm(bZ!uSo<mZey0!>?^5zO?Y2-Aj!t=QI+}m4efM7o#>p-+WlsPynQT=(0Va3|(&xZ;W=)io-$z?ThVpcpB7>V4z&;ly;6|MA3}H)mNX8Jn)n%p^`xIQltDlCBFu08+bjOKKpu2v<u(eYhG^dHNJ|8_Aoi@_8)K+s6r@}l9D4t(QqlkwdB^`!t%awBhcbRdP%1QWFc}qA5m%<XfdA);L%T+w8L!K&-YqA1b#^2u8@EWLa|0&**j>otZ;?eTT8VSu~za#2H|wLy0Q@0mTz$h`gpZ|eS`l7bbM=NYqNwU39V4}+HD5O^S2{EmSBL^H*4!#wX5svKbW-dQmgl%vuic5T3xGOt8H#&8t9KYaB4fPeYoFCSamw5s-R2tfhNaKnb$r+uNoOVwZKfK0d!0JT#3&>vp@>dSf3JeMO7Hq*pj<u44Yd7j&`HIwzXM#80yeuDn9AJ5ukc!;}>96N=tmv4=qV9KSEUnKWd~?X#DQ*uuGlXftBNMX8r}#t=1mKzs4O9enW{h!@AU61p=AhQFr$p;F_^&9U){u#e+HW07IB$gUV(Sq9XMtGvH4wvnp1iQwwF6bkYmsXu*y_%z_N|KZ)mZ=76baREColbu<JmO*NsA>UC)_^@9v9{oLQa3EYd68h-!-dyh7gYYXDV-5fa+8Z<zAz*_VwPmZ1+=Q%9k`a*J|*jAODgPR%cqdt=sruHlJ*k7cpqz!@SP1q}vX1#}+wf@O3v*E?BUnv$Z5RjjFAO>O0C{KT<)9~r0`CK8sk!z2gQotQQ#*-O@7jZUO?Z{eP8X*N?>~C9-cS@9}%5`6vSg9y0p61YNMi)@PN~#$#5{cB!894Bd-K1YopLl}-bzpQwy(TeD`kxZB7W}8cZX85qe5&mpdd%KD9nYC16FDwQS<S@xd8cBHO_^FkU-N9v_qXAHoEfuf-|DhBwoiBQE6&2G#_OHo<@wN)d+btc*}Gu$j^;QyhN1V=4)1U8?GzVV9ga1`f9XiW_hh4TGjilB2^8d0AZQ}-=F*MYY8A5^A0mMtmmcmN=jG*m$|lxn2Cc}?#lg{4-W$J4ysc^(>`SUlpVJJ_COm2aw9!VdRl;`>jy9GXfsP$7T+Zqe=2yafDV@~+CR+Rlm50>gkU0{OHn&hZNE*$;+|0~O@px=r6g7i51N4{gQLUWgF>&ALSqi-zs%;P&B8u7MD1X=l{0lv~Jbz}dHr<Ws3~34xfCq%;UKmCNj6sS}Ka0rd8DUM5aVu7wX#R(dUZLGM+-)|3du0^3O3l$>Z&0`wC7l7?k%PTfOT35zhk)1=l-<Ya#3^8{V1obQ;W3lDmp3_GXbeBNr%}}CG2UVKCSsyyP226%n&L!uA7CXDv#AWXeuR6<d-d9@3}Wh^Ga@Ja^)qrHz0hXe6BR+jFObJYIZ+=g)yw$LwXpl_)kmoq4~~{m>QOXqXqL!KR=#YuAcgOcUkcU3X)C7Z8DH&>1KkAKic4b){q>U)Be4|vVS!Af?{o6U`x>0={ncvi{f*Vl+WRaey^lKi`@L>&#Fc`IVYC-PJN2_F0Zzh;J?a2!4$fNGF2XNf{d!!*`tvgtxP8xkMw#xNvV8ZflXBL9kbsf>9JT7-&=8AcM~J;_kkxBL+G`;80JhR_&{wtJ@7}NTZD{#0PE_y`>T1fQLm^G#QvFHF{$PihS;Q0%_70N6h5$nNB;t^DEtQay1`ki8hgvB&+j9uTVJQ{#=~^|N%Ht7+%G;s59EF)8U6w)!@6Djm8~&W5KQCdmDA!ue={(jnOL~kf=7Q8M7e0Y*WnZ<2#>FOQci<aqVrgB$dq%<x%X)07#;{W_SfN%R$;hGphR&i_ze3s|9zl%ByU8yz(S)wKpE7Lhx%tRc86{#Fd*$JAN@rrK0P5JgJ4RW`v-IqT3O|$CML8h6327rbwis{}hzWvh$H><6p>*X&`0rwv&~zVA1n|$1fl_P1><seNU%dLDWc9R2**eVN>CE*Sr7gn7;%F=v092EHiY4#0lSZd<hEACq5vTDz>1~XN%OY^%ANTvMA-dn=y@OHb?w~-uMijo5YJ0!Zd$%O3eGNf!JTD=q!yHz=)9_BF;Cj|Vsp*UcilB&ON>s4Ycy;G#n$WmVTm#uy3E#rj(GpJ0=PJ7V*_nNaGz|R^#s>7krlA6c)wB_tXeK&rbqcdHGm)87k@DJm1nrm;a>ES+XTYz`mzaX;S&~tw2{%HI&OMCQJJj^i+QxTwDi1@3__6v3Wl^k$dYwFShpZLnEv8l+(;B7l-a%t<z!11Z2xv+h9>_=dAf*K$yWmBLgvRz4@9g0?+61q2u2@Or90}TZCH9}F80B2Fw?%`4Bmvp5(W}rcn~nb?o!khQ@|k@NVJQ$PFQ78?ctj*zry0dt?ZMFxX$S#TQ)qOCF`KZ1i&l`kLex+DSTNlCOX9)5w75u6V1h!@qPbE&A2OEb29NO(adAo+IelJCa}ZboG@!$OaXJWeKxSyHEpoeF`(eDX{=?0T7hPseVl#Z1<`;tsFLr6Si{KW`W_-Wfze`JApg*_Emu5jtzdtDL_mkv3E|)kc%tq;`BJ3gQXPRR}VBul7+igpXa1`#|iOIbdmcwt)eRuvlm+^MSfDQBb8vTz9r$@v1e!tOEAET^u2TNDplTeT2%%n=p2Z6%jfX_r3tJ{&bx<x$fYzt`P<?k=ec+4U1M0+kU)REy;@-_dv@J{p&@tFBb5ym;p&Jw5L5fv}aWnwT0M5}oeNu5Qz2S4Ccgq(uUh97#0gCZQ1cRRv|YNJ4J<y<}go{geS{~+0`EakZ+P?|<CMhBcg?M$_J4yCwwj-FvLF?kC)#1{M)emU%A`W7X3;#S!pfpB|KrY$_R=WL4<Vo)Nojd5Bw>md#nrLBrg+?7$jF$OH+TJLU$_dGoI9)Co9_1uF1-D|?Nc1|)LikV1iZD-CaR*8b^3r+|fc@FX6|LBuBl>^9!e8rzq9y5ony|uMbt=NT-Fh4Dpx_VYEMDY-Dnp$}K3Yp<`A`zM(UVX$Sg<xSWbgpY^kt3B4yPa4{U&I5T`_`~^4>THXaeY}k?XCDQUt6|Q%2Xt4wgwc<CCzbS7kQXJQ`v=Vf1sFgPrHYsPHV5#YqWK3AFk*{Rg9$3=P=C|ob-Sr6qh#|F?=JHWs_oe@xq+6B2I7#Wcr)Jql0OC7f+Y)rIL9v5GyrK>qY+{0{s);Sg63GwWm@N3tOXno9B3~e{2#hcT!&Ac&mDKrM6i~mCZUU`x4r}l){O!#vZ06y9x3xsYPSwq5v{GdCA*iPR6B-jx@^Oz`!VeBvazR(`KXD$2LLX+e^3}0GA!*k`>VG?6_^_f<hGjM)N?wKGD?EoUXb}8utCxAdy^e6ey)5I5{L`eQDy<C9Wdx9NF#eHrhC72};Fs3gT9jrmaJX2|gT(l-HbfG~pcChm+R6A*XYewM(?=JhPV{Q#s~x+BS_j48R(!X^5NujEhFWLs|QrmIpnmCa5a-85DqlYB-&R_G>i5{^|6S2^&Pv)0ip-HWQ0JJgs7Y(g&P6fR;i3LbRz;`harvlOwd!U_DVi*hhGrbL(*AG<w)xxMG)7)B!ZoYHcQ{`G!2MxdG@!7{Z;*B&J%Z-JCBcIj6yUS7x0WwN71HYEm&pN$bozNy3xiep5wILG0ISm*J!tj;tGUOT%Y*^jHRv&T9Aeu%E7++-x=Obh-puuDW>P=JS3kae{C|rIymY59VXLcCMsIXr4G9P^`8x)-BAjMaXkGsdf}3dOR8=(!vuCrXdaTb5o<uf62|OkqqGQnHtpGl2gPjy+axH%&qCr62mssH9l_%0)ZTnk4w&!$F?Z7Z2KJlqMFkehTY^}IRnVkoWsKyCi4zP__(}hK9F1DL-Ou?%7-{O{cW1{jBWF*0sH#!4jS~{maz80Xg9@r<|V*MtcF&ICX?pp<fg)eq$i%HiD8)FB9Rg`0sM>&62a+jPJeg$NAK*Zh2*HQH%y6|+=y@-E0-&;4In&Fz6SPZuA?fybBkx19)+`g^y&luhYK;4i5f`NEQ?Xj2a;p|Ya)56VDX`f`SUlH8U;B>B<k#pZ!gCk%oW5a#=XWs){XPV$M<U$2a7#`G(vNBk*Ev`ly5|YUNkThirXln^zC^Gu~}^OF?oeP%i#<CZksBHNst&{qph|mAvD`(if6&9i!LJ36h5xtFO*IL)U27!XGZyWmOWhiBbyz!I_N&=qOcTT{Pzcn38#2prWxmk<eX2wMW~@ai<Ix)lK~x7;0Ec&fFfq~9d7x-prt<{|4<@03$a#tPRp~mzU*mrK4urvgV@G-GnX56@fQ*fUf{aBOqu{GR7Ds6G92Ps8pdz))BqDi&Z8L0&QM&S&K5G;qG)5Hi%HrV9wdi}+t??X!x7tjE)M)>IW1rjI~5W(kOpuDpE8(&#dhCX*?cSA$Zqqn3)23`Bkn>=!eIlA3*ubfnWZ*La|do1mWR-um1_0(FrJ^1*k}|U<cU8)h3ga8lfU@bx|SfBts|Sf-)P^Z7XM3Ddd_VpnE@s>vk>F*#S8?Yq#d)9&uMTU5V*-N9KJb5)Bs5z<#A!&gG{Tkf}EW33%t4i<eMh6(&=EGrh9|<urZ)SKRU60!kze3aWr9`qSbh=qUOw;b&EPA_V0Yn6LtfgnqYQzBB~X30%VsQ4bf_YR|_=wJG|{}X7a>)){WpKr=ID-a^?P7A8<O{g8slodDFr%HM$UYM?)<3jz%B6W`->xv)^Vm?TOj~0UOC)HGK0WLv#>OdfR~8`hO(C;HjVk=?j>VRD83&k`+Dlc@jy&0YXWgCzmKN4P;iX1iVD`ni=I}lZrjpp%LW$hgcS5&))yWu+i?`VHf;kP8v`=le26-<A2CjcsAc~(MAs~96+KNlFbx~Ym`#4`NGU=x)Acm{Rf&hwbo(tKxF-owT~$?C`Zs`5LFaNhmFDA0Vk<{5~#Bymm4Qd<f@0rB4$lxaCV+xTL>-Tf=?J}_EVBR1bv8Rhm9a_rnD5ZbJz$xo)WoRo!-n{lZGe3Z8OjVbeGagZt5z-Wtf~}7+&_;RbLJJjm|(yrc$214$czY3bQajCvc=klC3HI!i9kvMyz%aKmRwgPXTT7r*cmSCXr{Er4=#TkR%}{8?$TFAF)<LeTqt)58&OY*>o2agwZ%ekp=y}>Z6x7j1~2uUIbN%5<VMh)&D^H(8ppj(Vz@kZ&K~}ZUfb*j$Y;_f6LVPWqKe4TBUOLknN@0lyLuq?=Di0)mqeMP{~-ZBQ+iel3nt`$?WEf+iJ|l;5L=eaG?K^Q8;pMLaRN)<kR!Ev=0hJ$z$~x>q2d6i2`GtEGY%_S-5|&%|GTF;hD>3hz>sP9;R5MrDuEYMz2l1An7P5kx-NR5O`oP3>w!<pCdyOTd>Xw>I_IXN7US3luWHtk>n0T;Y7oR@I6N7Nft=mVCrQ3S@6DerG^wBN%jKEwuQGD)_-pCG66RdYw%+c`A=vN=JIT4Ws0EVUJ#k{isoIL9-T#BSQ%x7DTLp3tzoBsHhMN5tFg?U`#gBi>w+vkNZbxTC9)#?q0`S*he_K~%n>Ff-yA@OUJMTC(lux6!NjOtv-AmJ)VcVXe4kT+7tQy6OPAtvtz>D@@~@=1c$Mh=5sBi5qoJ1c8-3YJIOmifOZmeu@zVW>27vtk8P8fURQu$98KIREa|rD!%nIt|bM`rq@GSY95^Ib%3S!q}XpLhTVU_j{;&(@wggwMc1GEuD_}WxJT<lA-9zC`wU-KT31qUg1qlKigBA2?7nla%YtMg6gS~)w$LSkThJKmT=%V`!|tOTypHUGV|H+BvON9rdzt_&7U#w#{Q?iyEF$oHep(LHQ#Si>8k<<`J7TAP`(T`TM?v_(mO+hkx#IelTP_OU1&H%xeqA5sv}1WqPy*UcG>HKK~88*<wMGITE00y{AGEvt!kRe0;x9O%W7JNS%i9Gli`vH_Y4nc7GsYd+{eS9_{OZE&;KM8jA5*Q9GXujwL5dz<FG-X_9N)I0gY*ETD+$So~@2a=0QT2{>~tS4nP_$#eQ4s|3AIQ0T++!%ECb>R)|YM%=v64>T=<kuwQG_X$ZQY1&9NDmWT7^E#mRpaiRV|fOL_mi=17;aEk0uj?3Mv1kJx96_fwY0pD!esDlnsx0B^6iL15dNz$JN*tEx+K9UeSL~8NB^8Q^^<J)1-AKRRx~!vdKp`<F&N-!c42Uwuz-hn)1640bXA6BC52<?{)c=6IrwDw#_tqFWESs}L0nmR`cT&R|NT5zU75nu_law(N{23{(AC1@=<Lz{mLMoTU<@bEXfaOlHbT6wuT9e2zfn{>T#vbpA&~JTE&W*_n0`*zwlV?uTHRc~wXsx-m)2KT>symF{RdjQ6Vhrw9osa;>7mW9yN#i?;m#z@|2N?aKOy^CAm#z-j)CJc9bu$Xb#t?}wOO3J3mXd$31EGBO0!U8Nq*|H1bM}L1*_O&3WZkx<Ky2tI76bnE=nTPuMr^2!YMWS00$^=F4kkR4xUUxl4OhcoLy)Vs{10BJV`LiiGv`Mcgm*mJa7T~R7*gx6A`76zHQHkWM^n!u;uXa66{uWs|Ez6x#47V^@-*8Hh72b1!9mK1T`mT)qd|yBzi1S)`4790|V(mB0C{mz52x>(F3zr7CbC0>l0%^FczRmruQe6{7whv<1t(Q7b|n`+O3rpk=wG!x=-DpCo=yK!4ih^BZZK|sEtIhO#d{!pG%idoYC*(R9t6^jY{s}rR63$qFki)#}3T8?ebU9An{W5CZhZX3Pw{D42sf1Md~+9#Gld<KTqM+r0TTm=#QqBWDG__Atx9gvpq2=l=BIS1R&5Wq0U+;!hd2!m%<8L6n;zvb_*okM=A-TSR50eKkPs7gep^sUjcI|{ixNWoEF_4R{MP~C%}1c|4xM>joaNub5MAXaz#k3qqLW45)tn(BuZWoFJ-P&-lGf_UrA~8Kbchl`eXn{xf#P^rD9!_v|?ri_ZG6zJUkv;JL-FUQYJNL>7&!egke3^Nk!*q6_cfDYx~0iZh;s92Ge2D>@Kxce4WNfjPx~@Mn^uvbs-cbGO6W|kCoWCQM<KSi#bWBat4*VWs2-CC(LUjv`3wW`Iyn3rZq$qmPjbg(Dg~Y3f+Exlj03&hCq<cACaE_Ol;~WRz~n+0zZ<g^vsO7-x6D9!}$wAs+0l`%*u?}T4OE&Bl#=w?VsBSRF4ATvM0otUj1iw2?4Vt&ifQOR4Sk4^-e6(!C`{Q;YbYji7SuADd2BPprHzBnG`w-VmGf?^k)b#a*hx1fiX3E30)z1P`TNFCb&^#d&!~y)gHfi6YGj>Zmn-%m7{C*l^R9(V>^}5dt5_0Gnvmg0H2#gNOyY3xbDzcl+GuFXLdSaaeNhAi>r^xpYB!8=w-22Vx*?h;oW8nGtBTGjAsy0Js&LWdj`4z(@&|}1Ibrb8Hq-vfAJ6D{oRlrGrmxHa4;IS+5|j<)9CKLqw>Z0g=qeb*_pZdH{O`b>DiMwne%2QqYoCrB=q?bD^sa`-t@@g7<{VpCi=Nqe(4my{3P8J-*~AIZe5N+sk`Y#C>|?LV#-*!7EqKp^Vn7ft^W0x@t;|{V%GUw$-Lwrkz{#=n2LOQJQpP@1Pz~F<*Y#wX4uG32@&;;v*JQ|7T>)O#5TZc2|X!t`!GzU8meqJ#u6CB!!D)LjoOWYDzR$+HtM(4^W$S8c<$13N<EpD%2W}tT&-OyVGbyC+=_^q5aMLfO;fY&wEldw@x;YQh!5k|@ibGA)m{-Z!lC2iFoGduel;7FE-U3EAZ6!o)xLn%i!!gtmd-Zr&^Y2Zv8<CgM~wxE4hyL#uRac6qwV_U=B*mlobsq~et3+tnp|1Gj_v(diZ3Ety+3g5N$5R;tF^5g>&xrV%G;GN48JW#CYEMWHIzh=Vj+k^2mJ|GRHkcOe5|K1rUEkkOgw4NG=gas($p@yBj4kD9|YtNW3B#$(dmPQnSimOQnjj-BJZv6MpE7*74Y^Y!^3>RF<r#$#E@V(pJAk^eLhDulWKuE7&Bs8dBpZ(6=M;p$}qXe9(E9O$+zsx&sA=d9R?(Y%|0N@4o6KqW&0|+(`+|^z|zPLlL1A1pI^Wd|0)CGNU<IQTDXs5&AnE4G>GpI9BkV-w1fZoFV=ff^Ee(1N6iOFVgH=9AJJZk3s(fx=dQcwxvkZ|bkK9qxaGG?WJ|5VJ0A<FJQ5TR7ZXUT@JNIv7KHc})nIXpI=%X6(W<h=-S{hXxo#hk`i)+nU1XR|%JmsgvBT-*+U;qqVW$jVTwO{`UXHQGY_EzLex^y9^0LG}c4E~cqq1gSP(iehfDimLSu36d(5Vm!&=T?*8_ovB2WT3z;RG(w5_D*ke<iwqqRWI(u7!>6{6ZbQzfw>KEy!YE#m)vQO32h-QkFc4x)(FfQ+iWQq&%?v6t8||ud&Y(MmDTR#xm}SNe<>_<}O=^r$n*DHwg38Q>2XO$G1D^pKz;~kVXa_%>Ktu{tLqFIGl^)3&@U1!}>)I`M&U0B(O@vrE3!C<%AbMgVCO07v?cIkoNXCeLm{i%=|RE5&z>Sze~9i*@y*NeD?vSS{Rg1T!#6n*}17p2(z#76@;6T_e~e#Ou;KfXs$s6D{vt?9TM(t9eiASd(3~*#<s^q1#)Vc)^cmKoj#uCd=lcCI{%Xk3gQx;NX!mT-WdzB34;9BaZETSLC0Y&oQ|Vxwkk#VoDY-wXS(d1g<HYVNZP7NC&l5Izg1VeH_n|bn6;T{jOk2qHB$$~vAk<z>eCG2SpF)u&7#2;mKe{jK}J*Z!Ann%c!E-2*>iCMoZLHImr}Gsj?_-20NX6fuwl#TTA6I^P2E+rT6d5|2W*Z;QS7lj!cU|e$;b9;s5qBwrC7ThsJVwDbpbNpm+$}ctDhoJHbtB(xy8-#m1TcKj^*^t|3sW79}?a+7rgrk-CXHw353OORHxHG;XkV147ikK%A1>F9YFD-l}@1S<(aIa@P0@ThB!d1yQE*N-;9@Q*H)@qwPpQ68)ne1T;^i9(A4F5?l8~p^&TT3pr#(`EJ8{BDIGS$$dbm3<3IOWwj&E{qcK1mQ(@^Es0U|YQMM@JqcT^Vw9j6<(aH7*`&iqkuVO*(n=93|uf10l^bLeFGn?&xeL3E&tyb5z>Px<^&)wa-jcgxEH~jkN#t|22*GUH1e(P&DZ*9e!H>zLD%&=~vw-?e*zxKH~;f~o}SJ#(st$YLBQcR(zzs;K~^)0S4{k8WgNeGxfGB{{_)BBk3Qi^Ub#4EMs>$MGps=`Nzs*SCO7~0ft4zRWs1L1$=0VHi5C-2=|Q?dG+TyK@MM|@N=-~L8Y>{FG|t?q|%06Q=>w{SVK3x?USglu+=ER9ns6q2@43c`z0{!2TAV8>x$butDkc1Ex>w<w{8shA6rfKHS+2wF*S`a5zS39#%(A`-q}NyDmL_QFUlDII`C)p2qcij(WhC7;1bih6ML*MNci#I^?Oe>m{gx3LY1&Ha=$(QWay?1{?DW=x81qgkV3>i5aaidm4P=6)80uDLG&bmQGe?D@f^uzWaUJuYd74|yAki<7%6{&%=6z8S}bgEZBvIGChDWI53E%qI%HNc;URraV_(-21E#t~DTHKp3p@IlX?1U%|ta<0$-QxA2?VK6^JN4qfo9)VG5bbnS^_aEbsxlV%`^GG#bqd+kv(0j(kFKL`e+Jwz(PETRX&?x@us8rL_~NAAFL_EC`0GPz&!vos1;SXy7(s=c+PHfZcQ76w9O(JPk74LT@fYkYi?whfZGC>6c`)Sy);3}}9=yl>$}Ui`C<f-DrA1vNAee5sx_yn+*#@?B-!4%TB=h23>i2rN3THG;-fO|m$^-fX!|Qq8N-lZ!b``UMdK%hl?L%_u92+=jPqR_l0LzZvll!;%Rn3&>lb7JQ?a19|A?28g{&^{aIdM&BMM4W0#%m^+Xh1R*6L!W%4k$JM2!+Rd#h)~Wgighdd~2GdxViY|4JJPyk4Yf8ffj!?iJ!fDPS9>hQj)CZ=uP~Vy7Ik9`UF9#e$97yb4S?aju#@0Bs;#A#%Cya`7!m-_Kg0(7|ot`WPOWk&3H^53sRM?AMATEZGiT^20P+N^sXT?BdhlDy+=FD&&E2jpUlT;Ks{3Ibvz`zoOe$6+d3_6WoMZLy}QW(3@Z0g(rJKl<9L(CGU3J~yV3I8$3h8@yNC0*vOQlWb8p@-G-n^LsS3yR8bq)Ouvz^Zw-v&+D?B|_duNfk$Djhs-CW2#%M-(|{9zZnm^cau)#PP@Ac)LSTWU|_tp{)5_Ds<NtGLxVpf#7+tXBEglO6~8~bK#%yJRKQt}hNMuZAjJPszH`uv-@nru9*lN@6%OO#c6I7Ujj8u$roJ2RJe-?}Ks2^mNG)o=(HbPd2CPx*FsUITVIdT9*sLwx+5je1)`UUVZUZJEMG*rU;$pxOcd*;;V(t0RS)RjQtUj-(AlDL_;^M(*f4_B92utveRJtpA_J~hBE_>v(me)%O7;(D+Ypi;+U&IoZ&A78`ww2;V)vHlsfXOgq0GxMsZ+ovK(a{=+ShvP>y>p@BdCAcI&csV^t+>G-S8(Yn<9>qLi~C{fWCt|jh@*SV$co1S{dbbsoo@%LQfJWiUK*F232%17u~~CHA-nryD@ouH+2%iB_Z+zpp0GUhGd79axyGQpF9zb360dd|lLuZ&d3%g8|A{1e^9#fh|K^zInXQN|uJW))!i^*P8rg8$jd_j~1?><tzLx)v?Xsu5Y!$nzMkv(gWZJMSWb$P|iUTUe_}0;I59RL0VXxo_^{TodkbalH13_awQjak~u{BJk%IIG?%LO<{3<r1@?N^@67hiAulAb|*ZMpW=ww}?B?tm8*Ur<IcyD^baV~IDUrN)>lTV5?|6)3G+`7|P*(sF!;Sbyw}$(WZk0tJpiKtX`V*(p2JqfVmZpY&YaebSkBq_1!_%Jc`7iru#TH7Ce2;)C>q2@o*Kw`<jn7!ylU2}Gg6uZz)mAFH)NLw$Usx_KiC?=5g<-d-YEbTCr~IFWV;8Za|NRp<76Ec8^{*jV4#j6w&qlhdgi3>ZXuS<=K$Z!aMI!Ca}rX>|p*VN<?MqxZtI2O6eWzFps_U$5h5o3-1uwRm%>UR&F$U#l-kI|2j9*ou7lBQ*~Th5y@_My;`7E-CCnu{0PR7K*TVp=9TCnvN7Xj6yeczpQN6_EPIKT|zh+Az~n}B=6MvauF+wVkDvbJFWvEZt&B7!+yGfL1?4S-A?y@M?b%{_Jg(cAFib~7&t@Q2L9!9LY9fG^{wg(?XRp#tF4y=+q87D6qS1H%e49##UcHf2AAQzl)9j54++k9qFv;ySyMe@wYIfUU)n4x+YYR3TJ*cCnZ&(q{>zSK9D!%*?>mM=+}l=v-C+iCHzGj&v~5PItnSh?VDCK!7DgVFN*@l()#VLXUz{lN<BctMxw1+lERIJ>bU(@}yps4^&ZxOLAKGvYw7S-Z7*~|01{bTkVZNb*4{T?nx|Qa0Fb#!ZAt|~8o}5?rB#)(5I<rl6lKU<YwOT0P<?-^hvzow-vKXNtEYj{6-(~7Bvn-$p#Ssl6EVqn88^SVmJJ4O3#IyoFD;;)$J9ay*y+Uycc*w?9yanw?v+j`EEx3^z5<e{dx3D+bZ7V!>#c4v2EaIEDuCCNKZ`78<)YXaiI%a_Hn~)j!WQm@WFtpd`1iMLqf<Yhav^Vy#G75<A0e`%RZy-B|my{}pCt!GxV2PL{peZ*|wzmsWA5Ws3{?V=9jouIOzxag1Yy<=$@Z^f5sus2s89*7rt~sH^t%+?t@FHlPKwFh?G~A#1j;GxHN*C^=08P;#5R}7+WIz&7A|?QS;O5Fr&nA~6Xg*xMx|l7AtSg1vt*B@D4QYuUMxnJTC3Je_cDIZ>k%6a*`(c9-Ztf&{Bk|X>iqhj}O}MMRWfz}ME`4WzfBy27{NnTJQuE*bb})OX6kvF+s*i{wyqni)a7Hzly`XiZ(a$U|-(&4P^OIRXAlP1{z%wV0whEbJ%IpBYc1G>Cy(n{+lbP?%?ak#ciWoO=Rd)Q#ls<89R^#g0GBXJ-A&;l;U(Z;zNUU56z?;J7I1c-{iA;PVnI05^c0ys5(o{BGx9DWV{`VA2=hNWLs+$da;za$LY;875v58?yzTlYN6A2f$sx(>p+sgV9+_XzKYD+)ZbZkSI3%hrasH%3zvk6cpx_|JX5@w2CckfoPo=QQ#m$F3K{irG^4@2qzL}Tnh3y~qpGa>4PpnOwqS9q;jU!j`s4K#?BVF<@XZz3)nZan3<zhT6c<H|vJ5k82x5q~|D?p@#U=jn$aRs3N2YP|U-6`s1bQonu!ZWod^kZxiSb~fs@&5#Pp6jD~L)K{pXEQiIccPYF{G7W`ZF0J2uI|?)qS+IF)b+x+jHd(m3{X0`s9+3pq;BYh;VhuuAB>a`nQUVtfVl^wAh5*?N=K9*Rm&u6=#Y)M|`kKtb(R!)rjCRZUyC@4RZPic$2TeA<y0U(i6tZkjEVhagAhI1^&LUOgQ&8isa9(V?T|haLDFSOt;*Cy{0eoVLILUV<u8~tHdb-sEgi~H!S-;L5il$hpEX-v;5&aOR0Sj=uhMvkWQicT>3;s@}F$-gqP(6>vc1S8Kmgn5zdg1Z`&7*;j<eOsdST@#wILTP#?fM!l_15~v+c`t(Gi~m7yLh9a+#h%EO3E>1nP%F_9~o)U%XO^&ef1XeQQz>W+iljy39C@;EhWQ;n@iQTHP|G2KZ4yR0WkhS?d|cy${W@4%p4w3)YHZ+m-QB`)a<NTAVYhVN_c%^{nkxX5LQ;yNx{l<F^qC~{T2qfEv?^L+X~G<?S}IQdHd%21cSu4u;&Q|%8XNwBK>9lNNO|l@c)GgMsuyYT3>k^KY;gPEc0WA662w*E`wH8zg@G&<+hPODrIz1(y?z)I3}-gs`&lN+~sW3#sYJ@x&jIW)n3Y9d;PYo46_WHv(KJW^mZsV1Gs+BLmdC8wy_>Y@Z7cO;J;RCTQ#_-xN0UDAWR)cu4)HeH`)&dtpP?{Gy=3_1deGY5Ic^C*NYhmJIVBo_LHS8iiPu;nyPJRSj<#BumLRswLIW!NI?pqpdB2vKr8A5x?qQQG`H63uwk-3z}xHBU<{l3O3;QD=3-8C9EkL5-2m;3sLBkXD}e-;Wn+t|w^uw8)YfJU47Gl(j<*oqdTgzMPTs5<W_h&jnksjQkdNqqS?_zhvhO@Gr19*uT}@wk#?r<%fKq4FX}vp23iw9eJn)th`V;*p|CHcx+;?-XKb+Od!aTCK%_@@1m;++vk&*=ralILjmdqX&7O|6u35OoonE$k}?Z;UxI6@Tb4V<91M3>BF&Z^AYcCoFETU$3^ydVk6Sv9|@gP>=!zHy!Ivf}q=Xg{n7X`UWlUF`>~q{N7Q8VjB$&_u*>az626W)^ry9p$%p{bp?qY0bBqM1Pq+wP_Gj`_W72m8<xRxOx+q6mYfpdX)q#oU}7Dbq~9;CrZiWbW11zzQoPIb$1(XkLA_+Cc0PRwdyJy#P>5VZQjhjl`Ze7`cOkD8B9t1snqOud!0r>7nDaIZ%NU=Z&iUfHj&?N)mh{aV=*%dZ6&_wScfmBY{EWa7~VrL&XrrsxPq#ol7LonT4t$}=q^glWH5vsVly)R43mI0_OuS?>Dn!fm58s^s`y{vbR_8VUd!c)#n(wwF;XtbMpe`>JZ-9<<YeHW5mQE*WKM!mJK^fe)Iw>N5j0XAI)a1SA-<m0RK~P_>AdZG7YWU&a1K^jY$pb5h#79GYxRv)q~-L$)NiG|gKldtiFe_;8k8`Aps*b-*KWtv&Gl<6T*{kO6e5Wgz;HH^enFzE-41tT4}H@QR4-We-rHmHZefBx&WI5W8KP1Z5DRyxSu3?ORCMN9!y0e^5V??*F4xztZ=wdeS=+d+)U6wD-(26i0grOnDCX}hZ7hXRE?7+o1D4=dSR<*FYqjD}8bULlH`zxkY6xtAJQB{ud)W>c)9I)^jZmo+v~YCT4%3)8*~uueVP(C#%m?m}>9e1pPK;cYF?LMIosuzp<mp0=EqhFf0)33G_MaK!kOV+c?x%RwU(#Ry(Jqf8u}})n+evtPDzQ*H<j4)S$c&>r35wXc!#d1|>3E>duIDxJgPJfFs!LCX=P|Zp_4Qyh=(YARPrlMmO8anYkfss-_{X~wMe)FUn4v~P@C`1tsmOdN2#MH@>6~MW=qC<&?*!*+bp6&ANYp^q=rE-k_Hw<m*)oi_cZycAZWSDZMy<VUh!>pMnF;y@4tk9Q`)Ykni34sJ)M4@e`tgqkZ@^0aeSg$>|L^~vyaIS?i_XGL(Ah@q&0FLP+pMusj5T7KuxAR{xq`YjJ2z2Bz{S@nvuv~eqndU6T*LqO=Hi)|8T|hlz-h>llB06EACRN!%OFRIXcv)Kk{lIFF-a_OJPy=Qa2z>akV9wEK<JMRDv*w2)+zo!CCjo&$7SO#aOKEKpO-GudA8mlpXY%<sk)iW=LaOyG0JEN9Ow1*uo`x3R|+n$7+G$yo_j9n7n^`JFy2Kr>*@e_Z-N5%r4H)^plP!I%7nlX)w1`*&lNzyQ_NAuh$xM7U0lphZSgZ*0i#14RsmlumIFz||D_>}N5SrcAtH)CCx8(OdVemd$vW9w7iR&w`;v3vgN^EHX>1Y(n{g>6;an**j5=0%57`&CQQ&nho@Y(bm*&gL0Jg@%;y47a>}CBgKXXI1qhlA&Y6?drP<RyXf&2qnNdtJ&Envq-sg6`QL=GTPKDY5w+oYW4_LVZasFTJpu%PJ0STxO65GgyW%rww{@8H1zql6^>G-(EemNY13b?Gb0UO<5nY;N!>uNgs{V;!8&jomEuL&*TC(=v{cywhJ|{1aRLb%2-oI`cMV*tPc5t1*j7xk5C2z5*8~W+^SqiKR-+5xQY{A!}t%4?XiqNx1}UGtapj(XnPy`2nM{^0ZI*+A@4W3qMfW0Q|yYuR9%A6!r}$^JS@{YFhz=D9kD?3iz?NALi^7qLB56)hIcMV^g#-w38%FG<y#FjlBfvR@p{d9T22K@2qv))D<#}q+L(As%RF1Wu>2$LC&}O_Io}v-9+DI1L?+W+4{1mj#YLutgeD=wR!6%8^p<IB$II7Ec!KfZO!bfs_x;=_bP{=Aj4xFipS;N6+l(E6J>uCR4Oma3oQF+W>}s%&Z<Zq_6~ytiEpFoe{ExZHD<>#x^vTA(d>0ZiwnNYK_#hYDn!J2?7bLNAdDC`B|6uIEcVdsMsf(cSzv`Eq4G*I1rnx2I*uh8^-%Q$-G>H-5&-@k<A)?kU<0)O#=vXDhXt!H-^M7g<ygCEm1mGx$nr)G>rMZHE1`G@4>F$=XI*19g^_J;ZHN!tHrG-w_Pb=!)rM~W&l6g`VHl~N#d)x1OsAk{VILkhMrOm%t<l&UjxcIgInSGc8<D8o9e7fc=3+o)abUNE^RNU2OsSvXoeCS@8bV`(yRBX?Y36Jo=!ERVM6aT<o^Bc|7mnXpRkiV!DE=%P?b$spRbgc|YpMRO*VeT^#AjYhzf<A})!_pLR)Gd87}i_$)%w=k@oIHz=>~ZK*RNfxFV)e@ME%l<CaZ1MQtHY#rAt`Iqm$fs1RRRD4hkmZIiR>e-T|D@Fxflkkc^|ePsLIy!HUGKn0J&J)m&U#gKMG&lUutkoZ7F-=tZ3axsA=&BiQPGp{QDn!nzqR(1NOxrI32%GQ-1Q%>tDOxYqo)7{+aG0vW)Pst5`?L+TFm#E9mdpt;lu+LSxqT!<z}6twc3Hk1>!zP5!T67DmQ;Y#&tZ6y_Psik%N3{9|_E7?KHK2}U#8=hKiMJ&QF;1ZXM#T{1MOH~O{wkY6Zod*RkhaUF3SSkujH3d@8o!yrb(AiJgjok$OuOO<UdvyvUR0Gpzpvs_o-HTT)p{cP}-K-I=kBR;9hX+v<v57%TePhK71>tuBk{Jh_aV#?>{=_jrgMkYBAz^4-&<~H2xS|PQR=b}HYZiA{`Z<Vt5*f{+0f*(<VxPZTB~@<TTG>)&X$ik$7i=mXo0o|&9I<v$qYsA#4#^-MbtJhfEShvcS7!ZwW$w}?^P5l_JYAUg4^BGH+^YUTr4k5YkwGgJlSq(Wy42UnUYkU*UAXE|Rf{QAe-K79u<ZUJ*4M7pHb|OWg%zyd1j+$!n_WEKq+Wc`Z66x<+HNwupTJtoM6(|E0jB+OAS=m7mGK$F-tVwlYAYF3wypXSM@1tMrn(qs4K5KT5iSdy7D^FIe{AG9QY7r?MShiG$@mi()6q81!gk?j$BsQ6iHBg+(T%j4xV0O%K>V)pbz5KF@ov%bEtb|di8KD6KujQjgpqGZ0vJ=jSwo$Ozt}8pmuD~Scr>D;9(ja<*-JXs0)-W}=uS&QDYwkw=pcN;6awi~v4=OSM3yXR+T1~j+LaV$vWeUrlRxPTsND~3b0}G?UUD9#>Z(|dj#FrL7=^rO3cb8MeO3-0qKQU*Q!?_#kH3Yk(jR|Iry}KtzN2YYMqwghe__(oQ^ED8Pqh+YCuqJ|B@?UwrB5T=9+$3QqwO;SKh@=R6{z6mjcO^I)VK{t3-7RZF__X`W#6kRv}!xl;thOdLS;RR!#^q|+zO-}-@c{2U=I|-wY61Sy@@LG_)p0W(Z+!lJO!I@m}gB$W;7l?#|69zLW%>xm=UEa{_o9ru!KLLmvR{85U<)FCf$hs8BFsvtj|ZpG17xD-`|i4Dpx3=1Y>KV1S<pvjT{{@NY>vpHdcW2;VTODrcJ<Px@I7+r%d;5%sQd4!Vz;K-W6&~DxH}SI|KG~2xQCiKroA`Xcz&6zFA#VzEK<43&;IJ70CUG7j@efu0SrR)o*T99$KO1HS&;4&=K+9(8|*hDoz~<-$&khh)fSQNIX!3=VV0T`$lVlSI#&k*{k7w;pV1!XUJI^T_?{YiIM}Al#@-&j@0S#XUAB1FCOmi@ZKA6M-6w;R&CV)JrNuqN~+LR*$DDYW1BviCU*TPp}c*r%JzOJ1uPyJ*VrOwI8d>B2XsWG*6Y6t;C;4K&ueAu?0<J=cVYI@?zu8{OtE?mYyZ@~2I*)B$@rFjDtP0Kt=iI!wfd6SHr{xnOduvOQ<alr!y5~<^-G3P|I)pbt!pX@uA~xz0t#PvOlaAse=~&>1VlMqy)R-Yy<eMh;u?y8@pV75FzTgvIaNWy_Si{^NBQSGuE;?lx67wLr4b-Bw`UeXwz3^2;9xF|lwh%>i;}ynR*c{ZS~6<Ow6q%8UO-@|D%Hn4uJLfdCSm7Mrnn;Rj0ICz3Qqr>0zW=<SfIro#ldR;j-A$#?Q=;Pf^Syk@NfY&fQ@?niNQ9up!xApYC$$?pp-7vXcZm{uzf~a;u|Jie-Z})i%x9(OR5jZdz!fxct|hwr2^b+ijD)rL4Fu9ZO9}9Wc+E$@%^z>r)2m^=DL*S_NjdMVM)~-qzmI`s-B4W*q_VoXMl=wAf}}f?MHYbs7M4Bk@&zx6~fnWfu058n|+~yF~znkf~0KpPne<<bNmMnpg!Gn9}&G+)gQd|Em9<H=OE!eTHPS{(eh~T?($V9YmC6xslu=+a(3bzcyrrIe%W)bga*I-QkK(VI*)k=i6cCh4;1I1SJO#^j~DG)k0)G{geyF=ko|#TWKUEI9&X1oUGYFy7kMn1a;EGNf`xucLjyj-LI)nimvTBuih^VCdiOzNCi%|nZ1bEMoby1MzvpYM90=uqDS$7MJ$S-3@lw~}iGAu%sPb%@z039DdKUOi&g6Vf^e5fub4H6Uw8^(D9jUK3wX<<$X1>vYR=+}Or(1>xO~yL7TWQw8bfC=E!S(vqjayd(VQZMh4!@F*YH2H2UjN}51`67;3)M@V`aa&Pc(*8jH9jK5hCpg{;{)J!`1laCAju!ce>eM${h?lYc+ePj4_kZi`5jEI4+MS55wVxM2}5c&6IAq7E?*RkI%uLq)gM&wcapu~09^x?ee#Y@R3DTAcB5jHGPyJEC4z~Bj;UTh*>4?94feXdq-k~DI;4T4niK`yK?GXc@F1elhjwc>3ea42c|KB+Ls7sr=CFE%WoPeq4+FHtlKE7cRR4|w{PR5rV2nOMffV{s?_;n^>Wxvq4KI{9=TTmJchnswehJ()QmE|}1LP(ZKc@;zmP{#mw^8mnB>9KF;JPNc3BUGPA{xy%Pk?T$A}fLmm4IU_%DH$~R5R)eZ&I;b{BHay5pgPD6p0J_RPI+<lXwfW((1QCjfnZIj2*_8KK=8NVa+ERh*KC4EwI9uqP58x7lOpr;$=9u=;?ydq5T&9SIgC{>ew;qq8N5r;+O7Nb>)yClgyoe^?OFz=(QLIGQ3{f3Ztvl&00}ecw6^?-oxU!u~IKz5?g<wit3<LX5m#^YnbZa3K+S7K3>W6F&U}$Q&0X6{yZVTTG^ZGe`3p+o?_xk7H6sAr&K}f*C^6?CSp>eE>y1Ipz|}L;Dw2YovkdzS|@EZ9bT4bUuZedtym%@v!3MvCio3AaBGSAoXxU>a4~>q=c)eLA@HZlwqSbYmX7vl>pQNyldA|*bN=$xcIj$-KNa{^smzqmzrspQ^8xQN3ZPwtnr5jdFwQTKDo#u-rQ$R$IHjf0;cLC+;=94SPOkgpboD%i)MAR0y_A!XqPP>Q<h#@;vXC8KeH0cw+m^T4o<ijIBwZ+;=;1N*BnQv<Ld6bd^n|I7<*-z=Sy<~yrr?aK+pHk{PkdBhi#wYfO|jSL4M+WilUti0_w)~29XKB#ENP@2gcl3C`|RIP$q<ZH-9Yz8J4u7aTCxqscHBcWy2&9~VQI}m>GkK;Ls5hmxsV1u)jbtUM8iozQ*h}ZIcm1<B%XNWZ|lSgkPiE;M!WT%czGKj#<lkMxeRoxGicrE5a1igkLV-|cHzw21&MDXQC@HfZ80%&gCQKNRAewm{=7yK2rSE)(f!=GMEGB*ocX?9Gx_9PMClkteq$<iJCMTGF?~dm-4|3~A_f1;A@f;ao9F2y6`o)`=<oS(Ozr9Kco2Nmw<z&>K2NR2&iRh07asky48k&(hGGMd1ujE2D7NE&K?TB2{1+XjMyp78i=Su_!#a7RPK?F9Q^%%YvFeNA`{6~09n%xL4}F;2Q?y;4o7u5XIjLGP=uDOHusyRdb&r(8Da=it3NOl7Y()W8A-Xm=S%(ZJ^|wlWEIHgOgqU~$e>hx$Ke|w1ukze9c4X>!PLCDel_5tXQ2XT7znp$@`md+|<Ma<$BIQw#(~Y&p?>1ru#N&98$l-gIi_i`@;D{yNm=_g5NAqQWvf-}<CevzaJd&Y3j0Psvl5Wf&)BG5CK#wpte9RvT4|j&3rUs->CP%;}C1i$kOgJzLv}t@LY0!PtrdV;i{k3kek8Z+Xuia=J!es;;8?}BAlO+Z^ND#O(P+=mVdcU=(UO9lPIzVC!BrLtr)}H)8BGm2vL}+0YzDhO{JPYP$zC|QI{xIGf+^c|SFdeS0EX233T$-_C!D`DkvR?__waDEG2}pqe7O{JHz2NplPSYcCrl!QWDY(AUV}#q`u+e|Ou_w8sL`R+lgh&Ya`R(MmeKQ!7lhza%=l-SHGnJjW&~DhgyVN*1_ibyJ^)j1|&a=)+_{|J274sA?%%w)<r8yjpY4SV+$zP~(OOr`BkTJaGYEUStZ!yw*Nh0?%3*rR!%UE=yudy%4nFsm1qhmnF><)*RxqG9VJ<pH?iZpL^6!aF3W6`Ogh8>%a&($SC*!mVgg7>&LhZm`^KHalv<D9bv)}Kd?zK(717i-$m%6a;RTP@qKaqq&7O}*>;TRP*<&eIXsGW2a}%IQgWxo^^px~k`E8a&T1Ol)@w%2xx;sqh(nB^pfSpxNCU9a41^(<0*C(v?|xc<c@JZtkN%ox(r%1VbAf9S1Bo@_eX(p{N2Gqg0elv@*r#uchIh^|yBZ!*Xq@&hh_#uwP#T6h|$=*Zi4@toFU13Zb?~svr8kqx2mKgDs~Xc*mua5mK|t{^0u@^9u+!!`g$*n^lVLGS7WqW}^4QvO~Dr4?|?kVfik?VPlkjSia}rSC7pButR8$a}om(gW!M;R)J2IWCvUf=^UjM$L78|MD*Qx&JX^fDSn$e*{M(~P5`VRWoP863|Mwrh)R8%BSTGCn(5q8bY^Cur;89~>JD$b|J#P}&agcvRyuv5rb4M@v)uO1vuhk^%&1tIk=Ur<o{=~|TO%>;op&Sze@>1I|C+EEks=KD!(6&}&B9^hh{BSTSh}yIdAYwRh$#ADS^g4*ShHhniH}sfs;mVkU%X}w3u;IzLYP_<XbzJ|bM>QznS>rY1}QXBxxrE5{K_mmVY=5Qkm}x0iW#UfI8@vrE6_Xdl18jzQ&Wxd8%pR_Drec7<f|*1D6KYDmpU$iJCOXeZvQy!s%?XuVhiEDD^__7RjtD(yr<FdLlCg=s^C$c9l)C!9DI%WXZc?<9<dJp&E^G%Wy=7PZr)opVlb`}h_S?^+ZtO6V@zEMrh^HpV_+xBf~V^P*k--R{7je^VL1oTJwpO!Yxf%Wk_HTf;g=qfPnD4~!~jb1hbo%!9|?#>fV53WqBw&YWaINQL+Ch+QAeIh6A~P!Lm<&{SV||wQH6!`6u%%Y2UlOxfsIdq>vZlIa`wnB*>B<MvZ^JZ`Lf-*bD*kVoRx;=8;iC4zUn8hQM*F5tW*OU>ORah=~L|TndLt8RQZ-h9I-*R&WE=w!x#$`Z%V%5X#%5@yz;f!nrvligW`-h|5YXZVf6{l_UUs@I%k|O>ck-pU&E*-N#ouFmFT<%%E^WpFUiODqxwy|uwuJA98?Ab9#VLi@lHz}RH_mz-bU?QK90}<&~6;=HXA{&+)GzN<%p?)_`Df0E~39{w_D^oiX+SVWE+8u5hh~-u}l-Tt-w;<m11p_TYGn-l$A4Wa*R^uxt)uY<fX-016F=<J0(q~L^@a8+PO$8Z;>2EnZ@t5dX~;Xv|0sHeH#<uW9{hxB?Mj!&jRVwO-A8;#Q!?%^*MK}qTj6F1P$)m3Ia6f7us+IwJQq<uGhT}ZC#${V1_&dsUCZ`XtT@t6-#?j!4%M?8gcj?t%79!NE_&!>G{g)+O<{HyPQumI`mXm9bdjq8aVb=TOGfg-j#p}rC<(q1?{l`R}`a$VMBjA{S5t_b_6Kjbg)UO02@7$)OGs(*_oL)=H7T?Zl-)jfQQ{aIsbo#p$q{w5UjSYMo!o%MNB<T)hMLP`0=<JjM?zKeKmwn$xZpvjdIV7=3(AEr+JR0+5O|-^pEiFZ}LO&Bt_=*pFocIr_+Bu{r%}*;QxO({ikvmSqJ3(Fbb8B;zpvwKy*4zUR@d*57qmjvFx`;gM*xyK)oswq$;Ap=0y08<m0;2?cM>h+@q)niJ2r`gk(uinKk*~8}*MRb+;fouSME^e7!pLLS&RTs`||b!vj2ftpP~557fZ$<gBzhcO!Y8{aSt~s5jFC%DE?RqP#&~LrVNVB>bSggHeZ(v7ptviyYm{xI>4#z{Hyr!zOpBL@@GQYJ2BkM}d#`hZUq8+_UR93=9C4vs$fRE%G_b1Q5<%{@&QnzE=ThdCcj=egncMqsXGi&|s;A^`hWZy1D-_Q;@v+a$P7NIj?}ZDvFEx&GUq-@keZhc$lM&%U!II^U8eBYfxFT)qXQ-Lx{@+zi;mxr<itX42ZN1i?bUN&mQ#m(#NbjY_uyF5nb9#TJ7AzQ<^3`aM&%nPB09+K+XB$TnH0!pDuOX1@bj)@?q=ZY<cEV^VpLp+pFm9o5nUgEfeCC9$y}dnrgj~HwU@BOvzRfBC}zGogC~#Sk=-XN$wUp(X0<N5AODyFNcS{N-tFk83%~<ki#4pLfU1lm&{vxU$z!w;>-Do#vnH{=0v5F|I+;fU_s{9^7y>v-O+vpr%oYOxCish5+dC|u{gd3VHwbnFogPgIvQI91JXt*pVCSJ;V3=0k3Y81E64YUt0dwW^jcihL!Bb{7JQbwQ@W~j&kO9T`Ng&z&K*-23Mp6Bg_4uq$thCnX?qg6!c*&uvaYy>m)4H(s2xu~q+>b;U4X9*s&$wSa7ssYBD+QaVbA<i^h1POFvl6++UBL%Ni(W_-z|%fQ6utv!T$9Ntaz38nE-|OiqS4s^RgyT8ubXz+1*<pJ^LLi_Ub~#X36ETvF>OT`77aHBR%R>RwbzIz`V~SKN!RF+*mdz3Xkhjd1FS*MY@?**UeWs%DS~Z;}rIZmGm1)8^POYPgsm71=UVK43u!-(7cQ~KtCBFDAoGf%G-F{1*K!4t3Xn465jT31??S_+=6Ya?8oYAXPb3F{j=4~RI=ytXAc{StjY~jl**IYaB)AJmevN-4^6wr(_$WvfJPgK1CRh<ws3V|4QS91j_sn}?vPoSa))e<IO;`*Ku76;3yy8CxCpf8#4)??HQGq)4=QIa44L7*IwWj@v?3*8=<lY)0oOm^X(Y`s`x2iMdW%^?nL--H**z?{EyC7@(bR0w13+7T9ZnaD(27S)L6u|<8P>?-$>(x7b1xFf$e{C<$``7d40;vB&M9$5TY(75q9@*zX$};ZcD6&TnpTXEmtams;nGYr3rx$06;GD+3U(t__80?27wN%XyE|}#x~&_PU%pV8o0%`4l~knCZ%OUWXyS6sN}}HspXtzVnh?4l_1hrj9Sn!PL3w(b9}Y?kYL8H%(`a>ov!?3PACEtOsZ%n+Jt&%LAhwTV49j@<Rn+HvU6k&#-UJtyOD-_~VZkhyVKFQu$he~epcmk<@O+7#X;WJddhk24kK+ilN-O&g9bLLSa~?t(Lpv3qo!N4C_Z|L-m}Z__i*KztiI4V-{e+rEk@hT^zgeuIx%-Z)ppo<0SL2OTbrbr4TLb>%u*C~DTeN>y9PvVmBcf^qbZXks%<#e_%y6s)%cPPELv&B)#c)yKtiMpjIv%NnJkIcUzy@5@n~z@o%0d%yQ+ez%qr30;uEAYU1@{gp_<XCn0heujb)&krbi+1`(-t9#SCApl-wOPlE2CL&UZ_ZGH)?R?t6RpQlgT-$!gq)i#^|(E$9lAfWyAMJZOJ93lpM70Cd~+>%Au8q%o)WljG|SUtuzU5P=b)16Glm3=&#2@+69ibzvvYF?kAvw_l9E0g%>K;{C#Izu9dh6$DvYCof@<^j1_>JSN<wBRqdD3<_LJ%Nl!TU!F5JN!ln0!*{_Yer6})AvXU9Hgz^i%AkF5NHXx36|CP8BfuJ8`;L(t@|A8cjCya8US{0Cwl1=Qy;>JzMmWivJ1ut46Wx``tI_7*0hn5VCG4pPXL&38rXh*6q{$^G1d7T*jb)e+_eOxOdn!y1km9-joI$g>rbJFq`1NhH9Al!Bn^QJIdU!~`3sL4{scxO52(-JCMk$GF*tWNv!QM7xRfG}A3gasE{`csui3tRMwvU)S@%`<`J;2;0YI1P-xWrKS>O(CG*R{yhwBJ%22r5xk4We>M~#_dKXU?}A`PODn@NhfKhZYMM!rP?)q!ig-~@7Sy&)0Rmf_2i9O7x+74&%XSMfCqTs@#kEgs^<C587u(l^sHqTzV_NdPFDUCJT?DF8&}m&AXbHCBZgYyr%&*NhK%qD|2PiMX|yGCN32lJFhWLa<31Dc*l;cLD@CC!%$+-u#)@H3Q8egEm@IV04d94<UFU$2hum4r_ZrBt^xtgznmNY3E)WQ8b13j3g2FHGC&PY{5XSr>vUY;&Cnyj`24Q*@5Row33*DL=$|f|(LR>in3cp#!xCumpd>cq#7ftoXPzdRoVA%W%+y7CHLB`X{Ya3>uEk%4hL_9d^i5(-8;brjQL|N+|ri0%)xO}<<wm}qRybW1l!O^2CV1|0d^S!)Ly|xuA<jWMSCwG+x>6t%}af1cwO!{Oo-T@UWm%)rb`p4cq$lvk9uQ^!87${hfO2uT$G0?Q(7eH-d$&oz$?JBPRRpr`@fM`)<dJaXW7N-URa7Y66Fad#qWA4x(obq!iTKgE1XdsffTU{~%>mPx*Hw7!0){wH~KxY5!eA0?lhH<#x$qPAF*&kxoCc88UGFb_JzC_yq4qR=+R*cwAZV83L(IpclEf4ZGlS6O$mIzF%_3In-w-Nyrh)2{!JDsZyW!5PjwFgJaxynIQ{&upIT(<Th=c8OYE!&&n|2TEgTNl6oeJ84pzHlO~{NLz^F!{nnh<V?)Lbdn<JC4lzg$^#`_f4$Xxbf%!G-p&MEY16^iB;pTtk^aG`zD}GKCmL(ykBsv8^0`ZZ{GLZpf`R$6$qF6U3M2UF?0ESrgtZu{OW%B#v2diV+rNznfR@21m5wO^W{$vDW@X#&iFo))=^G{_)XTom5fn#g90P345n=c?ZRC&n}M{AwpY1oIHc_^LBv}Ot5`l4;L8(&J3|$cF~3|rGY>9;qHr*t1TF#)YMP8wa$O_V1#fVu%^SfU+>-1W3;Ie!2eusax}aANf?=1F2>Be99O#0>79ro*cV>{f`B{p}sHC=>h4OO7@{+lFMeFlil6ra-pCc_qC2Lw>`liE+($03MKO=>6$6HcY&oJ*Rud1uxA%8PQ^nh$QvqJ^J!kr8Da*eYvJ)>y(&j{}JMB^U)`t*^FO3H>N2A5<oP*j-Hmwry5N4E`9G-xwp77l;LWwXX?`~q+fjX+PnhrY*SzuK?5P~}BNojo*VhQ%{HKo)2z;fk`w4B@MRhNCR1hsIQWEOnk=`T{O+Gz7(6d3dDCa+)lk^jMRG3I<zsDiBPE{EH}1vIkOrp|+wB(@ZjK;MnI57<AD$r^37Sz^2eE>u?KiNC(-0E89E%SOol#^n0pc12dCe>~>XdWnP`uT@Wz)55)Tme~8k>LSHfXcCbm<n;hN9+OUo?XbrS$4u8ohkt%R>+)?rS^9yIj$1*4~uttHs{VO{P;K56VJq&PPg7BhRfSiySe9NNwo9mk{(c|v}4p;^%mz1%v%y68g-OKX=jUz)ZH4{e40i{foBT(o0?eI9aW-INXiNqlaD(kJIoF8arc4p@C{5+5r(O#J;k0NwnK=60rD!y)PQXu*kqAP!MBSJ=hoOAG)oleB0^OZ0*Q#vdwvwg&~^!AZWkq&d&jAEOQx8lhW|FbMh;G2Q(mdW%iGf;i|*Eu`{wT1#V_MMUijDM?DI?Etc4jzid@i;(Mp$+DI`ufQb<)mV^8kls*lfysteAUHN-nQ1-OGVZ#XkBbMw;!3UA9@U>+Ok`=Z<5q?S~h(}dC_vAqGINku&MIzDC{KnnYV>y55x1Z;fysq$3Ox<y|xYPT(NS0NZt9Cz-A)zBaIhLc-p#JHU;fJv(a!La}gl5MB~c#*rMYSii9XRI*yxCuyv4-Xt;W7bBpCe6c5ph*<)d|$+9Ak*kb!==2ceh90s6%&W~)$ppI2KxKD*6bA#Lc*w=RB4khoC*TLA(iV?D_v|E?4qy3(I)o~6BFmuQ||7IwzF;57U7p^rpu#W<I>+M71X_Z^_a1$MD9)+}*VVPpfF%ebzmnWV^ce~98$fLxT9W2aEFU*yKCC5RC7It7ymOOu43N}r##VgZS0%PEZH;siKUd*pQv>(>2m_-W6d*~QAOTo1fVSjUF!QG_Y#mcT-&ZHAaU`iz#w9qVH3Oy7Qx0;F|H~=bD*i6END)@INY=gK(!Lq98;L*qqXf(MJDX<h&ky}wgV(yT16T&CMa8SGFXf_j&H*u(u1YSJk&W@*v71)gzk6}M-q|#$5o(3;Z%|@A*$HpN|7+iS%src%ejhRrO!XsscC!Ym>cy^3V8aG&8scee@B#IAPEN<ET<^u)G_o-cORQU0k5kbTq8e^k@8u_t6PX3()1vS3rkRUsh&l|UAAh=(?`Z>Jyg$)S(?A1rUz`i_VLiy~N4zHS;iS;>$mX3J5Kb0aYgm?W{DT!gi5AYh-5hG9R``Z&Z@WP4yz-9A^6V?@k1%Mqf6x7baG!7hPX~OK6X+?v^eo{e<O(h+t#jX&@6BC9=z{%0+J&R>pI4iO<3t>(%p14A2xo|2;S%qm_I~NC2En#p<fJxMEi^ze&EJ)75?t`EY$Gka(prUw#-~~d-aLgMclv<GKqD)7jD3Yd!2D9A(u%l8on<%4J`}ApVKrT{J${VE{L#LBjKL3iRG99?VqMI9Fon=(l^P)X-cSkM_S~j(6=uo2Y_k5k298nV*kt{l!$c6+n*Ug{DfP}<EUV8mH?iRbuD55D%OP%h0uSH1ruJlC`Dz{nl-Er+e>g%%&K*W{qU#MhEJ>ye@KM;`MClp|YVCGUZ1~ZU;&e*o5Vi(JT3Ur^toWA7mO#!IKVRpsku9HiHghoG6XGCF#59uHl4l}oIP{n#gAZ1l5Bat3w{gxP5mNd`MAF};rr>NU<CP|3=?k`4hxL_*|=Y?=rwN6xF8XqJ4jA}M~@aoqdKG*H+xB7>qXt|boe1c`$`Cu_LJ3bSOKyZ65tBa8ZrEv$@PX397UorZ~+#j~K!k0)U&<o!`XjHguLC4?4x;d6NoMO8&`@vscs1z)v$8&YFFTo8HRf=cqS`v!qLo`=fAFggw)dO-ATw)u9iQh~DRnj26QQX)7BbAzgDzQW+P^H8R5{r$!2tR6@j~x+{ZXA_wAcMisye^G$loG-aDWefvWNF|gW@(XCBN-m9C;Qb0>7GcTHzT3of<!`=3mkTguA?Ho$k)cSB5mu|B?o@e9F$&1CL14FGbV~6uc@4Pps&=96{=%<h71)$7v>-5Zqe{CXvh5H-0j%WW;kh>YP_{kt5GFUmAyw$Yw>dZI=W!voZ`T*A4_y@AUdCkPDMdT-sHpa;dC$s+0|UbXb$&q1Y&C!%9ci()1rEV<uXiYA>NI#jcj?LkeAt5hdTRjyPS}lZ!%U-bsLkZ_8i9gvBGP7mamF~SV#b(GA!ewzbx`AMw6AUo+#?etaGkAET+6BGM3O-U3XZ{H|O#*<<Dn~2a1OEdJd>K$KS&nGKsS%KEwYMpUY8`bD%8ujZK+34xD(52S;stmbSmDAIGgcZM$&zQbheg)GPa5mGM|c+OhtdILCow&L8YTu4m><z5TrUA`qpDi{|DK)=aZuh2+9Ym(28=vt#5e9FwWL@2_j(NHufr_2QNA?Z1d|_ZuvCV)-Bt=1|A||250>*)41f2u}m`kkrx^b1fUpddGo@yvFh)cqkrLBe{kbd1hmsp{YUp9DY_}5_dm_3HUVv!iH0#h&O#`OnA@#N2z|W&#C;rT&au#pe?!V+=qy>C--ryV8t}XqSCe0YAyyq^dn|GT`J7lPu?A&N*s>gRBNj4dz)L;ty)$MF>}E<!aVIk0iFHJ=|7$R@${b&?o8Qd{&f1u>F-Yea{60%{BNiK75+{?mnYkJ)$m_zP&8w2Fm<6*PU1(v7&r?iT(phRZq0Wq#)^v2pMFmDywN64K~|q~nRykOb&`|d=AlmiYI3T@M(uXJ_QM!+665ui<#@Ap9gE*!0@!A4<96=i{x_*KN;tDATv&}fL#3!c$FRyr>3ZSMji1|1^K;Gf>EdU!E>zN+s+ymj<)>55-gv_&zK53fcz!)3Bt8>+l-;s#&C_4wd5cHGy|Anb2$wqD`vp})xCB2T%4nBM*cEKUW+6XWkqQcItaA80v6W(uKJ%6HjhGhP3fyJ71=9<??d0U`_;=0F*5g#L*%)v{2fgr&W>4TK{ez-S%j<P|0<8p$drj~M+G{RpNmM<gwj5uDLvj_*MO9$?a}f@0FBuk8k4Y2tcwN&+cLL~>(?5BN6DyP_s(2?Q;Td6wGjl)V1IsS5#Q-#ozjq4Xdy;oOrEl{iT6P=#${o1nfP4yKeq62A5pH??It+a?h1?0RyU?KUz(R@8)!6?o!1Cx$1@*t9aOiepSN=hOo3L02&P?^xRTcMdHIu;}NEk?nSe0Gt4RaE(LDhd%zy-U2qu`1vd>#d}bAidCoVEa}17znNiWY;{P&`(E#A4De(Nw>6H|b;PwO+EP$0{g~Wa!30F<NABNlL}HL7^JKfbgdt{UkrA_xa(aZ|O;@FXC2bzpJ}(UpH?b!Z7;}B<B)eXpZ*oHh1HN*}2uLi@^Z29{93}cwX<061dtCKQb^#>Vw5#7v%wX1#^GfJWot1GP}|0C78;9t>Y>$V40K+nTD)SOx4-(4R2DCylkbgngR0xlzQ28aV-XG>+!Yf%4RLT60cv2uPg?&x8myBRxMo%Mzw2a-eQ{z-<iL1Y5%+5U6`MrX*L>rbKhOqYh0RJxCDRs?*8n|?CySJc0Osy_mZ!SUlK7{%Zt+xqtjRGg_|1nct_$)m*yR*iz>e|Z~$IItvw*k*5QcJEH>X<;gW&V+Wd|tvqdh~2j_t4Uuw*vLEG-a?){S6JE(5m?jh(3NBHyH%-rRv+1aU?`NiP&&GmRLo{eXv8e!uOa5hXU3gC+E6PWjd;6VbMfFN}Cfs5}p_ArB^iPuNBKaD4+)o2eNkZuzNm#$w$AcX)(v#-au5bus!?I!llXB)`vHiDT8iKMIStu5Pyp>9`K>dRG49HzY(E9qjRo3xhGg9n}AL4p7a2Ad)2<K3Jz7vcEBBMJh8r6l2)up=cf%BzWxh%D>92L}vE9k^%7jKf;?+t#l4M*SWzN_D;hCuA(RXgC=BeR#h+HEbn)yKf*@q}Fe#;<(Vzy#4R^u+{1I;pxKP9se(Zf}$?"
def notebook_from_embedded_source():
    import base64, zlib, nbformat
    content=zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85.encode('ascii'))).decode('utf-8')
    nb=nbformat.reads(content,as_version=4)
    expected='EMBEDDED_SOURCE_B85 = "ML5_V730_EMBEDDED_SOURCE_PLACEHOLDER_9fa72"'
    actual='EMBEDDED_SOURCE_B85 = "'+EMBEDDED_SOURCE_B85+'"'
    matches=[c for c in nb.cells if c.cell_type=='code' and c.source.startswith('# ML-05 V7.3.0 · self-contained immutable notebook')]
    if len(matches)!=1 or expected not in matches[0].source:
        raise RuntimeError('V7.3: invalid embedded snapshot structure')
    matches[0].source=matches[0].source.replace(expected,actual,1)
    nbformat.validate(nb)
    if not any('stage2-ml5-v7.3.0-preserved-full-research-verified-release' in c.source for c in nb.cells if c.cell_type=='code'):
        raise RuntimeError('V7.3: invalid embedded snapshot version')
    return nb


**Совместимость:** некоторые внутренние имена `V4_`/`V5_` сохранены для воспроизводимости исследовательских процедур. Все результаты, манифест и публикация относятся к V7.3.0.

## 2. Среда Colab, зависимости и статус
CPU достаточен для первичного запуска. Библиотеки устанавливаются только при отсутствии. Результаты SMOKE не публикуются.

In [ ]:
import os, sys, time, json, re, hashlib, base64, io, math, gc, copy, traceback, platform, shutil, subprocess, importlib.util
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import quote

DEPENDENCIES={'psutil':'psutil','numpy':'numpy','pandas':'pandas','pyarrow':'pyarrow','duckdb':'duckdb',
 'sklearn':'scikit-learn','scipy':'scipy','catboost':'catboost','lightgbm':'lightgbm','statsmodels':'statsmodels',
 'networkx':'networkx','xlsxwriter':'XlsxWriter','nbformat':'nbformat','requests':'requests',
 'tqdm':'tqdm','joblib':'joblib','python_calamine':'python-calamine','matplotlib':'matplotlib'}
missing=[pkg for module,pkg in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
print('Среда: Python',platform.python_version(),'· проверка зависимостей завершена;',
      'отсутствуют:', ', '.join(missing) if missing else 'нет',flush=True)
if missing:
    print('Установка зависимостей; дальнейшее выполнение начнётся только после её завершения',flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing])
    print('Зависимости установлены',flush=True)
import numpy as np, pandas as pd, duckdb, pyarrow as pa, pyarrow.parquet as pq
import requests, joblib, networkx as nx, nbformat, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, PoissonRegressor, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Ограниченный DuckDB: промежуточные сортировки/агрегации — в локальные spill-файлы.
DB_TMP=ROOT/'duckdb_tmp';DB_TMP.mkdir(parents=True,exist_ok=True)
def duck_connect():
    con=duckdb.connect(database=':memory:')
    con.execute(f"SET memory_limit='{CFG.DUCKDB_MEMORY_LIMIT}'")
    con.execute(f'SET threads={int(CFG.DUCKDB_THREADS)}')
    con.execute("SET temp_directory='"+str(DB_TMP).replace("'","''")+"'")
    return con

def memory_point(stage):
    gc.collect()
    try:
        import psutil
        proc=psutil.Process(os.getpid())
        available=psutil.virtual_memory().available
        # Colab может использовать cgroup: psutil.virtual_memory() иногда отражает память хоста.
        try:
            cg=Path('/sys/fs/cgroup')
            if (cg/'memory.max').exists():
                cap=(cg/'memory.max').read_text().strip()
                current=int((cg/'memory.current').read_text().strip())
                if cap!='max':available=min(available,max(0,int(cap)-current))
            elif (cg/'memory/memory.limit_in_bytes').exists():
                cap=int((cg/'memory/memory.limit_in_bytes').read_text().strip())
                current=int((cg/'memory/memory.usage_in_bytes').read_text().strip())
                if cap<2**60:available=min(available,max(0,cap-current))
        except (OSError,ValueError):pass
        info={'stage':stage,'rss_gib':round(proc.memory_info().rss/2**30,2),
              'ram_available_gib':round(available/2**30,2),
              'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,2)}
        print('RAM',info,flush=True)
        if 'progress_event' in globals():progress_event('MEMORY',f'{stage}: доступно {info["ram_available_gib"]} GiB RAM, RSS {info["rss_gib"]} GiB',announce=True)
        if info['ram_available_gib']<CFG.MIN_FREE_RAM_GIB:
            raise MemoryError(f'Недостаточно RAM: свободно {info["ram_available_gib"]} GiB на стадии {stage}. Последняя диагностическая стадия записана в logs.')
    except ImportError:
        pass

START_TIME=time.monotonic()
RUN_SESSION_ID=hashlib.sha256(f'{time.time_ns()}|{os.getpid()}'.encode()).hexdigest()[:16]
STATE={'technical':'NOT_RUN','scientific':'NOT_RUN','publication':'NOT_RUN','overall':'ASSEMBLED_NOT_RUN'}
LOG=[];PANELS={};PANEL_FILES={};TABLE_QA=[];ISSUES=[];SOURCE_AUDIT=[];SEMANTICS=[];QUARANTINE=[]
PREDICTIONS=[];PRED_PARTS=[];PRED_TOTAL=0;METRICS=[];MODEL_RECORDS=[];MODEL_FILES={};FIGURES=[];SPLIT_AUDIT=[]
MODULES={f'ML-5.{i}':{'status':'NOT_STARTED','reason':'Ожидается DATA_GATE','n':0} for i in range(1,8)}

def stamp():return datetime.now(timezone.utc).isoformat()
def note(stage,message,status='INFO'):
    LOG.append({'utc':stamp(),'stage':stage,'status':status,'message':str(message)[:600]})
    if 'progress_event' in globals():progress_event(stage,str(message)[:150],announce=(status!='INFO'))
def fail(stage,why):
    note(stage,why,'FAIL')
    raise RuntimeError(f'{stage}: {why}')
def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024**2),b''):h.update(b)
    return h.hexdigest()
def norm_code(s):
    return s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
def number(s):return pd.to_numeric(s,errors='coerce')
def safe_year(s):return pd.to_numeric(s,errors='coerce').astype('Int64')
def nonmissing_code(s):
    x=norm_code(s)
    return x.notna() & x.ne('') & ~x.str.fullmatch('0+').fillna(False) & ~x.str.lower().isin(['nan','none','неизвестно','unknown'])
def canonical_hash(o):return hashlib.sha256(json.dumps(o,sort_keys=True,ensure_ascii=False,default=str).encode('utf-8')).hexdigest()
def _disk_schema(df):
    # Примитивные наблюдаемые типы сохраняются, смешанные метаданные — строки.
    fields=[]
    for c in df.columns:
        d=df[c].dtype
        if pd.api.types.is_bool_dtype(d): t=pa.bool_()
        elif pd.api.types.is_integer_dtype(d): t=pa.int64()
        elif pd.api.types.is_float_dtype(d): t=pa.float64()
        else: t=pa.string()
        fields.append(pa.field(str(c),t))
    return pa.schema(fields)

def write_df_chunks(df,path,chunk_rows=None):
    chunk_rows=int(chunk_rows or CFG.EXPORT_PARQUET_CHUNK_ROWS)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    schema=_disk_schema(df)
    writer=pq.ParquetWriter(str(path),schema=schema,compression='zstd',use_dictionary=True)
    try:
        for i in range(0,len(df),chunk_rows):
            part=df.iloc[i:i+chunk_rows].copy()
            for field in schema:
                c=field.name
                if pa.types.is_string(field.type):
                    part[c]=part[c].map(lambda v:None if v is None or v is pd.NA else
                        json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                        if isinstance(v,(dict,list,tuple,set)) else str(v))
                elif pa.types.is_integer(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype('Int64')
                elif pa.types.is_floating(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype(float)
                elif pa.types.is_boolean(field.type):part[c]=part[c].astype('boolean')
            writer.write_table(pa.Table.from_pandas(part,schema=schema,preserve_index=False,safe=True))
            del part
            if i and i%(chunk_rows*10)==0:progress_event('DISK',f'Parquet {path.name}',min(i+chunk_rows,len(df)),len(df),announce=False)
    finally:writer.close()
    return path

def qa_table(name,df,keys=None,allow_empty=False):
    if df is None:df=pd.DataFrame()
    if not isinstance(df,pd.DataFrame):fail('DATA_GATE',name+': ожидается DataFrame')
    if keys and not all(x in df for x in keys):fail('DATA_GATE',f'{name}: отсутствуют колонки ключа {keys}')
    duplicate_count=int(df.duplicated(subset=keys).sum()) if keys else None
    if duplicate_count:fail('DATA_GATE',f'{name}: дубли ключа {keys}: {duplicate_count}')
    if df.empty and not allow_empty:note('DATA_GATE',f'{name}: панель пуста','WARN')
    TABLE_QA.append({'table':name,'rows':int(len(df)),'columns':int(len(df.columns)),
                     'duplicate_keys':duplicate_count,'status':'PASS' if len(df)>0 else 'EMPTY',
                     'unique_keys':','.join(keys or []),
                     'storage':'PARQUET_FULL_PLUS_PREVIEW' if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS else 'MEMORY'})
    if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS:
        fp=write_df_chunks(df,ROOT/'derived'/'panels_full'/(name+'.parquet'))
        PANEL_FILES[name]=fp
        PANELS[name]=df.head(CFG.PANEL_PREVIEW_ROWS).copy()
        progress_event('DISK',f'{name}: {len(df):,} строк сохранено на диск; в ОЗУ только preview',announce=True)
    else:PANELS[name]=df
    return df

def limitation(module,reason,required=''):
    MODULES[module].update(status='NOT_FEASIBLE_FOR_PREDICTION',reason=reason)
    ISSUES.append({'module':module,'status':'NOT_FEASIBLE_FOR_PREDICTION','reason':reason,'required':required})

def diagnostic():
    mem='unknown'
    try:
        import psutil
        mem=round(psutil.virtual_memory().total/2**30,1)
    except Exception:pass
    info={'python':platform.python_version(),'cpu':os.cpu_count(),'ram_gib':mem,
          'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,1),
          'gpu_available': bool(shutil.which('nvidia-smi')),
          'versions':{}}
    note('ENV',json.dumps(info,ensure_ascii=False))
    print('Colab environment:',{k:v for k,v in info.items() if k!='versions'})
    return info
ENV=diagnostic()

# Прогресс V7.3.0: истинное число завершённых этапов и отдельно текущая операция.
# Этапы меняются только в mark_stage(). SQL/INPUT/DOWNLOAD/MEMORY — подоперации,
# поэтому они не сбрасывают таймер этапа и не подменяют его название.
import threading, html
from IPython.display import display, HTML
PIPELINE_STAGES=('INPUT','CATALOG_52','SEMANTICS','PANELS','FEATURES','CROSS_SOURCE','MODEL_DESIGN','UNIT_TESTS',
                 'SEVEN_MODULES','EXTENDED_MODULES','STATISTICS','VISUALS','EXPORT','QA_AND_PUBLICATION','DOWNLOAD_RESULTS')
_PROGRESS_LOCK=threading.RLock()
_PROGRESS_STOP=threading.Event()
_PROGRESS={
 'completed':[], 'phase':'BOOT', 'operation':'ENVIRONMENT', 'detail':'Подготовка окружения',
 'stage_started':time.monotonic(), 'operation_started':time.monotonic(),
 'current':None,'total':None,'last_render':0.,'last_log':0., 'last_log_text':''
}
try:
    _PROGRESS_HANDLE=display(HTML('<b>ML-05 V7.3.0 · старт</b>'),display_id=True)
except Exception:
    _PROGRESS_HANDLE=None

def _progress_html():
    with _PROGRESS_LOCK:
        s={k:list(v) if isinstance(v,list) else v for k,v in _PROGRESS.items()}
    elapsed=int(time.monotonic()-START_TIME)
    phase_seconds=int(time.monotonic()-s['stage_started'])
    operation_seconds=int(time.monotonic()-s['operation_started'])
    n=len(s['completed']); total_stages=len(PIPELINE_STAGES)
    phase_pct=100*n/total_stages
    current=s['current'];total=s['total']
    numeric= isinstance(current,(int,float)) and isinstance(total,(int,float)) and total>0
    unit_pct=min(100,max(0,100*current/total)) if numeric else None
    unit_label=(f'{int(current):,} / {int(total):,} · {unit_pct:.1f}%' if numeric else
                f'{int(current):,} обработано' if isinstance(current,(int,float)) else 'Ожидание завершения операции')
    # Никакой искусственной оценки процента во время fit/prediction/SHA без известного total.
    current_bar=(f'<div style="width:{unit_pct:.2f}%;height:11px;border-radius:8px;background:#0f9b83"></div>'
      if numeric else '<div style="width:30%;height:11px;border-radius:8px;background:#0f9b83;animation:ml05pulse 2.4s linear infinite"></div>')
    detail=html.escape(str(s['detail']))
    return f"""<div style="font-family:system-ui,Arial,sans-serif;max-width:980px;border:1px solid #cdd5df;border-radius:10px;padding:12px;line-height:1.45">
      <style>@keyframes ml05pulse {{0%{{transform:translateX(-110%)}}100%{{transform:translateX(430%)}}}}</style>
      <div style="display:flex;justify-content:space-between;gap:8px"><b>ML-05 V7.3.0 · Advanced Data Reconstruction</b><span>{elapsed//60:02d}:{elapsed%60:02d}</span></div>
      <div style="margin:9px 0 3px"><b>Этапы исследования</b> · {n} / {total_stages} · {phase_pct:.1f}%</div>
      <div style="background:#e4e9ef;border-radius:8px;height:12px;overflow:hidden"><div style="width:{phase_pct:.2f}%;height:12px;background:#2863b5"></div></div>
      <div style="margin:11px 0 3px"><b>Текущий этап: {html.escape(str(s['phase']))}</b> · {phase_seconds} с</div>
      <div><b>Операция {html.escape(str(s['operation']))}</b>: {detail}</div>
      <div style="background:#e4e9ef;border-radius:8px;height:11px;margin-top:7px;overflow:hidden">{current_bar}</div>
      <div style="font-size:12px;margin-top:5px;color:#65717e">{unit_label} · операция {operation_seconds} с. Известный объём — реальный процент; неизвестный — анимация без процента.</div>
    </div>"""

def progress_refresh(force=False):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        if not force and now-_PROGRESS['last_render']<1.5:return
        _PROGRESS['last_render']=now
    if _PROGRESS_HANDLE is not None:
        try:_PROGRESS_HANDLE.update(HTML(_progress_html()))
        except Exception:pass

def progress_event(stage=None,detail=None,current=None,total=None,announce=True):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        old_operation=_PROGRESS['operation']
        if stage in PIPELINE_STAGES:
            # Не возвращаться в предыдущий этап при nested helper calls.
            if PIPELINE_STAGES.index(stage)>=len(_PROGRESS['completed']):
                if stage!=_PROGRESS['phase']:
                    _PROGRESS['phase']=stage; _PROGRESS['stage_started']=now
        if stage and stage!=old_operation:
            _PROGRESS['operation']=stage
            _PROGRESS['operation_started']=now
        if detail is not None:
            _PROGRESS['detail']=str(detail)[:230]
        if current is not None:
            _PROGRESS['current']=int(current)
            _PROGRESS['total']=int(total) if total and total>0 else None
        elif stage!=old_operation or detail is not None:
            _PROGRESS['current']=None;_PROGRESS['total']=None
        line=f'[{datetime.now().strftime("%H:%M:%S")}] {_PROGRESS["phase"]}/{_PROGRESS["operation"]}: {_PROGRESS["detail"]}'
        if announce and (now-_PROGRESS['last_log']>=1.5 or stage in ('QA_AND_PUBLICATION','DOWNLOAD_RESULTS')):
            _PROGRESS['last_log']=now;_PROGRESS['last_log_text']=line
            print(line,flush=True)
            try:
                log_path=ROOT/'logs'/'live_progress.jsonl'
                with log_path.open('a',encoding='utf-8') as f:
                    f.write(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),
                        'phase':_PROGRESS['phase'],'operation':_PROGRESS['operation'],
                        'detail':_PROGRESS['detail'],'completed':len(_PROGRESS['completed']),
                        'current':_PROGRESS['current'],'total':_PROGRESS['total']},ensure_ascii=False)+'\n')
            except (OSError,NameError):pass
    progress_refresh(force=announce)

def mark_stage(stage):
    if stage not in PIPELINE_STAGES:fail('PROGRESS','Неизвестный этап '+stage)
    with _PROGRESS_LOCK:
        if stage in _PROGRESS['completed']:
            print('Этап уже завершён:',stage,flush=True)
            return
        expected=PIPELINE_STAGES[len(_PROGRESS['completed'])]
        if stage!=expected:fail('PROGRESS',f'Нарушен порядок: ожидался {expected}, получено {stage}')
        _PROGRESS['completed'].append(stage)
        _entry={'run_session_id':RUN_SESSION_ID,'utc':stamp(),'stage':stage,
                'completed':len(_PROGRESS['completed']),
                'source_commit':str(globals().get('SOURCE_COMMIT','NOT_PINNED'))}
        _path=ROOT/'logs'/'stages_v73.jsonl'
        with _path.open('a',encoding='utf-8') as _lf:
            _lf.write(json.dumps(_entry,ensure_ascii=False)+'\n')
    progress_event(stage,f'Этап {stage} успешно завершён',announce=True)

class _ProgressBarFacade:
    def close(self):
        _PROGRESS_STOP.set();progress_refresh(force=True)
MAIN_PROGRESS=_ProgressBarFacade()

def _progress_heartbeat():
    # Обновляет только виджет, не засоряет логи строками [Пульс].
    while not _PROGRESS_STOP.wait(2):progress_refresh(force=True)
_HEARTBEAT_THREAD=threading.Thread(target=_progress_heartbeat,daemon=True,name='ml05_progress')
_HEARTBEAT_THREAD.start()
progress_event('ENVIRONMENT','Среда и зависимости подготовлены',announce=True)

# Лёгкая машинная проверка интеграционных инвариантов в самом ноутбуке.
def validate_table_contract(name,frame,mandatory=(),key=()):
    if not isinstance(frame,pd.DataFrame):fail('CONTRACT',f'{name} is not DataFrame')
    lost=set(mandatory)-set(frame.columns)
    if lost:fail('CONTRACT',f'{name}: отсутствуют обязательные поля {sorted(lost)}')
    if len(frame) and key and frame.duplicated(list(key)).any():fail('CONTRACT',f'{name}: дубли натурального ключа')
    return True

# Проверка дискового ParquetWriter до начала тяжёлой загрузки GitHub.
_probe_path=ROOT/'derived'/'_memory_writer_probe.parquet'
_probe_frame=pd.DataFrame({'region_code':pd.Series(['01000000','02000000'],dtype='string'),
    'test_year':pd.Series([2024,2025],dtype='Int64'),
    'measure':pd.Series([1.5,2.5],dtype=float),
    'period':pd.Series(['historical',2025],dtype=object)})
write_df_chunks(_probe_frame,_probe_path,chunk_rows=1)
_probe_pf=pq.ParquetFile(_probe_path)
assert _probe_pf.metadata.num_rows==2 and _probe_pf.schema_arrow.names==list(_probe_frame.columns)
_probe_read=pd.read_parquet(_probe_path)
assert _probe_read['region_code'].tolist()==['01000000','02000000']
assert _probe_read['test_year'].tolist()==[2024,2025]
assert _probe_read['period'].tolist()==['historical','2025']
del _probe_frame,_probe_pf,_probe_read
_probe_path.unlink(missing_ok=True)
progress_event('MEMORY','Тест потоковой записи Parquet и типов: PASS',announce=True)


In [ ]:
# ML-05 V7.3.0 · обязательный ранний тест SQL DuckDB до скачивания Stage I.
_adr_e02_query='''SELECT CAST(id AS VARCHAR) organization_id, TRY_CAST(year AS INTEGER) observation_year,
       CAST(indicator_name AS VARCHAR) indicator_name,CAST(indicator_unit AS VARCHAR) indicator_unit,
       CAST(area AS VARCHAR) education_area,CAST(specialty_code AS VARCHAR) specialty_code,
       TRY_CAST(indicator_value AS DOUBLE) observed_value
    FROM read_parquet(?) WHERE regexp_matches(lower(COALESCE(CAST(indicator_name AS VARCHAR),'')),
       'прием|приём|принят|зачисл|поступив')'''
_ADR_E02_SUMMARY_SQL = '''SELECT indicator_name,indicator_unit,
        COUNT(*)::BIGINT AS raw_rows,COUNT(DISTINCT organization_id)::BIGINT AS organizations,
        COUNT(DISTINCT observation_year)::BIGINT AS "years",
        COUNT(*) FILTER (WHERE observed_value IS NULL) AS invalid_numeric_rows,
        COUNT(*) FILTER (WHERE observed_value < 0 OR observed_value != FLOOR(observed_value)) AS invalid_count_rows,
        COUNT(DISTINCT COALESCE(education_area,'?')) AS areas,
        COUNT(DISTINCT COALESCE(specialty_code,'?')) AS specialty_codes
        FROM read_parquet(?) GROUP BY 1,2 ORDER BY raw_rows DESC'''

_ADR_E02_TOTAL_SQL = '''SELECT organization_id,observation_year,indicator_name,indicator_unit,
        COUNT(*)::BIGINT AS n_reporting_cells,
        COUNT(DISTINCT COALESCE(education_area,'?')) AS n_areas,
        COUNT(DISTINCT COALESCE(specialty_code,'?')) AS n_specialties,
        BOOL_AND(education_area IS NULL OR trim(CAST(education_area AS VARCHAR))='' OR
            regexp_matches(lower(CAST(education_area AS VARCHAR)),'^(всего|итого|total|all)$')) AS area_total,
        BOOL_AND(specialty_code IS NULL OR trim(CAST(specialty_code AS VARCHAR))='' OR
            regexp_matches(lower(CAST(specialty_code AS VARCHAR)),'^(всего|итого|total|all)$')) AS specialty_total,
        MIN(observed_value) AS min_count, MAX(observed_value) AS max_count
        FROM read_parquet(?) WHERE indicator_name IN (/*VALID_NAMES*/)
        GROUP BY 1,2,3,4'''

def _adr_e02_copy_statement(source, destination):
    """Build DuckDB COPY with paths SQL-quoted; the SELECT itself is preflight-tested."""
    quoted_source="'"+str(source).replace("'","''")+"'"
    quoted_target="'"+str(destination).replace("'","''")+"'"
    expected=_adr_e02_query.replace('read_parquet(?)','read_parquet('+quoted_source+')')
    if expected==_adr_e02_query:
        raise RuntimeError('ADR_SQL: source placeholder substitution failed')
    return f'COPY ({expected}) TO {quoted_target} (FORMAT PARQUET, COMPRESSION ZSTD)'

# Test EXACT production queries before downloading 2.25M E02 rows.
progress_event('SQL_PREFLIGHT','E02: SQL preflight на 3 искусственных строках (без Stage I)',0,4,announce=True)
_adr_fixture=ROOT/'derived'/'_adr_sql_preflight_input.parquet'
_adr_fixture_out=ROOT/'derived'/'_adr_sql_preflight_output.parquet'
_adr_dummy=pd.DataFrame([
    dict(id='0001',year=2022,indicator_name='Число принятых студентов',indicator_unit='чел',area='',specialty_code='',indicator_value=17),
    dict(id='0002',year=2023,indicator_name='Число зачисленных',indicator_unit='чел',area='Всего',specialty_code='Всего',indicator_value=19),
    dict(id='0003',year=2024,indicator_name='Укомплектованность',indicator_unit='чел',area='',specialty_code='',indicator_value=12)
])
try:
    _adr_dummy.to_parquet(_adr_fixture,index=False)
    progress_event('SQL_PREFLIGHT','E02 SQL: фильтрация и COPY',1,4,announce=False)
    with duck_connect() as _adr_con:
        _adr_con.execute(_adr_e02_copy_statement(_adr_fixture,_adr_fixture_out))
        progress_event('SQL_PREFLIGHT','E02 SQL: сводка по индикаторам',2,4,announce=False)
        _adr_fixture_summary=_adr_con.execute(_ADR_E02_SUMMARY_SQL,[str(_adr_fixture_out)]).df()
        if len(_adr_fixture_summary)!=2 or 'years' not in _adr_fixture_summary.columns:
            raise RuntimeError('ADR_SQL: candidate aggregation semantics mismatch')
        progress_event('SQL_PREFLIGHT','E02 SQL: проверка уникальных итоговых строк',3,4,announce=False)
        _sql=_ADR_E02_TOTAL_SQL.replace('/*VALID_NAMES*/',"'Число принятых студентов'")
        _adr_fixture_totals=_adr_con.execute(_sql,[str(_adr_fixture_out)]).df()
        if len(_adr_fixture_totals)!=1 or int(_adr_fixture_totals.iloc[0].n_reporting_cells)!=1:
            raise RuntimeError('ADR_SQL: organization-year aggregation semantics mismatch')
    progress_event('SQL_PREFLIGHT','E02 SQL preflight: PASS',4,4,announce=True)
finally:
    _adr_fixture.unlink(missing_ok=True)
    _adr_fixture_out.unlink(missing_ok=True)
    del _adr_dummy


SQL_PREFLIGHT_PASSED=True
print('E02 SQL: production templates COPY, SUMMARY and TOTAL — PASS before downloads')


# V7.3 · Проверка реального плана всех Release assets ДО дорогой загрузки источников.
def v73_release_asset_plan(index_file,chunk_dir,chunk_index,small_zip):
    parts=chunk_index.get('parts')
    if not isinstance(parts,list) or not parts:
        raise RuntimeError('V7.3: нет частей evidence')
    root=Path(chunk_dir)
    uploads=[(Path(index_file),'application/json')]
    for record in parts:
        name=str(record['name'])
        if Path(name).name!=name or not name.startswith('ML5_EVIDENCE.zip.part'):
            raise RuntimeError('V7.3: unsafe Release asset name: '+name)
        uploads.append((root/name,'application/octet-stream'))
    uploads.append((Path(small_zip),'application/zip'))
    paths=[p for p,_ in uploads]
    if len(set(p.name for p in paths))!=len(paths):
        raise RuntimeError('V7.3: repeated Release asset names')
    if not all(isinstance(p,Path) and p.is_file() and p.stat().st_size>0 for p in paths):
        raise RuntimeError('V7.3: Release plan has missing or empty files')
    return uploads

# Full 8-asset plan is exercised with tiny real files, not a static string inspection.
import tempfile as _v73_tempfile
with _v73_tempfile.TemporaryDirectory(prefix='ml5_v73_release_preflight_') as _v73_t:
    _v73_t=Path(_v73_t);(_v73_t/'chunks').mkdir()
    _v73_index=_v73_t/'ML5_V730_RELEASE_PARTS.json';_v73_index.write_text('{}')
    _v73_reports=_v73_t/'ML5_V730_ANALYTICS_test.zip';_v73_reports.write_bytes(b'PK-TEST')
    _v73_recs=[]
    for _v73_i in range(1,7):
        _v73_n=f'ML5_EVIDENCE.zip.part{_v73_i:03d}'
        (_v73_t/'chunks'/_v73_n).write_bytes(f'valid-chunk-{_v73_i}'.encode())
        _v73_recs.append({'name':_v73_n})
    _v73_probe=v73_release_asset_plan(_v73_index,_v73_t/'chunks',{'parts':_v73_recs},_v73_reports)
    assert len(_v73_probe)==8 and all(p.exists() for p,_ in _v73_probe)
    assert all(isinstance(p,Path) for p,_ in _v73_probe)
    try:
        v73_release_asset_plan(_v73_index,_v73_t/'chunks',{'parts':[{'name':'../escape'}]},_v73_reports)
        raise AssertionError('unsafe path accepted')
    except RuntimeError:pass
V73_PUBLICATION_PLAN_PREFLIGHT_PASSED=True
print('V7.3: PASS ранний тест 8 assets, Path, уникальности имён и защиты каталогов')


## 3. Приватный GitHub: доступ, неизменяемый вход и SHA-256
Токен читается только из Colab Secret `TRIAD-RU-Colab`; скрытого ручного ввода нет. Источники считываются из закреплённого Stage I; чужие сегменты не изменяются.

In [ ]:
progress_event('INPUT','Проверка токена и GitHub без изменения репозитория')

def github_token():
    try:
        from google.colab import userdata
        value = userdata.get('TRIAD-RU-Colab')
    except Exception as exc:
        raise RuntimeError('Нужен секрет Colab TRIAD-RU-Colab и разрешение доступу ноутбука (Secrets).') from exc
    if not isinstance(value,str) or not value.strip():
        raise RuntimeError('Пустой или недоступный Colab Secret: TRIAD-RU-Colab')
    return value.strip()

_TOKEN=github_token()
BASE='https://api.github.com/repos/'+CFG.REPO
SESS=requests.Session()
SESS.headers.update({'Authorization':'Bearer '+_TOKEN,'Accept':'application/vnd.github+json',
                     'X-GitHub-Api-Version':'2022-11-28'})

def api_call(method,url,*,headers=None,stream=False,timeout=(30,600),**kwargs):
    last=None
    for attempt in range(4):
        try:
            if attempt:progress_event('INPUT',f'Повтор GitHub API, попытка {attempt+1}/4',announce=True)
            r=SESS.request(method,url,headers=headers,stream=stream,timeout=timeout,**kwargs)
            if r.status_code in (429,500,502,503,504):
                last=r.status_code;r.close();time.sleep(2**attempt);continue
            if r.status_code in (401,403,404):
                status=r.status_code;r.close()
                raise PermissionError(f'GitHub HTTP {status}: права, доступ или объект отсутствуют; URL скрыт')
            if not r.ok:
                status=r.status_code;r.close()
                raise RuntimeError(f'GitHub HTTP {status}: отклонён запрос; секреты и ответ не выводятся')
            return r
        except PermissionError:raise
        except requests.RequestException:
            if attempt==3:fail('NETWORK','Ошибка запроса GitHub после повторных попыток')
            time.sleep(2**attempt)
    fail('NETWORK',f'GitHub ответил временной ошибкой {last}')

def api_json(method,url,**kw):
    r=api_call(method,url,**kw)
    try:return r.json()
    finally:r.close()

def repo_head(branch):
    p=api_json('GET',BASE+'/git/ref/heads/'+quote(branch,safe='/'))
    return p['object']['sha']

if CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB:
    _preflight_repo=api_json('GET',BASE)
    if not _preflight_repo.get('private'):
        raise RuntimeError('WRITE_PREFLIGHT: репозиторий должен быть приватным')
    _rights=_preflight_repo.get('permissions') or {}
    if not (_rights.get('push') or _rights.get('admin') or _rights.get('maintain')):
        raise PermissionError('WRITE_PREFLIGHT: TRIAD-RU-Colab должен иметь Contents:write для приватного репозитория')
    progress_event('INPUT','GitHub: приватность и права на публикацию проверены ДО обучения',announce=True)
progress_event('INPUT','Фиксация HEAD исходного и целевого репозиториев')
SOURCE_COMMIT=repo_head(CFG.SOURCE_BRANCH)
TARGET_HEAD_AT_PREFLIGHT=repo_head(CFG.TARGET_BRANCH)
# Точная фиксация одного исходного снапшота Stage I.
def read_git_blob(path,ref=SOURCE_COMMIT,with_metadata=False):
    p=api_json('GET',BASE+'/contents/'+quote(path,safe='/'),params={'ref':ref})
    if p.get('type')!='file':fail('INPUT','Ожидался Git файл '+path)
    if p.get('content'):
        content=base64.b64decode(p['content'])
    else:
        # GitHub Contents для >1 MiB может не возвращать base64; /git/blobs гарантирует исходный blob.
        blob=api_json('GET',BASE+'/git/blobs/'+p['sha'])
        if blob.get('encoding')!='base64':fail('INPUT','Неизвестный тип Git blob')
        content=base64.b64decode(blob['content'])
    return (content,p) if with_metadata else content

progress_event('INPUT','Чтение dataset_index.json и проверка Git blob SHA')
index_bytes,index_info=read_git_blob(CFG.INPUT_INDEX,with_metadata=True)
if CFG.EXPECTED_INDEX_BLOB_SHA and index_info['sha']!=CFG.EXPECTED_INDEX_BLOB_SHA:
    fail('INPUT','Изменился Git blob каталога Stage I: требуется предметная проверка версии')
INDEX=json.loads(index_bytes)
if INDEX.get('repo')!=CFG.REPO or INDEX.get('version')!='7.2':
    fail('INPUT','Несовместимый каталог Stage I')
ASSETS=INDEX['assets']; ASSET_BY_ID={a['asset_id']:a for a in ASSETS}
if len(ASSET_BY_ID)!=len(ASSETS):fail('INPUT','Повторяющиеся asset_id')
for a in ASSETS:
    if not a.get('sha256_verified') or len(a.get('sha256',''))!=64:
        fail('INPUT','Каталог содержит непроверенный исходник '+a['asset_id'])
SOURCE_MANIFESTS={}
for k,v in INDEX['source_manifests'].items():
    progress_event('INPUT','Чтение манифеста '+str(k),announce=True)
    SOURCE_MANIFESTS[k]=json.loads(read_git_blob(v))
EXPECTED_COUNT=len(ASSETS)
INPUT_SHA={}

def download_stream(r,path):
    part=Path(str(path)+'.part');part.unlink(missing_ok=True)
    try:
        expected=int(r.headers.get('content-length') or 0)
        received=0;last_report=0.
        with open(part,'wb') as f:
            for block in r.iter_content(2*1024**2):
                if block:
                    f.write(block);received+=len(block)
                    if time.monotonic()-last_report>2:
                        progress_event('DOWNLOAD','Получение Release-asset',received,expected,announce=False)
                        last_report=time.monotonic()
        progress_event('DOWNLOAD','Release-asset получен',received,expected,announce=True)
        if part.stat().st_size==0:fail('INPUT','Получен пустой файл')
        part.replace(path)
    finally:
        r.close();part.unlink(missing_ok=True)

def fetch_release_asset(asset_id,path):
    u=BASE+'/releases/assets/'+str(asset_id)
    r=api_call('GET',u,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
    content_type=r.headers.get('content-type','').lower()
    if 'json' in content_type:
        obj=r.json();r.close()
        if not obj.get('browser_download_url'):fail('INPUT','Не найден адрес закрытого Release-asset')
        r=api_call('GET',obj['browser_download_url'],stream=True,timeout=(30,900))
        if 'json' in r.headers.get('content-type','').lower():
            r.close();fail('INPUT','Private release-asset возвращает JSON вместо бинарного файла')
    download_stream(r,path)
    if path.stat().st_size==0:fail('INPUT','Скачан пустой Release-asset')
    return path

def fetch_asset(a):
    name=hashlib.sha256(a['asset_id'].encode()).hexdigest()[:20]+'.parquet'
    path=ROOT/'inputs'/name
    progress_event('INPUT',f'Исходный набор {a["asset_id"]}: проверка кэша и SHA',announce=True)
    if not path.is_file():
        for _previous_root in ('/content/ml5_v7_1_0','/content/ml5_v7_0_0'):
            _prior=Path(_previous_root)/'inputs'/name
            if not _prior.is_file():continue
            if hash_file(_prior)!=a['sha256']:continue
            try:os.link(_prior,path)  # same Colab storage, no duplicate 1GB download
            except OSError:shutil.copy2(_prior,path)
            progress_event('INPUT',f'SHA-confirmed Stage I cache reused: {a["asset_id"]}',announce=False)
            break
    # Fast restart after interrupted 7.1.x run: only SHA-verified local copies are reusable.
    if not path.exists():
        for old_root in ('/content/ml5_v7_1_1/inputs','/content/ml5_v7_1_0/inputs'):
            old=Path(old_root)/name
            if old.is_file() and hash_file(old)==a['sha256']:
                try:os.link(old,path)
                except OSError:shutil.copy2(old,path)
                break
    if path.exists() and hash_file(path)==a['sha256']:
        INPUT_SHA[a['asset_id']]=a['sha256'];return path
    path.unlink(missing_ok=True)
    if shutil.disk_usage(str(ROOT)).free < CFG.STAGE1_DISK_RESERVE_GIB*2**30:
        fail('INPUT',f'Недостаточно свободного диска: требуется резерв {CFG.STAGE1_DISK_RESERVE_GIB} GiB; можно подключить дополнительный диск или увеличить доступное место')
    loc=a['locator']
    if loc['type']=='git':
        # Вся загрузка закреплена SOURCE_COMMIT, а не плавающим main.
        progress_event('DOWNLOAD',f'Git blob {a["asset_id"]}',announce=True)
        data=read_git_blob(loc['path'])
        path.write_bytes(data)
    elif loc['type']=='release':
        progress_event('DOWNLOAD',f'Release asset {a["asset_id"]}',announce=True)
        fetch_release_asset(loc['asset_id'],path)
    else:fail('INPUT',f"Неподдерживаемый locator {loc['type']}")
    if hash_file(path)!=a['sha256']:
        path.unlink(missing_ok=True);fail('INPUT','SHA-256 не совпал '+a['asset_id'])
    INPUT_SHA[a['asset_id']]=a['sha256']
    progress_event('INPUT',f'Проверено {a["asset_id"]} · {path.stat().st_size/2**20:.1f} МиБ',announce=True)
    total=sum((ROOT/'inputs'/f).stat().st_size for f in os.listdir(ROOT/'inputs') if (ROOT/'inputs'/f).is_file())
    if total/2**30>CFG.MAX_REMOTE_INPUT_GIB:fail('INPUT','Лимит локального кэша Stage I превышен')
    return path

def original_spo_workbook():
    x=[a for a in INDEX.get('extra_originals',[]) if a.get('role')=='migration_matrix_original_xlsx']
    if len(x)!=1:return None
    path=ROOT/'inputs'/'spo_original_e01.xlsx'
    if not path.exists():fetch_release_asset(x[0]['locator']['asset_id'],path)
    # В Stage I для оригинала нет SHA в dataset_index: это не равно подтверждению контрольной суммы.
    return path

def source_asset(segment=None,role=None,source_id=None,prefix=None):
    return [a for a in ASSETS if (segment is None or a['segment']==segment)
            and (role is None or a.get('role')==role)
            and (source_id is None or a.get('source_id')==source_id)
            and (prefix is None or a['asset_id'].startswith(prefix))]

def required_one(items,label):
    if len(items)!=1:fail('INPUT',f'{label}: ожидается 1 источник, найдено {len(items)}')
    return items[0]

def read_table(a,columns=None,where=None,limit=None):
    progress_event('SQL',f'Чтение {a["asset_id"]}, фильтр={bool(where)}, limit={limit}',announce=True)
    path=fetch_asset(a)
    cols=a.get('columns_manifest') or list(pq.ParquetFile(path).schema_arrow.names)
    if columns:
        missing=set(columns)-set(cols)
        if missing:fail('SCHEMA',f'{a["asset_id"]}: отсутствуют {sorted(missing)}')
        selected=','.join('"'+c.replace('"','""')+'"' for c in columns)
    else:selected='*'
    sql=f"SELECT {selected} FROM read_parquet(?)"
    if where:sql+=' WHERE '+where
    if limit:sql+=f' LIMIT {int(limit)}'
    con=duck_connect()
    try:
        if limit is None and int(a.get('rows_actual') or 0)>CFG.MAX_PANDAS_ROWS:
            rows=int(con.execute('SELECT COUNT(*) FROM ('+sql+') AS _guard',[str(path)]).fetchone()[0])
            if rows>CFG.MAX_PANDAS_ROWS:
                fail('RAM_GUARD',f'{a["asset_id"]}: запрос даёт {rows:,} строк; запрещено .df() в FULL, используйте DuckDB COPY/агрегацию')
        result=con.execute(sql,[str(path)]).df()
        progress_event('SQL',f'{a["asset_id"]}: получено {len(result):,} строк',announce=True)
        return result
    finally:con.close()

def sql_query(a,query):
    progress_event('SQL',f'Агрегация {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    con=duck_connect()
    try:
        result=con.execute(query,[str(path)]).df()
        progress_event('SQL',f'Агрегация {a["asset_id"]}: {len(result):,} строк',announce=True)
        return result
    finally:con.close()

stage1_sources=pd.DataFrame([{'asset_id':a['asset_id'],'segment':a['segment'],'source_id':a['source_id'],
 'role':a.get('role',''),'registered_rows':a.get('rows_actual'), 'locator':a['locator']['type'],
 'sha256':a['sha256'], 'columns':','.join(a.get('columns_manifest',[]))[:500]} for a in ASSETS])
qa_table('stage1_sources',stage1_sources,['asset_id'])
source_ids=set(a.get('source_id') for a in ASSETS)
for missing_id in ('D03','D06'):
    if missing_id not in source_ids:
        ISSUES.append({'module':'STAGE1','status':'SOURCE_NOT_INGESTED',
                       'reason':f'В каталоге нет {missing_id}; нельзя подменять внешними данными','required':missing_id})
note('INPUT',f'pin {SOURCE_COMMIT}; catalog {index_info["sha"]}; {EXPECTED_COUNT} assets')
print('INPUT preflight:', len(ASSETS),'objects; git commit',SOURCE_COMMIT[:12],'; no writes')

mark_stage('INPUT')


## 3А. Полная инвентаризация Stage I — 52/52

В режиме FULL каждое зарегистрированное Parquet скачивается, проверяется по SHA-256, схеме и числу строк, затем построчно оценивается годовой охват без переноса гигантских таблиц в Pandas. Единица анализа при этом сохраняется: **проверка файла не приписывает его значения образовательной миграции**. Годовые shards СПО рассматриваются как альтернативные представления, а не добавочные наблюдения.

In [ ]:
# Stage I FULL52 — strict SHA+Parquet+content profiles, disk-conscious, resumable download cache.
progress_event('CATALOG_52','Проверка ВСЕХ файлов Stage I, SHA, схем, лет и строк',announce=True)
FULL52_INVENTORY=[];FULL52_YEAR_STATS=[];FULL52_THEME_STATS=[];FULL52_ORIGINALS=[]

def _catalog_family(a):
    x=a['asset_id'];seg=a.get('segment','')
    if seg=='01':return 'MUNICIPAL_DEMOGRAPHY'
    if seg=='02':return 'MUNICIPAL_ECONOMY_LABOUR_EDUCATION_TAX'
    if seg=='03':
        if a.get('role')=='education_od_matrix_raw_cells':return 'SPO_ORIGINAL_OD_CELLS_NOT_ADDITIVE'
        if a.get('role')=='organization_indicators':return 'SPO_ALL_YEARS_OR_YEAR_SHARD_NOT_ADDITIVE'
        return 'SPO_REGISTRY_OD_REGIONAL'
    if seg=='04':return 'HIGHER_EDUCATION_VPO'
    if seg=='05':return 'GRADUATE_LABOUR_OUTCOMES'
    if seg=='06':return 'MUNICIPAL_SPATIAL_BDMO_TRANSPORT'
    return 'UNKNOWN'

def _source_year_scan(a,p,cols):
    key=a.get('time_key')
    if key not in cols:key=next((c for c in ('year','observation_year','graduation_year') if c in cols),None)
    if not key:return [],'NO_TIME_KEY'
    column='"'+key.replace('"','""')+'"'
    try:
        with duck_connect() as con:
            years=con.execute(f'''SELECT CAST({column} AS VARCHAR) observed_year,COUNT(*)::BIGINT n
                FROM read_parquet(?) GROUP BY 1 ORDER BY 1''',[str(p)]).fetchall()
        return [{'asset_id':a['asset_id'],'year_label':str(y),'rows':int(n),'time_column':key} for y,n in years],'SCANNED_FULL'
    except (duckdb.OutOfMemoryException,duckdb.IOException) as ex:
        ISSUES.append({'module':'SOURCE_AUDIT','status':'TIME_DISTRIBUTION_NOT_AVAILABLE',
                       'reason':a['asset_id']+' '+type(ex).__name__,'required':'more spill memory for time scan'})
        return [],'YEAR_SCAN_UNAVAILABLE'

for i,a in enumerate(ASSETS,1):
    aid=a['asset_id']
    if CFG.MODE!='FULL' and aid not in {'01::0','03::42f47d3cf2159992::000','03::89505d5993e6f4ad::000'}:
        FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
           'research_family':_catalog_family(a),'observed_rows':None,'expected_rows':a.get('rows_actual'),
           'physical_read_ok':False,'footer_rows_ok':False,'sha256_ok':False,
           'years_scanned':0,'year_scan_status':'NOT_SCANNED_IN_PILOT',
           'source_sha256':a['sha256'],'level_gate':'PILOT_NOT_FULL'})
        continue
    progress_event('CATALOG_52',f'[{i:02d}/52] {aid}: SHA + контроль Parquet',i,len(ASSETS),announce=True)
    source=fetch_asset(a)  # verifies SHA both fresh and cached; never silently skipped
    footer=pq.ParquetFile(source)
    cols=list(footer.schema_arrow.names);registered=int(a.get('rows_actual') or 0)
    found=int(footer.metadata.num_rows)
    if found!=registered:fail('FULL52',f'{aid}: rows {found:,} != Stage I {registered:,}')
    manifest_cols=list(a.get('columns_manifest') or [])
    if manifest_cols and set(cols)!=set(manifest_cols):fail('FULL52',f'{aid}: Parquet fields != Stage I catalog')
    # Content touch: inspect a physical row through a LIMIT, not whole table in pandas.
    with duck_connect() as con:
        touched=int(con.execute('SELECT COUNT(*) FROM (SELECT 1 FROM read_parquet(?) LIMIT 1)',[str(source)]).fetchone()[0])
    if touched!=int(found>0):fail('FULL52',f'{aid}: нельзя прочитать содержимое Parquet')
    yr,yr_status=_source_year_scan(a,source,cols) if CFG.CATALOG_SCAN_ALL_YEARS else ([], 'SCAN_DISABLED')
    if yr and sum(r['rows'] for r in yr)!=found:fail('FULL52',f'{aid}: сумма годового охвата не совпадает с footer')
    FULL52_YEAR_STATS.extend(yr)
    family=_catalog_family(a)
    FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
      'data_role':a.get('role') or a.get('data_role'),'research_family':family,'observed_rows':found,
      'expected_rows':registered,'columns':len(cols),'bytes':source.stat().st_size,
      'physical_read_ok':True,'footer_rows_ok':found==registered,'sha256_ok':INPUT_SHA.get(aid)==a['sha256'],
      'years_scanned':len(yr),'year_scan_status':yr_status,'first_year':yr[0]['year_label'] if yr else None,
      'last_year':yr[-1]['year_label'] if yr else None,'source_sha256':a['sha256'],
      'original_grain_status':a.get('grain_status'),'level_gate':'ORIGINAL_GRAIN_NOT_ASSUMED_ADDITIVE'})
    del footer
    if i%4==0:gc.collect();memory_point(f'catalog_{i}_of_{len(ASSETS)}')

if CFG.MODE=='FULL' and (len(FULL52_INVENTORY)!=len(ASSETS) or len(INPUT_SHA)!=len(ASSETS)):
    fail('FULL52','Неполное чтение/контроль входных файлов')
# Role-specific research trail: every source has a declared analytic role, with no double-counting.
for fam,grp in pd.DataFrame(FULL52_INVENTORY).groupby('research_family'):
    FULL52_THEME_STATS.append({'family':fam,'n_assets':len(grp), 'source_rows_scanned_by_footer':int(grp.observed_rows.sum()),
      'interpretation':'ALT_REPRESENTATIONS_NOT_ADDITIVE' if 'SHARD' in fam or 'CELLS' in fam else 'OWN_GRAIN',
      'sha_verified':int(grp.sha256_ok.sum())})
qa_table('full52_source_inventory',pd.DataFrame(FULL52_INVENTORY),['asset_id'])
qa_table('full52_year_coverage',pd.DataFrame(FULL52_YEAR_STATS),allow_empty=True)
qa_table('full52_theme_coverage',pd.DataFrame(FULL52_THEME_STATS),['family'])
# Original SPO workbook is outside the 52 Parquet assets. Record accessibility separately.
try:
    ori=original_spo_workbook()
    if ori:
        FULL52_ORIGINALS=[{'name':ori.name,'bytes':ori.stat().st_size,'sha256_locally_calculated':hash_file(ori),
          'registered_in_52':False,'status':'CONTENT_AVAILABLE_METHODOLOGY_NOT_YET_IDENTIFIED'}]
except Exception as ex:
    FULL52_ORIGINALS=[{'name':'original_spo_workbook','status':'NOT_AVAILABLE','reason':type(ex).__name__}]
qa_table('full52_raw_originals',pd.DataFrame(FULL52_ORIGINALS),allow_empty=True)
print('CATALOG 52:',len(INPUT_SHA),'SHA verified;',sum(t['observed_rows'] or 0 for t in FULL52_INVENTORY),
      'Parquet rows across files (NOT unique observations);',len(FULL52_YEAR_STATS),'asset-year groups')
mark_stage('CATALOG_52')


## 4. Измерение и определения исходных величин
Для E01 сохраняются фактические заголовки оригинального XLSX и исходных raw-таблиц. Нельзя автоматически называть показателем миграции населения или числом зачисленных наблюдение об образовании и текущем обучении.

In [ ]:
memory_point('before_semantics')
progress_event('SEMANTICS','Аудит определений E01/E02/ВПО/СПО')
# Семантика извлекается из значений исходных таблиц, а не из догадки по названию файла.
E01=required_one(source_asset(role='education_od_flows'),'E01 OD')
SPO_ORG=required_one(source_asset(role='education_organizations'),'E01 organizations')
SPO_IND=sorted(source_asset(segment='03',role='organization_indicators'),
               key=lambda a:0 if 'all_years' in a.get('member','') else 1)
SPO_MASTER=next((a for a in SPO_IND if 'all_years' in a.get('member','')),None)
SPO_REG=required_one(source_asset(role='regional_spo'),'SPO regional indicators')
E02=required_one(source_asset(prefix='04::E02'),'E02 universities')
VPO_211=required_one(source_asset(prefix='04::VPO_2_1_1'),'VPO-1 admissions table')
VPO_213=required_one(source_asset(prefix='04::VPO_2_1_3'),'VPO-1 graduates table')
GRAD_ORG=required_one(source_asset(prefix='05::E04::organization_total'),'E04 graduates')
GRAD_FIELD=required_one(source_asset(prefix='05::E05::field_specialty'),'E05 specialties')
MUN=required_one(source_asset(segment='01'),'Municipal demography')
TRANSPORT=required_one(source_asset(prefix='06::TRANSPORT'),'transport graph')
SPATIAL=required_one(source_asset(prefix='06::D04::D04_member00::spatial_reference'),'geography reference')

# Для FULL/PILOT10 производим сканирование всех строк выбранных источников колонночным движком.
def indicator_catalog(a,fields,cap=None):
    cols=[c for c in fields if c in a.get('columns_manifest',[])]
    if 'indicator_name' not in cols:fail('SEMANTICS','Нет колонки indicator_name: '+a['asset_id'])
    progress_event('SEMANTICS',f'Сканирование словаря {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    with duck_connect() as con:
        # В SMOKE ограничиваем ввод — это не окончательный словарь.
        sample=f' LIMIT {CFG.MAX_READ_ROWS_SMOKE}' if CFG.MODE=='SMOKE' else ''
        group=','.join('"'+c+'"' for c in cols)
        sql=f'SELECT {group}, count(*)::BIGINT as n FROM (SELECT {group} FROM read_parquet(?) {sample}) GROUP BY {group} ORDER BY n DESC'
        d=con.execute(sql,[str(path)]).df()
    d['asset_id']=a['asset_id'];d['catalog_exhaustive']=CFG.MODE!='SMOKE'
    progress_event('SEMANTICS',f'Словарь готов: {a["asset_id"]} · {len(d):,} записей',announce=True)
    return d

CAT_E02=indicator_catalog(E02,['indicator_name','indicator_unit','area'])
CAT_VPO=indicator_catalog(VPO_211,['indicator_name','admission_type','education_level','education_form'])
CAT_SPO=indicator_catalog(SPO_MASTER,['indicator_name','indicator_unit','indicator_section']) if SPO_MASTER else pd.DataFrame()
CAT_SPO_REG=indicator_catalog(SPO_REG,['indicator_name','indicator_unit','indicator_section'])
CAT_VPO_GRAD=indicator_catalog(VPO_213,['indicator_name','education_level','education_form'])
for name,tab in [('dictionary_e02',CAT_E02),('dictionary_vpo_admission',CAT_VPO),
                 ('dictionary_spo',CAT_SPO),('dictionary_spo_regional',CAT_SPO_REG),
                 ('dictionary_vpo_graduation',CAT_VPO_GRAD)]:qa_table(name,tab,allow_empty=True)

POS_ADMISSION=re.compile(r'(?i)(?:^|\W)(?:при[её]м|принят|зачисл|поступил|поступивш|принятых)(?:\w|\W|$)')
NEG_ADMISSION=re.compile(r'(?i)(?:заявлен|подавш|конкурс|при[её]мн\w* комисс|при[её]м на работу|планируем|места приема|контрольн\w+ цифр|ваканс)')
POS_GRAD=re.compile(r'(?i)(выпуск|окончивш|завершивш)')
NEG_TOTAL=re.compile(r'(?i)(процент|доля|удельн|на 100|отношение|коэффициент|темп|индекс|стоимост|тыс\. руб|млн)')

# Утверждение требует положительного семантического определения, единицы и отсутствия разрезов-двойников.
def classify_indicator(name,unit=None,category=None):
    value='' if pd.isna(name) else str(name).strip()
    u='' if pd.isna(unit) else str(unit).strip().lower()
    cat='' if pd.isna(category) else str(category)
    if POS_ADMISSION.search(value) and not NEG_ADMISSION.search(value) and not NEG_TOTAL.search(value):
        role='ADMISSION_CANDIDATE'
    elif POS_GRAD.search(value) and not NEG_TOTAL.search(value):role='GRADUATION_CANDIDATE'
    else:role='OTHER'
    unit_ok=not u or bool(re.search(r'чел|человек|единиц|численност|шт',u))
    # Автоматическая классификация создаёт кандидатов, но НЕ утверждает научную семантику.
    grade='ELIGIBLE_FOR_ROW_CHECK' if role!='OTHER' and unit_ok else 'QUARANTINE'
    if role!='OTHER' and not unit_ok:grade='UNEXPECTED_UNIT'
    return {'semantic_role':role,'semantic_gate':grade,
            'name_exact':value,'unit_exact':u,'category_exact':cat}

CANDIDATES=[]
for source,frame in [('E02',CAT_E02),('VPO_2_1_1',CAT_VPO),('SPO_ORG',CAT_SPO),('SPO_REG',CAT_SPO_REG),('VPO_GRAD',CAT_VPO_GRAD)]:
    for _,r in frame.iterrows():
        d=classify_indicator(r.get('indicator_name'),r.get('indicator_unit'),r.get('admission_type'))
        CANDIDATES.append({'source':source,'indicator_name':r.get('indicator_name'),'rows':int(r['n']),
                           'level':r.get('education_level'), 'form':r.get('education_form'),
                           'admission_type':r.get('admission_type'), 'unit':r.get('indicator_unit'),
                           'indicator_section':r.get('indicator_section'),'area':r.get('area'),**d})
CANDIDATE_CATALOG=qa_table('semantic_indicator_catalog',pd.DataFrame(CANDIDATES),allow_empty=True)

OD_SEMANTIC={
 'source_id':'E01','origin_definition':'region_of_previous_general_education',
 'destination_definition':'region_of_current_vocational_study',
 'label':'observed_E01_schooling_to_study_value',
 'admissions_confirmed':False,'permanent_migration_confirmed':False,
 'yearly_count_unit_confirmed':False,'unit':'reported_indicator_value',
 'methodological_status':'SOURCE_DESCRIPTION_REQUIRES_PRIMARY_CELL_VERIFICATION',
 'statistics_available_asof':'2026_SOURCE_VINTAGE_ONLY',
 'admission_target_gate':'NOT_CONFIRMED',
 'source_page':'https://tochno.st/datasets/monitoring_spo',
 'primary_workbook':'Stage I extra_originals/migration_matrix_original_xlsx'
}
ORIGINAL_SHEETS=[]
try:
    progress_event('SEMANTICS','Чтение первичного XLSX E01',announce=True)
    p=original_spo_workbook()
    if p:
        xf=pd.ExcelFile(p,engine='calamine')
        for sheet in xf.sheet_names:
            raw=pd.read_excel(p,sheet_name=sheet,header=None,nrows=25,engine='calamine')
            cells=' | '.join(str(v)[:100] for v in raw.to_numpy().reshape(-1)
                           if not pd.isna(v) and str(v).strip())[:2500]
            ORIGINAL_SHEETS.append({'sheet':sheet,'first_cells':cells,'file_sha256':hash_file(p)})
        OD_SEMANTIC['original_opened']=True
except Exception as ex:
    OD_SEMANTIC['original_opened']=False
    ISSUES.append({'module':'ML-5.1','status':'PRIMARY_SOURCE_REVIEW_PENDING',
                   'reason':'Оригинальный XLSX E01 не открыт; ошибка '+type(ex).__name__,
                   'required':'original E01 workbook access'})
qa_table('e01_original_headers',pd.DataFrame(ORIGINAL_SHEETS),allow_empty=True)
# Даже наличие слова «миграция» в заголовке не доказывает, что показатель — поток зачисления.
if not OD_SEMANTIC.get('original_opened'):
    note('SEM-E01','Оригинал не открыт: допустим ограниченный анализ наблюдаемой величины','WARN')
else:
    note('SEM-E01','Оригинальные заголовки записаны; семантика admission пока не подтверждена')

RAW_SPO_CELL_AUDIT=[]
for a in source_asset(role='education_od_matrix_raw_cells'):
    try:
        p=fetch_asset(a);file=pq.ParquetFile(p)
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'rows':file.metadata.num_rows,
                                   'columns':','.join(file.schema_arrow.names),'sha256':a['sha256']})
    except Exception as ex:
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'error':type(ex).__name__})
qa_table('e01_raw_cell_sources',pd.DataFrame(RAW_SPO_CELL_AUDIT),allow_empty=True)

# Проверка ручного списка после чтения фактических словарей; не считать пустоту списка отсутствием исходных данных.
def approved_names(source):
    manually={'E02':CFG.APPROVED_VO_ADMISSION_NAMES,'SPO_ORG':CFG.APPROVED_SPO_ADMISSION_NAMES}
    names=tuple(manually.get(source,()))
    existing=set(CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(source),'indicator_name'].dropna().astype(str))
    bad=set(names)-existing
    if bad:fail('SEMANTICS',f'{source}: утверждённые имена не найдены в данных: {sorted(bad)}')
    return sorted(set(names))

SEMANTIC_APPROVAL=pd.DataFrame([{'source':src,'indicator_name':n,'approved_for_admission':n in approved_names(src),
    'semantic_role':r,'unit':u} for src in ('E02','SPO_ORG')
    for n,r,u in CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(src),
        ['indicator_name','semantic_role','unit']].drop_duplicates().itertuples(index=False,name=None)])
qa_table('admission_approval_register',SEMANTIC_APPROVAL,allow_empty=True)
print('Semantic candidates:',len(CANDIDATE_CATALOG),'| E01 original sheets:',len(ORIGINAL_SHEETS),
      '| SPO raw-cell files:',len(RAW_SPO_CELL_AUDIT))

mark_stage('SEMANTICS')


## 5. Исследовательские панели и контролируемые сопоставления
Статистические уровни ВО, СПО, организация, филиал, регион и муниципалитет не смешиваются. Спорные организационные ID исключаются в карантин, ноль отличается от отсутствия.

In [ ]:
memory_point('before_panels')
progress_event('PANELS','Исходные панели: E01, организации, ВПО, E04/E05, муниципалитеты')
# Шесть исходных исследовательских панелей и независимые таблицы ошибок сопоставления.
OD_RAW=read_table(E01,['year','origin_region_code','origin_region_name',
                       'destination_region_code','destination_region_name','indicator_value'])
OD_RAW=OD_RAW.rename(columns={'origin_region_code':'origin_region','destination_region_code':'destination_region',
                              'indicator_value':'observed_value','year':'observation_year'})
OD_RAW['origin_region']=norm_code(OD_RAW['origin_region'])
OD_RAW['destination_region']=norm_code(OD_RAW['destination_region'])
OD_RAW['observation_year']=safe_year(OD_RAW['observation_year'])
OD_RAW['observed_value']=number(OD_RAW['observed_value'])
OD_RAW['sector']='SPO';OD_RAW['origin_definition']='previous_general_education_region'
OD_RAW['destination_definition']='current_study_region'
OD_RAW['target_definition']=OD_SEMANTIC['label']
OD_RAW['known_origin']=nonmissing_code(OD_RAW.origin_region)
OD_RAW['known_destination']=nonmissing_code(OD_RAW.destination_region)
OD_RAW['code_status']=np.select([(~OD_RAW.known_origin|~OD_RAW.known_destination).fillna(True).to_numpy(dtype=bool),
                                OD_RAW.origin_region.eq(OD_RAW.destination_region).fillna(False).to_numpy(dtype=bool)],
                                ['UNKNOWN_CODE','INTRA_REGION'],default='INTER_REGION')
if OD_RAW['observed_value'].lt(0).any():fail('SEM-E01','Отрицательные величины E01: нужна особая интерпретация')
if OD_RAW['observed_value'].isna().any():
    QUARANTINE.extend([{'table':'edu_od_year','reason':'missing_value','row':int(i)}
                       for i in OD_RAW.index[OD_RAW.observed_value.isna()].tolist()[:100]])
OD_RAW['explicit_zero']=OD_RAW.observed_value.eq(0)&OD_RAW.observed_value.notna()
OD_RAW['missing_edge']=False;OD_RAW['suppressed']=False
OD_RAW['source_asset_id']=E01['asset_id'];OD_RAW['source_vintage']=INDEX['created_at']
# Год может быть статистическим периодом учёбы; не переименовывать в admission_year.
OD=qa_table('edu_od_year',OD_RAW,
            ['observation_year','origin_region','destination_region'])
OD_INTER=qa_table('edu_od_interregional',OD[OD.code_status=='INTER_REGION'].copy(),
              ['observation_year','origin_region','destination_region'],allow_empty=True)
OD_INTRA=qa_table('edu_od_intraregional',OD[OD.code_status=='INTRA_REGION'].copy(),allow_empty=True)
OD_UNKNOWN=qa_table('edu_od_unknown_code',OD[OD.code_status=='UNKNOWN_CODE'].copy(),allow_empty=True)
if sum(map(len,[OD_INTER,OD_INTRA,OD_UNKNOWN]))!=len(OD):fail('DATA_GATE','OD: категории не исчерпывают данные')

counts=OD.groupby(['observation_year','code_status'],dropna=False).agg(
    rows=('observed_value','size'),total_value=('observed_value','sum')).reset_index()
qa_table('od_composition',counts,allow_empty=True)
EDGES=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].copy()
EDGES['edge_id']=EDGES.origin_region.astype(str)+'|'+EDGES.destination_region.astype(str)
edge_years={int(y):set(g.edge_id) for y,g in EDGES.groupby('observation_year')}
coverage=[];transitions=[]
for y in sorted(edge_years):
    keys=edge_years[y];past=edge_years.get(y-1,set())
    coverage.append({'year':y,'observed_edges':len(keys),'new_vs_previous':len(keys-past) if past else pd.NA,
                     'lost_vs_previous':len(past-keys) if past else pd.NA,
                     'intersection_vs_previous':len(keys&past) if past else pd.NA,
                     'jaccard_vs_previous':len(keys&past)/len(keys|past) if past and (keys|past) else np.nan,
                     'total_observed_value':float(EDGES.loc[EDGES.observation_year==y,'observed_value'].sum())})
    if past:
        transitions.extend([{'year':y,'edge_id':e,'transition':'NEW'} for e in sorted(keys-past)])
        transitions.extend([{'year':y,'edge_id':e,'transition':'NOT_REPORTED'} for e in sorted(past-keys)])
coverage_df=qa_table('od_edge_coverage',pd.DataFrame(coverage),['year'])
qa_table('od_edge_transitions',pd.DataFrame(transitions),allow_empty=True)
if len(coverage_df)>1:
    coverage_df['edge_growth_ratio']=coverage_df.observed_edges/coverage_df.observed_edges.shift(1)
    coverage_df['potential_break']=coverage_df.edge_growth_ratio.gt(1.5) | coverage_df.edge_growth_ratio.lt(0.67)
    if coverage_df.potential_break.any():
        ISSUES.append({'module':'SEM-E01','status':'COVERAGE_BREAK_UNRESOLVED',
         'reason':'Изменение регистрации OD-направлений: прямое межгодовое сравнение требует аудита полноты',
         'required':'Единые определения и покрытие матрицы по годам'})

# Реестр ВО из E02: комбинация исторического года, организационного id и географического положения.
# E02 длинная таблица показателей (2.25 млн строк): реестр читаем DISTINCT в DuckDB,
# не загружая миллионы повторяющихся идентификаторов в оперативную память.
with duck_connect() as con:
    progress_event('PANELS','E02: DISTINCT организаций из полной таблицы, без переноса 2.25 млн строк в pandas',announce=True)
    vo=con.execute('''SELECT DISTINCT id,name,inn,ogrn,region,oktmo,institution,parent_id,parent_name,year
                     FROM read_parquet(?)''',[str(fetch_asset(E02))]).df()
if CFG.MODE=='SMOKE':vo=vo.head(CFG.MAX_READ_ROWS_SMOKE).copy()
for c in ['id','parent_id','oktmo','inn','ogrn']:vo[c]=norm_code(vo[c])
vo['observation_year']=safe_year(vo['year'])
vo=vo.drop(columns=['year'])
vo=vo.loc[vo.id.notna()&vo.observation_year.notna()].drop_duplicates().copy()
vo['sector']='VO';vo['organization_id']=vo['id']
vo['campus_id']=vo['id'] # Только при явном собственном id в исходнике; головной id не переносить на филиал.
vo['parent_group_id']=vo.parent_id.fillna(vo.organization_id)
vo['entity_id']='VO|'+vo.organization_id.astype(str)
vo['campus_geo_status']=np.where(nonmissing_code(vo.oktmo),'REPORTING_OKTMO_NOT_VERIFIED_CAMPUS','UNKNOWN')
vo['entity_match_confidence']=1.0

sporeg=read_table(SPO_ORG,['year','region_name','region_oktmo','id','name','city','lat','lon','type','head','address','branches'])
sporeg['observation_year']=safe_year(sporeg.year);sporeg=sporeg.drop(columns=['year'])
sporeg['id']=norm_code(sporeg.id);sporeg['region_oktmo']=norm_code(sporeg.region_oktmo)
sporeg['sector']='SPO';sporeg['organization_id']=sporeg.id
sporeg['entity_id']='SPO|'+sporeg.id.fillna('UNRESOLVED')
sporeg['campus_geo_status']='POINT_WITHOUT_MUNICIPAL_OKTMO'
sporeg['entity_match_confidence']=np.where(sporeg.id.notna(),1.0,0.0)
sporeg['parent_group_id']=sporeg['head'].fillna(sporeg['id']).astype('string')
sporeg['name_normalized']=sporeg.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()

# Проверять уникальность натурального ключа после исключения полных дублей.
# Исторические конфликты не удаляются молча: полные дубли снимаются, остальные в карантин.
def quarantine_conflicting_registry(df,sector):
    key=['entity_id','observation_year']
    if df.empty:return df
    df=df.drop_duplicates().copy()
    unknown=df.entity_id.astype(str).str.contains('UNRESOLVED',regex=False) | df.entity_id.isna()
    if unknown.any():
        QUARANTINE.extend(df.loc[unknown].head(500).assign(source=sector+'_MISSING_ID').to_dict('records'))
        df=df.loc[~unknown].copy()
    counts=df.groupby(key,dropna=False).size()
    badkeys=set(counts[counts>1].index)
    if badkeys:
        badmask=pd.MultiIndex.from_frame(df[key]).isin(pd.MultiIndex.from_tuples(list(badkeys),names=key))
        QUARANTINE.extend(df.loc[badmask].head(1000).assign(source=sector+'_CONFLICTING_ID_YEAR').to_dict('records'))
        df=df.loc[~badmask].copy()
    return df
vo=quarantine_conflicting_registry(vo,'VO')
sporeg=quarantine_conflicting_registry(sporeg,'SPO')
for x,tab in [('VO',vo),('SPO',sporeg)]:
    if tab.empty:ISSUES.append({'module':'ML-5.2','status':'NO_REGISTRY',
          'reason':f'Исторический реестр {x} пуст после контроля конфликтных идентификаторов','required':'unique stable IDs'})
registry_columns=['sector','observation_year','entity_id','organization_id','parent_group_id','name','region','oktmo',
                  'city','address','campus_geo_status','entity_match_confidence']
vo2=vo.rename(columns={'region':'region','oktmo':'oktmo'})
spo2=sporeg.rename(columns={'region_name':'region','region_oktmo':'region_oktmo_source'})
reg_parts=[]
for frame in (vo2,spo2):
    part=frame.copy()
    for col in registry_columns:
        if col not in part:part[col]=pd.NA
    reg_parts.append(part[registry_columns])
REGISTRY=qa_table('edu_org_registry',pd.concat(reg_parts,ignore_index=True),
                  ['sector','entity_id','observation_year'],allow_empty=True)
REGISTRY_AUDIT=pd.DataFrame([{'sector':s,'records':len(g),'unique_org_ids':g.entity_id.nunique(),
    'years':g.observation_year.nunique(),'confirmed_campuses':int(g.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum())}
    for s,g in REGISTRY.groupby('sector')])
qa_table('registry_coverage_audit',REGISTRY_AUDIT,allow_empty=True)

# Словари по учреждениям: ни один показатель не принимается по непустоте конфигурационного кортежа.
def selected_names(source,role='ADMISSION_CANDIDATE'):
    # В словарях сохраняются кандидаты, но в прогноз попадают только утверждённые точные имена.
    sub=CANDIDATE_CATALOG[(CANDIDATE_CATALOG.source==source)&
         (CANDIDATE_CATALOG.semantic_role==role)&
         (CANDIDATE_CATALOG.semantic_gate=='ELIGIBLE_FOR_ROW_CHECK')]
    candidates=set(sub.indicator_name.dropna().astype(str).unique())
    if source in ('E02','SPO_ORG'):
        approved=set(approved_names(source));unapproved=sorted(candidates-approved)
        if unapproved:ISSUES.append({'module':'ML-5.2','status':'UNAPPROVED_ADMISSION_CANDIDATES',
          'reason':f'{source}: {len(unapproved)} неутверждённых исходных имён','required':'утвердить точные имена в конфигурации'})
        return sorted(candidates&approved)
    return sorted(candidates)

# Панель приема ВО на уровне E02: все положительные candidates + точный натуральный ключ + единица.
# При неоднозначности исходных area/specialty/indicator нужна quarantine, а не сложение строк.
def selective_records(a,cols,names):
    if not names:return pd.DataFrame(columns=cols)
    esc=lambda t:"'"+t.replace("'","''")+"'"
    name_list=','.join(esc(t) for t in names)
    return read_table(a,cols,where=f'indicator_name IN ({name_list})',
        limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)

vo_admission=selective_records(E02,['id','parent_id','year','name','region','oktmo',
 'area','specialty_code','specialty_section','indicator_name','indicator_unit','indicator_value'],
 selected_names('E02'))
if len(vo_admission):
    vo_admission['entity_id']='VO|'+norm_code(vo_admission.id)
    vo_admission['observation_year']=safe_year(vo_admission.year)
    vo_admission['value']=number(vo_admission.indicator_value)
    vo_admission['indicator_grain']='E02_ID_YEAR_AREA_SPECIALTY_INDICATOR'
    # Сохраняем полный programme-разрез до отбора только общих строк для орг-панели.
    vo_program_candidates=vo_admission.copy()
    # area — раздел мониторинга, а не доказанный подитог. Ограничиваем только specialty_code.
    for categorical in ('specialty_code',):
        v=vo_admission[categorical].astype('string').str.strip().str.lower()
        keep=v.isna() | v.eq('') | v.isin(['всего','итого','total','0','00'])
        rejected=vo_admission.loc[~keep].copy()
        if len(rejected):QUARANTINE.extend(rejected.head(100).assign(source='VO_NON_TOTAL_'+categorical).to_dict('records'))
        vo_admission=vo_admission.loc[keep].copy()

# SPO данные организации сопрягаются только по уникальному совпадению региона/названия/года.
spo_admission=selective_records(SPO_MASTER,['year','region_name','name','property','indicator_section',
 'indicator_name','indicator_unit','org_value'],selected_names('SPO_ORG')) if SPO_MASTER else pd.DataFrame()
if len(spo_admission):
    spo_admission['name_normalized']=spo_admission.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
    spo_admission['observation_year']=safe_year(spo_admission.year)
    spo_admission['region_name']=spo_admission.region_name.astype('string').str.strip()
    lookup=sporeg[['region_name','name_normalized','observation_year','entity_id']].drop_duplicates()
    count=lookup.groupby(['region_name','name_normalized','observation_year'],dropna=False).size().rename('matches').reset_index()
    lookup=lookup.merge(count,on=['region_name','name_normalized','observation_year'],validate='many_to_one')
    ambiguous=lookup[lookup.matches!=1].copy()
    if len(ambiguous):QUARANTINE.extend(ambiguous.head(500).assign(source='SPO_MATCH').to_dict('records'))
    lookup=lookup[lookup.matches==1].drop(columns=['matches'])
    spo_admission=spo_admission.merge(lookup,on=['region_name','name_normalized','observation_year'],
                                     how='left',validate='many_to_one')
    spo_admission['value']=number(spo_admission.org_value)
    spo_admission['indicator_grain']='SPO_ORG_YEAR_PROPERTY_INDICATOR'
    # property в мониторинге СПО может обозначать форму собственности организации,
    # а не разрез значения. Не удаляем реальные организации по этому полю.

# ВПО-1: полная точная агрегация, частями на диск. Лимит 512 МБ используется только для лёгких запросов.
VPO_FULL_PROGRAM_DESCRIPTION=pd.DataFrame()
VPO_FULL_PROGRAM_PATH=None
_vpo_names=selected_names('VPO_2_1_1')
if _vpo_names and CFG.MODE=='FULL':
    _vpo_path=Path(fetch_asset(VPO_211))
    _base=ROOT/'derived'; _base.mkdir(parents=True,exist_ok=True)
    _shards=_base/'vpo_year_shards'; _shards.mkdir(parents=True,exist_ok=True)
    VPO_FULL_PROGRAM_PATH=_base/'vpo_full_program_description.parquet'
    _esc=lambda value:"'"+str(value).replace("'","''")+"'"
    _where_names='indicator_name IN ('+','.join(_esc(x) for x in _vpo_names)+')'
    _src='read_parquet('+_esc(str(_vpo_path))+')'
    # Каждый год рассчитывается независимо; результаты объединяются без повторной агрегации.
    # Группы год×индикатор×специальность нельзя дробить между выходными файлами.
    _select='''SELECT TRY_CAST(year AS INTEGER) AS observation_year, specialty_code, indicator_name,
        object_level, education_form, education_level, admission_type,
        COUNT(*)::BIGINT AS observed_records,
        MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported_value,
        COUNT(DISTINCT object_name)::BIGINT AS distinct_geographic_names,
        FALSE AS totals_additive_confirmed'''
    _start_filter=f'({_where_names}) AND TRY_CAST(year AS INTEGER) IS NOT NULL'
    memory_point('before_vpo_full_aggregate')
    progress_event('PANELS','ВПО-1: проверка годов и полной численности',announce=True)
    with duck_connect() as _con:
        _con.execute("SET preserve_insertion_order=false")
        _years=[int(r[0]) for r in _con.execute(
            f'SELECT DISTINCT TRY_CAST(year AS INTEGER) AS y FROM {_src} WHERE {_start_filter} ORDER BY 1'
        ).fetchall()]
        _expected=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE {_start_filter}').fetchone()[0])
        _bad_years=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE ({_where_names}) AND TRY_CAST(year AS INTEGER) IS NULL').fetchone()[0])
    if _bad_years:
        ISSUES.append({'module':'ML-5.4','status':'VPO_UNDATED_EXCLUDED',
            'reason':f'ВПО-1: {_bad_years} строк без корректного года исключены из динамики',
            'required':'проверка первичных дат'})
    # Очистка только технических файлов этой конкретной агрегации.
    for _old in _shards.glob('vpo_*.parquet'): _old.unlink()
    _parts=[]

    def _vpo_copy(where,output):
        # Точный MEDIAN и COUNT(DISTINCT) без замены статистик приближёнными.
        _sql=f'''COPY ({_select} FROM {_src} WHERE ({where}) GROUP BY 1,2,3,4,5,6,7)
                    TO {_esc(str(output))} (FORMAT PARQUET, COMPRESSION ZSTD)'''
        output.unlink(missing_ok=True)
        try:
            with duck_connect() as _con:
                _con.execute("SET memory_limit='3GB'")
                _con.execute('SET threads=1')
                _con.execute('SET preserve_insertion_order=false')
                _con.execute("SET max_temp_directory_size='40GB'")
                _con.execute(_sql)
            return output
        except Exception:
            output.unlink(missing_ok=True)
            raise

    for _y in _years:
        _filter=f'{_start_filter} AND TRY_CAST(year AS INTEGER)={_y}'
        _outfile=_shards/f'vpo_year_{_y}.parquet'
        progress_event('PANELS',f'ВПО-1: полный расчёт за {_y} год',announce=True)
        try:
            _parts.append(_vpo_copy(_filter,_outfile))
        except duckdb.OutOfMemoryException:
            progress_event('PANELS',f'ВПО-1 {_y}: дробление по индикаторам и специальностям',announce=True)
            for _i,_indicator in enumerate(_vpo_names):
                _ifilter=f'{_filter} AND indicator_name={_esc(_indicator)}'
                _ipath=_shards/f'vpo_year_{_y}_indicator_{_i}.parquet'
                try:
                    _parts.append(_vpo_copy(_ifilter,_ipath))
                except duckdb.OutOfMemoryException:
                    # hash(specialty_code) фиксирован для каждой группы; 16 частей не пересекаются.
                    for _bucket in range(16):
                        _bfilter=f'''{_ifilter} AND hash(COALESCE(CAST(specialty_code AS VARCHAR),'__NULL__')) % 16={_bucket}'''
                        _bpath=_shards/f'vpo_year_{_y}_indicator_{_i}_part_{_bucket:02}.parquet'
                        _parts.append(_vpo_copy(_bfilter,_bpath))
    if not _parts:
        ISSUES.append({'module':'ML-5.4','status':'VPO_NO_MATCHES','reason':'Нет строк по утверждённым именам индикаторов'})
        VPO_FULL_PROGRAM_PATH=None
    else:
        _filelist='['+','.join(_esc(p) for p in _parts)+']'
        with duck_connect() as _con:
            _con.execute("SET preserve_insertion_order=false")
            _con.execute(f'''COPY (SELECT * FROM read_parquet({_filelist})) TO
                          {_esc(VPO_FULL_PROGRAM_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
            _actual=int(_con.execute('SELECT SUM(observed_records) FROM read_parquet(?)',
                                     [str(VPO_FULL_PROGRAM_PATH)]).fetchone()[0] or 0)
            VPO_FULL_PROGRAM_DESCRIPTION=_con.execute(
                'SELECT * FROM read_parquet(?) LIMIT 2000',[str(VPO_FULL_PROGRAM_PATH)]).df()
        if _actual!=_expected:
            fail('VPO_DATA_GATE',f'Потеря или дубль строк ВПО-1: источник {_expected}, агрегация {_actual}')
        _pf=pq.ParquetFile(VPO_FULL_PROGRAM_PATH)
        PANEL_FILES['vpo_program_description']=VPO_FULL_PROGRAM_PATH
        PANELS['vpo_program_description']=VPO_FULL_PROGRAM_DESCRIPTION
        TABLE_QA.append({'table':'vpo_program_description','rows':_pf.metadata.num_rows,
            'columns':len(_pf.schema_arrow.names),'duplicate_keys':None,'status':'PASS',
            'unique_keys':'','storage':'DUCKDB_YEAR_SHARDED_EXACT_AGGREGATE'})
        progress_event('PANELS',f'ВПО-1: {_expected:,} исходных строк, {_pf.metadata.num_rows:,} групп; контроль PASS',announce=True)
        del _pf
    _parts.clear()
    gc.collect()
# Полная агрегация выше; ниже только просмотр небольшой выборки, который не является полным источником.
vpo_admission=selective_records(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 _vpo_names) if CFG.MODE!='FULL' else (read_table(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 where='indicator_name IN ('+','.join("'"+x.replace("'","''")+"'" for x in _vpo_names)+')',limit=2000)
 if _vpo_names else pd.DataFrame())
if len(vpo_admission):
    vpo_admission['observation_year']=safe_year(vpo_admission.year)
    vpo_admission['value']=number(vpo_admission.indicator_value)
    vpo_admission['geography_grain']='REPORTED_OBJECT_LEVEL_'+vpo_admission.object_level.astype(str)

# Автоматический агрегат для прогнозирования строится только когда один уникальный показатель
# на ID/year, и источник действительно указывает итог, а не несопоставимые категории.
def unambiguous_org_admission(raw,sector):
    if raw.empty or not {'entity_id','observation_year','indicator_name','value'}.issubset(raw):return pd.DataFrame()
    x=raw.dropna(subset=['entity_id','observation_year','value']).copy()
    integerish=np.isclose(x.value.to_numpy(float),np.rint(x.value.to_numpy(float)),atol=1e-8,rtol=0)
    noncount=x.loc[~integerish].copy()
    if len(noncount):QUARANTINE.extend(noncount.head(200).assign(source=sector+'_NONINTEGER_ADMISSION').to_dict('records'))
    x=x.loc[x.value.ge(0)&integerish].copy()
    if x.empty:return pd.DataFrame()
    keys=['entity_id','observation_year']
    counts=x.groupby(keys,dropna=False).agg(n=('value','size'),n_indicators=('indicator_name','nunique')).reset_index()
    usable=counts[(counts.n==1)&(counts.n_indicators==1)][keys]
    good=x.merge(usable,on=keys,validate='many_to_one')
    if good.duplicated(keys).any():fail('SEMANTICS',f'{sector}: неоднозначный показатель приёма после фильтра')
    good=good.assign(sector=sector,target_definition='source_approved_count_of_admitted')
    bad=x.merge(usable.assign(used=True),on=keys,how='left')
    bad=bad[bad.used.isna()].drop(columns=['used'])
    if len(bad):QUARANTINE.extend(bad.head(100).assign(source=f'{sector}_MULTIPLE_ADMISSION_VALUES').to_dict('records'))
    return good[['sector','entity_id','observation_year','value','indicator_name','target_definition']].rename(columns={'value':'actual'})

org_parts=[unambiguous_org_admission(vo_admission,'VO'),unambiguous_org_admission(spo_admission,'SPO')]
ORG_YEAR=qa_table('edu_org_year',pd.concat(org_parts,ignore_index=True) if any(len(p) for p in org_parts)
                  else pd.DataFrame(columns=['sector','entity_id','observation_year','actual','indicator_name','target_definition']),
                  ['sector','entity_id','observation_year'],allow_empty=True)
qa_table('vo_admission_raw_candidates',vo_admission,allow_empty=True)
qa_table('spo_admission_raw_candidates',spo_admission,allow_empty=True)
qa_table('vpo_admission_audit',vpo_admission,allow_empty=True)

# Образовательные программы: источники E02 и ВПО-1 сохраняются без географической реконструкции OD×programme.
parts=[]
if 'vo_program_candidates' in globals() and len(vo_program_candidates):
    v=vo_program_candidates[['entity_id','observation_year','specialty_code','specialty_section','indicator_name','value']].copy()
    v['sector']='VO';v['geography_grain']='ORGANIZATION';parts.append(v)
if len(vpo_admission):
    v=vpo_admission[['observation_year','specialty_code','indicator_name','value','geography_grain']].copy()
    v['specialty_section']=pd.NA;v['entity_id']=pd.NA;v['sector']='VO_VPO';parts.append(v)
PROGRAM=qa_table('edu_program_year',pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(),allow_empty=True)

# Выпускники: FULL обрабатывает каждую строку E04 и E05 через DuckDB, в RAM — только итоги.
# Оригинальные E04/E05 остаются в Stage I (SHA фиксируются). Отчёт об исходных записях явный.
GRADUATE_GROUP_SUMMARY=pd.DataFrame();GRADUATE_FIELD_SUMMARY=pd.DataFrame()
GRAD_E04_VALID_ROWS=0;GRAD_E05_VALID_ROWS=0
GRAD_COLS=['object_level','object_name','oktmo','gender','education_level','year','university',
           'count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']
E05_COLS=['object_level','object_name','year','education_level','study_area','specialty_section',
          'specialty','specialty_code','count_graduate','percent_employed','average_salary_fact_med']
if CFG.MODE=='FULL':
    progress_event('PANELS','E04/E05: загрузка оригинальных файлов, агрегация на диске',announce=True)
    path_e04=fetch_asset(GRAD_ORG);path_e05=fetch_asset(GRAD_FIELD)
    with duck_connect() as con:
        progress_event('PANELS','E04: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_GROUP_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,gender,education_level,COUNT(*)::BIGINT AS n_source_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_observed_salary,
                 MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) AS median_employment_percentage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
          GROUP BY 1,2,3,4""",[str(path_e04)]).df()
        GRAD_E04_VALID_ROWS=int(GRADUATE_GROUP_SUMMARY.n_source_records.sum())
        progress_event('PANELS','E05: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_FIELD_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,education_level,CAST(specialty_code AS VARCHAR) AS specialty_code,
                 COUNT(*)::BIGINT AS observed_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_wage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
            AND specialty_code IS NOT NULL AND TRY_CAST(year AS INTEGER) IS NOT NULL
          GROUP BY 1,2,3,4""",[str(path_e05)]).df()
        GRAD_E05_VALID_ROWS=int(GRADUATE_FIELD_SUMMARY.observed_records.sum()) if len(GRADUATE_FIELD_SUMMARY) else 0
    progress_event('PANELS',f'E04/E05: обработано {GRAD_E04_VALID_ROWS:,}+{GRAD_E05_VALID_ROWS:,} допустимых строк',announce=True)
    GRADUATE_GROUP_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_GROUP_SUMMARY['employment_asof']='2024-12-31'
    GRADUATE_FIELD_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_FIELD_SUMMARY['employment_asof']='2024-12-31'
    qa_table('graduate_descriptive_statistics',GRADUATE_GROUP_SUMMARY,allow_empty=True)
    qa_table('graduate_specialty_description',GRADUATE_FIELD_SUMMARY,allow_empty=True)
    qa_table('graduate_full_scan_audit',pd.DataFrame([
       {'asset_id':GRAD_ORG['asset_id'],'source_rows':int(GRAD_ORG['rows_actual']),
        'valid_rows_in_statistics':GRAD_E04_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_ORG['sha256']},
       {'asset_id':GRAD_FIELD['asset_id'],'source_rows':int(GRAD_FIELD['rows_actual']),
        'valid_rows_in_statistics':GRAD_E05_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_FIELD['sha256']}]),allow_empty=True)
    # Доступен только явно обозначенный preview, не выдаваемый за полную базу.
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=1000)
    grad['graduation_year']=safe_year(grad['year'])
    grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes_preview',grad,allow_empty=True)
    del grad;gc.collect()
    note('MEMORY','E04/E05 FULL: все строки агрегированы в DuckDB; в Pandas только сводные таблицы и 1000 исходных строк')
else:
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    grad['graduation_year']=safe_year(grad['year']);grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes',grad,allow_empty=True)
    gradf=read_table(GRAD_FIELD,E05_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    gradf['graduation_year']=safe_year(gradf['year']);gradf['employment_asof']=pd.Timestamp('2024-12-31')
    gradf['specialty_code']=norm_code(gradf['specialty_code'])
    for c in ['count_graduate','percent_employed','average_salary_fact_med']:gradf[c]=number(gradf[c])
    gradf['source_asset_id']=GRAD_FIELD['asset_id']
    qa_table('graduate_specialty_outcomes',gradf,allow_empty=True)

# Освободить полный первичный E04/E05 сразу после расчёта сводных таблиц.
for _unused in ('gradf',):
    if _unused in globals():del globals()[_unused]
gc.collect();memory_point('panels_after_graduates')

# Муниципальная демография: точные исходные ОКТМО и исторические года; значения не размножаются по вузам.
mun=read_table(MUN,['oktmo','region','municipality','year','population','migration','migration_rate','births','deaths'])
mun['oktmo']=norm_code(mun.oktmo);mun['observation_year']=safe_year(mun.year)
for c in ['population','migration','migration_rate','births','deaths']:mun[c]=number(mun[c])
mun=mun.drop(columns=['year'])
mun['geography_grain']='MUNICIPALITY_FROM_D01'
mun=mun.dropna(subset=['oktmo','observation_year']).copy()
mun_unique=mun.groupby(['oktmo','observation_year']).size()
if (mun_unique>1).any():
    ISSUES.append({'module':'ML-5.7','status':'CONFLICTING_MUNICIPAL_KEYS',
      'reason':f'D01 дублирует ОКТМО×год: {int((mun_unique>1).sum())} ключей; исключены',
      'required':'reconciliation of historical municipality borders'})
    ok=mun_unique[mun_unique==1].index
    mun=mun.set_index(['oktmo','observation_year']).loc[ok].reset_index()
MUNI=qa_table('municipal_demography',mun,['oktmo','observation_year'],allow_empty=True)

# Только verified campus география; наличие без ОКТМО никогда не трактуется как отсутствие.
conf=vo.loc[vo.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS'),
  ['entity_id','observation_year','oktmo','parent_group_id']].dropna(subset=['oktmo']).copy()
conf=conf[nonmissing_code(conf.oktmo)]
conf=conf.drop_duplicates(['entity_id','observation_year'])
conf=conf.groupby(['oktmo','observation_year'],as_index=False).agg(
    confirmed_vo_sites=('entity_id','nunique'))
MUNI_EDU=MUNI.merge(conf,on=['oktmo','observation_year'],how='left',validate='one_to_one')
MUNI_EDU['education_coverage_known']=MUNI_EDU.confirmed_vo_sites.notna()
# Missing is unknown, NOT zero campuses.
MUNI_EDU['confirmed_spo_sites']=pd.Series(pd.NA,index=MUNI_EDU.index,dtype='Int64')
qa_table('municipality_education_context',MUNI_EDU,['oktmo','observation_year'],allow_empty=True)

qa_table('entity_match_quarantine',pd.DataFrame(QUARANTINE),allow_empty=True)
print('Panels: OD',len(OD),'inter',len(OD_INTER),'VO+SPO registry',len(REGISTRY),
      'org_year',len(ORG_YEAR),'program',len(PROGRAM),'graduates',len(GRADUATES),
      'municipal',len(MUNI_EDU))

# Финальная отметка этапа перенесена после всех его подмодулей


## 5А. Advanced Data Reconstruction · реконструкция первичных показателей и географии
Только наблюдаемые записи: E01 на общем покрытии, все кандидаты приёма E02, географические соответствия СПО с D04 и раздельные флаги пространственной неопределённости.

In [ ]:
# ML-05 V7.3.0 · Advanced Data Reconstruction: audited targets, balanced OD and spatial links.
# Reconstruction is evidence-gated. No inferred school admission, residence, campus or employment location is recorded as observed.
progress_event('PANELS','V7.3: первичная реконструкция по опубликованным наблюдениям',announce=True)
ADR_PREPROCESS_AUDIT=[]
ADR_ADMISSION_YEAR=pd.DataFrame()
ADR_ADMISSION_CANDIDATES=pd.DataFrame()
ADR_GEO_SPO=pd.DataFrame()
ADR_GEO_VERIFIED=pd.DataFrame()
ADR_OD_BALANCED=pd.DataFrame()
ADR_OD_PAIRWISE=pd.DataFrame()
ADR_SEMANTIC_PROVENANCE=[]

def adr_text(s):
    return (s.astype('string').fillna('').str.lower().str.replace('ё','е',regex=False)
            .str.replace(r'[^\w\s-]',' ',regex=True).str.replace(r'\s+',' ',regex=True).str.strip())

def adr_numbers(s):
    return pd.to_numeric(s.astype('string').str.replace(',','.',regex=False),errors='coerce')

def adr_log(stage,source,status,n_rows=0,rule='',reason=''):
    ADR_PREPROCESS_AUDIT.append({'procedure':stage,'source':source,'status':status,
        'n_observations':int(n_rows),'strict_rule':rule,'limitation':str(reason)[:300]})

# ML-5.1 / ML-5.3 / ML-5.6. All-year balanced panel on *actually observed* interregional E01 edges.
# This is an independent coverage-stability study, not synthetic completion of the OD matrix.
_adr_od=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna().copy()
_adr_od['observation_year']=pd.to_numeric(_adr_od['observation_year'],errors='coerce')
_adr_od=_adr_od.loc[_adr_od.observation_year.notna() & _adr_od.observed_value.ge(0)].copy()
_adr_od['observation_year']=_adr_od.observation_year.astype(int)
_adr_keys=['origin_region','destination_region']
_adr_years=sorted(_adr_od.observation_year.unique())
if _adr_od.duplicated(['observation_year']+_adr_keys).any():
    raise RuntimeError('ADR E01: duplicate observed OD edge-year')
if len(_adr_years)>=2:
    _adr_counts=_adr_od.groupby(_adr_keys).observation_year.nunique()
    _adr_shared=_adr_counts[_adr_counts==len(_adr_years)].index
    ADR_OD_BALANCED=_adr_od.set_index(_adr_keys).loc[_adr_shared].reset_index() if len(_adr_shared) else _adr_od.head(0).copy()
    _adr_bal=ADR_OD_BALANCED.groupby('observation_year',as_index=False).agg(
        n_common_edges=('observed_value','size'),total_on_common_edges=('observed_value','sum'),
        median_on_common_edges=('observed_value','median')) if len(ADR_OD_BALANCED) else pd.DataFrame()
    if len(_adr_bal):
        _adr_bal['all_years_shared']=len(_adr_years)
        _adr_bal['scope']='OBSERVED_BALANCED_EDGE_INTERSECTION_NOT_NATIONAL_TOTAL'
    qa_table('adr_od_balanced_edges_by_year',_adr_bal,allow_empty=True)
    if len(ADR_OD_BALANCED):
        qa_table('adr_od_balanced_edges',ADR_OD_BALANCED,['observation_year']+_adr_keys)
    _adr_pairs=[]
    for _left,_right in zip(_adr_years[:-1],_adr_years[1:]):
        if int(_right)!=int(_left)+1:continue
        _x=_adr_od.loc[_adr_od.observation_year.eq(_left),_adr_keys+['observed_value']].rename(columns={'observed_value':'prior'})
        _y=_adr_od.loc[_adr_od.observation_year.eq(_right),_adr_keys+['observed_value']].rename(columns={'observed_value':'current'})
        _both=_x.merge(_y,on=_adr_keys,validate='one_to_one')
        _adr_pairs.append({'prior_year':int(_left),'current_year':int(_right),
           'prior_observed_edges':len(_x),'current_observed_edges':len(_y),'common_edges':len(_both),
           'coverage_of_prior':len(_both)/len(_x) if len(_x) else np.nan,
           'coverage_of_current':len(_both)/len(_y) if len(_y) else np.nan,
           'MAE_persistence_common':float(np.abs(_both.current-_both.prior).mean()) if len(_both) else np.nan,
           'sum_prior_common':float(_both.prior.sum()),'sum_current_common':float(_both.current.sum()),
           'coverage_break_at_2023':int(_right)==2023,
           'claim_scope':'COMMON_REPORTED_EDGES_ONLY_WITHOUT_MISSING_ZERO_IMPUTATION'})
    ADR_OD_PAIRWISE=qa_table('adr_od_pairwise_comparable',pd.DataFrame(_adr_pairs),allow_empty=True)
    adr_log('E01_STABLE_OD',E01['asset_id'],'COMPLETE' if len(ADR_OD_BALANCED) else 'NO_BALANCED_EDGES',
        len(ADR_OD_BALANCED),'intersection of observed OD keys across all available years',
        'No permanent-residence movement or year-of-admission claim')
else:adr_log('E01_STABLE_OD',E01['asset_id'],'SKIPPED_NO_TIME_SUPPORT',len(_adr_od),'at least two years')

# ML-5.2: exhaustively inspect admission-labeled E02 rows at original organization+indicator+year grain.
# Single row may be admitted only when explicit source terminology, count unit and uniqueness all hold.
_adr_e02_out=ROOT/'derived'/'adr_e02_admission_candidates.parquet'
_adr_e02_tmp=ROOT/'derived'/'adr_e02_admission_candidates.incomplete.parquet'
_adr_e02_tmp.unlink(missing_ok=True)
# Stage I E02 read starts only AFTER SQL preflight succeeds.
_adr_e02_path=fetch_asset(E02)
progress_event('PANELS','E02: запись всех подходящих исходных строк на диск',announce=True)
with duck_connect() as _adr_con:
    _adr_con.execute(_adr_e02_copy_statement(_adr_e02_path,_adr_e02_tmp))
    _adr_total=int(_adr_con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(_adr_e02_tmp)]).fetchone()[0])
    ADR_ADMISSION_CANDIDATES=_adr_con.execute(_ADR_E02_SUMMARY_SQL,[str(_adr_e02_tmp)]).df()
    _adr_e02_tmp.replace(_adr_e02_out)
progress_event('PANELS',f'E02 проверено {_adr_total:,} кандидатных строк; индикаторов {len(ADR_ADMISSION_CANDIDATES)}',announce=True)
if len(ADR_ADMISSION_CANDIDATES):
    _adr_label=adr_text(ADR_ADMISSION_CANDIDATES.indicator_name)
    _adr_unit=adr_text(ADR_ADMISSION_CANDIDATES.indicator_unit)
    ADR_ADMISSION_CANDIDATES['explicit_count_noun']=_adr_label.str.contains(r'численност|количеств|число',regex=True)
    ADR_ADMISSION_CANDIDATES['explicit_admitted_people']=_adr_label.str.contains(r'принят|зачислен',regex=True)
    ADR_ADMISSION_CANDIDATES['ambiguous_meaning']=_adr_label.str.contains(r'заявлен|конкурс|контрольн|план|квот|приемн|доля|процент|на работу',regex=True)
    ADR_ADMISSION_CANDIDATES['unit_is_people']=_adr_unit.str.contains(r'чел|человек',regex=True)
    ADR_ADMISSION_CANDIDATES['semantic_candidate']=ADR_ADMISSION_CANDIDATES.explicit_count_noun & ADR_ADMISSION_CANDIDATES.explicit_admitted_people & ~ADR_ADMISSION_CANDIDATES.ambiguous_meaning & ADR_ADMISSION_CANDIDATES.unit_is_people
    ADR_ADMISSION_CANDIDATES['user_or_method_approval']=ADR_ADMISSION_CANDIDATES.indicator_name.isin(CFG.APPROVED_VO_ADMISSION_NAMES)
    ADR_ADMISSION_CANDIDATES['strict_source_label_supported']=ADR_ADMISSION_CANDIDATES.semantic_candidate & ADR_ADMISSION_CANDIDATES.invalid_numeric_rows.eq(0) & ADR_ADMISSION_CANDIDATES.invalid_count_rows.eq(0)
    ADR_ADMISSION_CANDIDATES['automatic_official_approval']=False  # Source label != externally verified cohort methodology.
qa_table('adr_organizational_admission_dictionary',ADR_ADMISSION_CANDIDATES,allow_empty=True)
adr_log('E02_ADMISSIONS',E02['asset_id'],'SCANNED_FULL',_adr_total,
    'count-unit, label, original reporting grain, single unambiguous organization-year observation',
    'Candidates without methodology approval are described, not asserted as admissions')

# Persist full candidates, but only an explicitly approved indicator with a unique reporting total can enter a new target.
_adr_approved=ADR_ADMISSION_CANDIDATES.loc[
    (ADR_ADMISSION_CANDIDATES['strict_source_label_supported'] & ADR_ADMISSION_CANDIDATES['user_or_method_approval']),
    'indicator_name'].dropna().astype(str).unique().tolist() if len(ADR_ADMISSION_CANDIDATES) else []
if _adr_approved:
    with duck_connect() as _adr_con:
        _adr_con.execute("SET memory_limit='1536MB'")
        _adr_literal_names=','.join("'"+x.replace("'","''")+"'" for x in _adr_approved)
        _adr_sel=_adr_con.execute(_ADR_E02_TOTAL_SQL.replace('/*VALID_NAMES*/',_adr_literal_names),
            [str(_adr_e02_out)]).df()
    _adr_sel=_adr_sel.loc[_adr_sel.organization_id.notna()&_adr_sel.observation_year.notna()&
        _adr_sel.n_reporting_cells.eq(1)&_adr_sel.area_total.fillna(False)&_adr_sel.specialty_total.fillna(False)&
        _adr_sel.min_count.notna()&_adr_sel.min_count.ge(0)&
        _adr_sel.min_count.eq(_adr_sel.max_count)&_adr_sel.min_count.mod(1).eq(0)].copy()
    _adr_sel=_adr_sel.drop_duplicates(['organization_id','observation_year'],keep=False)
    _adr_sel['entity_id']='VO|'+norm_code(_adr_sel.organization_id)
    _adr_sel['actual']=_adr_sel.min_count
    _adr_sel['target_definition']='SOURCE_LABEL_REPORTED_ADMISSION_COUNT_NOT_OFFICIAL_COHORT'
    _adr_sel['methodology_approval']=_adr_sel.indicator_name.isin(CFG.APPROVED_VO_ADMISSION_NAMES)
    _adr_sel['scientific_scope']='EXACT_E02_SOURCE_LABEL_AND_UNIQUE_ORG_YEAR_TOTAL'
    ADR_ADMISSION_YEAR=qa_table('adr_admission_organization_year',_adr_sel,
          ['entity_id','observation_year'],allow_empty=True)
else:
    adr_log('E02_ORG_ADMISSIONS',E02['asset_id'],'SKIPPED_NO_SEMANTICALLY_UNAMBIGUOUS_TARGET',0,
      'explicit source reported admitted count, unit people, unique organization-year TOTAL row',
      'No organization-year admission target invented from section/area counts')

# ML-5.7: relate *reported SPO coordinates* to D04 settlement identifiers, with an ambiguity flag.
# This never silently converts a spatial proxy into an officially verified campus location.
try:
    from sklearn.neighbors import BallTree
    _adr_sp=sporeg[['entity_id','observation_year','region_name','city','lat','lon']].copy()
    _adr_sp['lat_num']=adr_numbers(_adr_sp['lat']); _adr_sp['lon_num']=adr_numbers(_adr_sp['lon'])
    _adr_sp=_adr_sp.loc[_adr_sp.lat_num.between(40,85)&_adr_sp.lon_num.between(19,190)].copy()
    _adr_sp['region_key']=adr_text(_adr_sp.region_name)
    _adr_sp['city_key']=adr_text(_adr_sp.city)
    _adr_sp=_adr_sp.loc[_adr_sp.region_key.ne('')&_adr_sp.city_key.ne('')].copy()
    _adr_grid=fetch_asset(SPATIAL)
    with duck_connect() as _adr_con:
        _adr_d=_adr_con.execute('''SELECT CAST(source_oktmo AS VARCHAR) source_oktmo,
             CAST(region_name AS VARCHAR) region_name, CAST(municipality_name AS VARCHAR) municipality_name,
             CAST(settlement_name AS VARCHAR) settlement_name,
             TRY_CAST(latitude AS DOUBLE) latitude,TRY_CAST(longitude AS DOUBLE) longitude,
             CAST(coordinates_valid AS VARCHAR) coordinates_valid
             FROM read_parquet(?) WHERE TRY_CAST(latitude AS DOUBLE) BETWEEN 40 AND 85
               AND TRY_CAST(longitude AS DOUBLE) BETWEEN 19 AND 190''',[str(_adr_grid)]).df()
    _adr_d['region_key']=adr_text(_adr_d.region_name)
    _adr_d['city_key']=adr_text(_adr_d.settlement_name)
    _adr_d['mun_key']=adr_text(_adr_d.municipality_name)
    _adr_d['source_oktmo']=norm_code(_adr_d.source_oktmo)
    _adr_d=_adr_d.loc[nonmissing_code(_adr_d.source_oktmo)&_adr_d.region_key.ne('')].reset_index(drop=True)
    _adr_match=[]
    for _reg,_rows in _adr_sp.groupby('region_key',sort=False):
        _places=_adr_d.loc[_adr_d.region_key.eq(_reg)].copy()
        if len(_places)<2:continue
        _tree=BallTree(np.radians(_places[['latitude','longitude']].to_numpy(float)),metric='haversine',leaf_size=40)
        _dist,_ix=_tree.query(np.radians(_rows[['lat_num','lon_num']].to_numpy(float)),k=2)
        _places=_places.reset_index(drop=True)
        for _j,_rr in enumerate(_rows.itertuples(index=False)):
            _nearest=_places.iloc[int(_ix[_j,0])]
            _d1=float(_dist[_j,0]*6371.0088); _d2=float(_dist[_j,1]*6371.0088)
            _city=str(_rr.city_key)
            _city_match=(_city==str(_nearest.city_key) or _city==str(_nearest.mun_key))
            _high=bool(_city_match and _d1<=2.0 and _d2-_d1>=0.50)
            _adr_match.append({'entity_id':_rr.entity_id,'observation_year':_rr.observation_year,
                'reported_city':_rr.city,'reported_region':_rr.region_name,
                'candidate_municipal_oktmo':_nearest.source_oktmo,
                'nearest_centroid_km':_d1,'second_nearest_centroid_km':_d2,
                'city_name_consistent':bool(_city_match),'spatially_unambiguous':bool(_high),
                'source_id_spo':SPO_ORG['asset_id'],'source_id_geography':SPATIAL['asset_id'],
                'identity_status':'PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION' if _high else 'UNRESOLVED'})
    ADR_GEO_SPO=pd.DataFrame(_adr_match)
    if len(ADR_GEO_SPO):
        # Inconsistent historical assignments for the same institution-year are quarantined.
        _dup=ADR_GEO_SPO.duplicated(['entity_id','observation_year'],keep=False)
        ADR_GEO_SPO.loc[_dup,'spatially_unambiguous']=False
        ADR_GEO_SPO.loc[_dup,'identity_status']='CONFLICTING_HISTORICAL_LOCATION'
        ADR_GEO_VERIFIED=ADR_GEO_SPO.loc[ADR_GEO_SPO.spatially_unambiguous].drop_duplicates(['entity_id','observation_year']).copy()
    qa_table('adr_spo_geolocation_candidates',ADR_GEO_SPO,allow_empty=True)
    qa_table('adr_spo_geolocation_high_confidence',ADR_GEO_VERIFIED,allow_empty=True)
    adr_log('SPO_GEOCODE',SPATIAL['asset_id'],'PROXIMITY_RECONSTRUCTION',len(ADR_GEO_VERIFIED),
        'same region, city/settlement match, <2km, nearest separation >=0.5km',
        'Geocoded municipality is a reconstructed spatial proxy, not officially confirmed campus placement')
except (KeyError,ValueError,TypeError) as _geo_error:
    adr_log('SPO_GEOCODE',SPATIAL['asset_id'],'SKIPPED_INCOMPATIBLE_SPATIAL_FIELDS',0,'valid coordinates/municipal code',str(_geo_error))

# V7.3 · ещё до муниципального эксперимента: многолетний конфликт прокси-ОКТМО.
if len(ADR_GEO_VERIFIED):
    _v72_geo_conflict=(ADR_GEO_VERIFIED.groupby('entity_id').candidate_municipal_oktmo.nunique(dropna=True))
    _v72_geo_conflict=set(_v72_geo_conflict[_v72_geo_conflict>1].index)
    if _v72_geo_conflict:
        _v72_unstable=ADR_GEO_VERIFIED.loc[ADR_GEO_VERIFIED.entity_id.isin(_v72_geo_conflict)].copy()
        QUARANTINE.extend(_v72_unstable.head(100).assign(source='V72_UNVERIFIED_HISTORICAL_OKTMO_CHANGE').to_dict('records'))
        ADR_GEO_VERIFIED=ADR_GEO_VERIFIED.loc[~ADR_GEO_VERIFIED.entity_id.isin(_v72_geo_conflict)].copy()
        adr_log('SPO_GEOCODE_STABILITY',SPATIAL['asset_id'],'EXCLUDED_UNVERIFIED_HISTORICAL_CHANGE',
            len(_v72_unstable),'same candidate municipality for all reported years',
            'Historical move or inconsistent geocoding cannot be verified from proximity alone')

qa_table('adr_preprocessing_provenance',pd.DataFrame(ADR_PREPROCESS_AUDIT),allow_empty=True)
progress_event('PANELS',f'V7.3: common OD={len(ADR_OD_BALANCED):,}; approved admissions={len(ADR_ADMISSION_YEAR):,}; SPO geo proxies={len(ADR_GEO_VERIFIED):,}',announce=True)

mark_stage('PANELS')


## 6. Временные признаки, экономика и транспорт
Все признаки имеют исторический период и осторожный лаг доступности. Дистанции между городами остаются прокси, не истинным расстоянием регионов.

In [ ]:
memory_point('before_features')
progress_event('FEATURES','География, экономический контекст и исторические признаки')
# Экономический и пространственный контекст: только фактические данные, с пометкой уровня агрегации.
NAME_MAP=pd.concat([
    OD_INTER[['origin_region','origin_region_name']].rename(columns={'origin_region':'region_code','origin_region_name':'region_name'}),
    OD_INTER[['destination_region','destination_region_name']].rename(columns={'destination_region':'region_code','destination_region_name':'region_name'})
],ignore_index=True).drop_duplicates()
NAME_MAP['region_name_norm']=NAME_MAP.region_name.astype('string').str.lower().str.replace('ё','е').str.strip()
conflicts=NAME_MAP.groupby('region_code',dropna=False).region_name_norm.nunique(dropna=True)
if (conflicts>1).any():
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].head(150)
                      .assign(source='REGION_CODE_NAME_CONFLICT').to_dict('records'))
    NAME_MAP=NAME_MAP[~NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].copy()
NAME_MAP=NAME_MAP.drop_duplicates(subset=['region_code'])
if NAME_MAP.region_code.duplicated().any():fail('GEOGRAPHY','Конфликтная карта кодов субъектов')
# Название региона должно задавать единственный код. Иначе исключаем сомнительные соединения.
_name_counts=NAME_MAP.groupby('region_name_norm',dropna=False).region_code.nunique()
_ambiguous=set(_name_counts[_name_counts>1].index)
if _ambiguous:
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_name_norm.isin(_ambiguous)].head(100).assign(source='AMBIGUOUS_REGION_NAME').to_dict('records'))
    ISSUES.append({'module':'GEOGRAPHY','status':'AMBIGUOUS_REGION_NAME','reason':f'{len(_ambiguous)} названий соответствуют нескольким кодам','required':'territorial crosswalk'})
NAME_LOOKUP=NAME_MAP[NAME_MAP.region_name_norm.notna() & ~NAME_MAP.region_name_norm.isin(_ambiguous)].copy()
REGION_CONTEXT=[];DISTANCE=pd.DataFrame()

try:
    raw=read_table(TRANSPORT,['region_from','region_to','distance_km','transport_type'],
                   limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)
    raw['origin_region_name_norm']=raw.region_from.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['destination_region_name_norm']=raw.region_to.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['distance_proxy_km']=number(raw.distance_km)
    raw=raw[(raw.distance_proxy_km>0)&raw.distance_proxy_km.notna()]
    d=raw.groupby(['origin_region_name_norm','destination_region_name_norm'],as_index=False).agg(
        distance_proxy_km=('distance_proxy_km','median'),distance_measurements=('distance_proxy_km','size'))
    o=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'origin_region',
        'region_name_norm':'origin_region_name_norm'})
    t=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'destination_region',
        'region_name_norm':'destination_region_name_norm'})
    DISTANCE=d.merge(o,on='origin_region_name_norm',validate='many_to_one').merge(
        t,on='destination_region_name_norm',validate='many_to_one')[['origin_region','destination_region',
        'distance_proxy_km','distance_measurements']]
    DISTANCE=DISTANCE.drop_duplicates(['origin_region','destination_region'])
    note('GEOGRAPHY','Расстояние — медиана транспортных пар городов; НЕ истинное расстояние субъект-субъект')
except Exception as ex:
    note('GEOGRAPHY','Транспортный контекст недоступен: '+type(ex).__name__,'WARN')
qa_table('region_distance_proxy',DISTANCE,allow_empty=True)
# Карантин дополняется при географическом сопоставлении, исходная панель обновляется после этапа.
PANELS['entity_match_quarantine']=pd.DataFrame(QUARANTINE)

# Региональный контекст собирается только из источников с явно указанным субъектным grain.
D05_LABOR=source_asset(source_id='D05')
for a in D05_LABOR:
    if 'object_level' not in a.get('columns_manifest',[]) or 'indicator_name' not in a.get('columns_manifest',[]):continue
    try:
        path=fetch_asset(a)
        with duck_connect() as con:
            sql='''SELECT object_name, object_level, year, indicator_name, indicator_unit,
                         TRY_CAST(indicator_value AS DOUBLE) AS indicator_value,
                         COUNT(*) as duplicates
                    FROM read_parquet(?)
                    WHERE regexp_matches(lower(indicator_name), 'безработ|заработ|доход|зарплат|численность рабочей силы')
                    GROUP BY ALL'''
            v=con.execute(sql,[str(path)]).df()
        v['source_asset_id']=a['asset_id']
        v['observation_year']=safe_year(v.year)
        # Нельзя считать объект регионом только по совпадению названия; требуется документированный уровень.
        v=v[v.object_level.astype(str).str.contains(r'регион|субъект|region',case=False,regex=True,na=False)].copy()
        if len(v):REGION_CONTEXT.append(v)
    except Exception as ex:
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_SOURCE_GATED',
            'reason':a['asset_id']+': '+type(ex).__name__,'required':'correct unit and regional grain'})
ECON_RAW=pd.concat(REGION_CONTEXT,ignore_index=True) if REGION_CONTEXT else pd.DataFrame()
qa_table('economic_indicator_candidates',ECON_RAW,allow_empty=True)

# Точное сопоставление названий; отсутствие точной пары = NA, не произвольное присвоение.
ECON_PANEL=pd.DataFrame()
if len(ECON_RAW):
    e=ECON_RAW.copy()
    e['region_name_norm']=e.object_name.astype('string').str.lower().str.replace('ё','е').str.strip()
    e=e.merge(NAME_LOOKUP[['region_code','region_name_norm']],on='region_name_norm',how='inner',
              validate='many_to_one')
    e=e.dropna(subset=['indicator_value','observation_year'])
    e['indicator_key']=e.indicator_name.astype(str)+'|'+e.indicator_unit.astype(str)
    # Только единичное значение: похожие категории и единицы не складываются.
    natural=['region_code','observation_year','indicator_key']
    multiplicity=e.groupby(natural,dropna=False)['indicator_value'].transform('size')
    if 'duplicates' in e:
        multiplicity=multiplicity.where(e['duplicates'].fillna(1).le(1),other=2)
    rejected=e.loc[multiplicity.ne(1)].copy()
    if len(rejected):
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_GRAIN_CONFLICT',
           'reason':f'Исключено {len(rejected)} неоднозначных строк регион×год×индикатор',
           'required':'verified regional stratum and indicator definition'})
        QUARANTINE.extend(rejected.head(200).assign(source='ECON_CONFLICTING_REGION_YEAR_INDICATOR').to_dict('records'))
    e=e.loc[multiplicity.eq(1)].copy()
    if len(e) and e.duplicated(natural).any():fail('ECONOMY','Конфликт натурального ключа регион×год×индикатор')
    if not e.empty:
        top=e.groupby('indicator_key').region_code.nunique().sort_values(ascending=False).head(3).index
        e=e[e.indicator_key.isin(top)]
        ECON_PANEL=e[['region_code','observation_year','indicator_key','indicator_value','source_asset_id']]
qa_table('regional_economy',ECON_PANEL,allow_empty=True)

# Подготовка лага/скользящих сумм без доступа к целевому году.
def historical_pair_panel(observed,entity_fields,target_col,feature_lag=None,horizon=None):
    fl=CFG.FEATURE_PUBLICATION_LAG_YEARS if feature_lag is None else feature_lag
    h=CFG.HORIZON if horizon is None else horizon
    if observed.empty:return pd.DataFrame()
    o=observed.copy()
    if 'observation_year' not in o:fail('TIME','Нет observation_year')
    o['observation_year']=safe_year(o.observation_year)
    if o.duplicated(entity_fields+['observation_year']).any():fail('TIME','Дубли натурального временного ключа')
    out=o.copy()
    out['target_year']=out.observation_year.astype('Int64')
    out['origin_year']=out.target_year-h
    out['information_cutoff_year']=out.origin_year
    out['historical_year']=out.origin_year-fl
    out['historical_year_2']=out.historical_year-1
    out=out.rename(columns={target_col:'actual'})
    hist=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year',target_col:'prior_value'})
    hist2=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year_2',target_col:'prior2_value'})
    out=out.merge(hist,on=entity_fields+['historical_year'],how='left',validate='many_to_one')
    out=out.merge(hist2,on=entity_fields+['historical_year_2'],how='left',validate='many_to_one')
    out=out[out.prior_value.notna()&out.actual.notna()].copy()
    out['feature_available_year']=out.historical_year+fl
    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    out['horizon']=h
    out['source_vintage']=INDEX['created_at']
    out['vintage_warning']='RETROSPECTIVE_ONLY_HISTORICAL_VINTAGES_NOT_ARCHIVED'
    out['split_id']='UNASSIGNED'
    if not (out.feature_available_year<=out.information_cutoff_year).all():
        fail('TIME','Будущие признаки попали в historical panel')
    return out

OD_MODEL=historical_pair_panel(
    OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna(subset=['observed_value']).copy(),
    ['origin_region','destination_region'],'observed_value')
if not OD_MODEL.empty:
    OD_MODEL['entity_id']=OD_MODEL.origin_region.astype(str)+'|'+OD_MODEL.destination_region.astype(str)
    OD_MODEL['sector']='SPO'
    # Маргинали/число партнёров строго за доступный прошлый год.
    counts=OD_INTER.groupby(['observation_year','origin_region']).agg(
        origin_mass=('observed_value','sum'),origin_destinations=('destination_region','nunique')).reset_index()
    dests=OD_INTER.groupby(['observation_year','destination_region']).agg(
        destination_mass=('observed_value','sum'),destination_origins=('origin_region','nunique')).reset_index()
    outm=counts.rename(columns={'observation_year':'historical_year'})
    inm=dests.rename(columns={'observation_year':'historical_year'})
    OD_MODEL=OD_MODEL.merge(outm,on=['historical_year','origin_region'],validate='many_to_one',how='left')
    OD_MODEL=OD_MODEL.merge(inm,on=['historical_year','destination_region'],validate='many_to_one',how='left')
    if len(DISTANCE):OD_MODEL=OD_MODEL.merge(DISTANCE[['origin_region','destination_region','distance_proxy_km']],
                                          on=['origin_region','destination_region'],validate='many_to_one',how='left')
    # Лагирование экономических факторов проверяется датой доступности. Никаких future margins.
    if len(ECON_PANEL):
        e=ECON_PANEL.pivot(index=['region_code','observation_year'],columns='indicator_key',
                           values='indicator_value').reset_index()
        keys=[k for k in e.columns if k not in ('region_code','observation_year')]
        for direction in ('origin','destination'):
            f=e.rename(columns={'region_code':f'{direction}_region','observation_year':'historical_year',
                                **{k:f'{direction}_econ_{i}' for i,k in enumerate(keys)}})
            OD_MODEL=OD_MODEL.merge(f,on=[f'{direction}_region','historical_year'],
                                    how='left',validate='many_to_one')
    OD_MODEL['target_id']=OD_SEMANTIC['label']
    flagged=coverage_df.loc[coverage_df.potential_break.eq(True),'year'] if 'potential_break' in coverage_df else pd.Series(dtype=int)
    OD_MODEL['coverage_break_target_year']=OD_MODEL.target_year.isin(flagged).astype(bool)
    OD_MODEL['coverage_break_historical_year']=OD_MODEL.historical_year.isin(flagged).astype(bool)
qa_table('forecast_od_candidates',OD_MODEL,allow_empty=True)

# Отток: СУММА известных межрегиональных образовательных маршрутов, не migration_residence.
OUTFLOW=OD_INTER.dropna(subset=['observed_value']).groupby(['observation_year','origin_region'],as_index=False).agg(
    observed_interregional_routes=('observed_value','sum'),observed_destinations=('destination_region','nunique'))
OUTFLOW['sector']='SPO';OUTFLOW['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('outflow_observed',OUTFLOW,['observation_year','origin_region'],allow_empty=True)
OUTFLOW_MODEL=historical_pair_panel(OUTFLOW[['observation_year','origin_region','observed_interregional_routes']],
                                  ['origin_region'],'observed_interregional_routes')
if len(OUTFLOW_MODEL):
    OUTFLOW_MODEL['entity_id']=OUTFLOW_MODEL.origin_region.astype(str)
    OUTFLOW_MODEL['sector']='SPO';OUTFLOW_MODEL['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('forecast_outflow_candidates',OUTFLOW_MODEL,allow_empty=True)

ORG_MODEL=pd.DataFrame()
if len(ORG_YEAR):
    for sector,g in ORG_YEAR.groupby('sector'):
        m=historical_pair_panel(g[['entity_id','observation_year','actual']],['entity_id'],'actual')
        if len(m):
            m['sector']=sector
            m['target_id']='verified_organizational_admission_count'
            if ORG_MODEL.empty:ORG_MODEL=m
            else:ORG_MODEL=pd.concat([ORG_MODEL,m],ignore_index=True)
    if len(ORG_MODEL):
        rg=REGISTRY[['sector','entity_id','observation_year','parent_group_id','region']].rename(
            columns={'observation_year':'target_year'})
        ORG_MODEL=ORG_MODEL.merge(rg,on=['sector','entity_id','target_year'],how='left',validate='many_to_one')
qa_table('forecast_org_candidates',ORG_MODEL,allow_empty=True)

# Профиль программ по одному источнику; несопоставимые классификаторы не агрегируем.
PROGRAM_PROFILE=pd.DataFrame()
if len(PROGRAM) and {'specialty_code','value'}.issubset(PROGRAM):
    pp=PROGRAM[(PROGRAM.sector=='VO')&PROGRAM.specialty_code.notna()].copy()
    pp=pp.dropna(subset=['value','observation_year'])
    pp['specialty_code']=norm_code(pp.specialty_code)
    pp=pp[pp.value.ge(0)]
    # Не складывать разные индикаторы в одном направлении/году: оставляем строгую согласованную совокупность.
    natural=['entity_id','observation_year','specialty_code','indicator_name']
    multiplicity=pp.groupby(natural,dropna=False).size().rename('records').reset_index()
    ok=multiplicity.loc[multiplicity.records.eq(1),natural]
    excluded=int(len(pp)-len(ok))
    if excluded:ISSUES.append({'module':'ML-5.4','status':'AMBIGUOUS_PROGRAM_GRAIN',
      'reason':f'{excluded} строк нельзя однозначно агрегировать по программе; карантин без произвольного сложения',
      'required':'programme/level/form/classifier disaggregation'})
    pp=pp.merge(ok,on=natural,how='inner',validate='one_to_one')
    # Суммы на одном и том же идентификаторе индикатора допустимы лишь как суммы строк с непересекающимися организациями.
    grp=pp.groupby(['observation_year','specialty_code','indicator_name'],as_index=False).agg(
        observed_value=('value','sum'),organizations=('entity_id','nunique'))
    if not grp.empty:
        grp['sector']='VO';grp['source_grain_gate']='UNIQUE_ORG_YEAR_SPECIALTY_INDICATOR_NOT_POPULATION_TOTAL';PROGRAM_PROFILE=grp
qa_table('program_profile',PROGRAM_PROFILE,allow_empty=True)
# ВПО-1 может содержать реальную специализацию при отсутствии утверждённой организации-приёма: не отключать описание.
VPO_PROGRAM_DESCRIPTION=pd.DataFrame()
if len(vpo_admission):
    z=vpo_admission.dropna(subset=['observation_year','specialty_code','value']).copy()
    z['specialty_code']=norm_code(z.specialty_code)
    if len(z):
        VPO_PROGRAM_DESCRIPTION=z.groupby(['observation_year','specialty_code','indicator_name','object_level',
          'education_form','education_level','admission_type'],dropna=False,as_index=False).agg(
          observed_records=('value','size'),median_reported_value=('value','median'),
          distinct_geographic_names=('object_name','nunique'))
        VPO_PROGRAM_DESCRIPTION['totals_additive_confirmed']=False
if VPO_FULL_PROGRAM_PATH is None:
    qa_table('vpo_program_description',VPO_PROGRAM_DESCRIPTION,allow_empty=True)
else:
    VPO_PROGRAM_DESCRIPTION=VPO_FULL_PROGRAM_DESCRIPTION
    note('VPO','ВПО-1: полная агрегированная панель находится в evidence-Parquet; Pandas содержит только preview')

progress_event('FEATURES',f'Готовы исторические панели: OD {len(OD_MODEL):,}, отток {len(OUTFLOW_MODEL):,}',announce=True)
print('Feature coverage:',len(OD_MODEL),'OD historical rows,',len(OUTFLOW_MODEL),'outflow,',
      len(ORG_MODEL),'organization; economic indicators',len(ECON_PANEL),
      'distance pairs',len(DISTANCE))

memory_point('after_features')
mark_stage('FEATURES')


## 6А. Исследовательское использование всех групп источников

Этот блок выводит дополнительные контекстные показатели без смешения пространственных уровней: экономические и образовательные индикаторы БДМО, муниципальную историю ОКТМО, географические справочники, все E04/E05 и все разделы ВПО. Крупные группировки записываются непосредственно в Parquet. Математически корректное сопоставление сохраняет исходный `indicator_code` и ограничения на сложение пересекающихся разрезов.

In [ ]:
# FULL52 second pass: thematic *derived* panels at the original statistical grain.
# Each of 52 datasets has already passed a physical SHA + row-coverage audit above.
memory_point('before_cross_source');progress_event('CROSS_SOURCE','БДМО, ВПО, E04/E05, СПО и история ОКТМО',announce=True)
CONTEXT_PARQUET_FILES=[];D02_PROFILE=[];VPO_ALL_PROFILE=[];SPO_YEAR_CHECK=[];FAMILY_E04E05=[];SPATIAL_CROSS=[]
BDMO_EDU_PROXY=pd.DataFrame();BDMO_EDU_PROXY_CANDIDATES=pd.DataFrame()
BDMO_SPATIAL=pd.DataFrame();D04_SPATIAL=pd.DataFrame();HISTORY_MUNICIPAL=pd.DataFrame();BDMO_PROXY_FULL_VALID=False
GRAD_ALL_ASSETS=[]

def _sql_literal(v):return "'"+str(v).replace("'","''")+"'"

def thematic_sql_disk(asset,query,out_stem,preview=1500,mem='2GB'):
    # SELECT query may contain __INPUT__ placeholder; never use SELECT * to pandas.
    p=fetch_asset(asset);out=ROOT/'derived'/'full52_context'/(out_stem+'.parquet')
    out.parent.mkdir(parents=True,exist_ok=True)
    sql=query.replace('__INPUT__',_sql_literal(p))
    tmp=out.with_suffix('.partial.parquet');tmp.unlink(missing_ok=True)
    with duckdb.connect(database=':memory:') as con:
        con.execute(f"SET memory_limit='{mem}'")
        con.execute('SET threads=1');con.execute('SET preserve_insertion_order=false')
        con.execute('SET temp_directory='+_sql_literal(DB_TMP))
        con.execute('COPY ('+sql+') TO '+_sql_literal(tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
        data=con.execute('SELECT * FROM read_parquet(?) LIMIT '+str(int(preview)),[str(tmp)]).df()
    tmp.replace(out);pf=pq.ParquetFile(out)
    CONTEXT_PARQUET_FILES.append(out)
    return out,data,int(pf.metadata.num_rows)

# A. Every economic/BDMO source: content-level year/indicator/region coverage (but no false additive sums).
# 5-NDFL/7-NDFL cut across multiple tax forms; count/coverage ≠ observed number of residents.
for i,a in enumerate(source_asset(segment='02') if CFG.MODE=='FULL' else source_asset(prefix='02::labor'),1):
    aid=a['asset_id'];cols=set(a['columns_manifest']);progress_event('CROSS_SOURCE',f'D02 [{i}/11]: {aid}',i,11,announce=True)
    ind=next((c for c in ('indicator_code','indicator_name') if c in cols),None)
    yr='year' if 'year' in cols else None
    geo=next((c for c in ('region_name','report_region_name','object_name') if c in cols),None)
    if not ind or not yr:continue
    select_geo='CAST('+_sql_literal('UNSPECIFIED_LEVEL')+' AS VARCHAR)' if not geo else 'CAST("'+geo+'" AS VARCHAR)'
    level_where="AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект|region')" if geo=='object_name' and 'object_level' in cols else ''
    query=f'''SELECT CAST("{yr}" AS VARCHAR) observed_year,
      {select_geo} AS reported_geography,CAST("{ind}" AS VARCHAR) indicator_key,
      COUNT(*)::BIGINT AS reported_records,
      COUNT(DISTINCT CAST("{geo}" AS VARCHAR)) AS n_locations
      FROM read_parquet(__INPUT__) WHERE "{ind}" IS NOT NULL {level_where}
      GROUP BY 1,2,3'''
    if geo is None:query=query.replace('COUNT(DISTINCT CAST("None" AS VARCHAR))','COUNT(*)::BIGINT')
    try:
        p,preview,rows=thematic_sql_disk(a,query,'d02_'+str(i).zfill(2),preview=800,mem='2GB')
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':rows,
            'profile_parquet':p.name,'status':'GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT'})
    except Exception as ex:
        ISSUES.append({'module':'D02','status':'SOURCE_PROFILE_FAILURE','reason':aid+' '+type(ex).__name__+': '+str(ex)[:130],
                       'required':'repair dedicated source query; full SHA inventory still valid'})
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':None,
                            'status':'GROUP_PROFILE_FAILED'})
qa_table('d02_indicator_coverage',pd.DataFrame(D02_PROFILE),['asset_id'])

# B. All 11 E02/VPO tables: count by publication year and exact indicator name, each input processed independently.
for i,a in enumerate(source_asset(segment='04') if CFG.MODE=='FULL' else source_asset(prefix='04::E02'),1):
    cols=set(a.get('columns_manifest') or [])
    if 'indicator_name' not in cols:continue
    query='''SELECT TRY_CAST(year AS INTEGER) observation_year,CAST(indicator_name AS VARCHAR) indicator_name,
      COUNT(*)::BIGINT observed_rows
      FROM read_parquet(__INPUT__) WHERE indicator_name IS NOT NULL
      GROUP BY 1,2'''
    try:
        p,preview,rows=thematic_sql_disk(a,query,'vpo_'+str(i).zfill(2),preview=500,mem='2GB')
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'year_indicator_groups':rows,'parquet':p.name,'status':'PASS'})
    except Exception as ex:
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'status':'PROFILE_FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'VPO-1','status':'SOURCE_PROFILE_FAILURE','reason':a['asset_id']+' '+type(ex).__name__,
                       'required':'retry annual indicator group scan'})
qa_table('vpo_all_sections_audit',pd.DataFrame(VPO_ALL_PROFILE),['asset_id'])

# C. SPO all-years and independent yearly shards; diagnose overlaps without summing them.
spo_assets=source_asset(segment='03',role='organization_indicators')
annual=pd.DataFrame(FULL52_YEAR_STATS)
master_id=SPO_MASTER['asset_id'] if SPO_MASTER else None
if len(annual) and master_id:
    ref=annual.loc[annual.asset_id.eq(master_id)].set_index('year_label')['rows']
    for a in spo_assets:
        cur=annual[annual.asset_id.eq(a['asset_id'])]
        for year,sub in cur.groupby('year_label'):
            n=int(sub['rows'].sum());v=ref.get(year,pd.NA)
            SPO_YEAR_CHECK.append({'asset_id':a['asset_id'],'year':year,'rows_in_representation':n,
               'rows_in_master':v,'same_count':bool(n==v) if pd.notna(v) else None,
               'treatment':'NOT_ADDITIVE_WITH_MASTER'})
qa_table('spo_annual_vs_master',pd.DataFrame(SPO_YEAR_CHECK),allow_empty=True)

# D. All four E04/E05 datasets: by actual year, educational level, gender or area (never sum intersecting cross-sections).
for i,a in enumerate(source_asset(segment='05') if CFG.MODE=='FULL' else source_asset(segment='05')[:1],1):
    cols=set(a.get('columns_manifest') or [])
    group=['year','education_level']+[q for q in ('gender','study_area') if q in cols]
    keys=','.join('"'+k+'"' for k in group)
    q=f'''SELECT {keys},COUNT(*)::BIGINT source_records,
          MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) median_reported_graduates,
          MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) median_employed_percent,
          MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) median_wage
          FROM read_parquet(__INPUT__) GROUP BY {keys}'''
    try:
        p,d,n=thematic_sql_disk(a,q,'graduates_source_'+str(i),preview=2000,mem='2GB')
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
           'group_rows':n,'group_columns':','.join(group),'status':'PASS_NOT_ADDITIVE',
           'employment_asof':'2024-12-31','geography_of_work_known':False})
        qa_table('graduate_family_'+str(i),d,allow_empty=True)
        PANEL_FILES['graduate_family_'+str(i)]=p
    except Exception as ex:
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'status':'FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'ML-5.5','status':'FAMILY_PROFILE_FAILED','reason':a['asset_id']+' '+type(ex).__name__,
           'required':'recheck exact statistical strata'})
qa_table('e04_e05_full_family_audit',pd.DataFrame(FAMILY_E04E05),['asset_id'])

# E. Historical municipal reference: full SQL join, no 431k+ rows in Pandas.
for key,asset_prefix in [('D04','06::D04::D04_member00::spatial_reference'),
                         ('BDMO_REG','06::BDMO6::BDMO6_member00::municipality_registry_observed'),
                         ('BDMO_HIST','06::BDMO6::BDMO6_member00::municipality_history_observed')]:
    a=required_one(source_asset(prefix=asset_prefix),key)
    cols=a.get('columns_manifest') or []
    chosen=[x for x in ('source_oktmo','region_name','municipality_name','settlement_name','observation_year',
      'coordinates_valid','latitude','longitude','census_year','observed_name_change','observed_type_change',
      'boundary_change_not_verified') if x in cols]
    if chosen:
        frame=read_table(a,chosen,limit=2000)  # explicitly labelled preview; full source profiled above
        if key=='D04':D04_SPATIAL=frame
        elif key=='BDMO_REG':BDMO_SPATIAL=frame
        else:HISTORY_MUNICIPAL=frame
        qa_table('spatial_'+key+'_preview',frame,allow_empty=True)

registry_asset=required_one(source_asset(prefix='06::BDMO6::BDMO6_member00::municipality_registry_observed'),'BDMO REG')
municipal_path=fetch_asset(MUN); registry_path=fetch_asset(registry_asset)
query=f'''SELECT CAST(d.oktmo AS VARCHAR) oktmo, TRY_CAST(d.year AS INTEGER) observation_year,
    MIN(CAST(d.region AS VARCHAR)) region, MIN(CAST(d.municipality AS VARCHAR)) municipality,
    MIN(CAST(b.municipality_name AS VARCHAR)) municipality_name,
    COUNT(DISTINCT CAST(b.municipality_name AS VARCHAR)) n_registry_names
    FROM read_parquet({_sql_literal(municipal_path)}) d
    INNER JOIN read_parquet({_sql_literal(registry_path)}) b
      ON CAST(d.oktmo AS VARCHAR)=CAST(b.source_oktmo AS VARCHAR)
     AND TRY_CAST(d.year AS INTEGER)=TRY_CAST(b.observation_year AS INTEGER)
    GROUP BY 1,2'''
try:
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        joined=con.execute(query).df()  # upper bound: 31,445 D01 rows
    joined['municipality_names_equal']=joined['municipality'].astype(str).str.lower().str.replace('ё','е').str.strip().eq(
       joined['municipality_name'].astype(str).str.lower().str.replace('ё','е').str.strip())
    joined['spatial_gate']='EXACT_OKTMO_YEAR_NAME_COMPARE_ONLY_NOT_CAMPUS'
    SPATIAL_CROSS=joined
    qa_table('spatial_exact_crosswalk',joined,['oktmo','observation_year'],allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'GEOGRAPHY','status':'CROSSWALK_FAILED','reason':type(ex).__name__+': '+str(ex)[:140],
       'required':'inspect municipality-year and historical OKTMO definitions'})
    qa_table('spatial_exact_crosswalk',pd.DataFrame(),allow_empty=True)

# F. Educational infrastructure D02::education at municipality×year×indicator_code, conservative uniqueness.
edu_source=required_one(source_asset(prefix='02::education'),'BDMO educational indicators')
edu_cols=edu_source['columns_manifest']
# Selection is a *hypothesis candidate*, not an automatic campus register.
pattern='(профессиональн|высш[еиг]|колледж|техникум|университет|образовательн.*организац)'
q=f'''SELECT CAST(oktmo AS VARCHAR) oktmo,TRY_CAST(year AS INTEGER) observation_year,
  CAST(indicator_code AS VARCHAR) indicator_code,CAST(indicator_name AS VARCHAR) indicator_name,
  CAST(indicator_unit AS VARCHAR) indicator_unit,
  COUNT(*)::BIGINT n_raw_rows,
  MIN(TRY_CAST(indicator_value AS DOUBLE)) AS min_reported,
  MAX(TRY_CAST(indicator_value AS DOUBLE)) AS max_reported,
  MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported
  FROM read_parquet(__INPUT__)
  WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)), {_sql_literal(pattern)})
  AND oktmo IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5'''
try:
    if CFG.MODE!='FULL':raise ValueError('MUNICIPAL_ECONOMIC_FULL_SCAN_ONLY')
    fp,preview,n=thematic_sql_disk(edu_source,q,'bdmo_education_municipal',preview=1000,mem='2GB')
    # Bounded filtered result only. If high-cardinality, use SQL to filter exact one-row groups first.
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        nuniq=int(con.execute('SELECT COUNT(*) FROM read_parquet(?) WHERE n_raw_rows=1 AND min_reported=max_reported',[str(fp)]).fetchone()[0])
        if nuniq<=CFG.MAX_PANDAS_ROWS:
            BDMO_PROXY_FULL_VALID=True
            BDMO_EDU_PROXY_CANDIDATES=con.execute('''SELECT * FROM read_parquet(?)
              WHERE n_raw_rows=1 AND min_reported=max_reported AND median_reported>=0''',[str(fp)]).df()
        else:
            BDMO_EDU_PROXY_CANDIDATES=preview.copy()
            ISSUES.append({'module':'ML-5.7','status':'PROXY_PREVIEW_ONLY',
               'reason':f'{nuniq} unique municipal education rows exceed RAM cap','required':'SQL-only E0/E1'})
    BDMO_EDU_PROXY_CANDIDATES['source_definition']='BDMO_MUNICIPAL_EDUCATION_INDICATOR_NOT_CAMPUS'
    qa_table('bdmo_education_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'ML-5.7','status':'BDMO_EDUCATION_REVIEW_SKIPPED_OR_FAILED',
       'reason':type(ex).__name__+': '+str(ex)[:150],'required':'review BDMO indicator units/levels'})
    qa_table('bdmo_education_candidates',pd.DataFrame(),allow_empty=True)

# Human-readable quality conditions, all sourced from actually processed files.
qa_table('municipal_infrastructure_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
print('CROSS SOURCE:',len(D02_PROFILE),'D02 members;',len(VPO_ALL_PROFILE),'E02/VPO;',
      len(FAMILY_E04E05),'E04/E05;',len(SPATIAL_CROSS),'exact municipal code-year matches;',
      len(BDMO_EDU_PROXY_CANDIDATES),'municipal infrastructure candidates')
memory_point('after_cross_source')
# Финальная отметка этапа перенесена после всех его подмодулей


## 6Б. Содержательная проверка 52/52 источников
Пофайловое содержание без подмены единиц измерения; годовые СПО и raw-матрицы не суммируются повторно.

In [ ]:
# ML-05 V4 · independent full-content utilization register (52/52).
# These are measurements of the SOURCE itself, not automatically legal predictive features.
progress_event('CROSS_SOURCE','V4: физическое обследование значений всех наборов Stage I',announce=True)
V4_SOURCE_PROFILES=[];V4_SOURCE_ERRORS=[]
for _i,_a in enumerate(ASSETS,1):
    _id=_a['asset_id'];_p=fetch_asset(_a);_cols=list(pq.ParquetFile(_p).schema_arrow.names)
    _time=next((x for x in ('year','observation_year','graduation_year') if x in _cols),None)
    _measure=next((x for x in ('indicator_value','observed_value','count_graduate','population','value','migration') if x in _cols),None)
    _y=f'TRY_CAST("{_time}" AS INTEGER)' if _time else 'NULL::INTEGER'
    _v=f'TRY_CAST("{_measure}" AS DOUBLE)' if _measure else 'NULL::DOUBLE'
    _q=f'''SELECT {_y} year_label,COUNT(*)::BIGINT source_rows,
      COUNT({_v})::BIGINT numeric_values,
      SUM(CASE WHEN {_v}<0 THEN 1 ELSE 0 END)::BIGINT negative_values,
      SUM(CASE WHEN {_v}=0 THEN 1 ELSE 0 END)::BIGINT zero_values,
      SUM(CASE WHEN {_v} IS NULL THEN 1 ELSE 0 END)::BIGINT null_or_non_numeric,
      MIN({_v}) AS minimum_observed_value,
      MAX({_v}) AS maximum_observed_value,
      AVG({_v}) AS unweighted_mean_of_source_records,
      STDDEV_POP({_v}) AS dispersion_of_source_records
      FROM read_parquet({_sql_literal(_p)}) GROUP BY 1'''
    _out=ROOT/'derived'/'v4_source_values'/f'source_{_i:02d}.parquet';_out.parent.mkdir(parents=True,exist_ok=True)
    _tmp=_out.with_suffix('.partial.parquet');_tmp.unlink(missing_ok=True)
    progress_event('CROSS_SOURCE',f'V4 [{_i}/52] число, пропуски, нули: {_id}',_i,len(ASSETS),announce=(_i==1 or _i%5==0 or _i==len(ASSETS)))
    try:
        with duckdb.connect(':memory:') as _db:
            _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
            _db.execute('SET preserve_insertion_order=false')
            _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
            _db.execute('COPY ('+_q+') TO '+_sql_literal(_tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
            _r=_db.execute('SELECT COALESCE(SUM(source_rows),0),COUNT(*) FROM read_parquet(?)',[str(_tmp)]).fetchone()
        if int(_r[0])!=int(_a['rows_actual']):raise AssertionError(f'Content count {_r[0]} != {_a["rows_actual"]}')
        _tmp.replace(_out);CONTEXT_PARQUET_FILES.append(_out)
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':int(_r[0]),'time_field':_time,'numeric_field':_measure,
           'groups_by_year':int(_r[1]),'sha_verified':_id in INPUT_SHA,'content_status':'FULL_VALUE_COVERAGE_PASS',
           'model_eligibility':'REQUIRES_TARGET_SEMANTICS_AND_GRAIN','derived_parquet':_out.name})
    except Exception as _ex:
        _tmp.unlink(missing_ok=True)
        V4_SOURCE_ERRORS.append({'asset_id':_id,'error':type(_ex).__name__,'details':str(_ex)[:280]})
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':None,'time_field':_time,'numeric_field':_measure,'groups_by_year':None,
           'sha_verified':_id in INPUT_SHA,'content_status':'FAIL_CONTENT_SCAN','derived_parquet':''})
_v4_used=pd.DataFrame(V4_SOURCE_PROFILES)
qa_table('v4_all52_source_value_audit',_v4_used,['asset_id'])
qa_table('v4_source_scan_failures',pd.DataFrame(V4_SOURCE_ERRORS),allow_empty=True)
if CFG.MODE=='FULL' and (len(_v4_used)!=52 or len(V4_SOURCE_ERRORS)):
    fail('ALL52_CONTENT','One or more Stage I members failed full numeric content scan: '+str(V4_SOURCE_ERRORS[:2]))

# Source -> defensible research use. Each stage is recorded independently, no claim of 52 model predictors.
_v4_roles=[]
for _a in ASSETS:
    _seg=_a['segment'];_r=_a.get('role','') or ''
    _topics={'01':'Municipal demography and ML-5.7','02':'Economic/education context and ML-5.1/ML-5.7',
      '03':'SPO OD, organizational panels and network ML-5.1/2/3/6',
      '04':'VO monitoring, admissions and specialties ML-5.2/5.4',
      '05':'Graduate employability and outcomes ML-5.5',
      '06':'Historic OKTMO, places and geographic accessibility ML-5.1/5.7'}
    _direct=_a['asset_id'] in {E01['asset_id'],E02['asset_id'],VPO_211['asset_id'],
              GRAD_ORG['asset_id'],GRAD_FIELD['asset_id'],MUN['asset_id'],TRANSPORT['asset_id']}
    _v4_roles.append({'asset_id':_a['asset_id'],'segment':_seg,'role_in_program':_topics.get(_seg),
       'content_scanned':True,'scientific_content_used':'YES_SOURCE_SPECIFIC_PROFILE',
       'main_prediction_input':bool(_direct),
       'not_main_input_reason':('Derived/alternate reporting representation or incompatible key/grain;' 
                                'not pooled into target by assumption' if not _direct else ''),
       'prevent_double_counting':_seg=='03' and (_r=='organization_indicators' or _r=='education_od_matrix_raw_cells'),
       'source_sha':INPUT_SHA.get(_a['asset_id'],'')})
qa_table('v4_all52_research_use',pd.DataFrame(_v4_roles),['asset_id'])
print('V4 / source use:',len(V4_SOURCE_PROFILES),'complete value-audits;',len(V4_SOURCE_ERRORS),'failures')


In [ ]:
# V7.3.0 · источники: обследование содержимого и проверяемые аналитические роли.
# Ни один из 52 наборов не объявляется модельным признаком только потому, что SHA сошлась.
progress_event('CROSS_SOURCE','V7: контроль научного обследования всех 52 источников',announce=True)
V7_SOURCE_AUDIT=[]
_v7_profile={r['asset_id']:r for r in V4_SOURCE_PROFILES}
_v7_inventory={r['asset_id']:r for r in FULL52_INVENTORY}
_v7_roles={
 '01':'Муниципальная демография: временные величины и состав E0/E1',
 '02':'Доходы, занятость, бюджет и образование: раздельные источниковые распределения',
 '03':'СПО, OD, реестры, сопоставление матриц и годовых вариантов',
 '04':'Мониторинг ВО, приём и структура специальностей',
 '05':'Выпуск, занятость и зарплаты: отдельные непересекающиеся страты',
 '06':'История ОКТМО, география, поселения и транспорт'
}
for _a in ASSETS:
    _id=_a['asset_id'];_p=_v7_profile.get(_id,{});_i=_v7_inventory.get(_id,{})
    _full=_p.get('content_status')=='FULL_VALUE_COVERAGE_PASS'
    _asset_name=_p.get('derived_parquet','')
    _stats_path=ROOT/'derived'/'v4_source_values'/_asset_name
    V7_SOURCE_AUDIT.append({
      'asset_id':_id,'source_group':_a['segment'],'rows_in_manifest':int(_a['rows_actual']),
      'sha256':_a['sha256'],'verified_sha256':INPUT_SHA.get(_id)==_a['sha256'],
      'footer_rows_verified':bool(_i.get('footer_rows_ok')),
      'all_values_profiled':bool(_full),
      'numeric_profile_file':_asset_name,
      'numeric_profile_exists':bool(_stats_path.is_file()),
      'time_field':_p.get('time_field'),'numeric_field':_p.get('numeric_field'),
      'source_specific_statistical_use':_v7_roles.get(_a['segment'],'UNREGISTERED'),
      'predictor_role':'NOT_DETERMINED_UNTIL_MODEL_FIT',
      'scientific_constraint':'NO_CROSS_SOURCE_ADDITIVITY_WITHOUT_GRAIN_AND_UNIT_VALIDATION'
    })
V7_SOURCE_AUDIT=pd.DataFrame(V7_SOURCE_AUDIT)
qa_table('v7_all52_scientific_source_audit',V7_SOURCE_AUDIT,['asset_id'])
if CFG.MODE=='FULL' and (len(V7_SOURCE_AUDIT)!=52 or not
        (V7_SOURCE_AUDIT.verified_sha256 & V7_SOURCE_AUDIT.footer_rows_verified &
         V7_SOURCE_AUDIT.all_values_profiled & V7_SOURCE_AUDIT.numeric_profile_exists).all()):
    raise RuntimeError('V7: хотя бы один источник не прошёл полный числовой/временной/хеш-контроль')
print('V7: исходники 52/52 проверены, содержательные профили раздельны; модельная трассировка будет сформирована после обучения')

mark_stage('CROSS_SOURCE')


## 7. Предрегистрация, временные разбиения, модели
Сопоставляются persistence, среднее двух наблюдений, затухающий тренд, Ridge, CatBoost; PPML только для доказанных неотрицательных счётных величин. TEST не выбирает победителя.

In [ ]:
progress_event('MODEL_DESIGN','Разбиения, метрики, базовые методы, Ridge/CatBoost')
# ML-05 · план эксперимента и функции, одинаковые для всех доступных модулей.
EXPERIMENT_PLAN={
 'version':CFG.VERSION,'mode':CFG.MODE,'horizon':CFG.HORIZON,
 'source_commit':SOURCE_COMMIT,'source_index_blob':index_info['sha'],
 'feature_lag':CFG.FEATURE_PUBLICATION_LAG_YEARS,
 'label_lag':CFG.LABEL_PUBLICATION_LAG_YEARS,
 'candidate_models':['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm','ppml_if_observed_integer_counts','negbin_fixed_dispersion_if_integer_counts'],
 'admission_approval_required':True,'unknown_edges_never_zero':True,
 'E01_is_not_certified_admissions':True,
 'validation':'DEV only if its label is available by TEST origin; otherwise fixed preregistered comparison, no selection',
 'seeds_dev':CFG.DEV_SEEDS,'seeds_final':CFG.FINAL_SEEDS,
 'test_keys_policy':'strict common support; no synthetic missing OD edges',
 'source_vintage':INDEX['created_at'],
 'publication_policy':'only FULL + local checks + explicit opt-in + science gate',
}
PLAN_HASH=canonical_hash(EXPERIMENT_PLAN)
PRED_KEYS=['module','target_id','sector','entity_id','origin_year','horizon','target_year',
           'split_id','model','seed','information_cutoff_year','source_vintage',
           'geography_grain','observation_definition','actual','predicted']

# Предварительно заданные признаки; семантика источника не попадает в X как oracle.
CAND_NUM=['prior_value','prior2_value','origin_mass','destination_mass',
          'origin_destinations','destination_origins','distance_proxy_km',
          'confirmed_vo_sites','prev_population','prev_migration_rate',
          'origin_year']
CAND_CAT=['origin_region','destination_region','region','sector','parent_group_id']

def feature_spec(frame):
    nums=[c for c in CAND_NUM if c in frame.columns and number(frame[c]).notna().any()]
    cats=[c for c in CAND_CAT if c in frame.columns and frame[c].notna().any()]
    for c in frame.columns:
        if c.startswith(('origin_econ_','destination_econ_')) and number(frame[c]).notna().any():nums.append(c)
    if 'prior_value' not in nums:fail('MODEL','Нет prior_value: нельзя оценить историческую инерцию')
    return nums,cats

def x_prepare(frame,nums,cats):
    x=frame[nums+cats].copy()
    for c in nums:x[c]=number(x[c]).astype(float)
    for c in cats:x[c]=x[c].fillna('__MISSING__').astype(str)
    return x

def effective_mae(y,yp):
    y=np.asarray(y,dtype=float);p=np.asarray(yp,dtype=float)
    mask=np.isfinite(y)&np.isfinite(p)
    if not mask.any():return {'MAE':np.nan,'RMSE':np.nan,'WAPE':np.nan,'n':0}
    y,p=y[mask],p[mask]
    return {'MAE':float(np.mean(np.abs(y-p))),'RMSE':float(np.sqrt(np.mean((y-p)**2))),
            'WAPE':float(np.abs(y-p).sum()/np.abs(y).sum()) if np.abs(y).sum()>0 else np.nan,'n':int(len(y))}

# Проверка, что один и тот же target не оказался одновременно в train/dev/test.
def temporal_partition(frame,min_train=None,min_test=None):
    if frame.empty:return None,'EMPTY_FRAME'
    f=frame.sort_values('target_year').copy()
    years=sorted(int(x) for x in f.target_year.dropna().unique())
    if len(years)<2:return None,'INSUFFICIENT_TARGET_YEARS'
    test_year=years[-1];test_origin=test_year-CFG.HORIZON
    tr=f[(f.target_year<test_year)&
         (f.target_available_year<=test_origin)].copy()
    test=f[f.target_year==test_year].copy()
    min_train=int(CFG.MIN_TRAIN_ROWS if min_train is None else min_train)
    min_test=int(CFG.MIN_TEST_ROWS if min_test is None else min_test)
    if len(tr)<min_train or len(test)<min_test:
        return None,f'LOW_SUPPORT: train={len(tr)},test={len(test)}'
    # DEV может существовать только если доступен к моменту финального решения.
    available_dev=sorted(int(y) for y in tr.target_year.unique())
    dev=None
    if len(available_dev)>=2:
        dv=available_dev[-1];dev_origin=dv-CFG.HORIZON
        early=tr[(tr.target_year<dv)&(tr.target_available_year<=dev_origin)].copy()
        potential=tr[tr.target_year==dv].copy()
        if len(early)>=min_train and len(potential)>=CFG.MIN_DEV_ROWS:
            dev=potential;tr=early
    if tr.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность train')
    if test.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность test')
    if (tr.target_available_year>test_origin).any():fail('SPLIT','train label published after test origin')
    if (test.feature_available_year>test.information_cutoff_year).any():fail('SPLIT','future feature in test')
    if dev is not None and (dev.target_available_year>test_origin).any():fail('SPLIT','dev label leaked to test')
    out={'train':tr,'dev':dev,'test':test,'test_year':test_year,'test_origin':test_origin,
         'protocol':'DEV_SELECTED' if dev is not None else 'FIXED_NO_DEV',
         'retrospective_vintage':True}
    SPLIT_AUDIT.append({'module':f.get('module',pd.Series(['UNSET'])).iloc[0],
       'protocol':out['protocol'],'train_rows':len(tr),'dev_rows':0 if dev is None else len(dev),
       'test_rows':len(test),'test_year':test_year,'test_origin':test_origin,
       'labels_known_asof_test':True,'source_vintage_at_test':'NOT_ARCHIVED'})
    return out,None


# V4: fixed-dispersion NB GLM, not mislabeled as an estimated NB2 dispersion model.
from sklearn.base import BaseEstimator, RegressorMixin
class NegativeBinomialFixedDispersion(RegressorMixin, BaseEstimator):
    """NB2 (alpha fixed), penalized MLE with stable log-link and declared prediction-support bound.

    Count targets must be observed nonnegative integers. Fits a low-dimensional, numeric-only
    design with L2 penalty; never densifies the high-cardinality categorical one-hot matrix.
    The predeclared upper bound is 10×max(train count): clipping is reported as diagnostic,
    so predictions cannot silently explode into 1e68 while still receiving PASS.
    """
    def __init__(self, alpha=1.0, penalty=0.5, support_multiplier=10.0):
        self.alpha=float(alpha);self.penalty=float(penalty);self.support_multiplier=float(support_multiplier)
    def fit(self,X,y):
        from scipy.optimize import minimize
        from scipy.special import expit
        a=np.asarray(X,dtype=np.float64); y=np.asarray(y,dtype=np.float64).reshape(-1)
        if a.ndim==1:a=a.reshape(-1,1)
        if a.shape[0]!=len(y) or not len(y) or a.shape[1]>64:raise ValueError('NB2 incompatible design')
        if not np.isfinite(y).all() or (y<0).any() or not np.allclose(y,np.round(y),atol=1e-8,rtol=0):
            raise ValueError('NB2 requires observed integer nonnegative counts')
        a=np.clip(np.nan_to_num(a,nan=0.0,posinf=0.0,neginf=0.0),-8.,8.)
        a=np.column_stack([np.ones(len(y)),a]); r=1.0/self.alpha
        # Stable NB log-likelihood: (y+r)*softplus(eta-log(r))-y*(eta-log(r)), up to constants.
        scale=max(1.,float(np.mean(y+r)));x0=np.zeros(a.shape[1]);x0[0]=np.log(max(.001,float(np.mean(y))))
        def fun(beta):
            e=a@beta-np.log(r)
            loss=np.mean(((y+r)*np.logaddexp(0.,e)-y*e))/scale+0.5*self.penalty*np.dot(beta[1:],beta[1:])
            g=(a.T@((y+r)*expit(e)-y))/(len(y)*scale)
            g[1:]+=self.penalty*beta[1:]
            return float(loss),g
        bounds=[(-22.,22.)]+[(-6.,6.)]*(a.shape[1]-1)
        opt=minimize(fun,x0,method='L-BFGS-B',jac=True,bounds=bounds,options={'maxiter':350,'ftol':1e-10,'gtol':1e-7})
        if not opt.success or not np.isfinite(opt.fun):
            raise RuntimeError('NB2 regularized likelihood did not converge: '+str(opt.message))
        self.coef_=opt.x;self.fit_converged_=True;self.fit_objective_=float(opt.fun)
        self.train_max_=float(np.max(y));self.train_mean_=float(np.mean(y))
        self.n_fit_=len(y);self.count_truncated_=0;self.max_raw_prediction_=np.nan
        return self
    def predict(self,X):
        a=np.asarray(X,dtype=np.float64)
        if a.ndim==1:a=a.reshape(-1,1)
        a=np.clip(np.nan_to_num(a,nan=0.,posinf=0.,neginf=0.),-8.,8.)
        eta=np.column_stack([np.ones(len(a)),a])@self.coef_
        # The model's response space is constrained by a *predeclared* historical support bound.
        # This is not replacement of missing observations. Truncation frequency is an output QC.
        max_mean=max(1.,self.train_max_)*self.support_multiplier
        clipped=np.clip(eta,-30.,np.log(max_mean))
        self.count_truncated_=int(np.count_nonzero(clipped!=eta))
        self.truncated_fraction_=float(self.count_truncated_/max(1,len(a)))
        self.max_raw_logmu_=float(np.max(eta)) if len(a) else np.nan
        pred=np.exp(clipped)
        if not np.isfinite(pred).all():raise FloatingPointError('NB2 unstable response')
        return pred

def pipeline_fit_predict(model,train,test,seed=11,allow_ppml=False,nonnegative=True):
    nums,cats=feature_spec(train)
    if set(nums+cats)&set(['actual','target_year','target_available_year','observed_value']):
        fail('LEAKAGE','В модель попала фактическая целевая переменная или период метки')
    Xtr=x_prepare(train,nums,cats);Xte=x_prepare(test,nums,cats)
    y=number(train.actual).to_numpy(float)
    if not np.isfinite(y).all():fail('MODEL','Невалидные training labels')
    if model=='ridge':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',Ridge(alpha=30.,solver='lsqr'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='elasticnet':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.7)),
             ('model',ElasticNet(alpha=1.0,l1_ratio=.35,max_iter=4000,random_state=(None if int(seed)<0 else int(seed)),selection='cyclic'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='lightgbm':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median'))]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.6)),
             ('model',LGBMRegressor(n_estimators=220,num_leaves=15,max_depth=5,learning_rate=.05,
                       reg_lambda=8.,min_child_samples=15,verbosity=-1,n_jobs=1,random_state=(None if int(seed)<0 else int(seed))))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='ppml':
        if not allow_ppml or np.any(y<0):raise ValueError('PPML requires verified nonnegative counts')
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',PoissonRegressor(alpha=3.,max_iter=200))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='negbin':
        if not allow_ppml or np.any(y<0) or not np.isfinite(y).all():
            raise ValueError('Negative binomial GLM requires observed nonnegative finite counts')
        # NB2: numeric-only design; avoids high-cardinality one-hot IRLS explosion.
        # All numeric inputs are observed past information, not current target.
        obj=Pipeline([('prep',ColumnTransformer([
                 ('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                          ('scale',StandardScaler())]),nums)],sparse_threshold=0.)),
            ('model',NegativeBinomialFixedDispersion(alpha=1.,penalty=0.5,
                          support_multiplier=CFG.MAX_COUNT_PREDICTION_MULTIPLIER))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        cats=[];artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,
                  'dispersion':'FIXED_ALPHA_1_PENALIZED_NB2',
                  'extrapolation_rule':'PREDICT_MEAN_AT_MOST_10X_MAX_OBSERVED_TRAIN_COUNT',
                  'prediction_guard_events':int(obj.named_steps['model'].count_truncated_),
                  'prediction_guard_fraction':float(obj.named_steps['model'].truncated_fraction_)}
    elif model=='catboost':
        med={c:float(Xtr[c].median()) if Xtr[c].notna().any() else 0. for c in nums}
        for c in nums:
            Xtr[c]=Xtr[c].fillna(med[c]);Xte[c]=Xte[c].fillna(med[c])
        obj=CatBoostRegressor(iterations=60 if CFG.MODE=='SMOKE' else 260,
                  depth=5,learning_rate=.05,l2_leaf_reg=8.,loss_function='RMSE',
                  random_seed=int(seed),verbose=False,thread_count=1,used_ram_limit='2gb',allow_writing_files=False)
        obj.fit(Xtr,y,cat_features=cats);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,'medians':med}
    else:fail('MODEL','Неизвестная модель '+model)
    yp=np.asarray(yp,dtype=np.float64)
    if len(yp)!=len(test) or not np.isfinite(yp).all():raise FloatingPointError(model+' invalid prediction array')
    if nonnegative and (yp<0).any():yp=np.maximum(yp,0)
    return yp,artifact

def reload_predict(artifact,test,nonnegative=True):
    X=x_prepare(test,artifact['nums'],artifact['cats'])
    if artifact['kind']=='catboost':
        for c,v in artifact['medians'].items():X[c]=X[c].fillna(v)
    p=artifact['pipeline'].predict(X)
    return np.maximum(p,0) if nonnegative else p

def baseline_prediction(test,kind,nonnegative=True):
    a=number(test.prior_value).to_numpy(float)
    if kind=='persistence':b=a
    elif kind in ('rolling_mean_2','damped_trend'):
        other=number(test.prior2_value).to_numpy(float)
        b=np.where(np.isfinite(other),(a+other)/2 if kind=='rolling_mean_2' else a+.35*(a-other),a)
    else:fail('BASELINE','Неизвестный baseline '+kind)
    return np.maximum(b,0) if nonnegative else b

def predictions_frame(frame,module,model,yp,seed=-1,protocol='ROLLING_TIME'):
    out=pd.DataFrame({'module':module,'target_id':frame.get('target_id',pd.Series(['UNDEFINED']*len(frame))).to_numpy(),
     'sector':frame.get('sector',pd.Series(['UNKNOWN']*len(frame))).to_numpy(),
     'entity_id':frame.entity_id.astype(str).to_numpy(),
     'origin_year':frame.origin_year.to_numpy(),'horizon':CFG.HORIZON,'target_year':frame.target_year.to_numpy(),
     'split_id':protocol,'model':model,'seed':int(seed),
     'information_cutoff_year':frame.information_cutoff_year.to_numpy(),
     'feature_available_year':frame.feature_available_year.to_numpy(),
     'target_available_year':frame.target_available_year.to_numpy(),
     'source_vintage':INDEX['created_at'],
     'geography_grain':'ORIGIN_DESTINATION' if module=='ML-5.1' else ('REGION' if module=='ML-5.3' else 'ORGANIZATION'),
     'observation_definition':frame.get('target_id',pd.Series(['observed_value']*len(frame))).to_numpy(),
     'actual':number(frame.actual).to_numpy(float),'predicted':np.asarray(yp,dtype=float)})
    for c in ('origin_region','destination_region','parent_group_id','region'):
        if c in frame:out[c]=frame[c].astype('string').to_numpy()
    return out

def record_prediction(frame):
    # Прогнозы всех сидов сохраняются немедленно; массивы не остаются в RAM.
    global PRED_TOTAL
    out=ROOT/'derived'/'prediction_parts';out.mkdir(parents=True,exist_ok=True)
    path=out/f'part_{len(PRED_PARTS):05d}.parquet'
    write_df_chunks(frame,path)
    PRED_PARTS.append(str(path));PRED_TOTAL+=int(len(frame))
    del frame
    if len(PRED_PARTS)%10==0:
        gc.collect();memory_point(f'predictions_saved_{len(PRED_PARTS)}')

def fit_and_record(module,model,train,test,seed=-1,nonnegative=True,allow_ppml=False,protocol='ROLLING_TIME'):
    sector=str(test.sector.iloc[0]) if 'sector' in test and test.sector.nunique()==1 else 'MIXED'
    if model in ('persistence','rolling_mean_2','damped_trend'):
        pred=baseline_prediction(test,model,nonnegative);artifact=None
    else:
        pred,artifact=pipeline_fit_predict(model,train,test,seed=seed,allow_ppml=allow_ppml,
                                         nonnegative=nonnegative)
        ident={'module':module,'sector':sector,'protocol':protocol,
               'target_id':sorted(test.target_id.astype(str).unique().tolist()) if 'target_id' in test else [],
               'model':model,'seed':int(seed)}
        short=hashlib.sha256(json.dumps(ident,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16]
        name=f'{module.replace("-","").replace(".","_")}_{re.sub(r"[^A-Za-z0-9_]+","_",sector)[:28]}_{re.sub(r"[^A-Za-z0-9_]+","_",protocol)[:32]}_{model}_{seed}_{short}.joblib'
        path=ROOT/'models'/name
        if name in MODEL_FILES or path.exists():
            raise RuntimeError('DUPLICATE_MODEL_ARTIFACT: '+name)
        joblib.dump(artifact,path,compress=3)
        loaded=joblib.load(path)
        rp=reload_predict(loaded,test,nonnegative=nonnegative)
        if not np.allclose(rp,pred,rtol=1e-7,atol=1e-7):fail('MODEL_RELOAD',name+' discrepancy')
        MODEL_FILES[name]=str(path)
        MODEL_RECORDS.append({'module':module,'sector':sector,'protocol':protocol,'target_id':ident['target_id'],'model':model,'seed':seed,'path':name,
            'file_sha256':hash_file(path),'feature_num':artifact['nums'],'feature_cat':artifact['cats'],
            'training_rows':len(train),'test_rows':len(test),'reloaded':True,
            'prediction_guard_events':artifact.get('prediction_guard_events',0),
            'prediction_guard_fraction':artifact.get('prediction_guard_fraction',0.)})
    if len(pred)!=len(test):fail('SUPPORT','Модель изменила число тестовых ключей')
    record_prediction(predictions_frame(test,module,model,pred,seed,protocol))
    met=effective_mae(test.actual.to_numpy(float),pred)
    METRICS.append({'module':module,'sector':sector,'model':model,'protocol':protocol,'seed':seed,
       'target_year':int(test.target_year.max()),'source_vintage':INDEX['created_at'],
       **met})
    # Процент неизвестен (ветвление зависит от данных); выводим точный счётчик.
    progress_event(None,f'{module}: {model}, рассчитано {len(METRICS):,} метрик, сохранено {len(MODEL_FILES):,} моделей',
                   current=len(METRICS),total=None,announce=False)
    return pred

mark_stage('MODEL_DESIGN')


## 8. Встроенные интеграционные проверки без GitHub-публикации
11 автоматических тестов на малом искусственном наборе проверяют чистоту времени, восстановление моделей и карантин. Их численные значения не используются как научные результаты.

In [ ]:
progress_event('UNIT_TESTS','Проверка моделей, утечек и повторного чтения')
# Локальные встроенные тесты математической и организационной логики; не являются научными наблюдениями.
SELF_TESTS=[]
def unit_result(name,condition):
    SELF_TESTS.append({'test':name,'result':'PASS' if bool(condition) else 'FAIL'})
    if not bool(condition):fail('UNIT',name)

# Временной split — метки всегда доступны к соответствующему моменту принятия решения.
_fixture=pd.DataFrame([{'entity_id':f'TEST_{e:02d}','observation_year':y,
                        'response':float(e+2*(y-2010)), 'sector':'UNIT'}
                       for e in range(35) for y in range(2012,2026)])
_panel=historical_pair_panel(_fixture[['entity_id','observation_year','response']],['entity_id'],'response')
_panel['sector']='UNIT';_panel['module']='UNIT';_panel['region']='UNIT_REGION'
_panel['target_id']='ARTIFICIAL_TEST_ONLY'
_split,_reason=temporal_partition(_panel)
unit_result('TIME_PANEL_EXISTS',len(_panel)>100)
unit_result('NO_FUTURE_FEATURES',bool((_panel.feature_available_year<=_panel.information_cutoff_year).all()))
unit_result('SPLIT_AVAILABLE',_split is not None)
unit_result('TRAIN_LABELS_KNOWN_AS_OF_TEST',bool((_split['train'].target_available_year<=_split['test_origin']).all()))
unit_result('TEST_KEYS_UNIQUE',not _split['test'].duplicated(['entity_id','target_year']).any())
if _split['dev'] is not None:
    unit_result('DEV_LABELS_KNOWN_AS_OF_TEST',bool((_split['dev'].target_available_year<=_split['test_origin']).all()))

# Модельный прогноз должен повторяться без дообучения после восстановления состояния.
_train=_split['train'];_test=_split['test'];_y_ridge,_ridge_art=pipeline_fit_predict('ridge',_train,_test)
unit_result('RIDGE_RELOAD_EQUAL',np.allclose(_y_ridge,reload_predict(_ridge_art,_test),atol=1e-8,rtol=1e-8))
_cb_y,_cb_art=pipeline_fit_predict('catboost',_train,_test,seed=CFG.DEV_SEEDS[0])
unit_result('CATBOOST_RELOAD_EQUAL',np.allclose(_cb_y,reload_predict(_cb_art,_test),atol=1e-7,rtol=1e-7))
unit_result('MODEL_OUTPUT_FINITE',np.isfinite(_y_ridge).all() and np.isfinite(_cb_y).all())
_en_y,_en_art=pipeline_fit_predict('elasticnet',_train,_test,seed=-1)
unit_result('ELASTICNET_SENTINEL_NEGATIVE_SEED_SAFE',np.isfinite(_en_y).all() and
             np.allclose(_en_y,reload_predict(_en_art,_test)))
_nb_fixture=_train.copy();_nb_test=_test.copy()
for _q in (_nb_fixture,_nb_test):_q['actual']=np.maximum(np.rint(_q.actual),0)
_po_y,_po_art=pipeline_fit_predict('ppml',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('PPML_OBSERVED_INTEGER_FIT_AND_RELOAD',np.isfinite(_po_y).all() and
             np.allclose(_po_y,reload_predict(_po_art,_nb_test)))
_nb_y,_nb_art=pipeline_fit_predict('negbin',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('NEGBIN_FIXED_DISPERSION_RELOAD',np.isfinite(_nb_y).all() and
             np.allclose(_nb_y,reload_predict(_nb_art,_nb_test)))
unit_result('BASELINE_NOT_FUTURE',np.allclose(baseline_prediction(_test,'persistence'),_test.prior_value))

# Спорный id×год отбрасывается с содержательным карантином, не объединяется по имени.
_quarantine_n=len(QUARANTINE)
_bad_registry=pd.DataFrame([{'entity_id':'X','observation_year':2024,'name':'Вуз 1'},
                            {'entity_id':'X','observation_year':2024,'name':'Вуз 2'},
                            {'entity_id':'Y','observation_year':2024,'name':'Колледж'}])
_clean=quarantine_conflicting_registry(_bad_registry,'TEST_ONLY')
unit_result('AMBIGUOUS_REGISTRY_QUARANTINED',len(_clean)==1 and _clean.entity_id.iloc[0]=='Y')
del QUARANTINE[_quarantine_n:] # тестовые записи не являются реальными спорными организациями

qa_table('embedded_unit_tests',pd.DataFrame(SELF_TESTS),['test'])
assert all(r['result']=='PASS' for r in SELF_TESTS)
print('ML-05 embedded tests:',len(SELF_TESTS),'PASS (контрольные данные; не научные результаты)')
del _fixture,_panel,_split,_reason,_train,_test,_ridge_art,_cb_art,_en_art,_en_y,_po_art,_po_y,_nb_art,_nb_y,_nb_fixture,_nb_test,_y_ridge,_cb_y,_quarantine_n,_bad_registry,_clean

mark_stage('UNIT_TESTS')


## 9. Семь предметных блоков ML-5.1–ML-5.7
Научные блоки выполняются на фактически наблюдаемых источниках. При отсутствии target или подтверждённого grain предусмотрен описательный результат или мотивированный отказ от прогноза.

In [ ]:
memory_point('before_model_training')
progress_event('SEVEN_MODULES','Расчёт семи модулей; расширенные статусы',announce=True)
# Семь предметных экспериментов. Наличие модели не является условием выполнения описательной части.
MODULE_DIAGNOSTICS=[];CONFIRMATORY=[];HYPOTHESES=[];MODULE_SUBTASKS=[]

def blocked(module,why,required=''):
    limitation(module,why,required)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':False,
        'n_observed':0,'reason':why})

def two_way_mae_bootstrap(base,model,groups_a,groups_b,reps=399,seed=20261104):
    # Two-way pigeonhole bootstrap — строки получают произведение кратностей независимых блоков.
    a=pd.Series(groups_a).fillna('__UNKNOWN__').astype(str).to_numpy()
    b=pd.Series(groups_b).fillna('__UNKNOWN__').astype(str).to_numpy()
    unique_a=np.unique(a);unique_b=np.unique(b)
    if len(unique_a)<8 or len(unique_b)<8:return {'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    v=np.asarray(model,dtype=float);u=np.asarray(base,dtype=float)
    d=v-u;delta=float(d.mean());rng=np.random.default_rng(seed)
    boot=[]
    for _ in range(reps):
        ca=Counter(rng.choice(unique_a,len(unique_a),replace=True))
        cb=Counter(rng.choice(unique_b,len(unique_b),replace=True))
        w=np.array([ca[x]*cb[y] for x,y in zip(a,b)],dtype=float)
        if w.sum():boot.append(float(np.average(d,weights=w)))
    if len(boot)<100:return {'delta_MAE':delta,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    arr=np.asarray(boot)
    p=min(1.,2*min((np.sum(arr<=0)+1)/(len(arr)+1),(np.sum(arr>=0)+1)/(len(arr)+1)))
    return {'delta_MAE':delta,'ci_low':float(np.quantile(arr,.025)),
        'ci_high':float(np.quantile(arr,.975)),'p_value':float(p),
        'clusters_a':len(unique_a),'clusters_b':len(unique_b)}

def test_predictive(module,raw,nonnegative=True,count_confirmed=False):
    if raw.empty:return blocked(module,'Нет валидной целевой панели после семантического контроля',
       'entity_id, observed target years, known historical features')
    progress_event('MODELS',f'{module}: подготовка временного теста ({len(raw):,} строк)',announce=True)
    frame=raw.copy();frame['module']=module
    split,reason=temporal_partition(frame,min_train=CFG.REGION_MIN_TRAIN_ROWS if module=='ML-5.3' else None, min_test=CFG.REGION_MIN_TEST_ROWS if module=='ML-5.3' else None)
    if split is None:return blocked(module,reason,'train + independent test + as-of labels')
    train,dev,test=split['train'],split['dev'],split['test']
    cand=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm']
    if count_confirmed:cand+=['ppml','negbin']
    if CFG.MODE=='SMOKE':cand=['persistence','rolling_mean_2','ridge']
    if CFG.MODE=='PILOT10' and len(train)>10000:
        train=train.sample(n=10000,random_state=CFG.RANDOM_SEED)
    primary='persistence';dev_scores={};dev_selection=False
    if dev is not None:
        # DEV labels действительно доступны к времени TEST-предсказания.
        for model in cand:
            try:
                if model in ('persistence','rolling_mean_2','damped_trend'):
                    p=baseline_prediction(dev,model,nonnegative)
                else:
                    p,_=pipeline_fit_predict(model,train,dev,seed=CFG.DEV_SEEDS[0],
                                             allow_ppml=count_confirmed,nonnegative=nonnegative)
                dev_scores[model]=effective_mae(dev.actual,p)['MAE']
            except Exception as ex:
                note('DEV',module+' '+model+': '+type(ex).__name__,'WARN')
        if dev_scores:
            priority={m:i for i,m in enumerate(cand)}
            primary=min(dev_scores,key=lambda m:(round(dev_scores[m],8),priority[m]))
            dev_selection=True
            # dev labels допустимо добавить в обучение только при as-of совпадении.
            train=pd.concat([train,dev],ignore_index=True)
    else:
        note('TEMPORAL',module+': DEV не опубликован к дате прогноза; модели предзаданы, без выбора по TEST','WARN')
    pred_by_model={}
    for model in cand:
        progress_event('MODELS',f'{module}: расчёт {model}',announce=True)
        seeds=[-1]
        if model=='catboost':
            if CFG.MODE=='SMOKE':seeds=[CFG.DEV_SEEDS[0]]
            elif CFG.MODE=='PILOT10' and dev_selection and primary=='catboost':seeds=list(CFG.DEV_SEEDS)
            elif CFG.MODE=='FULL':seeds=list(CFG.FINAL_SEEDS)[:CFG.ML54_FINAL_SEEDS] if module=='ML-5.4' else list(CFG.FINAL_SEEDS)  # 50 seeds for all FULL temporal tasks
            else:seeds=[CFG.FINAL_SEEDS[0]]
        preds=[]
        for i_seed,seed in enumerate(seeds,1):
            progress_event('MODELS',f'{module}: {model}, сид {i_seed}/{len(seeds)}',i_seed,len(seeds),announce=(i_seed==1 or i_seed%5==0 or i_seed==len(seeds)))
            try:
                p=fit_and_record(module,model,train,test,seed=seed,
                    nonnegative=nonnegative,allow_ppml=count_confirmed)
                preds.append(p)
            except Exception as ex:
                ISSUES.append({'module':module,'status':'MODEL_FAILED',
                     'reason':model+': '+type(ex).__name__+' — '+str(ex)[:180],
                     'required':'Inspect modeling frame; no fabricated substitute'})
                if CFG.MODE=='FULL' and model in ('negbin','ppml','ridge','elasticnet','lightgbm'):
                    note('MODEL_QA',module+' '+model+' failed, recorded as scientific FAIL','FAIL')
                if model==primary:raise
        if not preds:continue
        pred_by_model[model]=np.mean(preds,axis=0)
        if model=='catboost' and len(preds)>1:
            record_prediction(predictions_frame(test,module,'catboost_ensemble',pred_by_model[model],-1))
            METRICS.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','model':'catboost_ensemble','protocol':'ROLLING_TIME',
                 'seed':-1,'target_year':split['test_year'],
                 **effective_mae(test.actual,pred_by_model[model])})
    if 'persistence' not in pred_by_model:fail('MODEL','Нет сопоставимого исторического baseline')
    y=test.actual.to_numpy(float)
    base_err=np.abs(y-pred_by_model['persistence'])
    primary_err=np.abs(y-pred_by_model.get(primary,pred_by_model['persistence']))
    if module=='ML-5.1':
        ga=test.origin_region;gb=test.destination_region
    elif module=='ML-5.3':
        ga=test.origin_region;gb=test.target_year.astype(str)
    else:
        ga=test.parent_group_id.fillna(test.entity_id) if 'parent_group_id' in test else test.entity_id
        gb=test.target_year.astype(str)
    if dev_selection and primary!='persistence':
        ci=two_way_mae_bootstrap(base_err,primary_err,ga,gb,
               reps=99 if CFG.MODE=='SMOKE' else CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
    else:
        ci={'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,
            'clusters_a':len(pd.Series(ga).dropna().unique()),
            'clusters_b':len(pd.Series(gb).dropna().unique())}
    chosen=effective_mae(y,pred_by_model.get(primary,pred_by_model['persistence']))
    original=effective_mae(y,pred_by_model['persistence'])
    CONFIRMATORY.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','test_year':split['test_year'],'test_n':len(test),
        'choice_basis':'DEV_ASOF' if dev_selection else 'NO_CONFIRMATORY_COMPARISON_NO_DEV',
        'selected_model':primary,'baseline_mae':original['MAE'],'selected_mae':chosen['MAE'],
        'delta_mae':chosen['MAE']-original['MAE'],**ci,'source_vintage_warning':True})
    progress_event('MODELS',f'{module}: тест завершён, инерция MAE={original["MAE"]:.3g}, выбранная={chosen["MAE"]:.3g}',announce=True)
    MODULES[module].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
        reason=('Ретроспективный тест с неподтверждёнными историческими версиями источника; '
                'DEV-проверка доступности соблюдена; улучшение не гарантировано'),
        n=len(test),test_year=split['test_year'],selected=primary)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':True,
        'n_observed':len(raw),'n_test':len(test),'test_year':split['test_year'],
        'dev_available_asof':dev_selection,'selected_model':primary,
        'mae_baseline':original['MAE'],'mae_selected':chosen['MAE']})
    if not dev_selection:ISSUES.append({'module':module,'status':'SEVERE_TEMPORAL_LIMIT',
       'reason':'Нет DEV-меток, опубликованных до cutoff TEST; сравнения альтернатив только разведочные',
       'required':'older archived vintages or longer observations'})
    return split

# ML-5.1: истинная единица E01 пока не подтверждена как приём; регрессируем именно E01_observed_value.
if len(OD_INTER):
    MODULE_DIAGNOSTICS.append({'module':'ML-5.1','prediction_feasible':None,
        'n_observed':len(OD_INTER),'target_definition':OD_SEMANTIC['label'],
        'unknown_codes':len(OD_UNKNOWN),'diagonal_excluded':len(OD_INTRA),
        'coverage_breaks':int(coverage_df.potential_break.fillna(False).sum())})
    test_predictive('ML-5.1',OD_MODEL,nonnegative=True,count_confirmed=bool(OD_MODEL.actual.ge(0).all() and np.isclose(OD_MODEL.actual,np.rint(OD_MODEL.actual),atol=1e-8,rtol=0).all()))
    if coverage_df.potential_break.fillna(False).any():
        ISSUES.append({'module':'ML-5.1','status':'UNRESOLVED_YEAR_COVERAGE_BREAK',
          'reason':'Смена числа OD-пар в 2023 г. не доказана как изменение реальной мобильности; модельные результаты — условно разведочные',
          'required':'semantics/coverage harmonization across 2022–2023'})
else:blocked('ML-5.1','Нет истинно наблюдаемых межрегиональных E01-рёбер')

# ML-5.2: отдельно ВО/СПО; не моделировать учреждения, если не удалось установить уникальную величину.
if len(ORG_YEAR):
    for sector,group in ORG_MODEL.groupby('sector') if len(ORG_MODEL) else []:
        if len(group):
            result={'status':'NOT_STARTED','reason':'source not evaluated','n':0}
            try:
                test_predictive('ML-5.2',group,nonnegative=True,count_confirmed=True)
                result=MODULES['ML-5.2'].copy()
            except Exception as ex:
                result={'status':'FAILED','reason':f'{sector}: {type(ex).__name__}: {str(ex)[:100]}','n':0}
                raise
            finally:
                MODULE_SUBTASKS.append({'module':'ML-5.2','sector':sector,**result})
            MODULE_DIAGNOSTICS.append({'module':'ML-5.2','sector':sector,'n_organizations':group.entity_id.nunique(),
                    'n_observed':len(group)})
    if MODULES['ML-5.2']['status']=='NOT_STARTED':
        limitation('ML-5.2','Установлены потенциальные индикаторы, но недостаточно исторических пар для модели',
                   'admissions×organization×year with unambiguous units')
else:
    limitation('ML-5.2','Предметный каталог E02/VPO/SPO прочитан, но однозначное зачисление организация×год не подтверждено',
               'source-admission indicator and organization with unique key')

if MODULE_SUBTASKS:
    qa_table('organization_sector_subtasks',pd.DataFrame(MODULE_SUBTASKS),allow_empty=True)
    statuses=[r['status'] for r in MODULE_SUBTASKS]
    if any(x.startswith('TESTED') for x in statuses):
        MODULES['ML-5.2']['status']='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT'
        MODULES['ML-5.2']['reason']='Отдельные сектора: '+', '.join(f"{r['sector']}={r['status']}" for r in MODULE_SUBTASKS)

# ML-5.3: только межрегиональные маршруты из региона получения общего образования.
if len(OUTFLOW_MODEL):test_predictive('ML-5.3',OUTFLOW_MODEL,nonnegative=True,count_confirmed=bool(OUTFLOW_MODEL.actual.ge(0).all() and np.isclose(OUTFLOW_MODEL.actual,np.rint(OUTFLOW_MODEL.actual),atol=1e-8,rtol=0).all()))
else:blocked('ML-5.3','Недостаточно наблюдений агрегированного межрегионального E01',
             'origin schooling region, destination study region, comparable years')

# ML-5.4: реальная программа, не фиктивное происхождение по специальности.
if len(PROGRAM_PROFILE):
    pp=PROGRAM_PROFILE.copy()
    pp['denominator']=pp.groupby(['observation_year','indicator_name']).observed_value.transform('sum')
    pp['relative_share_qualifier']='CONDITIONAL_ON_EXACT_INDICATOR_AND_OBSERVED_PROGRAM_RECORDS'
    pp['share_observed']=np.where(pp.denominator>0,pp.observed_value/pp.denominator,np.nan)
    pp['herfindahl_component']=pp.share_observed**2
    concentration=pp.groupby(['observation_year','indicator_name'],as_index=False).agg(
        reported_programs=('specialty_code','nunique'),reported_value=('observed_value','sum'),
        HHI=('herfindahl_component','sum'))
    qa_table('program_concentration',concentration,allow_empty=True)
    qa_table('program_distribution',pp,allow_empty=True)
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',reason='Программы — условный срез конкретных индикаторов и кодов; OD×programme отсутствует',n=len(pp))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
       'n_observed':len(pp),'reason':'Нельзя прогнозировать выбор программы по OD без OD×programme'})
elif len(VPO_PROGRAM_DESCRIPTION):
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',
        reason='Реальные направления ВПО-1 по категориям и статистическим уровням, без двойного суммирования и ненаблюдаемого OD×programme',
        n=len(VPO_PROGRAM_DESCRIPTION))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
      'n_observed':len(VPO_PROGRAM_DESCRIPTION),'reason':'Представлены медианы исходных разрезов; точные когорты программы не установлены'})
else:limitation('ML-5.4','Доступные словари программ не дали согласованного временного показателя',
                'stable specialty codes, verified target, level and year')

# ML-5.5: полная E04/E05 обрабатывается потоково в DuckDB; не сравнивать preview со всей базой.
if CFG.MODE=='FULL':
    if GRAD_E04_VALID_ROWS>0 and not GRADUATE_GROUP_SUMMARY.empty:
        MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
            reason='Полные E04/E05: статистические группировки всех наблюдений в DuckDB; employment_asof 2024-12-31; RETENTION_NOT_OBSERVABLE',
            n=GRAD_E04_VALID_ROWS)
        MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,
            'n_observed_e04':GRAD_E04_VALID_ROWS,'n_observed_e05':GRAD_E05_VALID_ROWS,
            'employment_asof':'2024-12-31','reason':'Срез на одной референсной дате; место работы не подтверждено'})
    else:
        limitation('ML-5.5','Агрегат E04 по всем строкам пуст после валидации показателя')
elif len(GRADUATES):
    gr=GRADUATES.dropna(subset=['count_graduate']).copy()
    gr=gr[gr.count_graduate.ge(0)]
    gr_summary=gr.groupby(['graduation_year','object_level','gender','education_level'],dropna=False).agg(
       n_source_records=('university','size'),median_reported_graduates=('count_graduate','median'),
       median_observed_salary=('average_salary_fact_med','median'),
       median_employment_percentage=('percent_employed','median')).reset_index()
    gr_summary['aggregate_totals_identified']=False
    qa_table('graduate_descriptive_statistics',gr_summary,allow_empty=True)
    if 'graduate_specialty_outcomes' in PANELS:
        f=PANELS['graduate_specialty_outcomes'].dropna(subset=['graduation_year','specialty_code'])
        f=f[f.count_graduate.notna() & f.count_graduate.ge(0)]
        source_groups=f.groupby(['graduation_year','object_level','education_level','specialty_code'],dropna=False).agg(
          observed_records=('specialty','size'),median_reported_graduates=('count_graduate','median'),
          median_wage=('average_salary_fact_med','median')).reset_index()
        source_groups['aggregate_totals_identified']=False
        qa_table('graduate_specialty_description',source_groups,allow_empty=True)
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
      reason='SMOKE/PILOT: описательная статистика по ограниченным данным; retention неизвестен',n=len(gr))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,'n_observed':len(gr),
      'employment_asof':'2024-12-31','reason':'Одна дата занятости; нет географии работы'})
else:limitation('ML-5.5','E04/E05: нет валидной описательной статистики')

# ML-5.6: динамическая сеть, петли исключены; сетевые изменения с учетом неодинакового покрытия.
NETWORK=[];NETWORK_EDGES=[];COMMUNITIES=[];NET_SETS={}
for year,sub in OD_INTER.dropna(subset=['observed_value']).groupby('observation_year'):
    progress_event('NETWORK',f'Год {year}: расчёт наблюдаемой сети',announce=True)
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if pd.notna(r.observed_value) and r.observed_value>0:
            g.add_edge(str(r.origin_region),str(r.destination_region),weight=float(r.observed_value))
    if nx.number_of_selfloops(g):fail('NETWORK','Обнаружена диагональ в межрегиональном графе')
    if not g.number_of_edges():continue
    pager=nx.pagerank(g,weight='weight')
    undirected=g.to_undirected()
    try:comms=nx.community.louvain_communities(undirected,weight='weight',seed=CFG.RANDOM_SEED)
    except Exception:comms=list(nx.community.greedy_modularity_communities(undirected,weight='weight'))
    membership={node:i for i,community in enumerate(comms) for node in community}
    NET_SETS[int(year)]=set((a,b) for a,b in g.edges())
    for node in g.nodes():
        incoming=g.in_degree(node,weight='weight');outgoing=g.out_degree(node,weight='weight')
        vals=np.asarray([float(d['weight']) for _,_,d in g.out_edges(node,data=True)],float)
        sh=vals/vals.sum() if vals.sum()>0 else np.asarray([],float)
        out_hhi=float((sh*sh).sum()) if len(sh) else np.nan
        out_entropy=float(-(sh*np.log(sh)).sum()) if len(sh) else np.nan
        NETWORK.append({'year':int(year),'region_code':node,'in_strength':incoming,'out_strength':outgoing,
          'outbound_HHI_observed':out_hhi,'outbound_entropy_observed':out_entropy,
          'pagerank':pager.get(node,np.nan),'community_id':membership.get(node,-1),
          'n_in':g.in_degree(node),'n_out':g.out_degree(node),'observed_edges':g.number_of_edges()})
    for a,b,d in g.edges(data=True):NETWORK_EDGES.append({'year':int(year),'origin':a,'destination':b,
        'value':d['weight']})
    COMMUNITIES.append({'year':int(year),'communities':len(comms),
        'n_regions':len(g.nodes()),'n_edges':g.number_of_edges(),
        'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==int(year),'potential_break'].fillna(False).any())})
qa_table('education_network_year',pd.DataFrame(NETWORK),allow_empty=True)
qa_table('education_network_edges',pd.DataFrame(NETWORK_EDGES),allow_empty=True)
qa_table('education_network_communities',pd.DataFrame(COMMUNITIES),allow_empty=True)
if len(NET_SETS)>=2:
    stability=[]
    for a,b in zip(sorted(NET_SETS)[:-1],sorted(NET_SETS)[1:]):
        x,y=NET_SETS[a],NET_SETS[b]
        stability.append({'year_from':a,'year_to':b,'edge_jaccard':len(x&y)/len(x|y) if x|y else np.nan,
                          'n_edges_from':len(x),'n_edges_to':len(y),
                          'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==b,'potential_break'].fillna(False).any())})
    qa_table('network_stability',pd.DataFrame(stability),allow_empty=True)
if len(NETWORK):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_VALID',reason='Сетевые показатели наблюдаемой E01 без самопетель; покрытие лет различается',n=len(NETWORK))
else:limitation('ML-5.6','Нет валидных взвешенных межрегиональных ребер E01')

# ML-5.7: одинаковые муниципалитеты для E0/E1; пропущенный факт нахождения кампуса нельзя считать нулём.
context=MUNI_EDU[MUNI_EDU.education_coverage_known].copy()
if (context.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1 and
        context.observation_year.nunique()>=CFG.MIN_YEARS_FOR_TEMPORAL and
        context.confirmed_vo_sites.nunique()>1):
    x=context[['oktmo','observation_year','migration_rate','population','confirmed_vo_sites']].rename(
      columns={'oktmo':'entity_id','migration_rate':'actual','population':'prev_population'})
    # Не подтягивать число университетов из целевого года: lag, as-of.
    m0=historical_pair_panel(x[['entity_id','observation_year','actual']],['entity_id'],'actual')
    lag=context.rename(columns={'oktmo':'entity_id','observation_year':'historical_year',
                                'confirmed_vo_sites':'confirmed_vo_sites',
                                'population':'prev_population'})
    m0=m0.merge(lag[['entity_id','historical_year','confirmed_vo_sites','prev_population']],
        on=['entity_id','historical_year'],validate='many_to_one',how='left')
    m0['target_id']='municipal_migration_rate';m0['sector']='MUNICIPAL_CONTEXT'
    qa_table('municipal_e0e1_model_frame',m0,allow_empty=True)
    # Прогнозная процедура E0/E1 — одна и та же модель и поддержка; отключаем инфра в E0.
    split,reason=temporal_partition(m0)
    if split:
        tr=split['train'];te=split['test'];features0=['prior_value','prev_population','origin_year']
        features1=features0+['confirmed_vo_sites']
        for version,features in [('E0',features0),('E1',features1)]:
            model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                            ('scaler',StandardScaler()),('reg',Ridge(alpha=20.))])
            model.fit(tr[features],tr.actual)
            predictions=model.predict(te[features])
            record_prediction(predictions_frame(te,'ML-5.7',version,predictions))
            METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_CONTEXT','model':version,'protocol':'COMMON_SUPPORT_MUNICIPAL',
              'seed':-1,'target_year':split['test_year'],**effective_mae(te.actual,predictions)})
            fp=ROOT/'models'/f'ML_5_7_{version}.joblib';joblib.dump({'model':model,'features':features},fp)
            reload=joblib.load(fp)
            assert np.allclose(reload['model'].predict(te[features]),predictions)
            MODEL_FILES[fp.name]=str(fp)
        MODULES['ML-5.7'].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
            reason='E0/E1 на одинаковом подтверждённом составе ОКТМО',n=len(te))
    else:limitation('ML-5.7',reason,'more confirmed municipality-years')
else:
    limitation('ML-5.7',f'Недостаточно подтвержденного кампусного покрытия: {context.oktmo.nunique()} ОКТМО; неизвестно != 0',
               'verified municipality campus registry with historical validity')

# Все семь модулей получают явный предметный статус; ни один не теряется.
for key,result in MODULES.items():
    if result['status']=='NOT_STARTED':
        limitation(key,'Данные просканированы, но применимый протокол не смог сформировать результат')
qa_table('seven_module_status',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
qa_table('module_diagnostics',pd.DataFrame(MODULE_DIAGNOSTICS),allow_empty=True)
print('MODULES:',{k:v['status'] for k,v in MODULES.items()})

memory_point('after_seven_modules')
mark_stage('SEVEN_MODULES')


## 9А. Дополнительные исследования ML-5.2–ML-5.7

Строго отделяются проверенный приём и неутверждённые кандидаты, описательная структура программ и индивидуальный выбор, занятость выпускников и территориальное закрепление. Региональные и муниципальные экспериментальные протоколы запускаются только на проверенной общей области наблюдения. Сетевые показатели проверяются на устойчивость. Отрицательные результаты фиксируются без замены тестовой модели задним числом.

In [ ]:
# ML-05: substantive completion of all seven modules with explicit evidence gates.
progress_event('EXTENDED_MODULES','Дополнительные исследования ML-5.2–ML-5.7 и сетевой стресс-тест',announce=True)
EXT_MODULES=[]

# ML-5.2: distinguish candidates, verified admission, and non-additive section splits.
if 'CANDIDATE_CATALOG' in globals() and len(CANDIDATE_CATALOG):
    cand=CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.semantic_role.eq('ADMISSION_CANDIDATE')].copy()
    sector_rows=cand.groupby('source',dropna=False).agg(indicator_names=('indicator_name','nunique'),
                         rows_in_source=('rows','sum')).reset_index() if len(cand) else pd.DataFrame()
    if len(sector_rows):
        sector_rows['approved_entity_year_targets']=len(ORG_YEAR)
        sector_rows['interpretation']='SOURCE_CANDIDATES_ONLY_NOT_VERIFIED_ADMISSIONS'
    qa_table('ml52_admission_candidates',sector_rows,allow_empty=True)
    if len(sector_rows) and not len(ORG_YEAR):
        MODULES['ML-5.2'].update(status='DESCRIPTIVE_ONLY_TARGET_UNAPPROVED',n=int(sector_rows.rows_in_source.sum()),
          reason='Семантические кандидаты действительно найдены, но утверждённой численности приёма организация×год нет')
        EXT_MODULES.append({'module':'ML-5.2','evidence':'SEMANTIC_CANDIDATES_NOT_COUNTS',
          'n_candidates':int(sector_rows.indicator_names.sum()),'prediction_status':'NOT_FEASIBLE'})

# ML-5.4: whole-dataset specialty histories, not a preview of the first 2,000 aggregate rows.
ML54_HISTORY=pd.DataFrame()
if VPO_FULL_PROGRAM_PATH and Path(VPO_FULL_PROGRAM_PATH).exists():
    p=Path(VPO_FULL_PROGRAM_PATH)
    q='''SELECT observation_year,CAST(specialty_code AS VARCHAR) specialty_code,
      CAST(indicator_name AS VARCHAR) indicator_name,
      SUM(observed_records)::BIGINT rows_in_source,
      COUNT(*)::BIGINT independently_reported_strata,
      MEDIAN(median_reported_value) median_of_source_stratum_medians
      FROM read_parquet(?)
      WHERE specialty_code IS NOT NULL AND TRY_CAST(observation_year AS INT) IS NOT NULL
      GROUP BY 1,2,3'''
    with duck_connect() as con:
        n=int(con.execute('SELECT COUNT(*) FROM ('+q+')',[str(p)]).fetchone()[0])
        if n<=CFG.MAX_PANDAS_ROWS:
            ML54_HISTORY=con.execute(q,[str(p)]).df()
        else:
            out=ROOT/'derived'/'ml54_full_history.parquet'
            con.execute('COPY ('+q.replace('read_parquet(?)','read_parquet('+_sql_literal(p)+')')+
              ') TO '+_sql_literal(out)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
            PANEL_FILES['ml54_specialty_history']=out
            ML54_HISTORY=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(out)]).df()
    qa_table('ml54_specialty_history',ML54_HISTORY,allow_empty=True)
    if n:
        MODULES['ML-5.4'].update(status='DESCRIPTIVE_FULL_SOURCE_NOT_OD_PROGRAMME',n=n,
          reason='Всё содержимое VPO-1 сгруппировано по годам/кодам/показателям; составное медианное значение не равно численности зачисленных')
    EXT_MODULES.append({'module':'ML-5.4','full_specialty_year_indicator_groups':n,
          'forecast_valid':'NO_VERIFIED_ADMISSIONS_TARGET','unit':'source_records_and_medians_not_additive'})

# Exploratory specialty continuity forecast for stable time series (not a verified admission forecast).
# Predictions are directly row counts of source records in a published statistical strata grid.
ML54_TEST=pd.DataFrame()
if len(ML54_HISTORY) and 'rows_in_source' in ML54_HISTORY and 'observation_year' in ML54_HISTORY:
    z=ML54_HISTORY.copy();z=z.dropna(subset=['observation_year','specialty_code','indicator_name'])
    z['observation_year']=pd.to_numeric(z.observation_year,errors='coerce')
    z=z.sort_values(['specialty_code','indicator_name','observation_year'])
    g=z.groupby(['specialty_code','indicator_name'],sort=False)
    z['lag1_rows']=g.rows_in_source.shift(1)
    z['lag1_year']=g.observation_year.shift(1)
    last=int(z.observation_year.max()) if len(z) else 0
    ML54_TEST=z[(z.observation_year.eq(last))&(z.lag1_year.eq(last-1))&z.lag1_rows.notna()].copy()
    if len(ML54_TEST):
        ML54_TEST['abs_error_persistence']=(ML54_TEST.rows_in_source-ML54_TEST.lag1_rows).abs()
        ML54_TEST['measure_definition']='VPO_REPORTING_STRATA_COVERAGE_ONLY'
        qa_table('ml54_history_persistence_diagnostic',ML54_TEST,allow_empty=True)
        EXT_MODULES.append({'module':'ML-5.4','coverage_persistence_year':last,
          'test_cells':len(ML54_TEST),'MAE_source_row_coverage':float(ML54_TEST.abs_error_persistence.mean()),
          'scientific_gate':'NOT_ADMISSIONS_OR_STUDENT_FLOW'})

# ML-5.5: all 4 E04/E05 sources, male/female strata kept separate, no artificial employment location.
GRAD_DISPARITIES=pd.DataFrame()
family_frames=[]
for i in range(1,5):
    name='graduate_family_'+str(i)
    path=PANEL_FILES.get(name)
    if path and Path(path).exists():
        with duck_connect() as con:
            f=con.execute('SELECT * FROM read_parquet(?) LIMIT 30000',[str(path)]).df()
        f['asset_id']=source_asset(segment='05')[i-1]['asset_id']
        family_frames.append(f)
if family_frames:
    GRAD_DISPARITIES=pd.concat(family_frames,ignore_index=True,sort=False)
    GRAD_DISPARITIES['employment_reference']='2024-12-31'
    GRAD_DISPARITIES['outcome_grain']='MEDIAN_OF_PUBLISHED_AGGREGATE_RECORDS'
    qa_table('ml55_outcome_disparities',GRAD_DISPARITIES,allow_empty=True)
    EXT_MODULES.append({'module':'ML-5.5','source_families_analysed':len(family_frames),
        'output_strata':len(GRAD_DISPARITIES),'time_reference':'2024-12-31',
        'retention':'NOT_OBSERVABLE_WITHOUT_WORK_LOCATION'})
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_ALL_FOUR_FAMILIES',
       reason='E04/E05 всех четырёх типов изучены по исходным разрезам, без двойного суммирования; закрепление неизвестно',n=len(GRAD_DISPARITIES))

# ML-5.6: reciprocity, removal robustness, SCC/WCC and stable network coverage.
ML56_STRESS=[]
for yr,sub in pd.DataFrame(NETWORK_EDGES).groupby('year') if len(NETWORK_EDGES) else []:
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if r.value>0 and r.origin!=r.destination:g.add_edge(str(r.origin),str(r.destination),weight=float(r.value))
    if not g.number_of_nodes():continue
    pg=nx.pagerank(g,weight='weight');leading=max(pg,key=pg.get)
    reciprocal=sum(1 for u,v in g.edges() if g.has_edge(v,u))
    w0=len(max(nx.weakly_connected_components(g),key=len))
    s0=len(max(nx.strongly_connected_components(g),key=len))
    h=g.copy();h.remove_node(leading)
    w1=len(max(nx.weakly_connected_components(h),key=len)) if h.number_of_nodes() else 0
    edges_before=g.number_of_edges();edges_after=h.number_of_edges()
    ML56_STRESS.append({'year':int(yr),'n_nodes':g.number_of_nodes(),'n_edges':edges_before,
       'reciprocity_directed_edge_share':reciprocal/edges_before if edges_before else np.nan,
       'largest_weak_component':w0,'largest_strong_component':s0,'removed_top_pagerank_region':leading,
       'weak_component_after_removal':w1,'edges_after_removal':edges_after,
       'lost_edges_percent':100*(edges_before-edges_after)/edges_before if edges_before else np.nan,
       'interpretation':'STRUCTURAL_STRESS_NOT_EMPIRICAL_MIGRATION_DISRUPTION'})
qa_table('ml56_network_stress',pd.DataFrame(ML56_STRESS),allow_empty=True)
if len(ML56_STRESS):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_FULL_NETWORK_STRESS',n=len(NETWORK),
      reason='Динамические сети, PageRank, Louvain, концентрация, reciprocity и устойчивость удаления центрального узла; сопоставимость лет ограничена')
    EXT_MODULES.append({'module':'ML-5.6','n_years':len(ML56_STRESS),'validation':'OBSERVED_EDGES_ONLY'})

# ML-5.7: BDMO education-infrastructure proxy E0/E1, same OKTMO-year rows, no fabricated university locations.
ML57_SUPPORT=pd.DataFrame();ML57_RESULT={}
if BDMO_PROXY_FULL_VALID and len(BDMO_EDU_PROXY_CANDIDATES) and len(MUNI):
    e=BDMO_EDU_PROXY_CANDIDATES.dropna(subset=['oktmo','observation_year','indicator_code','median_reported']).copy()
    e['oktmo']=norm_code(e.oktmo);e['observation_year']=safe_year(e.observation_year)
    e=e[e.n_raw_rows.eq(1)&e.median_reported.ge(0)].copy()
    if len(e):
        mig=MUNI[['oktmo','observation_year','migration_rate','population']].drop_duplicates(['oktmo','observation_year']).copy()
        mig['observation_year']=safe_year(mig.observation_year)
        # only same historically observed municipality, lookback=one year at origin.
        mig=mig.sort_values(['oktmo','observation_year'])
        mig['prev_year']=mig.observation_year-1
        past=mig[['oktmo','observation_year','migration_rate','population']].rename(
             columns={'observation_year':'prev_year','migration_rate':'lag_migration_rate','population':'lag_population'})
        mig=mig.merge(past,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
        for c in ('migration_rate','lag_migration_rate','lag_population'):mig[c]=number(mig[c])
        last=int(mig.observation_year.max()) if len(mig) else None
        # Candidate selection entirely on PRE-TEST historical years; never on test labels.
        prior=e[e.observation_year.lt(last)] if last is not None else e.iloc[:0]
        stats=prior.groupby('indicator_code',dropna=False).agg(n_muni=('oktmo','nunique'),
           n_years=('observation_year','nunique'),n_rows=('median_reported','size'),
           n_distinct=('median_reported','nunique')).reset_index()
        stats=stats[(stats.n_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
                    (stats.n_years>=CFG.MIN_YEARS_FOR_TEMPORAL)&(stats.n_distinct>1)]
        if len(stats):
            stats=stats.sort_values(['n_muni','n_years','n_rows','indicator_code'],ascending=[False,False,False,True])
            selected=str(stats.iloc[0].indicator_code)
            x=e[e.indicator_code.astype(str).eq(selected)][['oktmo','observation_year','median_reported']].copy()
            x=x.rename(columns={'observation_year':'prev_year','median_reported':'lag_education_proxy'})
            # municipality's published education indicator is used at t-1 only, not t.
            ML57_SUPPORT=mig.merge(x,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
            ML57_SUPPORT=ML57_SUPPORT.dropna(subset=['migration_rate','lag_migration_rate',
                               'lag_population','lag_education_proxy'])
            ML57_SUPPORT['indicator_code']=selected
            ML57_SUPPORT['source_definition']='BDMO_PUBLISHED_EDUCATION_PROXY_NOT_VERIFIED_CAMPUS'
            qa_table('ml57_common_support',ML57_SUPPORT,allow_empty=True)
            tr=ML57_SUPPORT[ML57_SUPPORT.observation_year.lt(last)].copy()
            te=ML57_SUPPORT[ML57_SUPPORT.observation_year.eq(last)].copy()
            if (len(tr)>=max(50,CFG.REGION_MIN_TRAIN_ROWS) and
                len(te)>=CFG.MIN_MUNICIPALITIES_E0E1 and tr.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1):
                # Identical E0/E1 keys and model, change ONLY historic education covariate.
                estimates={};baselines=te.lag_migration_rate.to_numpy(float)
                baseline_score=effective_mae(te.migration_rate,baselines)
                for vers,features in [('E0',['lag_migration_rate','lag_population']),
                                      ('E1',['lag_migration_rate','lag_population','lag_education_proxy'])]:
                    model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                    model.fit(tr[features].astype(float),tr.migration_rate.to_numpy(float))
                    pred=model.predict(te[features].astype(float));estimates[vers]=effective_mae(te.migration_rate,pred)
                    fp=ROOT/'models'/('ML_5_7_proxy_'+vers+'.joblib')
                    artifact={'pipeline':model,'features':features,'proxy_indicator_code':selected}
                    joblib.dump(artifact,fp,compress=3);rp=joblib.load(fp)['pipeline'].predict(te[features].astype(float))
                    if not np.allclose(rp,pred):fail('ML-5.7','Model serialization discrepancy')
                    MODEL_FILES[fp.name]=str(fp)
                    MODEL_RECORDS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                      'seed':-1,'path':fp.name,'reloaded':True,'training_rows':len(tr),'test_rows':len(te),
                      'feature_num':features,'feature_cat':[],'file_sha256':hash_file(fp)})
                    out=te[['oktmo','observation_year','migration_rate','prev_year']].copy()
                    out=out.rename(columns={'oktmo':'entity_id','observation_year':'target_year',
                         'migration_rate':'actual','prev_year':'origin_year'})
                    out['information_cutoff_year']=out.origin_year
                    out['feature_available_year']=out.origin_year
                    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
                    out['target_id']='D01_MUNICIPAL_MIGRATION_RATE'
                    out['sector']='MUNICIPAL_BDMO_PROXY'
                    pf=predictions_frame(out,'ML-5.7',vers,pred,seed=-1,protocol='RETROSPECTIVE_COMMON_SUPPORT')
                    pf['geography_grain']='MUNICIPALITY';record_prediction(pf)
                    METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                       'protocol':'RETROSPECTIVE_COMMON_SUPPORT','seed':-1,'target_year':last,
                       'source_vintage':INDEX['created_at'],**estimates[vers]})
                ML57_RESULT={'status':'RETROSPECTIVE_PROXY_E0E1','year':last,'n_train':len(tr),
                  'n_test':len(te),'n_muni_test':te.oktmo.nunique(),'indicator_code':selected,
                  'inertia_MAE':baseline_score['MAE'],'e0_MAE':estimates['E0']['MAE'],
                  'e1_MAE':estimates['E1']['MAE'],'delta_MAE_E1_minus_E0':estimates['E1']['MAE']-estimates['E0']['MAE']}
                MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',n=len(te),
                    reason='E0/E1 на общих ОКТМО с опубликованным BDMO-показателем: не доказана фактическая образовательная площадка; причинные выводы запрещены')
if not ML57_RESULT:
    ML57_RESULT={'status':'NOT_FEASIBLE_WITH_VERIFIED_GEOGRAPHY',
      'reason':'Недостаточная муниципальная область наблюдения или отсутствующий сопоставимый показатель инфраструктуры'}
EXT_MODULES.append({'module':'ML-5.7',**ML57_RESULT})
# ML-5.1 generalization checks: both time and territory blocked, no test-based hyperparameter selection.
SPATIAL_MODEL_RESULTS=[]
if len(OD_MODEL):
    frame=OD_MODEL.copy()
    frame['module']='ML-5.1'
    years=sorted(int(y) for y in frame.target_year.dropna().unique())
    if len(years)>=2:
        ty=years[-1];origin=ty-CFG.HORIZON
        origins=sorted(frame.origin_region.dropna().astype(str).unique())
        destinations=sorted(frame.destination_region.dropna().astype(str).unique())
        def stable_holdout(code,salt):
            return int(hashlib.sha256((salt+'|'+str(code)).encode()).hexdigest()[:8],16)%5==0
        oo={r for r in origins if stable_holdout(r,'ML5_ORIGIN_HOLDOUT')}
        dd={r for r in destinations if stable_holdout(r,'ML5_DEST_HOLDOUT')}
        protocols=[('TERRITORY_ONLY',frame.origin_region.astype(str).isin(oo),
                    ~frame.origin_region.astype(str).isin(oo)),
                   ('CROSSED',frame.origin_region.astype(str).isin(oo)&frame.destination_region.astype(str).isin(dd),
                    ~frame.origin_region.astype(str).isin(oo)&~frame.destination_region.astype(str).isin(dd))]
        for protocol,is_test,is_train in protocols:
            tr=frame[frame.target_year.lt(ty)&frame.target_available_year.le(origin)&is_train].copy()
            te=frame[frame.target_year.eq(ty)&is_test].copy()
            meta={'module':'ML-5.1','protocol':protocol,'train_rows':len(tr),'test_rows':len(te),
                  'n_origins':te.origin_region.nunique(),'n_destinations':te.destination_region.nunique(),
                  'date_origin':origin,'test_year':ty,'selection_basis':'PREDECLARED_GEOGRAPHIC_HASH'}
            if len(tr)<CFG.MIN_TRAIN_ROWS or len(te)<CFG.MIN_TEST_ROWS:
                meta['status']='NOT_FEASIBLE_SUPPORT';SPATIAL_MODEL_RESULTS.append(meta);continue
            # Persist the baseline on precisely the same test keys as each spatial model.
            pbase=fit_and_record('ML-5.1','persistence',tr,te,seed=-1,
                                 nonnegative=True,protocol=protocol)
            reference=effective_mae(te.actual,pbase)
            meta['baseline_mae']=reference['MAE'];meta['status']='TESTED_RETROSPECTIVE_EXPLORATORY'
            # Single fixed seed; no TEST choice; same held-out keys for all models.
            for model in ('ridge','elasticnet','lightgbm'):
                try:
                    yp=fit_and_record('ML-5.1',model,tr,te,seed=CFG.RANDOM_SEED,
                                     nonnegative=True,allow_ppml=False,protocol=protocol)
                    mt=effective_mae(te.actual,yp)
                    meta[model+'_MAE']=mt['MAE'];meta[model+'_minus_baseline']=mt['MAE']-reference['MAE']
                except (ValueError,RuntimeError) as ex:
                    ISSUES.append({'module':'ML-5.1','status':'SPATIAL_MODEL_FAILURE',
                        'reason':protocol+' '+model+': '+type(ex).__name__,'required':'inspect spatial support'})
            SPATIAL_MODEL_RESULTS.append(meta)
qa_table('ml51_spatial_protocols',pd.DataFrame(SPATIAL_MODEL_RESULTS),allow_empty=True)
if len(SPATIAL_MODEL_RESULTS):
    EXT_MODULES.append({'module':'ML-5.1','protocols':len(SPATIAL_MODEL_RESULTS),
      'tested':sum(x['status'].startswith('TESTED') for x in SPATIAL_MODEL_RESULTS),
      'interpretation':'SPATIAL_GENERALIZATION_RETROSPECTIVE_CAVEATS'})

qa_table('module_52_extension',pd.DataFrame(EXT_MODULES),allow_empty=True)
qa_table('seven_module_status_final',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
print('Extended modules:',[(k,v['status']) for k,v in MODULES.items()])
# Финальная отметка этапа перенесена после всех его подмодулей


## 9Б. Региональный приём по специальностям и вузовский контингент
Подтверждённые количества по ВПО-1, отличные от отсутствующего приёма организации.

In [ ]:
# V4: measured organization outcomes and complete regional specialty admissions, with separate scientific targets.
progress_event('EXTENDED_MODULES','V4: реальный приём по специальностям ВПО-1; организация — отдельная величина',announce=True)
V4_SUBEXPERIMENTS=[]

def v4_block(name,reason,required='',rows=0):
    V4_SUBEXPERIMENTS.append({'subexperiment':name,'status':'NOT_FEASIBLE_FOR_TARGET',
       'reason':reason,'required':required,'n':int(rows)})
    note('V4_DATA_GATE',name+': '+reason,'WARN')

# Source VPO_2_1_1 has exact textual 'Принято, всего'. This is NOT a campus/branch observation.
_v4_vpo_p=fetch_asset(VPO_211)
V4_ADMISSION_PATH=ROOT/'derived'/'v4_regional_specialty_admissions.parquet'
_v4_s=f'''SELECT TRY_CAST(year AS INTEGER) observation_year, CAST(object_level AS VARCHAR) object_level,
    CAST(object_name AS VARCHAR) object_name, CAST(specialty_code AS VARCHAR) specialty_code,
    CAST(education_level AS VARCHAR) education_level,CAST(education_form AS VARCHAR) education_form,
    CAST(admission_type AS VARCHAR) admission_type,CAST(university_type AS VARCHAR) university_type,
    COUNT(*)::BIGINT records_in_cell,MIN(TRY_CAST(indicator_value AS DOUBLE)) minimum,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) maximum
  FROM read_parquet({_sql_literal(_v4_vpo_p)})
  WHERE TRIM(CAST(indicator_name AS VARCHAR))='Принято, всего'
    AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
    AND specialty_code IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5,6,7,8'''
_v4_tmp=V4_ADMISSION_PATH.with_suffix('.partial.parquet');_v4_tmp.unlink(missing_ok=True)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='2048MB'");_db.execute('SET threads=1')
    _db.execute('SET temp_directory='+_sql_literal(DB_TMP));_db.execute('SET preserve_insertion_order=false')
    _db.execute('COPY ('+_v4_s+') TO '+_sql_literal(_v4_tmp)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
    _v4_counts=_db.execute('''SELECT COUNT(*),COUNT(*) FILTER (WHERE records_in_cell=1 AND minimum=maximum
      AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8),COUNT(DISTINCT observation_year)
      FROM read_parquet(?)''',[str(_v4_tmp)]).fetchone()
_v4_tmp.replace(V4_ADMISSION_PATH);CONTEXT_PARQUET_FILES.append(V4_ADMISSION_PATH)
qa_table('v4_specialty_admission_gate',pd.DataFrame([{'source_id':VPO_211['asset_id'],
   'full_source_rows':int(VPO_211['rows_actual']),'exact_name':'Принято, всего',
   'regional_specialty_strata':int(_v4_counts[0]),'valid_singleton_nonnegative_integer_strata':int(_v4_counts[1]),
   'years':int(_v4_counts[2]),'source_grain':'REGION_X_SPECIALTY_X_EDUCATION_LEVEL_X_FORM_X_ADMISSION_TYPE_X_UNIVERSITY_TYPE',
   'warning':'NOT_ORGANIZATION_ADMISSIONS_AND_NOT_OD_BY_SPECIALTY'}]))
V4_PROGRAM_FRAME=pd.DataFrame()
if int(_v4_counts[1])>0:
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _v4_q='''SELECT * FROM read_parquet(?) WHERE records_in_cell=1 AND minimum=maximum
             AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8
             AND observation_year IS NOT NULL AND specialty_code!='' '''
        _v4_n=int(_db.execute('SELECT COUNT(*) FROM ('+_v4_q+')',[str(V4_ADMISSION_PATH)]).fetchone()[0])
        if CFG.MODE=='FULL' and CFG.ML54_FULL_NO_STRATA_CAP:
            # ALL verified strata (not 'longest 15000'); 237k observed in V4.
            # Pandas materialization is restricted to verified regional/specialty cells,
            # not the 7.5 million source rows. Never truncate study population.
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        elif _v4_n<=CFG.MAX_PANDAS_ROWS:
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        else:
            raise RuntimeError('ML54_FULL_STRATA_REQUIRED: no discretionary sample allowed')
    if int(_v4_n)!=int(len(V4_PROGRAM_FRAME)):
        raise RuntimeError(f'ML54_COVERAGE_LOSS: eligible={_v4_n}, materialized={len(V4_PROGRAM_FRAME)}')
    qa_table('v5_ml54_full_strata_coverage',pd.DataFrame([{
        'verified_rows':int(_v4_n),'materialized_rows':int(len(V4_PROGRAM_FRAME)),
        'fraction':1.0,'bounded_sample_used':False}]))
if len(V4_PROGRAM_FRAME):
    _f=V4_PROGRAM_FRAME.copy()
    for _c in ['object_name','specialty_code','education_level','education_form','admission_type','university_type']:
        _f[_c]=_f[_c].fillna('__MISSING__').astype(str)
    _fields=['object_name','specialty_code','education_level','education_form','admission_type','university_type']
    _f['entity_id']=_f[_fields].agg('|'.join,axis=1)
    _f['value']=_f['minimum'].astype(float)
    # A unique reporting stratum must not be interpreted as the sum of its components.
    _f=_f.drop_duplicates(['entity_id','observation_year'],keep=False)
    _f=_f[['entity_id','observation_year','value','object_name','specialty_code','education_level']]
    _rows=historical_pair_panel(_f[['entity_id','observation_year','value']],['entity_id'],'value')
    if len(_rows):
        _rows['sector']='VO_REGIONAL_SPECIALTY'
        _rows['target_id']='VPO_2_1_1_REPORTED_ADMITTED_EXACT_STRATUM'
        _rows['geography_grain']='REPORTED_REGION_SPECIALTY'
        qa_table('v4_specialty_model_frame',_rows,allow_empty=True)
        test_predictive('ML-5.4',_rows,nonnegative=True,count_confirmed=True)
        if MODULES['ML-5.4']['status'].startswith('TESTED'):
            MODULES['ML-5.4']['reason']='Exact VPO admitted value regional specialty/level/form/admission_type stratum; not OD and not institution-level'
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS',
               'status':'MODEL_TESTED','n':len(_rows),'source':VPO_211['asset_id']})
        else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No historically available train/test for exact cells',rows=len(_rows))
    else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','Too few historical observations under publication lag',rows=len(_f))
else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No unique regional speciality admission strata',rows=int(_v4_counts[0]))
del V4_PROGRAM_FRAME;gc.collect()

# ML-5.2: do not auto-approve a number of admitted students from name-matched E02.
# A separate observed 'Приведенный контингент студентов' is a permissible alternative target,
# but it CANNOT be reported as organization admissions or interregional entrants.
_v4_e02_p=fetch_asset(E02)
_v4_org_target='Приведенный контингент студентов'
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1024MB'");_db.execute('SET threads=1')
    _org=_db.execute('''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
        COUNT(*)::BIGINT n_source_records,COUNT(DISTINCT CAST(area AS VARCHAR)) n_area,
        MIN(TRY_CAST(indicator_value AS DOUBLE)) low_value,MAX(TRY_CAST(indicator_value AS DOUBLE)) high_value
        FROM read_parquet(?) WHERE indicator_name=? AND id IS NOT NULL AND year IS NOT NULL
        GROUP BY 1,2''',[str(_v4_e02_p),_v4_org_target]).df()
qa_table('v4_organization_contingent_candidates',_org,allow_empty=True)
_org=_org[_org.n_source_records.eq(1)&_org.low_value.notna()&_org.low_value.eq(_org.high_value)&_org.low_value.ge(0)].copy()
if len(_org):
    _org['entity_id']='VO|'+norm_code(_org.id)
    _observed=_org[['entity_id','observation_year','low_value']].rename(columns={'low_value':'value'})
    _observed=_observed.drop_duplicates(['entity_id','observation_year'],keep=False)
    _panel=historical_pair_panel(_observed,['entity_id'],'value')
    if len(_panel):
        _panel['sector']='VO';_panel['target_id']='E02_REPORTED_STUDENT_CONTINGENT_NOT_ADMISSIONS'
        _panel['parent_group_id']=_panel['entity_id'];_panel['geography_grain']='ORGANIZATION'
        _split,_why=temporal_partition(_panel,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _split is not None:
            _tr=_split['train'];_te=_split['test']
            for _model in ('persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'):
                fit_and_record('ML-5.2',_model,_tr,_te,seed=CFG.RANDOM_SEED,nonnegative=True,
                  protocol='ALT_CONTINGENT_RETROSPECTIVE')
            qa_table('v4_organization_contingent_model_frame',_panel,allow_empty=True)
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.2_ALTERNATIVE_CONTINGENT','status':'MODEL_TESTED_NOT_ADMISSIONS',
                                     'n':len(_te),'target':_v4_org_target})
            MODULES['ML-5.2'].update(status='ALTERNATIVE_TARGET_TESTED_ADMISSIONS_NOT_FEASIBLE',
                  reason='E02 student contingent model evaluated; original number admitted still unavailable for organization × year',n=len(_te))
        else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT',str(_why),rows=len(_panel))
    else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','Insufficient stable organization/year contingent panel',rows=len(_org))
else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','No unique organization/year source value; cannot sum area splits',rows=len(_org))
v4_block('ML-5.2_ORGANIZATION_ADMISSIONS','No certified admitted-count × organization × year in Stage I; student contingent is not admissions',
         'exact admission count, organization/branch identifiers, year and verified indicator/units') if len(ORG_YEAR)==0 else None
qa_table('v4_additional_subexperiments',pd.DataFrame(V4_SUBEXPERIMENTS),allow_empty=True)
progress_event('EXTENDED_MODULES','V4: проверка фактических величин ВПО/ВО завершена',announce=True)


## 9Г. ML-5.2 · углублённый временной эксперимент по организации и образовательной страте

Целевая величина — реально опубликованный контингент Е02. Это **не** число зачисленных; наблюдения разных страт не суммируются и не выдаются за организационный итог.

In [ ]:
# V5 · дополнительный строгий эксперимент ML-5.2: контингент организация × образовательная страта.
# Не равно приёму, не суммируется по перекрывающимся направлениям, не подменяет ML-5.2 admission target.
progress_event('EXTENDED_MODULES','V5 ML-5.2: проверка 106 тыс. исходных наблюдений по образовательным стратам',announce=True)
V5_ORG_AREA_STATUS={'status':'NOT_FEASIBLE_SOURCE_GRAIN','source':E02['asset_id']}
_v5_52_file=fetch_asset(E02)
_v5_52_sql='''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
    COALESCE(CAST(area AS VARCHAR),'__NA__') area,
    COALESCE(CAST(specialty_code AS VARCHAR),'__NA__') specialty_code,
    COALESCE(CAST(specialty_section AS VARCHAR),'__NA__') specialty_section,
    COALESCE(CAST(indicator_unit AS VARCHAR),'__NA__') indicator_unit,
    COUNT(*)::BIGINT n_original_records,
    MIN(TRY_CAST(indicator_value AS DOUBLE)) min_value,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) max_value
    FROM read_parquet(?) WHERE TRIM(CAST(indicator_name AS VARCHAR))='Приведенный контингент студентов'
      AND id IS NOT NULL AND year IS NOT NULL GROUP BY 1,2,3,4,5,6'''
with duckdb.connect(':memory:') as _con:
    _con.execute("SET memory_limit='1536MB'");_con.execute('SET threads=1')
    _v5_52_raw=_con.execute(_v5_52_sql,[str(_v5_52_file)]).df()
_v5_52_eligible=_v5_52_raw.loc[
    _v5_52_raw.n_original_records.eq(1)&_v5_52_raw.min_value.notna()&
    _v5_52_raw.min_value.eq(_v5_52_raw.max_value)&_v5_52_raw.min_value.ge(0)].copy()
V5_ORG_AREA_STATUS.update(raw_strata=len(_v5_52_raw),uniquely_reported_strata=len(_v5_52_eligible),
    duplicate_or_ambiguous_strata=int(len(_v5_52_raw)-len(_v5_52_eligible)))
if len(_v5_52_eligible):
    # The full combination, including area and specialty, is the statistical entity.
    _v5_52_eligible['entity_id']='VOAREA|'+_v5_52_eligible[
        ['id','area','specialty_code','specialty_section','indicator_unit']].astype(str).agg(
        lambda row:hashlib.sha256(json.dumps(row.tolist(),ensure_ascii=False).encode()).hexdigest()[:24],axis=1)
    _v5_52_eligible=_v5_52_eligible.drop_duplicates(['entity_id','observation_year'],keep=False)
    _v5_52_obs=_v5_52_eligible[['entity_id','observation_year','min_value']].rename(columns={'min_value':'value'})
    _v5_52_panel=historical_pair_panel(_v5_52_obs,['entity_id'],'value')
    V5_ORG_AREA_STATUS['complete_source_entity_year_rows']=len(_v5_52_obs)
    V5_ORG_AREA_STATUS['eligible_lagged_rows']=len(_v5_52_panel)
    qa_table('v5_ml52_organization_area_source_strata',_v5_52_eligible,allow_empty=True)
    if len(_v5_52_panel):
        _v5_52_panel['sector']='VO_CONTINGENT_AREA';_v5_52_panel['target_id']='E02_CONTINGENT_ORGANIZATION_AREA_NOT_ADMISSIONS'
        _v5_52_panel['parent_group_id']=_v5_52_panel.entity_id
        _v5_52_panel['geography_grain']='ORGANIZATION_AREA_SPECIALTY'
        _v5_52_split,_v5_52_reason=temporal_partition(_v5_52_panel)
        if _v5_52_split is not None:
            progress_event('EXTENDED_MODULES','ML-5.2: полный разведочный временной эксперимент по контингенту',announce=True)
            test_predictive('ML-5.2',_v5_52_panel,nonnegative=True,count_confirmed=False)
            if MODULES['ML-5.2']['status'].startswith('TESTED'):
                MODULES['ML-5.2'].update(status='ALTERNATIVE_ORG_AREA_CONTINGENT_TESTED_NOT_ADMISSIONS',
                    reason='Observed E02 student contingent by organization/area/specialty; NOT admitted cohorts, not institutional total')
                V5_ORG_AREA_STATUS.update(status='ALTERNATIVE_RETROSPECTIVE_TESTED_NOT_ADMISSIONS',
                    n_test=int(len(_v5_52_split['test'])),target_year=int(_v5_52_split['test_year']))
        else:
            V5_ORG_AREA_STATUS.update(status='NOT_FEASIBLE_HISTORICAL_SUPPORT',reason=str(_v5_52_reason))
else:
    V5_ORG_AREA_STATUS['reason']='No uniquely reported organization-area-year source cell, no admissible target'
qa_table('v5_ml52_area_model_execution',pd.DataFrame([V5_ORG_AREA_STATUS]),allow_empty=True)
print('ML-5.2 alternative:',V5_ORG_AREA_STATUS['status'],'| scientific admission target still gated')
del _v5_52_raw,_v5_52_eligible
import gc;gc.collect()


## 9В. E0/E1 по муниципальной образовательной инфраструктуре и выпускники
Полный дисковый SQL по БДМО; отдельный групповой тест занятости без выводов о закреплении.

In [ ]:
# V4. ML-5.7 disk-backed E0/E1; ML-5.5 independently held-out geography test.
progress_event('EXTENDED_MODULES','V4: E0/E1 полный муниципальный SQL и выпускники E04',announce=True)
V4_MUNICIPAL_STATUS={};V4_GRADUATE_STATUS={}
_v4_bdmo_file=ROOT/'derived'/'full52_context'/'bdmo_education_municipal.parquet'
if CFG.MODE=='FULL' and _v4_bdmo_file.is_file():
    _mi=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _mi['oktmo']=norm_code(_mi.oktmo)
    for _c in ('migration_rate','population'):_mi[_c]=number(_mi[_c])
    _mi=_mi.dropna(subset=['oktmo','observation_year','migration_rate'])
    _mi=_mi.drop_duplicates(['oktmo','observation_year'],keep=False)
    _mi['target_year']=_mi.observation_year.astype(int)
    _mi['origin_year']=_mi.target_year-CFG.HORIZON
    _mi['feature_year']=_mi.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _hist=_mi[['oktmo','observation_year','migration_rate','population']].rename(columns={
        'observation_year':'feature_year','migration_rate':'lag_migration_rate','population':'lag_population'})
    _mi=_mi.merge(_hist,on=['oktmo','feature_year'],validate='many_to_one',how='inner')
    _mi=_mi.dropna(subset=['lag_migration_rate','lag_population'])
    _latest=int(_mi.target_year.max()) if len(_mi) else 0
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
        _db.register('m',_mi)
        _q='''SELECT m.oktmo,m.target_year,m.origin_year,m.feature_year,m.migration_rate actual,
            m.lag_migration_rate,m.lag_population,CAST(e.indicator_code AS VARCHAR) indicator_code,
            CAST(e.indicator_name AS VARCHAR) indicator_name,CAST(e.indicator_unit AS VARCHAR) indicator_unit,
            e.median_reported education_historic_value
            FROM m JOIN read_parquet(?) e ON CAST(m.oktmo AS VARCHAR)=CAST(e.oktmo AS VARCHAR)
               AND m.feature_year=e.observation_year
            WHERE e.n_raw_rows=1 AND e.min_reported=e.max_reported AND e.median_reported>=0
              AND e.indicator_code IS NOT NULL'''
        _matched=ROOT/'derived'/'v4_municipality_education_exact_support.parquet'
        _db.execute('COPY ('+_q.replace('read_parquet(?)','read_parquet('+_sql_literal(_v4_bdmo_file)+')')+
                    ') TO '+_sql_literal(_matched)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
        # Indicators are selected by training coverage, unit constancy and values, not TEST errors.
        _stats=_db.execute('''SELECT indicator_code,indicator_unit,MIN(indicator_name) AS example_indicator_name,
            COUNT(*) FILTER (WHERE target_year<=?) n_train_candidates,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year<=?) n_municipalities_train,
            COUNT(DISTINCT target_year) FILTER (WHERE target_year<=?) n_years_train,
            COUNT(DISTINCT education_historic_value) FILTER (WHERE target_year<=?) n_distinct_train,
            COUNT(*) FILTER (WHERE target_year=?) n_test_coverage,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year=?) n_municipalities_test,
            COUNT(DISTINCT indicator_name) n_indicator_names,
            COUNT(*) as rows_all
            FROM read_parquet(?) GROUP BY 1,2''',
          [_latest-2]*4+[_latest]*2+[str(_matched)]).df()
        _stats=_stats.sort_values(['n_municipalities_train','n_test_coverage','indicator_code'],ascending=[False,False,True])
        qa_table('v4_ml57_indicator_support',_stats,allow_empty=True)
        _eligible=_stats.loc[(_stats.n_train_candidates>=50)&(_stats.n_municipalities_train>=CFG.MIN_MUNICIPALITIES_E0E1)&
            (_stats.n_years_train>=2)&(_stats.n_distinct_train>1)&
            (_stats.n_municipalities_test>=CFG.MIN_MUNICIPALITIES_E0E1)&(_stats.n_indicator_names==1)]
        if len(_eligible):
            _eligible=_eligible.copy()
            _eligible['is_professional_education']=_eligible.example_indicator_name.astype(str).str.contains(
                r'профессиональн|высш|колледж|техникум|университет',case=False,regex=True,na=False)
            _eligible=_eligible.sort_values(['is_professional_education','n_municipalities_train','n_test_coverage','indicator_code'],
                ascending=[False,False,False,True])
            _one=_eligible.iloc[0];_selected=str(_one.indicator_code);_unit=str(_one.indicator_unit)
            V4_MUNICIPAL_INDICATOR_SEMANTIC='PROFESSIONAL_EDUCATION_CONTEXT' if bool(_one.is_professional_education) else 'GENERAL_EDUCATION_PROXY_ONLY'
            _rows=_db.execute('''SELECT * FROM read_parquet(?) WHERE indicator_code=?
                  AND COALESCE(indicator_unit,'')=COALESCE(?,'')''',[str(_matched),_selected,_unit]).df()
        else:_rows=pd.DataFrame();_selected=None
    CONTEXT_PARQUET_FILES.append(_matched)
    V4_MUNICIPAL_STATUS={'candidate_indicators':len(_stats),'matched_municipality_indicator_year_rows':int(pq.ParquetFile(_matched).metadata.num_rows),
       'target_year':_latest,'feasible_indicators':len(_eligible),'selected_code':_selected,'selected_semantic':locals().get('V4_MUNICIPAL_INDICATOR_SEMANTIC','UNAVAILABLE')}
    if len(_rows):
        _rows=_rows.sort_values(['oktmo','target_year','indicator_code'])
        _rows=_rows.drop_duplicates(['oktmo','target_year'],keep=False)
        _tr=_rows[_rows.target_year.le(_latest-2)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        _te=_rows[_rows.target_year.eq(_latest)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        if len(_tr)>=50 and len(_te)>=CFG.MIN_MUNICIPALITIES_E0E1:
            # Historical features are at target_year - 1 - publication lag; not current-year infrastructure.
            assert (_tr.feature_year<_tr.target_year).all() and (_te.feature_year<=_te.origin_year).all()
            assert (_tr.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS<=_te.origin_year.min()).all()
            qa_table('v4_ml57_exact_common_support',_rows,allow_empty=True)
            _p0=['lag_migration_rate','lag_population'];_p1=_p0+['education_historic_value']
            _results=[]
            for _label,_features in [('E0',_p0),('E1',_p1)]:
                _mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _mod.fit(_tr[_features].to_numpy(float),_tr.actual.to_numpy(float))
                _yp=_mod.predict(_te[_features].to_numpy(float))
                _f=ROOT/'models'/f'v4_ml57_{_label}.joblib'
                joblib.dump({'model':_mod,'features':_features,'indicator_code':_selected},_f,compress=3)
                _loaded=joblib.load(_f)
                assert np.allclose(_loaded['model'].predict(_te[_features].to_numpy(float)),_yp)
                MODEL_FILES[_f.name]=str(_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,'seed':-1,
                  'path':_f.name,'file_sha256':hash_file(_f),'feature_num':_features,'feature_cat':[],
                  'training_rows':len(_tr),'test_rows':len(_te),'reloaded':True})
                _p=pd.DataFrame({'entity_id':_te.oktmo.astype(str).to_numpy(),
                  'origin_year':_te.origin_year.to_numpy(),'target_year':_te.target_year.to_numpy(),
                  'information_cutoff_year':_te.origin_year.to_numpy(),
                  'feature_available_year':_te.origin_year.to_numpy(),
                  'target_available_year':_te.target_year.to_numpy()+CFG.LABEL_PUBLICATION_LAG_YEARS,
                  'sector':'BDMO_EDUCATION_PROXY','target_id':'D01_MUNICIPAL_MIGRATION_RATE',
                  'actual':_te.actual.to_numpy()})
                _pred=predictions_frame(_p,'ML-5.7',_label,_yp,protocol='RETROSPECTIVE_PROXY_E0E1')
                _pred['geography_grain']='MUNICIPALITY';record_prediction(_pred)
                _mt=effective_mae(_te.actual,_yp)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,
                    'protocol':'RETROSPECTIVE_PROXY_E0E1','seed':-1,'target_year':_latest,
                    'source_vintage':INDEX['created_at'],**_mt})
                _results.append({'variant':_label,'n_test':len(_te),'n_train':len(_tr),'municipalities_test':_te.oktmo.nunique(),**_mt})
            qa_table('v4_ml57_results',pd.DataFrame(_results),allow_empty=True)
            V4_MUNICIPAL_STATUS.update(status='E0_E1_TESTED_PROXY_NOT_VERIFIED_CAMPUS',
                 n_train=len(_tr),n_test=len(_te),MAE_E0=_results[0]['MAE'],MAE_E1=_results[1]['MAE'])
            MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',
                  reason='BDMO lagged '+V4_MUNICIPAL_INDICATOR_SEMANTIC+' E0/E1; same OKTMO-year support, DOES NOT verify campus or causal effect',n=len(_te))
        else:V4_MUNICIPAL_STATUS.update(status='NOT_FEASIBLE_TEMPORAL_SUPPORT',train=len(_tr),test=len(_te))
    else:V4_MUNICIPAL_STATUS['status']='NOT_FEASIBLE_EXACT_SHARED_SUPPORT'
else:V4_MUNICIPAL_STATUS={'status':'NOT_FEASIBLE_NO_BDMO_FULL_FILE'}
qa_table('v4_ml57_execution',pd.DataFrame([V4_MUNICIPAL_STATUS]),allow_empty=True)

# ML-5.5: out-of-region validation of employment rate on E04's original reporting rows.
# This is deliberately cross-sectional, not a forecast of retention or postgraduation residence.
progress_event('EXTENDED_MODULES','V4: трудоустройство E04, проверка вне регионов обучения',announce=True)
_gpath=fetch_asset(GRAD_ORG)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
    _gg=_db.execute('''SELECT CAST(object_name AS VARCHAR) region_name,
      CAST(object_level AS VARCHAR) object_level,CAST(university AS VARCHAR) university,
      CAST(gender AS VARCHAR) gender,CAST(education_level AS VARCHAR) education_level,
      TRY_CAST(year AS INTEGER) graduation_year,
      TRY_CAST(count_graduate AS DOUBLE) count_graduate,
      TRY_CAST(percent_employed AS DOUBLE) employed_pct
      FROM read_parquet(?) WHERE regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
      AND TRY_CAST(percent_employed AS DOUBLE) BETWEEN 0 AND 100
      AND TRY_CAST(count_graduate AS DOUBLE)>=0''',[str(_gpath)]).df()
_gkeys=['region_name','university','gender','education_level','graduation_year']
_gduplicates=_gg.duplicated(_gkeys,keep=False).sum()
_gg=_gg.drop_duplicates(_gkeys,keep=False).copy()
_gcount=_gg.region_name.nunique()
V4_GRADUATE_STATUS={'valid_individual_reporting_rows':len(_gg),'ambiguous_reporting_rows_dropped':int(_gduplicates),
                    'n_geographies':int(_gcount),'employment_reference':'2024-12-31','is_longitudinal':False}
if len(_gg)>=100 and _gcount>=8 and _gg.employed_pct.nunique()>1:
    from sklearn.model_selection import GroupShuffleSplit
    _sp=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=CFG.RANDOM_SEED)
    _a,_b=next(_sp.split(_gg,groups=_gg.region_name))
    _gtr=_gg.iloc[_a].copy();_gte=_gg.iloc[_b].copy()
    assert set(_gtr.region_name).isdisjoint(_gte.region_name)
    _cn=['count_graduate','graduation_year'];_cc=['education_level','gender']
    _Xtr=_gtr[_cn+_cc].copy();_Xte=_gte[_cn+_cc].copy()
    for _k in _cc:
        _Xtr[_k]=_Xtr[_k].fillna('UNKNOWN').astype(str);_Xte[_k]=_Xte[_k].fillna('UNKNOWN').astype(str)
    _gpred=[];_gy=_gte.employed_pct.to_numpy(float)
    _gbase=np.full(len(_gte),float(_gtr.employed_pct.mean()))
    _gpred.append(('region_blind_train_mean',_gbase))
    for _kind in ('ridge','lightgbm','catboost'):
        _num=Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())])
        if _kind=='catboost':
            _med={k:float(_Xtr[k].median()) for k in _cn}
            _A=_Xtr.copy();_B=_Xte.copy()
            for _k in _cn:
                _A[_k]=_A[_k].fillna(_med[_k]);_B[_k]=_B[_k].fillna(_med[_k])
            _mod=CatBoostRegressor(iterations=180,depth=4,learning_rate=.06,verbose=False,
                random_seed=CFG.RANDOM_SEED,thread_count=1,allow_writing_files=False)
            _mod.fit(_A,_gtr.employed_pct.to_numpy(float),cat_features=_cc)
            _gp=_mod.predict(_B)
            _art={'model':_mod,'medians':_med,'categoricals':_cc,'features':_cn+_cc}
        else:
            _prep=ColumnTransformer([('n',_num,_cn),('c',OneHotEncoder(handle_unknown='ignore'),_cc)])
            if _kind=='ridge':_est=Ridge(alpha=30.,solver='lsqr')
            else:_est=LGBMRegressor(n_estimators=170,max_depth=4,num_leaves=12,
                   n_jobs=1,verbosity=-1,random_state=CFG.RANDOM_SEED)
            _mod=Pipeline([('prep',_prep),('reg',_est)])
            _mod.fit(_Xtr,_gtr.employed_pct.to_numpy(float));_gp=_mod.predict(_Xte)
            _art={'model':_mod,'features':_cn+_cc}
        _gp=np.clip(_gp,0,100)
        _gpred.append((_kind,_gp))
        _mp=ROOT/'models'/f'v4_ml55_region_heldout_{_kind}.joblib'
        joblib.dump(_art,_mp,compress=3)
        _rt=joblib.load(_mp)
        if _kind=='catboost':
            _verify=_Xte.copy()
            for _k in _cn:_verify[_k]=_verify[_k].fillna(_rt['medians'][_k])
        else:_verify=_Xte
        assert np.allclose(np.clip(_rt['model'].predict(_verify),0,100),_gp)
        MODEL_FILES[_mp.name]=str(_mp)
        MODEL_RECORDS.append({'module':'ML-5.5','sector':'GRAD_E04_CROSS_SECTION',
            'model':_kind,'seed':CFG.RANDOM_SEED,'path':_mp.name,'file_sha256':hash_file(_mp),
            'training_rows':len(_gtr),'test_rows':len(_gte),'reloaded':True,'feature_num':_cn,'feature_cat':_cc})
    _gm=[];_gpframe=_gte[['region_name','university','graduation_year','employed_pct']].reset_index(drop=True).copy()
    for _kind,_yp in _gpred:
        _gpframe['pred_'+_kind]=_yp
        _gm.append({'module':'ML-5.5','model':_kind,'protocol':'REGION_HELD_OUT_CROSS_SECTION_2024',
            'n_test':len(_gte),'n_regions_test':_gte.region_name.nunique(),**effective_mae(_gy,_yp)})
    qa_table('v4_ml55_crosssection_predictions',_gpframe,allow_empty=True)
    qa_table('v4_ml55_crosssection_metrics',pd.DataFrame(_gm),allow_empty=True)
    V4_GRADUATE_STATUS.update(status='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION',n_test=len(_gte),n_train=len(_gtr),
       baseline_mae=_gm[0]['MAE'],tested_methods=len(_gpred))
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_AND_CROSS_SECTIONAL_TEST',
       reason='All four E04/E05 descriptive; E04 regional OOS employment test does not imply retention/temporal forecast',n=len(_gte))
else:V4_GRADUATE_STATUS['status']='NOT_FEASIBLE_REGION_HELDOUT_AFTER_EXACT_STRATA_GATE'
qa_table('v4_ml55_execution',pd.DataFrame([V4_GRADUATE_STATUS]),allow_empty=True)

# ML-5.6: historical node centrality is tested on common regions only; coverage break is an explicit caveat.
_v4_net=pd.DataFrame(NETWORK)
_v4_nresults=[]
if len(_v4_net) and _v4_net.year.nunique()>1:
    _v4_years=sorted(_v4_net.year.unique())
    for _prev,_now in zip(_v4_years[:-1],_v4_years[1:]):
        _p=_v4_net[_v4_net.year.eq(_prev)][['region_code','pagerank']].rename(columns={'pagerank':'prior_pagerank'})
        _n=_v4_net[_v4_net.year.eq(_now)][['region_code','pagerank']].rename(columns={'pagerank':'actual_pagerank'})
        _joined=_p.merge(_n,on='region_code',validate='one_to_one')
        _rho,_pv=(spearmanr(_joined.prior_pagerank,_joined.actual_pagerank) if len(_joined)>=3 else (np.nan,np.nan))
        _v4_nresults.append({'origin_year':int(_prev),'target_year':int(_now),'common_regions':len(_joined),
            'pagerank_persistence_mae':float(np.abs(_joined.prior_pagerank-_joined.actual_pagerank).mean()) if len(_joined) else np.nan,
            'pagerank_rank_spearman':float(_rho) if np.isfinite(_rho) else np.nan,
            'nominal_p_value_NOT_CONFIRMATORY':float(_pv) if np.isfinite(_pv) else np.nan,
            'coverage_break':'UNRESOLVED_2023_OR_OTHER_YEAR_COVERAGE'})
qa_table('v4_ml56_node_forecast',pd.DataFrame(_v4_nresults),allow_empty=True)


## 9Д. ML-5.7 · независимое географическое испытание образовательного муниципального контекста

Дополнительный тест E0/E1 по пространственно отложенным муниципалитетам. Образовательный признак берётся из БДМО в тот же наблюдаемый год и поэтому **не** является историческим прогнозным признаком. Это только диагностическая ассоциация, а не эффект университета или доказательство миграции.

In [ ]:
# V5 · ML-5.7: независимый *пространственный*, не временной E0/E1 diagnostic.
# General education proxy from BDMO; no academic-campus location, no residence effect inferred.
progress_event('EXTENDED_MODULES','V5 ML-5.7: пространственный E0/E1 на наблюдаемом образовательном контексте',announce=True)
V5_MUNICIPAL_CROSS_STATUS={'status':'NOT_IDENTIFIABLE_ON_AVAILABLE_CROSS_SECTION'}
if CFG.MODE=='FULL' and 'BDMO_EDU_PROXY_CANDIDATES' in globals() and len(BDMO_EDU_PROXY_CANDIDATES):
    _v5_e=BDMO_EDU_PROXY_CANDIDATES.copy()
    _v5_e['oktmo']=norm_code(_v5_e.oktmo);_v5_e['observation_year']=safe_year(_v5_e.observation_year)
    _v5_e=_v5_e.loc[_v5_e.n_raw_rows.eq(1)&_v5_e.min_reported.eq(_v5_e.max_reported)&
        _v5_e.median_reported.notna()&_v5_e.median_reported.ge(0)].copy()
    _v5_m=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _v5_m['oktmo']=norm_code(_v5_m.oktmo);_v5_m['observation_year']=safe_year(_v5_m.observation_year)
    for _k in ('migration_rate','population'):_v5_m[_k]=number(_v5_m[_k])
    _v5_m=_v5_m.dropna(subset=['oktmo','observation_year','migration_rate','population'])
    _v5_m=_v5_m.drop_duplicates(['oktmo','observation_year'],keep=False)
    _v5_join=_v5_m.merge(_v5_e,on=['oktmo','observation_year'],how='inner',validate='one_to_many')
    _v5_join['is_heldout']=_v5_join.oktmo.astype(str).map(lambda x:int(
         hashlib.sha256(('V5_MUNICIPAL_HOLDOUT|'+x).encode()).hexdigest()[:8],16)%5==0)
    _v5_join['indicator_code']=_v5_join.indicator_code.astype(str)
    _v5_join['indicator_unit']=_v5_join.indicator_unit.astype(str)
    _v5_stats=_v5_join.loc[~_v5_join.is_heldout].groupby(['indicator_code','indicator_unit'],dropna=False).agg(
        n_train_muni=('oktmo','nunique'),n_train_rows=('migration_rate','size'),
        n_train_years=('observation_year','nunique'),value_variation=('median_reported','nunique')).reset_index()
    _v5_stats=_v5_stats.sort_values(['n_train_muni','n_train_years','indicator_code'],ascending=[False,False,True])
    qa_table('v5_ml57_spatial_proxy_eligibility',_v5_stats,allow_empty=True)
    _v5_best=_v5_stats.loc[(_v5_stats.n_train_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
        (_v5_stats.value_variation>1)]
    if len(_v5_best):
        _v5_code=str(_v5_best.iloc[0].indicator_code);_v5_unit=str(_v5_best.iloc[0].indicator_unit)
        _v5_k=_v5_join.loc[_v5_join.indicator_code.eq(_v5_code)&_v5_join.indicator_unit.eq(_v5_unit)].copy()
        _v5_k=_v5_k.drop_duplicates(['oktmo','observation_year'],keep=False)
        _v5_train=_v5_k.loc[~_v5_k.is_heldout].copy();_v5_test=_v5_k.loc[_v5_k.is_heldout].copy()
        if len(_v5_train)>=100 and _v5_test.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1:
            _v5_features={'E0':['population','observation_year'],
                          'E1':['population','observation_year','median_reported']}
            _v5_out=[]
            for _variant,_cols in _v5_features.items():
                _v5_mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=30.0))])
                _v5_mod.fit(_v5_train[_cols].to_numpy(float),_v5_train.migration_rate.to_numpy(float))
                _v5_pred=_v5_mod.predict(_v5_test[_cols].to_numpy(float))
                _v5_f=ROOT/'models'/('v5_ml57_crosssection_'+_variant+'_'+hashlib.sha256(
                    (_v5_code+'|'+_v5_unit).encode()).hexdigest()[:10]+'.joblib')
                if _v5_f.exists() or _v5_f.name in MODEL_FILES:raise RuntimeError('V5 ML57 model artifact collision')
                joblib.dump({'model':_v5_mod,'features':_cols,'indicator_code':_v5_code,
                   'protocol':'CROSS_SECTION_NOT_TEMPORAL'},_v5_f,compress=3)
                _v5_loaded=joblib.load(_v5_f)
                if not np.allclose(_v5_pred,_v5_loaded['model'].predict(_v5_test[_cols].to_numpy(float))):
                    raise RuntimeError('V5 ML57 reloaded cross-sectional model mismatch')
                MODEL_FILES[_v5_f.name]=str(_v5_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST','model':_variant,'seed':CFG.RANDOM_SEED,
                  'path':_v5_f.name,'file_sha256':hash_file(_v5_f),'feature_num':_cols,'feature_cat':[],
                  'training_rows':len(_v5_train),'test_rows':len(_v5_test),'reloaded':True})
                _v5_pf=pd.DataFrame({'entity_id':_v5_test.oktmo.astype(str).to_numpy(),
                  'origin_year':_v5_test.observation_year.to_numpy(),'target_year':_v5_test.observation_year.to_numpy(),
                  'information_cutoff_year':_v5_test.observation_year.to_numpy(),
                  'feature_available_year':_v5_test.observation_year.to_numpy(),
                  'target_available_year':_v5_test.observation_year.to_numpy(),
                  'sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'target_id':'D01_MUNICIPAL_MIGRATION_RATE_CONTEMPORANEOUS_ASSOCIATION',
                  'actual':_v5_test.migration_rate.to_numpy(float)})
                _v5_pred_rows=predictions_frame(_v5_pf,'ML-5.7',_variant,_v5_pred,
                                   seed=CFG.RANDOM_SEED,protocol='SPATIAL_CROSS_SECTION_NOT_FORECAST')
                _v5_pred_rows['geography_grain']='MUNICIPALITY';record_prediction(_v5_pred_rows)
                _v5_met=effective_mae(_v5_test.migration_rate,_v5_pred)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'model':_variant,'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST',
                  'seed':CFG.RANDOM_SEED,'target_year':int(_v5_test.observation_year.max()),**_v5_met})
                _v5_out.append({'variant':_variant,'n_train':len(_v5_train),'n_test':len(_v5_test),
                       'heldout_municipalities':_v5_test.oktmo.nunique(),**_v5_met})
            qa_table('v5_ml57_spatial_E0_E1_results',pd.DataFrame(_v5_out),allow_empty=True)
            V5_MUNICIPAL_CROSS_STATUS={'status':'CROSS_SECTION_ASSOCIATION_TESTED_NOT_TEMPORAL',
                'selected_source_indicator':_v5_code,'unit':_v5_unit,
                'years':int(_v5_k.observation_year.nunique()),'n_train':len(_v5_train),
                'n_test':len(_v5_test),'MAE_E0':_v5_out[0]['MAE'],'MAE_E1':_v5_out[1]['MAE'],
                'target':'municipal_migration_rate','not_campus_metric':True}
            if MODULES['ML-5.7'].get('status','').startswith('NOT_FEASIBLE'):
                MODULES['ML-5.7'].update(status='SPATIAL_ASSOCIATION_ONLY_NOT_E0E1_TEMPORAL',
                    reason='Municipal held-out E0/E1 general education context, no temporal as-of claim and no verified VO/SPO campuses')
        else:V5_MUNICIPAL_CROSS_STATUS['reason']='Insufficient held-out municipality count for spatial test'
    else:V5_MUNICIPAL_CROSS_STATUS['reason']='No valid unique municipal indicator across enough training municipalities'
qa_table('v5_ml57_spatial_E0_E1_execution',pd.DataFrame([V5_MUNICIPAL_CROSS_STATUS]),allow_empty=True)
print('ML-5.7 supplemental:',V5_MUNICIPAL_CROSS_STATUS['status'],'| temporal E0/E1 remains a separate gate')


## V7 · дополнительные предметные проверки: OD, кандидаты приёма, муниципальные E0/E1

In [ ]:
# V7.3.0 · дополнительные проверки научной идентифицируемости и устойчивости.
progress_event('EXTENDED_MODULES','V7: сопоставимость OD, точные целевые показатели, муниципальная поддержка',announce=True)

# 5.1/5.3/5.6: диагностируем устойчивость только на ОБЩИХ фактически наблюдавшихся OD-парах.
# Отсутствующие рёбра никогда не превращаем в нули и не делаем вывод об общей динамике страны.
_v7_od=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna().copy()
_v7_od['observation_year']=pd.to_numeric(_v7_od.observation_year,errors='coerce')
_v7_od=_v7_od[_v7_od.observation_year.notna()].copy()
_v7_od['observation_year']=_v7_od.observation_year.astype(int)
if _v7_od.duplicated(['observation_year','origin_region','destination_region']).any():
    raise RuntimeError('V7 OD: неоднозначный ключ год×откуда×куда')
_v7_years=sorted(_v7_od.observation_year.unique())
V7_OD_COMPARABLE=[]
for _cy in _v7_years[1:]:
    _pr=_v7_od[_v7_od.observation_year.eq(_cy-1)]
    _cr=_v7_od[_v7_od.observation_year.eq(_cy)]
    _pair=_cr.merge(_pr[['origin_region','destination_region','observed_value']].rename(
        columns={'observed_value':'previous_value'}),on=['origin_region','destination_region'],how='inner',validate='one_to_one')
    _shared=int(len(_pair));_now=int(len(_cr));_old=int(len(_pr))
    V7_OD_COMPARABLE.append({'year':int(_cy),'observed_edges_previous':_old,'observed_edges_current':_now,
       'shared_edges':_shared,'previous_coverage':_shared/_old if _old else np.nan,
       'current_coverage':_shared/_now if _now else np.nan,
       'MAE_persistence_on_common_edges':float((_pair.observed_value-_pair.previous_value).abs().mean()) if _shared else np.nan,
       'sum_current_on_common_edges':float(_pair.observed_value.sum()),
       'sum_previous_on_common_edges':float(_pair.previous_value.sum()),
       'interpretation':'INTERSECTION_OF_OBSERVED_EDGES_ONLY_NOT_TOTAL_FLOW',
       'coverage_break_unresolved':bool(_old and _now and (_now/_old>1.5 or _now/_old<.67))})
qa_table('v7_od_common_support_sensitivity',pd.DataFrame(V7_OD_COMPARABLE),allow_empty=True)

# 5.2: полная поисковая проверка утверждаемых целевых индикаторов E02 (в отличие от контингента).
# Публикация названия, значения и числа наблюдений — доказательство кандидата, но НЕ его утверждение.
V7_ORG_ADMISSION_SEARCH=pd.DataFrame()
try:
    _v7_e02=fetch_asset(E02)
    with duck_connect() as _db:
        _v7_g=_db.execute('''SELECT CAST(indicator_name AS VARCHAR) indicator_name,
             CAST(indicator_unit AS VARCHAR) indicator_unit,
             COUNT(*)::BIGINT n_source_rows,COUNT(DISTINCT CAST(id AS VARCHAR)) n_organization_ids,
             COUNT(DISTINCT TRY_CAST(year AS INT)) n_years,
             COUNT(*) FILTER (WHERE TRY_CAST(indicator_value AS DOUBLE) IS NOT NULL) numeric_rows,
             COUNT(*) FILTER (WHERE TRY_CAST(indicator_value AS DOUBLE)<0) negative_rows,
             COUNT(DISTINCT COALESCE(CAST(area AS VARCHAR),'__NULL__')) n_area_values
          FROM read_parquet(?) WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)),
            'принят|приём|прием|зачислен|поступив') GROUP BY 1,2''',[str(_v7_e02)]).df()
    _v7_g['indicator_approved']=_v7_g.indicator_name.isin(tuple(CFG.APPROVED_VO_ADMISSION_NAMES))
    _v7_g['has_organization_year_provenance']=_v7_g.n_organization_ids.gt(0)&_v7_g.n_years.ge(2)
    _v7_g['is_certified_admission_total']=False  # name matching and counts never certify statistical semantics
    _v7_g['gate_reason']='REQUIRES_PUBLISHED_METHOD_AND_UNIQUE_ORGANIZATION_YEAR_COHORT'
    V7_ORG_ADMISSION_SEARCH=_v7_g
except Exception as _ex:
    ISSUES.append({'module':'ML-5.2','status':'V7_ADMISSION_DISCOVERY_FAILURE',
                   'reason':type(_ex).__name__+': '+str(_ex)[:160],
                   'required':'repair E02 indicator inventory scan'})
    if CFG.MODE=='FULL': raise
qa_table('v7_organization_admission_discovery',V7_ORG_ADMISSION_SEARCH,allow_empty=True)

# 5.7: строго показываем, почему E0/E1 с учебными площадками выполним или нет.
_v7_muni_status=globals().get('V4_MUNICIPAL_STATUS',{})
_v7_spatial=globals().get('V5_MUNICIPAL_CROSS_STATUS',{})
_v7_campus_n=int(REGISTRY.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum()) if len(REGISTRY) else 0
V7_MUNICIPAL_SUPPORT=pd.DataFrame([{
    'module':'ML-5.7','municipal_demography_rows':int(len(MUNI)),
    'bdmo_proxy_rows':int(len(BDMO_EDU_PROXY_CANDIDATES)),
    'verified_education_campus_records':_v7_campus_n,
    'temporal_e0e1_status':str(_v7_muni_status.get('status','NOT_VERIFIED')),
    'spatial_e0e1_status':str(_v7_spatial.get('status','NOT_VERIFIED')),
    'original_campus_e0e1_identifiable':bool(_v7_campus_n>0 and MODULES['ML-5.7']['status'].startswith('TESTED')),
    'interpretation':'MUNICIPAL_PROXIES_NOT_IDENTICAL_TO_HIGHER_EDUCATION_CAMPUS_CAPACITY',
    'requirement':'VERIFIED_CAMPUS_OKTMO_BY_YEAR_AND_COMPARABLE_MUNICIPAL_OUTCOME'
}])
qa_table('v7_municipal_identification_gate',V7_MUNICIPAL_SUPPORT,['module'])
print('V7: общая поддержка OD',len(V7_OD_COMPARABLE),'пар лет; кандидатов E02',len(V7_ORG_ADMISSION_SEARCH),
      '; подтверждённых учебных площадок',_v7_campus_n)


## 9Е. Дополнительные проверяемые эксперименты V7.1
При наличии подтверждённого целевого показателя проверяется новый временной прогноз; при отсутствии цель исключается без искусственных значений. Сохраняются все ранее выполненные модели V7.

In [ ]:
# ML-05 V7.3.0 · All defensible new tests, retaining the completed V7 seven-module models.
progress_event('EXTENDED_MODULES','V7.3: новые проверяемые подэксперименты ML-5.1–ML-5.7',announce=True)
ADR_EXPERIMENTS=[]

def adr_record_exp(module,code,status,n_train=0,n_test=0,reason='',source_ids='',scientific_claim=''):
    ADR_EXPERIMENTS.append({'module':module,'experiment':code,'status':status,
        'train_rows':int(n_train),'test_rows':int(n_test),'source_ids':source_ids,
        'scientific_claim':scientific_claim,'reason':str(reason)[:500]})

# ML-5.1/3/6: independent stability diagnostics use the same recorded source E01 only.
if len(ADR_OD_PAIRWISE):
    _adr_break=ADR_OD_PAIRWISE.loc[ADR_OD_PAIRWISE.coverage_break_at_2023.eq(True)]
    _adr_ok_years=ADR_OD_PAIRWISE.loc[(ADR_OD_PAIRWISE.coverage_of_prior>=.8)&
                                      (ADR_OD_PAIRWISE.coverage_of_current>=.8)].copy()
    qa_table('adr_od_high_overlap_year_comparisons',_adr_ok_years,allow_empty=True)
    adr_record_exp('ML-5.1','OD_HIGH_OVERLAP_STABILITY','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),
        '2023 coverage break recorded separately; no missing OD zero replacement',E01['asset_id'],
        'Only the intersection of actually reported regional schooling-study edges')
    adr_record_exp('ML-5.3','OUTFLOW_BALANCED_OD_SUPPORT','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),'OD annual marginal totals not comparable without common support',E01['asset_id'],
        'Observed outflow from prior schooling region, not permanent population outmigration')
    adr_record_exp('ML-5.6','NETWORK_BALANCED_GRAPH_SUPPORT','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),'Temporal graph robustness restricted to comparable reported edges',E01['asset_id'],
        'Observed OD network topology, not residential migration infrastructure')
else:
    adr_record_exp('ML-5.1','OD_HIGH_OVERLAP_STABILITY','SKIPPED_NO_MATCHED_YEARS',reason='No contiguous years of observed OD pairs',source_ids=E01['asset_id'])

# Additional testable V7.1 sensitivity: fixed-source common-edge (and origin marginal) time forecasts.
# Each protocol has a separate artifact ID and only uses observed rows; no TEST-based model choice.
def adr_balanced_time_experiment(module,observation,entity_fields,target,sector,target_id,protocol,min_train=100,min_test=20):
    if not len(observation):
        adr_record_exp(module,protocol,'SKIPPED_NO_COMMON_SOURCE_EDGES',source_ids=E01['asset_id'])
        return
    _f=historical_pair_panel(observation,entity_fields,target)
    if _f.empty:
        adr_record_exp(module,protocol,'SKIPPED_NO_HISTORICAL_LAG',source_ids=E01['asset_id'])
        return
    _f['entity_id']=_f[entity_fields].astype(str).agg('|'.join,axis=1)
    _f['sector']=sector;_f['target_id']=target_id
    _split,_reason=temporal_partition(_f,min_train=min_train,min_test=min_test)
    if _split is None:
        adr_record_exp(module,protocol,'SKIPPED_TIME_SUPPORT',len(_f),0,_reason,E01['asset_id'])
        return
    _tr=_split['train'];_te=_split['test']
    _models=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm']
    # PPML/NB are already evaluated exhaustively in the primary V7 OD protocol;
    # the balanced sensitivity adds independent, predeclared non-count-specific checks.
    _before=len(METRICS)
    for _model in _models:
        progress_event('EXTENDED_MODULES',f'{protocol}: {_model}',announce=True)
        fit_and_record(module,_model,_tr,_te,seed=-1,protocol=protocol,
                       nonnegative=True,allow_ppml=False)
    adr_record_exp(module,protocol,'MODEL_COMPLETED',len(_tr),len(_te),
        f'{len(METRICS)-_before} predeclared methods; no selection from TEST',E01['asset_id'],
        'Observed stable OD subpopulation, not a national movement count or source-year admission')

if len(ADR_OD_BALANCED):
    _adr_observed=ADR_OD_BALANCED[['observation_year','origin_region','destination_region','observed_value']].copy()
    adr_balanced_time_experiment('ML-5.1',_adr_observed,['origin_region','destination_region'],
        'observed_value','SPO_BALANCED_OD','E01_BALANCED_OBSERVED_EDGES','ADR_BALANCED_OD_TIME',100,20)
    _adr_balanced_outflow=_adr_observed.groupby(['observation_year','origin_region'],as_index=False).agg(
        balanced_observed_outflow=('observed_value','sum'))
    adr_balanced_time_experiment('ML-5.3',_adr_balanced_outflow,['origin_region'],
        'balanced_observed_outflow','SPO_BALANCED_ORIGIN','E01_BALANCED_ORIGIN_OUTFLOW','ADR_BALANCED_OUTFLOW_TIME',
        CFG.REGION_MIN_TRAIN_ROWS,CFG.REGION_MIN_TEST_ROWS)
else:
    for _m,_protocol in [('ML-5.1','ADR_BALANCED_OD_TIME'),('ML-5.3','ADR_BALANCED_OUTFLOW_TIME')]:
        adr_record_exp(_m,_protocol,'SKIPPED_NO_ALL_YEAR_OD_SUPPORT',0,0,
          'No OD edge appears in every source year; no imputation',E01['asset_id'])

# ML-5.2: actual admissions are modeled only if official indicator labels have been explicitly approved
# and the organizational-year target has no specialty/area duplication.
if len(ADR_ADMISSION_YEAR):
    _adr_approved_frame=ADR_ADMISSION_YEAR[['entity_id','observation_year','actual']].rename(columns={'actual':'observed_admitted'}).copy()
    _adr_org_frame=historical_pair_panel(_adr_approved_frame,['entity_id'],'observed_admitted')
    if len(_adr_org_frame):
        _adr_org_frame['sector']='VO_VERIFIED_ADMISSION'
        _adr_org_frame['target_id']='CERTIFIED_VO_UNIQUE_ORG_YEAR_ADMISSIONS'
        _adr_org_frame['parent_group_id']=_adr_org_frame.entity_id
        _adr_org_frame['historical_target_has_semantic_approval']=True
        _adr_split,_adr_split_reason=temporal_partition(_adr_org_frame,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _adr_split is not None:
            _adr_ex_before=len(METRICS)
            test_predictive('ML-5.2',_adr_org_frame,nonnegative=True,count_confirmed=True)
            _adr_new_models=len(METRICS)-_adr_ex_before
            adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','MODEL_COMPLETED',
                len(_adr_split['train']),len(_adr_split['test']),f'{_adr_new_models} new metric rows',E02['asset_id'],
                'Source-approved admissions by organization and year; not student-stock or applications')
        else:
            adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_TIME_SUPPORT',
                len(_adr_org_frame),0,_adr_split_reason,E02['asset_id'])
    else:
        adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_NO_HISTORICAL_PAIRS',
            len(ADR_ADMISSION_YEAR),0,'No historically available count lag',E02['asset_id'])
else:
    adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_SEMANTICALLY_UNVERIFIED',
        0,0,'Published source data did not certify unique admissions by organization-year; V7 contingent experiment is retained',E02['asset_id'],
        'Candidate labels cannot be converted into real admissions without verified definition')

# ML-5.4/5.5 existing V7 tests already use all matching raw source rows; no invented additional target.
for _adr_mod,_adr_exp,_adr_asset in [('ML-5.4','FULL_SPECIALTY_STRATA_MODEL',VPO_211['asset_id']),
     ('ML-5.5','ALL_FOUR_GRADUATE_FAMILIES',GRAD_ORG['asset_id'])]:
    _adr_success=bool(any(r.get('module')==_adr_mod for r in MODEL_RECORDS) or
                      (_adr_mod=='ML-5.5' and len(PANELS.get('graduate_descriptive_statistics',pd.DataFrame()))))
    adr_record_exp(_adr_mod,_adr_exp,'V7_COMPLETED' if _adr_success else 'SKIPPED_UNAVAILABLE_TARGET',
        reason='Existing V7 observational tests retained; no graduate job-location inference',source_ids=_adr_asset)

# ML-5.7: optional independent E0/E1 *associational* forecasting on geocoded SPO proxies.
# Only available observed SPO institution-years are used. Neither absent organizations nor sites are filled with zero.
ADR_ML57_GEO_STATUS={'status':'SKIPPED_NO_SPATIAL_PROXY','n_spo_location_proxies':len(ADR_GEO_VERIFIED)}
if len(ADR_GEO_VERIFIED):
    _adr_sites=ADR_GEO_VERIFIED[['entity_id','observation_year','candidate_municipal_oktmo']].copy()
    _adr_sites['oktmo']=norm_code(_adr_sites.candidate_municipal_oktmo)
    _adr_sites['observation_year']=safe_year(_adr_sites.observation_year)
    _adr_sites=_adr_sites.dropna(subset=['oktmo','observation_year']).drop_duplicates()
    _adr_site_counts=_adr_sites.groupby(['oktmo','observation_year'],as_index=False).agg(
        observed_geocoded_spo_entities=('entity_id','nunique'))
    qa_table('adr_spo_site_year_observed_counts',_adr_site_counts,['oktmo','observation_year'],allow_empty=True)
    _adr_m=MUNI[['oktmo','observation_year','population','migration_rate']].copy()
    _adr_m['oktmo']=norm_code(_adr_m.oktmo)
    _adr_m['observation_year']=safe_year(_adr_m.observation_year)
    _adr_m['actual']=number(_adr_m.migration_rate)
    _adr_m['population']=number(_adr_m.population)
    _adr_m=_adr_m.dropna(subset=['oktmo','observation_year','actual']).drop_duplicates(['oktmo','observation_year'],keep=False)
    _adr_m['target_year']=_adr_m.observation_year.astype('Int64')
    _adr_m['origin_year']=_adr_m.target_year-CFG.HORIZON
    _adr_m['feature_year']=_adr_m.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _adr_lags=_adr_m[['oktmo','observation_year','actual','population']].rename(columns={
        'observation_year':'feature_year','actual':'lag_migration_rate','population':'lag_population'})
    _adr_m=_adr_m.merge(_adr_lags,on=['oktmo','feature_year'],how='inner',validate='many_to_one')
    _adr_sites_lag=_adr_site_counts.rename(columns={'observation_year':'feature_year'})
    _adr_m=_adr_m.merge(_adr_sites_lag,on=['oktmo','feature_year'],how='inner',validate='many_to_one')
    _adr_m=_adr_m.dropna(subset=['lag_migration_rate','lag_population','observed_geocoded_spo_entities'])
    _adr_m['entity_id']='MUNICIPAL|'+_adr_m.oktmo.astype(str)
    _adr_m['sector']='SPO_SPATIAL_PROXY'
    _adr_m['target_id']='D01_OBSERVED_MUNICIPAL_MIGRATION_RATE'
    _adr_m['information_cutoff_year']=_adr_m.origin_year
    _adr_m['target_available_year']=_adr_m.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    _adr_m['feature_available_year']=_adr_m.feature_year+CFG.FEATURE_PUBLICATION_LAG_YEARS
    _adr_m['prior_value']=_adr_m.lag_migration_rate
    _adr_m['source_vintage']=INDEX['created_at']
    qa_table('adr_ml57_spo_proxy_common_support',_adr_m,allow_empty=True)
    _adr_years57=sorted(_adr_m.target_year.dropna().astype(int).unique()) if len(_adr_m) else []
    if len(_adr_years57)>=2:
        _adr_testyear=max(_adr_years57)
        _adr_origin=_adr_testyear-CFG.HORIZON
        _adr_tr=_adr_m.loc[(_adr_m.target_year<_adr_testyear)&
             (_adr_m.target_available_year<=_adr_origin)].copy()
        _adr_te=_adr_m.loc[_adr_m.target_year.eq(_adr_testyear)].copy()
        if len(_adr_tr)>=80 and len(_adr_te)>=CFG.MIN_MUNICIPALITIES_E0E1 and _adr_tr.observed_geocoded_spo_entities.nunique()>1:
            _adr_results=[]
            for _variant,_xcols in [('E0',['lag_migration_rate','lag_population']),
                                     ('E1',['lag_migration_rate','lag_population','observed_geocoded_spo_entities'])]:
                _fit=Pipeline([('imputer',SimpleImputer(strategy='median')),
                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _fit.fit(_adr_tr[_xcols].to_numpy(float),_adr_tr.actual.to_numpy(float))
                _pred=_fit.predict(_adr_te[_xcols].to_numpy(float))
                _saved=ROOT/'models'/f'adr_ml57_spo_geocode_{_variant}.joblib'
                if _saved.name in MODEL_FILES:raise RuntimeError('ADR ML57 artifact name collision')
                joblib.dump({'model':_fit,'features':_xcols,'scientific_scope':'SPO_GEO_PROXY_ASSOCIATION'},_saved,compress=3)
                if not np.allclose(joblib.load(_saved)['model'].predict(_adr_te[_xcols].to_numpy(float)),_pred):
                    raise RuntimeError('ADR ML57 model reload mismatch')
                MODEL_FILES[_saved.name]=str(_saved)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'SPO_SPATIAL_PROXY',
                    'model':'adr_'+_variant,'protocol':'SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL',
                    'seed':-1,'path':_saved.name,'file_sha256':hash_file(_saved),'feature_num':_xcols,
                    'feature_cat':[],'training_rows':len(_adr_tr),'test_rows':len(_adr_te),'reloaded':True})
                _pf=pd.DataFrame({'entity_id':_adr_te.entity_id.to_numpy(),
                     'origin_year':_adr_te.origin_year.to_numpy(),'target_year':_adr_te.target_year.to_numpy(),
                     'information_cutoff_year':_adr_te.information_cutoff_year.to_numpy(),
                     'feature_available_year':_adr_te.feature_available_year.to_numpy(),
                     'target_available_year':_adr_te.target_available_year.to_numpy(),
                     'sector':'SPO_SPATIAL_PROXY','target_id':'D01_OBSERVED_MUNICIPAL_MIGRATION_RATE',
                     'actual':_adr_te.actual.to_numpy()})
                _predrows=predictions_frame(_pf,'ML-5.7','adr_'+_variant,_pred,protocol='SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL')
                _predrows['geography_grain']='MUNICIPALITY';record_prediction(_predrows)
                _met=effective_mae(_adr_te.actual,_pred)
                METRICS.append({'module':'ML-5.7','sector':'SPO_SPATIAL_PROXY','model':'adr_'+_variant,
                    'protocol':'SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL','seed':-1,'target_year':_adr_testyear,**_met})
                _adr_results.append({'variant':_variant,'n_train':len(_adr_tr),'n_test':len(_adr_te),**_met})
            qa_table('adr_ml57_spatial_e0e1_metrics',pd.DataFrame(_adr_results),allow_empty=True)
            ADR_ML57_GEO_STATUS={'status':'TESTED_ASSOCIATION_NOT_CONFIRMED_CAMPUS',
                'n_train':len(_adr_tr),'n_test':len(_adr_te),'n_spo_location_proxies':len(ADR_GEO_VERIFIED),
                'MAE_E0':_adr_results[0]['MAE'],'MAE_E1':_adr_results[1]['MAE']}
            adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1','MODEL_COMPLETED',len(_adr_tr),len(_adr_te),
                 'Same-support as-of tests; SPO site count only a proximity proxy',
                 SPO_ORG['asset_id']+';'+SPATIAL['asset_id']+';'+MUN['asset_id'],
                 'Predictive association of reported proxy, not causal effect or confirmed campus exposure')
        else:
            ADR_ML57_GEO_STATUS={'status':'SKIPPED_INSUFFICIENT_TIME_OR_SPATIAL_SUPPORT',
              'n_train':len(_adr_tr),'n_test':len(_adr_te)}
    if ADR_ML57_GEO_STATUS.get('status','').startswith('SKIPPED'):
        adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1',ADR_ML57_GEO_STATUS['status'],
           reason='No as-of historical support for unique high-confidence geocoded SPO entities',
           source_ids=SPO_ORG['asset_id']+';'+SPATIAL['asset_id']+';'+MUN['asset_id'])
else:
    adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1','SKIPPED_NO_MATCHED_SPATIAL_SITES',
       reason='No high-confidence spatial links in the actual source; no fabricated campus locations',
       source_ids=SPO_ORG['asset_id']+';'+SPATIAL['asset_id'])

qa_table('adr_ml57_geographic_execution',pd.DataFrame([ADR_ML57_GEO_STATUS]),allow_empty=True)
qa_table('adr_all_feasible_experiments',pd.DataFrame(ADR_EXPERIMENTS),allow_empty=True)
print('V7.1 reconstruction experiments:',len(ADR_EXPERIMENTS),'| new model procedures:',
      sum(x['status']=='MODEL_COMPLETED' for x in ADR_EXPERIMENTS), '| retained all V7 modules')


## 9Ж. V7.3 · предобработка семи исследовательских направлений

Проверки выполняются по наблюдаемым значениям. Неоднозначные соответствия не включаются в целевые переменные.

In [ ]:
# ML-05 V7.3.0 · систематическая верификация предобработки: ML-5.1–ML-5.7.
# Работает ТОЛЬКО с уже прочитанными первичными показателями; нет новых/синтетических наблюдений.
progress_event('EXTENDED_MODULES','V7.3: 7 модулей · семантика, годы, классификаторы, география, сопоставимость',announce=True)
V72_RECONSTRUCTION=[]
def v72_audit(module,experiment,status,rows=0,source='',note='',evidence=''):
    V72_RECONSTRUCTION.append({'module':module,'experiment':experiment,'status':status,
        'observed_rows':int(rows),'primary_source':str(source),'rule':str(evidence),'interpretation':str(note)})

# ML-5.1: изменения покрытия интерпретируются отдельно от изменения численности.
V72_OD_COVERAGE=pd.DataFrame()
if len(ADR_OD_PAIRWISE):
    _p=ADR_OD_PAIRWISE.copy().sort_values(['prior_year','current_year'])
    _p['joint_coverage_min']=_p[['coverage_of_prior','coverage_of_current']].min(axis=1)
    _p['ratio_on_common_edges']=np.where(_p.sum_prior_common.gt(0),
        _p.sum_current_common/_p.sum_prior_common,np.nan)
    _p['coverage_change_factor']=np.where(_p.prior_observed_edges.gt(0),
        _p.current_observed_edges/_p.prior_observed_edges,np.nan)
    _p['coverage_shift_flag']=(_p.joint_coverage_min.lt(.8)|
        _p.coverage_change_factor.lt(.75)|_p.coverage_change_factor.gt(1.33))
    _p['historic_2023_incomplete']=_p.current_year.eq(2023)|_p.prior_year.eq(2023)
    _p['missing_edges_imputed_zero']=False
    _p['sum_claim']='COMMON_REPORTED_OD_EDGES_ONLY'
    V72_OD_COVERAGE=qa_table('v72_od_year_pair_coverage_decomposition',_p,allow_empty=True)
    v72_audit('ML-5.1','E01_COVERAGE_CHANGE_SEPARATION','PASS',len(_p),E01['asset_id'],
              'Оценка динамики только на пересечении зарегистрированных направлений',
              'no imputation; common-support sum; explicit coverage flags')
else:
    v72_audit('ML-5.1','E01_COVERAGE_CHANGE_SEPARATION','SKIPPED_NO_ADJACENT_YEARS',source=E01['asset_id'])

# ML-5.2: решение о пригодности индикатора по наблюдаемым типам и уникальности,
# а не по одному совпадению русского названия.
V72_ADMISSION_DECISION=pd.DataFrame()
if len(ADR_ADMISSION_CANDIDATES):
    _a=ADR_ADMISSION_CANDIDATES.copy()
    _approval=_a['user_or_method_approval'].fillna(False).astype(bool)
    _semantic=_a['strict_source_label_supported'].fillna(False).astype(bool)
    _a['source_semantics_verified']=_semantic
    _a['methodology_approved']=_approval
    _a['allow_model_observed_admissions']=_semantic&_approval
    _a['decision']=np.select([_a.allow_model_observed_admissions,_semantic],
                            ['SOURCE_AND_APPROVAL_PASS','SEMANTIC_CANDIDATE_NOT_METHOD_APPROVED'],
                            default='REJECT_AMBIGUOUS_UNIT_LABEL_OR_VALUE')
    _a['no_sum_of_programme_subtotals']=True
    V72_ADMISSION_DECISION=qa_table('v72_org_admission_indicator_decision',_a,allow_empty=True)
    v72_audit('ML-5.2','E02_ADMISSION_PROOF','PASS_WITH_EXCLUSIONS',len(_a),E02['asset_id'],
        'Единица измерения, значение, утверждение определения и уникальная организационно-годовая ячейка',
        'No approval means no admission target, even for semantically plausible labels')
else:
    v72_audit('ML-5.2','E02_ADMISSION_PROOF','SKIPPED_NO_MATCHING_LABELS',source=E02['asset_id'])

# ML-5.3: региональный отток не смешивается с населением или всей совокупностью направлений.
V72_OUTFLOW_BALANCED=pd.DataFrame()
if len(ADR_OD_BALANCED):
    _bo=ADR_OD_BALANCED.groupby(['observation_year','origin_region'],as_index=False).agg(
        observed_common_edge_outflow=('observed_value','sum'),observed_common_destinations=('destination_region','nunique'))
    _bo['definition']='SAME_REPORTED_EDGES_IN_EVERY_SOURCE_YEAR_NOT_POPULATION_OUTMIGRATION'
    V72_OUTFLOW_BALANCED=qa_table('v72_origin_common_support_outflow',_bo,
                                  ['observation_year','origin_region'],allow_empty=True)
    v72_audit('ML-5.3','REGIONAL_OUTFLOW_CONSTANT_EDGES','PASS',len(_bo),E01['asset_id'],
        'Суммируются лишь наблюдавшиеся ежегодно направления', 'Do not infer total migration from E01')
else:v72_audit('ML-5.3','REGIONAL_OUTFLOW_CONSTANT_EDGES','SKIPPED_NO_SHARED_EDGES',source=E01['asset_id'])

# ML-5.4: аудит смены классификатора по опубликованным кодам — без неутверждённых синонимов.
V72_SPECIALTY_BRIDGE=pd.DataFrame()
if len(ML54_HISTORY):
    _hist=ML54_HISTORY[['observation_year','specialty_code','indicator_name']].copy()
    _hist['year']=pd.to_numeric(_hist.observation_year,errors='coerce').astype('Int64')
    _hist['raw_code']=_hist.specialty_code.astype('string').str.strip().str.replace(' ','',regex=False)
    _hist['standardized_code']=(_hist.raw_code.str.replace(r'[^0-9A-Za-z]','',regex=True)
                                .str.upper().replace('',pd.NA))
    _hist['code_format_valid']=_hist.standardized_code.str.fullmatch(r'[0-9A-Z]{4,12}',na=False)
    _hist['source_indicator']=_hist.indicator_name.astype('string')
    _group=_hist.dropna(subset=['year','standardized_code']).groupby(
        ['standardized_code','year'],as_index=False).agg(
            n_source_indicator_names=('source_indicator','nunique'),n_source_rows=('raw_code','size'),
            n_raw_notations=('raw_code','nunique'),valid_format=('code_format_valid','all'))
    _group['classification_bridge_status']=np.where(_group.valid_format,
        'NORMALIZED_EXACT_SPELLING_ONLY_NOT_CROSS_STANDARD','UNVERIFIED_CODE_FORMAT')
    _group['source_table_scope']=('FULL_SOURCE_AGGREGATE' if 'ml54_specialty_history' not in PANEL_FILES
                                  else 'AVAILABLE_IN_MEMORY_PREVIEW_ONLY')
    V72_SPECIALTY_BRIDGE=qa_table('v72_specialty_code_observed_crosswalk',_group,allow_empty=True)
    v72_audit('ML-5.4','SPECIALTY_CLASSIFIER_SPELLING_AUDIT','PASS',len(_group),VPO_211['asset_id'],
        'Меняются только написания одного кода; разные коды не объявляются одной специальностью',
        'Separate historical code coverage from claimed student admissions')
else:v72_audit('ML-5.4','SPECIALTY_CLASSIFIER_SPELLING_AUDIT','SKIPPED_NO_CODE_HISTORY',source=VPO_211['asset_id'])

# ML-5.5: статистика только внутри исходных E04/E05 семей; несовпадающие срезы не складываются.
V72_GRADUATE_FAMILY=pd.DataFrame()
if len(GRAD_DISPARITIES):
    _g=GRAD_DISPARITIES.copy()
    _k='asset_id' if 'asset_id' in _g else next((c for c in ('source_id','family') if c in _g),None)
    if _k:
        _measures=[c for c in _g if pd.api.types.is_numeric_dtype(_g[c]) and
                   not any(x in c.lower() for x in ('year','rows','count','n_source'))]
        _out=[]
        for _fam,_part in _g.groupby(_k,dropna=False,sort=False):
            _item={'source_family':str(_fam),'observed_aggregate_strata':len(_part),
                'summing_families_forbidden':True,'workplace_geography_observed':False}
            for _col in _measures[:6]:
                _valid=pd.to_numeric(_part[_col],errors='coerce').dropna()
                _item[_col+'_median_of_observed_strata']=float(_valid.median()) if len(_valid) else np.nan
            _out.append(_item)
        V72_GRADUATE_FAMILY=qa_table('v72_graduate_family_independent_descriptives',
            pd.DataFrame(_out),['source_family'],allow_empty=True)
        v72_audit('ML-5.5','E04_E05_NONADDITIVE_GROUPS','PASS',len(V72_GRADUATE_FAMILY),
             ';'.join(a['asset_id'] for a in source_asset(segment='05')),
             'Медианы агрегированных срезов, отдельно для каждой первичной семьи',
             'No inference of workplace region or graduate retention')
    else:v72_audit('ML-5.5','E04_E05_NONADDITIVE_GROUPS','SKIPPED_NO_SOURCE_FAMILY_FIELD')
else:v72_audit('ML-5.5','E04_E05_NONADDITIVE_GROUPS','SKIPPED_NO_GRADUATE_SUMMARY')

# ML-5.6: оценка сопоставимой сети ограничена общими наблюдавшимися узлами и рёбрами.
V72_NETWORK_STABLE=pd.DataFrame()
if len(NETWORK_EDGES):
    _ne=pd.DataFrame(NETWORK_EDGES)
    _years=sorted(_ne.year.dropna().unique())
    _rows=[]
    for _a,_b in zip(_years[:-1],_years[1:]):
        if int(_b)!=int(_a)+1:continue
        _aedge=_ne.loc[_ne.year.eq(_a),['origin','destination','value']].copy()
        _bedge=_ne.loc[_ne.year.eq(_b),['origin','destination','value']].copy()
        if _aedge.duplicated(['origin','destination']).any() or _bedge.duplicated(['origin','destination']).any():
            raise RuntimeError('V72 NETWORK: повторное направление внутри наблюдаемого года')
        _anodes=set(_aedge.origin.astype(str))|set(_aedge.destination.astype(str))
        _bnodes=set(_bedge.origin.astype(str))|set(_bedge.destination.astype(str))
        _shared_nodes=_anodes&_bnodes
        _left=_aedge[_aedge.origin.astype(str).isin(_shared_nodes)&_aedge.destination.astype(str).isin(_shared_nodes)]
        _right=_bedge[_bedge.origin.astype(str).isin(_shared_nodes)&_bedge.destination.astype(str).isin(_shared_nodes)]
        _joined=_left.merge(_right,on=['origin','destination'],suffixes=('_prev','_curr'),validate='one_to_one')
        _rows.append({'prior_year':int(_a),'current_year':int(_b),'n_shared_nodes':len(_shared_nodes),
             'edges_on_shared_nodes_prior':len(_left),'edges_on_shared_nodes_current':len(_right),
             'same_observed_edges':len(_joined),
             'prior_strength_common':float(_joined.value_prev.sum()),
             'current_strength_common':float(_joined.value_curr.sum()),
             'interpretation':'SAME_NODES_AND_EDGES_ONLY_NO_UNOBSERVED_EDGE_ZERO_FILL'})
    V72_NETWORK_STABLE=qa_table('v72_network_same_node_edge_dynamics',pd.DataFrame(_rows),allow_empty=True)
    v72_audit('ML-5.6','NETWORK_SAME_NODE_EDGE_YEAR_PAIRS',
         'PASS' if len(_rows) else 'SKIPPED_NO_ADJACENT_YEARS',len(_rows),E01['asset_id'],
         'Изменение по неизменной наблюдаемой топологии; не общая динамика сети')
else:v72_audit('ML-5.6','NETWORK_SAME_NODE_EDGE_YEAR_PAIRS','SKIPPED_NO_EDGES',source=E01['asset_id'])

# ML-5.7: адрес/координаты рядом с поселением — ещё не подтверждённый адрес кампуса;
# проверяем согласованность исторических прокси и не выдаём их за первичные ОКТМО.
V72_SPATIAL_CONFIDENCE=pd.DataFrame()
if len(ADR_GEO_SPO):
    _geo=ADR_GEO_SPO.copy()
    _geo['municipality_proxy_only']=True
    _geo['confirmed_campus_location']=False
    _geo['spatial_confidence_band']=np.select([
        _geo.spatially_unambiguous.fillna(False),
        _geo.city_name_consistent.fillna(False)],['PROXIMITY_HIGH_NOT_CERTIFIED','CITY_ONLY_UNVERIFIED'],
        default='UNRESOLVED')
    _valid=_geo[_geo.spatially_unambiguous.fillna(False)].copy()
    _c=_valid.groupby('entity_id').candidate_municipal_oktmo.nunique() if len(_valid) else pd.Series(dtype=int)
    _conflict=set(_c[_c>1].index)
    _geo['multi_year_oktmo_conflict']=_geo.entity_id.isin(_conflict)
    _geo.loc[_geo.multi_year_oktmo_conflict,'spatial_confidence_band']='INCONSISTENT_OVER_TIME'
    _geo['may_enter_official_campus_target']=False
    V72_SPATIAL_CONFIDENCE=qa_table('v72_spo_municipal_proxy_confidence',_geo,allow_empty=True)
    if _conflict:
        QUARANTINE.extend(_geo[_geo.multi_year_oktmo_conflict].head(100).assign(
             source='ML57_SPATIAL_MULTIPERIOD_CONFLICT').to_dict('records'))
    v72_audit('ML-5.7','D04_SPO_SPATIAL_PROXY_STABILITY','PASS_WITH_QUARANTINE',len(_geo),
        SPO_ORG['asset_id']+';'+SPATIAL['asset_id'],
        'Проверено постоянство прокси-ОКТМО; подтверждённых кампусов эта процедура не создаёт',
        'NEVER treat nearest settlement as verified study campus')
else:v72_audit('ML-5.7','D04_SPO_SPATIAL_PROXY_STABILITY','SKIPPED_NO_GEO_MATCHES',source=SPATIAL['asset_id'])

V72_RECON_DF=qa_table('v72_all_seven_reconstruction_evidence',pd.DataFrame(V72_RECONSTRUCTION),
                      ['module','experiment'])
print('V7.3 · семь модулей: проверено',len(V72_RECON_DF),'подисследований,',
      sum(str(x['status']).startswith('PASS') for x in V72_RECONSTRUCTION),'с подтверждёнными результатами')

mark_stage('EXTENDED_MODULES')


## 10. Независимые метрики, canary, статистика
Метрики пересчитываются по прогнозным строкам. При отсутствии законного DEV сравнения не становятся подтверждающими. Отрицательные результаты сохраняются.

In [ ]:
progress_event('STATISTICS','Независимые метрики, канарейки, проверки гипотез')
# Независимые сквозные проверки: источники/будущее/статистика/содержательные выводы.
CANARIES=[]

def test_future_mutation(raw,keys,value_col,module):
    if raw.empty or 'observation_year' not in raw or value_col not in raw:
        return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет валидной временной панели'}
    r=raw[keys+['observation_year',value_col]].dropna(subset=['observation_year']).copy()
    if r.empty or r.observation_year.nunique()<2:return {'module':module,'status':'NOT_APPLICABLE','reason':'Мало лет'}
    r=r.drop_duplicates(keys+['observation_year'])
    if r.empty:return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет уникальных ключей'}
    p1=historical_pair_panel(r,keys,value_col)
    mx=int(r.observation_year.max());r2=r.copy()
    r2.loc[r2.observation_year.eq(mx),value_col]=number(r2.loc[r2.observation_year.eq(mx),value_col]).fillna(0)+10**8
    p2=historical_pair_panel(r2,keys,value_col)
    # Проверяется строгое отсутствие будущих лагов; отдельный тест экономических и сетевых join — ниже.
    if len(p1) and (p1.feature_available_year>p1.information_cutoff_year).any():
        fail('CANARY','Исходная панель содержит будущие признаки: '+module)
    # До мутации будущего итоговые X прежних origins должны быть идентичны.
    common=[c for c in keys+['target_year','prior_value','prior2_value','historical_year'] if c in p1 and c in p2]
    early1=p1[p1.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    early2=p2[p2.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    if not early1.equals(early2):fail('CANARY','Мутация будущего изменила прошлые признаки: '+module)
    return {'module':module,'status':'PASS','n_untouched':len(early1),'mutated_year':mx}

CANARIES.append(test_future_mutation(OD_INTER,['origin_region','destination_region'],'observed_value','ML-5.1'))
CANARIES.append(test_future_mutation(OUTFLOW,['origin_region'],'observed_interregional_routes','ML-5.3'))
if len(ORG_YEAR):CANARIES.append(test_future_mutation(ORG_YEAR,['sector','entity_id'],'actual','ML-5.2'))
if len(PROGRAM_PROFILE):
    prog_test=PROGRAM_PROFILE.drop_duplicates(['specialty_code','indicator_name','observation_year'])
    CANARIES.append(test_future_mutation(prog_test,['specialty_code','indicator_name'],'observed_value','ML-5.4'))
if len(GRADUATES):
    CANARIES.append({'module':'ML-5.5','status':'DESCRIPTIVE_ONLY',
      'reason':'Единственная дата измерения занятости; модель предсказаний не запускалась'})
if len(MUNI):CANARIES.append(test_future_mutation(MUNI,['oktmo'],'migration_rate','ML-5.7'))
CANARIES.append({'module':'ECONOMIC_CONTEXT','status':'STATIC_OR_ASOF_AUDIT',
 'reason':'Использована только экономическая панель observation_year=historical_year; future rows not joined'})
if len(OD_MODEL):
    # Отдельный тест исходных маргиналей: признаки прошлого года должны совпадать с исходными агрегатами.
    src_mass=OD_INTER.groupby(['observation_year','origin_region'],as_index=False).observed_value.sum()
    tmp=OD_MODEL[['origin_region','historical_year','origin_mass']].dropna().drop_duplicates()
    exp=src_mass.rename(columns={'observation_year':'historical_year','observed_value':'expected_mass'})
    comp=tmp.merge(exp,on=['historical_year','origin_region'],how='left',validate='many_to_one')
    if len(comp) and not np.allclose(comp.origin_mass,comp.expected_mass,equal_nan=True):
        fail('CANARY','Региональные маргинали не совпали с историческим исходным годом')
    CANARIES.append({'module':'OD_MARGINS','status':'PASS' if len(comp) else 'NOT_APPLICABLE',
       'verified_origin_year_margins':int(len(comp))})
qa_table('future_mutation_canaries',pd.DataFrame(CANARIES),allow_empty=True)

# Весь стек прогнозов, включая 50 сидов, объединяется НА ДИСКЕ.
PRED=pd.DataFrame();RECALCULATED=[];PRED_STORAGE_FILE=None
if PRED_PARTS:
    progress_event('STATISTICS',f'Независимая проверка {PRED_TOTAL:,} строк из {len(PRED_PARTS)} файлов',announce=True)
    PRED_STORAGE_FILE=ROOT/'derived'/'education_predictions.parquet'
    # read_parquet(list) обрабатывается потоком в DuckDB, не через pandas.concat.
    with duck_connect() as con:
        _files='['+','.join("'"+str(p).replace("'","''")+"'" for p in PRED_PARTS)+']'
        _q=f"SELECT * FROM read_parquet({_files},union_by_name=true)"
        _fp=str(PRED_STORAGE_FILE).replace("'","''")
        con.execute(f"COPY ({_q}) TO '{_fp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        _num=con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _num!=PRED_TOTAL:fail('REPRO',f'Прогнозов в Parquet {_num} вместо {PRED_TOTAL}')
        _invalid=con.execute('''SELECT COUNT(*) FROM read_parquet(?)
            WHERE predicted IS NULL OR actual IS NULL OR NOT isfinite(predicted) OR NOT isfinite(actual)
            OR feature_available_year>information_cutoff_year''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _invalid:fail('REPRO',f'Невалидных прогнозов: {_invalid}')
        _duplicates=con.execute('''SELECT COUNT(*) FROM (
            SELECT module,sector,entity_id,target_year,model,seed,split_id,COUNT(*) AS n
            FROM read_parquet(?) GROUP BY ALL HAVING COUNT(*)>1)''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _duplicates:fail('SUPPORT',f'Повторные натуральные ключи прогнозов: {_duplicates}')
        # Точные MAE/RMSE/WAPE по ВСЕМ строкам всех файлов.
        RECALCULATED=con.execute('''SELECT module,sector,model,seed,target_year,split_id,
            AVG(ABS(actual-predicted)) AS MAE,
            SQRT(AVG(POWER(actual-predicted,2))) AS RMSE,
            SUM(ABS(actual-predicted))/NULLIF(SUM(ABS(actual)),0) AS WAPE,
            COUNT(*)::BIGINT AS n
            FROM read_parquet(?) GROUP BY 1,2,3,4,5,6''',[str(PRED_STORAGE_FILE)]).df().to_dict('records')
        PRED=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(PRED_STORAGE_FILE)]).df()
    PANEL_FILES['education_predictions']=PRED_STORAGE_FILE
    final=pd.DataFrame(RECALCULATED)
    for r in final.itertuples(index=False):
        original=[m for m in METRICS if m['module']==r.module and m.get('sector')==r.sector
                  and m['model']==r.model and m['seed']==r.seed and m['target_year']==r.target_year and m.get('protocol')==r.split_id]
        if not original:fail('REPRO','Нет строки метрики для '+r.module+' '+r.model)
        for m in original:
            if not np.isclose(m['MAE'],r.MAE,atol=1e-7,rtol=1e-7):fail('REPRO','MAE не совпал: '+r.module+' '+r.model)
qa_table('metrics_recomputed_from_predictions',pd.DataFrame(RECALCULATED),allow_empty=True)

# Предрегистрация Holm: только подтверждающие сравнения, для которых есть валидный p.
CONFIRM=pd.DataFrame(CONFIRMATORY)
if not CONFIRM.empty:
    valid=CONFIRM.p_value.notna()&np.isfinite(CONFIRM.p_value)
    CONFIRM['p_holm']=np.nan
    if valid.any():
        CONFIRM.loc[valid,'p_holm']=multipletests(CONFIRM.loc[valid,'p_value'].to_numpy(float),
                                                method='holm')[1]
    CONFIRM['interpretation']=np.where(CONFIRM.delta_mae.lt(0),'LOWER_MAE_ON_SELECTED_COMMON_SUPPORT',
                                      np.where(CONFIRM.delta_mae.gt(0),'HIGHER_MAE_ON_SELECTED_COMMON_SUPPORT','EQUAL_MAE'))
    CONFIRM['historical_vintage_validated']=False
    CONFIRM['claim_level']='EXPLORATORY_RETROSPECTIVE_SOURCE_2026'

qa_table('confirmatory_comparisons',CONFIRM,allow_empty=True)

# Доказательный реестр 7 гипотез: отдельно результат и предел идентификации.
for i in range(1,8):
    module=f'ML-5.{i}'
    status=MODULES[module]
    subset=CONFIRM[CONFIRM.module.eq(module)] if len(CONFIRM) else pd.DataFrame()
    if len(subset):
        r=subset.iloc[0]
        if r.choice_basis!='DEV_ASOF':
            conclusion='Подтверждающий отбор невозможен: DEV-метки недоступны на момент TEST; сравнения моделей разведочные'
            metric=f'baseline_MAE={r.baseline_mae:.5g}; confirmatory_delta=NOT_APPLICABLE'
        else:
            conclusion=('выбранная на DEV модель не превзошла инерцию на ретроспективном тесте' if r.delta_mae>=0
                        else 'в ретроспективном тесте MAE ниже инерции; исторические версии данных не подтверждены')
            metric=f'baseline_MAE={r.baseline_mae:.5g}; selected_MAE={r.selected_mae:.5g}; delta={r.delta_mae:+.5g}'
    else:conclusion=status['reason'];metric='Нет корректного независимого прогнозного теста'
    HYPOTHESES.append({'hypothesis':f'H5.{i}','module':module,'status':status['status'],
        'metric_or_basis':metric,'scientific_conclusion':conclusion,
        'source_observability':('E01_schooling_to_study_NOT_residence' if i in (1,3,6)
             else 'source_specific_unverified_retention' if i==5 else 'source_specific'),
        'causal_identification':'NOT_ESTABLISHED'})
qa_table('seven_hypotheses',pd.DataFrame(HYPOTHESES),['hypothesis'])
qa_table('split_audit',pd.DataFrame(SPLIT_AUDIT),allow_empty=True)
qa_table('model_metrics',pd.DataFrame(METRICS),allow_empty=True)
qa_table('model_registry',pd.DataFrame(MODEL_RECORDS),allow_empty=True)

# Статусы: техническая полнота != научное подтверждение семи гипотез.
executed_all=all(m['status']!='NOT_STARTED' for m in MODULES.values())
failed_modules=[k for k,m in MODULES.items() if m['status']=='FAILED']
research_full=all(m['status']=='TESTED_FULL' for m in MODULES.values()) and OD_SEMANTIC['yearly_count_unit_confirmed']
STATE['technical']='PASS_ASSEMBLED' if executed_all and not failed_modules else 'FAIL'
STATE['scientific']='FULL_RESEARCH_PASS' if research_full and CFG.MODE=='FULL' else (
    'FULL_EXECUTION_WITH_LIMITATIONS' if CFG.MODE=='FULL' and executed_all and not failed_modules else 'PILOT_OR_PARTIAL')
STATE['overall']=STATE['scientific']

print('Technical / scientific statuses:',STATE['technical'],STATE['scientific'])
print('Confirmed comparisons:',len(CONFIRM),'| exported predictions:',PRED_TOTAL)

# Финальная отметка этапа перенесена после всех его подмодулей


## 10А. Парные сравнения и проверка завершённости программы
Разделение вычислительной полноты и научной идентифицируемости; отсутствующие измерения не фабрикуются.

In [ ]:
# V4 · audit model completion independently of the notebook's technical PASS.
progress_event('STATISTICS','V4: численные сравнения на общих ключах, матрица научной полноты',announce=True)
V4_PAIRED_COMPARISONS=[]
if PRED_STORAGE_FILE and Path(PRED_STORAGE_FILE).is_file():
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _unique=_db.execute('''SELECT DISTINCT module,sector,split_id,model FROM read_parquet(?) ORDER BY 1,2,3,4''',
                           [str(PRED_STORAGE_FILE)]).df()
        _combos=_unique[~_unique.model.isin(['persistence','rolling_mean_2','damped_trend','catboost'])].copy()
        for _rec in _combos.itertuples(index=False):
            _pred=_db.execute('''WITH b AS (SELECT entity_id,target_year,actual,predicted baseline,
                     origin_region,destination_region,parent_group_id FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model='persistence' AND seed=-1),
                 v AS (SELECT entity_id,target_year,actual,predicted as candidate FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model=?)
                  SELECT b.*,v.candidate FROM b JOIN v USING(entity_id,target_year)
                  WHERE ABS(b.actual-v.actual)<1e-7''',
               [str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,
                str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,_rec.model]).df()
            if not len(_pred):continue
            _a=np.abs(_pred.actual.to_numpy(float)-_pred.baseline.to_numpy(float))
            _v=np.abs(_pred.actual.to_numpy(float)-_pred.candidate.to_numpy(float))
            if _rec.module=='ML-5.1' and {'origin_region','destination_region'}.issubset(_pred):
                _g1=_pred.origin_region.fillna('__UNK__').astype(str)
                _g2=_pred.destination_region.fillna('__UNK__').astype(str)
                _ci=two_way_mae_bootstrap(_a,_v,_g1,_g2,reps=CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
            else:
                _g1=(_pred.parent_group_id.fillna(_pred.entity_id).astype(str) if
                     'parent_group_id' in _pred else _pred.entity_id.astype(str))
                # Only one test target_year; two-way geographic-year bootstrap not identified.
                _unique_groups=_g1.nunique()
                if _unique_groups>=8:
                    _rng=np.random.default_rng(CFG.RANDOM_SEED)
                    _groups=_g1.to_numpy();_base=np.asarray(_v-_a,float)
                    _levels=np.unique(_groups);_boot=[]
                    for _ in range(CFG.BOOTSTRAP_REPS):
                        _mult=Counter(_rng.choice(_levels,len(_levels),replace=True))
                        _weights=np.fromiter((_mult[v] for v in _groups),dtype=float,count=len(_groups))
                        if _weights.sum()>0:_boot.append(float(np.average(_base,weights=_weights)))
                    _vals=np.asarray(_boot)
                    _ci={'delta_MAE':float(np.mean(_base)),'ci_low':float(np.quantile(_vals,.025)) if len(_vals)>20 else np.nan,
                         'ci_high':float(np.quantile(_vals,.975)) if len(_vals)>20 else np.nan,
                         'p_value':min(1.0,2*min((np.count_nonzero(_vals<=0)+1)/(len(_vals)+1),
                            (np.count_nonzero(_vals>=0)+1)/(len(_vals)+1))) if len(_vals)>20 else np.nan,
                         'clusters_a':_unique_groups,'clusters_b':None}
                else:_ci={'delta_MAE':float(np.mean(_v-_a)),'ci_low':np.nan,'ci_high':np.nan,
                           'p_value':np.nan,'clusters_a':_unique_groups,'clusters_b':None}
            V4_PAIRED_COMPARISONS.append({'module':_rec.module,'sector':_rec.sector,
                'split_id':_rec.split_id,'model':_rec.model,'n_common_keys':len(_pred),
                'persistence_MAE':float(_a.mean()),'candidate_MAE':float(_v.mean()),**_ci,
                'inference_type':'EXPLORATORY_PAIRED_RETROSPECTIVE_NOT_CONFIRMATORY',
                'historical_source_vintage_at_prediction':'NOT_ARCHIVED'})
_V4_BOOT=pd.DataFrame(V4_PAIRED_COMPARISONS)
if len(_V4_BOOT):
    _mask=_V4_BOOT['p_value'].notna()&np.isfinite(_V4_BOOT['p_value'])
    _V4_BOOT['p_holm_exploratory']=np.nan
    if _mask.any():_V4_BOOT.loc[_mask,'p_holm_exploratory']=multipletests(
        _V4_BOOT.loc[_mask,'p_value'].to_numpy(float),method='holm')[1]
qa_table('v4_exploratory_paired_comparisons',_V4_BOOT,allow_empty=True)

V4_METHOD_EXECUTIONS=[]
# Check every required model separately, not merely that a module produced one metric.
_v4_contract_base={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble'}
_v4_expected={}
for _m in ('ML-5.1','ML-5.3','ML-5.4'):
    _actual={z.get('model') for z in METRICS if z.get('module')==_m and z.get('protocol')=='ROLLING_TIME'}
    if _actual:
        _req=_v4_contract_base.copy()
        # PPML/NB only if their observed-source integer semantics passed the count gate.
        _frame=OD_MODEL if _m=='ML-5.1' else OUTFLOW_MODEL if _m=='ML-5.3' else None
        _count_eligible=bool(_frame is not None and len(_frame) and
             number(_frame.actual).ge(0).all() and
             np.isclose(number(_frame.actual),np.rint(number(_frame.actual)),atol=1e-8,rtol=0).all())
        if _m=='ML-5.4':
            _count_eligible=MODULES[_m]['status'].startswith('TESTED')
        if _count_eligible:_req.update({'ppml','negbin'})
        _v4_expected[_m]=_req
if any(z.get('module')=='ML-5.2' and z.get('protocol')=='ALT_CONTINGENT_RETROSPECTIVE' for z in METRICS):
    _v4_expected['ML-5.2']={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'}
if any(z.get('module')=='ML-5.7' and z.get('protocol')=='RETROSPECTIVE_PROXY_E0E1' for z in METRICS):
    _v4_expected['ML-5.7']={'E0','E1'}
for _module in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _r=[m for m in METRICS if m.get('module')==_module]
    _failed=[x for x in ISSUES if x.get('module')==_module and x.get('status') in
            ('MODEL_FAILED','FAMILY_PROFILE_FAILED','SOURCE_PROFILE_FAILURE','SPATIAL_MODEL_FAILURE')]
    _expected=_v4_expected.get(_module,set())
    _actually={z.get('model') for z in _r}
    _missing=sorted(_expected-_actually)
    if _missing:_failed.append({'reason':'Feasible models without metrics: '+','.join(_missing),
                               'status':'METHOD_NOT_EXECUTED'})
    if _module=='ML-5.5' and V4_GRADUATE_STATUS.get('status')=='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION':
        if len(PANELS.get('v4_ml55_crosssection_metrics',pd.DataFrame()))<4:
            _failed.append({'reason':'E04 cross-sectional baseline/Ridge/LightGBM/CatBoost incomplete'})
    if _module=='ML-5.6' and len(NETWORK) and not len(PANELS.get('v4_ml56_node_forecast',pd.DataFrame())):
        _failed.append({'reason':'Node centrality persistence test missing'})
    _feasible=bool(_r) or MODULES[_module].get('status','').startswith('DESCRIPTIVE')
    V4_METHOD_EXECUTIONS.append({'module':_module,'module_status':MODULES[_module].get('status'),
        'models_with_metrics':len(set(x.get('model') for x in _r)),
        'metric_rows':len(_r),'saved_model_artifacts':sum(m.get('module')==_module for m in MODEL_RECORDS),
        'prediction_rows_validated':bool(any(x.get('module')==_module for x in RECALCULATED)),
        'required_models':','.join(sorted(_expected)), 'missing_required_models':','.join(_missing),
        'computational_failures':len(_failed),
        'failure_reason':' | '.join(str(x.get('reason',''))[:120] for x in _failed[:3]),
        'target_is_scientifically_identified':bool(_module in ('ML-5.4','ML-5.6')),
        'historical_vintage_validated':False,
        'scientific_completion':'RETROSPECTIVE_OR_DESCRIPTIVE_WITH_GATES'})
qa_table('v4_methods_execution_ledger',pd.DataFrame(V4_METHOD_EXECUTIONS),['module'])
V4_COMPUTATIONAL_COMPLETE=bool(not V4_SOURCE_ERRORS and not any(x['computational_failures'] for x in V4_METHOD_EXECUTIONS))
V4_SCIENTIFIC_COMPLETE=bool(V4_COMPUTATIONAL_COMPLETE and
                           all(MODULES[m]['status']=='TESTED_FULL' for m in MODULES) and
                           OD_SEMANTIC.get('yearly_count_unit_confirmed',False) and
                           len(ORG_YEAR)>0)
STATE['scientific']='FULL_RESEARCH_PASS' if V4_SCIENTIFIC_COMPLETE else (
    'FULL_COMPUTATIONS_WITH_DOCUMENTED_DATA_LIMITS' if V4_COMPUTATIONAL_COMPLETE and CFG.MODE=='FULL'
     else 'FULL_PARTIAL_COMPUTATIONS_OR_MODEL_FAILURES')
STATE['overall']=STATE['scientific']
qa_table('v4_scientific_acceptance',pd.DataFrame([{'version':CFG.VERSION,'source_assets':len(ASSETS),
    'value_scans_passed':sum(x['content_status']=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'source_assets_not_in_model':sum(not x['main_prediction_input'] for x in _v4_roles),
    'seven_modules_reported':len(MODULES),'computational_failures':sum(x['computational_failures'] for x in V4_METHOD_EXECUTIONS),
    'full_computations_without_failures':V4_COMPUTATIONAL_COMPLETE,'all_seven_scientifically_identified':V4_SCIENTIFIC_COMPLETE,
    'scientific_status':STATE['scientific'],'human_approval_needed_for_admissions':len(ORG_YEAR)==0,
    'OD_2023_coverage_break_unresolved':bool(coverage_df.potential_break.fillna(False).any()),
    'historical_vintages_not_archived':True,'retention_unobservable_without_work_region':True}]))
print('V4 acceptability:',STATE['scientific'],'| calculations with actual errors:',not V4_COMPUTATIONAL_COMPLETE)

# Rolling-origin feasibility is measured year by year. Lack of archived source vintages forbids a confirmatory claim.
_v4_roll=[]
for _m,_f in [('ML-5.1',OD_MODEL),('ML-5.3',OUTFLOW_MODEL)]:
    if _f is None or not len(_f):continue
    _years=sorted(int(y) for y in _f.target_year.dropna().unique())
    for _y in _years[1:]:
        _cut=_y-CFG.HORIZON
        _train=_f[_f.target_year.lt(_y)&_f.target_available_year.le(_cut)]
        _test=_f[_f.target_year.eq(_y)]
        _gate=len(_train)>=(CFG.REGION_MIN_TRAIN_ROWS if _m=='ML-5.3' else CFG.MIN_TRAIN_ROWS) and \
              len(_test)>=(CFG.REGION_MIN_TEST_ROWS if _m=='ML-5.3' else CFG.MIN_TEST_ROWS)
        _v4_roll.append({'module':_m,'target_year':_y,'n_asof_train':len(_train),
             'n_test_observed':len(_test),'legally_feasible_given_publication_lag':bool(_gate),
             'source_vintage_asof_origin':'NOT_ARCHIVED',
             'scientific_status':'RETROSPECTIVE_ONLY' if _gate else 'NOT_FEASIBLE_SMALL_HISTORICAL_TRAIN'})
qa_table('v4_rolling_origin_feasibility',pd.DataFrame(_v4_roll),allow_empty=True)


# 10Б. Контракт V5: полнота моделей, численная состоятельность и доказательная база

Этот блок проверяет результаты независимо от общей отметки выполнения ноутбука. Модели со сбоями, отсутствующими прогнозами или нестабильными значениями не засчитываются. Нет исходных показателей — `NOT_IDENTIFIABLE`, а не вымышленный результат.

In [ ]:
# ML-05 V5 · fail-closed model registry, numerical diagnostics and evidence matrix.
progress_event('STATISTICS','V5: полный контракт моделей, протоколов и проверка числа сидов',announce=True)
V5_MODEL_QA=[];V5_MODULE_ACCEPTANCE=[]
_artifact_paths=[m['path'] for m in MODEL_RECORDS]
if len(_artifact_paths)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: different model experiments have duplicate output paths')
if len(MODEL_FILES)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: model registry and actual model files differ')
for _m in MODEL_RECORDS:
    _fp=ROOT/'models'/_m['path']
    _ok=_fp.is_file() and _m.get('file_sha256')==hash_file(_fp) and _m.get('reloaded')
    V5_MODEL_QA.append({'module':_m['module'],'model':_m['model'],'protocol':_m.get('protocol','CUSTOM'),
        'seed':_m['seed'],'path':_m['path'],'sha_verified':bool(_ok),
        'guarded_predictions':_m.get('prediction_guard_events',0),
        'guard_fraction':_m.get('prediction_guard_fraction',0.)})
if V5_MODEL_QA and not all(x['sha_verified'] for x in V5_MODEL_QA):
    raise RuntimeError('V5: saved models changed or reload failed')
qa_table('v5_model_files_independently_verified',pd.DataFrame(V5_MODEL_QA),allow_empty=True)

# Reconcile required model names and their independent saved metrics, not a mere code branch.
for _m in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _mm=[r for r in METRICS if r.get('module')==_m]
    _issues=[r for r in ISSUES if r.get('module')==_m and r.get('status') in
      ('MODEL_FAILED','SPATIAL_MODEL_FAILURE','SOURCE_PROFILE_FAILURE','FAMILY_PROFILE_FAILED')]
    _status=MODULES[_m].get('status','UNKNOWN')
    _preds=sum(int(x.get('n',0)) for x in _mm)
    _time={x['model'] for x in _mm if x.get('protocol')=='ROLLING_TIME'}
    _required={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble','ppml','negbin'}
    _targets_confirmed=bool(_status.startswith('TESTED'))
    if _m in ('ML-5.1','ML-5.3','ML-5.4') and _targets_confirmed:
        _missing=sorted(_required-_time)
        _seeds={int(x['seed']) for x in _mm if x.get('model')=='catboost' and x.get('protocol')=='ROLLING_TIME'}
        _missing_seeds=sorted(set(int(z) for z in CFG.FINAL_SEEDS)-_seeds) if CFG.MODE=='FULL' else []
    else:_missing=[];_missing_seeds=[]
    _numeric_invalid=[x['model'] for x in _mm if not np.isfinite(x.get('MAE',np.nan)) or
        not np.isfinite(x.get('RMSE',np.nan)) or float(x.get('MAE',0))<0]
    _failures=bool(_issues or _missing or _missing_seeds or _numeric_invalid)
    _scientific=('OBSERVED_TARGET_TESTED_RETROSPECTIVELY' if _targets_confirmed else
                 'DESCRIPTIVE_OR_PROXY_TEST_ONLY' if _mm or 'DESCRIPTIVE' in _status or 'PROXY' in _status else
                 'NOT_IDENTIFIABLE_WITH_AVAILABLE_STAGE_I')
    V5_MODULE_ACCEPTANCE.append({'module':_m,'status':_status,'metrics':len(_mm),
        'model_file_count':sum(x['module']==_m for x in MODEL_RECORDS),'failed_methods':len(_issues),
        'missing_methods':','.join(_missing),'missing_final_seeds':','.join(map(str,_missing_seeds)),
        'nonfinite_metrics':','.join(_numeric_invalid),
        'computational_pass':not _failures,'scientific_scope':_scientific,
        'original_target_fully_identified':bool(_m=='ML-5.4' and _targets_confirmed),
        'historical_source_vintage_validated':False})

V5_LEDGER=pd.DataFrame(V5_MODULE_ACCEPTANCE)
qa_table('v5_seven_module_acceptance',V5_LEDGER,['module'])

# Source evidence: 52 original datasets are all profiled but not uniformly identifiable as model features.
V5_SOURCE_EVIDENCE=pd.DataFrame(V4_SOURCE_PROFILES)
if len(V5_SOURCE_EVIDENCE)!=len(ASSETS) or not all(r.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for r in V4_SOURCE_PROFILES):
    if CFG.MODE=='FULL':raise RuntimeError('V5 source coverage is not 52/52')
qa_table('v5_all52_source_evidence',V5_SOURCE_EVIDENCE,allow_empty=True)
V5_COMPUTATIONAL_PASS=bool(not V4_SOURCE_ERRORS and V5_LEDGER.computational_pass.all())
V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS=bool(V5_COMPUTATIONAL_PASS and all(V5_LEDGER.original_target_fully_identified))
V5_ACCEPTANCE_STATUS='FULL_COMPUTATIONS_WITH_SCIENTIFIC_LIMITS' if V5_COMPUTATIONAL_PASS else 'MODEL_OR_EVIDENCE_FAILURE'
STATE['scientific']=V5_ACCEPTANCE_STATUS
print('V5: 52-source integrity',len(V5_SOURCE_EVIDENCE),'/',len(ASSETS),
      '| all computational methods',V5_COMPUTATIONAL_PASS,'| scientifically identified 7/7',V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)


## V7 · независимая трассировка источников и моделей: 52/52 + научные границы семи модулей

In [ ]:
# V7.3.0 · проверяемая цепочка Stage I → исследовательская панель → сохранённая модель.
# Не называем технически обследованные 52 источника 52 модельными признаками.
progress_event('STATISTICS','V7: проверяемая трассировка 52 источников и 7 модулей',announce=True)

_v7_source_panels={}
def _v7_map(asset_or_list,panels,methods,relationship='PRIMARY_SOURCE'):
    for _a in (asset_or_list if isinstance(asset_or_list,(tuple,list)) else [asset_or_list]):
        if _a is None:continue
        _id=_a['asset_id'] if isinstance(_a,dict) else str(_a)
        _v7_source_panels.setdefault(_id,[]).append({'panels':list(panels),'modules':list(methods),'relationship':relationship})
_v7_map(E01,['edu_od_year','forecast_od_candidates','outflow_observed','education_network_year'],['ML-5.1','ML-5.3','ML-5.6'])
_v7_map(E02,['edu_org_registry','v7_organization_admission_discovery','v4_organization_contingent_candidates'],['ML-5.2'])
_v7_map(SPO_ORG,['edu_org_registry'],['ML-5.2'])
_v7_map(SPO_MASTER,['spo_admission_raw_candidates'],['ML-5.2'])
_v7_map(VPO_211,['vpo_program_description','v4_specialty_model_frame','v7_organization_admission_discovery'],['ML-5.4','ML-5.2'])
_v7_map(GRAD_ORG,['graduate_descriptive_statistics','v4_ml55_crosssection_metrics'],['ML-5.5'])
_v7_map(GRAD_FIELD,['graduate_specialty_description'],['ML-5.5'])
_v7_map(source_asset(segment='05'),['e04_e05_full_family_audit'],['ML-5.5'],'FULL_FAMILY_DESCRIPTION')
_v7_map(MUN,['municipal_demography','municipality_education_context'],['ML-5.7'])
_v7_map(TRANSPORT,['region_distance_proxy'],['ML-5.1'],'CONTEXT_PROXIED_NOT_LITERAL_REGION_DISTANCES')
_v7_map(source_asset(segment='02'),['d02_indicator_coverage'],['ML-5.1','ML-5.7'],'SEPARATE_THEMATIC_CONTEXT')
_v7_map(source_asset(segment='04'),['vpo_all_sections_audit'],['ML-5.2','ML-5.4'],'SEPARATE_VPO_SECTIONS')
_v7_map(source_asset(segment='03',role='education_od_matrix_raw_cells'),['e01_raw_cell_sources'],['ML-5.1'],'SOURCE_CROSSCHECK_NOT_ADDITIVE')
_v7_map(source_asset(segment='06'),['spatial_exact_crosswalk','spatial_D04_preview','spatial_BDMO_HIST_preview'],['ML-5.1','ML-5.7'],'SPATIAL_CONTEXT_WITH_MATCHING_GATE')

# Confirm only *actually used* predictive sources, based on target panel and model feature registry.
_v7_executed={r['module'] for r in MODEL_RECORDS}
_v7_real={}
_v7_real[E01['asset_id']]={'ML-5.1','ML-5.3'} & _v7_executed
_v7_real[VPO_211['asset_id']]={'ML-5.4'} & _v7_executed
_v7_real[E02['asset_id']]={'ML-5.2'} & _v7_executed
_v7_real[GRAD_ORG['asset_id']]={'ML-5.5'} & _v7_executed
_v7_real[MUN['asset_id']]={'ML-5.7'} & _v7_executed
_v7_od_features=set()
for _r in MODEL_RECORDS:
    if _r.get('module')=='ML-5.1':
        _v7_od_features.update(str(f) for f in (_r.get('feature_num') or []))
if 'distance_proxy_km' in _v7_od_features and len(DISTANCE):
    _v7_real[TRANSPORT['asset_id']]={'ML-5.1'}
if any(x.startswith(('origin_econ_','destination_econ_')) for x in _v7_od_features) and len(ECON_PANEL):
    for _id in ECON_PANEL.source_asset_id.dropna().astype(str).unique():
        _v7_real[str(_id)]={'ML-5.1'}

# Model ancestry table: source inputs for every saved trained model, no guessing from source names.
V7_MODEL_LINEAGE=[]
for _r in MODEL_RECORDS:
    _mod=_r.get('module','')
    _src=sorted(_id for _id,_mods in _v7_real.items() if _mod in _mods)
    if not _src and _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.7'):
        ISSUES.append({'module':_mod,'status':'V7_LINEAGE_MISSING','reason':'No verified source linkage for recorded model '+str(_r.get('path')),'required':'exact lineage'})
    V7_MODEL_LINEAGE.append({'module':_mod,'model':_r.get('model'),'protocol':_r.get('protocol'),
      'seed':_r.get('seed'),'model_file':_r.get('path'),'model_sha256':_r.get('file_sha256'),
      'asset_ids':';'.join(_src),'n_sources':len(_src),
      'feature_columns':';'.join([str(x) for x in (_r.get('feature_num') or [])]),
      'feature_categories':';'.join([str(x) for x in (_r.get('feature_cat') or [])]),
      'claim_scope':'AS_STORED_MODEL_FIT_AND_PANEL_DERIVATION_NOT_CAUSAL_SOURCE_ATTRIBUTION'})
V7_MODEL_LINEAGE=pd.DataFrame(V7_MODEL_LINEAGE)
qa_table('v7_model_lineage',V7_MODEL_LINEAGE,allow_empty=True)

V7_SOURCE_LINEAGE=[]
for _a in ASSETS:
    _id=_a['asset_id'];_profiles=_v7_source_panels.get(_id,[])
    _num=V7_SOURCE_AUDIT.loc[V7_SOURCE_AUDIT.asset_id.eq(_id)].iloc[0]
    _panels=sorted({p for x in _profiles for p in x['panels']})
    _modules=sorted({m for x in _profiles for m in x['modules']})
    _direct=sorted(_v7_real.get(_id,set()))
    _derived=ROOT/'derived'/'v4_source_values'/_num.numeric_profile_file
    V7_SOURCE_LINEAGE.append({'asset_id':_id,'segment':_a['segment'],
      'sha256':_a['sha256'],'rows_actual':int(_a['rows_actual']),
      'content_profile_sha256':hash_file(_derived) if _derived.is_file() else '',
      'source_coverage_pass':bool(_num.all_values_profiled and _num.verified_sha256 and _num.footer_rows_verified),
      'source_specific_analysis':'FULL_VALUE_DISTRIBUTION_YEAR_IF_AVAILABLE',
      'analytical_panels':';'.join(_panels),
      'research_modules_possible':';'.join(_modules),
      'actual_saved_model_modules':';'.join(_direct),
      'direct_model_input':bool(_direct),'model_usage_semantics':('PROVEN_PANEL_CHAIN' if _direct else 'THEMATIC_OR_CONTEXT_ONLY'),
      'nonmodel_reason':('MISMATCHED_STATISTICAL_GRAIN_OR_OVERLAPPING_SOURCE_OR_NOT_IDENTIFIED_AS_PREDICTOR'
                         if not _direct else ''),
      'not_invented_source_join':False})
V7_SOURCE_LINEAGE=pd.DataFrame(V7_SOURCE_LINEAGE)
qa_table('v7_source_lineage_52',V7_SOURCE_LINEAGE,['asset_id'])
if CFG.MODE=='FULL' and (len(V7_SOURCE_LINEAGE)!=len(ASSETS) or
    not V7_SOURCE_LINEAGE.source_coverage_pass.all() or
    not V7_MODEL_LINEAGE.n_sources.ge(1).all()):
    raise RuntimeError('V7 LINEAGE: исходники, модели или реестр происхождения не согласованы')

V7_SCIENCE_GATES=[]
for _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7'):
    _row=V5_LEDGER.loc[V5_LEDGER.module.eq(_mod)].iloc[0]
    _raw={
      'ML-5.1':'E01_REGION_OF_PRIOR_SCHOOLING_VS_CURRENT_SPO_STUDY_NOT_RESIDENCE',
      'ML-5.2':'VERIFIED_ADMISSIONS_BY_ORGANIZATION_YEAR_NOT_STUDENT_CONTINGENT',
      'ML-5.3':'OBSERVED_E01_OUTFLOW_SUM_OF_REPORTED_EDGES_NOT_RESIDENTIAL_MIGRATION',
      'ML-5.4':'REGIONAL_SPECIALTY_ADMITTED_COUNT_NOT_ORG_BY_SPECIALTY_OD',
      'ML-5.5':'EMPLOYMENT_SNAPSHOT_NOT_RETENTION_OR_LOCATION_OF_WORK',
      'ML-5.6':'DYNAMIC_OBSERVED_OD_NETWORK_WITH_COVERAGE_BREAK',
      'ML-5.7':'VERIFIED_EDUCATIONAL_CAMPUSES_AND_PREPERIOD_CAPACITY_BY_OKTMO'}[_mod]
    _original_ident=bool(_row.get('original_target_fully_identified',False))
    _status=MODULES[_mod].get('status','UNREPORTED')
    V7_SCIENCE_GATES.append({'module':_mod,'target_definition':_raw,
      'computational_pass':bool(_row.computational_pass),
      'original_scientific_target_identified':_original_ident,
      'scientific_status':str(_row.scientific_scope),
      'module_execution_status':_status,
      'model_records':int(V7_MODEL_LINEAGE.module.eq(_mod).sum()),
      'historical_vintage_proven':False,
      'remaining_identification_dependency':('ORIGINAL_SOURCE_DEFINITION_OR_SPATIAL_CONFIRMATION_REQUIRED' if not _original_ident else ''),
      'no_synthetic_data':True})
V7_SCIENCE_GATES=pd.DataFrame(V7_SCIENCE_GATES)
qa_table('v7_seven_scientific_gates',V7_SCIENCE_GATES,['module'])
print('V7: 52/52 lineages; true direct model sources',int(V7_SOURCE_LINEAGE.direct_model_input.sum()),
      '/52; models with source lineage',len(V7_MODEL_LINEAGE),'/',len(MODEL_RECORDS))


## 10В. Независимая приёмка реконструкций
Состав и происхождение признаков, годовые разрезы, 52/52 источника, фактические и пропущенные эксперименты; доказательные статусы без подмены научного результата технической готовностью.

In [ ]:
# ML-05 V7.3.0 · Per-experiment acceptance without inventing any missing research target.
progress_event('STATISTICS','V7.3: проверка предобработки, 52 источников и всех выполнимых экспериментов',announce=True)
ADR_RECONSTRUCTED_LINEAGE=V7_MODEL_LINEAGE.copy()
if len(ADR_RECONSTRUCTED_LINEAGE):
    for _idx,_r in ADR_RECONSTRUCTED_LINEAGE.iterrows():
        _sources=set(str(_r.asset_ids).split(';')) if pd.notna(_r.asset_ids) else set()
        if _r.module=='ML-5.7' and str(_r.model) in ('adr_E0','adr_E1'):
            _sources.update([MUN['asset_id'],SPO_ORG['asset_id'],SPATIAL['asset_id']])
        if _r.module=='ML-5.2':
            _sources.add(E02['asset_id'])
        _sources={s for s in _sources if s in ASSET_BY_ID}
        ADR_RECONSTRUCTED_LINEAGE.at[_idx,'asset_ids']=';'.join(sorted(_sources))
        ADR_RECONSTRUCTED_LINEAGE.at[_idx,'n_sources']=len(_sources)
    qa_table('adr_reconstructed_model_lineage',ADR_RECONSTRUCTED_LINEAGE,allow_empty=True)

# A fixed research contract: existing V7 calculations and all added observable subexperiments.
_adrex=pd.DataFrame(ADR_EXPERIMENTS)
_adr_qa=[]
for _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7'):
    _ev=_adrex.loc[_adrex.module.eq(_mod)] if len(_adrex) else pd.DataFrame()
    _bad=_ev.loc[_ev.status.str.startswith('FAILED',na=False)] if len(_ev) else pd.DataFrame()
    _v5=V5_LEDGER.loc[V5_LEDGER.module.eq(_mod)]
    _existing=bool(len(_v5) and _v5.iloc[0].computational_pass)
    _ran=bool(len(_ev) and _ev.status.isin(['MODEL_COMPLETED','V7_COMPLETED','DESCRIPTIVE_COMPLETED']).any())
    _scientific=bool(len(_v5) and _v5.iloc[0].original_target_fully_identified)
    _adr_qa.append({'module':_mod,'v7_computational_pass':_existing,'v71_added_experiments':len(_ev),
        'v71_observed_experiment_completed':_ran,'v71_new_failures':len(_bad),
        'original_scientific_target_fully_identified':_scientific,
        'scientific_scope':'ONLY_PUBLISHED_OBSERVED_TARGETS_AND_LEGALLY_MATCHED_PROXIES',
        'missing_data_not_fabricated':True,'missing_data_experiments_skipped':int(_ev.status.str.startswith('SKIPPED',na=False).sum()) if len(_ev) else 0})
ADR_SEVEN_MODULE_FINAL=qa_table('adr_v71_seven_module_acceptance',pd.DataFrame(_adr_qa),['module'])
ADR_COMPUTATIONAL_PASS=bool(V5_COMPUTATIONAL_PASS and ADR_SEVEN_MODULE_FINAL.v71_new_failures.eq(0).all())
ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED=bool(ADR_COMPUTATIONAL_PASS and ADR_SEVEN_MODULE_FINAL.original_scientific_target_fully_identified.all())
ADR_SOURCE_AUDIT=pd.DataFrame([{'asset_id':a['asset_id'],'sha256_expected':a['sha256'],
        'sha256_checked':INPUT_SHA.get(a['asset_id']),
        'is_source_checked':INPUT_SHA.get(a['asset_id'])==a['sha256'],
        'source_analytical_role':V7_SOURCE_LINEAGE.loc[V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']),'model_usage_semantics'].iloc[0]
            if V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']).any() else 'UNMAPPED',
        'source_methodological_grain':a.get('grain_status','UNVERIFIED'),
        'direct_model_input':bool(V7_SOURCE_LINEAGE.loc[V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']),'direct_model_input'].iloc[0])
            if V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']).any() else False} for a in ASSETS])
qa_table('adr_source_52_completeness',ADR_SOURCE_AUDIT,['asset_id'])

# Exact original inputs are immutable: compare expected Stage I fingerprint and do not reinterpret source rows as new observations.
if CFG.MODE=='FULL' and (len(ADR_SOURCE_AUDIT)!=52 or not ADR_SOURCE_AUDIT.is_source_checked.all()):
    raise RuntimeError('ADR SOURCE_GATE: source coverage failure')
if ADR_COMPUTATIONAL_PASS and len(ADR_RECONSTRUCTED_LINEAGE):
    if not ADR_RECONSTRUCTED_LINEAGE.n_sources.ge(1).all():
        raise RuntimeError('ADR MODEL_GATE: saved model lacks verified source lineage')
if len(ADR_ADMISSION_YEAR) and not (ADR_ADMISSION_YEAR.area_total.fillna(False).all() and
        ADR_ADMISSION_YEAR.specialty_total.fillna(False).all() and ADR_ADMISSION_YEAR.n_reporting_cells.eq(1).all()):
    raise RuntimeError('ADR SEMANTIC_GATE: source-reported admission cell is not an unambiguous total')
if len(ADR_GEO_VERIFIED) and not ADR_GEO_VERIFIED.identity_status.eq('PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION').all():
    raise RuntimeError('ADR SPATIAL_GATE: proxy falsely asserted as confirmed campus')

ADR_FINAL_REPORT={
 'version':CFG.VERSION,'source_catalog_entries':len(ASSETS),'source_sha_passed':int(ADR_SOURCE_AUDIT.is_source_checked.sum()),
 'all_seven_modules_reported':len(ADR_SEVEN_MODULE_FINAL)==7,'v7_technical_methods_retained':V5_COMPUTATIONAL_PASS,
 'v71_additional_experiment_count':len(ADR_EXPERIMENTS),'v71_additional_model_experiment_count':sum(
    r['status']=='MODEL_COMPLETED' for r in ADR_EXPERIMENTS),
 'v71_computational_pass':ADR_COMPUTATIONAL_PASS,
 'seven_original_scientific_targets_identified':ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED,
 'OD_all_year_balanced_edges':int(len(ADR_OD_BALANCED)),
 'VO_source_approved_org_year_admissions':int(len(ADR_ADMISSION_YEAR)),
 'SPO_spatially_reconstructed_site_year_proxies':int(len(ADR_GEO_VERIFIED)),
 'municipal_geocode_E0E1':ADR_ML57_GEO_STATUS,
 'methodological_rule':'MISSING_NOT_OBSERVED_IS_SKIPPED_NOT_ZERO_FILLED_AND_NOT_FABRICATED'
}
qa_table('adr_final_scientific_diagnostics',pd.DataFrame([{
    k:(json.dumps(v,ensure_ascii=False,default=str) if isinstance(v,(dict,list)) else v)
    for k,v in ADR_FINAL_REPORT.items()}]),allow_empty=True)
print('V7.3: source coverage',ADR_FINAL_REPORT['source_sha_passed'],'/52 | additional experiments',
    ADR_FINAL_REPORT['v71_additional_experiment_count'],'| computational gate',ADR_COMPUTATIONAL_PASS,
    '| scientific 7/7',ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED)


## 10Г. V7.3 · независимая приёмка исследовательских выводов

Проверка временной доступности, сохранённых моделей и всей цепочки от первичного источника до результата. Выбор по TEST не допускается.

In [ ]:
# ML-05 V7.3.0 · независимая научная/временная/публикационная диагностика.
progress_event('STATISTICS','V7.3: временные границы, выбор моделей, источник→признак→результат',announce=True)
import importlib.metadata as _v72_metadata

V72_REPRODUCIBILITY={
 'version':CFG.VERSION,'python':platform.python_version(),'mode':CFG.MODE,
 'source_commit':SOURCE_COMMIT,'source_catalog_blob_sha':index_info['sha'],
 'catalog_n_assets':len(ASSETS),'sha_verified_n':len(INPUT_SHA),
 'seed_main':int(CFG.RANDOM_SEED),'seeds_final':list(CFG.FINAL_SEEDS),
 'training_model_count':len(MODEL_RECORDS),'metric_rows':len(METRICS),
 'prediction_rows':int(PRED_TOTAL),'source_vintage':str(INDEX.get('created_at')),
 'prequential_historical_source_vintage_archived':False,
 'dependencies':{pkg:(_v72_metadata.version(pkg) if _v72_metadata.packages_distributions() else 'unknown')
                 for pkg in ()}
}
for _package in ('numpy','pandas','duckdb','pyarrow','scikit-learn','scipy','catboost',
                 'lightgbm','statsmodels','networkx','nbformat','xlsxwriter','requests'):
    try:V72_REPRODUCIBILITY['dependencies'][_package]=_v72_metadata.version(_package)
    except _v72_metadata.PackageNotFoundError:V72_REPRODUCIBILITY['dependencies'][_package]='NOT_INSTALLED'

# Explicit as-of protocol for each prediction test; not a real source-archived forecast.
V72_ASOF_AUDIT=pd.DataFrame()
_v72_hist=[]
for _m in METRICS:
    _ty=_m.get('target_year');_tag=_m.get('source_vintage',INDEX.get('created_at'))
    _ty_num=pd.to_numeric(_ty,errors='coerce')
    _vint=pd.to_datetime(str(_tag),utc=True,errors='coerce')
    _real_asof=bool(pd.notna(_ty_num) and pd.notna(_vint) and _vint.year<=int(_ty_num)-CFG.HORIZON)
    _v72_hist.append({'module':str(_m.get('module')),'model':str(_m.get('model')),
        'protocol':str(_m.get('protocol')),'seed':int(_m.get('seed',-1)),
        'target_year':int(_ty_num) if pd.notna(_ty_num) else None,
        'source_vintage':str(_tag),'archived_source_before_forecast_origin':_real_asof,
        'interpretation':('POINT_IN_TIME_SOURCE_VINTAGE' if _real_asof else
                          'RETROSPECTIVE_ONLY_CURRENT_SOURCE_VINTAGE')})
V72_ASOF_AUDIT=qa_table('v72_each_metric_asof_vintage_gate',pd.DataFrame(_v72_hist),allow_empty=True)
V72_REPRODUCIBILITY['n_point_in_time_validated_metrics']=int(V72_ASOF_AUDIT.archived_source_before_forecast_origin.sum()) if len(V72_ASOF_AUDIT) else 0
V72_REPRODUCIBILITY['n_retrospective_only_metrics']=int((~V72_ASOF_AUDIT.archived_source_before_forecast_origin).sum()) if len(V72_ASOF_AUDIT) else 0

# Paired evidence is already calculated from the same prediction entity-year keys, with Holm adjustment.
# Champion selection MUST NOT maximize performance on TEST when DEV labels were unavailable.
V72_MODEL_SELECTION=[]
_v72_m=pd.DataFrame(METRICS)
if len(_v72_m):
    _v72_m['MAE']=pd.to_numeric(_v72_m.MAE,errors='coerce')
    _v72_m['target_year']=pd.to_numeric(_v72_m.target_year,errors='coerce')
    _v72_m=_v72_m.loc[_v72_m.MAE.notna()&np.isfinite(_v72_m.MAE)].copy()
    for (_mod,_sector,_protocol,_year),_all in _v72_m.groupby(['module','sector','protocol','target_year'],dropna=False):
        _values=_all.groupby('model',dropna=False).MAE.agg(['mean','count']).reset_index()
        if not len(_values):continue
        _rank=_values.sort_values(['mean','model'],kind='stable')
        _best=_rank.iloc[0]
        _base=_values.loc[_values.model.eq('persistence')]
        _base=float(_base.iloc[0]['mean']) if len(_base) else np.nan
        _selected='persistence' if np.isfinite(_base) else 'NO_PREDECLARED_BASELINE'
        _choice='PREDECLARED_PERSISTENCE_NOT_TEST_OPTIMIZED'
        if len(CONFIRM):
            _confirm=CONFIRM.loc[(CONFIRM.module.eq(_mod)) &
                                 (CONFIRM.sector.eq(_sector)) &
                                 (CONFIRM.choice_basis.eq('DEV_ASOF'))]
            if len(_confirm):
                _candidate=str(_confirm.iloc[0].selected_model)
                if _candidate in set(_values.model.astype(str)):
                    _selected=_candidate;_choice='DEV_ASOF_PRESELECTED'
        _cur=_values.loc[_values.model.eq(_selected)]
        _selected_mae=float(_cur.iloc[0]['mean']) if len(_cur) else np.nan
        V72_MODEL_SELECTION.append({'module':_mod,'sector':_sector,'protocol':_protocol,
            'test_year':int(_year) if pd.notna(_year) else None,'selected_model_without_test_optimization':_selected,
            'selection_basis':_choice,'selected_test_MAE_retrospective':_selected_mae,
            'persistence_MAE_retrospective':_base,'best_TEST_MAE_for_diagnostics_only':float(_best['mean']),
            'best_TEST_model_not_selected':str(_best['model']),
            'n_test_models_reported':int(len(_values)),
            'historical_source_vintage_validated':False,
            'conclusion':'EXPLORATORY_RETROSPECTIVE_NO_TEST_BASED_DEPLOYMENT'})
V72_MODEL_SELECTION=qa_table('v72_ex_ante_model_selection_audit',
    pd.DataFrame(V72_MODEL_SELECTION),allow_empty=True)

# Independent natural-key model lineage, now with an explicit source/transform/panel link per model.
V72_MODEL_PROVENANCE=[]
_v72_sources={str(x.asset_id):x for x in V7_SOURCE_LINEAGE.itertuples(index=False)}
for _r in V7_MODEL_LINEAGE.itertuples(index=False):
    _ids=sorted({x for x in str(_r.asset_ids or '').split(';') if x in ASSET_BY_ID})
    for _asset in _ids:
        _s=_v72_sources[_asset]
        V72_MODEL_PROVENANCE.append({'module':_r.module,'model_file':_r.model_file,
          'model_sha256':_r.model_sha256,'asset_id':_asset,'source_sha256':ASSET_BY_ID[_asset]['sha256'],
          'transform_or_panel_ref':str(_s.analytical_panels),
          'numerical_features':str(_r.feature_columns),'categorical_features':str(_r.feature_categories),
          'source_relation':'PANEL_DERIVATION_NOT_CAUSAL_FEATURE_ATTRIBUTION',
          'reconstructed_target':False,
          'source_verified_sha256':INPUT_SHA.get(_asset)==ASSET_BY_ID[_asset]['sha256']})
V72_MODEL_PROVENANCE=qa_table('v72_verified_source_panel_model_links',pd.DataFrame(V72_MODEL_PROVENANCE),
                              ['model_file','asset_id'],allow_empty=True)

# Seven scientific gates, with observed tests and explicit skipped original targets side by side.
V72_MODULE_ACCEPTANCE=[]
for _row in V7_SCIENCE_GATES.itertuples(index=False):
    _matches=V72_RECON_DF[V72_RECON_DF.module.eq(_row.module)]
    _results=ADR_SEVEN_MODULE_FINAL.loc[ADR_SEVEN_MODULE_FINAL.module.eq(_row.module)]
    _base_ok=bool(_row.computational_pass)
    _new_errors=int(_matches.status.str.startswith('FAILED').sum()) if len(_matches) else 0
    V72_MODULE_ACCEPTANCE.append({'module':_row.module,
       'baseline_v711_computational_pass':_base_ok,
       'new_v72_validated_substudies':int(_matches.status.str.startswith('PASS').sum()) if len(_matches) else 0,
       'new_v72_skipped_no_source':int(_matches.status.str.startswith('SKIPPED').sum()) if len(_matches) else 0,
       'new_computational_failures':_new_errors,
       'original_target_identified':bool(_row.original_scientific_target_identified),
       'historical_vintage_validated':bool(_row.historical_vintage_proven),
       'honest_scientific_scope':str(_row.scientific_status),
       'acceptance':'COMPUTATIONAL_PASS_SCIENTIFIC_GATED' if _base_ok and not _new_errors else 'COMPUTATIONAL_FAIL',
       'no_missing_source_value_inferred':True})
V72_MODULE_ACCEPTANCE=qa_table('v72_seven_modules_scientific_acceptance',
                                pd.DataFrame(V72_MODULE_ACCEPTANCE),['module'])

V72_ENGINEERING_GATES={
 'all52_sources_sha':len(ASSETS)==52 and len(INPUT_SHA)==52 and
        all(INPUT_SHA.get(a['asset_id'])==a['sha256'] for a in ASSETS),
 'all_7_scientific_decisions_documented':len(V72_MODULE_ACCEPTANCE)==7 and
        V72_MODULE_ACCEPTANCE.module.nunique()==7,
 'preserved_v711_completed_models':len(MODEL_RECORDS)>=233,
 'preserved_v711_metrics':len(METRICS)>=254,
 'same_model_files_and_lineage':len(MODEL_RECORDS)==len(V7_MODEL_LINEAGE) and
        len(V72_MODEL_PROVENANCE)>=len(V7_MODEL_LINEAGE),
 'source_to_model_sha_all_valid':bool(len(V72_MODEL_PROVENANCE) and
        V72_MODEL_PROVENANCE.source_verified_sha256.all()),
 'reconstruction_has_no_failure':bool(len(V72_RECON_DF) and
        not V72_RECON_DF.status.str.startswith('FAILED').any()),
 'no_automatic_org_admission_from_unapproved_label':bool(not len(ADR_ADMISSION_YEAR) or
        bool(CFG.APPROVED_VO_ADMISSION_NAMES)),
 'no_geographic_proxy_becomes_official_campus':bool(not len(V72_SPATIAL_CONFIDENCE) or
        not V72_SPATIAL_CONFIDENCE.confirmed_campus_location.any()),
 'no_test_optimal_model_automatically_selected':bool(not len(V72_MODEL_SELECTION) or
        V72_MODEL_SELECTION.selection_basis.isin([
            'PREDECLARED_PERSISTENCE_NOT_TEST_OPTIMIZED','DEV_ASOF_PRESELECTED']).all()),
 'all_model_testing_vintage_scopes_explicit':len(V72_ASOF_AUDIT)==len(METRICS),
}
qa_table('v72_engineering_decision_gates',pd.DataFrame([
    {'gate':k,'passed':bool(v)} for k,v in V72_ENGINEERING_GATES.items()]),['gate'])
if CFG.MODE=='FULL' and not all(V72_ENGINEERING_GATES.values()):
    bad=[k for k,v in V72_ENGINEERING_GATES.items() if not v]
    raise RuntimeError('V7.2 independent QA failure: '+', '.join(bad))
V72_REPRODUCIBILITY['v72_gates']=V72_ENGINEERING_GATES
V72_REPRODUCIBILITY['new_observed_reconstruction_checks']=len(V72_RECON_DF)
V72_REPRODUCIBILITY['new_original_targets_verified']=int(V72_MODULE_ACCEPTANCE.original_target_identified.sum())
print('V7.3: 52 source records,',len(V72_MODEL_PROVENANCE),'model-source links,',
      len(V72_ASOF_AUDIT),'time-vintage controls,',len(V72_MODULE_ACCEPTANCE),'module decisions')


### 11б. V7.3.0 — новые количественные исследования: парные ошибки прогнозов, кластерный bootstrap, коррекция Holm и декомпозиция ошибок без повторного обучения 233 моделей

In [ ]:
# ML-05 V7.3.0 · NEW EMPIRICAL EXPERIMENTS: paired error differences on identical observations.
# The 233 established model artifacts and their metrics are NOT changed.
# Predictions from 50 seeds are averaged by entity-year before testing: seeds are not 50 independent observations.
# Whole test supports, no row sampling; cluster resampling avoids declaring each OD edge independent.
progress_event('STATISTICS','V7.3: реальные парные тесты прогнозных ошибок, bootstrap и Holm',announce=True)
from scipy.stats import norm as _v73_norm

V73_PAIRED_TESTS=[]
V73_ERROR_STRATA=[]
V73_PAIRED_AUDIT={
    'version':CFG.VERSION,'prediction_rows_input':int(PRED_TOTAL),
    'existing_metric_rows_before':int(len(METRICS)),
    'existing_models_before':int(len(MODEL_RECORDS)),
    'inference':'cluster-randomization sign-flips, grouped by reported geographical unit; exploratory retrospective',
    'selection_policy':'predeclared persistence; no model chosen by TEST performance',
    'resampling_repetitions':int(CFG.BOOTSTRAP_REPS),
    'raw_test_observations_not_independent_across_models':True,
    'multiple_testing':'Holm-Bonferroni across all qualified paired comparisons'
}

if PRED_STORAGE_FILE is None or not Path(PRED_STORAGE_FILE).is_file():
    raise RuntimeError('V7.3: нет полного Parquet прогнозов; парные эксперименты невозможны')
_v73_collapsed=ROOT/'derived'/'v73_predictions_ensemble_by_observation.parquet'
_v73_temp=_v73_collapsed.with_suffix('.tmp.parquet')
_v73_temp.unlink(missing_ok=True)
with duck_connect() as _v73_db:
    _v73_sql="""SELECT module,sector,split_id,target_id,target_year,entity_id,model,
           MIN(actual)::DOUBLE AS actual,MAX(actual)::DOUBLE AS actual_max,
           AVG(predicted)::DOUBLE AS prediction,
           COUNT(*)::BIGINT AS seed_runs,
           COUNT(DISTINCT seed)::BIGINT AS distinct_seeds,
           MAX(COALESCE(CAST(origin_region AS VARCHAR),CAST(parent_group_id AS VARCHAR),CAST(entity_id AS VARCHAR))) AS cluster_id
      FROM read_parquet(?)
      GROUP BY module,sector,split_id,target_id,target_year,entity_id,model"""
    # DuckDB streams COPY to disk; no 2.75M-row Pandas materialization.
    _v73_literal="'"+str(PRED_STORAGE_FILE).replace("'","''")+"'"
    _v73_target="'"+str(_v73_temp).replace("'","''")+"'"
    _v73_db.execute('COPY ('+_v73_sql.replace('read_parquet(?)','read_parquet('+_v73_literal+')')+') TO '+_v73_target+' (FORMAT PARQUET, COMPRESSION ZSTD)')
    _v73_pre=_v73_db.execute('''SELECT COUNT(*)::BIGINT,COUNT(*) FILTER(WHERE actual IS DISTINCT FROM actual_max)::BIGINT,
        COUNT(*) FILTER(WHERE seed_runs<>distinct_seeds)::BIGINT FROM read_parquet(?)''',[str(_v73_temp)]).fetchone()
    if _v73_pre[1] or _v73_pre[2]:
        raise RuntimeError(f'V7.3: несогласованность фактической величины или дубли сидов: {_v73_pre}')
    _v73_groups=_v73_db.execute('''SELECT DISTINCT module,sector,split_id,target_id,target_year
        FROM read_parquet(?) WHERE model='persistence' ORDER BY 1,2,3,4,5''',[str(_v73_temp)]).df()
_v73_temp.replace(_v73_collapsed)
V73_PAIRED_AUDIT['unique_model_observations']=int(_v73_pre[0])
V73_PAIRED_AUDIT['predeclared_baseline_groups']=int(len(_v73_groups))

# Reusable statistics, tested on a synthetic pair before querying the research data.
def v73_cluster_randomization(delta,cluster,reps=399,seed=1703):
    vals=pd.DataFrame({'d':np.asarray(delta,dtype=float),'g':np.asarray(cluster,dtype=str)})
    vals=vals[np.isfinite(vals.d)].copy()
    by=vals.groupby('g',sort=True).d.agg(['sum','count'])
    groups=int(len(by));rows=int(len(vals))
    if groups<8 or rows<25:return {'p_value':np.nan,'ci_low':np.nan,'ci_high':np.nan,'n_clusters':groups,'n_rows':rows,'inference_status':'INSUFFICIENT_INDEPENDENT_CLUSTERS'}
    rng=np.random.default_rng(int(seed))
    sums=by['sum'].to_numpy(float);counts=by['count'].to_numpy(float)
    diff=float(sums.sum()/counts.sum())
    bootstrap=np.empty(int(reps),dtype=float)
    flipped=np.empty(int(reps),dtype=float)
    for i in range(int(reps)):
        idx=rng.integers(0,groups,size=groups)
        bootstrap[i]=float(sums[idx].sum()/counts[idx].sum())
        flip=rng.choice(np.array([-1.,1.]),size=groups)
        flipped[i]=float((sums*flip).sum()/counts.sum())
    lo,hi=np.quantile(bootstrap,[0.025,0.975]).tolist()
    p=(1+int(np.count_nonzero(np.abs(flipped)>=abs(diff)-1e-12)))/(int(reps)+1)
    return {'p_value':p,'ci_low':float(lo),'ci_high':float(hi),'n_clusters':groups,'n_rows':rows,'inference_status':'EXPLORATORY_CLUSTER_SIGN_FLIP'}

_v73_probe=v73_cluster_randomization([2.0,-1.0]*15,[f'g{i}' for i in range(30)],reps=99)
assert _v73_probe['n_rows']==30 and _v73_probe['n_clusters']==30 and 0<_v73_probe['p_value']<=1

with duck_connect() as _v73_db:
    for _v73_i,_grp in enumerate(_v73_groups.itertuples(index=False),1):
        _keys=[str(_grp.module),str(_grp.sector),str(_grp.split_id),str(_grp.target_id),int(_grp.target_year)]
        progress_event('STATISTICS',f'V7.3: парные наблюдения {_v73_i}/{len(_v73_groups)} · {_keys[0]} / {_keys[2]}',_v73_i,len(_v73_groups),announce=False)
        _v73_paired=_v73_db.execute('''SELECT a.model,a.entity_id,a.cluster_id,a.actual,
             a.prediction AS candidate,b.prediction AS baseline,a.seed_runs,a.distinct_seeds
          FROM read_parquet(?) a INNER JOIN read_parquet(?) b
            ON a.module=b.module AND a.sector=b.sector AND a.split_id=b.split_id
             AND a.target_id=b.target_id AND a.target_year=b.target_year AND a.entity_id=b.entity_id
          WHERE b.model='persistence' AND a.model<>'persistence'
            AND a.module=? AND a.sector=? AND a.split_id=? AND a.target_id=? AND a.target_year=?''',
          [str(_v73_collapsed),str(_v73_collapsed),*_keys]).df()
        if _v73_paired.empty:continue
        for _v73_model,_v73_df in _v73_paired.groupby('model',sort=True):
            _v73_actual=_v73_df.actual.to_numpy(float)
            _v73_current=_v73_df.candidate.to_numpy(float)
            _v73_base=_v73_df.baseline.to_numpy(float)
            _v73_ok=np.isfinite(_v73_actual)&np.isfinite(_v73_current)&np.isfinite(_v73_base)
            if not _v73_ok.all():raise RuntimeError('V7.3: нечисловые парные прогнозы')
            _v73_diff=np.abs(_v73_actual-_v73_current)-np.abs(_v73_actual-_v73_base)
            _v73_cluster=_v73_df.cluster_id.fillna(_v73_df.entity_id).astype(str).to_numpy()
            _v73_stats=v73_cluster_randomization(_v73_diff,_v73_cluster,CFG.BOOTSTRAP_REPS,
                seed=int(CFG.RANDOM_SEED)+len(V73_PAIRED_TESTS))
            _v73_mae=float(np.abs(_v73_actual-_v73_current).mean())
            _v73_basemae=float(np.abs(_v73_actual-_v73_base).mean())
            V73_PAIRED_TESTS.append({'module':_keys[0],'sector':_keys[1],'protocol':_keys[2],
                'target_id':_keys[3],'test_year':_keys[4],'model':str(_v73_model),'comparator':'persistence',
                'n_paired_observations':int(len(_v73_diff)),
                'mean_candidate_MAE':_v73_mae,'mean_baseline_MAE':_v73_basemae,
                'MAE_delta_candidate_minus_baseline':float(_v73_diff.mean()),
                'candidate_better_on_test_descriptive_only':bool(_v73_mae<_v73_basemae),
                'n_model_seed_runs':int(_v73_df.seed_runs.max()),
                'source_vintage_verified_for_origin':False,
                **_v73_stats})
            _v73_strata=pd.cut(_v73_actual,[-np.inf,0.5,5.5,25.5,100.5,np.inf],
                labels=['0','1-5','6-25','26-100','>100'])
            _v73_part=pd.DataFrame({'stratum':_v73_strata,'actual':_v73_actual,
                'candidate_error':np.abs(_v73_actual-_v73_current),
                'baseline_error':np.abs(_v73_actual-_v73_base)})
            for _s,_part in _v73_part.groupby('stratum',observed=True):
                V73_ERROR_STRATA.append({'module':_keys[0],'sector':_keys[1],'protocol':_keys[2],
                   'target_id':_keys[3],'test_year':_keys[4],'model':str(_v73_model),'stratum':str(_s),
                   'n':int(len(_part)),'candidate_MAE':float(_part.candidate_error.mean()),
                   'persistence_MAE':float(_part.baseline_error.mean()),
                   'delta_MAE':float((_part.candidate_error-_part.baseline_error).mean()),
                   'all_observations_retained':True})
        del _v73_paired

V73_PAIRED_TESTS=pd.DataFrame(V73_PAIRED_TESTS)
V73_ERROR_STRATA=pd.DataFrame(V73_ERROR_STRATA)
if V73_PAIRED_TESTS.empty:raise RuntimeError('V7.3: нет ни одного парного сравнения с инерцией')
_v73_p=pd.to_numeric(V73_PAIRED_TESTS.p_value,errors='coerce')
_v73_valid=_v73_p.notna()&np.isfinite(_v73_p)&_v73_p.between(0,1)
V73_PAIRED_TESTS['p_holm']=np.nan
if _v73_valid.any():
    V73_PAIRED_TESTS.loc[_v73_valid,'p_holm']=multipletests(_v73_p[_v73_valid].to_numpy(float),method='holm')[1]
V73_PAIRED_TESTS['interpretation']=np.where(
    V73_PAIRED_TESTS.p_holm.lt(.05)&V73_PAIRED_TESTS.MAE_delta_candidate_minus_baseline.lt(0),
    'EXPLORATORY_ADVANTAGE_CANDIDATE',
    np.where(V73_PAIRED_TESTS.p_holm.lt(.05)&V73_PAIRED_TESTS.MAE_delta_candidate_minus_baseline.gt(0),
             'EXPLORATORY_ADVANTAGE_PERSISTENCE','NO_SIGNIFICANT_PAIRED_ADVANTAGE_OR_INSUFFICIENT_CLUSTERS'))
V73_PAIRED_AUDIT['paired_comparisons']=int(len(V73_PAIRED_TESTS))
V73_PAIRED_AUDIT['eligible_p_values']=int(_v73_valid.sum())
V73_PAIRED_AUDIT['error_strata']=int(len(V73_ERROR_STRATA))
V73_PAIRED_AUDIT['models_after']=int(len(MODEL_RECORDS))
V73_PAIRED_AUDIT['metrics_after']=int(len(METRICS))
V73_PAIRED_AUDIT['training_artifacts_unchanged']=bool(len(MODEL_RECORDS)==V73_PAIRED_AUDIT['existing_models_before'] and len(METRICS)==V73_PAIRED_AUDIT['existing_metric_rows_before'])
if not V73_PAIRED_AUDIT['training_artifacts_unchanged']:
    raise RuntimeError('V7.3: новые статистические тесты не должны изменять результаты обученных моделей')
qa_table('v73_paired_model_error_tests',V73_PAIRED_TESTS,allow_empty=False)
qa_table('v73_error_stratum_decomposition',V73_ERROR_STRATA,allow_empty=False)
qa_table('v73_prediction_pair_audit',pd.DataFrame([V73_PAIRED_AUDIT]))
print('V7.3: новые эмпирические результаты:',len(V73_PAIRED_TESTS),'парных тестов,',
      len(V73_ERROR_STRATA),'страт; Holm p-values:',int(_v73_valid.sum()),'| старые модели сохранены:',len(MODEL_RECORDS))


# STATISTICS заканчивается только после НОВЫХ V7.3 экспериментов.
mark_stage("STATISTICS")


## 11. Научные изображения
Диаграммы привязаны к исходному показателю и его периоду, без фиктивной геометрии и трактовки отсутствующей связи как нулевой.

In [ ]:
progress_event('VISUALS','Визуализация фактических E01, сетей и тестовых ошибок')
# Фигуры только на реально наблюдаемых исходных величинах; без псевдогеографических карт.
plt.rcParams.update({'font.family':'DejaVu Sans','figure.figsize':(10,5),'axes.grid':True})

def figure_save(stem):
    path=ROOT/'figures'/(stem+'.png')
    plt.tight_layout();plt.savefig(path,dpi=160,bbox_inches='tight');plt.close()
    FIGURES.append(str(path))
    return path

# Композиция берётся из неизменяемого PANELS, а не из локального counts (он изменяется в FEATURES).
od_comp=PANELS.get('od_composition',pd.DataFrame()).copy()
required={'observation_year','code_status','total_value'}
if not required.issubset(od_comp.columns):
    if {'observation_year','code_status','observed_value'}.issubset(OD.columns):
        od_comp=OD.groupby(['observation_year','code_status'],dropna=False,as_index=False).agg(total_value=('observed_value','sum'))
    else:fail('VISUALS','Потеряна первичная схема E01 / code_status')
if not od_comp.empty:
    od_comp['total_value']=pd.to_numeric(od_comp.total_value,errors='coerce')
    od_comp=od_comp.dropna(subset=['observation_year','code_status','total_value'])
    tab=od_comp.groupby(['observation_year','code_status']).total_value.sum().unstack(fill_value=0)
    tab=tab.reindex(columns=['INTER_REGION','INTRA_REGION','UNKNOWN_CODE'],fill_value=0).sort_index()
    if len(tab):
        fig,ax=plt.subplots(figsize=(10,5));tab.plot(kind='bar',stacked=True,ax=ax)
        ax.set(title='E01: состав зарегистрированного показателя (не постоянная миграция)',xlabel='Отчётный год',ylabel='Наблюдаемое значение E01')
        ax.legend(['Межрегиональные','Внутрирегиональные','Неопределённые коды'],fontsize=8)
        ax.tick_params(axis='x',rotation=0)
        figure_save('01_od_value_composition')
if len(coverage_df):
    cv=coverage_df.copy();cv['year']=pd.to_numeric(cv.year,errors='coerce');cv['observed_edges']=pd.to_numeric(cv.observed_edges,errors='coerce')
    cv=cv.dropna(subset=['year','observed_edges']).sort_values('year')
    if len(cv):
        fig,ax=plt.subplots(figsize=(10,4));ax.plot(cv.year,cv.observed_edges,'o-')
        flagged=cv.get('potential_break',pd.Series(False,index=cv.index)).fillna(False).astype(bool)
        for r in cv.loc[flagged].itertuples():
            ax.annotate('Изменение покрытия',(r.year,r.observed_edges),xytext=(0,12),textcoords='offset points',ha='center',fontsize=8)
        ax.set(title='E01: число зарегистрированных межрегиональных направлений',xlabel='Год',ylabel='Наблюдаемые пары регионов')
        ax.set_xticks(sorted(cv.year.astype(int).unique()))
        figure_save('02_edge_coverage')
if len(NETWORK):
    net=pd.DataFrame(NETWORK);year=int(net.year.max())
    x=net[net.year==year].nlargest(15,'in_strength').sort_values('in_strength')
    if len(x):
        fig,ax=plt.subplots(figsize=(11,6));ax.barh(x.region_code.astype(str),x.in_strength)
        ax.set(title=f'E01: крупнейшие регионы обучения, {year}',xlabel='Сумма исходного E01',ylabel='Код региона')
        figure_save('03_destination_centres')
if len(CONFIRM):
    q=CONFIRM[['module','baseline_mae','selected_mae']].copy()
    q['baseline_mae']=pd.to_numeric(q.baseline_mae,errors='coerce');q['selected_mae']=pd.to_numeric(q.selected_mae,errors='coerce')
    q=q.dropna();q=q[q.baseline_mae.gt(0)&q.selected_mae.ge(0)].copy()
    q['ratio']=q.selected_mae/q.baseline_mae;q=q[np.isfinite(q.ratio)]
    if len(q):
        fig,ax=plt.subplots(figsize=(10,5));at=np.arange(len(q));w=.34
        ax.bar(at-w/2,np.ones(len(q)),width=w,label='Инерция = 1,0')
        ax.bar(at+w/2,q.ratio,width=w,label='Выбранная до теста модель')
        ax.axhline(1,color='gray',linewidth=1,alpha=.7)
        ax.set_xticks(at);ax.set_xticklabels(q.module.astype(str).tolist())
        ax.set(ylabel='MAE / MAE инерции',title='Относительная ошибка на общих тестовых ключах')
        ax.legend();figure_save('04_mae_preselected_models')
print('Generated figures:',len(FIGURES))

mark_stage('VISUALS')


## 12. Экспорт: полный evidence в GitHub, аналитический Excel и воспроизводимый встроенный ноутбук

In [ ]:
memory_point('before_export')
progress_event('EXPORT','Excel, evidence-Parquet, модели, ноутбук и манифест')
import zipfile
from itertools import chain
EXCEL=ROOT/'exports'/'ML5_EDUCATION_MIGRATION_RESULTS.xlsx'
EVIDENCE=ROOT/'exports'/'ML5_EVIDENCE.zip'
NOTEBOOK=ROOT/'exports'/'ML5_EDUCATION_MIGRATION.ipynb'
MANIFEST=ROOT/'exports'/'run_manifest.json'

# Экспорт строго типизирован: смешанный int/string test_year не допускается.
STRING_HINTS={'entity_id','parent_group_id','origin_region','destination_region','region_code',
              'source_asset_id','oktmo','organization_id','indicator_name','test_period_label',
              'object_oktmo','specialty_code','area','campus_id','region_oktmo'}
INT_HINTS={'target_year','origin_year','observation_year','historical_year','historical_year_2',
           'information_cutoff_year','feature_available_year','target_available_year',
           'test_year','year','horizon','seed','graduation_year','salary_reference_year'}

def prepare_for_parquet(name,frame):
    f=frame.copy()
    f.columns=[str(c) for c in f.columns]
    if len(set(f.columns))!=len(f.columns):fail('EXPORT','Дубли названий столбцов '+name)
    if 'test_year' in f and f['test_year'].dtype in ('object','string'):
        original=f['test_year'].copy();numeric_year=number(original)
        descriptive=original.where(original.notna() & numeric_year.isna()).astype('string')
        if 'test_period_label' not in f:f['test_period_label']=descriptive.fillna('')
        else:f['test_period_label']=f['test_period_label'].astype('string').fillna(descriptive).fillna('')
        f['test_year']=numeric_year.astype('Int64')
    for c in f.columns:
        if c in INT_HINTS:
            z=number(f[c]);invalid=f[c].notna()&z.isna()
            if invalid.any():fail('PARQUET_SCHEMA',f'{name}.{c}: нечисловые годы/сиды')
            f[c]=z.astype('Int64')
        elif c in STRING_HINTS:f[c]=f[c].astype('string')
        elif pd.api.types.is_object_dtype(f[c]):
            sample=f[c].dropna()
            kinds={type(t) for t in sample.head(2000)}
            if not kinds or kinds<={str}:
                f[c]=f[c].astype('string')
            elif kinds<={int,float,np.int64,np.float64}:
                f[c]=number(f[c])
            elif any(isinstance(t,(dict,list,tuple,set)) for t in sample.head(2000)):
                f[c]=f[c].map(lambda v:json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                                    if isinstance(v,(dict,list,tuple,set)) else None if v is None else str(v)).astype('string')
            elif all(isinstance(t,(str,bytes,int,float,complex,np.number)) for t in sample.head(2000)):
                # Смешанный административный код переводится в строку только как metadata,
                # строгие числовые признаки/метрики не трогаем.
                if re.search(r'(value|actual|predicted|MAE|RMSE|salary|rate|population)',c,re.I):
                    fail('PARQUET_SCHEMA',f'{name}.{c}: смешанный тип числового поля')
                f[c]=f[c].astype('string')
            else:
                f[c]=f[c].map(lambda v:str(v) if pd.notna(v) else None).astype('string')
    return f

# Регрессионный тест известной ArrowInvalid: числа и текстовый исторический период.
_schema_probe=prepare_for_parquet('group_protocols',pd.DataFrame({
    'test_year':[2025,'historical',pd.NA],
    'test_period_label':['',pd.NA,''],
    'entity_id':['001','002','003']}))
if str(_schema_probe.test_year.dtype)!='Int64' or _schema_probe.test_period_label.iloc[1]!='historical':
    fail('PARQUET_SCHEMA','Смешанные test_year снова повреждают метку исторического периода')
del _schema_probe

EVIDENCE_PARQUET={}
def parquet_pack(name,frame):
    progress_event('EXPORT',f'Parquet {name}: {len(frame):,} строк, порциями',announce=True)
    # Физически пишем по 10 тыс. строк; никакой pa.Table из полного DataFrame.
    clean=prepare_for_parquet(name,frame)
    path=ROOT/'derived'/(name+'.parquet')
    write_df_chunks(clean,path)
    pf=pq.ParquetFile(path)
    if pf.metadata.num_rows!=len(clean):fail('EXPORT','row count changed in '+name)
    if pf.schema_arrow.names!=list(clean.columns):fail('EXPORT','schema changed in '+name)
    sample=pf.read_row_group(0).slice(0,min(100,len(clean))).to_pandas()
    if len(sample)!=min(len(clean),100):fail('EXPORT','roundtrip failed '+name)
    EVIDENCE_PARQUET[name]={'rows':len(clean),'columns':len(clean.columns),
        'schema':str(pf.schema_arrow),'sha256':hash_file(path),'path':str(path)}
    del sample,clean,pf;gc.collect()
    return path

def excel_table(writer,source,sheet,cap=None):
    if not isinstance(source,pd.DataFrame) or source.empty:return False
    # XlsxWriter constant_memory + последовательная запись строк (pandas.to_excel с ним несовместим).
    cap=int(cap or CFG.EXCEL_PREVIEW_ROWS)
    x=source.head(cap)
    ws=writer.book.add_worksheet(sheet[:31]);ws.freeze_panes(1,0)
    header=writer.book.add_format({'bold':True,'bg_color':'#294862','font_color':'#FFFFFF','text_wrap':True,'valign':'vcenter'})
    ws.write_row(0,0,[str(c) for c in x.columns],header)
    ws.set_row(0,30)
    for j,c in enumerate(x.columns):ws.set_column(j,j,min(43,max(11,len(str(c))+2)))
    for idx,row in enumerate(x.itertuples(index=False,name=None),1):
        values=[]
        for v in row:
            if v is None or v is pd.NA:values.append(None)
            elif isinstance(v,(dict,list,tuple,set)):values.append(json.dumps(v,ensure_ascii=False,default=str))
            elif isinstance(v,(datetime,pd.Timestamp)):values.append(str(v))
            elif isinstance(v,(np.integer,)):values.append(int(v))
            elif isinstance(v,(np.floating,)):values.append(float(v) if np.isfinite(v) else None)
            elif isinstance(v,(int,float,bool,str)):values.append(v if not isinstance(v,float) or np.isfinite(v) else None)
            else:values.append(str(v))
        ws.write_row(idx,0,values)
    ws.autofilter(0,0,len(x),len(x.columns)-1)
    return True

V7_EXCEL_EXTRA=[
 ('v7_all52_scientific_source_audit','V7 научный аудит 52'),
 ('v7_od_common_support_sensitivity','V7 OD общие пары'),
 ('v7_organization_admission_discovery','V7 кандидаты приема'),
 ('v7_municipal_identification_gate','V7 муниципалитеты'),
 ('v7_source_lineage_52','V7 происхождение 52'),
 ('v7_model_lineage','V7 модели источники'),
 ('v7_seven_scientific_gates','V7 семь целей')]
EXCEL_TABLES=[
    ('v73_paired_model_error_tests','V73 парные модели'),
    ('v73_error_stratum_decomposition','V73 страты ошибок'),
    ('v73_prediction_pair_audit','V73 приёмка прогнозов'),
    ('adr_preprocessing_provenance','V71 происхождение реконструкций'),
    ('adr_od_pairwise_comparable','V71 OD сопоставимые пары'),
    ('adr_od_balanced_edges_by_year','V71 OD общие рёбра'),
    ('adr_organizational_admission_dictionary','V71 поиск приёма E02'),
    ('adr_admission_organization_year','V71 приём подтвержден'),
    ('adr_spo_geolocation_candidates','V71 география СПО'),
    ('adr_spo_geolocation_high_confidence','V71 география высокая'),
    ('adr_all_feasible_experiments','V71 все эксперименты'),
    ('adr_ml57_geographic_execution','V71 E0 E1 география'),
    ('adr_ml57_spatial_e0e1_metrics','V71 E0 E1 результаты'),
    ('adr_v71_seven_module_acceptance','V71 7 модулей'),
    ('adr_source_52_completeness','V71 источники 52'),
    ('adr_reconstructed_model_lineage','V71 происхождение моделей'),
    ('adr_final_scientific_diagnostics','V71 научные границы'),

 ('v4_all52_source_value_audit','V4 данные 52 из 52'),
 ('v4_all52_research_use','V4 использование 52'),
 ('v4_source_scan_failures','V4 ошибки источников'),
 ('v4_specialty_admission_gate','V4 прием спец. данные'),
 ('v4_specialty_model_frame','V4 прогноз спец. база'),
 ('v4_organization_contingent_candidates','V4 контингент ВО'),
 ('v4_organization_contingent_model_frame','V4 прогноз контингента'),
 ('v4_additional_subexperiments','V4 дополнительные задачи'),
 ('v4_ml57_indicator_support','V4 охват БДМО'),
 ('v4_ml57_exact_common_support','V4 E0 E1 состав'),
 ('v4_ml57_results','V4 E0 E1 метрики'),
 ('v4_ml57_execution','V4 E0 E1 статус'),
 ('v4_ml55_crosssection_metrics','V4 занятость тест'),
 ('v4_ml55_execution','V4 выпуск тест'),
 ('v4_ml56_node_forecast','V4 сеть тест'),
 ('v4_exploratory_paired_comparisons','V4 парные сравнения'),
 ('v4_methods_execution_ledger','V4 выполнение методов'),
 ('v4_rolling_origin_feasibility','V4 скользящие годы'),
 ('v4_scientific_acceptance','V4 научная приемка'),
 ('full52_source_inventory', 'Все 52 источника'),
 ('seven_module_status_final', 'Семь модулей итог'),
 ('ml51_spatial_protocols', 'Пространственный тест OD'),
 ('ml54_history_persistence_diagnostic', 'Тренды спец. диагностика'),
 ('ml52_admission_candidates', 'Кандидаты приема'),
 ('full52_year_coverage', 'Все годы источников'),
 ('full52_theme_coverage', 'Охват тематик'),
 ('full52_raw_originals', 'Оригинальные XLSX'),
 ('bdmo_education_candidates', 'БДМО образование'),
 ('municipal_infrastructure_candidates', 'Инфраструктура ОКТМО'),
 ('spatial_exact_crosswalk', 'География ОКТМО'),
 ('e04_e05_full_family_audit', 'Все массивы выпускников'),
 ('vpo_all_sections_audit', 'Все разделы ВПО'),
 ('spo_annual_vs_master', 'СПО годовые сверки'),
 ('d02_indicator_coverage', 'Экономика 11 источников'),
 ('module_52_extension', 'Доп модули'),
 ('ml54_specialty_history', 'История специальностей'),
 ('ml55_outcome_disparities', 'Выпускники различия'),
 ('ml56_network_stress', 'Устойчивость сети'),
 ('ml57_common_support','БДМО E0 E1 исходные'),

 ('stage1_sources','Источники Stage I'),('semantic_indicator_catalog','Семантика индикаторов'),
 ('admission_approval_register','Утверждение индикаторов'),('registry_coverage_audit','Охват реестров'),
 ('organization_sector_subtasks','ВО СПО итоги модулей'),
 ('e01_original_headers','E01 первичные заголовки'),('e01_raw_cell_sources','E01 raw-проверка'),
 ('od_composition','Состав OD'),('od_edge_coverage','Покрытие OD'),
 ('od_edge_transitions','Новые и исчезнувшие OD'),('edu_org_registry','Реестр организаций'),
 ('edu_org_year','Приём ВО СПО'),('vpo_admission_audit','Приём ВПО-1'),
 ('program_profile','Программы'),('program_concentration','Концентрация программ'),
 ('vpo_program_description','ВПО программы отдельно'),
 ('graduate_full_scan_audit','E04 E05 полное сканирование'),('graduate_descriptive_statistics','Занятость выпускников'),('graduate_specialty_description','Выпуск по специальностям'),('municipality_education_context','Муниципалитеты E0 E1'),
 ('economic_indicator_candidates','Экономика источники'),('regional_economy','Региональные факторы'),
 ('region_distance_proxy','Транспорт и расстояния'),('education_network_year','Сетевые показатели'),
 ('education_network_communities','Сообщества'),('network_stability','Динамика сети'),
 ('seven_module_status','Семь модулей'),('module_diagnostics','Статусы модулей'),
 ('seven_hypotheses','Семь гипотез'),('model_metrics','Метрики'),
 ('confirmatory_comparisons','Подтверждающие тесты'),('split_audit','Временные разбиения'),
 ('future_mutation_canaries','Защита от утечек'),('metrics_recomputed_from_predictions','Независимый пересчёт'),
 ('entity_match_quarantine','Спорные совпадения'),('graduate_outcomes','Выпускники исходные'),
 ('edu_od_interregional','Рёбра межрегиональные')]

EXCEL_TABLES += V7_EXCEL_EXTRA
EXCEL_TABLES += [
 ('v72_od_year_pair_coverage_decomposition','V72 OD годовые пары'),
 ('v72_org_admission_indicator_decision','V72 E02 приём семантика'),
 ('v72_origin_common_support_outflow','V72 отток общие рёбра'),
 ('v72_specialty_code_observed_crosswalk','V72 коды специальностей'),
 ('v72_graduate_family_independent_descriptives','V72 E04 E05 отдельно'),
 ('v72_network_same_node_edge_dynamics','V72 сеть общие связи'),
 ('v72_spo_municipal_proxy_confidence','V72 география карантин'),
 ('v72_all_seven_reconstruction_evidence','V72 предобработка'),
 ('v72_each_metric_asof_vintage_gate','V72 временные винтажи'),
 ('v72_ex_ante_model_selection_audit','V72 выбор моделей'),
 ('v72_verified_source_panel_model_links','V72 источник модель'),
 ('v72_seven_modules_scientific_acceptance','V72 семь модулей'),
 ('v72_engineering_decision_gates','V72 контроль программы')
]

passport=pd.DataFrame([
  {'Параметр':'Проект','Значение':'Демографический советник · ML-05 Образовательная миграция'},
  {'Параметр':'Версия','Значение':CFG.VERSION},
  {'Параметр':'Режим','Значение':CFG.MODE},
  {'Параметр':'Снапшот Stage I','Значение':SOURCE_COMMIT},
  {'Параметр':'Каталог Stage I','Значение':index_info['sha']},
  {'Параметр':'Наборы в каталоге','Значение':len(ASSETS)},
  {'Параметр':'Наблюдаемая OD-связка','Значение':'Регион общего образования → регион СПО; не является фактом переезда'},
  {'Параметр':'E01 зачисления','Значение':'Не подтверждены первичным определением'},
  {'Параметр':'Статус научного исполнения','Значение':STATE['scientific']},
  {'Параметр':'Ограничение временного теста','Значение':'Исторические source vintages не доступны; ретроспективная проверка'},
  {'Параметр':'Публикация','Значение':'Только FULL при локальной приёмке и явно включённом разрешении'},
])

# При повторном запуске после ошибки NOTEBOOK_CAPTURE не пересоздаём огромный evidence.
def _completed_evidence_available():
    if not all(p.is_file() and p.stat().st_size>100 for p in (EXCEL,EVIDENCE)):return False
    try:
        with zipfile.ZipFile(EVIDENCE) as z:
            cfg=json.loads(z.read('metadata/config.json'))
            names=z.namelist()
            return (cfg.get('VERSION')==CFG.VERSION and 'metadata/v5_module_acceptance.json' in names
                    and 'predictions/education_predictions.parquet' in names
                    and any(p.startswith('models/') for p in names)
                    and any(p.startswith('panels/') for p in names))
    except Exception:return False

_REUSE_EXPORT=_completed_evidence_available()
if _REUSE_EXPORT:
    progress_event('EXPORT','Готовые Excel/evidence V7.3.0 обнаружены: повторная упаковка не требуется',announce=True)
    # Восстанавливаем реестр Parquet, сброшенный в начале этой ячейки.
    with zipfile.ZipFile(EVIDENCE) as z:
        for member in z.namelist():
            if not member.endswith('.parquet') or not member.startswith(('panels/','predictions/')):continue
            key=Path(member).stem
            p=Path(PRED_STORAGE_FILE) if key=='education_predictions' and PRED_STORAGE_FILE else Path(PANEL_FILES.get(key,ROOT/'derived'/(key+'.parquet')))
            if not p.is_file():
                p=ROOT/'derived'/(key+'.parquet');p.parent.mkdir(parents=True,exist_ok=True)
                with z.open(member) as source,p.open('wb') as target:shutil.copyfileobj(source,target,4*1024**2)
            pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
              'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
    progress_event('EXPORT',f'Восстановлен реестр {len(EVIDENCE_PARQUET)} Parquet без пересчёта',announce=True)

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка многолистового Excel',announce=True)
    with pd.ExcelWriter(EXCEL,engine='xlsxwriter',engine_kwargs={'options':{'constant_memory':True,'strings_to_formulas':False,'strings_to_urls':False}}) as writer:
        excel_table(writer,passport,'Паспорт')
        for _i,(key,title) in enumerate(EXCEL_TABLES,1):
            progress_event('EXPORT',f'Excel {_i}/{len(EXCEL_TABLES)}: {key}',_i,len(EXCEL_TABLES),announce=False)
            excel_table(writer,PANELS.get(key),title)
        excel_table(writer,pd.DataFrame(TABLE_QA),'Контроль таблиц')
        excel_table(writer,pd.DataFrame(ISSUES),'Ограничения')
        excel_table(writer,pd.DataFrame(LOG),'Журнал диагностики')

METHODOLOGY='''# ML-05 V7.3.0. Научный протокол

V7.3: дополнительные сравнения на одинаковых тестовых наблюдениях с усреднением 50 сидов.
Статистические выводы по кластерным перестановкам и Holm только разведочные, не исторический прогноз.

Восемь различных событий мобильности не отождествляются.
E01 region_of_previous_general_education × region_of_current_study НЕ является доказанным годовым приёмом.
Ретроспективные модели оценивают только измеренную E01 величину; нулевые отсутствующие рёбра не создаются.
Разбиения train/DEV/TEST подчинены доступности меток по консервативному лагу. Если DEV не был известен к test origin,
победителя из теста не выбирают; persistence остаётся предзаданной основной моделью.
Историческая версия источника 2026 не доказывает фактическую доступность значений в 2021–2025.
Сети описывают только наблюдаемые связи регионов предыдущего образования и обучения.
Выпускники E04/E05 — отдельный employment_reference 2024-12-31: место работы не известно, retention нельзя оценить.
Пересекающиеся географические/гендерные агрегаты E04/E05 нельзя суммировать, поэтому описаны медианы исходных строк по разрезам.
В пространственном E0/E1 отсутствие подтверждённой образовательной площадки не равно отсутствию площадки.
Экономические и транспортные признаки включаются только при доказанном grain и лаге.
Отрицательные результаты сохраняются, причинные эффекты по прогнозной важности не идентифицируются.
'''

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка evidence-ZIP и контроль отдельных Parquet',announce=True)
    with zipfile.ZipFile(EVIDENCE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
        z.writestr('research_methods.md',METHODOLOGY)
        z.writestr('metadata/config.json',json.dumps(asdict(CFG),ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/plan.json',json.dumps(EXPERIMENT_PLAN,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/semantics_e01.json',json.dumps(OD_SEMANTIC,ensure_ascii=False,indent=2))
        z.writestr('metadata/qa.json',json.dumps(TABLE_QA,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/source_hashes.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/issues.json',json.dumps(ISSUES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/module_status.json',json.dumps(MODULES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/splits.json',json.dumps(SPLIT_AUDIT,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_sha.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/v5_module_acceptance.json',V5_LEDGER.to_json(orient='records',force_ascii=False,indent=2))
        z.writestr('metadata/v5_model_registry.json',json.dumps(MODEL_RECORDS,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_protocol.md', 'Каждый из 52 Parquet: SHA-256, число строк, schema, годовой охват. '+'Годовые shards СПО не суммируются с общей панелью. Другие уровни данных не приписываются образовательному OD.')
        _export_panels=list(PANELS.items())
        for _i,(key,df) in enumerate(_export_panels,1):
            progress_event('EXPORT',f'Evidence {_i}/{len(_export_panels)}: {key}',_i,len(_export_panels),announce=False)
            if not isinstance(df,pd.DataFrame) or df.empty:continue
            if key in PANEL_FILES:
                p=Path(PANEL_FILES[key]);pf=pq.ParquetFile(p)
                EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                    'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            else:p=parquet_pack(key,df)
            z.write(p,'panels/'+key+'.parquet')
        if PRED_STORAGE_FILE is not None:
            p=Path(PRED_STORAGE_FILE);pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET['education_predictions']={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            z.write(p,'predictions/education_predictions.parquet')
        # Large source-grouped context: include every successfully built Parquet, not only Excel preview.
        for p in CONTEXT_PARQUET_FILES:
            if not Path(p).exists():fail('EXPORT','Отсутствует исходный контекстный Parquet '+str(p))
            z.write(p,'context/'+Path(p).name)
        for path in FIGURES:z.write(path,'figures/'+Path(path).name)
        for name,path in MODEL_FILES.items():z.write(path,'models/'+name)
        z.writestr('metadata/v73_paired_inference.json',json.dumps(V73_PAIRED_AUDIT,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/v72_reproducibility.json',json.dumps(V72_REPRODUCIBILITY,ensure_ascii=False,indent=2,default=str))
        z.writestr('metadata/v72_seven_module_acceptance.json',V72_MODULE_ACCEPTANCE.to_json(orient='records',force_ascii=False,indent=2))
        z.writestr('logs/run_events.json',json.dumps(LOG,ensure_ascii=False,indent=2))
        z.write(ROOT/'logs'/'stages_v73.jsonl','logs/stages_v73.jsonl')

# V6 SOURCE CAPTURE: восстанавливаем полный код из самого ноутбука; Colab get_ipynb НЕ используется.
progress_event('EXPORT','Сохранение встроенного неизменяемого V7.3.0 notebook',announce=True)
snap=notebook_from_embedded_source()
for _cell in snap.cells:
    if _cell.cell_type=='code':
        _cell.outputs=[]
        _cell.execution_count=None
nbformat.validate(snap)
clean_text=nbformat.writes(snap)
_token=globals().get('_TOKEN')
if (isinstance(_token,str) and len(_token)>15 and _token in clean_text) or re.search(r'(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}',clean_text):
    raise RuntimeError('V7.3.0 SECURITY: notebook contains GitHub token; export blocked')
_tmp_notebook=NOTEBOOK.with_suffix('.tmp.ipynb')
nbformat.write(snap,_tmp_notebook)
_tmp_notebook.replace(NOTEBOOK)
_saved_nb=nbformat.read(NOTEBOOK,as_version=4)
if [c.source for c in _saved_nb.cells]!=[c.source for c in snap.cells]:
    raise RuntimeError('V7.3.0: notebook roundtrip changed sources')
progress_event('EXPORT',f'V7.3.0 notebook saved, {len(snap.cells)} cells',announce=True)

# Manifest создаётся после локальных артефактов и содержит их реальные SHA; его собственный SHA нельзя включить в него же.
output_hashes={p.name:hash_file(p) for p in [EXCEL,EVIDENCE,NOTEBOOK] if p.exists()}
manifest={
 'version':CFG.VERSION,'mode':CFG.MODE,'created_utc':stamp(),
 'source_repo':CFG.REPO,'source_commit':SOURCE_COMMIT,'source_index_git_blob_sha':index_info['sha'],
 'source_index_version':INDEX['version'],'source_hashes':INPUT_SHA,
 'context_derived_files':{Path(p).name:{'rows':pq.ParquetFile(p).metadata.num_rows,'sha256':hash_file(p)} for p in CONTEXT_PARQUET_FILES},
 'catalog_52_coverage':{'registered':len(ASSETS),'sha_verified':len(INPUT_SHA),
 'verified_every_asset':len(INPUT_SHA)==len(ASSETS),
 'year_profiled':len(FULL52_YEAR_STATS)},
 'experiment_plan_sha256':PLAN_HASH,'v5_science_gates':{'all52_source_content_verified':not V4_SOURCE_ERRORS,'all_feasible_computations_succeeded':V5_COMPUTATIONAL_PASS,'all_seven_scientifically_identified':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'model_failure_count':int((~V5_LEDGER.computational_pass).sum())},'status':STATE.copy(),'module_status':MODULES,
 'observed_od_counts':{'all':len(OD),'inter':len(OD_INTER),'intra':len(OD_INTRA),'unknown':len(OD_UNKNOWN)},
 'real_prediction_rows':PRED_TOTAL,'v5_model_file_count':len(MODEL_FILES),
 'v5_module_acceptance':V5_MODULE_ACCEPTANCE,
 'v5_original_seven_target_identifiable':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'metric_rows':len(METRICS),'panels':{
      k:{'rows':v['rows'],'columns':v['columns'],'sha256':v['sha256']} for k,v in EVIDENCE_PARQUET.items()},
 'adr_advanced_data_reconstruction':ADR_FINAL_REPORT,
 'v72_reproducibility':V72_REPRODUCIBILITY,
 'v73_paired_inference_audit':V73_PAIRED_AUDIT,
 'v73_paired_comparisons':len(V73_PAIRED_TESTS),
 'v73_error_strata':len(V73_ERROR_STRATA),
 'v72_module_decisions':V72_MODULE_ACCEPTANCE.to_dict('records'),
 'v72_new_preprocessing':V72_RECON_DF.to_dict('records'),
 'adr_experiment_ledger':ADR_EXPERIMENTS,
 'output_sha256':output_hashes,'runtime_seconds':round(time.monotonic()-START_TIME,1),
 'limitations':ISSUES,'publication_status':'NOT_PUBLISHED',
 'commit_self_reference':'Commit SHA can be reported after Git commit; cannot be embedded in the same hashed blob'
}
progress_event('EXPORT','Манифест с SHA-256 локальных результатов',announce=True)
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Local package:',EXCEL.name,round(EXCEL.stat().st_size/1e6,2),'MB;',
      EVIDENCE.name,round(EVIDENCE.stat().st_size/1e6,2),'MB; notebook:',NOTEBOOK.exists())

memory_point('after_export')
mark_stage('EXPORT')


## 13. Строгая локальная приёмка семи модулей, численных результатов и 52 источников

In [ ]:
progress_event('QA_AND_PUBLICATION','Локальная приёмка, воспроизводимость и безопасная публикация')
# Финальный QA запускается до первого GitHub-запроса на запись.
LOCAL_CHECKS=[]
def check(name,condition,why=''):
    ok=bool(condition)
    LOCAL_CHECKS.append({'check':name,'status':'PASS' if ok else 'FAIL','reason':why})
    if not ok:note('QA',name+': '+why,'FAIL')
    return ok

check('DUCKDB_SQL_PREFLIGHT_PRODUCTION_QUERIES',bool(globals().get('SQL_PREFLIGHT_PASSED')),
      'Production COPY, candidate SUMMARY and org-year TOTAL must parse and execute before Stage I download')
check('INPUT_SOURCE_PIN',bool(SOURCE_COMMIT and index_info['sha']==CFG.EXPECTED_INDEX_BLOB_SHA))
check('SEVEN_MODULES_REPORTED',len(MODULES)==7 and all(v['status']!='NOT_STARTED' for v in MODULES.values()))
check('EXPECTED_CATALOG',len(ASSETS)==52)
check('FULL52_SHA_COVERAGE',CFG.MODE!='FULL' or (len(INPUT_SHA)==len(ASSETS) and all(a['asset_id'] in INPUT_SHA for a in ASSETS)))
check('FULL52_ROW_COVERAGE',CFG.MODE!='FULL' or (len(FULL52_INVENTORY)==len(ASSETS) and all(r.get('footer_rows_ok') for r in FULL52_INVENTORY)))
check('FULL52_YEAR_DISTRIBUTION',CFG.MODE!='FULL' or all(r['year_scan_status'] in ('SCANNED_FULL','NO_TIME_KEY') for r in FULL52_INVENTORY))
check('FULL52_D02_PROFILE',CFG.MODE!='FULL' or (len(D02_PROFILE)==11 and all(r['status']=='GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT' for r in D02_PROFILE)))
check('FULL52_VPO_PROFILE',CFG.MODE!='FULL' or (len(VPO_ALL_PROFILE)==11 and all(r['status']=='PASS' for r in VPO_ALL_PROFILE)))
check('FULL52_E04_E05_PROFILE',CFG.MODE!='FULL' or (len(FAMILY_E04E05)==4 and all(r['status']=='PASS_NOT_ADDITIVE' for r in FAMILY_E04E05)))
check('FULL52_CONTEXT_EXPORT',CFG.MODE!='FULL' or len(CONTEXT_PARQUET_FILES)>=26)
check('V4_SOURCE_VALUES_52',CFG.MODE!='FULL' or (len(V4_SOURCE_PROFILES)==52 and not V4_SOURCE_ERRORS))
check('V4_MODEL_FAILURES_ZERO',V4_COMPUTATIONAL_COMPLETE,'Failed model or source value analysis is a real computational failure')
check('V5_COMPLETE_COMPUTATIONAL_CONTRACT',V5_COMPUTATIONAL_PASS,'All feasible models and checks must complete; no hidden failures')
check('V5_UNIQUE_MODEL_ARTIFACTS',len(MODEL_FILES)==len(MODEL_RECORDS),'No overwritten spatial model files')
check('V5_ALL_MODELS_VERIFIED',bool(V5_MODEL_QA) and all(x['sha_verified'] for x in V5_MODEL_QA))
check('V5_SEVEN_MODULE_LEDGER',len(V5_LEDGER)==7 and V5_LEDGER.module.nunique()==7)
check('V5_ALL52_SOURCE_EVIDENCE',CFG.MODE!='FULL' or len(V5_SOURCE_EVIDENCE)==52)
check('V5_ML54_NO_SAMPLE',CFG.MODE!='FULL' or (len(PANELS.get('v5_ml54_full_strata_coverage',pd.DataFrame()))==1))
check('V4_FINAL_LEDGER',len(V4_METHOD_EXECUTIONS)==7)
check('V4_SCIENTIFIC_TRUTHFUL',not V4_SCIENTIFIC_COMPLETE or (len(ORG_YEAR)>0 and OD_SEMANTIC.get('yearly_count_unit_confirmed',False)))
check('E01_PRIMARY_OPENED',CFG.MODE!='FULL' or bool(OD_SEMANTIC.get('original_opened')))
check('SEMANTIC_APPROVAL_GATED',not len(ORG_YEAR) or all(
    name in set(CFG.APPROVED_VO_ADMISSION_NAMES)|set(CFG.APPROVED_SPO_ADMISSION_NAMES)
    for name in ORG_YEAR.indicator_name.dropna().astype(str).unique()))
check('EDGES_PARTITION',len(OD)==len(OD_INTER)+len(OD_INTRA)+len(OD_UNKNOWN))
check('INTRA_EXCLUDED',OD_INTER.origin_region.ne(OD_INTER.destination_region).all())
check('NO_FUTURE_FEATURES',all(v.get('status')!='FAIL' for v in CANARIES))
check('CANARIES_EXECUTED',any(v.get('module')=='ML-5.1' and v.get('status')=='PASS' for v in CANARIES) and
      any(v.get('module')=='ML-5.3' and v.get('status')=='PASS' for v in CANARIES))
check('NO_TEST_SELECTED_MODEL',not len(CONFIRM) or
      (CONFIRM.choice_basis.isin(['DEV_ASOF','NO_CONFIRMATORY_COMPARISON_NO_DEV']).all() and
       CONFIRM.loc[CONFIRM.choice_basis.eq('NO_CONFIRMATORY_COMPARISON_NO_DEV'),
                   'selected_model'].eq('persistence').all()))
check('NEGATIVE_FINDINGS_PRESERVED',len(HYPOTHESES)==7)
check('ZIP_CRC',zipfile.ZipFile(EVIDENCE).testzip() is None)
with zipfile.ZipFile(EXCEL) as z:
    check('EXCEL_OPENS',z.testzip() is None and 'xl/workbook.xml' in z.namelist())
check('MODEL_RELOAD',all(m.get('reloaded') for m in MODEL_RECORDS))
check('NOTEBOOK_CAPTURED',NOTEBOOK.exists(),'Необходимо получить очищенный самостоятельный Colab notebook')
if NOTEBOOK.exists():
    try:
        nb=nbformat.read(NOTEBOOK,as_version=4)
        nbformat.validate(nb)
        content=NOTEBOOK.read_text(encoding='utf-8')
        suspicious=re.findall(r'\b(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}\b',content)
        check('NOTEBOOK_NO_TOKEN',not suspicious and _TOKEN not in content)
        check('NOTEBOOK_NO_OUTPUTS',sum(len(c.get('outputs',[])) for c in nb.cells if c.cell_type=='code')==0)
        check('NOTEBOOK_SELF_CONTAINED',not re.search(r'(?m)^\s*(?:%run|%%writefile)\b',content))
        check('NOTEBOOK_REQUIRED_SECTION_COUNT',sum(c.cell_type=='code' for c in nb.cells)>=12)
        check('NOTEBOOK_REASONABLE_SIZE',NOTEBOOK.stat().st_size<2_000_000)
    except Exception as ex:check('NOTEBOOK_VALIDATE',False,type(ex).__name__)

# Проверяем независимые Parquet исходя из физического ZIP, не доверяя только собственным логам.
with zipfile.ZipFile(EVIDENCE) as z:
    for name,info in EVIDENCE_PARQUET.items():
        target=('predictions/' if name=='education_predictions' else 'panels/')+name+'.parquet'
        if target not in z.namelist():
            check('PARQUET_'+name,False,'Нет файла в evidence');continue
        try:
            # Не z.read(): весь Parquet в bytes может мгновенно исчерпать RAM.
            p=Path(info['path']);pf=pq.ParquetFile(p)
            member=z.getinfo(target)
            check('PARQUET_'+name,int(pf.metadata.num_rows)==info['rows'] and
                  len(pf.schema_arrow.names)==info['columns'] and
                  int(member.file_size)==p.stat().st_size and hash_file(p)==info['sha256'])
        except Exception as ex:check('PARQUET_'+name,False,type(ex).__name__)

# V7: repeat independent source/model lineage checks on physical exported tables.
check('V7_SOURCE_AUDIT_ALL52',CFG.MODE!='FULL' or
      (len(V7_SOURCE_AUDIT)==52 and V7_SOURCE_AUDIT.verified_sha256.all() and
       V7_SOURCE_AUDIT.all_values_profiled.all() and V7_SOURCE_AUDIT.numeric_profile_exists.all()))
check('V7_LINEAGE_52_UNIQUE',len(V7_SOURCE_LINEAGE)==len(ASSETS) and
      V7_SOURCE_LINEAGE.asset_id.nunique()==len(ASSETS) and
      V7_SOURCE_LINEAGE.source_coverage_pass.all())
check('V7_ALL_MODELS_LINKED',len(V7_MODEL_LINEAGE)==len(MODEL_RECORDS) and
      (V7_MODEL_LINEAGE.n_sources.ge(1).all() if len(V7_MODEL_LINEAGE) else True),
      'Every recorded model needs a traceable source input')
check('V7_ORIGINAL_7_GATES_REPORTED',len(V7_SCIENCE_GATES)==7 and
      V7_SCIENCE_GATES.module.nunique()==7 and
      V7_SCIENCE_GATES.computational_pass.all())
check('V7_OD_COMPARABLE_COMMON_SUPPORT',bool(len(V7_OD_COMPARABLE)) and
      all(x['shared_edges']<=min(x['observed_edges_previous'],x['observed_edges_current'])
          for x in V7_OD_COMPARABLE))
check('V71_PUBLISH_CHECK_SEPARATED_FROM_LOCAL_QA',
      bool(MANIFEST.is_file() and EXCEL.is_file() and EVIDENCE.is_file() and
           manifest.get('version')==CFG.VERSION),
      'Remote publication is checked only after uploading and reading the GitHub Release')
check('V71_ADVANCED_SOURCE_COVERAGE',len(ADR_SOURCE_AUDIT)==52 and ADR_SOURCE_AUDIT.is_source_checked.all())
check('V71_SEVEN_MODULE_RECONSTRUCTION_LEDGER',len(ADR_SEVEN_MODULE_FINAL)==7 and ADR_SEVEN_MODULE_FINAL.module.nunique()==7)
check('V71_RECONSTRUCTION_MODEL_AND_SEMANTICS',ADR_COMPUTATIONAL_PASS,
      'All actually executed reconstruction models passed; missing source targets are explicitly skipped')
check('V71_NO_INVENTED_ADMISSIONS',not len(ADR_ADMISSION_YEAR) or bool(CFG.APPROVED_VO_ADMISSION_NAMES))
check('V71_NO_INVENTED_CAMPUSES',not len(ADR_GEO_VERIFIED) or
      ADR_GEO_VERIFIED.identity_status.eq('PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION').all())

# V7.3 · new independent gates (scientific limits are not technical failures).
for _name,_passed in V72_ENGINEERING_GATES.items():
    check('V72_'+_name.upper(),_passed,'Independently computed source, model and scientific contract')
check('V72_RESEARCH_MODULES_SEVEN',len(V72_MODULE_ACCEPTANCE)==7)
check('V72_RECONSTRUCTIONS_RECORDED',len(V72_RECON_DF)>=7)
check('V72_SOURCE_VINTAGE_NOT_MISLABELED',
      len(V72_ASOF_AUDIT)==len(METRICS) and
      not V72_MODEL_SELECTION.historical_source_vintage_validated.any(),
      'Retrospective results must not be relabeled as real-time historical forecasts')
check('V73_RELEASE_PLAN_PREFLIGHT',bool(globals().get('V73_PUBLICATION_PLAN_PREFLIGHT_PASSED')), '8 assets path validation must pass before input download')
check('V73_REAL_PAIRED_TESTS',len(V73_PAIRED_TESTS)>0 and V73_PAIRED_TESTS.n_paired_observations.ge(1).all())
check('V73_MODEL_RESULT_PRESERVATION',bool(V73_PAIRED_AUDIT.get('training_artifacts_unchanged')) and len(MODEL_RECORDS)>=233 and len(METRICS)>=254)
check('V73_PAIRED_SOURCE_COVERAGE',V73_PAIRED_AUDIT.get('prediction_rows_input')==PRED_TOTAL and V73_PAIRED_AUDIT.get('predeclared_baseline_groups',0)>0)
check('V73_INFERENCE_MULTIPLICITY',bool(V73_PAIRED_TESTS.p_holm.dropna().between(0,1).all()))
_v73_stage_file=ROOT/'logs'/'stages_v73.jsonl'
_v73_stage_entries=[json.loads(x) for x in _v73_stage_file.read_text(encoding='utf-8').splitlines()] if _v73_stage_file.is_file() else []
_v73_session_stages=[x['stage'] for x in _v73_stage_entries if x.get('run_session_id')==RUN_SESSION_ID]
check('V73_DURABLE_STAGE_JOURNAL',_v73_session_stages==list(PIPELINE_STAGES)[:13],
      'Expected 13 completed research/export stages in this run session before QA and GitHub')
LOCAL_QA=pd.DataFrame(LOCAL_CHECKS)
progress_event('QA_AND_PUBLICATION',f'Проверено {len(LOCAL_QA):,} контролей; ошибок: {int(LOCAL_QA.status.eq("FAIL").sum())}',announce=True)
# Перезаписываем локальный manifest после QA, но Excel/ZIP уже фиксированы по SHA.
manifest['local_qa']=LOCAL_CHECKS
manifest['status']['technical']='PASS' if LOCAL_QA.status.eq('PASS').all() else 'FAIL'
manifest['status']['publication']='NOT_ATTEMPTED'
manifest['status']['overall']=(STATE['scientific'] if manifest['status']['technical']=='PASS'
                              else 'PARTIAL_LOCAL_QA')
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Independent QA:',int(LOCAL_QA.status.eq('PASS').sum()),'PASS /',len(LOCAL_QA),'checks')

# Публикация отсюда не выполняется, даже при FAIL локальной QA.
# Новый финальный этап публикует артефакты с честным статусом QA; сами данные не теряются.
PUBLICATION_RECEIPT={'status':'PENDING_FINAL_STAGE'}
STATE['publication']='PENDING_FINAL_STAGE'

print('ML-05 V7.3.0:',STATE['scientific'],'| technical:',manifest['status']['technical'],
      '| GitHub:',STATE['publication'])
print('Output paths:',EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)

mark_stage('QA_AND_PUBLICATION')
# Heartbeat работает и во время сборки/загрузки итогового ZIP; закрываем после последней ячейки.


## Приёмочный контракт V7.3.0

- **TECHNICAL_PASS**: все 52 исходных набора проверены по SHA, 7 модулей имеют журналы исполнения, обязательные по данным расчёты завершены, модельные файлы сверены по SHA и локальная QA без ошибок.
- **SCIENTIFIC_LIMITED**: измеряемая цель не идентифицируется в первичных данных. Этот статус честно сохраняется в отчёте и не подменяется иной величиной.
- **REPLACE**: если локальные вычисления и QA успешны, публикуется новый Git-коммит, заменяющий только целевой ML-05 сегмент. При техническом FAIL полные результаты могут быть сохранены как черновой Release, но старая рабочая версия не удаляется.


- **SQL_PREFLIGHT**: рабочие DuckDB-запросы E02 сначала исполняются на маленьком наборе, затем на полном источнике; ошибка синтаксиса приводит к остановке до многогигабайтных операций.

## 14. Итог: компактная автозагрузка отчёта → проверенный GitHub Release → атомарная замена старого ML-05

In [ ]:
# ML-05 V7.3.0 · COMPLETE PIPELINE / VERIFIED GITHUB REPLACEMENT / COMPACT DOWNLOAD
# FULL -> 52 SHA verified -> seven modules -> QA -> small report -> draft Release -> atomic segment replace.
# Does not delete Stage I, unrelated projects or historical git commits. Every destructive change is prefix-scoped.
import json, os, math, hashlib, base64, zipfile, shutil, re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

progress_event('DOWNLOAD_RESULTS','V7.3: итоговый QA, компактная аналитика и безопасная замена ML-05',announce=True)
if CFG.MODE != 'FULL': raise RuntimeError('V7.3: для публикации необходим FULL')
if not CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB: raise RuntimeError('V7.3: публикация в приватный GitHub выключена')
if not CFG.STORE_DERIVED_RESTRICTED_DATA: raise RuntimeError('V7.3: не разрешено публиковать производные результаты')
if not api_json('GET',BASE).get('private'): raise RuntimeError('V7.3: репозиторий не приватный')
if len(ASSETS)!=52 or len(INPUT_SHA)!=52 or len(V5_SOURCE_EVIDENCE)!=52:
    raise RuntimeError('V7.3: 52 источника не подтверждены SHA и профилями')
if len(V5_LEDGER)!=7 or V5_LEDGER.module.nunique()!=7:
    raise RuntimeError('V7.3: отсутствует одна или несколько приёмок семи модулей')

out=ROOT/'exports'; out.mkdir(parents=True,exist_ok=True)
needed=[Path(p) for p in (EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)]
for p in needed:
    if not p.is_file() or p.stat().st_size==0: raise RuntimeError('V7.3: отсутствует файл '+str(p))
_saved=json.loads(Path(MANIFEST).read_text(encoding='utf-8'))
if _saved.get('version')!=CFG.VERSION or _saved.get('source_commit')!=SOURCE_COMMIT:
    raise RuntimeError('V7.3: манифест не соответствует активному запуску')
for p in (EXCEL,EVIDENCE,NOTEBOOK):
    if _saved.get('output_sha256',{}).get(Path(p).name)!=hash_file(p):
        raise RuntimeError('V7.3: нарушена контрольная сумма '+str(p))

# Source capture is a deterministic embedded copy of this notebook, not a stale Colab get_ipynb.
recovered=notebook_from_embedded_source()
if hash_file(NOTEBOOK)!=hashlib.sha256(nbformat.writes(recovered).encode('utf-8')).hexdigest():
    # nbformat serialization may differ in insignificant line breaks. Revalidate actual source instead.
    _check=nbformat.read(NOTEBOOK,as_version=4)
    if sum(c.cell_type=='code' for c in _check.cells)!=sum(c.cell_type=='code' for c in recovered.cells):
        raise RuntimeError('V7.3: сохранённый ноутбук не совпал с встроенным источником')
    _check_src=[c.source for c in _check.cells]
    if _check_src != [c.source for c in recovered.cells]:
        raise RuntimeError('V7.3: исходный код ноутбука отличается от эталона')

sha_evidence=hash_file(EVIDENCE)
run_key=hashlib.sha256((CFG.VERSION+'|'+SOURCE_COMMIT+'|'+sha_evidence).encode()).hexdigest()[:20]
release_tag='ml05-v7.3.0-full-'+run_key
prefix=CFG.OUTPUT_PREFIX.rstrip('/')+'/'
if prefix!='stage2/ml05_education/': raise RuntimeError('V7.3: удаление разрешено ТОЛЬКО в stage2/ml05_education/')
qa_ok=bool(len(LOCAL_QA) and LOCAL_QA.status.eq('PASS').all())
compute_ok=bool(V5_COMPUTATIONAL_PASS and ADR_COMPUTATIONAL_PASS)
science_ok=bool(V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)
can_replace=bool(qa_ok and compute_ok)

# V7 source/model lineage is derived from exact input panels and stored feature lists.
# No false claim that all 52 sources enter predictive X; all 52 have full numerical research profiles.
source_ledger=V7_SOURCE_LINEAGE.to_dict('records')
adr_ledger_csv=out/'ML5_V730_RECONSTRUCTED_EXPERIMENTS.csv'
pd.DataFrame(ADR_EXPERIMENTS).to_csv(adr_ledger_csv,index=False,encoding='utf-8-sig')
adr_summary_file=out/'ML5_V730_RECONSTRUCTION_ACCEPTANCE.json'
adr_summary_file.write_text(json.dumps(ADR_FINAL_REPORT,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
adr_lineage_csv=out/'ML5_V730_RECONSTRUCTED_MODEL_LINEAGE.csv'
ADR_RECONSTRUCTED_LINEAGE.to_csv(adr_lineage_csv,index=False,encoding='utf-8-sig')
source_csv=out/'ML5_V730_SOURCE_COVERAGE_52.csv'
V7_SOURCE_LINEAGE.to_csv(source_csv,index=False,encoding='utf-8-sig')
lineage_csv=out/'ML5_V730_MODEL_LINEAGE.csv'
V7_MODEL_LINEAGE.to_csv(lineage_csv,index=False,encoding='utf-8-sig')
gates_csv=out/'ML5_V730_SEVEN_SCIENTIFIC_GATES.csv'
V7_SCIENCE_GATES.to_csv(gates_csv,index=False,encoding='utf-8-sig')
source_stats_csv=out/'ML5_V730_SOURCE_RESEARCH_52.csv'
V7_SOURCE_AUDIT.to_csv(source_stats_csv,index=False,encoding='utf-8-sig')
module_csv=out/'ML5_V730_SEVEN_MODULES.csv';V5_LEDGER.to_csv(module_csv,index=False,encoding='utf-8-sig')
metrics_csv=out/'ML5_V730_METRICS.csv';pd.DataFrame(METRICS).to_csv(metrics_csv,index=False,encoding='utf-8-sig')
models_csv=out/'ML5_V730_MODELS.csv';pd.DataFrame(V5_MODEL_QA).to_csv(models_csv,index=False,encoding='utf-8-sig')
qa_csv=out/'ML5_V730_TECHNICAL_QA.csv';LOCAL_QA.to_csv(qa_csv,index=False,encoding='utf-8-sig')
issues_csv=out/'ML5_V730_SCIENTIFIC_LIMITATIONS.csv';pd.DataFrame(ISSUES).to_csv(issues_csv,index=False,encoding='utf-8-sig')
# V7.3 · reproducibility and independent research acceptance documents.
v72_files={}
for _name,_df in {
    'V72_RECONSTRUCTION_EVIDENCE':V72_RECON_DF,
    'V72_SOURCE_PANEL_MODEL':V72_MODEL_PROVENANCE,
    'V72_SEVEN_MODULE_DECISIONS':V72_MODULE_ACCEPTANCE,
    'V72_ASOF_VINTAGE':V72_ASOF_AUDIT,
    'V72_MODEL_SELECTION':V72_MODEL_SELECTION,
    'V72_OD_COVERAGE':V72_OD_COVERAGE,
    'V72_ADMISSION_DECISIONS':V72_ADMISSION_DECISION,
    'V72_GEO_CONFIDENCE':V72_SPATIAL_CONFIDENCE,
    'V72_ENGINEERING_GATES':pd.DataFrame([{'gate':k,'passed':v} for k,v in V72_ENGINEERING_GATES.items()])
}.items():
    _path=out/('ML5_'+_name+'.csv')
    _df.to_csv(_path,index=False,encoding='utf-8-sig')
    v72_files[_name]=_path
_v72_repro_file=out/'ML5_V72_REPRODUCIBILITY.json'
_v72_repro_file.write_text(json.dumps(V72_REPRODUCIBILITY,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
v72_files['V72_REPRODUCIBILITY']=_v72_repro_file


v73_paired_csv=out/'ML5_V730_PAIRED_TESTS.csv'
V73_PAIRED_TESTS.to_csv(v73_paired_csv,index=False,encoding='utf-8-sig')
v73_strata_csv=out/'ML5_V730_ERROR_STRATA.csv'
V73_ERROR_STRATA.to_csv(v73_strata_csv,index=False,encoding='utf-8-sig')
v73_audit_json=out/'ML5_V730_PAIRED_AUDIT.json'
v73_audit_json.write_text(json.dumps(V73_PAIRED_AUDIT,ensure_ascii=False,default=str,indent=2),encoding='utf-8')

summary={
    'version':CFG.VERSION,'run_key':run_key,'git_source_commit':SOURCE_COMMIT,
    'catalog_assets':len(ASSETS),'sha_verified':len(INPUT_SHA),
    'content_profiled':sum(x.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'direct_model_sources':int(V7_SOURCE_LINEAGE.direct_model_input.sum()),
    'model_lineage_records':len(V7_MODEL_LINEAGE),'source_profiles_researched':int(V7_SOURCE_AUDIT.all_values_profiled.sum()),
    'seven_modules':V5_MODULE_ACCEPTANCE,'v7_scientific_gates':V7_SCIENCE_GATES.to_dict('records'),'qa_pass':int(LOCAL_QA.status.eq('PASS').sum()),
    'qa_fail':int(LOCAL_QA.status.eq('FAIL').sum()),'computational_pass':can_replace,
    'original_seven_targets_scientifically_identified':science_ok,
    'prediction_rows':PRED_TOTAL,'metric_rows':len(METRICS),'model_count':len(MODEL_FILES),
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,
    'scientific_caveat':'Невозможный по имеющимся данным прогноз не подменяется иным целевым показателем.',
    'adr_reconstruction':ADR_FINAL_REPORT,
    'v72_engineering_gates':V72_ENGINEERING_GATES,
    'v73_publication_preflight':V73_PUBLICATION_PLAN_PREFLIGHT_PASSED,
    'v72_scientific_module_acceptance':V72_MODULE_ACCEPTANCE.to_dict('records'),
    'v72_reproduction_fingerprint':V72_REPRODUCIBILITY,
    'v73_paired_tests':V73_PAIRED_AUDIT
}
summary_file=out/'ML5_V730_FINAL_ACCEPTANCE.json'
summary_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# Reports are downloaded before remote upload; complete evidence and heavy models NEVER enter this ZIP.
plots=sorted({Path(p) for p in FIGURES if Path(p).is_file()},key=lambda p:p.name)
reports=[Path(EXCEL),Path(NOTEBOOK),Path(MANIFEST),source_csv,lineage_csv,gates_csv,source_stats_csv,module_csv,metrics_csv,models_csv,qa_csv,issues_csv,summary_file,adr_ledger_csv,adr_summary_file,adr_lineage_csv]+list(v72_files.values())+[v73_paired_csv,v73_strata_csv,v73_audit_json]+plots
small_zip=out/('ML5_V730_ANALYTICS_'+run_key+'.zip')
with zipfile.ZipFile(small_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for p in reports:
        z.write(p,('figures/' if p in plots else 'reports/')+p.name)
    z.writestr('reports/checksums.json',json.dumps({p.name:hash_file(p) for p in reports},indent=2))
with zipfile.ZipFile(small_zip) as z:
    if z.testzip() is not None: raise RuntimeError('V7.3: повреждён отчётный ZIP')
if small_zip.stat().st_size>100*2**20:raise RuntimeError('V7.3: отчётный архив >100 MiB, проверьте содержимое')
progress_event('DOWNLOAD_RESULTS',f'V7.3: готов отчётный архив {small_zip.stat().st_size/2**20:.1f} МиБ',announce=True)
print('АНАЛИТИКА:',small_zip,'|','EVIDENCE только в GitHub:',EVIDENCE.stat().st_size/2**20,'МиБ',flush=True)
if CFG.DOWNLOAD_COMPACT_REPORTS:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(small_zip))
    except ImportError:
        from IPython.display import FileLink,display
        display(FileLink(str(small_zip)))

# Precompute resumable chunks on disk; verify recombined SHA before GitHub upload.
size_part=int(CFG.GH_RELEASE_PART_MIB)*2**20
if not 16<=int(CFG.GH_RELEASE_PART_MIB)<=256:raise RuntimeError('V7.3: part size вне 16..256 MiB')
chunk_dir=out/('release_chunks_'+run_key);chunk_dir.mkdir(parents=True,exist_ok=True)
chunk_index={'version':CFG.VERSION,'source_evidence':Path(EVIDENCE).name,
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,'parts':[]}
with Path(EVIDENCE).open('rb') as src:
    total=math.ceil(EVIDENCE.stat().st_size/size_part)
    for i in range(total):
        name=f'ML5_EVIDENCE.zip.part{i+1:03d}'
        p=chunk_dir/name
        n=min(size_part,EVIDENCE.stat().st_size-i*size_part)
        if p.exists() and p.stat().st_size==n:
            src.seek(n,1)
        else:
            tmp=p.with_suffix('.part.tmp')
            with tmp.open('wb') as dest:
                remaining=n
                while remaining:
                    buf=src.read(min(4*2**20,remaining))
                    if not buf:raise IOError('V7.3: преждевременный конец evidence')
                    dest.write(buf);remaining-=len(buf)
            tmp.replace(p)
        chunk_index['parts'].append({'name':name,'bytes':n,'sha256':hash_file(p)})
        progress_event('DOWNLOAD_RESULTS',f'Локальная часть evidence {i+1}/{total}',announce=True)
verify_hash=hashlib.sha256()
for part in chunk_index['parts']:
    with (chunk_dir/part['name']).open('rb') as f:
        for buf in iter(lambda:f.read(4*2**20),b''):verify_hash.update(buf)
if verify_hash.hexdigest()!=sha_evidence:raise RuntimeError('V7.3: разбитый evidence SHA не совпадает с исходным')
index_file=out/'ML5_V730_RELEASE_PARTS.json'
index_file.write_text(json.dumps(chunk_index,ensure_ascii=False,indent=2),encoding='utf-8')

# Helpers for GitHub. An existing asset is reused ONLY if GitHub's SHA-256 and size match.
def remote_release_assets(release_id):
    return {a['name']:a for a in api_json('GET',BASE+f'/releases/{release_id}/assets',params={'per_page':100})}

def upload_release_asset(release_id,p,mime):
    p=Path(p);sha=hash_file(p)
    already=remote_release_assets(release_id).get(p.name)
    if already and already.get('state')=='uploaded' and already.get('digest')=='sha256:'+sha and int(already.get('size',-1))==p.stat().st_size:
        progress_event('DOWNLOAD_RESULTS','GitHub SHA уже совпадает: '+p.name,announce=True)
        return already
    if already:
        resp=SESS.delete(BASE+'/releases/assets/'+str(already['id']),timeout=(30,120))
        status=resp.status_code;resp.close()
        if status!=204:raise RuntimeError('V7.3: не удалён повреждённый Release-asset '+p.name)
    upload_url=f'https://uploads.github.com/repos/{CFG.REPO}/releases/{release_id}/assets'
    progress_event('DOWNLOAD_RESULTS',f'GitHub: {p.name} · {p.stat().st_size/2**20:.1f} МиБ',announce=True)
    try:
        with p.open('rb') as f:
            resp=SESS.post(upload_url,params={'name':p.name},headers={'Content-Type':mime},data=f,timeout=(30,3600))
        status=resp.status_code
        if status==201:obj=resp.json()
        else:obj=None
        resp.close()
    except Exception:
        obj=None
    if obj is None:
        obj=remote_release_assets(release_id).get(p.name)
    if not obj or obj.get('state')!='uploaded' or obj.get('digest')!='sha256:'+sha or int(obj.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V7.3: ошибка GitHub asset '+p.name+'; можно повторить только финальную ячейку')
    return obj

branch=CFG.TARGET_BRANCH
remote_head=repo_head(branch)
if remote_head!=TARGET_HEAD_AT_PREFLIGHT:
    # A rerun after the successful commit is allowed, but unrelated changes must not be overwritten.
    try:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        already_done=(existing.get('run_key')==run_key and existing.get('evidence_sha256')==sha_evidence)
    except Exception:
        already_done=False
    if not already_done:raise RuntimeError('V7.3: ветка изменилась после preflight. Не перезаписываю чужие изменения.')
else:already_done=False

# Atomic replacement is permitted ONLY if all computational checks and local QA passed.
# Otherwise publish a diagnostic draft Release; leave old active segment unchanged.
if not can_replace:
    print('V7.3: QA/вычисления не полностью пройдены. Отчёты скачаны. Старый рабочий сегмент НЕ удаляю.',flush=True)

publication_plan=out/'ML5_V730_PUBLICATION_PLAN.json'
planned=None
if publication_plan.is_file():
    try:
        j=json.loads(publication_plan.read_text('utf-8'))
        if j.get('run_key')==run_key and j.get('base_sha')==TARGET_HEAD_AT_PREFLIGHT:planned=j
    except Exception:pass

publication_receipt={
    'version':CFG.VERSION,'run_key':run_key,'release_tag':release_tag,
    'release_url':f'https://github.com/{CFG.REPO}/releases/tag/{release_tag}',
    'evidence_sha256':sha_evidence,'parts_manifest':index_file.name,
    'source_commit':SOURCE_COMMIT,'source_count':52,'computational_pass':can_replace,
    'scientific_targets_all_identified':science_ok,
    'reconstruction_computational_pass':ADR_COMPUTATIONAL_PASS,'target_prefix':prefix,
    'replacement_policy':'atomic current ML-05 tree only; preserve git history and Stage I',
    'remote_verification_state':'PREPARED_NOT_YET_VERIFIED'
}
receipt_path=out/'publication_receipt.json'
receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
pubmanifest=out/'run_manifest_publication.json'
pubm=json.loads(Path(MANIFEST).read_text('utf-8'))
pubm['publication_status']='PENDING_GITHUB_VERIFICATION' if can_replace else 'DRAFT_PARTIAL_FAIL'
pubm['status']['publication']=pubm['publication_status']
pubm['publication_reference']={'release_tag':release_tag,'evidence_sha256':sha_evidence}
pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

if already_done:
    _remote_manifest=read_git_blob(prefix+'run_manifest.json',ref=remote_head)
    _remote_receipt=read_git_blob(prefix+'publication_receipt.json',ref=remote_head)
    if json.loads(_remote_manifest).get('status',{}).get('publication')!='PUBLISHED_GITHUB_VERIFIED':
        raise RuntimeError('V7.3: final verified status missing on a re-run')
    pubmanifest.write_bytes(_remote_manifest)
    receipt_path.write_bytes(_remote_receipt)

# What remains in active stage2/ml05_education AFTER successful update.
target_files={
    prefix+'ML5_EDUCATION_MIGRATION.ipynb':Path(NOTEBOOK),
    prefix+'ML5_EDUCATION_MIGRATION_RESULTS.xlsx':Path(EXCEL),
    prefix+'run_manifest.json':pubmanifest,
    prefix+'publication_receipt.json':receipt_path,
    prefix+index_file.name:index_file,
    prefix+summary_file.name:summary_file,
    prefix+source_csv.name:source_csv,
    prefix+lineage_csv.name:lineage_csv,
    prefix+gates_csv.name:gates_csv,
    prefix+source_stats_csv.name:source_stats_csv,
    prefix+module_csv.name:module_csv,
    prefix+metrics_csv.name:metrics_csv,
    prefix+qa_csv.name:qa_csv,
    prefix+adr_ledger_csv.name:adr_ledger_csv,
    prefix+adr_summary_file.name:adr_summary_file,
    prefix+adr_lineage_csv.name:adr_lineage_csv,
    **{prefix+p.name:p for p in v72_files.values()},
    prefix+'ML5_V730_PAIRED_TESTS.csv':v73_paired_csv,
    prefix+'ML5_V730_ERROR_STRATA.csv':v73_strata_csv,
    prefix+'ML5_V730_PAIRED_AUDIT.json':v73_audit_json,
}
if not all(k.startswith(prefix) for k in target_files): raise RuntimeError('V7.3: нарушена изоляция сегмента')

# Prepare a candidate Git commit *before* creating a release; tag points to V7.3 source.
commit_sha=remote_head if already_done else None
if can_replace and not already_done:
    if planned:
        candidate=api_json('GET',BASE+'/git/commits/'+planned['candidate_sha'])
        commit_sha=planned['candidate_sha']
        if candidate['parents'][0]['sha']!=remote_head:raise RuntimeError('V7.3: кандидат не является потомком HEAD')
    else:
        old_tree=api_json('GET',BASE+'/git/trees/'+remote_head,params={'recursive':1})
        if old_tree.get('truncated'):raise RuntimeError('V7.3: неполное дерево Git, удаление запрещено')
        old_paths={x['path'] for x in old_tree.get('tree',[]) if x.get('type')=='blob' and x.get('path','').startswith(prefix)}
        obsolete=sorted(old_paths-set(target_files))
        changes=[]
        for filename,path in sorted(target_files.items()):
            binary=Path(path).read_bytes() # Small files only; evidence is Release-only.
            if len(binary)>45*2**20:raise RuntimeError('V7.3: слишком большой Git blob '+filename)
            blob=api_json('POST',BASE+'/git/blobs',json={'content':base64.b64encode(binary).decode(),'encoding':'base64'})
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':blob['sha']})
        for filename in obsolete:
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':None})
        old_commit=api_json('GET',BASE+'/git/commits/'+remote_head)
        tree=api_json('POST',BASE+'/git/trees',json={'base_tree':old_commit['tree']['sha'],'tree':changes})
        candidate=api_json('POST',BASE+'/git/commits',json={
            'message':f'ML-05 V7.3.0 FULL verified replacement: {run_key} (old segment files retired)',
            'tree':tree['sha'],'parents':[remote_head]})
        commit_sha=candidate['sha']
        plan={'run_key':run_key,'base_sha':remote_head,'candidate_sha':commit_sha,
              'obsolete_paths':obsolete,'new_files':sorted(target_files)}
        publication_plan.write_text(json.dumps(plan,ensure_ascii=False,indent=2),encoding='utf-8')
        progress_event('DOWNLOAD_RESULTS',f'Кандидат Git commit готов, старых файлов к замене {len(obsolete)}',announce=True)

# Create a draft Release. The tag MUST point to the prepared V7.3 commit.
all_releases=api_json('GET',BASE+'/releases',params={'per_page':100})
release=next((x for x in all_releases if x.get('tag_name')==release_tag),None)
if release is None:
    release=api_json('POST',BASE+'/releases',json={
        'tag_name':release_tag,'target_commitish':commit_sha or remote_head,
        'name':'ML-05 V7.3.0 FULL '+run_key,'draft':True,'prerelease':True,
        'body':f'Stage I 52/52. Computational PASS: {can_replace}. Scientific 7/7 identified: {science_ok}. '
               f'Evidence SHA-256 {sha_evidence}. Full results belong to this private repository.'})
release_id=release['id']
_v73_uploads=v73_release_asset_plan(index_file,chunk_dir,chunk_index,small_zip)
for _v72_i,(p,mime) in enumerate(_v73_uploads,1):
    progress_event('DOWNLOAD_RESULTS',f'GitHub asset {_v72_i}/{len(_v73_uploads)}: {Path(p).name}',_v72_i-1,len(_v73_uploads),announce=True)
    upload_release_asset(release_id,p,mime)
    progress_event('DOWNLOAD_RESULTS',f'Проверено {_v72_i}/{len(_v73_uploads)} assets',_v72_i,len(_v73_uploads),announce=True)
assets=remote_release_assets(release_id)
needed_assets=[p for p,_mime in _v73_uploads]
for p in needed_assets:
    a=assets.get(p.name)
    if not a or a.get('state')!='uploaded' or a.get('digest')!='sha256:'+hash_file(p) or int(a.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V7.3: удалённый Release-asset не прошёл приёмку '+p.name)

if can_replace:
    if not already_done:
        if repo_head(branch)!=remote_head:raise RuntimeError('V7.3: конфликт обновления Git ветки, старый сегмент не изменён')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={'sha':commit_sha,'force':False})
        if repo_head(branch)!=commit_sha:raise RuntimeError('V7.3: не подтверждён Git HEAD после коммита')
    # Check EVERY committed file by round-trip SHA256 from GitHub, then ensure old files are absent.
    for filename,path in target_files.items():
        remote=read_git_blob(filename,ref=commit_sha)
        if hashlib.sha256(remote).hexdigest()!=hash_file(path):
            raise RuntimeError('V7.3: GitHub roundtrip SHA mismatch '+filename)
    finaltree=api_json('GET',BASE+'/git/trees/'+commit_sha,params={'recursive':1})
    if finaltree.get('truncated'):raise RuntimeError('V7.3: Git tree truncated in final check')
    now={x['path'] for x in finaltree['tree'] if x['type']=='blob' and x['path'].startswith(prefix)}
    if now!=set(target_files):raise RuntimeError('V7.3: остались устаревшие файлы ML-05 или пропали новые')
    if release.get('draft'):
        release=api_json('PATCH',BASE+'/releases/'+str(release_id),json={'draft':False,'prerelease':True})
    published=api_json('GET',BASE+'/releases/'+str(release_id))
    if published.get('draft') or published.get('tag_name')!=release_tag:
        raise RuntimeError('V7.3: новый Release не подтвердился')
    # Confirm the tag points to the new commit, not the retired V2 commit.
    tagref=api_json('GET',BASE+'/git/ref/tags/'+quote(release_tag,safe=''))
    tagsha=tagref.get('object',{}).get('sha')
    if tagsha!=(commit_sha if not already_done else existing.get('prepared_candidate_sha')):
        raise RuntimeError('V7.3: release tag does not point to the checked code commit')
    # V7.3: second metadata-only commit records independently verified publication.
    # The first verified commit atomically replaces only the ML-05 tree; this second one updates
    # the publication state after (not before) the GitHub/Release round-trip succeeds.
    if not already_done:
        pubm['status']['publication']='PUBLISHED_GITHUB_VERIFIED'
        pubm['publication_status']='PUBLISHED_GITHUB_VERIFIED'
        pubm['publication_reference']['candidate_commit_sha']=commit_sha
        pubm['publication_reference']['release_id']=release_id
        pubm['publication_reference']['verification']='GIT_TREE_AND_RELEASE_ASSET_DIGESTS_PASS'
        pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
        publication_receipt.update({'remote_verification_state':'PASS_GITHUB',
            'prepared_candidate_sha':commit_sha,'release_id':release_id,
            'release_digests_confirmed':True,'stage1_unchanged':True})
        receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
        _base_commit=api_json('GET',BASE+'/git/commits/'+commit_sha)
        _changes=[]
        for _path,_file in ((prefix+'run_manifest.json',pubmanifest),
                            (prefix+'publication_receipt.json',receipt_path)):
            _data=Path(_file).read_bytes()
            _blob=api_json('POST',BASE+'/git/blobs',json={
                'content':base64.b64encode(_data).decode(),'encoding':'base64'})
            _changes.append({'path':_path,'mode':'100644','type':'blob','sha':_blob['sha']})
        _tree=api_json('POST',BASE+'/git/trees',json={
            'base_tree':_base_commit['tree']['sha'],'tree':_changes})
        _receipt_commit=api_json('POST',BASE+'/git/commits',json={
            'message':'ML-05 V7.3.0: verified release and GitHub publication receipt '+run_key,
            'tree':_tree['sha'],'parents':[commit_sha]})['sha']
        if repo_head(branch)!=commit_sha:
            raise RuntimeError('V7.3: concurrent branch modification before final receipt')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={
            'sha':_receipt_commit,'force':False})
        if repo_head(branch)!=_receipt_commit:
            raise RuntimeError('V7.3: final publication receipt commit was not installed')
        for _path,_file in ((prefix+'run_manifest.json',pubmanifest),
                            (prefix+'publication_receipt.json',receipt_path)):
            if hashlib.sha256(read_git_blob(_path,ref=_receipt_commit)).hexdigest()!=hash_file(_file):
                raise RuntimeError('V7.3: final publication receipt failed round-trip '+_path)
        commit_sha=_receipt_commit
        print('V7.3: final GitHub publication receipt independently verified',commit_sha,flush=True)
    else:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        if existing.get('remote_verification_state')!='PASS_GITHUB':
            raise RuntimeError('V7.3: rerun found incomplete metadata receipt; manual recovery required')
        commit_sha=remote_head
    STATE['publication']='PASS_GITHUB_V73_VERIFIED'
    print('ЗАМЕНА ML-05 ПОДТВЕРЖДЕНА:',commit_sha,flush=True)
    print('Только новый набор файлов в',prefix,'; старые файлы удалены из текущего дерева',flush=True)
else:
    STATE['publication']='DRAFT_REVIEW_ONLY_OLD_SEGMENT_PRESERVED'
    print('Новая доказательная база в черновом Release. Старый сегмент сохранён из-за FAIL.',flush=True)

receipt={**publication_receipt,'commit_sha':commit_sha,'release_id':release_id,
    'final_publication_status':STATE['publication'],
    'time_utc':datetime.now(timezone.utc).isoformat()}
(out/'ML5_V730_PUBLICATION_CONFIRMATION.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print('GitHub Release:',publication_receipt['release_url'],flush=True)
print('Небольшой отчётный архив:',small_zip,flush=True)
print('Научная приёмка:', 'PASS' if science_ok else 'LIMITED_BY_OBSERVABILITY',flush=True)
if can_replace:mark_stage('DOWNLOAD_RESULTS')
else:progress_event('DOWNLOAD_RESULTS','Публикация частичного исследования; 100% исполнения не подтверждены',announce=True)
bar=globals().get('MAIN_PROGRESS')
if bar is not None: bar.close()
